### Import packeges

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
from scipy.interpolate import griddata
from pathlib import Path
import h5py
import sys
from pathlib import Path

# Where am I running?
try:
    # Normal script
    here = Path(__file__).resolve().parent
except NameError:
    # Notebook / REPL
    here = Path.cwd()

phys_const_path = (here / '..' / 'phys_const').resolve()
sys.path.append(str(phys_const_path))

nsm_plots_path = (here / '..' / 'nsm_plots').resolve()
sys.path.append(str(nsm_plots_path))

nsm_plots_postproc = (here / '..' / 'nsm_instabilities').resolve()
sys.path.append(str(nsm_plots_postproc))

import phys_const as pc
import plot_functions as pf
import functions_angular_crossings as fac

### File path

In [ ]:
##################################################################################################################
# Simulation paths

direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_0'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-5'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-4'
direct = '/home/erick/gw170817_1.00ye_locsim/CFI/paper_cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_cdt_10.0m_att_1e-3'

parfile = '/plt00000_particles'
ncell = (1,1,1) # number of cells in each direction
domain = (1e5, 1e5, 1e5) # cm
num_particles_per_energy_bin = 92
num_energy_bins = 13
direct_interpolated_data = direct

##################################################################################################################

all_par_files = sorted(glob.glob(direct + '/plt*_particles'))
plt_numbers = [f.split('/')[-1].split('plt')[1].split('_particles')[0] for f in all_par_files]
plt_numbers = sorted(plt_numbers, key=lambda x: int(x))
all_par_files = [f'plt{plt_num}_particles' for plt_num in plt_numbers]

finaldir = direct.split('/')[-1]

# Calculate the volume of a single cell and the total volume
cellvolume = (domain[0] / ncell[0]) * (domain[1] / ncell[1]) * (domain[2] / ncell[2]) # cm^3
domainvolume = cellvolume * (ncell[0] * ncell[1] * ncell[2]) # cm^3

### Define cell index to be analyzed

In [ ]:
cell_index_i = 0
cell_index_j = 0
cell_index_k = 0

### Cell to be analized

In [ ]:
# Get the cell indices
cell_file_names = glob.glob(direct + parfile + '/cell_*_*_*')
cell_file_names = [file_name.split('/')[-1] for file_name in cell_file_names]
x_cell_ind = np.array([int(file_name.split('_')[1]) for file_name in cell_file_names])
y_cell_ind = np.array([int(file_name.split('_')[2]) for file_name in cell_file_names])
z_cell_ind = np.array([int((file_name.split('_')[3]).split('.')[0]) for file_name in cell_file_names])
cell_indices = np.array(list(zip(x_cell_ind, y_cell_ind, z_cell_ind)))
print('Number of cells:', len(cell_indices))
print(f'shape of cell_indices: {cell_indices.shape}')

# Get the cell indices for ix fix but iy and iz varying all available cells
x_idx_slice = cell_index_i
mask_yz_slice = cell_indices[:,0] == x_idx_slice # fixing the x index in this value
cell_indices_yz_slice = cell_indices[mask_yz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_yz_slice[:,1], cell_indices_yz_slice[:,2], color='b')
ax.scatter(cell_index_j, cell_index_k, color='r')
ax.set_xlabel('$i_y$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_x=${x_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iy fix but ix and iz varying all available cells
y_idx_slice = cell_index_j
mask_xz_slice = cell_indices[:,1] == y_idx_slice # fixing the y index in this value
cell_indices_xz_slice = cell_indices[mask_xz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xz_slice[:,0], cell_indices_xz_slice[:,2], color='b')
ax.scatter(cell_index_i, cell_index_k, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_y=${y_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iz fix but ix and iy varying all available cells
z_idx_slice = cell_index_k
mask_xy_slice = cell_indices[:,2] == z_idx_slice # fixing the z index in this value
cell_indices_xy_slice = cell_indices[mask_xy_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xy_slice[:,0], cell_indices_xy_slice[:,1], color='b')
ax.scatter(cell_index_i, cell_index_j, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_y$')
ax.set_title(f'Cell indices for fixed $i_z=${z_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Combine all cell indices from the three slices
# cell_indices_all = np.concatenate((cell_indices_yz_slice, cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.concatenate((cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.unique(cell_indices_all, axis=0)

### Getting temperature, electron fraction and density

In [ ]:
rho_ye_T_h5py = h5py.File(direct+'/rho_Ye_T.hdf5', 'r')

Temperature_MeV = np.array(rho_ye_T_h5py['/T_Mev'])[cell_index_i,cell_index_j,cell_index_k]
rho_g_cm3 = np.array(rho_ye_T_h5py['/rho_g|ccm'])[cell_index_i,cell_index_j,cell_index_k]
Ye = np.array(rho_ye_T_h5py['/Ye'])[cell_index_i,cell_index_j,cell_index_k]

print(f'rho_cons = {rho_g_cm3} # g/ccm')
print(f'T_cons = {Temperature_MeV} # MeV')
print(f'Ye_cons = {Ye} # n_electron - n_positron / n_barions')

neutrons_number_density_cm3 = rho_g_cm3 * (1.0 - Ye) / pc.PhysConst.Mp
protons_number_density_cm3 = rho_g_cm3 * Ye / pc.PhysConst.Mp

rho_ye_T_h5py.close()

### Compute ELN number density

In [ ]:
energybinsMeV = np.array([
    1, 3, 5.2382, 8.0097, 11.442, 15.691, 20.953, 27.468,
    35.536, 45.525, 57.895, 73.212, 92.178,
    # 115.66, 144.74, 180.75, 225.33, 280.54
])

energybinstopMeV = np.array([
    2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 102.67, 
    # 128.65, 160.83, 200.67, 250, 311.08
])

energybinsbottomMeV = np.array([
    0, 2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 
    # 102.67, 128.65, 160.83, 200.67, 250
])

nu_e_opacities_inv_cm = [1.1279864349828499e-08, 4.7428025628186796e-08, 1.321984832248114e-07, 3.294158114604995e-07, 7.528396092747342e-07, 1.5644452508683583e-06, 2.9487331095815796e-06, 5.1520838790595784e-06, 8.605493196949606e-06, 1.4043114431085709e-05, 2.262451577257334e-05, 3.6135649577238473e-05, 5.731900449884377e-05, 9.036764140607461e-05, 0.00014163060110313624, 0.00022059108455224182, 0.00034117744506685334, 0.0005234543477497763]
nu_x_opacities_inv_cm = [2.174662492044255e-10, 3.0391841308327543e-10, 4.466580398128411e-10, 6.371159236091845e-10, 8.969420647822444e-10, 1.2650799494987814e-09, 1.777602214007199e-09, 2.3794705306483863e-09, 3.149801121595646e-09, 4.208531959563822e-09, 5.307878395583215e-09, 6.9122806344757855e-09, 8.655465405273294e-09, 1.0987165360395685e-08, 1.3820008452372534e-08, 1.7330779319237295e-08, 2.1665700738968832e-08, 2.7031770968769257e-08]
nubar_e_opacities_inv_cm = [1.3566096423813977e-09, 4.4755275717913815e-09, 2.1546754963019424e-08, 5.47531006400814e-08, 1.1123883054511091e-07, 2.042363205567264e-07, 3.5482012940187117e-07, 5.920655243201202e-07, 9.526436748501559e-07, 1.4809548374270669e-06, 2.2289900021493824e-06, 3.2538736229147686e-06, 4.6123319631062746e-06, 6.3540654191625565e-06, 8.519538465802936e-06, 1.1153146580678764e-05, 1.4349950240540414e-05, 1.83608347661424e-05]

nu_e_opacities_inv_cm = np.array(nu_e_opacities_inv_cm)
nu_x_opacities_inv_cm = np.array(nu_x_opacities_inv_cm)
nubar_e_opacities_inv_cm = np.array(nubar_e_opacities_inv_cm)

nu_e_opacities_inv_cm = nu_e_opacities_inv_cm[0:13]
nu_x_opacities_inv_cm = nu_x_opacities_inv_cm[0:13]
nubar_e_opacities_inv_cm = nubar_e_opacities_inv_cm[0:13]

### Define function to get number density as a function of time

In [ ]:
def get_number_density_per_energy_bins(i, j, k, particlefile):

    parcellname = '/cell_' + str(i) + '_' + str(j) + '_' + str(k) + '.h5'

    particles_h5py = h5py.File(direct+particlefile+parcellname, 'r')
    N00_Re    = np.array(particles_h5py['/N00_Re'   ]) # particles
    N00_Rebar = np.array(particles_h5py['/N00_Rebar']) # particles
    N11_Re    = np.array(particles_h5py['/N11_Re'   ]) # particles
    N11_Rebar = np.array(particles_h5py['/N11_Rebar']) # particles
    N22_Re    = np.array(particles_h5py['/N22_Re'   ]) # particles
    N22_Rebar = np.array(particles_h5py['/N22_Rebar']) # particles
    time_s    = np.array(particles_h5py['/time'     ])[0] # seconds
    pupt      = np.array(particles_h5py['/pupt'     ]) # ergs
    pupx     = np.array(particles_h5py['/pupx'     ]) / pupt # unitless
    pupy     = np.array(particles_h5py['/pupy'     ]) / pupt # unitless
    pupz     = np.array(particles_h5py['/pupz'     ]) / pupt # unitless
    energyMeV = pupt / ( 1e6 * pc.CGSUnitsConst.eV ) # MeV     
    particles_h5py.close()

    n00_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n00_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n11_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3 
    n11_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3

    for t in range(num_energy_bins):
        energymask = (energyMeV >= energybinsbottomMeV[t]) & (energyMeV < energybinstopMeV[t])
        if np.sum(energymask) != num_particles_per_energy_bin:
            print(f"Warning: Energy bin {t} has {np.sum(energymask)} particles, expected {num_particles_per_energy_bin}")
        n00_Re_split_energy_bins   [t] = np.sum(N00_Re[energymask])    / cellvolume # particles / cm^3
        n00_Rebar_split_energy_bins[t] = np.sum(N00_Rebar[energymask]) / cellvolume # particles / cm^3
        n11_Re_split_energy_bins   [t] = np.sum(N11_Re[energymask])    / cellvolume # particles / cm^3
        n11_Rebar_split_energy_bins[t] = np.sum(N11_Rebar[energymask]) / cellvolume # particles / cm^3
        n22_Re_split_energy_bins   [t] = np.sum(N22_Re[energymask])    / cellvolume # particles / cm^3
        n22_Rebar_split_energy_bins[t] = np.sum(N22_Rebar[energymask]) / cellvolume # particles / cm^3

    return n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, time_s

### Plot the time evolution of the neutrino energy spectrum

In [ ]:

dE  = ( energybinstopMeV    - energybinsbottomMeV    ) * ( 1e6 * pc.CGSUnitsConst.eV )    # erg
dE3 = ( energybinstopMeV**3 - energybinsbottomMeV**3 ) * ( 1e6 * pc.CGSUnitsConst.eV )**3 # erg^3
E = ( energybinsMeV ) * ( 1e6 * pc.CGSUnitsConst.eV ) # erg
dOmega = 4.0 * np.pi

phase_space_factor = dOmega * dE * E**2  / pc.PhysConst.hc**3

In [ ]:
all_f00_Re_split_energy_bins    = []
all_f00_Rebar_split_energy_bins = []
all_f11_Re_split_energy_bins    = []
all_f11_Rebar_split_energy_bins = []
all_f22_Re_split_energy_bins    = []
all_f22_Rebar_split_energy_bins = []
tiempo_s = []

for dirpar in all_par_files:
    n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, t_s = get_number_density_per_energy_bins(cell_index_i, cell_index_j, cell_index_k,'/'+dirpar)
    all_f00_Re_split_energy_bins   .append(np.array(n00_Re_split_energy_bins)   / phase_space_factor)
    all_f00_Rebar_split_energy_bins.append(np.array(n00_Rebar_split_energy_bins)/ phase_space_factor)
    all_f11_Re_split_energy_bins   .append(np.array(n11_Re_split_energy_bins)   / phase_space_factor)
    all_f11_Rebar_split_energy_bins.append(np.array(n11_Rebar_split_energy_bins)/ phase_space_factor)
    all_f22_Re_split_energy_bins   .append(np.array(n22_Re_split_energy_bins)   / phase_space_factor)
    all_f22_Rebar_split_energy_bins.append(np.array(n22_Rebar_split_energy_bins)/ phase_space_factor)
    tiempo_s.append(t_s)

print('tiempo_s[-1]=',tiempo_s[-1])
all_f00_Re_split_energy_bins    = np.array(all_f00_Re_split_energy_bins)
all_f00_Rebar_split_energy_bins = np.array(all_f00_Rebar_split_energy_bins)
all_f11_Re_split_energy_bins    = np.array(all_f11_Re_split_energy_bins)
all_f11_Rebar_split_energy_bins = np.array(all_f11_Rebar_split_energy_bins)
all_f22_Re_split_energy_bins    = np.array(all_f22_Re_split_energy_bins)
all_f22_Rebar_split_energy_bins = np.array(all_f22_Rebar_split_energy_bins)
tiempo_s = np.array(tiempo_s)

print(f'energybinsMeV = {energybinsMeV}')
print(f'energybinsMeV.shape = {energybinsMeV.shape}')
print(f'all_f00_Re_split_energy_bins.shape = {all_f00_Re_split_energy_bins.shape}')
print(f'all_f00_Re_split_energy_bins[1].shape = {all_f00_Re_split_energy_bins[1].shape}')
print(f'tiempo_s.shape = {tiempo_s.shape}')

net_absorption_emission_n00_inv_s_inv_cm3 = np.zeros_like(tiempo_s)
net_absorption_emission_n00bar_inv_s_inv_cm3 = np.zeros_like(tiempo_s)

for i in range(len(tiempo_s)):
    net_absorption_emission_n00_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Re_split_energy_bins[i] - all_f00_Re_split_energy_bins[0]) * phase_space_factor )
    net_absorption_emission_n00bar_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Rebar_split_energy_bins[i] - all_f00_Rebar_split_energy_bins[0]) * phase_space_factor )

net_absorption_emission_protons_inv_s_inv_cm3   = +1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_neutrons_inv_s_inv_cm3  = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_electrons_inv_s_inv_cm3 = -1 * (                                                net_absorption_emission_n00_inv_s_inv_cm3 ) 
net_absorption_emission_positrons_inv_s_inv_cm3 = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3                                             ) 

print(f'net_absorption_emission_protons_inv_s_inv_cm3_ = {list(net_absorption_emission_protons_inv_s_inv_cm3)}')
print(f'tiempo_s_ = {list(tiempo_s)}')
print(f'...')

delta_tiempo_s = tiempo_s[1:] - tiempo_s[:-1]
cumulative_net_absorption_emission_protons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_protons_inv_s_inv_cm3[:-1] * delta_tiempo_s)
cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_neutrons_inv_s_inv_cm3[:-1] * delta_tiempo_s)

Ye_dot_inv_s = ( pc.PhysConst.Mp / rho_g_cm3 ) * ( net_absorption_emission_n00bar_inv_s_inv_cm3 -  net_absorption_emission_n00_inv_s_inv_cm3)
cumulative_Ye_inv_s = np.cumsum(Ye_dot_inv_s[:-1] * delta_tiempo_s)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_n00_inv_s_inv_cm3, label=r'$\dot{n}_{\nu_e}$')
ax.plot(tiempo_s, net_absorption_emission_n00bar_inv_s_inv_cm3, label=r'$\dot{n}_{\bar{\nu}_e}$')
ax.plot(tiempo_s, net_absorption_emission_electrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^-}$')
ax.plot(tiempo_s, net_absorption_emission_positrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^+}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
# ax.set_xlim(0, 0.005)
ax.set_title('Emission-absorption-driven changes\nwith flavor-conversion rates excluded.\nElectrons, positrons, neutrons,\nand protons are not integrated over time.\nAntineutrinos and neutrinos are integrated over time.\n')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=3, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n - n_{t_0} \,[\mathrm{cm}^{-3}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], neutrons_number_density_cm3 + cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.plot(tiempo_s[1:], protons_number_density_cm3 + cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n\,[\mathrm{{cm}}^{{-3}}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
# ax.set_xscale('log')
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, Ye_dot_inv_s)
ax.set_xlabel(r'$t\,[\mathrm{s}]$', fontsize=28)
ax.set_ylabel(r'$\dot{Y}_e\;[\mathrm{s}^{-1}]$', fontsize=28)
leg = ax.legend(framealpha=0.0, fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_Ye_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for cumulative change in Ye
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_Ye_inv_s, color='tab:blue', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e-Y_e^{t_0}$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_delta_Ye_cumulative.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for Ye evolution
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], Ye + cumulative_Ye_inv_s, color='tab:green', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_Ye_evolution.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
net_absorption_emission_protons_inv_s_inv_cm3_1emin5 = [0.0, -2.7417988622711993e+23, -5.69593643182399e+23, -8.900431590442068e+23, -1.217842750056133e+24, -1.5478397570246527e+24, -1.8877974891587434e+24, -2.20948967907077e+24, -2.4951351888339565e+24, -2.741130928257859e+24, -2.957430684881714e+24, -3.1543191282041187e+24, -3.3275719899698517e+24, -3.510120390080477e+24, -3.69941153356465e+24, -3.9127895712067644e+24, -4.1855465937702134e+24, -4.469345942676455e+24, -4.781984687118294e+24, -5.136667878607e+24, -5.490484001908446e+24, -5.833605765686578e+24, -6.184137645909753e+24, -6.510110664775622e+24, -6.805420669461881e+24, -7.061621735023414e+24, -7.282140145737677e+24, -7.463613119988068e+24, -7.60534967369099e+24, -7.7330252300202e+24, -7.858142070590711e+24, -7.985546400783027e+24, -8.133134166221871e+24, -8.299645995029768e+24, -8.516366427635309e+24, -8.763608133814187e+24, -9.038030388064199e+24, -9.334340034459525e+24, -9.639147684790045e+24, -9.968678146671154e+24, -1.0309963309646358e+25, -1.0642533287391279e+25, -1.0973029005389435e+25, -1.1303442697021672e+25, -1.164725682482186e+25, -1.1985655669553875e+25, -1.2323669137078903e+25, -1.2682140496562403e+25, -1.305031882973849e+25, -1.3446016318595462e+25, -1.387308870052804e+25, -1.4326662808010322e+25, -1.4806512027932734e+25, -1.5314095198343713e+25, -1.5844184375465875e+25, -1.6395965915950586e+25, -1.6970512164804366e+25, -1.7554023154659644e+25, -1.8153051361088072e+25, -1.8760572266183814e+25, -1.9371843556256746e+25, -2.0000013177797434e+25, -2.0642693706177105e+25, -2.129198740775023e+25, -2.1961914899434422e+25, -2.2639396540445895e+25, -2.3339126772695984e+25, -2.406031339436279e+25, -2.4790837048443433e+25, -2.5561642618111647e+25, -2.6342262159837392e+25, -2.7157068403282758e+25, -2.7984895086798876e+25, -2.8834634974047406e+25, -2.9709341316896187e+25, -3.059900722408278e+25, -3.1508897175124325e+25, -3.24460259515844e+25, -3.338398791424104e+25, -3.4365888315008014e+25, -3.537589474454929e+25, -3.6418993407699185e+25, -3.748618410996289e+25, -3.858692878976627e+25, -3.9717478320392755e+25, -4.08915686435366e+25, -4.209678182312776e+25, -4.3342120811677995e+25, -4.46283471357546e+25, -4.5938938045823426e+25, -4.730143059439638e+25, -4.869646235640131e+25, -5.012451770593179e+25, -5.16099596967901e+25, -5.313436567206554e+25, -5.470590779623683e+25, -5.632636440661561e+25, -5.799214772842424e+25, -5.971187907406803e+25, -6.147430032095172e+25, -6.330122046631195e+25, -6.5157820775548e+25, -6.708403723644303e+25, -6.908161583828004e+25, -7.111774330023553e+25, -7.322156819508733e+25, -7.538338030009291e+25, -7.760408974630888e+25, -7.989211086113659e+25, -8.225648346845898e+25, -8.469304195752335e+25, -8.720241426336727e+25, -8.97960097377279e+25, -9.246971365443205e+25, -9.523394197598632e+25, -9.805157180627782e+25, -1.0097199699925162e+26, -1.0397680913172592e+26, -1.070603209160705e+26, -1.1023384196924668e+26, -1.1350485587161899e+26, -1.1689974916197004e+26, -1.2038442783362177e+26, -1.2397403850330517e+26, -1.2766710192824516e+26, -1.3149518146945156e+26, -1.354178118096208e+26, -1.3947566935554317e+26, -1.4365356294669027e+26, -1.4795179866610862e+26, -1.5238249354374133e+26, -1.569543321036504e+26, -1.6165759237742713e+26, -1.6652294936452452e+26, -1.715232964232185e+26, -1.7668761337055855e+26, -1.8199975580650194e+26, -1.874745531129955e+26, -1.931189887786165e+26, -1.9893002995728186e+26, -2.0492126797617397e+26, -2.1106831332960732e+26, -2.1742977448932315e+26, -2.23965481553415e+26, -2.3071508048053802e+26, -2.3768224090197578e+26, -2.44851459763936e+26, -2.522500288162297e+26, -2.598766202712528e+26, -2.6773476486348354e+26, -2.7583610787947046e+26, -2.8418137105344705e+26, -2.9278414412920037e+26, -3.0164834132339262e+26, -3.1077148341101434e+26, -3.2018222289044615e+26, -3.2988590438010994e+26, -3.3988834985853096e+26, -3.501910151324867e+26, -3.608197422879293e+26, -3.7176250886522504e+26, -3.830489955029137e+26, -3.946787983087623e+26, -4.066699362430401e+26, -4.190349309832334e+26, -4.3178618004878646e+26, -4.449210695455332e+26, -4.5847943400796645e+26, -4.724357015419728e+26, -4.868172924855458e+26, -5.016461288665145e+26, -5.169271566016733e+26, -5.326990471258383e+26, -5.4894349942274864e+26, -5.656936632797093e+26, -5.8295545137515405e+26, -6.007547925542544e+26, -6.190996701598085e+26, -6.380235607071035e+26, -6.575239135101173e+26, -6.776144233110905e+26, -6.983330093960569e+26, -7.197009809487311e+26, -7.417355643153923e+26, -7.644470229574755e+26, -7.878770959240496e+26, -8.12012010774699e+26, -8.369132367875319e+26, -8.625595482177053e+26, -8.890180415384285e+26, -9.16302701740621e+26, -9.444300764928644e+26, -9.73424167089076e+26, -1.0033273226966712e+27, -1.0341400833663167e+27, -1.0659189669708539e+27, -1.0986895893618905e+27, -1.1324912015843534e+27, -1.1673371329207389e+27, -1.2032535451041535e+27, -1.2402947076638478e+27, -1.2784904522369107e+27, -1.3178649163234432e+27, -1.3584791682674456e+27, -1.4003558920749764e+27, -1.4435386759609867e+27, -1.4880635609654081e+27, -1.5339965850956368e+27, -1.5813491094839536e+27, -1.630173511553778e+27, -1.680513101931408e+27, -1.732428982926517e+27, -1.7859812477938862e+27, -1.8411890544189341e+27, -1.898136640694473e+27, -1.9568483189869744e+27, -2.017399022659344e+27, -2.0798300570952242e+27, -2.1442456973516786e+27, -2.2106544691116282e+27, -2.279140662949147e+27, -2.3497869730633266e+27, -2.422631984273123e+27, -2.49774800034579e+27, -2.575236371877184e+27, -2.65513850541535e+27, -2.7375554970120767e+27, -2.822549894506502e+27, -2.910214745652925e+27, -3.000619778420852e+27, -3.0938542250601823e+27, -3.1900392996605964e+27, -3.28922666635494e+27, -3.3915304848277597e+27, -3.4970423544206655e+27, -3.6058701175413095e+27, -3.71812115063792e+27, -3.8338918803898646e+27, -3.953308460552997e+27, -4.076474623380203e+27, -4.203513049647897e+27, -4.3345452851055507e+27, -4.469704341052999e+27, -4.6091198763222713e+27, -4.7529115289954597e+27, -4.90123942322422e+27, -5.054237975197109e+27, -5.212052004591579e+27, -5.374835122115497e+27, -5.542746443277271e+27, -5.715947810881773e+27, -5.894618992996953e+27, -6.07891658468772e+27, -6.269033548911941e+27, -6.465131945536551e+27, -6.667421383140868e+27, -6.876107347556815e+27, -7.091367806132425e+27, -7.313418957784764e+27, -7.54248640070583e+27, -7.778793897417544e+27, -8.02255436540969e+27, -8.274024765748904e+27, -8.533437441356206e+27, -8.801055319176172e+27, -9.07713628525432e+27, -9.361940640363649e+27, -9.655752591092962e+27, -9.958858819368144e+27, -1.0271554270486849e+28, -1.0594144178648923e+28, -1.0926944789877864e+28, -1.1270278836679132e+28, -1.1624493766933013e+28, -1.1989920044096204e+28, -1.2366924125557408e+28, -1.2755867248288267e+28, -1.3157146395909889e+28, -1.3571131101490224e+28, -1.3998246967298168e+28, -1.4438901515976385e+28, -1.4893526539803685e+28, -1.536257422357296e+28, -1.58465020386531e+28, -1.6345794064482324e+28, -1.686091538024945e+28, -1.7392390665571245e+28, -1.7940736541807135e+28, -1.8506499161998667e+28, -1.909023192093828e+28, -1.9692498366594573e+28, -2.0313895533802558e+28, -2.0955042047141423e+28, -2.1616559953432963e+28, -2.229910537833557e+28, -2.300335189436469e+28, -2.372998957675147e+28, -2.447973441744474e+28, -2.52533203962801e+28, -2.605151925275559e+28, -2.6875110136918157e+28, -2.772491859547217e+28, -2.8601771610791143e+28, -2.9506531685506373e+28, -3.044010289144264e+28, -3.140339397402758e+28, -3.2397378829411747e+28, -3.3423006783884348e+28, -3.4481326256295003e+28, -3.557335730739993e+28, -3.670019065559628e+28, -3.7862944640465125e+28, -3.906275845298482e+28, -4.030083331357359e+28, -4.157838518164318e+28, -4.2896681940693035e+28, -4.425703472595158e+28, -4.566077937214026e+28, -4.710931588840994e+28, -4.860407810844222e+28, -5.014654678605111e+28, -5.173826247731616e+28, -5.3380782863817545e+28, -5.507576126161478e+28, -5.682486216405983e+28, -5.862983583267907e+28, -6.049247337256537e+28, -6.241461173156097e+28, -6.43981764097098e+28, -6.644513154795335e+28, -6.8557508644907385e+28, -7.0737426491008235e+28, -7.298703664991465e+28, -7.530858155538816e+28, -7.770437402655871e+28, -8.017678923209815e+28, -8.272830776902254e+28, -8.536144803362232e+28, -8.807885398567851e+28, -9.08832169375623e+28, -9.377735041637447e+28, -9.676412694542823e+28, -9.984652299513173e+28, -1.0302762221615312e+29, -1.0631058358091342e+29, -1.0969869724760334e+29, -1.1319533953816395e+29, -1.168040005554279e+29, -1.205282884723991e+29, -1.2437192000807208e+29, -1.283387377555229e+29, -1.3243269056759357e+29, -1.3665789149320658e+29, -1.4101855651055414e+29, -1.4551903840340634e+29, -1.501638502767562e+29, -1.5495761738027544e+29, -1.5990517434873924e+29, -1.650114471806366e+29, -1.7028154128937638e+29, -1.7572074534195256e+29, -1.813345078385742e+29, -1.8712843295781607e+29, -1.9310835499056542e+29, -1.9928024686292355e+29, -2.056503009505641e+29, -2.122249061051003e+29, -2.1901064176089545e+29, -2.260143203607283e+29, -2.332429741179919e+29, -2.4070384912529013e+29, -2.484044350300195e+29, -2.563524554881022e+29, -2.6455590027517425e+29, -2.7302299165287275e+29, -2.817622536155967e+29, -2.907824466558205e+29, -3.0009263959662958e+29, -3.097021941619732e+29, -3.1962075884359496e+29, -3.2985830871695295e+29, -3.4042514849096146e+29, -3.5133189293647245e+29, -3.625895062904111e+29, -3.742093278834625e+29, -3.862030555242779e+29, -3.9858274422278075e+29, -4.1136085871397156e+29, -4.24550259845377e+29, -4.381642540585576e+29, -4.522165325727395e+29, -4.667212487376941e+29, -4.8169302804892635e+29, -4.971469413135461e+29, -5.130985850574897e+29, -5.295640279979679e+29, -5.465598738520235e+29, -5.641032540979197e+29, -5.82211876080302e+29, -6.009039995455579e+29, -6.20198489866092e+29, -6.401148075735104e+29, -6.606730709336791e+29, -6.818940044754808e+29, -7.037990685385973e+29, -7.26410348028971e+29, -7.497507113695873e+29, -7.738437212751035e+29, -7.98713718581845e+29, -8.243858226972724e+29, -8.508859935357013e+29, -8.7824101195011e+29, -9.064785417724811e+29, -9.356271191503695e+29, -9.657162304884773e+29, -9.967762993591961e+29, -1.0288387610928372e+30, -1.0619360428194379e+30, -1.096101641115416e+30, -1.1313701373654306e+30, -1.1677772171842842e+30, -1.2053597452049049e+30, -1.2441557821289144e+30, -1.2842045988907787e+30, -1.3255467827625977e+30, -1.3682241918067857e+30, -1.4122800627608222e+30, -1.4577590455357113e+30, -1.5047072268937048e+30, -1.5531721874714274e+30, -1.6032030664366363e+30, -1.6548505840747574e+30, -1.7081670957586627e+30, -1.7632066962237404e+30, -1.820025186508761e+30, -1.8786802414376436e+30, -1.9392313335788794e+30, -2.0017399229377812e+30, -2.066269444324248e+30, -2.132885385705446e+30, -2.2016553733441116e+30, -2.2726492413915085e+30, -2.3459390625045018e+30, -2.4215992715470584e+30, -2.4997067100575346e+30, -2.580340717141287e+30, -2.66358322331015e+30, -2.749518805927959e+30, -2.838234776745133e+30, -2.929821322036298e+30, -3.0243715248690705e+30, -3.1219815089236783e+30, -3.2227505085839004e+30, -3.326780991675576e+30, -3.4341787644348143e+30, -3.545053049109117e+30, -3.659516632056456e+30, -3.7776859853048667e+30, -3.899681305462766e+30, -4.025626763572836e+30, -4.155650531866642e+30, -4.2898849469716076e+30, -4.4284666617985696e+30, -4.571536772908801e+30, -4.7192409228347985e+30, -4.8717295468069527e+30, -5.0291579542674597e+30, -5.191686475033945e+30, -5.359480695698743e+30, -5.532711539621726e+30, -5.711555537808811e+30, -5.896194906023594e+30, -6.086817846962171e+30, -6.283618646891335e+30, -6.486797914496943e+30, -6.696562787458441e+30, -6.913127149626e+30, -7.13671181004168e+30, -7.367544806098222e+30, -7.605861557944936e+30, -7.85190516424343e+30, -8.105926609710999e+30, -8.368185087578652e+30, -8.63894816127248e+30, -8.918492151261449e+30, -9.207102359755926e+30, -9.505073349582966e+30, -9.812709295321252e+30, -1.013032423181453e+31, -1.0458242472479817e+31, -1.079679883635469e+31, -1.1146339049185183e+31, -1.150722010568325e+31, -1.1879810592101361e+31, -1.226449112793851e+31, -1.2661654685021652e+31, -1.3071707048219765e+31, -1.3495067198578117e+31, -1.3932167767943193e+31, -1.4383455479781913e+31, -1.484939157084058e+31, -1.5330452316349068e+31, -1.5827129494294798e+31, -1.6339930899665562e+31, -1.6869380832746855e+31, -1.7416020719037176e+31, -1.7980409546547374e+31, -1.856312456892651e+31, -1.9164761802084735e+31, -1.9785936661954303e+31, -2.0427284645311755e+31, -2.108946191128459e+31, -2.177314598980503e+31, -2.2479036507959254e+31, -2.320785584809162e+31, -2.396034992969353e+31, -2.4737288981187317e+31, -2.5539468289583836e+31, -2.6367709096027448e+31, -2.722285934873906e+31, -2.8105794635816775e+31, -2.901741908111024e+31, -2.9958666236618755e+31, -3.0930500091697743e+31, -3.193391603310286e+31, -3.2969941875762626e+31, -3.4039638935522325e+31, -3.51441030873529e+31, -3.6284465938784353e+31, -3.746189595938889e+31, -3.867759968681534e+31, -3.9932822979236273e+31, -4.122885229526561e+31, -4.256701602247004e+31, -4.394868583768644e+31, -4.537527811564137e+31, -4.68482554071932e+31, -4.836912788721829e+31, -4.99394550184055e+31, -5.156084704264964e+31, -5.323496668957264e+31, -5.496353090033772e+31, -5.67483125866211e+31, -5.859114244448781e+31, -6.049391082843452e+31, -6.245856974017766e+31, -6.448713481632955e+31, -6.658168737260128e+31, -6.874437661628367e+31, -7.097742179830634e+31, -7.328311453796231e+31, -7.566382114963232e+31, -7.812198511325333e+31, -8.066012957703962e+31, -8.328085995077199e+31, -8.598686661162202e+31, -8.878092763539941e+31, -9.166591171020294e+31, -9.464478105619609e+31, -9.77205944879301e+31, -1.0089651057939819e+32, -1.0417579088849096e+32, -1.0756180334289727e+32, -1.1105802571404261e+32, -1.1466804918394198e+32, -1.1839558205774881e+32, -1.2224445358689344e+32, -1.2621861789708901e+32, -1.3032215809474736e+32, -1.345592904430569e+32, -1.3893436875103688e+32, -1.4345188880199547e+32, -1.4811649305820367e+32, -1.5293297537498939e+32, -1.579062859964599e+32, -1.63041536643877e+32, -1.6834400578155407e+32, -1.738191440365785e+32, -1.7947257987707286e+32, -1.853101253661033e+32, -1.9133778214893628e+32, -1.9756174770005276e+32, -2.039884216323728e+32, -2.1062441236372735e+32, -2.174765439228851e+32, -2.245518629169995e+32, -2.3185764589545723e+32, -2.394014068055303e+32, -2.4719090471149136e+32, -2.552341518793709e+32, -2.635394219375653e+32, -2.7211525850781755e+32, -2.8097048392505813e+32, -2.9011420842701915e+32, -2.9955583948843467e+32, -3.093050915136523e+32, -3.1937199591463456e+32, -3.2976691141293296e+32, -3.4050053472844565e+32, -3.515839116277801e+32, -3.630284482922594e+32, -3.74845923122057e+32, -3.870484988342305e+32, -3.996487350542092e+32, -4.126596012276698e+32, -4.260944900383638e+32, -4.3996723116824896e+32, -4.5429210565061904e+32, -4.690838604952206e+32, -4.8435772395384526e+32, -5.001294212428563e+32, -5.1641519070615656e+32, -5.33231800641976e+32, -5.50596566506757e+32, -5.6852736883654544e+32, -5.8704267165447496e+32, -6.0616154155736434e+32, -6.259036673194132e+32, -6.462893802493252e+32, -6.673396751193654e+32, -6.890762318521861e+32, -7.11521437819815e+32, -7.346984110147481e+32, -7.58631023812497e+32, -7.833439276109967e+32, -8.088625782788045e+32, -8.352132623569114e+32, -8.624231241939145e+32, -8.905201938799212e+32, -9.195334162564149e+32, -9.494926806005282e+32, -9.804288515839386e+32, -1.0123738009817387e+33, -1.045360440572647e+33, -1.0794227560740784e+33, -1.1145958421291851e+33, -1.1509159385021815e+33, -1.1884204674077267e+33, -1.2271480721042445e+33, -1.2671386566503093e+33, -1.3084334271088307e+33, -1.3510749338811731e+33, -1.3951071156710913e+33, -1.4405753446352185e+33, -1.487526473169738e+33, -1.5360088821608554e+33, -1.5860725307938896e+33, -1.6377690079932017e+33, -1.691151585542324e+33, -1.7462752729430833e+33, -1.8031968740027603e+33, -1.8619750453481743e+33, -1.92267035672918e+33, -1.9853453534052184e+33, -2.0500646204578763e+33, -2.116894849241385e+33, -2.1859049059707488e+33, -2.2571659025632741e+33, -2.3307512697589935e+33, -2.4067368325931655e+33, -2.485200888373274e+33, -2.5662242871052748e+33, -2.649890514624189e+33, -2.7362857783296336e+33, -2.825499095769547e+33, -2.91762238602166e+33, -3.012750564115825e+33, -3.1109816384562773e+33, -3.2124168113907097e+33, -3.317160583102475e+33, -3.4253208587747274e+33, -3.5370090592810583e+33, -3.6523402354077725e+33, -3.771433185846128e+33, -3.894410578874142e+33, -4.0213990780820285e+33, -4.1525294720820486e+33, -4.287936808465479e+33, -4.427760531939061e+33, -4.572144627165013e+33, -4.721237765894672e+33, -4.8751934591280356e+33, -5.034170213926139e+33, -5.198331695474725e+33, -5.367846894172089e+33, -5.542890298241175e+33, -5.723642071771766e+33, -5.910288238545141e+33, -6.103020871717347e+33, -6.302038289586431e+33, -6.507545257642864e+33, -6.719753197088203e+33, -6.93888040000025e+33, -7.165152251364206e+33, -7.398801458268556e+33, -7.640068286342248e+33, -7.889200803731989e+33, -8.146455132882331e+33, -8.412095710331762e+33, -8.686395554727976e+33, -8.969636543410646e+33, -9.262109697764267e+33, -9.564115477516607e+33, -9.875964084554936e+33, -1.0197975776176722e+34, -1.053048118833294e+34, -1.0873821669020781e+34, -1.1228349622251445e+34, -1.1594428862821973e+34, -1.19724349822154e+34, -1.236275572599478e+34, -1.2765791383142102e+34, -1.318195518741312e+34, -1.3611673731431041e+34, -1.4055387393526903e+34, -1.4513550778076664e+34, -1.4986633169393298e+34, -1.547511899978526e+34, -1.5979508332155562e+34, -1.6500317357524287e+34, -1.7038078907884105e+34, -1.7593342984961534e+34, -1.8166677305195241e+34, -1.8758667861445855e+34, -1.9369919501992626e+34, -2.0001056527134972e+34, -2.065272330411709e+34, -2.132558490065838e+34, -2.2020327737764127e+34, -2.273766026233859e+34, -2.347831364003183e+34, -2.424304246905207e+34, -2.503262551525358e+34, -2.5847866469399396e+34, -2.6689594726946464e+34, -2.75586661909599e+34, -2.845596409890479e+34, -2.938239987384773e+34, -3.033891400064667e+34, -3.1326476927896577e+34, -3.2346089996144796e+34, -3.3398786393099636e+34, -3.4485632136515935e+34, -3.560772708536116e+34, -3.676620598002354e+34, -3.79622395121371e+34, -3.9197035424864857e+34, -4.0471839644182245e+34, -4.178793744192832e+34, -4.314665463138374e+34, -4.454935879598709e+34, -4.5997460551909565e+34, -4.749241484525023e+34, -4.9035722284478175e+34, -5.062893050889952e+34, -5.227363559373089e+34, -5.39714834925602e+34, -5.572417151783949e+34, -5.753344986008395e+34, -5.940112314631704e+34, -6.132905203856453e+34, -6.331915487287462e+34, -6.537340933955432e+34, -6.749385420496633e+34, -6.9682591075820565e+34, -7.19417862059243e+34, -7.427367234638541e+34, -7.668055063920675e+34, -7.916479255508412e+34, -8.172884187529665e+34, -8.43752167183279e+34, -8.710651161108316e+34, -8.992539960516783e+34, -9.283463443797849e+34, -9.583705273871241e+34, -9.893557627925045e+34, -1.0213321426952142e+35, -1.0543306569722137e+35, -1.0883832171121734e+35, -1.1235226804837295e+35, -1.1597828750282406e+35, -1.1971986243706324e+35, -1.2358057733373133e+35, -1.2756412138703446e+35, -1.3167429113243664e+35, -1.359149931130471e+35, -1.402902465810772e+35, -1.4480418623237193e+35, -1.494610649717818e+35, -1.5426525670710999e+35, -1.5922125916878393e+35, -1.643336967523168e+35, -1.6960732338045286e+35, -1.7504702538112098e+35, -1.806578243775796e+35, -1.8644488018625296e+35, -1.9241349371768593e+35, -1.985691098756052e+35, -2.0491732044856473e+35, -2.11463866988261e+35, -2.1821464366812187e+35, -2.251757001153821e+35, -2.323532442091032e+35, -2.3975364483636023e+35, -2.473834345979997e+35, -2.5524931245490736e+35, -2.6335814630504184e+35, -2.717169754809825e+35, -2.8033301315674254e+35, -2.892136486523476e+35, -2.9836644962350236e+35, -3.0779916412334546e+35, -3.1751972252201366e+35, -3.2753623926946048e+35, -3.378570144857e+35, -3.484905353620475e+35, -3.5944547735598525e+35, -3.707307051613047e+35, -3.823552734344145e+35, -3.943284272565422e+35, -4.066596023108205e+35, -4.193584247522691e+35, -4.3243471074752105e+35, -4.458984656602947e+35, -4.597598828578037e+35, -4.74029342112022e+35, -4.887174075689944e+35, -5.038348252582975e+35, -5.193925201140521e+35, -5.354015924776738e+35, -5.518733140521436e+35, -5.688191232763527e+35, -5.862506200876799e+35, -6.0417956004000514e+35, -6.226178477441684e+35, -6.415775295968819e+35, -6.6107078576419845e+35, -6.811099213849621e+35, -7.017073569599673e+35, -7.228756178919548e+35, -7.446273231424872e+35, -7.669751729712914e+35, -7.89931935724959e+35, -8.135104336420659e+35, -8.377235276430063e+35, -8.625841010741021e+35, -8.881050423770025e+35, -9.142992266559838e+35, -9.41179496118292e+35, -9.68758639364548e+35, -9.970493695093657e+35, -1.0260643011153756e+36, -1.0558159259271736e+36, -1.0863165873956689e+36, -1.1175784539876808e+36, -1.1496134912801924e+36, -1.182433432843699e+36, -1.2160497499248416e+36, -1.2504736199443502e+36, -1.2857158938326123e+36, -1.3217870622322603e+36, -1.3586972206042377e+36, -1.3964560332814002e+36, -1.4350726965220504e+36, -1.4745559006238844e+36, -1.5149137911677734e+36, -1.5561539294699838e+36, -1.59828325233036e+36, -1.6413080311742107e+36, -1.685233830694663e+36, -1.7300654671126934e+36, -1.775806966181489e+36, -1.8224615210718556e+36, -1.8700314502851763e+36, -1.9185181557497083e+36, -1.967922081265446e+36, -2.0182426714716425e+36, -2.0694783315191392e+36, -2.1216263876377972e+36, -2.1746830487960035e+36, -2.2286433696555472e+36, -2.2835012150304608e+36, -2.3392492260624614e+36, -2.395878788328778e+36, -2.4533800020997544e+36, -2.5117416549642056e+36, -2.5709511970390158e+36, -2.630994718977362e+36, -2.691856932984803e+36, -2.753521157047586e+36, -2.815969302568529e+36, -2.8791818655975103e+36, -2.94313792183119e+36, -3.007815125544367e+36, -3.07318971260001e+36, -3.139236507668392e+36, -3.205928935767911e+36, -3.2732390382199533e+36, -3.3411374930892075e+36, -3.409593640157997e+36, -3.4785755104599285e+36, -3.548049860372934e+36, -3.6179822102466646e+36, -3.6883368875129944e+36, -3.7590770742019194e+36, -3.830164858758705e+36, -3.901561292031679e+36, -3.9732264472737294e+36, -4.0451194839757275e+36, -4.117198715324813e+36, -4.1894216790576774e+36, -4.261745211456266e+36, -4.3341255242133957e+36, -4.406518283876426e+36, -4.478878693560921e+36, -4.5511615766112095e+36, -4.623321461872383e+36, -4.695312670228776e+36, -4.76708940205592e+36, -4.838605825228451e+36, -4.909816163324132e+36, -4.9806747836642637e+36, -5.051136284833667e+36, -5.121155583329075e+36, -5.1906879989920366e+36, -5.2596893388931567e+36, -5.328115979346324e+36, -5.395924945746323e+36, -5.463073989939206e+36, -5.529521664852811e+36, -5.595227396134299e+36, -5.660151550562032e+36, -5.7242555010211e+36, -5.787501687854246e+36, -5.849853676423086e+36, -5.911276210738153e+36, -5.971735263040172e+36, -6.031198079238587e+36, -6.089633220136856e+36, -6.147010598397437e+36, -6.203301511221653e+36, -6.258478668741791e+36, -6.31251621814335e+36, -6.36538976355581e+36, -6.417076381768894e+36, -6.467554633849025e+36, -6.516804572747261e+36, -6.56480774700491e+36, -6.611547200676936e+36, -6.65700746960553e+36, -6.701174574187277e+36, -6.744036008786722e+36, -6.785580727957544e+36, -6.825799129639246e+36, -6.864683035502699e+36, -6.902225668622138e+36, -6.938421628654324e+36, -6.973266864707202e+36, -7.00675864608116e+36, -7.038895531065729e+36, -7.069677333973343e+36, -7.099105090589383e+36, -7.12718102221498e+36, -7.153908498475272e+36, -7.179291999061387e+36, -7.203337074569521e+36, -7.226050306595043e+36, -7.247439267233596e+36, -7.267512478134791e+36, -7.286279369247882e+36, -7.303750237391496e+36, -7.319936204772771e+36, -7.334849177574116e+36, -7.34850180471842e+36, -7.360907436916527e+36, -7.372080086093272e+36, -7.382034385281436e+36, -7.390785549065763e+36, -7.398349334652245e+36, -7.404742003630987e+36, -7.409980284494484e+36, -7.41408133596651e+36, -7.417062711190742e+36, -7.418942322822104e+36, -7.419738409058257e+36, -7.41946950064294e+36, -7.418154388867932e+36, -7.415812094595124e+36, -7.412461838315776e+36, -7.408123011259615e+36, -7.402815147561996e+36, -7.39655789749407e+36, -7.389371001756697e+36, -7.381274266836041e+36, -7.372287541415341e+36, -7.362430693834956e+36, -7.35172359058984e+36, -7.340186075851465e+36, -7.327837951999376e+36, -7.314698961144944e+36, -7.300788767629365e+36, -7.28612694147529e+36, -7.270732942771137e+36, -7.25462610696543e+36, -7.237825631048024e+36, -7.220350560593917e+36, -7.202219777645011e+36, -7.183451989404565e+36, -7.164065717718718e+36, -7.144079289319171e+36, -7.12351082680118e+36, -7.102378240310611e+36, -7.080699219914153e+36, -7.058491228626706e+36, -7.035771496070406e+36, -7.012557012739526e+36, -6.988864524846427e+36, -6.964710529723523e+36, -6.940111271757023e+36, -6.915082738828466e+36, -6.889640659240626e+36, -6.863800499104735e+36, -6.837577460166848e+36, -6.810986478051258e+36, -6.784042220899777e+36, -6.756759088386283e+36, -6.729151211086247e+36, -6.701232450181934e+36, -6.673016397484235e+36, -6.644516375753025e+36, -6.615745439298204e+36, -6.586716374844519e+36, -6.557441702643685e+36, -6.527933677817854e+36, -6.498204291919448e+36, -6.46826527469235e+36, -6.438128096020767e+36, -6.407803968051884e+36, -6.377303847479734e+36, -6.346638437977585e+36, -6.315818192767234e+36, -6.284853317313722e+36, -6.253753772134813e+36, -6.222529275714759e+36, -6.191189307512759e+36, -6.159743111056522e+36, -6.128199697112309e+36, -6.096567846922864e+36, -6.064856115505201e+36, -6.033072835000893e+36, -6.001226118071427e+36, -5.969323861332037e+36, -5.937373748817447e+36, -5.905383255473693e+36, -5.873359650670083e+36, -5.841310001726149e+36, -5.809241177448421e+36, -5.777159851672325e+36, -5.745072506804834e+36, -5.71298543736359e+36, -5.680904753508791e+36, -5.648836384564015e+36, -5.616786082522862e+36, -5.584759425537946e+36, -5.552761821389629e+36, -5.520798510931617e+36, -5.48887457151099e+36, -5.456994920360284e+36, -5.425164317959586e+36, -5.393387371366606e+36, -5.361668537513119e+36, -5.330012126465814e+36, -5.2984223046504623e+36, -5.266903098037739e+36, -5.2354583952897675e+36, -5.204091950866023e+36, -5.172807388087908e+36, -5.14160820216094e+36, -5.1104977631539534e+36, -5.079479318934555e+36, -5.0485559980604137e+36, -5.0177308126258114e+36, -4.9870066610630885e+36, -4.956386330898824e+36, -4.925872501464343e+36, -4.895467746560489e+36, -4.8651745370765784e+36, -4.834995243563534e+36, -4.804932138761089e+36, -4.774987400079305e+36, -4.745163112034466e+36, -4.7154612686395085e+36, -4.685883775749179e+36, -4.656432453360353e+36, -4.627109037867519e+36, -4.597915184274035e+36, -4.5688524683593344e+36, -4.5399223888025334e+36, -4.5111263692629167e+36, -4.4824657604176245e+36, -4.453941841956982e+36, -4.4255558245382097e+36, -4.3973088516975986e+36, -4.369202001722077e+36, -4.3412362894803744e+36, -4.313412668214497e+36, -4.2857320312920436e+36, -4.2581952139198465e+36, -4.230802994819575e+36, -4.2035560978659024e+36, -4.176455193687652e+36, -4.1495009012327563e+36, -4.1226937892974116e+36, -4.096034378020077e+36, -4.069523140341034e+36, -4.043160503427791e+36, -4.0169468500673635e+36, -3.990882520025599e+36, -3.964967811374433e+36, -3.939202981787472e+36, -3.913588249804637e+36, -3.888123796066416e+36, -3.8628097645181176e+36, -3.8376462635851286e+36, -3.8126333673191417e+36, -3.7877711165165775e+36, -3.7630595198092657e+36, -3.738498554728138e+36, -3.7140881687404765e+36, -3.689828280261293e+36, -3.6657187796392594e+36, -3.6417595301178566e+36, -3.6179503687721323e+36, -3.59429110742175e+36, -3.570781533520688e+36, -3.547421411024145e+36, -3.524210481233198e+36, -3.5011484636176576e+36, -3.4782350566175664e+36, -3.4554699384238956e+36, -3.4328527677388715e+36, -3.410383184516325e+36, -3.388060810682689e+36, -3.365885250838887e+36, -3.343856092943685e+36, -3.321972908978885e+36, -3.3002352555967874e+36, -3.2786426747503684e+36, -3.257194694306512e+36, -3.235890828642788e+36, -3.2147305792280397e+36, -3.193713435187369e+36, -3.1728388738516006e+36, -3.1521063612920003e+36, -3.131515352840099e+36, -3.111065293593505e+36, -3.0907556189076826e+36, -3.070585754874145e+36, -3.050555118785507e+36, -3.030663119587583e+36, -3.010909158318851e+36, -2.9912926285377525e+36, -2.971812916737987e+36, -2.952469402752107e+36, -2.9332614601438297e+36, -2.914188456589243e+36, -2.895249754247244e+36, -2.8764447101193794e+36, -2.857772676399609e+36, -2.839233000813956e+36, -2.820825026950525e+36, -2.802548094580012e+36, -2.78440153996706e+36, -2.7663846961725804e+36, -2.7484968933474194e+36, -2.730737459017473e+36, -2.713105718360572e+36, -2.69560099447528e+36, -2.6782226086419447e+36, -2.660969880575998e+36, -2.6438421286740007e+36, -2.6268386702523675e+36, -2.609958821779173e+36, -2.5932018990990278e+36, -2.5765672176515438e+36, -2.560054092683054e+36, -2.54366183945241e+36, -2.5273897734304424e+36, -2.5112372104936758e+36, -2.4952034671122187e+36, -2.4792878605320916e+36, -2.4634897089522428e+36, -2.4478083316961678e+36, -2.4322430493785167e+36, -2.416793184066768e+36, -2.401458059438051e+36, -2.3862370009313364e+36, -2.3711293358951095e+36, -2.356134393730687e+36, -2.341251506031265e+36, -2.3264800067168448e+36, -2.3118192321651918e+36, -2.297268521338955e+36, -2.2828272159090126e+36, -2.2684946603742242e+36, -2.2542702021776501e+36, -2.2401531918194385e+36, -2.22614298296641e+36, -2.2122389325584305e+36, -2.198440400911826e+36, -2.1847467518197322e+36, -2.1711573526497045e+36, -2.1576715744384325e+36, -2.144288791983968e+36, -2.1310083839352424e+36, -2.1178297328791984e+36, -2.104752225425548e+36, -2.091775252289176e+36, -2.0788982083704357e+36, -2.0661204928332367e+36, -2.0534415091811725e+36, -2.0408606653316703e+36, -2.0283773736882692e+36, -2.0159910512111066e+36, -2.0037011194857053e+36, -1.9915070047900573e+36, -1.979408138160285e+36, -1.9674039554546017e+36, -1.955493897416052e+36, -1.9436774097337405e+36, -1.9319539431028634e+36, -1.9203229532834606e+36, -1.908783901158084e+36, -1.897336252788313e+36, -1.8859794794702053e+36, -1.87471305778886e+36, -1.8635364696720765e+36, -1.8524492024430657e+36, -1.8414507488724234e+36, -1.8305406072294156e+36, -1.8197182813324834e+36, -1.8089832805992418e+36, -1.798335120095729e+36, -1.7877733205853994e+36, -1.7772974085773406e+36, -1.7669069163743476e+36, -1.7566013821204506e+36, -1.7463803498482263e+36, -1.7362433695257473e+36, -1.726189997103454e+36, -1.71621979456068e+36, -1.7063323299521974e+36, -1.6965271774545838e+36, -1.6868039174126393e+36, -1.677162136385685e+36, -1.6676014271940524e+36, -1.6581213889655994e+36, -1.648721627182326e+36, -1.639401753727266e+36, -1.6301613869315037e+36, -1.6210001516215247e+36, -1.611917679166801e+36, -1.6029136075277722e+36, -1.5939875813042003e+36, -1.585139251783896e+36, -1.5763682769919566e+36, -1.5676743217405053e+36, -1.5590570576788547e+36, -1.5505161633444147e+36, -1.5420513242140516e+36, -1.5336622327562046e+36, -1.5253485884835727e+36, -1.5171100980066158e+36, -1.5089464750877065e+36, -1.500857440696186e+36, -1.4928427230641423e+36, -1.4849020577430693e+36, -1.4770351876613755e+36, -1.4692418631829327e+36, -1.461521842166383e+36, -1.4538748900256723e+36, -1.4463007797913373e+36, -1.4387992921730825e+36, -1.4313702156232145e+36, -1.4240133464014372e+36, -1.4167284886405378e+36, -1.4095154544134385e+36, -1.4023740638013483e+36, -1.3953041449632008e+36, -1.3883055342062803e+36, -1.3813780760582727e+36, -1.3745216233405053e+36, -1.3677360372426396e+36, -1.36102118739862e+36, -1.3543769519642304e+36, -1.3478032176958536e+36, -1.3412998800308242e+36, -1.3348668431692736e+36, -1.3285040201574527e+36, -1.3222113329725878e+36, -1.3159887126093352e+36, -1.309836099167755e+36, -1.3037534419430509e+36, -1.2977406995167063e+36, -1.291797839849573e+36, -1.285924840376322e+36, -1.2801216881019169e+36, -1.2743883796995595e+36, -1.268724921610521e+36, -1.263131330145697e+36, -1.2576076315889098e+36, -1.2521538623021084e+36, -1.2467700688322801e+36, -1.2414563080202617e+36, -1.2362126471113726e+36, -1.231039163867956e+36, -1.2259359466837163e+36, -1.2209030947000153e+36, -1.2159407179240821e+36, -1.211048937349023e+36, -1.2062278850759218e+36, -1.2014777044376688e+36, -1.1967985501248951e+36, -1.1921905883137662e+36, -1.1876539967956731e+36, -1.1831889651089524e+36, -1.1787956946724834e+36, -1.1744743989212152e+36, -1.1702253034437146e+36, -1.1660486461214868e+36, -1.1619446772703705e+36, -1.1579136597837552e+36, -1.153955869277656e+36, -1.1500715942377976e+36, -1.1462611361684065e+36, -1.1425248097429645e+36, -1.1388629429567027e+36, -1.1352758772809183e+36, -1.1317639678190846e+36, -1.1283275834646026e+36, -1.1249671070603485e+36, -1.1216829355598144e+36, -1.1184754801898908e+36, -1.1153451666152186e+36, -1.1122924351040262e+36, -1.1093177406955026e+36, -1.1064215533685012e+36, -1.1036043582116466e+36, -1.1008666555946976e+36, -1.0982089613411299e+36, -1.0956318069018335e+36, -1.0931357395298504e+36, -1.0907213224561759e+36, -1.0883891350662671e+36, -1.0861397730774641e+36, -1.0839738487170256e+36, -1.0818919909007322e+36, -1.0798948454119518e+36, -1.0779830750809616e+36, -1.0761573599645985e+36, -1.0744183975257934e+36, -1.0727669028131807e+36, -1.071203608640314e+36, -1.0697292657646099e+36, -1.0683446430656948e+36, -1.0670505277228984e+36, -1.0658477253919753e+36, -1.0647370603805856e+36, -1.063719375822528e+36, -1.0627955338503667e+36, -1.0619664157663144e+36, -1.0612329222111224e+36, -1.0605959733307338e+36, -1.0600565089404265e+36, -1.0596154886861876e+36, -1.0592738922030525e+36, -1.059032719270075e+36, -1.0588929899617631e+36, -1.0588557447953853e+36, -1.058922044874205e+36, -1.0590929720259197e+36, -1.0593696289362224e+36, -1.0597531392769531e+36, -1.060244647828637e+36, -1.060845320596673e+36, -1.0615563449212346e+36, -1.0623789295800467e+36, -1.0633143048837223e+36, -1.0643637227633655e+36, -1.0655284568496363e+36, -1.0668098025431551e+36, -1.0682090770753771e+36, -1.0697276195596771e+36, -1.0713667910319424e+36, -1.0731279744802255e+36, -1.0750125748627445e+36, -1.0770220191137158e+36, -1.0791577561364243e+36, -1.0814212567827689e+36, -1.0838140138187353e+36, -1.0863375418751424e+36, -1.0889933773827606e+36, -1.0917830784914225e+36, -1.0947082249719936e+36, -1.0977704181007895e+36, -1.1009712805254608e+36, -1.104312456111629e+36, -1.1077956097693921e+36, -1.1114224272589628e+36, -1.1151946149744714e+36, -1.119113899705141e+36, -1.1231820283728777e+36, -1.1274007677454447e+36, -1.1317719041241571e+36, -1.1362972430052797e+36, -1.1409786087140785e+36, -1.1458178440105564e+36, -1.1508168096658922e+36, -1.1559773840085307e+36, -1.1613014624388771e+36, -1.1667909569116677e+36, -1.1724477953847241e+36, -1.1782739212333055e+36, -1.1842712926286895e+36, -1.190441881880104e+36, -1.1967876747387965e+36, -1.2033106696631547e+36, -1.210012877043795e+36, -1.2168963183874105e+36, -1.2239630254583737e+36, -1.2312150393768955e+36, -1.2386544096726494e+36, -1.2462831932927473e+36, -1.2541034535629428e+36, -1.2621172591009633e+36, -1.2703266826809548e+36, -1.2787338000478693e+36, -1.2873406886807956e+36, -1.2961494265042215e+36, -1.3051620905461849e+36, -1.3143807555423252e+36, -1.3238074924849937e+36, -1.3334443671163424e+36, -1.3432934383646948e+36, -1.3533567567232227e+36, -1.3636363625702948e+36, -1.3741342844306645e+36, -1.384852537176856e+36, -1.3957931201702232e+36, -1.4069580153409367e+36, -1.4183491852067657e+36, -1.4299685708299267e+36, -1.441818089711926e+36, -1.4538996336260416e+36, -1.4662150663874826e+36, -1.4787662215609964e+36, -1.491554900106225e+36, -1.5045828679608534e+36, -1.517851853561933e+36, -1.5313635453058345e+36, -1.545119588947413e+36, -1.5591215849389804e+36, -1.573371085710124e+36, -1.58786959288924e+36, -1.6026185544679897e+36, -1.6176193619100142e+36, -1.6328733472053647e+36, -1.648381779872349e+36, -1.664145863908593e+36, -1.680166734693436e+36, -1.696445455843768e+36, -1.7129830160258676e+36, -1.7297803257256864e+36, -1.7468382139805997e+36, -1.7641574250755056e+36, -1.7817386152066535e+36, -1.799582349116513e+36, -1.8176890967036458e+36, -1.8360592296111142e+36, -1.8546930177979865e+36, -1.873590626097968e+36, -1.892752110769957e+36, -1.91217741604532e+36, -1.9318663706768802e+36, -1.951818684494948e+36, -1.9720339449758967e+36, -1.9925116138290523e+36, -2.0132510236077217e+36, -2.0342513743507358e+36, -2.0555117302605942e+36, -2.0770310164250394e+36, -2.098808015588739e+36, -2.1208413649818805e+36, -2.143129553213061e+36, -2.1656709172335582e+36, -2.1884636393805275e+36, -2.2115057445066164e+36, -2.2347950972038457e+36, -2.2583293991293512e+36, -2.2821061864411903e+36, -2.306122827351833e+36, -2.3303765198077767e+36, -2.3548642893030148e+36, -2.3795829868346192e+36, -2.4045292870086223e+36, -2.4296996863039704e+36, -2.4550905015028272e+36, -2.480697868295052e+36, -2.5065177400646907e+36, -2.532545886866088e+36, -2.5587778945974346e+36, -2.5852091643786956e+36, -2.6118349121414884e+36, -2.6386501684375807e+36, -2.6656497784726275e+36, -2.69282840237168e+36, -2.7201805156822774e+36, -2.7477004101209175e+36, -2.775382194568152e+36, -2.803219796317081e+36, -2.831206962579844e+36, -2.859337262255908e+36, -2.887604087965672e+36, -2.916000658352357e+36, -2.944520020654462e+36, -2.973155053550839e+36, -3.0018984702793074e+36, -3.030742822029716e+36, -3.059680501611288e+36, -3.088703747393631e+36, -3.1178046475200795e+36, -3.146975144391454e+36, -3.176207039417386e+36, -3.2054919980320326e+36, -3.2348215549699477e+36, -3.2641871197973626e+36, -3.293579982693437e+36, -3.322991320475188e+36, -3.352412202859233e+36, -3.381833598952808e+36, -3.4112463839655943e+36, -3.4406413461336124e+36, -3.470009193845435e+36, -3.49934056296047e+36, -3.5286260243084254e+36, -3.5578560913586086e+36, -3.5870212280468144e+36, -3.616111856747395e+36, -3.6451183663773396e+36, -3.67403112061878e+36, -3.70284046624609e+36, -3.731536741542893e+36, -3.7601102847944977e+36, -3.7885514428406084e+36, -3.816850579672843e+36, -3.844998085061689e+36, -3.872984383197036e+36, -3.9007999413264904e+36, -3.9284352783754167e+36, -3.955880973532838e+36, -3.9831276747871925e+36, -4.0101661073960254e+36, -4.0369870822739116e+36, -4.0635815042829454e+36, -4.089940380410465e+36, -4.116054827818902e+36, -4.1419160817529266e+36, -4.1675155032894746e+36, -4.1928445869167374e+36, -4.217894967928286e+36, -4.242658429619597e+36, -4.267126910274162e+36, -4.291292509927279e+36, -4.315147496896286e+36, -4.3386843140662976e+36, -4.3618955849215225e+36, -4.3847741193127066e+36, -4.4073129189521276e+36, -4.4295051826280363e+36, -4.451344311131672e+36, -4.4728239118900547e+36, -4.4939378032993575e+36, -4.514680018753591e+36, -4.535044810364958e+36, -4.5550266523722656e+36, -4.5746202442352133e+36, -4.593820513412643e+36, -4.6126226178241016e+36, -4.631021947994333e+36, -4.649014128881477e+36, -4.666595021390386e+36, -4.6837607235728725e+36, -4.7005075715180774e+36, -4.7168321399360266e+36, -4.732731242438746e+36, -4.7482019315235676e+36, -4.763241498263944e+36, -4.7778474717137476e+36, -4.7920176180313583e+36, -4.80574993933062e+36, -4.8190426722658686e+36, -4.8318942863590797e+36, -4.844303482077176e+36, -4.8562691886682126e+36, -4.867790561765193e+36, -4.8788669807669016e+36, -4.889498046005084e+36, -4.8996835757076375e+36, -4.9094236027678017e+36, -4.918718371329149e+36, -4.927568333196694e+36, -4.935974144084178e+36, -4.943936659707907e+36, -4.9514569317372265e+36, -4.958536203612185e+36, -4.965175906238299e+36, -4.9713776535688015e+36, -4.977143238084311e+36, -4.9824746261800074e+36, -4.987373953469987e+36, -4.991843520018476e+36, -4.995885785507507e+36, -4.999503364350199e+36, -5.002699020758679e+36, -5.005475663775683e+36, -5.0078363422781104e+36, -5.0097842399611797e+36, -5.011322670311039e+36, -5.0124550715736633e+36, -5.013185001727566e+36, -5.013516133467583e+36, -5.0134522492064974e+36, -5.012997236101277e+36, -5.012155081110093e+36, -5.010929866086259e+36, -5.0093257629146004e+36, -5.007347028695893e+36, -5.004998000984142e+36, -5.0022830930817945e+36, -4.9992067893970404e+36, -4.9957736408676994e+36, -4.9919882604553394e+36, -4.987855318713267e+36, -4.98337953943178e+36, -4.9785656953636965e+36, -4.9734186040327826e+36, -4.9679431236277984e+36, -4.9621441489842727e+36, -4.956026607655907e+36, -4.949595456077676e+36, -4.942855675821706e+36, -4.9358122699476863e+36, -4.928470259448458e+36, -4.920834679792035e+36, -4.912910577560351e+36, -4.9047030071856093e+36, -4.8962170277842136e+36, -4.8874577000885124e+36, -4.8784300834763404e+36, -4.8691392330982365e+36, -4.8595901971017186e+36, -4.849788013952489e+36, -4.839737709851681e+36, -4.829444296248493e+36, -4.8189127674474885e+36, -4.808148098309325e+36, -4.797155242044089e+36, -4.7859391280960374e+36, -4.7745046601183836e+36, -4.7628567140368914e+36, -4.751000136201023e+36, -4.73893974162089e+36, -4.72668031228872e+36, -4.714226595583206e+36, -4.7015833027551475e+36, -4.688755107492599e+36, -4.675746644564034e+36, -4.6625625085376853e+36, -4.649207252575199e+36, -4.635685387298115e+36, -4.62200137972509e+36, -4.60815965227823e+36, -4.5941645818566694e+36, -4.5800204989755655e+36, -4.5657316869686763e+36, -4.551302381252745e+36, -4.536736768651847e+36, -4.522038986779869e+36, -4.507213123479325e+36, -4.4922632163147733e+36, -4.477193252118995e+36, -4.4620071665902065e+36, -4.446708843938522e+36, -4.431302116580098e+36, -4.415790764877023e+36, -4.400178516921489e+36, -4.3844690483625326e+36, -4.368665982273676e+36, -4.352772889059901e+36, -4.3367932864024875e+36, -4.320730639240001e+36, -4.3045883597841463e+36, -4.2883698075687505e+36, -4.272078289530725e+36, -4.255717060121321e+36, -4.239289321446557e+36, -4.2227982234352615e+36, -4.2062468640335315e+36, -4.1896382894243275e+36, -4.172975494270934e+36, -4.1562614219830434e+36, -4.139498965004373e+36, -4.122690965120598e+36, -4.105840213786437e+36, -4.0889494524710664e+36, -4.072021373020392e+36, -4.055058618035567e+36, -4.038063781266546e+36, -4.0210394080197855e+36, -4.0039879955791365e+36, -3.9869119936391535e+36, -3.9698138047497337e+36, -3.9526957847715304e+36, -3.9355602433410925e+36, -3.918409444345134e+36, -3.901245606403052e+36, -3.884070903357097e+36, -3.866887464769429e+36, -3.8496973764253194e+36, -3.8325026808421335e+36, -3.815305377783129e+36, -3.7981074247757183e+36, -3.780910737633506e+36, -3.7637171909816926e+36, -3.7465286187851923e+36, -3.7293468148789794e+36, -3.712173533500309e+36, -3.695010489822183e+36, -3.677859360487816e+36, -3.660721784145488e+36, -3.643599361983542e+36, -3.626493658265119e+36, -3.6094062008621924e+36, -3.592338481788662e+36, -3.5752919577321224e+36, -3.5582680505840344e+36, -3.54126814796797e+36, -3.524293603765663e+36, -3.507345738640759e+36, -3.49042584055959e+36, -3.4735351653093735e+36, -3.456674937013027e+36, -3.4398463486407205e+36, -3.4230505625179437e+36, -3.406288710829791e+36, -3.38956189612136e+36, -3.3728711917942186e+36, -3.356217642598478e+36, -3.339602265120744e+36, -3.32302604826751e+36, -3.3064899537440065e+36, -3.289994916528328e+36, -3.2735418453409005e+36, -3.257131623108903e+36, -3.240765107425945e+36, -3.224443131006531e+36, -3.208166502135539e+36, -3.191936005112544e+36, -3.1757524006908997e+36, -3.159616426511552e+36, -3.143528797531665e+36, -3.127490206447757e+36, -3.11150132411361e+36, -3.095562799952741e+36, -3.079675262365522e+36, -3.0638393191308066e+36, -3.0480555578022125e+36, -3.032324546098934e+36, -3.0166468322911454e+36, -3.001022945579977e+36, -2.9854533964721064e+36, -2.9699386771488655e+36, -2.954479261830055e+36, -2.939075607132347e+36, -2.9237281524223206e+36, -2.9084373201641664e+36, -2.893203516262158e+36, -2.878027130397779e+36, -2.8629085363616126e+36, -2.847848092380091e+36, -2.832846141437014e+36, -2.817903011589945e+36, -2.8030190162815093e+36, -2.7881944546456806e+36, -2.7734296118090055e+36, -2.7587247591868724e+36, -2.74408015477488e+36, -2.7294960434353455e+36, -2.71497265717892e+36, -2.7005102154414914e+36, -2.686108925356344e+36, -2.671768982021608e+36, -2.6574905687631508e+36, -2.6432738573928383e+36, -2.6291190084622578e+36, -2.6150261715120932e+36, -2.6009954853169327e+36, -2.587027078125826e+36, -2.5731210678984704e+36, -2.559277562537239e+36, -2.5454966601149006e+36, -2.5317784490983045e+36, -2.518123008567907e+36, -2.5045304084333095e+36, -2.4910007096448265e+36, -2.47753396440112e+36, -2.4641302163530214e+36, -2.4507895008034285e+36, -2.4375118449036533e+36, -2.4242972678458553e+36, -2.411145781051998e+36, -2.398057388359156e+36, -2.385032086201324e+36, -2.372069863787674e+36, -2.359170703277496e+36, -2.346334579951714e+36, -2.3335614623810864e+36, -2.320851312591169e+36, -2.3082040862240473e+36, -2.2956197326969294e+36, -2.2830981953576085e+36, -2.2706394116369415e+36, -2.258243313198191e+36, -2.245909826083598e+36, -2.2336388708578438e+36, -2.2214303627488654e+36, -2.209284211785693e+36, -2.197200322933661e+36, -2.185178596226801e+36, -2.1732189268976667e+36, -2.1613212055045027e+36, -2.1494853180558104e+36, -2.1377111461324536e+36, -2.1259985670072514e+36, -2.114347453762161e+36, -2.1027576754030174e+36, -2.0912290969719894e+36, -2.0797615796576714e+36, -2.068354980903004e+36, -2.0570091545108524e+36, -2.04572395074749e+36, -2.0344992164439164e+36, -2.0233347950951137e+36, -2.012230526957156e+36, -2.0011862491423768e+36, -1.990201795712509e+36, -1.9792769977699506e+36, -1.9684116835469598e+36, -1.9576056784931782e+36, -1.9468588053611817e+36, -1.936170884290288e+36, -1.925541732888529e+36, -1.914971166313009e+36, -1.9044589973484513e+36, -1.8940050364841013e+36, -1.8836090919890768e+36, -1.8732709699860076e+36, -1.8629904745231432e+36, -1.852767407644974e+36, -1.8426015694612527e+36, -1.8324927582145818e+36, -1.8224407703466084e+36, -1.812445400562652e+36, -1.802506441895091e+36, -1.7926236857652973e+36, -1.782796922044235e+36, -1.7730259391117966e+36, -1.7633105239148027e+36, -1.7536504620237834e+36, -1.7440455376884943e+36, -1.7344955338922397e+36, -1.7250002324049844e+36, -1.7155594138353938e+36, -1.7061728576815996e+36, -1.6968403423809607e+36, -1.6875616453587088e+36, -1.6783365430754802e+36, -1.6691648110738732e+36, -1.6600462240239342e+36, -1.6509805557675894e+36, -1.6419675793622563e+36, -1.6330070671233193e+36, -1.624098790665732e+36, -1.615242520944708e+36, -1.606438028295521e+36, -1.5976850824723228e+36, -1.5889834526862504e+36, -1.580332907642457e+36, -1.5717332155766374e+36, -1.563184144290389e+36, -1.5546854611860233e+36, -1.5462369333004563e+36, -1.5378383273384243e+36, -1.5294894097049076e+36, -1.5211899465367817e+36, -1.512939703733863e+36, -1.5047384469891376e+36, -1.496585941818342e+36, -1.488481953588894e+36, -1.480426247548139e+36, -1.4724185888509428e+36, -1.464458742586684e+36, -1.4565464738055617e+36, -1.4486815475443942e+36, -1.4408637288517348e+36, -1.433092782812456e+36, -1.4253684745717472e+36, -1.417690569358536e+36, -1.410058832508398e+36, -1.402473029485962e+36, -1.3949329259066508e+36, -1.387438287558104e+36, -1.3799888804209562e+36, -1.372584470689155e+36, -1.3652248247898303e+36, -1.3579097094027168e+36, -1.3506388914789726e+36, -1.3434121382597646e+36, -1.3362292172941874e+36, -1.3290898964569304e+36, -1.3219939439654332e+36, -1.3149411283965868e+36, -1.307931218703136e+36, -1.3009639842296067e+36, -1.2940391947278925e+36, -1.2871566203723568e+36, -1.2803160317747083e+36, -1.2735171999984298e+36, -1.266759896572789e+36, -1.2600438935066444e+36, -1.2533689633017887e+36, -1.2467348789659901e+36, -1.2401414140257075e+36, -1.2335883425385022e+36, -1.2270754391050842e+36, -1.2206024788810796e+36, -1.2141692375885006e+36, -1.2077754915269114e+36, -1.2014210175842744e+36, -1.1951055932475573e+36, -1.1888289966129944e+36, -1.1825910063962066e+36, -1.1763914019418247e+36, -1.1702299632331307e+36, -1.1641064709011687e+36, -1.1580207062338327e+36, -1.1519724511845197e+36, -1.1459614883806776e+36, -1.1399876011320492e+36, -1.1340505734386841e+36, -1.1281501899987348e+36, -1.1222862362160198e+36, -1.116458498207401e+36, -1.1106667628098706e+36, -1.1049108175875261e+36, -1.0991904508382377e+36, -1.0935054516001987e+36, -1.0878556096582044e+36, -1.0822407155498049e+36, -1.0766605605711987e+36, -1.0711149367829947e+36, -1.0656036370157546e+36, -1.0601264548753938e+36, -1.0546831847483234e+36, -1.0492736218065303e+36, -1.0438975620123961e+36, -1.0385548021234272e+36, -1.0332451396966983e+36, -1.0279683730933039e+36, -1.0227243014824975e+36, -1.017512724845803e+36, -1.0123334439808202e+36, -1.007186260505096e+36, -1.002070976859626e+36, -9.969873963123459e+35, -9.919353229615014e+35, -9.869145617387482e+35, -9.819249184122647e+35, -9.769661995896364e+35, -9.720382127206393e+35, -9.671407660998897e+35, -9.622736688694234e+35, -9.574367310210286e+35, -9.526297633985931e+35, -9.478525777002351e+35, -9.431049864803957e+35, -9.383868031517202e+35, -9.33697841986916e+35, -9.29037918120483e+35, -9.24406847550266e+35, -9.19804447139035e+35, -9.152305346158091e+35, -9.106849285771732e+35, -9.061674484884802e+35, -9.016779146849516e+35, -8.972161483726466e+35, -8.927819716293962e+35, -8.883752074055717e+35, -8.839956795248748e+35, -8.79643212684873e+35, -8.7531763245761e+35, -8.710187652900074e+35, -8.667464385042793e+35, -8.625004802981736e+35, -8.582807197452002e+35, -8.54086986794725e+35, -8.499191122720466e+35, -8.457769278783233e+35, -8.416602661904616e+35, -8.375689606609527e+35, -8.335028456175999e+35, -8.29461756263165e+35, -8.254455286749735e+35, -8.214539998044354e+35, -8.174870074765098e+35, -8.135443903890868e+35, -8.096259881122878e+35, -8.057316410877625e+35, -8.018611906278276e+35, -7.980144789146359e+35, -7.94191348999253e+35, -7.903916448006262e+35, -7.866152111045329e+35, -7.828618935625148e+35, -7.79131538690662e+35, -7.754239938683913e+35, -7.717391073372365e+35, -7.680767281993918e+35, -7.644367064164464e+35, -7.6081889280785e+35, -7.572231390494833e+35, -7.536492976720954e+35, -7.500972220596868e+35, -7.465667664479474e+35, -7.430577859224781e+35, -7.395701364170939e+35, -7.36103674712058e+35, -7.326582584322431e+35, -7.292337460452824e+35, -7.258299968596157e+35, -7.224468710225554e+35, -7.19084229518314e+35, -7.157419341659258e+35, -7.124198476171998e+35, -7.09117833354589e+35, -7.058357556890234e+35, -7.025734797577332e+35, -6.993308715220464e+35, -6.961077977650255e+35, -6.929041260892885e+35, -6.897197249145688e+35, -6.865544634754088e+35, -6.834082118187329e+35, -6.802808408013713e+35, -6.771722220876601e+35, -6.740822281468849e+35, -6.710107322507663e+35, -6.679576084709103e+35, -6.64922731676158e+35, -6.619059775300491e+35, -6.589072224881334e+35, -6.559263437952268e+35, -6.529632194828326e+35, -6.500177283663124e+35, -6.470897500421645e+35, -6.441791648852253e+35, -6.412858540459032e+35, -6.384096994472723e+35, -6.35550583782295e+35, -6.3270839051089655e+35, -6.298830038570732e+35, -6.270743088059806e+35, -6.2428219110097904e+35, -6.215065372406669e+35, -6.187472344758998e+35, -6.160041708067815e+35, -6.1327723497965815e+35, -6.1056631648404e+35, -6.078713055495597e+35, -6.051920931429243e+35, -6.0252857096473475e+35, -5.998806314464627e+35, -5.972481677472705e+35, -5.946310737508639e+35, -5.9202924406236314e+35, -5.894425740051051e+35, -5.86870959617458e+35, -5.843142976495888e+35, -5.8177248556031994e+35, -5.7924542151380676e+35, -5.767330043763633e+35, -5.742351337131741e+35, -5.717517097850401e+35, -5.692826335450998e+35, -5.668278066355073e+35, -5.643871313841965e+35, -5.6196051080149476e+35, -5.59547848576887e+35, -5.571490490756428e+35, -5.547640173354178e+35, -5.5239265906304024e+35, -5.500348806310442e+35, -5.476905890743785e+35, -5.4535969208698734e+35, -5.430420980184794e+35, -5.4073771587069346e+35, -5.384464552943823e+35, -5.361682265857506e+35, -5.339029406830661e+35, -5.3165050916331324e+35, -5.2941084423871585e+35, -5.2718385875337256e+35, -5.249694661797876e+35, -5.2276758061551024e+35, -5.205781167796738e+35, -5.184009900095747e+35, -5.1623611625725535e+35, -5.140834120860528e+35, -5.119427946671907e+35, -5.0981418177631815e+35, -5.076974917901041e+35, -5.055926436827731e+35, -5.034995570226791e+35, -5.0141815196887215e+35, -4.993483492676523e+35, -4.9729007024915566e+35, -4.95243236823872e+35, -4.932077714792571e+35, -4.911835972762674e+35, -4.891706378459504e+35, -4.871688173860052e+35, -4.8517806065733185e+35, -4.8319829298064904e+35, -4.8122944023301486e+35, -4.79271428844478e+35, -4.773241857946004e+35, -4.753876386090722e+35, -4.7346171535626756e+35, -4.7154634464391005e+35, -4.696414556155935e+35, -4.677469779474332e+35, -4.658628418446543e+35, -4.639889780382168e+35, -4.621253177814168e+35, -4.602717928465218e+35, -4.584283355214203e+35, -4.565948786062048e+35, -4.547713554098803e+35, -4.5295769974694586e+35, -4.511538459340984e+35, -4.493597287869098e+35, -4.4757528361641214e+35, -4.4580044622585765e+35, -4.440351529073689e+35, -4.422793404386462e+35, -4.40532946079647e+35, -4.3879590756932894e+35, -4.370681631223108e+35, -4.353496514256465e+35, -4.336403116355678e+35, -4.319400833741656e+35, -4.30248906726213e+35, -4.285667222358808e+35, -4.268934709035271e+35, -4.252290941824992e+35, -4.2357353397585384e+35, -4.2192673263326986e+35, -4.2028863294776894e+35, -4.18659178152552e+35, -4.17038311917872e+35, -4.1542597834784405e+35, -4.138221219773178e+35, -4.122266877687207e+35, -4.106396211089683e+35, -4.090608678063245e+35, -4.074903740872919e+35, -4.0592808659358055e+35, -4.043739523789716e+35, -4.028279189062704e+35, -4.012899340442684e+35, -3.9975994606469235e+35, -3.982379036391824e+35, -3.967237558362553e+35, -3.952174521183185e+35, -3.93718942338692e+35, -3.922281767386202e+35, -3.907451059442835e+35, -3.892696809638971e+35, -3.878018531847148e+35, -3.863415743701574e+35, -3.8488879665686394e+35, -3.8344347255180545e+35, -3.8200555492942374e+35, -3.8057499702871854e+35, -3.7915175245041076e+35, -3.777357751541087e+35, -3.763270194554627e+35, -3.74925440023357e+35, -3.73530991877109e+35, -3.721436303836941e+35, -3.707633112549216e+35, -3.693899905447566e+35, -3.680236246465129e+35, -3.666641702901415e+35, -3.653115845395175e+35, -3.639658247897431e+35, -3.6262684876446665e+35, -3.612946145131749e+35, -3.599690804085912e+35, -3.5865020514398436e+35, -3.5733794773056924e+35, -3.5603226749490374e+35, -3.547331240762399e+35, -3.534404774240114e+35, -3.521542877952111e+35, -3.508745157518744e+35, -3.496011221585086e+35, -3.483340681796089e+35, -3.470733152771011e+35, -3.4581882520791444e+35, -3.4457056002144185e+35, -3.433284820571262e+35, -3.4209255394198925e+35, -3.408627385881969e+35, -3.396389991906548e+35, -3.384212992246242e+35, -3.372096024432849e+35, -3.360038728754085e+35, -3.348040748230259e+35, -3.3361017285900916e+35, -3.324221318248223e+35, -3.312399168281719e+35, -3.3006349324071787e+35, -3.2889282669582156e+35, -3.27727883086263e+35, -3.2656862856197197e+35, -3.2541502952785765e+35, -3.242670526415184e+35, -3.2312466481109195e+35, -3.2198783319305116e+35, -3.208565251900305e+35, -3.1973070844867634e+35, -3.1861035085749596e+35, -3.1749542054477146e+35, -3.1638588587636595e+35, -3.15281715453734e+35, -3.141828781117635e+35, -3.130893429167297e+35, -3.1200107916427374e+35, -3.1091805637733526e+35, -3.098402443041538e+35, -3.0876761291624473e+35, -3.077001324064199e+35, -3.066377731868239e+35, -3.0558050588697193e+35, -3.0452830135178796e+35, -3.0348113063967827e+35, -3.024389650206258e+35, -3.014017759743076e+35, -3.003695351881591e+35, -2.9934221455555784e+35, -2.983197861739463e+35, -2.9730222234294872e+35, -2.9628949556262576e+35, -2.952815785315976e+35, -2.9427844414527658e+35, -2.9328006549405104e+35, -2.9228641586157528e+35, -2.912974687229551e+35, -2.903131977430435e+35, -2.8933357677467917e+35, -2.883585798570188e+35, -2.8738818121380753e+35, -2.8642235525169794e+35, -2.854610765586031e+35, -2.8450431990199818e+35, -2.835520602273319e+35, -2.826042726563454e+35, -2.8166093248548763e+35, -2.8072201518431872e+35, -2.7978749639390013e+35, -2.7885735192523927e+35, -2.7793155775771845e+35, -2.7701009003755375e+35, -2.7609292507625113e+35, -2.7518003934912315e+35, -2.742714094937254e+35, -2.7336701230841126e+35, -2.724668247508253e+35, -2.7157082393644413e+35, -2.7067898713711578e+35, -2.697912917796415e+35, -2.6890771544432714e+35, -2.6802823586354164e+35, -2.671528309203816e+35, -2.662814786472321e+35, -2.6541415722438812e+35, -2.6455084497871055e+35, -2.636915203822752e+35, -2.6283616205099855e+35, -2.6198474874335858e+35, -2.611372593590469e+35, -2.602936729376593e+35, -2.594539686574526e+35, -2.5861812583400694e+35, -2.5778612391898482e+35, -2.5695794249888654e+35, -2.5613356129381167e+35, -2.5531296015617104e+35, -2.5449611906954786e+35, -2.536830181474428e+35, -2.5287363763206563e+35, -2.5206795789318626e+35, -2.512659594269624e+35, -2.5046762285472597e+35, -2.4967292892189152e+35, -2.4888185849680098e+35, -2.4809439256954987e+35, -2.4731051225095344e+35, -2.465301987713567e+35, -2.4575343347959674e+35, -2.449801978418946e+35, -2.442104734407902e+35, -2.4344424197406875e+35, -2.426814852537163e+35, -2.419221852048599e+35, -2.411663238647912e+35, -2.4041388338187254e+35, -2.3966484601457402e+35, -2.3891919413045233e+35, -2.381769102051838e+35, -2.374379768215573e+35, -2.3670237666853014e+35, -2.3597009254024954e+35, -2.3524110733508912e+35, -2.3451540405477038e+35, -2.337929658033432e+35, -2.3307377578635927e+35, -2.3235781730988346e+35, -2.3164507377963534e+35, -2.3093552870006667e+35, -2.302291656735441e+35, -2.2952596839940162e+35, -2.288259206730989e+35, -2.2812900638540174e+35, -2.2743520952146936e+35, -2.267445141601024e+35, -2.26056904472844e+35, -2.2537236472320714e+35, -2.246908792658386e+35, -2.240124325457234e+35, -2.2333700909741215e+35, -2.226645935441978e+35, -2.219951705973985e+35, -2.2132872505552697e+35, -2.2066524180360278e+35, -2.200047058123349e+35, -2.1934710213741653e+35, -2.186924159188074e+35, -2.1804063237997187e+35, -2.173917368271725e+35, -2.1674571464879393e+35, -2.1610255131460966e+35, -2.1546223237508774e+35, -2.148247434607346e+35, -2.1419007028138443e+35, -2.1355819862556264e+35, -2.1292911435978633e+35, -2.1230280342796082e+35, -2.1167925185069017e+35, -2.1105844572464585e+35, -2.1044037122197157e+35, -2.098250145895744e+35, -2.092123621486455e+35, -2.086024002939068e+35, -2.0799511549309533e+35, -2.073904942863297e+35, -2.067885232855397e+35, -2.0618918917389735e+35, -2.0559247870521357e+35, -2.0499837870337487e+35, -2.044068760618061e+35, -2.038179577428947e+35, -2.0323161077743433e+35, -2.0264782226410786e+35, -2.0206657936896048e+35, -2.014878693248176e+35, -2.0091167943084177e+35, -2.003379970519322e+35, -1.9976680961826847e+35, -1.9919810462484932e+35, -1.986318696308607e+35, -1.9806809225930533e+35, -1.975067601965123e+35, -1.9694786119156716e+35, -1.9639138305595273e+35, -1.958373136629746e+35, -1.952856409474021e+35, -1.9473635290494884e+35, -1.9418943759179498e+35, -1.9364488312425306e+35, -1.9310267767822486e+35, -1.925628094888034e+35, -1.920252668498584e+35, -1.9149003811359465e+35, -1.909571116901473e+35, -1.904264760471663e+35, -1.8989811970939315e+35, -1.8937203125831052e+35, -1.8884819933167782e+35, -1.8832661262319456e+35, -1.8780725988206696e+35, -1.8729012991268943e+35, -1.86775211574195e+35, -1.8626249378014953e+35, -1.8575196549813733e+35, -1.852436157494199e+35, -1.847374336085914e+35, -1.8423340820318093e+35, -1.8373152871335064e+35, -1.8323178437154837e+35, -1.8273416446214354e+35, -1.8223865832110475e+35, -1.817452553356478e+35, -1.8125394494398626e+35, -1.807647166348927e+35, -1.802775599474871e+35, -1.797924644708596e+35, -1.7930941984377814e+35, -1.78828415754402e+35, -1.7834944193995324e+35, -1.7787248818643233e+35, -1.7739754432831908e+35, -1.7692460024831646e+35, -1.7645364587698787e+35, -1.7598467119254093e+35, -1.7551766622056776e+35, -1.750526210336898e+35, -1.745895257513618e+35, -1.7412837053954866e+35, -1.7366914561051407e+35, -1.7321184122251008e+35, -1.7275644767957593e+35, -1.7230295533121292e+35, -1.7185135457222054e+35, -1.7140163584235818e+35, -1.7095378962617697e+35, -1.7050780645275448e+35, -1.70063676895442e+35, -1.6962139157165395e+35, -1.691809411425928e+35, -1.687423163131085e+35, -1.6830550783137624e+35, -1.678705064887274e+35, -1.674373031194496e+35, -1.670058886005234e+35, -1.6657625385141177e+35, -1.661483898339007e+35, -1.6572228755183613e+35, -1.6529793805093635e+35, -1.6487533241864472e+35, -1.6445446178382217e+35, -1.6403531731662925e+35, -1.6361789022833047e+35, -1.6320217177105626e+35, -1.6278815323767583e+35, -1.6237582596153758e+35, -1.6196518131636691e+35, -1.6155621071598874e+35, -1.6114890561426491e+35, -1.6074325750478398e+35, -1.6033925792081776e+35, -1.5993689843504915e+35, -1.5953617065943891e+35, -1.5913706624506394e+35, -1.5873957688196525e+35, -1.583436942989157e+35, -1.5794941026335004e+35, -1.5755671658113392e+35, -1.571656050964322e+35, -1.5677606769156322e+35, -1.563880962868233e+35, -1.5600168284034931e+35, -1.556168193479689e+35, -1.552334978430286e+35, -1.5485171039626774e+35, -1.5447144911569095e+35, -1.5409270614639138e+35, -1.537154736704166e+35, -1.5333974390663668e+35, -1.5296550911060118e+35, -1.5259276157440125e+35, -1.5222149362654618e+35, -1.518516976318204e+35, -1.5148336599114512e+35, -1.5111649114145441e+35, -1.5075106555557313e+35, -1.5038708174206482e+35, -1.5002453224514555e+35, -1.4966340964452598e+35, -1.4930370655528402e+35, -1.4894541562778945e+35, -1.485885295475061e+35, -1.4823304103495755e+35, -1.4787894284552988e+35, -1.4752622776941805e+35, -1.4717488863147431e+35, -1.4682491829111282e+35, -1.464763096421594e+35, -1.4612905561279727e+35, -1.457831491653804e+35, -1.4543858329641911e+35, -1.4509535103637596e+35, -1.4475344544961858e+35, -1.4441285963429033e+35, -1.4407358672220676e+35, -1.4373561987873944e+35, -1.43398952302727e+35, -1.4306357722638698e+35, -1.42729487915147e+35, -1.423966776676478e+35, -1.4206513981553148e+35, -1.4173486772342304e+35, -1.4140585478878507e+35, -1.4107809444185391e+35, -1.40751580145515e+35, -1.4042630539519652e+35, -1.4010226371882362e+35, -1.3977944867667378e+35, -1.3945785386129604e+35, -1.3913747289743297e+35, -1.3881829944189654e+35, -1.3850032718351034e+35, -1.381835498429831e+35, -1.3786796117286365e+35, -1.3755355495739009e+35, -1.3724032501243696e+35, -1.369282651854491e+35, -1.366173693552743e+35, -1.3630763143214943e+35, -1.3599904535759823e+35, -1.356916051043017e+35, -1.3538530467607083e+35, -1.3508013810770716e+35, -1.3477609946494887e+35, -1.3447318284435956e+35, -1.3417138237329264e+35, -1.338706922097507e+35, -1.335711065423247e+35, -1.3327261959010762e+35, -1.3297522560261274e+35, -1.3267891885969902e+35, -1.3238369367147146e+35, -1.3208954437820944e+35, -1.317964653502765e+35, -1.3150445098804097e+35, -1.3121349572181488e+35, -1.3092359401172741e+35, -1.306347403477101e+35, -1.3034692924931907e+35, -1.300601552657826e+35, -1.2977441297580945e+35, -1.2948969698753287e+35, -1.2920600193852662e+35, -1.2892332249555312e+35, -1.2864165335465897e+35, -1.2836098924098242e+35, -1.2808132490869132e+35, -1.2780265514097388e+35, -1.2752497474986826e+35, -1.272482785762586e+35, -1.2697256148972221e+35, -1.2669781838853841e+35, -1.2642404419954389e+35, -1.2615123387809787e+35, -1.258793824079581e+35, -1.2560848480126329e+35, -1.2533853609840887e+35, -1.250695313680048e+35, -1.2480146570674955e+35, -1.2453433423941068e+35, -1.242681321187262e+35, -1.2400285452531448e+35, -1.2373849666761932e+35, -1.2347505378181618e+35, -1.2321252113174763e+35, -1.2295089400885756e+35, -1.2269016773209263e+35, -1.224303376478386e+35, -1.2217139912984399e+35, -1.219133475791586e+35, -1.216561784240239e+35, -1.2139988711982842e+35, -1.211444691490345e+35, -1.208899200210852e+35, -1.2063623527231937e+35, -1.2038341046594841e+35, -1.2013144119193721e+35, -1.1988032306690597e+35, -1.1963005173415209e+35, -1.1938062286345375e+35, -1.191320321510815e+35, -1.188842753197223e+35, -1.1863734811832511e+35, -1.1839124632213554e+35, -1.181459657325489e+35, -1.1790150217702303e+35, -1.1765785150910217e+35, -1.1741500960822677e+35, -1.1717297237971944e+35, -1.1693173575471782e+35, -1.1669129569008334e+35, -1.1645164816831504e+35, -1.1621278919748585e+35, -1.159747148111981e+35, -1.1573742106845924e+35, -1.1550090405365567e+35, -1.1526515987642813e+35, -1.150301846716618e+35, -1.147959745993487e+35, -1.1456252584454814e+35, -1.1432983461733622e+35, -1.140978971526615e+35, -1.1386670971035519e+35, -1.1363626857498833e+35, -1.1340657005582771e+35, -1.1317761048679372e+35, -1.1294938622632316e+35, -1.1272189365734676e+35, -1.1249512918718995e+35, -1.1226908924750656e+35, -1.1204377029421606e+35, -1.1181916880741934e+35, -1.1159528129133143e+35, -1.1137210427419078e+35, -1.1114963430821992e+35, -1.109278679695261e+35, -1.107068018580333e+35, -1.1048643259741236e+35, -1.10266756835018e+35, -1.1004777124180463e+35, -1.098294725122511e+35, -1.096118573643109e+35, -1.0939492253927761e+35, -1.0917866480182738e+35, -1.0896308093980845e+35, -1.0874816776428808e+35, -1.0853392210940636e+35, -1.0832034083232478e+35, -1.0810742081317502e+35, -1.0789515895496094e+35, -1.0768355218347341e+35, -1.0747259744727932e+35, -1.072622917175961e+35, -1.0705263198821488e+35, -1.0684361527549042e+35, -1.066352386181895e+35, -1.0642749907747897e+35, -1.0622039373682945e+35, -1.0601391970195377e+35, -1.0580807410073216e+35, -1.056028540831288e+35, -1.0539825682114902e+35, -1.0519427950874706e+35, -1.0499091936176144e+35, -1.047881736178252e+35, -1.045860395363472e+35, -1.043845143983837e+35, -1.041835955066139e+35, -1.039832801852268e+35, -1.0378356577988886e+35, -1.0358444965766742e+35, -1.0338592920691025e+35, -1.0318800183728847e+35, -1.0299066497958244e+35, -1.0279391608574674e+35, -1.0259775262872983e+35, -1.0240217210250665e+35, -1.0220717202193591e+35, -1.0201274992269628e+35, -1.0181890336127572e+35, -1.0162562991481914e+35, -1.014329271811376e+35, -1.012407927785977e+35, -1.0104922434605812e+35, -1.0085821954279814e+35, -1.0066777604848357e+35, -1.0047789156302727e+35, -1.0028856380662373e+35, -1.0009979051957285e+35, -9.991156946230535e+34, -9.972389841524782e+34, -9.953677517880203e+34, -9.935019757323428e+34, -9.916416343865773e+34, -9.897867063494362e+34, -9.879371704161776e+34, -9.86093005578864e+34, -9.842541910245686e+34, -9.824207061355193e+34, -9.805925304884368e+34, -9.78769643853114e+34, -9.769520261928517e+34, -9.75139657662982e+34, -9.733325186104668e+34, -9.715305895732952e+34, -9.697338512798107e+34, -9.679422846483255e+34, -9.661558707858044e+34, -9.643745909879181e+34, -9.625984267380414e+34, -9.608273597066821e+34, -9.590613717509087e+34, -9.573004449136348e+34, -9.555445614232222e+34, -9.537937036921895e+34, -9.520478543177762e+34, -9.503069960797971e+34, -9.485711119413315e+34, -9.46840185047476e+34, -9.451141987247545e+34, -9.433931364805355e+34, -9.416769820027343e+34, -9.399657191584928e+34, -9.382593319944179e+34, -9.365578047351673e+34, -9.348611217835075e+34, -9.3316926771931e+34, -9.314822272989312e+34, -9.29799985455158e+34, -9.28122527295611e+34, -9.264498381029402e+34, -9.247819033342873e+34, -9.231187086201093e+34, -9.214602397637892e+34, -9.198064827415358e+34, -9.181574237009729e+34, -9.16513048961078e+34, -9.148733450117744e+34, -9.132382985128078e+34, -9.116078962934652e+34, -9.099821253518816e+34, -9.083609728547724e+34, -9.06744426136299e+34, -9.051324726980412e+34, -9.03525100208064e+34, -9.019222965007249e+34, -9.00324049575538e+34, -8.987303475969082e+34, -8.971411788942142e+34, -8.955565319596089e+34, -8.939763954493353e+34, -8.924007581818453e+34, -8.908296091378165e+34, -8.892629374595144e+34, -8.877007324500139e+34, -8.861429835731156e+34, -8.845896804522935e+34, -8.830408128704663e+34, -8.81496370769122e+34, -8.799563442483938e+34, -8.78420723565618e+34, -8.768894991357777e+34, -8.753626615301342e+34, -8.7384020147614e+34, -8.723221098567968e+34, -8.708083777100868e+34, -8.692989962282362e+34, -8.677939567580584e+34, -8.662932507988558e+34, -8.647968700035326e+34, -8.633048061768235e+34, -8.618170512755933e+34, -8.603335974079063e+34, -8.588544368325163e+34, -8.573795619583989e+34, -8.559089653443388e+34, -8.544426396981119e+34, -8.529805778763737e+34, -8.515227728837482e+34, -8.500692178724226e+34, -8.486199061419664e+34, -8.471748311382258e+34, -8.457339864532478e+34, -8.442973658247062e+34, -8.428649631349732e+34, -8.41436772411374e+34, -8.400127878250423e+34, -8.385930036903126e+34, -8.37177414465052e+34, -8.357660147490288e+34, -8.34358799284252e+34, -8.329557629543423e+34, -8.315569007835172e+34, -8.301622079364821e+34, -8.287716797179138e+34, -8.273853115720268e+34, -8.260030990814344e+34, -8.246250379676982e+34, -8.232511240898109e+34, -8.218813534442866e+34, -8.205157221646075e+34, -8.191542265202907e+34, -8.177968629169432e+34, -8.164436278952377e+34, -8.150945181308679e+34, -8.137495304336048e+34, -8.124086617469563e+34, -8.110719091481449e+34, -8.097392698464802e+34, -8.084107411839228e+34, -8.070863206340703e+34, -8.0576600580149025e+34, -8.04449794421765e+34, -8.031376843602538e+34, -8.018296736123391e+34, -8.0052576030218135e+34, -7.992259426826343e+34, -7.979302191347116e+34, -7.966385881668223e+34, -7.953510484145816e+34, -7.940675986397238e+34, -7.927882377302868e+34, -7.915129646995873e+34, -7.902417786857139e+34, -7.88974678951228e+34, -7.877116648824735e+34, -7.864527359890003e+34, -7.85197891903147e+34, -7.839471323792728e+34, -7.827004572936628e+34, -7.814578666430809e+34, -7.802193605456247e+34, -7.789849392385475e+34, -7.777546030790062e+34, -7.765283525427077e+34, -7.753061882237749e+34, -7.740881108340257e+34, -7.7287412120240005e+34, -7.7166422027424e+34, -7.704584091108287e+34, -7.692566888890673e+34, -7.680590609005189e+34, -7.668655265508251e+34, -7.656760873593408e+34, -7.644907449583811e+34, -7.633095010927137e+34, -7.621323576187586e+34, -7.609593165043093e+34, -7.597903798275466e+34, -7.586255497765224e+34, -7.574648286488192e+34, -7.563082188503352e+34, -7.551557228955257e+34, -7.54007343405691e+34, -7.528630831090627e+34, -7.517229448401054e+34, -7.505869315386127e+34, -7.494550462488717e+34, -7.483272921196523e+34, -7.472036724029704e+34, -7.460841904534868e+34, -7.449688497280606e+34, -7.438576537847258e+34, -7.427506062822362e+34, -7.416477109793388e+34, -7.4054897173388875e+34, -7.3945439250233315e+34, -7.383639773386223e+34, -7.372777303941131e+34, -7.361956559163113e+34, -7.351177582481377e+34, -7.340440418274116e+34, -7.329745111859431e+34, -7.319091709486882e+34, -7.308480258332923e+34, -7.297910806489372e+34, -7.28738340295758e+34, -7.276898097637359e+34, -7.266454941323388e+34, -7.256053985695781e+34, -7.24569528330762e+34, -7.235378887582569e+34, -7.2251048528034055e+34, -7.214873234103331e+34, -7.204684087455961e+34, -7.1945374696716335e+34, -7.184433438383086e+34, -7.174372052041625e+34, -7.164353369902304e+34, -7.154377452019422e+34, -7.1444443592366485e+34, -7.134554153175728e+34, -7.124706896229482e+34, -7.1149026515507665e+34, -7.105141483043341e+34, -7.095423455354746e+34, -7.085748633860578e+34, -7.076117084661977e+34, -7.0665288745699145e+34, -7.05698407109672e+34, -7.0474827424469335e+34, -7.038024957511685e+34, -7.02861078584413e+34, -7.01924029766556e+34, -7.009913563845942e+34, -7.000630655892591e+34, -6.991391645944283e+34, -6.982196606757388e+34, -6.973045611695384e+34, -6.963938734717145e+34, -6.954876050367306e+34, -6.945857633764298e+34, -6.936883560588026e+34, -6.927953907070977e+34, -6.919068749982526e+34, -6.910228166622617e+34, -6.901432234802423e+34, -6.892681032843973e+34, -6.883974639553213e+34, -6.875313134223448e+34, -6.866696596610275e+34, -6.858125106928143e+34, -6.849598745831944e+34, -6.841117594407413e+34, -6.832681734158892e+34, -6.824291246995116e+34, -6.815946215218341e+34, -6.807646721506717e+34, -6.799392848907966e+34, -6.791184680822748e+34, -6.783022300988222e+34, -6.774905793470856e+34, -6.76683524265058e+34, -6.758810733203585e+34, -6.750832350094928e+34, -6.742900178559063e+34, -6.735014304091911e+34, -6.72717481243141e+34, -6.719381789544413e+34, -6.711635321616408e+34, -6.703935495032923e+34, -6.696282396367185e+34, -6.68867611236616e+34, -6.681116729934926e+34, -6.673604336123368e+34, -6.666139018107202e+34, -6.658720863181351e+34, -6.651349958736584e+34, -6.644026392250319e+34, -6.636750251268029e+34, -6.6295216233925695e+34, -6.622340596261947e+34, -6.615207257542177e+34, -6.608121694904171e+34, -6.601083996017827e+34, -6.594094248523325e+34, -6.58715254002859e+34, -6.580258958086672e+34, -6.573413590181687e+34, -6.5666165237133435e+34, -6.5598678459793485e+34, -6.553167644162578e+34, -6.54651600531261e+34, -6.5399130163307635e+34, -6.53335876395479e+34, -6.526853334740813e+34, -6.520396815048238e+34, -6.513989291025926e+34, -6.507630848591877e+34, -6.5013215734197335e+34, -6.495061550920922e+34, -6.4888508662277815e+34, -6.482689604183198e+34, -6.476577849315322e+34, -6.470515685825954e+34, -6.46450319757565e+34, -6.458540468062516e+34, -6.452627580411815e+34, -6.446764617352742e+34, -6.440951661205139e+34, -6.435188793868942e+34, -6.4294760967954885e+34, -6.4238136509812535e+34, -6.418201536946047e+34, -6.412639834719226e+34, -6.407128623820521e+34, -6.401667983249331e+34, -6.39625799145942e+34, -6.39089872634988e+34, -6.385590265246573e+34, -6.380332684884175e+34, -6.375126061393293e+34, -6.36997047027954e+34, -6.364865986411817e+34, -6.359812684002964e+34, -6.354810636595584e+34, -6.349859917046303e+34, -6.344960597506138e+34, -6.340112749408057e+34, -6.3353164434511535e+34, -6.330571749583361e+34, -6.325878736984823e+34, -6.321237474055932e+34, -6.316648028396105e+34, -6.312110466793447e+34, -6.3076248552089325e+34, -6.303191258755594e+34, -6.298809741688901e+34, -6.29448036739115e+34, -6.290203198354881e+34, -6.2859782961647495e+34, -6.281805721487982e+34, -6.277685534062866e+34, -6.273617792670894e+34, -6.269602555135188e+34, -6.2656398783013885e+34, -6.261729818022019e+34, -6.257872429145998e+34, -6.254067765500199e+34, -6.250315879879931e+34, -6.246616824033304e+34, -6.242970648647639e+34, -6.239377403337216e+34, -6.235837136627528e+34, -6.232349895945533e+34, -6.228915727606558e+34, -6.225534676797501e+34, -6.222206787570993e+34, -6.218932102826735e+34, -6.215710664303183e+34, -6.212542512562341e+34, -6.209427686984659e+34, -6.20636622574929e+34, -6.203358165829021e+34, -6.200403542974039e+34, -6.197502391703335e+34, -6.194654745297245e+34, -6.1918606357819335e+34, -6.189120093920455e+34, -6.18643314920443e+34, -6.183799829841602e+34, -6.181220162748934e+34, -6.178694173541078e+34, -6.176221886519412e+34, -6.173803324671998e+34, -6.1714385096509e+34, -6.169127461775192e+34, -6.166870200017841e+34, -6.1646667420001525e+34, -6.162517103979124e+34, -6.160421300846915e+34, -6.15837934611644e+34, -6.156391251921364e+34, -6.154457029003832e+34, -6.152576686711408e+34, -6.150750232986855e+34, -6.148977674366789e+34, -6.147259015976029e+34, -6.145594261516922e+34, -6.143983413269421e+34, -6.142426472083525e+34, -6.140923437374963e+34, -6.139474307125112e+34, -6.138079077867839e+34, -6.136737744693878e+34, -6.135450301246207e+34, -6.1342167397131405e+34, -6.133037050827054e+34, -6.131911223865372e+34, -6.130839246640818e+34, -6.129821105508744e+34, -6.128856785354432e+34, -6.127946269603481e+34, -6.12708954020939e+34, -6.126286577662046e+34, -6.1255373609789065e+34, -6.124841867712325e+34, -6.12420007394405e+34, -6.123611954287003e+34, -6.123077481887365e+34, -6.122596628420485e+34, -6.122169364101005e+34, -6.121795657675617e+34, -6.121475476429063e+34, -6.121208786183407e+34, -6.120995551303041e+34, -6.120835734695735e+34, -6.120729297818085e+34, -6.120676200673e+34, -6.120676401818065e+34, -6.120729858368528e+34, -6.120836525998987e+34, -6.1209963589484e+34, -6.121209310028207e+34, -6.12147533062128e+34, -6.121794370690485e+34, -6.122166378785693e+34, -6.122591302042126e+34, -6.123069086198063e+34, -6.123599675587524e+34, -6.124183013159577e+34, -6.124819040471965e+34, -6.125507697709835e+34, -6.12624892368844e+34, -6.127042655857075e+34, -6.127888830310634e+34, -6.12878738179792e+34, -6.129738243729092e+34, -6.130741348181755e+34, -6.131796625911363e+34, -6.13290400636596e+34, -6.134063417684143e+34, -6.135274786710563e+34, -6.1365380390095405e+34, -6.137853098863988e+34, -6.139219889299269e+34, -6.140638332081873e+34, -6.142108347735638e+34, -6.143629855551583e+34, -6.1452027735982425e+34, -6.1468270187322685e+34, -6.148502506612984e+34, -6.1502291517098865e+34, -6.152006867315918e+34, -6.153835565561286e+34, -6.155715157422126e+34, -6.157645552735059e+34, -6.1596266602105395e+34, -6.16165838744243e+34, -6.163740640923547e+34, -6.165873326057335e+34, -6.168056347171458e+34, -6.170289607531564e+34, -6.172573009352546e+34, -6.174906453816134e+34, -6.177289841080184e+34, -6.179723070295683e+34, -6.182206039620479e+34, -6.184738646229708e+34, -6.187320786334874e+34, -6.1899523551991055e+34, -6.192633247144422e+34, -6.1953633555720995e+34, -6.198142572978148e+34, -6.200970790962748e+34, -6.203847900251471e+34, -6.2067737907058895e+34, -6.209748351339604e+34, -6.212771470334849e+34, -6.215843035053986e+34, -6.218962932061776e+34, -6.222131047134119e+34, -6.225347265275535e+34, -6.22861147073367e+34, -6.231923547020228e+34, -6.235283376918153e+34, -6.238690842502338e+34, -6.24214582515607e+34, -6.245648205583947e+34, -6.249197863827203e+34, -6.2527946792837595e+34, -6.256438530719283e+34, -6.260129296283454e+34, -6.263866853529802e+34, -6.267651079426851e+34, -6.27148185037616e+34, -6.275359042226999e+34, -6.279282530293036e+34, -6.283252189366766e+34, -6.287267893737475e+34, -6.29132951720432e+34, -6.295436933094845e+34, -6.299590014275928e+34, -6.303788633173422e+34, -6.308032661787893e+34, -6.3123219717077745e+34, -6.316656434124654e+34, -6.321035919849288e+34, -6.325460299330547e+34, -6.329929442662983e+34, -6.334443219609987e+34, -6.33900149961071e+34, -6.343604151804555e+34, -6.348251045036749e+34, -6.352942047880618e+34, -6.35767702864694e+34, -6.362455855400273e+34, -6.367278395976886e+34, -6.372144517992411e+34, -6.37705408886276e+34, -6.382006975813571e+34, -6.387003045898946e+34, -6.392042166011984e+34, -6.397124202895973e+34, -6.402249023165057e+34, -6.4074164933147135e+34, -6.412626479733529e+34, -6.417878848716104e+34, -6.423173466481674e+34, -6.428510199183081e+34, -6.433888912916386e+34, -6.439309473742742e+34, -6.444771747691354e+34, -6.45027560078037e+34, -6.455820899021474e+34, -6.4614075084383275e+34, -6.467035295075171e+34, -6.472704125009914e+34, -6.478413864366136e+34, -6.484164379322957e+34, -6.489955536127346e+34, -6.49578720110804e+34, -6.501659240681569e+34, -6.507571521365989e+34, -6.513523909791822e+34, -6.519516272712368e+34, -6.525548477014321e+34, -6.531620389724067e+34, -6.5377318780243305e+34, -6.543882809259912e+34, -6.550073050947756e+34, -6.556302470785903e+34, -6.562570936666013e+34, -6.568878316676289e+34, -6.575224479117559e+34, -6.581609292507571e+34, -6.588032625589746e+34, -6.594494347343291e+34, -6.600994326989286e+34, -6.607532434002333e+34, -6.614108538113964e+34, -6.620722509321197e+34, -6.627374217896845e+34, -6.634063534394344e+34, -6.64079032965606e+34, -6.647554474817715e+34, -6.654355841318594e+34, -6.661194300905516e+34, -6.668069725638378e+34, -6.674981987900086e+34, -6.681930960399595e+34, -6.688916516178726e+34, -6.695938528617863e+34, -6.70299687143907e+34, -6.710091418712224e+34, -6.71722204486625e+34, -6.724388624683874e+34, -6.73159103331193e+34, -6.738829146267013e+34, -6.746102839438261e+34, -6.753411989088095e+34, -6.760756471862352e+34, -6.768136164789208e+34, -6.775550945288492e+34, -6.783000691166312e+34, -6.790485280627215e+34, -6.7980045922734405e+34, -6.805558505107858e+34, -6.813146898537262e+34, -6.820769652375491e+34, -6.828426646845404e+34, -6.836117762581764e+34, -6.843842880634246e+34, -6.851601882467283e+34, -6.859394649962965e+34, -6.867221065426213e+34, -6.875081011580638e+34, -6.882974371574011e+34, -6.8909010289803865e+34, -6.89886086779773e+34, -6.906853772450906e+34, -6.914879627796322e+34, -6.9229383191161295e+34, -6.931029732123561e+34, -6.939153752965627e+34, -6.947310268213971e+34, -6.955499164878401e+34, -6.963720330398303e+34, -6.971973652644484e+34, -6.980259019924212e+34, -6.988576320972497e+34, -6.996925444961751e+34, -7.005306281493914e+34, -7.013718720604301e+34, -7.022162652761167e+34, -7.030637968863647e+34, -7.039144560244048e+34, -7.047682318662668e+34, -7.056251136312076e+34, -7.064850905814274e+34, -7.07348152022208e+34, -7.0821428730131185e+34, -7.0908348580951895e+34, -7.099557369801674e+34, -7.108310302891895e+34, -7.117093552551763e+34, -7.125907014387794e+34, -7.134750584431497e+34, -7.143624159136078e+34, -7.152527635373868e+34, -7.1614609104396915e+34, -7.170423882044517e+34, -7.1794164483173e+34, -7.188438507804522e+34, -7.197489959464245e+34, -7.206570702672509e+34, -7.215680637214908e+34, -7.224819663290293e+34, -7.233987681507224e+34, -7.243184592883837e+34, -7.252410298846089e+34, -7.261664701226478e+34, -7.270947702262931e+34, -7.280259204599917e+34, -7.289599111283591e+34, -7.298967325765034e+34, -7.308363751891623e+34, -7.317788293919358e+34, -7.327240856497158e+34, -7.336721344675065e+34, -7.346229663903163e+34, -7.355765720025327e+34, -7.365329419283397e+34, -7.374920668315884e+34, -7.3845393741555725e+34, -7.394185444230213e+34, -7.4038587863597e+34, -7.413559308762173e+34, -7.423286920044802e+34, -7.433041529210482e+34, -7.442823045653302e+34, -7.452631379163563e+34, -7.462466439919697e+34, -7.47232813849752e+34, -7.482216385863414e+34, -7.492131093377681e+34, -7.502072172795862e+34, -7.512039536265257e+34, -7.522033096330597e+34, -7.532052765931351e+34, -7.5420984584000425e+34, -7.552170087473013e+34, -7.562267567279258e+34, -7.5723908123478355e+34, -7.582539737613517e+34, -7.592714258406118e+34, -7.602914290461499e+34, -7.6131397499228575e+34, -7.623390553336997e+34, -7.633666617662452e+34, -7.643967860265295e+34, -7.654294198927002e+34, -7.664645551843087e+34, -7.675021837626753e+34, -7.685422975311251e+34, -7.695848884352103e+34, -7.706299484630154e+34, -7.716774696457772e+34, -7.727274440576024e+34, -7.737798638160584e+34, -7.74834721082763e+34, -7.758920080634672e+34, -7.769517170084335e+34, -7.780138402128315e+34, -7.7907837001715875e+34, -7.8014529880790305e+34, -7.812146190176075e+34, -7.822863231253828e+34, -7.833604036577135e+34, -7.844368531882295e+34, -7.855156643391386e+34, -7.865968297808799e+34, -7.876803422331256e+34, -7.887661944653149e+34, -7.898543792969367e+34, -7.909448895983362e+34, -7.920377182913543e+34, -7.931328583495119e+34, -7.9423030279925455e+34, -7.9533004472015865e+34, -7.964320772455077e+34, -7.9753639356325745e+34, -7.986429869167879e+34, -7.997518506051876e+34, -8.008629779841207e+34, -8.019763624667778e+34, -8.03091997524538e+34, -8.042098766874928e+34, -8.05329993545484e+34, -8.064523417485856e+34, -8.075769150084419e+34, -8.087037070985323e+34, -8.098327118552417e+34, -8.109639231786694e+34, -8.120973350338062e+34, -8.132329414507478e+34, -8.1437073652603335e+34, -8.155107144236089e+34, -8.166528693754894e+34, -8.177971956828501e+34, -8.189436877168914e+34, -8.200923399199255e+34, -8.212431468061608e+34, -8.223961029627269e+34, -8.235512030508739e+34, -8.247084418066921e+34, -8.258678140422452e+34, -8.270293146465245e+34, -8.281929385869769e+34, -8.293586809097044e+34, -8.305265367411955e+34, -8.316965012891218e+34, -8.328685698435576e+34, -8.3404273777772e+34, -8.352190005498756e+34, -8.36397353703448e+34, -8.375777928687375e+34, -8.387603137641188e+34, -8.399449121965967e+34, -8.411315840638514e+34, -8.42320325354375e+34, -8.435111321494735e+34, -8.447040006241017e+34, -8.45898927047753e+34, -8.470959077861808e+34, -8.482949393022321e+34, -8.494960181570073e+34, -8.506991410116402e+34, -8.519043046273721e+34, -8.5311150586778e+34, -8.543207416996747e+34, -8.5553200919432e+34, -8.567453055280348e+34, -8.57960627984705e+34, -8.5917797395566e+34, -8.603973409418758e+34, -8.616187265546064e+34, -8.62842128516964e+34, -8.640675446646829e+34, -8.652949729479386e+34, -8.665244114321971e+34, -8.677558582993738e+34, -8.689893118492292e+34, -8.70224770500756e+34, -8.714622327927842e+34, -8.72701697385796e+34, -8.739431630630435e+34, -8.751866287312583e+34, -8.764320934225042e+34, -8.776795562948146e+34, -8.789290166339874e+34, -8.801804738540006e+34, -8.814339274987993e+34, -8.826893772432057e+34, -8.83946822894197e+34, -8.852062643919271e+34, -8.864677018106988e+34, -8.877311353607445e+34, -8.889965653884444e+34, -8.902639923781932e+34, -8.915334169531787e+34, -8.928048398763024e+34, -8.940782620517977e+34, -8.95353684525463e+34, -8.966311084865635e+34, -8.979105352685014e+34, -8.99191966349556e+34, -9.004754033542347e+34, -9.01760848054434e+34, -9.030483023699079e+34, -9.04337768369541e+34, -9.056292482722093e+34, -9.069227444478456e+34, -9.082182594180212e+34, -9.095157958572983e+34, -9.108153565936667e+34, -9.12116944609519e+34, -9.134205630426397e+34, -9.14726215186893e+34, -9.160339044927502e+34, -9.173436345686367e+34, -9.186554091813261e+34, -9.199692322563164e+34, -9.212851078791997e+34, -9.226030402960675e+34, -9.239230339136995e+34, -9.252450933009845e+34, -9.265692231888948e+34, -9.278954284713983e+34, -9.292237142057555e+34, -9.305540856130222e+34, -9.31886548079102e+34, -9.332211071544227e+34, -9.34557768554622e+34, -9.358965381611725e+34, -9.37237422021578e+34, -9.385804263497847e+34, -9.399255575264529e+34, -9.412728220989213e+34, -9.426222267821968e+34, -9.439737784583281e+34, -9.453274841773039e+34, -9.466833511563755e+34, -9.480413867812621e+34, -9.494015986050667e+34, -9.50763994349177e+34, -9.521285819029954e+34, -9.534953693235315e+34, -9.548643648362722e+34, -9.562355768338451e+34, -9.576090138768497e+34, -9.589846846934085e+34, -9.60362598178767e+34, -9.61742763395077e+34, -9.631251895715553e+34, -9.645098861032311e+34, -9.65896862551567e+34, -9.672861286437354e+34, -9.686776942715087e+34, -9.700715694917351e+34, -9.714677645252957e+34, -9.728662897565218e+34, -9.742671557326745e+34, -9.75670373163455e+34, -9.770759529198013e+34, -9.784839060339268e+34, -9.798942436977003e+34, -9.813069772622524e+34, -9.827221182373042e+34, -9.841396782897294e+34, -9.855596692428834e+34, -9.869821030756559e+34, -9.884069919214744e+34, -9.89834348066654e+34, -9.912641839501403e+34, -9.926965121615165e+34, -9.941313454403569e+34, -9.95568696674557e+34, -9.970085788990444e+34, -9.98451005294863e+34, -9.998959891871984e+34, -1.0013435440440992e+35, -1.0027936834750929e+35, -1.0042464212295667e+35, -1.0057017711950067e+35, -1.0071597473955996e+35, -1.0086203639905133e+35, -1.0100836352718256e+35, -1.0115495756630002e+35, -1.0130181997172507e+35, -1.0144895221152256e+35, -1.015963557663271e+35, -1.017440321291617e+35, -1.0189198280518718e+35, -1.020402093115667e+35, -1.0218871317717283e+35, -1.0233749594245077e+35, -1.024865591591261e+35, -1.0263590438999874e+35, -1.0278553320878353e+35, -1.029354471997463e+35, -1.030856479575657e+35, -1.0323613708704332e+35, -1.0338691620288384e+35, -1.0353798692939461e+35, -1.036893509002654e+35, -1.0384100975828764e+35, -1.0399296515509461e+35, -1.0414521875089645e+35, -1.0429777221417515e+35, -1.0445062722143938e+35, -1.0460378545690213e+35, -1.0475724861222198e+35, -1.049110183861752e+35, -1.0506509648440488e+35, -1.0521948461906071e+35, -1.053741845085326e+35, -1.0552919787711254e+35, -1.056845264547129e+35, -1.0584017197651896e+35, -1.0599613618263754e+35, -1.0615242081784225e+35, -1.0630902763114959e+35, -1.0646595837558889e+35, -1.0662321480775209e+35, -1.0678079868754696e+35, -1.0693871177774935e+35, -1.0709695584376504e+35, -1.072555326531531e+35, -1.07414443975377e+35, -1.0757369158135458e+35, -1.0773327724315889e+35, -1.07893202733589e+35, -1.0805346982583842e+35, -1.0821408029311307e+35, -1.0837503590822043e+35, -1.085363384432426e+35, -1.0869798966908087e+35, -1.0885999135512873e+35, -1.0902234526882834e+35, -1.0918505317533434e+35, -1.0934811683705058e+35, -1.095115380132766e+35, -1.0967531845978583e+35, -1.0983945992840632e+35, -1.1000396416667212e+35, -1.1016883291733622e+35, -1.1033406791797635e+35, -1.1049967090062302e+35, -1.1066564359133154e+35, -1.1083198770970051e+35, -1.109987049685191e+35, -1.1116579707332253e+35, -1.1133326572197666e+35, -1.1150111260424174e+35, -1.1166933940132134e+35, -1.1183794778548796e+35, -1.1200693941962979e+35, -1.1217631595678218e+35, -1.1234607903975101e+35, -1.1251623030067216e+35, -1.1268677136052925e+35, -1.1285770382880156e+35, -1.1302902930294405e+35, -1.1320074936799486e+35, -1.1337286559616928e+35, -1.1354537954634888e+35, -1.13718292763735e+35, -1.1389160677932794e+35, -1.140653231095502e+35, -1.142394432558151e+35, -1.1441396870403852e+35, -1.1458890092426352e+35, -1.1476424137018476e+35, -1.1493999147876791e+35, -1.151161526697496e+35, -1.1529272634527688e+35, -1.1546971388944797e+35, -1.156471166678622e+35, -1.15824936027257e+35, -1.1600317329501694e+35, -1.161818297788114e+35, -1.1636090676612013e+35, -1.1654040552388251e+35, -1.1672032729801224e+35, -1.1690067331303251e+35, -1.1708144477167994e+35, -1.172626428544162e+35, -1.1744426871914304e+35, -1.17626323500713e+35, -1.1780880831055821e+35, -1.1799172423629345e+35, -1.181750723413759e+35, -1.1835885366460816e+35, -1.1854306921989588e+35, -1.1872771999573696e+35, -1.1891280695494582e+35, -1.1909833103424955e+35, -1.1928429314389502e+35, -1.194706941673416e+35, -1.1965753496084746e+35, -1.1984481635320822e+35, -1.2003253914528123e+35, -1.2022070410978353e+35, -1.2040931199084348e+35, -1.2059836350376804e+35, -1.2078785933464935e+35, -1.20977800140058e+35, -1.2116818654678464e+35, -1.213590191514598e+35, -1.2155029852031419e+35, -1.2174202518887107e+35, -1.2193419966164783e+35, -1.2212682241188855e+35, -1.2231989388127914e+35, -1.2251341447971498e+35, -1.2270738458502422e+35, -1.2290180454270288e+35, -1.230966746656968e+35, -1.2329199523418808e+35, -1.2348776649528293e+35, -1.2368398866291605e+35, -1.2388066191753679e+35, -1.2407778640596706e+35, -1.242753622412035e+35, -1.244733895021668e+35, -1.2467186823364096e+35, -1.2487079844600921e+35, -1.250701801151509e+35, -1.252700131822537e+35, -1.2547029755373203e+35, -1.2567103310100799e+35, -1.2587221966047646e+35, -1.2607385703335444e+35, -1.2627594498555997e+35, -1.2647848324770039e+35, -1.2668147151487515e+35, -1.268849094467094e+35, -1.2708879666722142e+35, -1.2729313276479942e+35, -1.2749791729219088e+35, -1.2770314976640677e+35, -1.2790882966872977e+35, -1.2811495644475106e+35, -1.2832152950429224e+35, -1.2852854822148202e+35, -1.287360119347401e+35, -1.2894391994681855e+35, -1.2915227152488527e+35, -1.293610659005192e+35, -1.2957030226981121e+35, -1.2977997979345107e+35, -1.2999009759679001e+35, -1.3020065476996486e+35, -1.3041165036801117e+35, -1.3062308341094902e+35, -1.3083495288397003e+35, -1.3104725773757638e+35, -1.312599968876875e+35, -1.3147316921585768e+35, -1.3168677356946615e+35, -1.3190080876188434e+35, -1.3211527357267457e+35, -1.323301667478136e+35, -1.3254548699996292e+35, -1.3276123300862903e+35, -1.3297740342049088e+35, -1.3319399684960303e+35, -1.3341101187771201e+35, -1.336284470545238e+35, -1.3384630089801306e+35, -1.3406457189470624e+35, -1.3428325850004276e+35, -1.3450235913864525e+35, -1.347218722047512e+35, -1.3494179606249789e+35, -1.3516212904630594e+35, -1.353828694612623e+35, -1.3560401558351729e+35, -1.3582556566068053e+35, -1.3604751791220565e+35, -1.3626987052985748e+35, -1.364926216781076e+35, -1.3671576949458809e+35, -1.3693931209057175e+35, -1.3716324755140357e+35, -1.373875739369958e+35, -1.3761228928228868e+35, -1.3783739159778541e+35, -1.3806287887004567e+35, -1.3828874906216377e+35, -1.385150001143725e+35, -1.3874162994449425e+35, -1.3896863644853716e+35, -1.3919601750126446e+35, -1.3942377095672326e+35, -1.3965189464881997e+35, -1.398803863919306e+35, -1.4010924398146952e+35, -1.4033846519449888e+35, -1.4056804779035227e+35, -1.4079798951119424e+35, -1.4102828808275368e+35, -1.4125894121486781e+35, -1.4148994660211352e+35, -1.4172130192453108e+35, -1.4195300484823346e+35, -1.421850530260788e+35, -1.424174440983397e+35, -1.4265017569336995e+35, -1.4288324542832732e+35, -1.431166509097924e+35, -1.4335038973454948e+35, -1.4358445949022254e+35, -1.4381885775601614e+35, -1.4405358210342322e+35, -1.4428863009689723e+35, -1.4452399929468507e+35, -1.4475968724942153e+35, -1.4499569150894753e+35, -1.4523200961701558e+35, -1.4546863911405168e+35, -1.4570557753782978e+35, -1.4594282242431862e+35, -1.4618037130834353e+35, -1.4641822172438768e+35, -1.4665637120731655e+35, -1.468948172931604e+35, -1.4713355751982036e+35, -1.4737258942788657e+35, -1.476119105613894e+35, -1.478515184685013e+35, -1.4809141070236098e+35, -1.4833158482180189e+35, -1.4857203839213384e+35, -1.4881276898585959e+35, -1.4905377418350161e+35, -1.492950515742911e+35, -1.4953659875698974e+35, -1.4977841334058178e+35, -1.5002049294504646e+35, -1.502628352021704e+35, -1.5050543775619238e+35, -1.5074829826461207e+35, -1.5099141439892736e+35, -1.5123478384534799e+35, -1.5147840430556201e+35, -1.5172227349741345e+35, -1.519663891556886e+35, -1.5221074903278855e+35, -1.5245535089944604e+35, -1.5270019254549009e+35, -1.529452717804448e+35, -1.5319058643434111e+35, -1.5343613435830296e+35, -1.5368191342529036e+35, -1.5392792153076278e+35, -1.5417415659332767e+35, -1.5442061655540563e+35, -1.5466729938389707e+35, -1.5491420307080832e+35, -1.5516132563387728e+35, -1.5540866511719444e+35, -1.5565621959184296e+35, -1.5590398715647131e+35, -1.5615196593788594e+35, -1.5640015409165909e+35, -1.5664854980267401e+35, -1.5689715128568665e+35, -1.5714595678587165e+35, -1.57394964579363e+35, -1.5764417297378865e+35, -1.5789358030873047e+35, -1.5814318495629863e+35, -1.583929853215232e+35, -1.5864297984292466e+35, -1.5889316699285954e+35, -1.5914354527806763e+35, -1.5939411324001333e+35, -1.5964486945535273e+35, -1.598958125362974e+35, -1.6014694113101374e+35, -1.6039825392398773e+35, -1.6064974963635617e+35, -1.6090142702629504e+35, -1.6115328488921877e+35, -1.6140532205823176e+35, -1.6165753740430019e+35, -1.6190992983656035e+35, -1.6216249830254864e+35, -1.6241524178848596e+35, -1.6266815931938731e+35, -1.6292124995937072e+35, -1.6317451281173959e+35, -1.6342794701921031e+35, -1.6368155176399296e+35, -1.6393532626795086e+35, -1.6418926979269741e+35, -1.6444338163963974e+35, -1.6469766115007937e+35, -1.6495210770523293e+35, -1.6520672072626464e+35, -1.654614996742675e+35, -1.6571644405026371e+35, -1.659715533951546e+35, -1.662268272896489e+35, -1.6648226535421348e+35, -1.667378672489142e+35, -1.669936326733496e+35, -1.6724956136645847e+35, -1.675056531063871e+35, -1.6776190771029122e+35, -1.680183250341151e+35, -1.682749049723287e+35, -1.6853164745777128e+35, -1.687885524612577e+35, -1.690456199913721e+35, -1.693028500940761e+35, -1.6956024285244146e+35, -1.6981779838620514e+35, -1.7007551685145584e+35, -1.7033339844016138e+35, -1.7059144337978705e+35, -1.708496519328224e+35, -1.7110802439629292e+35, -1.7136656110127396e+35, -1.716252624123764e+35, -1.7188412872719625e+35, -1.7214316047574468e+35, -1.724023581198899e+35, -1.726617221526936e+35, -1.7292125309781553e+35, -1.7318095150885446e+35, -1.7344081796868837e+35, -1.7370085308872123e+35, -1.7396105750824253e+35, -1.742214318936092e+35, -1.744819769375292e+35, -1.7474269335827592e+35, -1.7500358189884348e+35, -1.7526464332614095e+35, -1.755258784301746e+35, -1.757872880231271e+35, -1.7604887293848503e+35, -1.763106340301411e+35, -1.7657257217141686e+35, -1.768346882541821e+35, -1.77096983187803e+35, -1.7735945789821874e+35, -1.7762211332684727e+35, -1.778849504296222e+35, -1.7814797017592344e+35, -1.784111735474563e+35, -1.7867456153723115e+35, -1.7893813514841417e+35, -1.7920189539319164e+35, -1.7946584329164813e+35, -1.7972997987057224e+35, -1.7999430616227657e+35, -1.8025882320342806e+35, -1.8052353203382008e+35, -1.807884336950904e+35, -1.810535292295333e+35, -1.813188196787715e+35, -1.8158430608255038e+35, -1.818499894773382e+35, -1.821158708950831e+35, -1.8238195136188264e+35, -1.8264823189658376e+35, -1.8291471350948415e+35, -1.8318139720091638e+35, -1.8344828395987054e+35, -1.8371537476262404e+35, -1.8398267057127227e+35, -1.8425017233235567e+35, -1.845178809753881e+35, -1.847857974114274e+35, -1.8505392253159513e+35, -1.8532225720562558e+35, -1.8559080228038964e+35, -1.8585955857832843e+35, -1.8612852689608382e+35, -1.863977080028498e+35, -1.866671026389457e+35, -1.86936711514271e+35, -1.8720653530671982e+35, -1.874765746607013e+35, -1.877468301855379e+35, -1.880173024539771e+35, -1.8828799200052673e+35, -1.8855889931998924e+35, -1.8883002486583717e+35, -1.8910136904860468e+35, -1.8937293223439298e+35, -1.8964471474321692e+35, -1.899167168474134e+35, -1.901889387701145e+35, -1.9046138068358442e+35, -1.9073404270764875e+35, -1.910069249081257e+35, -1.9128002729519684e+35, -1.9155334982180958e+35, -1.9182689238212885e+35, -1.9210065480986884e+35, -1.923746368767824e+35, -1.9264883829099052e+35, -1.92923258695459e+35, -1.9319789766637118e+35, -1.9347275471158713e+35, -1.9374782926903663e+35, -1.9402312070511365e+35, -1.9429862831321385e+35, -1.9457435131207338e+35, -1.948502888442529e+35, -1.9512643997462135e+35, -1.9540280368873413e+35, -1.9567937889140406e+35, -1.9595616440509357e+35, -1.9623315896844668e+35, -1.9651036123476284e+35, -1.9678776977053337e+35, -1.9706538305390106e+35, -1.973431994732672e+35, -1.9762121732577195e+35, -1.9789943481582288e+35, -1.9817785005376803e+35, -1.984564610543794e+35, -1.9873526573546815e+35, -1.9901426191648405e+35, -1.9929344731718352e+35, -1.995728195561939e+35, -1.9985237614973838e+35, -2.001321145102853e+35, -2.0041203194518565e+35, -2.0069212565544387e+35, -2.009723927344152e+35, -2.0125283016653855e+35, -2.0153343482611823e+35, -2.018142034760707e+35, -2.020951327667786e+35, -2.023762192348585e+35, -2.0265745930201644e+35, -2.0293884927394793e+35, -2.0322038533919126e+35, -2.03502063568012e+35, -2.0378387991140396e+35, -2.040658301999618e+35, -2.0434791014295487e+35, -2.0463011532722e+35, -2.0491244121627683e+35, -2.0519488314937327e+35, -2.0547743634047174e+35, -2.0576009587750374e+35, -2.0604285672134754e+35, -2.0632571370511196e+35, -2.0660866153324387e+35, -2.068916947807762e+35, -2.0717480789253327e+35, -2.0745799518245535e+35, -2.0774125083285022e+35, -2.080245688937408e+35, -2.0830794328219897e+35, -2.085913677817661e+35, -2.0887483604185823e+35, -2.0915834157718583e+35, -2.094418777672624e+35, -2.0972543785589753e+35, -2.1000901495075943e+35, -2.102926020229216e+35, -2.1057619190645075e+35, -2.1085977729808872e+35, -2.1114335075689002e+35, -2.1142690470395197e+35, -2.1171043142207297e+35, -2.119939230556317e+35, -2.1227737161030457e+35, -2.1256076895290016e+35, -2.1284410681131065e+35, -2.1312737677426984e+35, -2.134105702914476e+35, -2.1369367867331608e+35, -2.1397669309116887e+35, -2.1425960457719066e+35, -2.1454240402450346e+35, -2.1482508218730264e+35, -2.1510762968095808e+35, -2.1539003698224694e+35, -2.1567229442951696e+35, -2.159543922229892e+35, -2.162363204249622e+35, -2.1651806896022322e+35, -2.167996276163459e+35, -2.1708098604408335e+35, -2.173621337578366e+35, -2.176430601360384e+35, -2.179237544217553e+35, -2.182042057231261e+35, -2.1848440301395077e+35, -2.1876433513437077e+35, -2.1904399079136965e+35, -2.1932335855957132e+35, -2.1960242688188915e+35, -2.1988118407026083e+35, -2.2015961830648966e+35, -2.204377176429693e+35, -2.207154700036015e+35, -2.2099286318465754e+35, -2.212698848556921e+35, -2.215465225605004e+35, -2.218227637180963e+35, -2.220985956237528e+35, -2.2237400545003848e+35, -2.2264898024789352e+35, -2.229235069477959e+35, -2.231975723608676e+35, -2.2347116318010202e+35, -2.23744265981537e+35, -2.2401686722558045e+35, -2.242889532582032e+35, -2.245605103123262e+35, -2.2483152450918194e+35, -2.2510198185959324e+35, -2.2537186826551486e+35, -2.256411695213896e+35, -2.259098713156474e+35, -2.2617795923221803e+35, -2.2644541875204587e+35, -2.2671223525467315e+35, -2.269783940197952e+35, -2.2724388022894513e+35, -2.2750867896707767e+35, -2.2777277522428338e+35, -2.2803615389749328e+35, -2.2829879979218684e+35, -2.285606976241793e+35, -2.2882183202139044e+35, -2.2908218752566884e+35, -2.2934174859461964e+35, -2.2960049960345556e+35, -2.2985842484694417e+35, -2.3011550854121984e+35, -2.303717348258142e+35, -2.3062708776555918e+35, -2.3088155135260993e+35, -2.3113510950840043e+35, -2.313877460857049e+35, -2.3163944487069792e+35, -2.3189018958498e+35, -2.32139963887698e+35, -2.323887513776683e+35, -2.326365355954694e+35, -2.3288330002563537e+35, -2.3312902809875657e+35, -2.3337370319375353e+35, -2.336173086399995e+35, -2.3385982771959377e+35, -2.341012436695757e+35, -2.3434153968418202e+35, -2.345806989171249e+35, -2.348187044838407e+35, -2.350555394638408e+35, -2.3529118690297817e+35, -2.3552562981579043e+35, -2.3575885118784192e+35, -2.359908339780594e+35, -2.3622156112112285e+35, -2.364510155297896e+35, -2.366791800973095e+35, -2.369060376998324e+35, -2.3713157119878193e+35, -2.373557634433048e+35, -2.375785972726114e+35, -2.3780005551851337e+35, -2.3802012100775745e+35, -2.382387765645675e+35, -2.3845600501300545e+35, -2.386717891794806e+35, -2.3888611189518297e+35, -2.3909895599856536e+35, -2.393103043377987e+35, -2.395201397732544e+35, -2.3972844517997433e+35, -2.3993520345015412e+35, -2.401403974956168e+35, -2.4034401025032567e+35, -2.405460246727862e+35, -2.407464237486577e+35, -2.409451904930893e+35, -2.411423079533578e+35, -2.4133775921119535e+35, -2.415315273853914e+35, -2.4172359563423922e+35, -2.4191394715794934e+35, -2.4210256520121486e+35, -2.4228943305562797e+35, -2.4247453406215255e+35, -2.4265785161361243e+35, -2.4283936915711182e+35, -2.4301907019649775e+35, -2.4319693829485393e+35, -2.433729570768552e+35, -2.4354711023125938e+35, -2.437193815133403e+35, -2.4388975474727428e+35, -2.440582138285505e+35, -2.442247427264156e+35, -2.443893254862225e+35, -2.4455194623184882e+35, -2.447125891680452e+35, -2.448712385828121e+35, -2.450278788497669e+35, -2.45182494430494e+35, -2.4533506987683634e+35, -2.454855898332467e+35, -2.4563403903912355e+35, -2.457804023310652e+35, -2.4592466464517978e+35, -2.460668110193605e+35, -2.4620682659552803e+35, -2.4634469662190265e+35, -2.4648040645520425e+35, -2.4661394156292408e+35, -2.4674528752548246e+35, -2.4687443003842664e+35, -2.4700135491463287e+35, -2.4712604808643586e+35, -2.472484956077927e+35, -2.4736868365641176e+35, -2.4748659853585226e+35, -2.476022266776283e+35, -2.477155546433255e+35, -2.4782656912656698e+35, -2.4793525695517538e+35, -2.4804160509313968e+35, -2.4814560064264654e+35, -2.4824723084604707e+35, -2.4834648308788258e+35, -2.4844334489678493e+35, -2.485378039474971e+35, -2.4862984806271267e+35, -2.487194652150304e+35, -2.488066435288069e+35, -2.488913712820745e+35, -2.4897363690834738e+35, -2.490534289984347e+35, -2.4913073630231038e+35, -2.492055477308698e+35, -2.4927785235764916e+35, -2.493476394206798e+35, -2.4941489832414337e+35, -2.494796186401036e+35, -2.495417901102122e+35, -2.49601402647341e+35, -2.49658446337272e+35, -2.4971291144032026e+35, -2.4976478839291896e+35, -2.4981406780924745e+35, -2.4986074048276673e+35, -2.499047973877733e+35, -2.499462296809263e+35, -2.49985028702797e+35, -2.500211859792782e+35, -2.5005469322310996e+35, -2.500855423352961e+35, -2.5011372540651667e+35, -2.5013923471858115e+35, -2.5016206274573492e+35, -2.501822021560622e+35, -2.5019964581285045e+35, -2.5021438677583594e+35, -2.5022641830254578e+35, -2.5023573384956204e+35, -2.5024232707379675e+35, -2.5024619183366215e+35, -2.5024732219036604e+35, -2.5024571240900133e+35, -2.5024135695978764e+35, -2.5023425051921973e+35, -2.5022438797111887e+35, -2.5021176440778287e+35, -2.5019637513110006e+35, -2.501782156535311e+35, -2.501572816992062e+35, -2.501335692049178e+35, -2.5010707432110814e+35, -2.5007779341283592e+35, -2.500457230607595e+35, -2.5001086006205693e+35, -2.4997320143126677e+35, -2.499327444012723e+35, -2.498894864241073e+35, -2.498434251717875e+35, -2.4979455853717773e+35, -2.497428846347464e+35, -2.4968840180132107e+35, -2.4963110859696366e+35, -2.4957100380550897e+35, -2.4950808643545975e+35, -2.494423557205324e+35, -2.4937381112043106e+35, -2.4930245232139498e+35, -2.492282792369073e+35, -2.4915129200822345e+35, -2.490714910049778e+35, -2.4898887682575983e+35, -2.4890345029859574e+35, -2.4881521248148268e+35, -2.4872416466289314e+35, -2.4863030836219937e+35, -2.4853364533016233e+35, -2.4843417754930927e+35, -2.483319072343508e+35, -2.482268368325324e+35, -2.4811896902405676e+35, -2.4800830672228787e+35, -2.478948530741719e+35, -2.477786114604443e+35, -2.4765958549593064e+35, -2.4753777902978474e+35, -2.47413196145642e+35, -2.472858411618831e+35, -2.4715571863175515e+35, -2.470228333435031e+35, -2.4688719032054326e+35, -2.4674879482149498e+35, -2.466076523402834e+35, -2.4646376860618404e+35, -2.463171495838257e+35, -2.461678014732119e+35, -2.4601573070966974e+35, -2.458609439638412e+35, -2.457034481415549e+35, -2.455432503837385e+35, -2.4538035806633177e+35, -2.4521477880007362e+35, -2.4504652043038898e+35, -2.448755910371239e+35, -2.4470199893439842e+35, -2.4452575267026864e+35, -2.4434686102648438e+35, -2.4416533301825565e+35, -2.4398117789376325e+35, -2.437944051339731e+35, -2.4360502445211043e+35, -2.4341304579338918e+35, -2.4321847933444187e+35, -2.4302133548296004e+35, -2.4282162487715583e+35, -2.426193583852522e+35, -2.424145471049428e+35, -2.422072023628233e+35, -2.4199733571381982e+35, -2.41784958940477e+35, -2.415700840524175e+35, -2.4135272328557695e+35, -2.4113288910153413e+35, -2.409105941867521e+35, -2.406858514518454e+35, -2.404586740307356e+35, -2.4022907527986302e+35, -2.399970687773292e+35, -2.3976266832201203e+35, -2.3952588793262554e+35, -2.3928674184684015e+35, -2.3904524452029564e+35, -2.388014106255535e+35, -2.3855525505109647e+35, -2.38306792900311e+35, -2.3805603949035474e+35, -2.3780301035103042e+35, -2.375477212236629e+35, -2.3729018805993914e+35, -2.3703042702062637e+35, -2.3676845447441154e+35, -2.3650428699660417e+35, -2.3623794136780744e+35, -2.3596943457262105e+35, -2.356987837982872e+35, -2.3542600643324178e+35, -2.3515112006576e+35, -2.3487414248246926e+35, -2.3459509166682835e+35, -2.34313985797695e+35, -2.340308432476976e+35, -2.337456825817212e+35, -2.334585225552346e+35, -2.331693821126877e+35, -2.3287828038584937e+35, -2.3258523669208037e+35, -2.322902705325953e+35, -2.3199340159071326e+35, -2.316946497300663e+35, -2.313940349927828e+35, -2.3109157759756255e+35, -2.3078729793788635e+35, -2.3048121658001226e+35, -2.3017335426108607e+35, -2.298637318871029e+35, -2.2955237053095746e+35, -2.2923929143037097e+35, -2.2892451598581986e+35, -2.2860806575841677e+35, -2.2828996246782665e+35, -2.27970227990064e+35, -2.2764888435533462e+35, -2.2732595374576512e+35, -2.2700145849321367e+35, -2.2667542107695412e+35, -2.263478641213894e+35, -2.2601881039369494e+35, -2.2568828280145627e+35, -2.2535630439032848e+35, -2.2502289834153576e+35, -2.2468808796947673e+35, -2.2435189671925487e+35, -2.240143481641396e+35, -2.2367546600307014e+35, -2.233352740581022e+35, -2.2299379627180127e+35, -2.2265105670468253e+35, -2.2230707953251856e+35, -2.2196188904376712e+35, -2.216155096368072e+35, -2.2126796581726625e+35, -2.2091928219534918e+35, -2.2056948348300363e+35, -2.2021859449119906e+35, -2.198666401271061e+35, -2.1951364539131245e+35, -2.1915963537493203e+35, -2.188046352568129e+35, -2.1844867030059966e+35, -2.180917658518689e+35, -2.177339473351728e+35, -2.1737524025117942e+35, -2.1701567017361582e+35, -2.1665526274638287e+35, -2.1629404368047494e+35, -2.159320387510838e+35, -2.1556927379447382e+35, -2.1520577470500607e+35, -2.1484156743202738e+35, -2.1447667797688694e+35, -2.1411113238974867e+35, -2.13744956766589e+35, -2.1337817724602196e+35, -2.1301082000623052e+35, -2.126429112617883e+35, -2.1227447726055657e+35, -2.1190554428049248e+35, -2.1153613862653423e+35, -2.1116628662737675e+35, -2.1079601463235814e+35, -2.104253490082167e+35, -2.100543161358996e+35, -2.096829424074325e+35, -2.0931125422260406e+35, -2.089392779858759e+35, -2.0856704010306707e+35, -2.0819456697821086e+35, -2.0782188501027566e+35, -2.074490205899995e+35, -2.0707600009663574e+35, -2.0670284989473916e+35, -2.063295963309133e+35, -2.059562657306359e+35, -2.0558288439501414e+35, -2.052094785975416e+35, -2.0483607458093926e+35, -2.0446269855388016e+35, -2.0408937668782326e+35, -2.0371613511379418e+35, -2.033429999191809e+35, -2.029699971445593e+35, -2.0259715278048853e+35, -2.0222449276436147e+35, -2.0185204297718778e+35, -2.0147982924045897e+35, -2.0110787731305987e+35, -2.007362128880056e+35, -2.0036486158940664e+35, -1.999938489693269e+35, -1.996232005046444e+35, -1.992529415940292e+35, -1.9888309755479226e+35, -1.9851369361987325e+35, -1.9814475493475422e+35, -1.9777630655445107e+35, -1.9740837344049047e+35, -1.97040980457911e+35, -1.966741523722574e+35, -1.9630791384664265e+35, -1.959422894388036e+35, -1.955773035981361e+35, -1.9521298066282814e+35, -1.9484934485695022e+35, -1.9448642028758767e+35, -1.9412423094202027e+35, -1.9376280068487505e+35, -1.934021532553116e+35, -1.9304231226426982e+35, -1.9268330119174368e+35, -1.9232514338396914e+35, -1.919678620508062e+35, -1.9161148026299698e+35, -1.912560209495275e+35, -1.909015068950324e+35, -1.9054796073713865e+35, -1.901954049639221e+35, -1.898438619113792e+35, -1.894933537608514e+35, -1.8914390253658e+35, -1.8879553010322733e+35, -1.8844825816343413e+35, -1.8810210825544186e+35, -1.8775710175068535e+35, -1.874132598514749e+35, -1.870706035886623e+35, -1.867291538193633e+35, -1.863889312247356e+35, -1.8604995630771716e+35, -1.8571224939084763e+35, -1.8537583061413278e+35, -1.8504071993291948e+35, -1.847069371157565e+35, -1.8437450174242403e+35, -1.8404343320182352e+35, -1.8371375069006794e+35, -1.8338547320842536e+35, -1.8305861956151933e+35, -1.827332083553444e+35, -1.824092579954449e+35, -1.8208678668510476e+35, -1.8176581242355443e+35, -1.8144635300423087e+35, -1.8112842601308346e+35, -1.808120488268248e+35, -1.804972386113632e+35, -1.801840123201719e+35, -1.7987238669273225e+35, -1.7956237825297187e+35, -1.792540033077836e+35, -1.7894727794555547e+35, -1.7864221803477786e+35, -1.783388392226087e+35, -1.7803715693355767e+35, -1.7773718636816427e+35, -1.774389425016864e+35, -1.7714244008294235e+35, -1.7684769363298448e+35, -1.7655471744403742e+35, -1.7626352557834307e+35, -1.7597413186701236e+35, -1.7568654990905397e+35, -1.7540079307027023e+35, -1.7511687448234507e+35, -1.7483480704187787e+35, -1.745546034094242e+35, -1.7427627600872076e+35, -1.739998370257734e+35, -1.737252984080751e+35, -1.73452671863879e+35, -1.731819688614567e+35, -1.7291320062838e+35, -1.7264637815091108e+35, -1.723815121734182e+35, -1.7211861319771336e+35, -1.7185769148257955e+35, -1.7159875704323954e+35, -1.713418196509278e+35, -1.7108688883240617e+35, -1.7083397386960947e+35, -1.705830837992839e+35, -1.703342274126276e+35, -1.700874132550993e+35, -1.698426496260513e+35, -1.6959994457861e+35, -1.6935930591942713e+35, -1.6912074120862367e+35, -1.6888425775958e+35, -1.6864986263896445e+35, -1.6841756266664576e+35, -1.681873644156748e+35, -1.6795927421235883e+35, -1.6773329813633657e+35, -1.6750944202064287e+35, -1.6728771145187006e+35, -1.670681117703388e+35, -1.6685064807030937e+35, -1.6663532520022656e+35, -1.6642214776298473e+35, -1.662111201162221e+35, -1.6600224637272062e+35, -1.6579553040069624e+35, -1.6559097582429454e+35, -1.6538858602398964e+35, -1.6518836413703847e+35, -1.6499031305803688e+35, -1.6479443543944936e+35, -1.646007336921306e+35, -1.6440920998601113e+35, -1.642198662506744e+35, -1.6403270417603889e+35, -1.6384772521308873e+35, -1.6366493057454557e+35, -1.6348432123570658e+35, -1.6330589793515728e+35, -1.6312966117565777e+35, -1.6295561122496902e+35, -1.6278374811676178e+35, -1.626140716514856e+35, -1.6244658139735295e+35, -1.6228127669130929e+35, -1.6211815664002514e+35, -1.6195722012093522e+35, -1.6179846578328305e+35, -1.6164189204923652e+35, -1.6148749711499787e+35, -1.6133527895196711e+35, -1.6118523530784302e+35, -1.6103736370794596e+35, -1.608916614563156e+35, -1.6074812563710503e+35, -1.6060675311573298e+35, -1.6046754054027626e+35, -1.6033048434278313e+35, -1.6019558074064756e+35, -1.600628257379527e+35, -1.599322151269584e+35, -1.5980374448945769e+35, -1.5967740919831584e+35, -1.5955320441891112e+35, -1.5943112511066143e+35, -1.5931116602854858e+35, -1.5919332172470918e+35, -1.5907758654998259e+35, -1.5896395465554953e+35, -1.5885241999454975e+35, -1.5874297632369169e+35, -1.5863561720502678e+35, -1.5853033600751106e+35, -1.5842712590888498e+35, -1.5832597989723156e+35, -1.5822689077288543e+35, -1.5812985115012075e+35, -1.5803485345902434e+35, -1.5794188994723714e+35, -1.5785095268184809e+35, -1.5776203355122334e+35, -1.576751242669039e+35, -1.5759021636548029e+35, -1.575073012105491e+35, -1.5742636999457758e+35, -1.5734741374090239e+35, -1.5727042330567051e+35, -1.5719538937981427e+35, -1.5712230249110555e+35, -1.5705115300607519e+35, -1.5698193113211542e+35, -1.569146269194883e+35, -1.568492302634251e+35, -1.5678573090611365e+35, -1.5672411843886115e+35, -1.5666438230418764e+35, -1.5660651179786292e+35, -1.5655049607115183e+35, -1.5649632413283133e+35, -1.5644398485141866e+35, -1.563934669572562e+35, -1.5634475904478731e+35, -1.5629784957463304e+35, -1.5625272687582235e+35, -1.5620937914799417e+35, -1.561677944636267e+35, -1.5612796077017129e+35, -1.5608986589237639e+35, -1.560534975344779e+35, -1.5601884328241025e+35, -1.5598589060609762e+35, -1.5595462686168407e+35, -1.5592503929378869e+35, -1.5589711503778033e+35, -1.5587084112205337e+35, -1.5584620447026073e+35, -1.558231919036425e+35, -1.5580179014327118e+35, -1.5578198581234919e+35, -1.5576376543850014e+35, -1.5574711545604002e+35, -1.5573202220829586e+35, -1.557184719498391e+35, -1.5570645084885575e+35, -1.5569594498939962e+35, -1.5568694037367817e+35, -1.5567942292435455e+35, -1.5567337848684338e+35, -1.5566879283158912e+35, -1.5566565165637687e+35, -1.5566394058859097e+35, -1.5566364518750882e+35, -1.5566475094659059e+35, -1.5566724329573557e+35, -1.5567110760359937e+35, -1.5567632917978664e+35, -1.5568289327718082e+35, -1.556907850941794e+35, -1.5569998977691687e+35, -1.5571049242158246e+35, -1.5572227807657272e+35, -1.5573533174480024e+35, -1.5574963838584005e+35, -1.557651829182321e+35, -1.5578195022163392e+35, -1.5579992513902526e+35, -1.5581909247890381e+35, -1.5583943701749729e+35, -1.5586094350088939e+35, -1.5588359664720384e+35, -1.5590738114874468e+35, -1.5593228167416107e+35, -1.5595828287055642e+35, -1.55985369365613e+35, -1.5601352576966313e+35, -1.5604273667786528e+35, -1.5607298667218478e+35, -1.5610426032354603e+35, -1.5613654219383102e+35, -1.561698168379528e+35, -1.5620406880584948e+35, -1.5623928264453737e+35, -1.5627544290008877e+35, -1.5631253411960688e+35, -1.5635054085320757e+35, -1.5638944765598096e+35, -1.564292390899184e+35, -1.5646989972582399e+35, -1.5651141414521634e+35, -1.5655376694228527e+35, -1.5659694272566804e+35, -1.566409261203845e+35, -1.5668570176961156e+35, -1.5673125433657128e+35, -1.5677756850630647e+35, -1.5682462898742958e+35, -1.5687242051396653e+35, -1.56920927847061e+35, -1.5697013577675012e+35, -1.5702002912359858e+35, -1.5707059274049513e+35, -1.5712181151424956e+35, -1.5717367036733887e+35, -1.572261542594802e+35, -1.57279248189254e+35, -1.5733293719578597e+35, -1.5738720636021876e+35, -1.574420408074004e+35, -1.5749742570731377e+35, -1.5755334627670607e+35, -1.5760978778052626e+35, -1.5766673553346405e+35, -1.5772417490138018e+35, -1.5778209130278789e+35, -1.5784047021025351e+35, -1.5789929715187196e+35, -1.5795855771258012e+35, -1.5801823753555934e+35, -1.5807832232362972e+35, -1.5813879784052856e+35, -1.581996499122511e+35, -1.5826086442834528e+35, -1.5832242734316372e+35, -1.583843246772156e+35, -1.5844654251823124e+35, -1.585090670225532e+35, -1.5857188441623991e+35, -1.5863498099627798e+35, -1.5869834313169816e+35, -1.5876195726484406e+35, -1.5882580991229227e+35, -1.5888988766617372e+35, -1.5895417719510544e+35, -1.59018665245318e+35, -1.5908333864171941e+35, -1.5914818428887557e+35, -1.5921318917209842e+35, -1.592783403583663e+35, -1.5934362499734074e+35, -1.5940903032232952e+35, -1.5947454365119188e+35, -1.5954015238733813e+35, -1.5960584402053396e+35, -1.5967160612789948e+35, -1.5973742637467726e+35, -1.5980329251519693e+35, -1.5986919239362199e+35, -1.599351139448453e+35, -1.6000104519524837e+35, -1.600669742635099e+35, -1.6013288936139576e+35, -1.6019877879452014e+35, -1.602646309630381e+35, -1.6033043436245693e+35, -1.6039617758431082e+35, -1.6046184931685557e+35, -1.6052743834575426e+35, -1.6059293355478124e+35, -1.6065832392644824e+35, -1.6072359854266047e+35, -1.6078874658534092e+35, -1.608537573370617e+35, -1.6091862018163396e+35, -1.6098332460472758e+35, -1.6104786019438427e+35, -1.611122166416639e+35, -1.6117638374116106e+35, -1.6124035139153735e+35, -1.6130410959605703e+35, -1.6136764846314956e+35, -1.6143095820683627e+35, -1.614940291472814e+35, -1.6155685171130719e+35, -1.6161941643279216e+35, -1.6168171395319047e+35, -1.6174373502197439e+35, -1.6180547049708191e+35, -1.6186691134535065e+35, -1.6192804864292577e+35, -1.6198887357573473e+35, -1.620493774398316e+35, -1.6210955164183137e+35, -1.6216938769931411e+35, -1.6222887724116913e+35, -1.6228801200800742e+35, -1.6234678385251156e+35, -1.6240518473982349e+35, -1.6246320674787701e+35, -1.6252084206772981e+35, -1.6257808300395002e+35, -1.6263492197493196e+35, -1.62691351513188e+35, -1.6274736426574022e+35, -1.6280295299438504e+35, -1.6285811057603766e+35, -1.6291283000299907e+35, -1.6296710438330809e+35, -1.6302092694100736e+35, -1.6307429101646183e+35, -1.631271900665821e+35, -1.6317961766520845e+35, -1.632315675033066e+35, -1.6328303338930061e+35, -1.6333400924930213e+35, -1.633844891274182e+35, -1.6343446718599038e+35, -1.6348393770587062e+35, -1.6353289508669353e+35, -1.6358133384712946e+35, -1.6362924862510624e+35, -1.636766341781258e+35, -1.6372348538348071e+35, -1.6376979723849596e+35, -1.6381556486080122e+35, -1.638607834885783e+35, -1.6390544848078048e+35, -1.639495553174285e+35, -1.63993099599781e+35, -1.6403607705067696e+35, -1.6407848351466153e+35, -1.6412031495833906e+35, -1.6416156747053295e+35, -1.6420223726259892e+35, -1.6424232066860071e+35, -1.6428181414558807e+35, -1.6432071427384143e+35, -1.6435901775709428e+35, -1.6439672142278415e+35, -1.6443382222231559e+35, -1.6447031723126328e+35, -1.6450620364965055e+35, -1.6454147880218921e+35, -1.6457614013850103e+35, -1.646101852333999e+35, -1.6464361178709363e+35, -1.6467641762549704e+35, -1.6470860070042981e+35, -1.6474015908986296e+35, -1.6477109099820774e+35, -1.6480139475654413e+35, -1.6483106882288714e+35, -1.6486011178242506e+35, -1.6488852234781337e+35, -1.6491629935935698e+35, -1.6494344178535926e+35, -1.6496994872234e+35, -1.6499581939528488e+35, -1.6502105315792565e+35, -1.6504564949299683e+35, -1.65069608012521e+35, -1.6509292845804103e+35, -1.651156107009364e+35, -1.6513765474262416e+35, -1.6515906071490781e+35, -1.6517982888017295e+35, -1.6519995963173006e+35, -1.652194534940225e+35, -1.6523831112294721e+35, -1.6525653330610068e+35, -1.65274120963083e+35, -1.6529107514573359e+35, -1.6530739703844892e+35, -1.6532308795843996e+35, -1.6533814935599951e+35, -1.6535258281477277e+35, -1.6536639005208656e+35, -1.6537957291916685e+35, -1.6539213340144479e+35, -1.6540407361879997e+35, -1.6541539582590018e+35, -1.6542610241239494e+35, -1.6543619590324952e+35, -1.654456789590081e+35, -1.6545455437603117e+35, -1.6546282508678524e+35, -1.6547049416012225e+35, -1.6547756480154654e+35, -1.6548404035344825e+35, -1.6548992429536585e+35, -1.6549522024429372e+35, -1.6549993195487883e+35, -1.6550406331972547e+35, -1.6550761836960039e+35, -1.655106012736934e+35, -1.6551301633985902e+35, -1.6551486801488924e+35, -1.655161608846772e+35, -1.655168996744966e+35, -1.655170892492139e+35, -1.6551673461351488e+35, -1.6551584091210554e+35, -1.6551441342991703e+35, -1.6551245759231861e+35, -1.6550997896530312e+35, -1.655069832557012e+35, -1.6550347631132083e+35, -1.6549946412116728e+35, -1.6549495281555157e+35, -1.6548994866632167e+35, -1.6548445808695814e+35, -1.6547848763274162e+35, -1.6547204400087288e+35, -1.6546513403060346e+35, -1.6545776470334577e+35, -1.6544994314278466e+35, -1.6544167661498871e+35, -1.6543297252843998e+35, -1.6542383843416329e+35, -1.6541428202574209e+35, -1.6540431113940094e+35, -1.6539393375402724e+35, -1.6538315799120805e+35, -1.6537199211520744e+35, -1.6536044453302946e+35, -1.6534852379432687e+35, -1.653362385914252e+35, -1.653235977592571e+35, -1.6531061027531334e+35, -1.6529728525956443e+35, -1.6528363197438002e+35, -1.6526965982439584e+35, -1.6525537835642921e+35, -1.6524079725933255e+35, -1.652259263638142e+35, -1.6521077564230118e+35, -1.651953552087227e+35, -1.6517967531830985e+35, -1.6516374636737646e+35, -1.6514757889305708e+35, -1.6513118357307362e+35, -1.6511457122540993e+35, -1.6509775280803447e+35, -1.6508073941856564e+35, -1.6506354229393803e+35, -1.6504617281001689e+35, -1.6502864248121254e+35, -1.650109629601111e+35, -1.6499314603695616e+35, -1.6497520363928108e+35, -1.6495714783139048e+35, -1.6493899081385202e+35, -1.6492074492300199e+35, -1.6490242263034718e+35, -1.648840365420709e+35, -1.648655993983411e+35, -1.6484712407275291e+35, -1.64828623571675e+35, -1.6481011103355602e+35, -1.6479159972821134e+35, -1.6477310305614956e+35, -1.6475463454777786e+35, -1.6473620786264811e+35, -1.6471783678862763e+35, -1.6469953524108666e+35, -1.6468131726203294e+35, -1.6466319701922005e+35, -1.6464518880522673e+35, -1.6462730703654487e+35, -1.646095662525438e+35, -1.6459198111454282e+35, -1.645745664047199e+35, -1.645573370250833e+35, -1.6454030799638228e+35, -1.64523494456987e+35, -1.6450691166169127e+35, -1.6449057498062407e+35, -1.6447449989800318e+35, -1.6445870201082874e+35, -1.6444319702772049e+35, -1.644280007675619e+35, -1.6441312915815036e+35, -1.6439859823488429e+35, -1.643844241393418e+35, -1.643706231178602e+35, -1.6435721152008587e+35, -1.6434420579747165e+35, -1.6433162250178968e+35, -1.6431947828354891e+35, -1.6430778989043992e+35, -1.6429657416569706e+35, -1.6428584804646972e+35, -1.642756285621719e+35]
tiempo_s_1emin5 = [0.0, 8.005538284800001e-07, 1.6011076569600003e-06, 2.4016614854399944e-06, 3.2022153139199845e-06, 4.002769142399974e-06, 4.803322970879964e-06, 5.603876799359954e-06, 6.404430627839944e-06, 7.2049844563199345e-06, 8.005538284799943e-06, 8.806092113279974e-06, 9.606645941760005e-06, 1.0407199770240035e-05, 1.1207753598720066e-05, 1.2008307427200097e-05, 1.2808861255680128e-05, 1.3609415084160158e-05, 1.440996891264019e-05, 1.521052274112022e-05, 1.6011076569600174e-05, 1.6811630398080124e-05, 1.7612184226560073e-05, 1.8412738055040023e-05, 1.9213291883519972e-05, 2.001384571199992e-05, 2.081439954047987e-05, 2.161495336895982e-05, 2.241550719743977e-05, 2.321606102591972e-05, 2.401661485439967e-05, 2.4817168682879618e-05, 2.5617722511359568e-05, 2.6418276339839517e-05, 2.7218830168319466e-05, 2.8019383996799416e-05, 2.8819937825279365e-05, 2.9620491653759315e-05, 3.0421045482239264e-05, 3.1221599310719214e-05, 3.202215313919916e-05, 3.282270696767911e-05, 3.362326079615906e-05, 3.442381462463901e-05, 3.522436845311896e-05, 3.602492228159891e-05, 3.682547611007886e-05, 3.762602993855881e-05, 3.842658376703876e-05, 3.922713759551871e-05, 4.002769142399866e-05, 4.082824525247861e-05, 4.1628799080958556e-05, 4.2429352909438506e-05, 4.3229906737918455e-05, 4.4030460566398404e-05, 4.4831014394878354e-05, 4.56315682233583e-05, 4.643212205183825e-05, 4.72326758803182e-05, 4.803322970879815e-05, 4.88337835372781e-05, 4.963433736575805e-05, 5.0434891194238e-05, 5.123544502271795e-05, 5.20359988511979e-05, 5.283655267967785e-05, 5.36371065081578e-05, 5.443766033663775e-05, 5.5238214165117696e-05, 5.6038767993597646e-05, 5.6839321822077595e-05, 5.7639875650557545e-05, 5.8440429479037494e-05, 5.9240983307517444e-05, 6.004153713599739e-05, 6.084209096447734e-05, 6.164264479295754e-05, 6.244319862143782e-05, 6.324375244991809e-05, 6.404430627839837e-05, 6.484486010687864e-05, 6.564541393535892e-05, 6.644596776383919e-05, 6.724652159231947e-05, 6.804707542079974e-05, 6.884762924928001e-05, 6.964818307776029e-05, 7.044873690624056e-05, 7.124929073472084e-05, 7.204984456320111e-05, 7.285039839168139e-05, 7.365095222016166e-05, 7.445150604864194e-05, 7.525205987712221e-05, 7.605261370560249e-05, 7.685316753408276e-05, 7.765372136256304e-05, 7.845427519104331e-05, 7.925482901952359e-05, 8.005538284800386e-05, 8.085593667648414e-05, 8.165649050496441e-05, 8.245704433344468e-05, 8.325759816192496e-05, 8.405815199040523e-05, 8.485870581888551e-05, 8.565925964736578e-05, 8.645981347584606e-05, 8.726036730432633e-05, 8.806092113280661e-05, 8.886147496128688e-05, 8.966202878976716e-05, 9.046258261824743e-05, 9.12631364467277e-05, 9.206369027520798e-05, 9.286424410368826e-05, 9.366479793216853e-05, 9.44653517606488e-05, 9.526590558912908e-05, 9.606645941760935e-05, 9.686701324608963e-05, 9.76675670745699e-05, 9.846812090305018e-05, 9.926867473153045e-05, 0.00010006922856001073, 0.000100869782388491, 0.00010167033621697128, 0.00010247089004545155, 0.00010327144387393183, 0.0001040719977024121, 0.00010487255153089238, 0.00010567310535937265, 0.00010647365918785293, 0.0001072742130163332, 0.00010807476684481347, 0.00010887532067329375, 0.00010967587450177402, 0.0001104764283302543, 0.00011127698215873457, 0.00011207753598721485, 0.00011287808981569512, 0.0001136786436441754, 0.00011447919747265567, 0.00011527975130113595, 0.00011608030512961622, 0.0001168808589580965, 0.00011768141278657677, 0.00011848196661505705, 0.00011928252044353732, 0.0001200830742720176, 0.00012088362810049787, 0.00012168418192897814, 0.00012248473575745807, 0.0001232852895859377, 0.00012408584341441732, 0.00012488639724289694, 0.00012568695107137656, 0.0001264875048998562, 0.0001272880587283358, 0.00012808861255681544, 0.00012888916638529506, 0.00012968972021377468, 0.0001304902740422543, 0.00013129082787073393, 0.00013209138169921356, 0.00013289193552769318, 0.0001336924893561728, 0.00013449304318465243, 0.00013529359701313205, 0.00013609415084161168, 0.0001368947046700913, 0.00013769525849857093, 0.00013849581232705055, 0.00013929636615553017, 0.0001400969199840098, 0.00014089747381248942, 0.00014169802764096905, 0.00014249858146944867, 0.0001432991352979283, 0.00014409968912640792, 0.00014490024295488754, 0.00014570079678336717, 0.0001465013506118468, 0.00014730190444032642, 0.00014810245826880604, 0.00014890301209728566, 0.0001497035659257653, 0.0001505041197542449, 0.00015130467358272454, 0.00015210522741120416, 0.00015290578123968379, 0.0001537063350681634, 0.00015450688889664303, 0.00015530744272512266, 0.00015610799655360228, 0.0001569085503820819, 0.00015770910421056153, 0.00015850965803904115, 0.00015931021186752078, 0.0001601107656960004, 0.00016091131952448003, 0.00016171187335295965, 0.00016251242718143928, 0.0001633129810099189, 0.00016411353483839852, 0.00016491408866687815, 0.00016571464249535777, 0.0001665151963238374, 0.00016731575015231702, 0.00016811630398079664, 0.00016891685780927627, 0.0001697174116377559, 0.00017051796546623552, 0.00017131851929471514, 0.00017211907312319477, 0.0001729196269516744, 0.000173720180780154, 0.00017452073460863364, 0.00017532128843711326, 0.00017612184226559289, 0.0001769223960940725, 0.00017772294992255213, 0.00017852350375103176, 0.00017932405757951138, 0.000180124611407991, 0.00018092516523647063, 0.00018172571906495026, 0.00018252627289342988, 0.0001833268267219095, 0.00018412738055038913, 0.00018492793437886875, 0.00018572848820734838, 0.000186529042035828, 0.00018732959586430762, 0.00018813014969278725, 0.00018893070352126687, 0.0001897312573497465, 0.00019053181117822612, 0.00019133236500670575, 0.00019213291883518537, 0.000192933472663665, 0.00019373402649214462, 0.00019453458032062424, 0.00019533513414910387, 0.0001961356879775835, 0.00019693624180606311, 0.00019773679563454274, 0.00019853734946302236, 0.000199337903291502, 0.0002001384571199816, 0.00020093901094846124, 0.00020173956477694086, 0.00020254011860542048, 0.0002033406724339001, 0.00020414122626237973, 0.00020494178009085936, 0.00020574233391933898, 0.0002065428877478186, 0.00020734344157629823, 0.00020814399540477785, 0.00020894454923325748, 0.0002097451030617371, 0.00021054565689021673, 0.00021134621071869635, 0.00021214676454717597, 0.0002129473183756556, 0.00021374787220413522, 0.00021454842603261485, 0.00021534897986109447, 0.0002161495336895741, 0.00021695008751805372, 0.00021775064134653334, 0.00021855119517501297, 0.0002193517490034926, 0.00022015230283197222, 0.00022095285666045184, 0.00022175341048893146, 0.0002225539643174111, 0.0002233545181458907, 0.00022415507197437034, 0.00022495562580284996, 0.00022575617963132958, 0.0002265567334598092, 0.00022735728728828883, 0.00022815784111676846, 0.00022895839494524808, 0.0002297589487737277, 0.00023055950260220733, 0.00023136005643068695, 0.00023216061025916658, 0.0002329611640876462, 0.00023376171791612583, 0.00023456227174460545, 0.00023536282557308507, 0.0002361633794015647, 0.00023696393323004432, 0.00023776448705852395, 0.00023856504088700357, 0.0002393655947154832, 0.00024016614854396282, 0.00024096670237244244, 0.00024176725620092207, 0.0002425678100294017, 0.00024336836385788132, 0.0002441689176863609, 0.0002449694715148392, 0.00024577002534331753, 0.00024657057917179586, 0.0002473711330002742, 0.0002481716868287525, 0.0002489722406572308, 0.00024977279448570915, 0.00025057334831418747, 0.0002513739021426658, 0.0002521744559711441, 0.00025297500979962244, 0.00025377556362810076, 0.0002545761174565791, 0.0002553766712850574, 0.00025617722511353573, 0.00025697777894201406, 0.0002577783327704924, 0.0002585788865989707, 0.000259379440427449, 0.00026017999425592735, 0.00026098054808440567, 0.000261781101912884, 0.0002625816557413623, 0.00026338220956984064, 0.00026418276339831896, 0.0002649833172267973, 0.0002657838710552756, 0.00026658442488375393, 0.00026738497871223226, 0.0002681855325407106, 0.0002689860863691889, 0.0002697866401976672, 0.00027058719402614555, 0.00027138774785462387, 0.0002721883016831022, 0.0002729888555115805, 0.00027378940934005884, 0.00027458996316853716, 0.0002753905169970155, 0.0002761910708254938, 0.00027699162465397213, 0.00027779217848245046, 0.0002785927323109288, 0.0002793932861394071, 0.0002801938399678854, 0.00028099439379636375, 0.0002817949476248421, 0.0002825955014533204, 0.0002833960552817987, 0.00028419660911027704, 0.00028499716293875537, 0.0002857977167672337, 0.000286598270595712, 0.00028739882442419033, 0.00028819937825266866, 0.000288999932081147, 0.0002898004859096253, 0.00029060103973810363, 0.00029140159356658195, 0.0002922021473950603, 0.0002930027012235386, 0.0002938032550520169, 0.00029460380888049524, 0.00029540436270897357, 0.0002962049165374519, 0.0002970054703659302, 0.00029780602419440854, 0.00029860657802288686, 0.0002994071318513652, 0.0003002076856798435, 0.00030100823950832183, 0.00030180879333680015, 0.0003026093471652785, 0.0003034099009937568, 0.0003042104548222351, 0.00030501100865071344, 0.00030581156247919177, 0.0003066121163076701, 0.0003074126701361484, 0.00030821322396462674, 0.00030901377779310506, 0.0003098143316215834, 0.0003106148854500617, 0.00031141543927854003, 0.00031221599310701835, 0.0003130165469354967, 0.000313817100763975, 0.0003146176545924533, 0.00031541820842093164, 0.00031621876224940997, 0.0003170193160778883, 0.0003178198699063666, 0.00031862042373484494, 0.00031942097756332326, 0.0003202215313918016, 0.0003210220852202799, 0.00032182263904875823, 0.00032262319287723655, 0.0003234237467057149, 0.0003242243005341932, 0.0003250248543626715, 0.00032582540819114985, 0.00032662596201962817, 0.0003274265158481065, 0.0003282270696765848, 0.00032902762350506314, 0.00032982817733354146, 0.0003306287311620198, 0.0003314292849904981, 0.00033222983881897643, 0.00033303039264745475, 0.0003338309464759331, 0.0003346315003044114, 0.0003354320541328897, 0.00033623260796136805, 0.00033703316178984637, 0.0003378337156183247, 0.000338634269446803, 0.00033943482327528134, 0.00034023537710375966, 0.000341035930932238, 0.0003418364847607163, 0.00034263703858919463, 0.00034343759241767295, 0.0003442381462461513, 0.0003450387000746296, 0.0003458392539031079, 0.00034663980773158625, 0.00034744036156006457, 0.0003482409153885429, 0.0003490414692170212, 0.00034984202304549954, 0.00035064257687397786, 0.0003514431307024562, 0.0003522436845309345, 0.00035304423835941283, 0.00035384479218789115, 0.0003546453460163695, 0.0003554458998448478, 0.0003562464536733261, 0.00035704700750180445, 0.00035784756133028277, 0.0003586481151587611, 0.0003594486689872394, 0.00036024922281571774, 0.00036104977664419606, 0.0003618503304726744, 0.0003626508843011527, 0.00036345143812963103, 0.00036425199195810936, 0.0003650525457865877, 0.000365853099615066, 0.0003666536534435443, 0.00036745420727202265, 0.00036825476110050097, 0.0003690553149289793, 0.0003698558687574576, 0.00037065642258593594, 0.00037145697641441426, 0.0003722575302428926, 0.0003730580840713709, 0.00037385863789984923, 0.00037465919172832756, 0.0003754597455568059, 0.0003762602993852842, 0.0003770608532137625, 0.00037786140704224085, 0.00037866196087071917, 0.0003794625146991975, 0.0003802630685276758, 0.00038106362235615414, 0.00038186417618463246, 0.0003826647300131108, 0.0003834652838415891, 0.00038426583767006743, 0.00038506639149854576, 0.0003858669453270241, 0.0003866674991555024, 0.0003874680529839807, 0.00038826860681245905, 0.00038906916064093737, 0.0003898697144694157, 0.000390670268297894, 0.00039147082212637234, 0.00039227137595485066, 0.000393071929783329, 0.0003938724836118073, 0.00039467303744028563, 0.00039547359126876396, 0.0003962741450972423, 0.0003970746989257206, 0.0003978752527541989, 0.00039867580658267725, 0.00039947636041115557, 0.0004002769142396339, 0.0004010774680681122, 0.00040187802189659054, 0.00040267857572506887, 0.0004034791295535472, 0.0004042796833820255, 0.00040508023721050383, 0.00040588079103898216, 0.0004066813448674605, 0.0004074818986959388, 0.0004082824525244171, 0.00040908300635289545, 0.0004098835601813738, 0.0004106841140098521, 0.0004114846678383304, 0.00041228522166680874, 0.00041308577549528707, 0.0004138863293237654, 0.0004146868831522437, 0.00041548743698072204, 0.00041628799080920036, 0.0004170885446376787, 0.000417889098466157, 0.00041868965229463533, 0.00041949020612311365, 0.000420290759951592, 0.0004210913137800703, 0.0004218918676085486, 0.00042269242143702694, 0.00042349297526550527, 0.0004242935290939836, 0.0004250940829224619, 0.00042589463675094024, 0.00042669519057941856, 0.0004274957444078969, 0.0004282962982363752, 0.00042909685206485353, 0.00042989740589333185, 0.0004306979597218102, 0.0004314985135502885, 0.0004322990673787668, 0.00043309962120724514, 0.00043390017503572347, 0.0004347007288642018, 0.0004355012826926801, 0.00043630183652115844, 0.00043710239034963676, 0.0004379029441781151, 0.0004387034980065934, 0.00043950405183507173, 0.00044030460566355005, 0.0004411051594920284, 0.0004419057133205067, 0.000442706267148985, 0.00044350682097746334, 0.00044430737480594167, 0.00044510792863442, 0.0004459084824628983, 0.00044670903629137664, 0.00044750959011985496, 0.0004483101439483333, 0.0004491106977768116, 0.00044991125160528993, 0.00045071180543376825, 0.0004515123592622466, 0.0004523129130907249, 0.0004531134669192032, 0.00045391402074768155, 0.00045471457457615987, 0.0004555151284046382, 0.0004563156822331165, 0.00045711623606159484, 0.00045791678989007316, 0.0004587173437185515, 0.0004595178975470298, 0.00046031845137550813, 0.00046111900520398645, 0.0004619195590324648, 0.0004627201128609431, 0.0004635206666894214, 0.00046432122051789975, 0.00046512177434637807, 0.0004659223281748564, 0.0004667228820033347, 0.00046752343583181304, 0.00046832398966029136, 0.0004691245434887697, 0.000469925097317248, 0.00047072565114572633, 0.00047152620497420465, 0.000472326758802683, 0.0004731273126311613, 0.0004739278664596396, 0.00047472842028811795, 0.00047552897411659627, 0.0004763295279450746, 0.0004771300817735529, 0.00047793063560203124, 0.00047873118943050956, 0.0004795317432589879, 0.0004803322970874662, 0.00048113285091594453, 0.00048193340474442285, 0.0004827339585729012, 0.0004835345124013795, 0.0004843350662298578, 0.00048513562005833615, 0.00048593617388681447, 0.0004867367277152928, 0.0004875372815437711, 0.0004883378353722495, 0.0004891383892007278, 0.0004899389430292061, 0.0004907394968576845, 0.0004915400506861628, 0.0004923406045146411, 0.0004931411583431194, 0.0004939417121715978, 0.0004947422660000761, 0.0004955428198285544, 0.0004963433736570327, 0.000497143927485511, 0.0004979444813139894, 0.0004987450351424677, 0.000499545588970946, 0.0005003461427994243, 0.0005011466966279027, 0.000501947250456381, 0.0005027478042848593, 0.0005035483581133376, 0.000504348911941816, 0.0005051494657702943, 0.0005059500195987726, 0.0005067505734272509, 0.0005075511272557292, 0.0005083516810842076, 0.0005091522349126859, 0.0005099527887411642, 0.0005107533425696425, 0.0005115538963981209, 0.0005123544502265992, 0.0005131550040550775, 0.0005139555578835558, 0.0005147561117120342, 0.0005155566655405125, 0.0005163572193689908, 0.0005171577731974691, 0.0005179583270259474, 0.0005187588808544258, 0.0005195594346829041, 0.0005203599885113824, 0.0005211605423398607, 0.0005219610961683391, 0.0005227616499968174, 0.0005235622038252957, 0.000524362757653774, 0.0005251633114822524, 0.0005259638653107307, 0.000526764419139209, 0.0005275649729676873, 0.0005283655267961657, 0.000529166080624644, 0.0005299666344531223, 0.0005307671882816006, 0.0005315677421100789, 0.0005323682959385573, 0.0005331688497670356, 0.0005339694035955139, 0.0005347699574239922, 0.0005355705112524706, 0.0005363710650809489, 0.0005371716189094272, 0.0005379721727379055, 0.0005387727265663839, 0.0005395732803948622, 0.0005403738342233405, 0.0005411743880518188, 0.0005419749418802971, 0.0005427754957087755, 0.0005435760495372538, 0.0005443766033657321, 0.0005451771571942104, 0.0005459777110226888, 0.0005467782648511671, 0.0005475788186796454, 0.0005483793725081237, 0.000549179926336602, 0.0005499804801650804, 0.0005507810339935587, 0.000551581587822037, 0.0005523821416505153, 0.0005531826954789937, 0.000553983249307472, 0.0005547838031359503, 0.0005555843569644286, 0.000556384910792907, 0.0005571854646213853, 0.0005579860184498636, 0.0005587865722783419, 0.0005595871261068203, 0.0005603876799352986, 0.0005611882337637769, 0.0005619887875922552, 0.0005627893414207335, 0.0005635898952492119, 0.0005643904490776902, 0.0005651910029061685, 0.0005659915567346468, 0.0005667921105631252, 0.0005675926643916035, 0.0005683932182200818, 0.0005691937720485601, 0.0005699943258770385, 0.0005707948797055168, 0.0005715954335339951, 0.0005723959873624734, 0.0005731965411909517, 0.0005739970950194301, 0.0005747976488479084, 0.0005755982026763867, 0.000576398756504865, 0.0005771993103333434, 0.0005779998641618217, 0.0005788004179903, 0.0005796009718187783, 0.0005804015256472567, 0.000581202079475735, 0.0005820026333042133, 0.0005828031871326916, 0.00058360374096117, 0.0005844042947896483, 0.0005852048486181266, 0.0005860054024466049, 0.0005868059562750832, 0.0005876065101035616, 0.0005884070639320399, 0.0005892076177605182, 0.0005900081715889965, 0.0005908087254174749, 0.0005916092792459532, 0.0005924098330744315, 0.0005932103869029098, 0.0005940109407313881, 0.0005948114945598665, 0.0005956120483883448, 0.0005964126022168231, 0.0005972131560453014, 0.0005980137098737798, 0.0005988142637022581, 0.0005996148175307364, 0.0006004153713592147, 0.0006012159251876931, 0.0006020164790161714, 0.0006028170328446497, 0.000603617586673128, 0.0006044181405016063, 0.0006052186943300847, 0.000606019248158563, 0.0006068198019870413, 0.0006076203558155196, 0.000608420909643998, 0.0006092214634724763, 0.0006100220173009546, 0.0006108225711294329, 0.0006116231249579113, 0.0006124236787863896, 0.0006132242326148679, 0.0006140247864433462, 0.0006148253402718245, 0.0006156258941003029, 0.0006164264479287812, 0.0006172270017572595, 0.0006180275555857378, 0.0006188281094142162, 0.0006196286632426945, 0.0006204292170711728, 0.0006212297708996511, 0.0006220303247281295, 0.0006228308785566078, 0.0006236314323850861, 0.0006244319862135644, 0.0006252325400420427, 0.0006260330938705211, 0.0006268336476989994, 0.0006276342015274777, 0.000628434755355956, 0.0006292353091844344, 0.0006300358630129127, 0.000630836416841391, 0.0006316369706698693, 0.0006324375244983477, 0.000633238078326826, 0.0006340386321553043, 0.0006348391859837826, 0.000635639739812261, 0.0006364402936407393, 0.0006372408474692176, 0.0006380414012976959, 0.0006388419551261742, 0.0006396425089546526, 0.0006404430627831309, 0.0006412436166116092, 0.0006420441704400875, 0.0006428447242685659, 0.0006436452780970442, 0.0006444458319255225, 0.0006452463857540008, 0.0006460469395824791, 0.0006468474934109575, 0.0006476480472394358, 0.0006484486010679141, 0.0006492491548963924, 0.0006500497087248708, 0.0006508502625533491, 0.0006516508163818274, 0.0006524513702103057, 0.0006532519240387841, 0.0006540524778672624, 0.0006548530316957407, 0.000655653585524219, 0.0006564541393526974, 0.0006572546931811757, 0.000658055247009654, 0.0006588558008381323, 0.0006596563546666106, 0.000660456908495089, 0.0006612574623235673, 0.0006620580161520456, 0.0006628585699805239, 0.0006636591238090023, 0.0006644596776374806, 0.0006652602314659589, 0.0006660607852944372, 0.0006668613391229156, 0.0006676618929513939, 0.0006684624467798722, 0.0006692630006083505, 0.0006700635544368288, 0.0006708641082653072, 0.0006716646620937855, 0.0006724652159222638, 0.0006732657697507421, 0.0006740663235792205, 0.0006748668774076988, 0.0006756674312361771, 0.0006764679850646554, 0.0006772685388931338, 0.0006780690927216121, 0.0006788696465500904, 0.0006796702003785687, 0.000680470754207047, 0.0006812713080355254, 0.0006820718618640037, 0.000682872415692482, 0.0006836729695209603, 0.0006844735233494387, 0.000685274077177917, 0.0006860746310063953, 0.0006868751848348736, 0.000687675738663352, 0.0006884762924918303, 0.0006892768463203086, 0.0006900774001487869, 0.0006908779539772652, 0.0006916785078057436, 0.0006924790616342219, 0.0006932796154627002, 0.0006940801692911785, 0.0006948807231196569, 0.0006956812769481352, 0.0006964818307766135, 0.0006972823846050918, 0.0006980829384335702, 0.0006988834922620485, 0.0006996840460905268, 0.0007004845999190051, 0.0007012851537474834, 0.0007020857075759618, 0.0007028862614044401, 0.0007036868152329184, 0.0007044873690613967, 0.0007052879228898751, 0.0007060884767183534, 0.0007068890305468317, 0.00070768958437531, 0.0007084901382037884, 0.0007092906920322667, 0.000710091245860745, 0.0007108917996892233, 0.0007116923535177016, 0.00071249290734618, 0.0007132934611746583, 0.0007140940150031366, 0.0007148945688316149, 0.0007156951226600933, 0.0007164956764885716, 0.0007172962303170499, 0.0007180967841455282, 0.0007188973379740066, 0.0007196978918024849, 0.0007204984456309632, 0.0007212989994594415, 0.0007220995532879198, 0.0007229001071163982, 0.0007237006609448765, 0.0007245012147733548, 0.0007253017686018331, 0.0007261023224303115, 0.0007269028762587898, 0.0007277034300872681, 0.0007285039839157464, 0.0007293045377442248, 0.0007301050915727031, 0.0007309056454011814, 0.0007317061992296597, 0.000732506753058138, 0.0007333073068866164, 0.0007341078607150947, 0.000734908414543573, 0.0007357089683720513, 0.0007365095222005297, 0.000737310076029008, 0.0007381106298574863, 0.0007389111836859646, 0.000739711737514443, 0.0007405122913429213, 0.0007413128451713996, 0.0007421133989998779, 0.0007429139528283562, 0.0007437145066568346, 0.0007445150604853129, 0.0007453156143137912, 0.0007461161681422695, 0.0007469167219707479, 0.0007477172757992262, 0.0007485178296277045, 0.0007493183834561828, 0.0007501189372846612, 0.0007509194911131395, 0.0007517200449416178, 0.0007525205987700961, 0.0007533211525985744, 0.0007541217064270528, 0.0007549222602555311, 0.0007557228140840094, 0.0007565233679124877, 0.0007573239217409661, 0.0007581244755694444, 0.0007589250293979227, 0.000759725583226401, 0.0007605261370548794, 0.0007613266908833577, 0.000762127244711836, 0.0007629277985403143, 0.0007637283523687926, 0.000764528906197271, 0.0007653294600257493, 0.0007661300138542276, 0.0007669305676827059, 0.0007677311215111843, 0.0007685316753396626, 0.0007693322291681409, 0.0007701327829966192, 0.0007709333368250976, 0.0007717338906535759, 0.0007725344444820542, 0.0007733349983105325, 0.0007741355521390108, 0.0007749361059674892, 0.0007757366597959675, 0.0007765372136244458, 0.0007773377674529241, 0.0007781383212814025, 0.0007789388751098808, 0.0007797394289383591, 0.0007805399827668374, 0.0007813405365953158, 0.0007821410904237941, 0.0007829416442522724, 0.0007837421980807507, 0.000784542751909229, 0.0007853433057377074, 0.0007861438595661857, 0.000786944413394664, 0.0007877449672231423, 0.0007885455210516207, 0.000789346074880099, 0.0007901466287085773, 0.0007909471825370556, 0.000791747736365534, 0.0007925482901940123, 0.0007933488440224906, 0.0007941493978509689, 0.0007949499516794473, 0.0007957505055079256, 0.0007965510593364039, 0.0007973516131648822, 0.0007981521669933605, 0.0007989527208218389, 0.0007997532746503172, 0.0008005538284787955, 0.0008013543823072738, 0.0008021549361357522, 0.0008029554899642305, 0.0008037560437927088, 0.0008045565976211871, 0.0008053571514496655, 0.0008061577052781438, 0.0008069582591066221, 0.0008077588129351004, 0.0008085593667635787, 0.0008093599205920571, 0.0008101604744205354, 0.0008109610282490137, 0.000811761582077492, 0.0008125621359059704, 0.0008133626897344487, 0.000814163243562927, 0.0008149637973914053, 0.0008157643512198837, 0.000816564905048362, 0.0008173654588768403, 0.0008181660127053186, 0.0008189665665337969, 0.0008197671203622753, 0.0008205676741907536, 0.0008213682280192319, 0.0008221687818477102, 0.0008229693356761886, 0.0008237698895046669, 0.0008245704433331452, 0.0008253709971616235, 0.0008261715509901019, 0.0008269721048185802, 0.0008277726586470585, 0.0008285732124755368, 0.0008293737663040151, 0.0008301743201324935, 0.0008309748739609718, 0.0008317754277894501, 0.0008325759816179284, 0.0008333765354464068, 0.0008341770892748851, 0.0008349776431033634, 0.0008357781969318417, 0.00083657875076032, 0.0008373793045887984, 0.0008381798584172767, 0.000838980412245755, 0.0008397809660742333, 0.0008405815199027117, 0.00084138207373119, 0.0008421826275596683, 0.0008429831813881466, 0.000843783735216625, 0.0008445842890451033, 0.0008453848428735816, 0.0008461853967020599, 0.0008469859505305383, 0.0008477865043590166, 0.0008485870581874949, 0.0008493876120159732, 0.0008501881658444515, 0.0008509887196729299, 0.0008517892735014082, 0.0008525898273298865, 0.0008533903811583648, 0.0008541909349868432, 0.0008549914888153215, 0.0008557920426437998, 0.0008565925964722781, 0.0008573931503007565, 0.0008581937041292348, 0.0008589942579577131, 0.0008597948117861914, 0.0008605953656146697, 0.0008613959194431481, 0.0008621964732716264, 0.0008629970271001047, 0.000863797580928583, 0.0008645981347570614, 0.0008653986885855397, 0.000866199242414018, 0.0008669997962424963, 0.0008678003500709747, 0.000868600903899453, 0.0008694014577279313, 0.0008702020115564096, 0.000871002565384888, 0.0008718031192133663, 0.0008726036730418446, 0.0008734042268703229, 0.0008742047806988012, 0.0008750053345272796, 0.0008758058883557579, 0.0008766064421842362, 0.0008774069960127145, 0.0008782075498411929, 0.0008790081036696712, 0.0008798086574981495, 0.0008806092113266278, 0.0008814097651551061, 0.0008822103189835845, 0.0008830108728120628, 0.0008838114266405411, 0.0008846119804690194, 0.0008854125342974978, 0.0008862130881259761, 0.0008870136419544544, 0.0008878141957829327, 0.0008886147496114111, 0.0008894153034398894, 0.0008902158572683677, 0.000891016411096846, 0.0008918169649253243, 0.0008926175187538027, 0.000893418072582281, 0.0008942186264107593, 0.0008950191802392376, 0.000895819734067716, 0.0008966202878961943, 0.0008974208417246726, 0.0008982213955531509, 0.0008990219493816293, 0.0008998225032101076, 0.0009006230570385859, 0.0009014236108670642, 0.0009022241646955426, 0.0009030247185240209, 0.0009038252723524992, 0.0009046258261809775, 0.0009054263800094558, 0.0009062269338379342, 0.0009070274876664125, 0.0009078280414948908, 0.0009086285953233691, 0.0009094291491518475, 0.0009102297029803258, 0.0009110302568088041, 0.0009118308106372824, 0.0009126313644657608, 0.0009134319182942391, 0.0009142324721227174, 0.0009150330259511957, 0.000915833579779674, 0.0009166341336081524, 0.0009174346874366307, 0.000918235241265109, 0.0009190357950935873, 0.0009198363489220657, 0.000920636902750544, 0.0009214374565790223, 0.0009222380104075006, 0.000923038564235979, 0.0009238391180644573, 0.0009246396718929356, 0.0009254402257214139, 0.0009262407795498922, 0.0009270413333783706, 0.0009278418872068489, 0.0009286424410353272, 0.0009294429948638055, 0.0009302435486922839, 0.0009310441025207622, 0.0009318446563492405, 0.0009326452101777188, 0.0009334457640061972, 0.0009342463178346755, 0.0009350468716631538, 0.0009358474254916321, 0.0009366479793201104, 0.0009374485331485888, 0.0009382490869770671, 0.0009390496408055454, 0.0009398501946340237, 0.0009406507484625021, 0.0009414513022909804, 0.0009422518561194587, 0.000943052409947937, 0.0009438529637764154, 0.0009446535176048937, 0.000945454071433372, 0.0009462546252618503, 0.0009470551790903286, 0.000947855732918807, 0.0009486562867472853, 0.0009494568405757636, 0.0009502573944042419, 0.0009510579482327203, 0.0009518585020611986, 0.0009526590558896769, 0.0009534596097181552, 0.0009542601635466336, 0.0009550607173751119, 0.0009558612712035902, 0.0009566618250320685, 0.0009574623788605468, 0.0009582629326890252, 0.0009590634865175035, 0.0009598640403459818, 0.0009606645941744601, 0.0009614651480029385, 0.0009622657018314168, 0.0009630662556598951, 0.0009638668094883734, 0.0009646673633168518, 0.0009654679171453301, 0.0009662684709738084, 0.0009670690248022867, 0.000967869578630765, 0.0009686701324592434, 0.0009694706862877217, 0.0009702712401162, 0.0009710717939446783, 0.0009718723477731567, 0.000972672901601635, 0.0009734734554301133, 0.0009742740092585916, 0.00097507456308707, 0.0009758751169155483, 0.0009766756707440267, 0.000977476224572505, 0.0009782767784009834, 0.0009790773322294617, 0.00097987788605794, 0.0009806784398864183, 0.0009814789937148966, 0.000982279547543375, 0.0009830801013718533, 0.0009838806552003316, 0.00098468120902881, 0.0009854817628572883, 0.0009862823166857666, 0.000987082870514245, 0.0009878834243427232, 0.0009886839781712016, 0.0009894845319996799, 0.0009902850858281582, 0.0009910856396566365, 0.0009918861934851148, 0.0009926867473135932, 0.0009934873011420715, 0.0009942878549705498, 0.0009950884087990281, 0.0009958889626275065, 0.0009966895164559848, 0.0009974900702844631, 0.0009982906241129414, 0.0009990911779414198, 0.000999891731769898, 0.0010006922855983764, 0.0010014928394268547, 0.001002293393255333, 0.0010030939470838114, 0.0010038945009122897, 0.001004695054740768, 0.0010054956085692463, 0.0010062961623977247, 0.001007096716226203, 0.0010078972700546813, 0.0010086978238831596, 0.001009498377711638, 0.0010102989315401163, 0.0010110994853685946, 0.001011900039197073, 0.0010127005930255512, 0.0010135011468540296, 0.001014301700682508, 0.0010151022545109862, 0.0010159028083394645, 0.0010167033621679429, 0.0010175039159964212, 0.0010183044698248995, 0.0010191050236533778, 0.0010199055774818562, 0.0010207061313103345, 0.0010215066851388128, 0.0010223072389672911, 0.0010231077927957695, 0.0010239083466242478, 0.001024708900452726, 0.0010255094542812044, 0.0010263100081096827, 0.001027110561938161, 0.0010279111157666394, 0.0010287116695951177, 0.001029512223423596, 0.0010303127772520744, 0.0010311133310805527, 0.001031913884909031, 0.0010327144387375093, 0.0010335149925659877, 0.001034315546394466, 0.0010351161002229443, 0.0010359166540514226, 0.001036717207879901, 0.0010375177617083793, 0.0010383183155368576, 0.001039118869365336, 0.0010399194231938142, 0.0010407199770222926, 0.0010415205308507709, 0.0010423210846792492, 0.0010431216385077275, 0.0010439221923362059, 0.0010447227461646842, 0.0010455232999931625, 0.0010463238538216408, 0.0010471244076501191, 0.0010479249614785975, 0.0010487255153070758, 0.0010495260691355541, 0.0010503266229640324, 0.0010511271767925108, 0.001051927730620989, 0.0010527282844494674, 0.0010535288382779457, 0.001054329392106424, 0.0010551299459349024, 0.0010559304997633807, 0.001056731053591859, 0.0010575316074203373, 0.0010583321612488157, 0.001059132715077294, 0.0010599332689057723, 0.0010607338227342506, 0.001061534376562729, 0.0010623349303912073, 0.0010631354842196856, 0.001063936038048164, 0.0010647365918766423, 0.0010655371457051206, 0.001066337699533599, 0.0010671382533620772, 0.0010679388071905555, 0.0010687393610190339, 0.0010695399148475122, 0.0010703404686759905, 0.0010711410225044688, 0.0010719415763329472, 0.0010727421301614255, 0.0010735426839899038, 0.0010743432378183821, 0.0010751437916468605, 0.0010759443454753388, 0.001076744899303817, 0.0010775454531322954, 0.0010783460069607737, 0.001079146560789252, 0.0010799471146177304, 0.0010807476684462087, 0.001081548222274687, 0.0010823487761031654, 0.0010831493299316437, 0.001083949883760122, 0.0010847504375886003, 0.0010855509914170787, 0.001086351545245557, 0.0010871520990740353, 0.0010879526529025136, 0.001088753206730992, 0.0010895537605594703, 0.0010903543143879486, 0.001091154868216427, 0.0010919554220449052, 0.0010927559758733836, 0.0010935565297018619, 0.0010943570835303402, 0.0010951576373588185, 0.0010959581911872969, 0.0010967587450157752, 0.0010975592988442535, 0.0010983598526727318, 0.0010991604065012101, 0.0010999609603296885, 0.0011007615141581668, 0.0011015620679866451, 0.0011023626218151234, 0.0011031631756436018, 0.00110396372947208, 0.0011047642833005584, 0.0011055648371290367, 0.001106365390957515, 0.0011071659447859934, 0.0011079664986144717, 0.00110876705244295, 0.0011095676062714283, 0.0011103681600999067, 0.001111168713928385, 0.0011119692677568633, 0.0011127698215853416, 0.00111357037541382, 0.0011143709292422983, 0.0011151714830707766, 0.001115972036899255, 0.0011167725907277333, 0.0011175731445562116, 0.00111837369838469, 0.0011191742522131682, 0.0011199748060416465, 0.0011207753598701249, 0.0011215759136986032, 0.0011223764675270815, 0.0011231770213555598, 0.0011239775751840382, 0.0011247781290125165, 0.0011255786828409948, 0.0011263792366694731, 0.0011271797904979515, 0.0011279803443264298, 0.001128780898154908, 0.0011295814519833864, 0.0011303820058118647, 0.001131182559640343, 0.0011319831134688214, 0.0011327836672972997, 0.001133584221125778, 0.0011343847749542564, 0.0011351853287827347, 0.001135985882611213, 0.0011367864364396913, 0.0011375869902681697, 0.001138387544096648, 0.0011391880979251263, 0.0011399886517536046, 0.001140789205582083, 0.0011415897594105613, 0.0011423903132390396, 0.001143190867067518, 0.0011439914208959962, 0.0011447919747244746, 0.0011455925285529529, 0.0011463930823814312, 0.0011471936362099095, 0.0011479941900383879, 0.0011487947438668662, 0.0011495952976953445, 0.0011503958515238228, 0.0011511964053523012, 0.0011519969591807795, 0.0011527975130092578, 0.0011535980668377361, 0.0011543986206662144, 0.0011551991744946928, 0.001155999728323171, 0.0011568002821516494, 0.0011576008359801277, 0.001158401389808606, 0.0011592019436370844, 0.0011600024974655627, 0.001160803051294041, 0.0011616036051225194, 0.0011624041589509977, 0.001163204712779476, 0.0011640052666079543, 0.0011648058204364326, 0.001165606374264911, 0.0011664069280933893, 0.0011672074819218676, 0.001168008035750346, 0.0011688085895788243, 0.0011696091434073026, 0.001170409697235781, 0.0011712102510642592, 0.0011720108048927376, 0.0011728113587212159, 0.0011736119125496942, 0.0011744124663781725, 0.0011752130202066508, 0.0011760135740351292, 0.0011768141278636075, 0.0011776146816920858, 0.0011784152355205641, 0.0011792157893490425, 0.0011800163431775208, 0.001180816897005999, 0.0011816174508344774, 0.0011824180046629558, 0.001183218558491434, 0.0011840191123199124, 0.0011848196661483907, 0.001185620219976869, 0.0011864207738053474, 0.0011872213276338257, 0.001188021881462304, 0.0011888224352907823, 0.0011896229891192607, 0.001190423542947739, 0.0011912240967762173, 0.0011920246506046956, 0.001192825204433174, 0.0011936257582616523, 0.0011944263120901306, 0.001195226865918609, 0.0011960274197470872, 0.0011968279735755656, 0.001197628527404044, 0.0011984290812325222, 0.0011992296350610005, 0.0012000301888894789, 0.0012008307427179572, 0.0012016312965464355, 0.0012024318503749138, 0.0012032324042033922, 0.0012040329580318705, 0.0012048335118603488, 0.0012056340656888271, 0.0012064346195173054, 0.0012072351733457838, 0.001208035727174262, 0.0012088362810027404, 0.0012096368348312187, 0.001210437388659697, 0.0012112379424881754, 0.0012120384963166537, 0.001212839050145132, 0.0012136396039736104, 0.0012144401578020887, 0.001215240711630567, 0.0012160412654590453, 0.0012168418192875236, 0.001217642373116002, 0.0012184429269444803, 0.0012192434807729586, 0.001220044034601437, 0.0012208445884299153, 0.0012216451422583936, 0.001222445696086872, 0.0012232462499153502, 0.0012240468037438286, 0.0012248473575723069, 0.0012256479114007852, 0.0012264484652292635, 0.0012272490190577418, 0.0012280495728862202, 0.0012288501267146985, 0.0012296506805431768, 0.0012304512343716551, 0.0012312517882001335, 0.0012320523420286118, 0.0012328528958570901, 0.0012336534496855684, 0.0012344540035140468, 0.001235254557342525, 0.0012360551111710034, 0.0012368556649994817, 0.00123765621882796, 0.0012384567726564384, 0.0012392573264849167, 0.001240057880313395, 0.0012408584341418733, 0.0012416589879703517, 0.00124245954179883, 0.0012432600956273083, 0.0012440606494557866, 0.001244861203284265, 0.0012456617571127433, 0.0012464623109412216, 0.0012472628647697, 0.0012480634185981782, 0.0012488639724266566, 0.001249664526255135, 0.0012504650800836132, 0.0012512656339120915, 0.0012520661877405699, 0.0012528667415690482, 0.0012536672953975265, 0.0012544678492260048, 0.0012552684030544832, 0.0012560689568829615, 0.0012568695107114398, 0.0012576700645399181, 0.0012584706183683964, 0.0012592711721968748, 0.001260071726025353, 0.0012608722798538314, 0.0012616728336823097, 0.001262473387510788, 0.0012632739413392664, 0.0012640744951677447, 0.001264875048996223, 0.0012656756028247014, 0.0012664761566531797, 0.001267276710481658, 0.0012680772643101363, 0.0012688778181386147, 0.001269678371967093, 0.0012704789257955713, 0.0012712794796240496, 0.001272080033452528, 0.0012728805872810063, 0.0012736811411094846, 0.001274481694937963, 0.0012752822487664412, 0.0012760828025949196, 0.0012768833564233979, 0.0012776839102518762, 0.0012784844640803545, 0.0012792850179088329, 0.0012800855717373112, 0.0012808861255657895, 0.0012816866793942678, 0.0012824872332227461, 0.0012832877870512245, 0.0012840883408797028, 0.0012848888947081811, 0.0012856894485366594, 0.0012864900023651378, 0.001287290556193616, 0.0012880911100220944, 0.0012888916638505727, 0.001289692217679051, 0.0012904927715075294, 0.0012912933253360077, 0.001292093879164486, 0.0012928944329929643, 0.0012936949868214427, 0.001294495540649921, 0.0012952960944783993, 0.0012960966483068776, 0.001296897202135356, 0.0012976977559638343, 0.0012984983097923126, 0.001299298863620791, 0.0013000994174492693, 0.0013008999712777476, 0.001301700525106226, 0.0013025010789347042, 0.0013033016327631825, 0.0013041021865916609, 0.0013049027404201392, 0.0013057032942486175, 0.0013065038480770958, 0.0013073044019055742, 0.0013081049557340525, 0.0013089055095625308, 0.0013097060633910091, 0.0013105066172194875, 0.0013113071710479658, 0.001312107724876444, 0.0013129082787049224, 0.0013137088325334007, 0.001314509386361879, 0.0013153099401903574, 0.0013161104940188357, 0.001316911047847314, 0.0013177116016757924, 0.0013185121555042707, 0.001319312709332749, 0.0013201132631612273, 0.0013209138169897057, 0.001321714370818184, 0.0013225149246466623, 0.0013233154784751406, 0.001324116032303619, 0.0013249165861320973, 0.0013257171399605756, 0.001326517693789054, 0.0013273182476175322, 0.0013281188014460106, 0.0013289193552744889, 0.0013297199091029672, 0.0013305204629314455, 0.0013313210167599239, 0.0013321215705884022, 0.0013329221244168805, 0.0013337226782453588, 0.0013345232320738371, 0.0013353237859023155, 0.0013361243397307938, 0.0013369248935592721, 0.0013377254473877504, 0.0013385260012162288, 0.001339326555044707, 0.0013401271088731854, 0.0013409276627016637, 0.001341728216530142, 0.0013425287703586204, 0.0013433293241870987, 0.001344129878015577, 0.0013449304318440553, 0.0013457309856725337, 0.001346531539501012, 0.0013473320933294903, 0.0013481326471579686, 0.001348933200986447, 0.0013497337548149253, 0.0013505343086434036, 0.001351334862471882, 0.0013521354163003603, 0.0013529359701288386, 0.001353736523957317, 0.0013545370777857952, 0.0013553376316142735, 0.0013561381854427519, 0.0013569387392712302, 0.0013577392930997085, 0.0013585398469281868, 0.0013593404007566652, 0.0013601409545851435, 0.0013609415084136218, 0.0013617420622421001, 0.0013625426160705785, 0.0013633431698990568, 0.001364143723727535, 0.0013649442775560134, 0.0013657448313844917, 0.00136654538521297, 0.0013673459390414484, 0.0013681464928699267, 0.001368947046698405, 0.0013697476005268834, 0.0013705481543553617, 0.00137134870818384, 0.0013721492620123183, 0.0013729498158407967, 0.001373750369669275, 0.0013745509234977533, 0.0013753514773262316, 0.00137615203115471, 0.0013769525849831883, 0.0013777531388116666, 0.001378553692640145, 0.0013793542464686232, 0.0013801548002971016, 0.0013809553541255799, 0.0013817559079540582, 0.0013825564617825365, 0.0013833570156110149, 0.0013841575694394932, 0.0013849581232679715, 0.0013857586770964498, 0.0013865592309249281, 0.0013873597847534065, 0.0013881603385818848, 0.0013889608924103631, 0.0013897614462388414, 0.0013905620000673198, 0.001391362553895798, 0.0013921631077242764, 0.0013929636615527547, 0.001393764215381233, 0.0013945647692097114, 0.0013953653230381897, 0.001396165876866668, 0.0013969664306951464, 0.0013977669845236247, 0.001398567538352103, 0.0013993680921805813, 0.0014001686460090596, 0.001400969199837538, 0.0014017697536660163, 0.0014025703074944946, 0.001403370861322973, 0.0014041714151514513, 0.0014049719689799296, 0.001405772522808408, 0.0014065730766368862, 0.0014073736304653646, 0.0014081741842938429, 0.0014089747381223212, 0.0014097752919507995, 0.0014105758457792778, 0.0014113763996077562, 0.0014121769534362345, 0.0014129775072647128, 0.0014137780610931911, 0.0014145786149216695, 0.0014153791687501478, 0.001416179722578626, 0.0014169802764071044, 0.0014177808302355828, 0.001418581384064061, 0.0014193819378925394, 0.0014201824917210177, 0.001420983045549496, 0.0014217835993779744, 0.0014225841532064527, 0.001423384707034931, 0.0014241852608634093, 0.0014249858146918877, 0.001425786368520366, 0.0014265869223488443, 0.0014273874761773226, 0.001428188030005801, 0.0014289885838342793, 0.0014297891376627576, 0.001430589691491236, 0.0014313902453197142, 0.0014321907991481926, 0.0014329913529766709, 0.0014337919068051492, 0.0014345924606336275, 0.0014353930144621059, 0.0014361935682905842, 0.0014369941221190625, 0.0014377946759475408, 0.0014385952297760192, 0.0014393957836044975, 0.0014401963374329758, 0.0014409968912614541, 0.0014417974450899324, 0.0014425979989184108, 0.001443398552746889, 0.0014441991065753674, 0.0014449996604038457, 0.001445800214232324, 0.0014466007680608024, 0.0014474013218892807, 0.001448201875717759, 0.0014490024295462374, 0.0014498029833747157, 0.001450603537203194, 0.0014514040910316723, 0.0014522046448601506, 0.001453005198688629, 0.0014538057525171073, 0.0014546063063455856, 0.001455406860174064, 0.0014562074140025423, 0.0014570079678310206, 0.001457808521659499, 0.0014586090754879772, 0.0014594096293164556, 0.0014602101831449339, 0.0014610107369734122, 0.0014618112908018905, 0.0014626118446303688, 0.0014634123984588472, 0.0014642129522873255, 0.0014650135061158038, 0.0014658140599442821, 0.0014666146137727605, 0.0014674151676012388, 0.0014682157214297171, 0.0014690162752581954, 0.0014698168290866738, 0.001470617382915152, 0.0014714179367436304, 0.0014722184905721087, 0.001473019044400587, 0.0014738195982290654, 0.0014746201520575437, 0.001475420705886022, 0.0014762212597145003, 0.0014770218135429787, 0.001477822367371457, 0.0014786229211999353, 0.0014794234750284136, 0.001480224028856892, 0.0014810245826853703, 0.0014818251365138486, 0.001482625690342327, 0.0014834262441708052, 0.0014842267979992836, 0.001485027351827762, 0.0014858279056562402, 0.0014866284594847185, 0.0014874290133131969, 0.0014882295671416752, 0.0014890301209701535, 0.0014898306747986318, 0.0014906312286271102, 0.0014914317824555885, 0.0014922323362840668, 0.0014930328901125451, 0.0014938334439410234, 0.0014946339977695018, 0.00149543455159798, 0.0014962351054264584, 0.0014970356592549367, 0.001497836213083415, 0.0014986367669118934, 0.0014994373207403717, 0.00150023787456885, 0.0015010384283973284, 0.0015018389822258067, 0.001502639536054285, 0.0015034400898827633, 0.0015042406437112416, 0.00150504119753972, 0.0015058417513681983, 0.0015066423051966766, 0.001507442859025155, 0.0015082434128536333, 0.0015090439666821116, 0.00150984452051059, 0.0015106450743390682, 0.0015114456281675466, 0.0015122461819960249, 0.0015130467358245032, 0.0015138472896529815, 0.0015146478434814599, 0.0015154483973099382, 0.0015162489511384165, 0.0015170495049668948, 0.0015178500587953731, 0.0015186506126238515, 0.0015194511664523298, 0.0015202517202808081, 0.0015210522741092864, 0.0015218528279377648, 0.001522653381766243, 0.0015234539355947214, 0.0015242544894231997, 0.001525055043251678, 0.0015258555970801564, 0.0015266561509086347, 0.001527456704737113, 0.0015282572585655913, 0.0015290578123940697, 0.001529858366222548, 0.0015306589200510263, 0.0015314594738795046, 0.001532260027707983, 0.0015330605815364613, 0.0015338611353649396, 0.001534661689193418, 0.0015354622430218963, 0.0015362627968503746, 0.001537063350678853, 0.0015378639045073312, 0.0015386644583358095, 0.0015394650121642879, 0.0015402655659927662, 0.0015410661198212445, 0.0015418666736497228, 0.0015426672274782012, 0.0015434677813066795, 0.0015442683351351578, 0.0015450688889636361, 0.0015458694427921145, 0.0015466699966205928, 0.001547470550449071, 0.0015482711042775494, 0.0015490716581060277, 0.001549872211934506, 0.0015506727657629844, 0.0015514733195914627, 0.001552273873419941, 0.0015530744272484194, 0.0015538749810768977, 0.001554675534905376, 0.0015554760887338543, 0.0015562766425623327, 0.001557077196390811, 0.0015578777502192893, 0.0015586783040477676, 0.001559478857876246, 0.0015602794117047243, 0.0015610799655332026, 0.001561880519361681, 0.0015626810731901592, 0.0015634816270186376, 0.0015642821808471159, 0.0015650827346755942, 0.0015658832885040725, 0.0015666838423325509, 0.0015674843961610292, 0.0015682849499895075, 0.0015690855038179858, 0.0015698860576464641, 0.0015706866114749425, 0.0015714871653034208, 0.0015722877191318991, 0.0015730882729603774, 0.0015738888267888558, 0.001574689380617334, 0.0015754899344458124, 0.0015762904882742907, 0.001577091042102769, 0.0015778915959312474, 0.0015786921497597257, 0.001579492703588204, 0.0015802932574166823, 0.0015810938112451607, 0.001581894365073639, 0.0015826949189021173, 0.0015834954727305956, 0.001584296026559074, 0.0015850965803875523, 0.0015858971342160306, 0.001586697688044509, 0.0015874982418729873, 0.0015882987957014656, 0.001589099349529944, 0.0015898999033584222, 0.0015907004571869005, 0.0015915010110153789, 0.0015923015648438572, 0.0015931021186723355, 0.0015939026725008138, 0.0015947032263292922, 0.0015955037801577705, 0.0015963043339862488, 0.0015971048878147271, 0.0015979054416432055, 0.0015987059954716838, 0.001599506549300162, 0.0016003071031286404, 0.0016011076569571187, 0.001601908210785597, 0.0016027087646140754, 0.0016035093184425537, 0.001604309872271032, 0.0016051104260995104, 0.0016059109799279887, 0.001606711533756467, 0.0016075120875849453, 0.0016083126414134237, 0.001609113195241902, 0.0016099137490703803, 0.0016107143028988586, 0.001611514856727337, 0.0016123154105558153, 0.0016131159643842936, 0.001613916518212772, 0.0016147170720412502, 0.0016155176258697286, 0.0016163181796982069, 0.0016171187335266852, 0.0016179192873551635, 0.0016187198411836419, 0.0016195203950121202, 0.0016203209488405985, 0.0016211215026690768, 0.0016219220564975551, 0.0016227226103260335, 0.0016235231641545118, 0.0016243237179829901, 0.0016251242718114684, 0.0016259248256399468, 0.001626725379468425, 0.0016275259332969034, 0.0016283264871253817, 0.00162912704095386, 0.0016299275947823384, 0.0016307281486108167, 0.001631528702439295, 0.0016323292562677733, 0.0016331298100962517, 0.00163393036392473, 0.0016347309177532083, 0.0016355314715816866, 0.001636332025410165, 0.0016371325792386433, 0.0016379331330671216, 0.0016387336868956, 0.0016395342407240783, 0.0016403347945525566, 0.001641135348381035, 0.0016419359022095132, 0.0016427364560379916, 0.0016435370098664699, 0.0016443375636949482, 0.0016451381175234265, 0.0016459386713519048, 0.0016467392251803832, 0.0016475397790088615, 0.0016483403328373398, 0.0016491408866658181, 0.0016499414404942965, 0.0016507419943227748, 0.001651542548151253, 0.0016523431019797314, 0.0016531436558082098, 0.001653944209636688, 0.0016547447634651664, 0.0016555453172936447, 0.001656345871122123, 0.0016571464249506014, 0.0016579469787790797, 0.001658747532607558, 0.0016595480864360363, 0.0016603486402645147, 0.001661149194092993, 0.0016619497479214713, 0.0016627503017499496, 0.001663550855578428, 0.0016643514094069063, 0.0016651519632353846, 0.001665952517063863, 0.0016667530708923412, 0.0016675536247208196, 0.0016683541785492979, 0.0016691547323777762, 0.0016699552862062545, 0.0016707558400347329, 0.0016715563938632112, 0.0016723569476916895, 0.0016731575015201678, 0.0016739580553486462, 0.0016747586091771245, 0.0016755591630056028, 0.0016763597168340811, 0.0016771602706625594, 0.0016779608244910378, 0.001678761378319516, 0.0016795619321479944, 0.0016803624859764727, 0.001681163039804951, 0.0016819635936334294, 0.0016827641474619077, 0.001683564701290386, 0.0016843652551188644, 0.0016851658089473427, 0.001685966362775821, 0.0016867669166042993, 0.0016875674704327776, 0.001688368024261256, 0.0016891685780897343, 0.0016899691319182126, 0.001690769685746691, 0.0016915702395751693, 0.0016923707934036476, 0.001693171347232126, 0.0016939719010606042, 0.0016947724548890826, 0.0016955730087175609, 0.0016963735625460392, 0.0016971741163745175, 0.0016979746702029958, 0.0016987752240314742, 0.0016995757778599525, 0.0017003763316884308, 0.0017011768855169091, 0.0017019774393453875, 0.0017027779931738658, 0.001703578547002344, 0.0017043791008308224, 0.0017051796546593008, 0.001705980208487779, 0.0017067807623162574, 0.0017075813161447357, 0.001708381869973214, 0.0017091824238016924, 0.0017099829776301707, 0.001710783531458649, 0.0017115840852871273, 0.0017123846391156057, 0.001713185192944084, 0.0017139857467725623, 0.0017147863006010406, 0.001715586854429519, 0.0017163874082579973, 0.0017171879620864756, 0.001717988515914954, 0.0017187890697434322, 0.0017195896235719106, 0.001720390177400389, 0.0017211907312288672, 0.0017219912850573455, 0.0017227918388858239, 0.0017235923927143022, 0.0017243929465427805, 0.0017251935003712588, 0.0017259940541997372, 0.0017267946080282155, 0.0017275951618566938, 0.0017283957156851721, 0.0017291962695136504, 0.0017299968233421288, 0.001730797377170607, 0.0017315979309990854, 0.0017323984848275637, 0.001733199038656042, 0.0017339995924845204, 0.0017348001463129987, 0.001735600700141477, 0.0017364012539699554, 0.0017372018077984337, 0.001738002361626912, 0.0017388029154553903, 0.0017396034692838686, 0.001740404023112347, 0.0017412045769408253, 0.0017420051307693036, 0.001742805684597782, 0.0017436062384262603, 0.0017444067922547386, 0.001745207346083217, 0.0017460078999116952, 0.0017468084537401736, 0.0017476090075686519, 0.0017484095613971302, 0.0017492101152256085, 0.0017500106690540868, 0.0017508112228825652, 0.0017516117767110435, 0.0017524123305395218, 0.0017532128843680001, 0.0017540134381964785, 0.0017548139920249568, 0.0017556145458534351, 0.0017564150996819134, 0.0017572156535103918, 0.00175801620733887, 0.0017588167611673484, 0.0017596173149958267, 0.001760417868824305, 0.0017612184226527834, 0.0017620189764812617, 0.00176281953030974, 0.0017636200841382183, 0.0017644206379666967, 0.001765221191795175, 0.0017660217456236533, 0.0017668222994521316, 0.00176762285328061, 0.0017684234071090883, 0.0017692239609375666, 0.001770024514766045, 0.0017708250685945233, 0.0017716256224230016, 0.00177242617625148, 0.0017732267300799582, 0.0017740272839084365, 0.0017748278377369149, 0.0017756283915653932, 0.0017764289453938715, 0.0017772294992223498, 0.0017780300530508282, 0.0017788306068793065, 0.0017796311607077848, 0.0017804317145362631, 0.0017812322683647415, 0.0017820328221932198, 0.001782833376021698, 0.0017836339298501764, 0.0017844344836786547, 0.001785235037507133, 0.0017860355913356114, 0.0017868361451640897, 0.001787636698992568, 0.0017884372528210464, 0.0017892378066495247, 0.001790038360478003, 0.0017908389143064813, 0.0017916394681349597, 0.001792440021963438, 0.0017932405757919163, 0.0017940411296203946, 0.001794841683448873, 0.0017956422372773513, 0.0017964427911058296, 0.001797243344934308, 0.0017980438987627862, 0.0017988444525912646, 0.0017996450064197429, 0.0018004455602482212, 0.0018012461140766995, 0.0018020466679051779, 0.0018028472217336562, 0.0018036477755621345, 0.0018044483293906128, 0.0018052488832190911, 0.0018060494370475695, 0.0018068499908760478, 0.0018076505447045261, 0.0018084510985330044, 0.0018092516523614828, 0.001810052206189961, 0.0018108527600184394, 0.0018116533138469177, 0.001812453867675396, 0.0018132544215038744, 0.0018140549753323527, 0.001814855529160831, 0.0018156560829893093, 0.0018164566368177877, 0.001817257190646266, 0.0018180577444747443, 0.0018188582983032226, 0.001819658852131701, 0.0018204594059601793, 0.0018212599597886576, 0.001822060513617136, 0.0018228610674456143, 0.0018236616212740926, 0.001824462175102571, 0.0018252627289310492, 0.0018260632827595275, 0.0018268638365880059, 0.0018276643904164842, 0.0018284649442449625, 0.0018292654980734408, 0.0018300660519019192, 0.0018308666057303975, 0.0018316671595588758, 0.0018324677133873541, 0.0018332682672158325, 0.0018340688210443108, 0.001834869374872789, 0.0018356699287012674, 0.0018364704825297457, 0.001837271036358224, 0.0018380715901867024, 0.0018388721440151807, 0.001839672697843659, 0.0018404732516721374, 0.0018412738055006157, 0.001842074359329094, 0.0018428749131575723, 0.0018436754669860507, 0.001844476020814529, 0.0018452765746430073, 0.0018460771284714856, 0.001846877682299964, 0.0018476782361284423, 0.0018484787899569206, 0.001849279343785399, 0.0018500798976138772, 0.0018508804514423556, 0.0018516810052708339, 0.0018524815590993122, 0.0018532821129277905, 0.0018540826667562689, 0.0018548832205847472, 0.0018556837744132255, 0.0018564843282417038, 0.0018572848820701821, 0.0018580854358986605, 0.0018588859897271388, 0.0018596865435556171, 0.0018604870973840954, 0.0018612876512125738, 0.001862088205041052, 0.0018628887588695304, 0.0018636893126980087, 0.001864489866526487, 0.0018652904203549654, 0.0018660909741834437, 0.001866891528011922, 0.0018676920818404003, 0.0018684926356688787, 0.001869293189497357, 0.0018700937433258353, 0.0018708942971543136, 0.001871694850982792, 0.0018724954048112703, 0.0018732959586397486, 0.001874096512468227, 0.0018748970662967053, 0.0018756976201251836, 0.001876498173953662, 0.0018772987277821402, 0.0018780992816106185, 0.0018788998354390969, 0.0018797003892675752, 0.0018805009430960535, 0.0018813014969245318, 0.0018821020507530102, 0.0018829026045814885, 0.0018837031584099668, 0.0018845037122384451, 0.0018853042660669235, 0.0018861048198954018, 0.00188690537372388, 0.0018877059275523584, 0.0018885064813808368, 0.001889307035209315, 0.0018901075890377934, 0.0018909081428662717, 0.00189170869669475, 0.0018925092505232284, 0.0018933098043517067, 0.001894110358180185, 0.0018949109120086633, 0.0018957114658371417, 0.00189651201966562, 0.0018973125734940983, 0.0018981131273225766, 0.001898913681151055, 0.0018997142349795333, 0.0019005147888080116, 0.00190131534263649, 0.0019021158964649682, 0.0019029164502934466, 0.0019037170041219249, 0.0019045175579504032, 0.0019053181117788815, 0.0019061186656073599, 0.0019069192194358382, 0.0019077197732643165, 0.0019085203270927948, 0.0019093208809212732, 0.0019101214347497515, 0.0019109219885782298, 0.0019117225424067081, 0.0019125230962351864, 0.0019133236500636648, 0.001914124203892143, 0.0019149247577206214, 0.0019157253115490997, 0.001916525865377578, 0.0019173264192060564, 0.0019181269730345347, 0.001918927526863013, 0.0019197280806914914, 0.0019205286345199697, 0.001921329188348448, 0.0019221297421769263, 0.0019229302960054046, 0.001923730849833883, 0.0019245314036623613, 0.0019253319574908396, 0.001926132511319318, 0.0019269330651477963, 0.0019277336189762746, 0.001928534172804753, 0.0019293347266332312, 0.0019301352804617096, 0.0019309358342901879, 0.0019317363881186662, 0.0019325369419471445, 0.0019333374957756228, 0.0019341380496041012, 0.0019349386034325795, 0.0019357391572610578, 0.0019365397110895361, 0.0019373402649180145, 0.0019381408187464928, 0.001938941372574971, 0.0019397419264034494, 0.0019405424802319278, 0.001941343034060406, 0.0019421435878888844, 0.0019429441417173627, 0.001943744695545841, 0.0019445452493743194, 0.0019453458032027977, 0.001946146357031276, 0.0019469469108597543, 0.0019477474646882327, 0.001948548018516711, 0.0019493485723451893, 0.0019501491261736676, 0.001950949680002146, 0.0019517502338306243, 0.0019525507876591026, 0.001953351341487581, 0.0019541518953160595, 0.001954952449144538, 0.001955753002973016, 0.0019565535568014944, 0.0019573541106299728, 0.001958154664458451, 0.0019589552182869294, 0.0019597557721154077, 0.001960556325943886, 0.0019613568797723644, 0.0019621574336008427, 0.001962957987429321, 0.0019637585412577993, 0.0019645590950862777, 0.001965359648914756, 0.0019661602027432343, 0.0019669607565717126, 0.001967761310400191, 0.0019685618642286693, 0.0019693624180571476, 0.001970162971885626, 0.0019709635257141042, 0.0019717640795425826, 0.001972564633371061, 0.0019733651871995392, 0.0019741657410280175, 0.001974966294856496, 0.001975766848684974, 0.0019765674025134525, 0.001977367956341931, 0.001978168510170409, 0.0019789690639988875, 0.001979769617827366, 0.001980570171655844, 0.0019813707254843224, 0.0019821712793128008, 0.001982971833141279, 0.0019837723869697574, 0.0019845729407982357, 0.001985373494626714, 0.0019861740484551924, 0.0019869746022836707, 0.001987775156112149, 0.0019885757099406274, 0.0019893762637691057, 0.001990176817597584, 0.0019909773714260623, 0.0019917779252545407, 0.001992578479083019, 0.0019933790329114973, 0.0019941795867399756, 0.001994980140568454, 0.0019957806943969323, 0.0019965812482254106, 0.001997381802053889, 0.0019981823558823672, 0.0019989829097108456, 0.001999783463539324, 0.002000584017367802, 0.0020013845711962805, 0.002002185125024759, 0.002002985678853237, 0.0020037862326817155, 0.002004586786510194, 0.002005387340338672, 0.0020061878941671505, 0.002006988447995629, 0.002007789001824107, 0.0020085895556525854, 0.0020093901094810638, 0.002010190663309542, 0.0020109912171380204, 0.0020117917709664987, 0.002012592324794977, 0.0020133928786234554, 0.0020141934324519337, 0.002014993986280412, 0.0020157945401088903, 0.0020165950939373687, 0.002017395647765847, 0.0020181962015943253, 0.0020189967554228036, 0.002019797309251282, 0.0020205978630797603, 0.0020213984169082386, 0.002022198970736717, 0.0020229995245651953, 0.0020238000783936736, 0.002024600632222152, 0.0020254011860506302, 0.0020262017398791085, 0.002027002293707587, 0.002027802847536065, 0.0020286034013645435, 0.002029403955193022, 0.0020302045090215, 0.0020310050628499785, 0.002031805616678457, 0.002032606170506935, 0.0020334067243354135, 0.0020342072781638918, 0.00203500783199237, 0.0020358083858208484, 0.0020366089396493267, 0.002037409493477805, 0.0020382100473062834, 0.0020390106011347617, 0.00203981115496324, 0.0020406117087917184, 0.0020414122626201967, 0.002042212816448675, 0.0020430133702771533, 0.0020438139241056317, 0.00204461447793411, 0.0020454150317625883, 0.0020462155855910666, 0.002047016139419545, 0.0020478166932480233, 0.0020486172470765016, 0.00204941780090498, 0.0020502183547334582, 0.0020510189085619366, 0.002051819462390415, 0.002052620016218893, 0.0020534205700473715, 0.00205422112387585, 0.002055021677704328, 0.0020558222315328065, 0.002056622785361285, 0.002057423339189763, 0.0020582238930182415, 0.00205902444684672, 0.002059825000675198, 0.0020606255545036764, 0.0020614261083321548, 0.002062226662160633, 0.0020630272159891114, 0.0020638277698175897, 0.002064628323646068, 0.0020654288774745464, 0.0020662294313030247, 0.002067029985131503, 0.0020678305389599813, 0.0020686310927884597, 0.002069431646616938, 0.0020702322004454163, 0.0020710327542738946, 0.002071833308102373, 0.0020726338619308513, 0.0020734344157593296, 0.002074234969587808, 0.0020750355234162863, 0.0020758360772447646, 0.002076636631073243, 0.0020774371849017212, 0.0020782377387301995, 0.002079038292558678, 0.002079838846387156, 0.0020806394002156345, 0.002081439954044113, 0.002082240507872591, 0.0020830410617010695, 0.002083841615529548, 0.002084642169358026, 0.0020854427231865045, 0.0020862432770149828, 0.002087043830843461, 0.0020878443846719394, 0.0020886449385004177, 0.002089445492328896, 0.0020902460461573744, 0.0020910465999858527, 0.002091847153814331, 0.0020926477076428094, 0.0020934482614712877, 0.002094248815299766, 0.0020950493691282443, 0.0020958499229567227, 0.002096650476785201, 0.0020974510306136793, 0.0020982515844421576, 0.002099052138270636, 0.0020998526920991143, 0.0021006532459275926, 0.002101453799756071, 0.0021022543535845492, 0.0021030549074130276, 0.002103855461241506, 0.002104656015069984, 0.0021054565688984625, 0.002106257122726941, 0.002107057676555419, 0.0021078582303838975, 0.002108658784212376, 0.002109459338040854, 0.0021102598918693325, 0.002111060445697811, 0.002111860999526289, 0.0021126615533547674, 0.0021134621071832458, 0.002114262661011724, 0.0021150632148402024, 0.0021158637686686807, 0.002116664322497159, 0.0021174648763256374, 0.0021182654301541157, 0.002119065983982594, 0.0021198665378110724, 0.0021206670916395507, 0.002121467645468029, 0.0021222681992965073, 0.0021230687531249856, 0.002123869306953464, 0.0021246698607819423, 0.0021254704146104206, 0.002126270968438899, 0.0021270715222673773, 0.0021278720760958556, 0.002128672629924334, 0.0021294731837528122, 0.0021302737375812906, 0.002131074291409769, 0.002131874845238247, 0.0021326753990667255, 0.002133475952895204, 0.002134276506723682, 0.0021350770605521605, 0.002135877614380639, 0.002136678168209117, 0.0021374787220375955, 0.0021382792758660738, 0.002139079829694552, 0.0021398803835230304, 0.0021406809373515088, 0.002141481491179987, 0.0021422820450084654, 0.0021430825988369437, 0.002143883152665422, 0.0021446837064939004, 0.0021454842603223787, 0.002146284814150857, 0.0021470853679793353, 0.0021478859218078137, 0.002148686475636292, 0.0021494870294647703, 0.0021502875832932486, 0.002151088137121727, 0.0021518886909502053, 0.0021526892447786836, 0.002153489798607162, 0.0021542903524356402, 0.0021550909062641186, 0.002155891460092597, 0.002156692013921075, 0.0021574925677495535, 0.002158293121578032, 0.00215909367540651, 0.0021598942292349885, 0.002160694783063467, 0.002161495336891945, 0.0021622958907204235, 0.002163096444548902, 0.00216389699837738, 0.0021646975522058584, 0.0021654981060343368, 0.002166298659862815, 0.0021670992136912934, 0.0021678997675197717, 0.00216870032134825, 0.0021695008751767284, 0.0021703014290052067, 0.002171101982833685, 0.0021719025366621634, 0.0021727030904906417, 0.00217350364431912, 0.0021743041981475983, 0.0021751047519760766, 0.002175905305804555, 0.0021767058596330333, 0.0021775064134615116, 0.00217830696728999, 0.0021791075211184683, 0.0021799080749469466, 0.002180708628775425, 0.0021815091826039032, 0.0021823097364323816, 0.00218311029026086, 0.002183910844089338, 0.0021847113979178165, 0.002185511951746295, 0.002186312505574773, 0.0021871130594032515, 0.00218791361323173, 0.002188714167060208, 0.0021895147208886865, 0.002190315274717165, 0.002191115828545643, 0.0021919163823741214, 0.0021927169362025998, 0.002193517490031078, 0.0021943180438595564, 0.0021951185976880347, 0.002195919151516513, 0.0021967197053449914, 0.0021975202591734697, 0.002198320813001948, 0.0021991213668304263, 0.0021999219206589047, 0.002200722474487383, 0.0022015230283158613, 0.0022023235821443396, 0.002203124135972818, 0.0022039246898012963, 0.0022047252436297746, 0.002205525797458253, 0.0022063263512867312, 0.0022071269051152096, 0.002207927458943688, 0.0022087280127721662, 0.0022095285666006445, 0.002210329120429123, 0.002211129674257601, 0.0022119302280860795, 0.002212730781914558, 0.002213531335743036, 0.0022143318895715145, 0.002215132443399993, 0.002215932997228471, 0.0022167335510569494, 0.0022175341048854278, 0.002218334658713906, 0.0022191352125423844, 0.0022199357663708627, 0.002220736320199341, 0.0022215368740278194, 0.0022223374278562977, 0.002223137981684776, 0.0022239385355132544, 0.0022247390893417327, 0.002225539643170211, 0.0022263401969986893, 0.0022271407508271676, 0.002227941304655646, 0.0022287418584841243, 0.0022295424123126026, 0.002230342966141081, 0.0022311435199695593, 0.0022319440737980376, 0.002232744627626516, 0.0022335451814549942, 0.0022343457352834726, 0.002235146289111951, 0.002235946842940429, 0.0022367473967689075, 0.002237547950597386, 0.002238348504425864, 0.0022391490582543425, 0.002239949612082821, 0.002240750165911299, 0.0022415507197397775, 0.002242351273568256, 0.002243151827396734, 0.0022439523812252124, 0.0022447529350536908, 0.002245553488882169, 0.0022463540427106474, 0.0022471545965391257, 0.002247955150367604, 0.0022487557041960824, 0.0022495562580245607, 0.002250356811853039, 0.0022511573656815173, 0.0022519579195099957, 0.002252758473338474, 0.0022535590271669523, 0.0022543595809954306, 0.002255160134823909, 0.0022559606886523873, 0.0022567612424808656, 0.002257561796309344, 0.0022583623501378223, 0.0022591629039663006, 0.002259963457794779, 0.0022607640116232572, 0.0022615645654517355, 0.002262365119280214, 0.002263165673108692, 0.0022639662269371705, 0.002264766780765649, 0.002265567334594127, 0.0022663678884226055, 0.002267168442251084, 0.002267968996079562, 0.0022687695499080405, 0.0022695701037365188, 0.002270370657564997, 0.0022711712113934754, 0.0022719717652219537, 0.002272772319050432, 0.0022735728728789104, 0.0022743734267073887, 0.002275173980535867, 0.0022759745343643454, 0.0022767750881928237, 0.002277575642021302, 0.0022783761958497803, 0.0022791767496782587, 0.002279977303506737, 0.0022807778573352153, 0.0022815784111636936, 0.002282378964992172, 0.0022831795188206503, 0.0022839800726491286, 0.002284780626477607, 0.0022855811803060852, 0.0022863817341345636, 0.002287182287963042, 0.00228798284179152, 0.0022887833956199985, 0.002289583949448477, 0.002290384503276955, 0.0022911850571054335, 0.002291985610933912, 0.00229278616476239, 0.0022935867185908685, 0.002294387272419347, 0.002295187826247825, 0.0022959883800763034, 0.0022967889339047818, 0.00229758948773326, 0.0022983900415617384, 0.0022991905953902167, 0.002299991149218695, 0.0023007917030471734, 0.0023015922568756517, 0.00230239281070413, 0.0023031933645326083, 0.0023039939183610867, 0.002304794472189565, 0.0023055950260180433, 0.0023063955798465216, 0.002307196133675, 0.0023079966875034783, 0.0023087972413319566, 0.002309597795160435, 0.0023103983489889133, 0.0023111989028173916, 0.00231199945664587, 0.0023128000104743482, 0.0023136005643028265, 0.002314401118131305, 0.002315201671959783, 0.0023160022257882615, 0.00231680277961674, 0.002317603333445218, 0.0023184038872736965, 0.002319204441102175, 0.002320004994930653, 0.0023208055487591315, 0.0023216061025876098, 0.002322406656416088, 0.0023232072102445664, 0.0023240077640730447, 0.002324808317901523, 0.0023256088717300014, 0.0023264094255584797, 0.002327209979386958, 0.0023280105332154364, 0.0023288110870439147, 0.002329611640872393, 0.0023304121947008713, 0.0023312127485293497, 0.002332013302357828, 0.0023328138561863063, 0.0023336144100147846, 0.002334414963843263, 0.0023352155176717413, 0.0023360160715002196, 0.002336816625328698, 0.0023376171791571762, 0.0023384177329856546, 0.002339218286814133, 0.002340018840642611, 0.0023408193944710895, 0.002341619948299568, 0.002342420502128046, 0.0023432210559565245, 0.002344021609785003, 0.002344822163613481, 0.0023456227174419595, 0.002346423271270438, 0.002347223825098916, 0.0023480243789273944, 0.0023488249327558728, 0.002349625486584351, 0.0023504260404128294, 0.0023512265942413077, 0.002352027148069786, 0.0023528277018982644, 0.0023536282557267427, 0.002354428809555221, 0.0023552293633836993, 0.0023560299172121777, 0.002356830471040656, 0.0023576310248691343, 0.0023584315786976126, 0.002359232132526091, 0.0023600326863545693, 0.0023608332401830476, 0.002361633794011526, 0.0023624343478400043, 0.0023632349016684826, 0.002364035455496961, 0.0023648360093254392, 0.0023656365631539176, 0.002366437116982396, 0.002367237670810874, 0.0023680382246393525, 0.002368838778467831, 0.002369639332296309, 0.0023704398861247875, 0.002371240439953266, 0.002372040993781744, 0.0023728415476102225, 0.0023736421014387008, 0.002374442655267179, 0.0023752432090956574, 0.0023760437629241358, 0.002376844316752614, 0.0023776448705810924, 0.0023784454244095707, 0.002379245978238049, 0.0023800465320665274, 0.0023808470858950057, 0.002381647639723484, 0.0023824481935519623, 0.0023832487473804407, 0.002384049301208919, 0.0023848498550373973, 0.0023856504088658756, 0.002386450962694354, 0.0023872515165228323, 0.0023880520703513106, 0.002388852624179789, 0.0023896531780082672, 0.0023904537318367456, 0.002391254285665224, 0.002392054839493702, 0.0023928553933221805, 0.002393655947150659, 0.002394456500979137, 0.0023952570548076155, 0.002396057608636094, 0.002396858162464572, 0.0023976587162930505, 0.002398459270121529, 0.002399259823950007, 0.0024000603777784854, 0.0024008609316069638, 0.002401661485435442, 0.0024024620392639204, 0.0024032625930923987, 0.002404063146920877, 0.0024048637007493554, 0.0024056642545778337, 0.002406464808406312, 0.0024072653622347904, 0.0024080659160632687, 0.002408866469891747, 0.0024096670237202253, 0.0024104675775487036, 0.002411268131377182, 0.0024120686852056603, 0.0024128692390341386, 0.002413669792862617, 0.0024144703466910953, 0.0024152709005195736, 0.002416071454348052, 0.0024168720081765302, 0.0024176725620050086, 0.002418473115833487, 0.002419273669661965, 0.0024200742234904435, 0.002420874777318922, 0.0024216753311474, 0.0024224758849758785, 0.002423276438804357, 0.002424076992632835, 0.0024248775464613135, 0.002425678100289792, 0.00242647865411827, 0.0024272792079467484, 0.0024280797617752268, 0.002428880315603705, 0.0024296808694321834, 0.0024304814232606617, 0.00243128197708914, 0.0024320825309176184, 0.0024328830847460967, 0.002433683638574575, 0.0024344841924030533, 0.0024352847462315317, 0.00243608530006001, 0.0024368858538884883, 0.0024376864077169666, 0.002438486961545445, 0.0024392875153739233, 0.0024400880692024016, 0.00244088862303088, 0.0024416891768593582, 0.0024424897306878366, 0.002443290284516315, 0.002444090838344793, 0.0024448913921732715, 0.00244569194600175, 0.002446492499830228, 0.0024472930536587065, 0.002448093607487185, 0.002448894161315663, 0.0024496947151441415, 0.00245049526897262, 0.002451295822801098, 0.0024520963766295764, 0.0024528969304580548, 0.002453697484286533, 0.0024544980381150114, 0.0024552985919434897, 0.002456099145771968, 0.0024568996996004464, 0.0024577002534289247, 0.002458500807257403, 0.0024593013610858814, 0.0024601019149143597, 0.002460902468742838, 0.0024617030225713163, 0.0024625035763997946, 0.002463304130228273, 0.0024641046840567513, 0.0024649052378852296, 0.002465705791713708, 0.0024665063455421863, 0.0024673068993706646, 0.002468107453199143, 0.0024689080070276212, 0.0024697085608560996, 0.002470509114684578, 0.002471309668513056, 0.0024721102223415345, 0.002472910776170013, 0.002473711329998491, 0.0024745118838269695, 0.002475312437655448, 0.002476112991483926, 0.0024769135453124045, 0.002477714099140883, 0.002478514652969361, 0.0024793152067978394, 0.0024801157606263178, 0.002480916314454796, 0.0024817168682832744, 0.0024825174221117527, 0.002483317975940231, 0.0024841185297687094, 0.0024849190835971877, 0.002485719637425666, 0.0024865201912541443, 0.0024873207450826227, 0.002488121298911101, 0.0024889218527395793, 0.0024897224065680576, 0.002490522960396536, 0.0024913235142250143, 0.0024921240680534926, 0.002492924621881971, 0.0024937251757104493, 0.0024945257295389276, 0.002495326283367406, 0.0024961268371958842, 0.0024969273910243625, 0.002497727944852841, 0.002498528498681319, 0.0024993290525097975, 0.002500129606338276, 0.002500930160166754, 0.0025017307139952325, 0.002502531267823711, 0.002503331821652189, 0.0025041323754806675, 0.0025049329293091458, 0.002505733483137624, 0.0025065340369661024, 0.0025073345907945807, 0.002508135144623059, 0.0025089356984515374, 0.0025097362522800157, 0.002510536806108494, 0.0025113373599369724, 0.0025121379137654507, 0.002512938467593929, 0.0025137390214224073, 0.0025145395752508857, 0.002515340129079364, 0.0025161406829078423, 0.0025169412367363206, 0.002517741790564799, 0.0025185423443932773, 0.0025193428982217556, 0.002520143452050234, 0.0025209440058787122, 0.0025217445597071906, 0.002522545113535669, 0.002523345667364147, 0.0025241462211926255, 0.002524946775021104, 0.002525747328849582, 0.0025265478826780605, 0.002527348436506539, 0.002528148990335017, 0.0025289495441634955, 0.002529750097991974, 0.002530550651820452, 0.0025313512056489304, 0.0025321517594774088, 0.002532952313305887, 0.0025337528671343654, 0.0025345534209628437, 0.002535353974791322, 0.0025361545286198004, 0.0025369550824482787, 0.002537755636276757, 0.0025385561901052353, 0.0025393567439337137, 0.002540157297762192, 0.0025409578515906703, 0.0025417584054191486, 0.002542558959247627, 0.0025433595130761053, 0.0025441600669045836, 0.002544960620733062, 0.0025457611745615403, 0.0025465617283900186, 0.002547362282218497, 0.0025481628360469752, 0.0025489633898754535, 0.002549763943703932, 0.00255056449753241, 0.0025513650513608885, 0.002552165605189367, 0.002552966159017845, 0.0025537667128463235, 0.002554567266674802, 0.00255536782050328, 0.0025561683743317585, 0.0025569689281602368, 0.002557769481988715, 0.0025585700358171934, 0.0025593705896456717, 0.00256017114347415, 0.0025609716973026284, 0.0025617722511311067, 0.002562572804959585, 0.0025633733587880634, 0.0025641739126165417, 0.00256497446644502, 0.0025657750202734983, 0.0025665755741019767, 0.002567376127930455, 0.0025681766817589333, 0.0025689772355874116, 0.00256977778941589, 0.0025705783432443683, 0.0025713788970728466, 0.002572179450901325, 0.0025729800047298032, 0.0025737805585582816, 0.00257458111238676, 0.002575381666215238, 0.0025761822200437165, 0.002576982773872195, 0.002577783327700673, 0.0025785838815291515, 0.00257938443535763, 0.002580184989186108, 0.0025809855430145865, 0.002581786096843065, 0.002582586650671543, 0.0025833872045000214, 0.0025841877583284998, 0.002584988312156978, 0.0025857888659854564, 0.0025865894198139347, 0.002587389973642413, 0.0025881905274708914, 0.0025889910812993697, 0.002589791635127848, 0.0025905921889563263, 0.0025913927427848047, 0.002592193296613283, 0.0025929938504417613, 0.0025937944042702396, 0.002594594958098718, 0.0025953955119271963, 0.0025961960657556746, 0.002596996619584153, 0.0025977971734126313, 0.0025985977272411096, 0.002599398281069588, 0.0026001988348980662, 0.0026009993887265445, 0.002601799942555023, 0.002602600496383501, 0.0026034010502119795, 0.002604201604040458, 0.002605002157868936, 0.0026058027116974145, 0.002606603265525893, 0.002607403819354371, 0.0026082043731828495, 0.0026090049270113278, 0.002609805480839806, 0.0026106060346682844, 0.0026114065884967627, 0.002612207142325241, 0.0026130076961537194, 0.0026138082499821977, 0.002614608803810676, 0.0026154093576391544, 0.0026162099114676327, 0.002617010465296111, 0.0026178110191245893, 0.0026186115729530677, 0.002619412126781546, 0.0026202126806100243, 0.0026210132344385026, 0.002621813788266981, 0.0026226143420954593, 0.0026234148959239376, 0.002624215449752416, 0.0026250160035808942, 0.0026258165574093726, 0.002626617111237851, 0.002627417665066329, 0.0026282182188948075, 0.002629018772723286, 0.002629819326551764, 0.0026306198803802425, 0.002631420434208721, 0.002632220988037199, 0.0026330215418656775, 0.002633822095694156, 0.002634622649522634, 0.0026354232033511124, 0.0026362237571795908, 0.002637024311008069, 0.0026378248648365474, 0.0026386254186650257, 0.002639425972493504, 0.0026402265263219824, 0.0026410270801504607, 0.002641827633978939, 0.0026426281878074174, 0.0026434287416358957, 0.002644229295464374, 0.0026450298492928523, 0.0026458304031213306, 0.002646630956949809, 0.0026474315107782873, 0.0026482320646067656, 0.002649032618435244, 0.0026498331722637223, 0.0026506337260922006, 0.002651434279920679, 0.0026522348337491572, 0.0026530353875776356, 0.002653835941406114, 0.002654636495234592, 0.0026554370490630705, 0.002656237602891549, 0.002657038156720027, 0.0026578387105485055, 0.002658639264376984, 0.002659439818205462, 0.0026602403720339405, 0.002661040925862419, 0.002661841479690897, 0.0026626420335193754, 0.0026634425873478538, 0.002664243141176332, 0.0026650436950048104, 0.0026658442488332887, 0.002666644802661767, 0.0026674453564902454, 0.0026682459103187237, 0.002669046464147202, 0.0026698470179756803, 0.0026706475718041587, 0.002671448125632637, 0.0026722486794611153, 0.0026730492332895936, 0.002673849787118072, 0.0026746503409465503, 0.0026754508947750286, 0.002676251448603507, 0.0026770520024319852, 0.0026778525562604636, 0.002678653110088942, 0.00267945366391742, 0.0026802542177458985, 0.002681054771574377, 0.002681855325402855, 0.0026826558792313335, 0.002683456433059812, 0.00268425698688829, 0.0026850575407167685, 0.002685858094545247, 0.002686658648373725, 0.0026874592022022034, 0.0026882597560306818, 0.00268906030985916, 0.0026898608636876384, 0.0026906614175161167, 0.002691461971344595, 0.0026922625251730734, 0.0026930630790015517, 0.00269386363283003, 0.0026946641866585084, 0.0026954647404869867, 0.002696265294315465, 0.0026970658481439433, 0.0026978664019724216, 0.0026986669558009, 0.0026994675096293783, 0.0027002680634578566, 0.002701068617286335, 0.0027018691711148133, 0.0027026697249432916, 0.00270347027877177, 0.0027042708326002482, 0.0027050713864287266, 0.002705871940257205, 0.002706672494085683, 0.0027074730479141615, 0.00270827360174264, 0.002709074155571118, 0.0027098747093995965, 0.002710675263228075, 0.002711475817056553, 0.0027122763708850315, 0.00271307692471351, 0.002713877478541988, 0.0027146780323704664, 0.0027154785861989448, 0.002716279140027423, 0.0027170796938559014, 0.0027178802476843797, 0.002718680801512858, 0.0027194813553413364, 0.0027202819091698147, 0.002721082462998293, 0.0027218830168267713, 0.0027226835706552497, 0.002723484124483728, 0.0027242846783122063, 0.0027250852321406846, 0.002725885785969163, 0.0027266863397976413, 0.0027274868936261196, 0.002728287447454598, 0.0027290880012830762, 0.0027298885551115546, 0.002730689108940033, 0.0027314896627685112, 0.0027322902165969895, 0.002733090770425468, 0.002733891324253946, 0.0027346918780824245, 0.002735492431910903, 0.002736292985739381, 0.0027370935395678595, 0.002737894093396338, 0.002738694647224816, 0.0027394952010532945, 0.0027402957548817728, 0.002741096308710251, 0.0027418968625387294, 0.0027426974163672077, 0.002743497970195686, 0.0027442985240241644, 0.0027450990778526427, 0.002745899631681121, 0.0027467001855095994, 0.0027475007393380777, 0.002748301293166556, 0.0027491018469950343, 0.0027499024008235127, 0.002750702954651991, 0.0027515035084804693, 0.0027523040623089476, 0.002753104616137426, 0.0027539051699659043, 0.0027547057237943826, 0.002755506277622861, 0.0027563068314513392, 0.0027571073852798176, 0.002757907939108296, 0.002758708492936774, 0.0027595090467652525, 0.002760309600593731, 0.002761110154422209, 0.0027619107082506875, 0.002762711262079166, 0.002763511815907644, 0.0027643123697361225, 0.002765112923564601, 0.002765913477393079, 0.0027667140312215574, 0.0027675145850500358, 0.002768315138878514, 0.0027691156927069924, 0.0027699162465354707, 0.002770716800363949, 0.0027715173541924274, 0.0027723179080209057, 0.002773118461849384, 0.0027739190156778623, 0.0027747195695063407, 0.002775520123334819, 0.0027763206771632973, 0.0027771212309917756, 0.002777921784820254, 0.0027787223386487323, 0.0027795228924772106, 0.002780323446305689, 0.0027811240001341673, 0.0027819245539626456, 0.002782725107791124, 0.0027835256616196022, 0.0027843262154480805, 0.002785126769276559, 0.002785927323105037, 0.0027867278769335155, 0.002787528430761994, 0.002788328984590472, 0.0027891295384189505, 0.002789930092247429, 0.002790730646075907, 0.0027915311999043855, 0.0027923317537328638, 0.002793132307561342, 0.0027939328613898204, 0.0027947334152182987, 0.002795533969046777, 0.0027963345228752554, 0.0027971350767037337, 0.002797935630532212, 0.0027987361843606904, 0.0027995367381891687, 0.002800337292017647, 0.0028011378458461253, 0.0028019383996746037, 0.002802738953503082, 0.0028035395073315603, 0.0028043400611600386, 0.002805140614988517, 0.0028059411688169953, 0.0028067417226454736, 0.002807542276473952, 0.0028083428303024302, 0.0028091433841309086, 0.002809943937959387, 0.002810744491787865, 0.0028115450456163435, 0.002812345599444822, 0.0028131461532733, 0.0028139467071017785, 0.002814747260930257, 0.002815547814758735, 0.0028163483685872135, 0.002817148922415692, 0.00281794947624417, 0.0028187500300726484, 0.0028195505839011268, 0.002820351137729605, 0.0028211516915580834, 0.0028219522453865617, 0.00282275279921504, 0.0028235533530435184, 0.0028243539068719967, 0.002825154460700475, 0.0028259550145289533, 0.0028267555683574317, 0.00282755612218591, 0.0028283566760143883, 0.0028291572298428666, 0.002829957783671345, 0.0028307583374998233, 0.0028315588913283016, 0.00283235944515678, 0.0028331599989852583, 0.0028339605528137366, 0.002834761106642215, 0.0028355616604706932, 0.0028363622142991715, 0.00283716276812765, 0.002837963321956128, 0.0028387638757846065, 0.002839564429613085, 0.002840364983441563, 0.0028411655372700415, 0.00284196609109852, 0.002842766644926998, 0.0028435671987554765, 0.0028443677525839548, 0.002845168306412433, 0.0028459688602409114, 0.0028467694140693897, 0.002847569967897868, 0.0028483705217263464, 0.0028491710755548247, 0.002849971629383303, 0.0028507721832117814, 0.0028515727370402597, 0.002852373290868738, 0.0028531738446972163, 0.0028539743985256947, 0.002854774952354173, 0.0028555755061826513, 0.0028563760600111296, 0.002857176613839608, 0.0028579771676680863, 0.0028587777214965646, 0.002859578275325043, 0.0028603788291535212, 0.0028611793829819996, 0.002861979936810478, 0.002862780490638956, 0.0028635810444674345, 0.002864381598295913, 0.002865182152124391, 0.0028659827059528695, 0.002866783259781348, 0.002867583813609826, 0.0028683843674383045, 0.002869184921266783, 0.002869985475095261, 0.0028707860289237394, 0.0028715865827522178, 0.002872387136580696, 0.0028731876904091744, 0.0028739882442376527, 0.002874788798066131, 0.0028755893518946094, 0.0028763899057230877, 0.002877190459551566, 0.0028779910133800444, 0.0028787915672085227, 0.002879592121037001, 0.0028803926748654793, 0.0028811932286939576, 0.002881993782522436, 0.0028827943363509143, 0.0028835948901793926, 0.002884395444007871, 0.0028851959978363493, 0.0028859965516648276, 0.002886797105493306, 0.0028875976593217842, 0.0028883982131502626, 0.002889198766978741, 0.002889999320807219, 0.0028907998746356975, 0.002891600428464176, 0.002892400982292654, 0.0028932015361211325, 0.002894002089949611, 0.002894802643778089, 0.0028956031976065675, 0.0028964037514350458, 0.002897204305263524, 0.0028980048590920024, 0.0028988054129204808, 0.002899605966748959, 0.0029004065205774374, 0.0029012070744059157, 0.002902007628234394, 0.0029028081820628724, 0.0029036087358913507, 0.002904409289719829, 0.0029052098435483073, 0.0029060103973767857, 0.002906810951205264, 0.0029076115050337423, 0.0029084120588622206, 0.002909212612690699, 0.0029100131665191773, 0.0029108137203476556, 0.002911614274176134, 0.0029124148280046122, 0.0029132153818330906, 0.002914015935661569, 0.002914816489490047, 0.0029156170433185255, 0.002916417597147004, 0.002917218150975482, 0.0029180187048039605, 0.002918819258632439, 0.002919619812460917, 0.0029204203662893955, 0.002921220920117874, 0.002922021473946352, 0.0029228220277748304, 0.0029236225816033088, 0.002924423135431787, 0.0029252236892602654, 0.0029260242430887437, 0.002926824796917222, 0.0029276253507457004, 0.0029284259045741787, 0.002929226458402657, 0.0029300270122311354, 0.0029308275660596137, 0.002931628119888092, 0.0029324286737165703, 0.0029332292275450486, 0.002934029781373527, 0.0029348303352020053, 0.0029356308890304836, 0.002936431442858962, 0.0029372319966874403, 0.0029380325505159186, 0.002938833104344397, 0.0029396336581728752, 0.0029404342120013536, 0.002941234765829832, 0.00294203531965831, 0.0029428358734867885, 0.002943636427315267, 0.002944436981143745, 0.0029452375349722235, 0.002946038088800702, 0.00294683864262918, 0.0029476391964576585, 0.002948439750286137, 0.002949240304114615, 0.0029500408579430934, 0.0029508414117715718, 0.00295164196560005, 0.0029524425194285284, 0.0029532430732570067, 0.002954043627085485, 0.0029548441809139634, 0.0029556447347424417, 0.00295644528857092, 0.0029572458423993983, 0.0029580463962278767, 0.002958846950056355, 0.0029596475038848333, 0.0029604480577133116, 0.00296124861154179, 0.0029620491653702683, 0.0029628497191987466, 0.002963650273027225, 0.0029644508268557032, 0.0029652513806841816, 0.00296605193451266, 0.0029668524883411382, 0.0029676530421696165, 0.002968453595998095, 0.002969254149826573, 0.0029700547036550515, 0.00297085525748353, 0.002971655811312008, 0.0029724563651404865, 0.002973256918968965, 0.002974057472797443, 0.0029748580266259214, 0.0029756585804543998, 0.002976459134282878, 0.0029772596881113564, 0.0029780602419398347, 0.002978860795768313, 0.0029796613495967914, 0.0029804619034252697, 0.002981262457253748, 0.0029820630110822264, 0.0029828635649107047, 0.002983664118739183, 0.0029844646725676613, 0.0029852652263961396, 0.002986065780224618, 0.0029868663340530963, 0.0029876668878815746, 0.002988467441710053, 0.0029892679955385313, 0.0029900685493670096, 0.002990869103195488, 0.0029916696570239662, 0.0029924702108524446, 0.002993270764680923, 0.002994071318509401, 0.0029948718723378795, 0.002995672426166358, 0.002996472979994836, 0.0029972735338233145, 0.002998074087651793, 0.002998874641480271, 0.0029996751953087495, 0.003000475749137228, 0.003001276302965706, 0.0030020768567941844, 0.0030028774106226628, 0.003003677964451141, 0.0030044785182796194, 0.0030052790721080977, 0.003006079625936576, 0.0030068801797650544, 0.0030076807335935327, 0.003008481287422011, 0.0030092818412504893, 0.0030100823950789677, 0.003010882948907446, 0.0030116835027359243, 0.0030124840565644026, 0.003013284610392881, 0.0030140851642213593, 0.0030148857180498376, 0.003015686271878316, 0.0030164868257067943, 0.0030172873795352726, 0.003018087933363751, 0.0030188884871922292, 0.0030196890410207075, 0.003020489594849186, 0.003021290148677664, 0.0030220907025061425, 0.003022891256334621, 0.003023691810163099, 0.0030244923639915775, 0.003025292917820056, 0.003026093471648534, 0.0030268940254770125, 0.0030276945793054908, 0.003028495133133969, 0.0030292956869624474, 0.0030300962407909257, 0.003030896794619404, 0.0030316973484478824, 0.0030324979022763607, 0.003033298456104839, 0.0030340990099333174, 0.0030348995637617957, 0.003035700117590274, 0.0030365006714187523, 0.0030373012252472307, 0.003038101779075709, 0.0030389023329041873, 0.0030397028867326656, 0.003040503440561144, 0.0030413039943896223, 0.0030421045482181006, 0.003042905102046579, 0.0030437056558750572, 0.0030445062097035356, 0.003045306763532014, 0.003046107317360492, 0.0030469078711889705, 0.003047708425017449, 0.003048508978845927, 0.0030493095326744055, 0.003050110086502884, 0.003050910640331362, 0.0030517111941598405, 0.003052511747988319, 0.003053312301816797, 0.0030541128556452754, 0.0030549134094737538, 0.003055713963302232, 0.0030565145171307104, 0.0030573150709591887, 0.003058115624787667, 0.0030589161786161454, 0.0030597167324446237, 0.003060517286273102, 0.0030613178401015803, 0.0030621183939300587, 0.003062918947758537, 0.0030637195015870153, 0.0030645200554154936, 0.003065320609243972, 0.0030661211630724503, 0.0030669217169009286, 0.003067722270729407, 0.0030685228245578853, 0.0030693233783863636, 0.003070123932214842, 0.0030709244860433202, 0.0030717250398717985, 0.003072525593700277, 0.003073326147528755, 0.0030741267013572335, 0.003074927255185712, 0.00307572780901419, 0.0030765283628426685, 0.003077328916671147, 0.003078129470499625, 0.0030789300243281035, 0.0030797305781565818, 0.00308053113198506, 0.0030813316858135384, 0.0030821322396420167, 0.003082932793470495, 0.0030837333472989734, 0.0030845339011274517, 0.00308533445495593, 0.0030861350087844084, 0.0030869355626128867, 0.003087736116441365, 0.0030885366702698433, 0.0030893372240983217, 0.0030901377779268, 0.0030909383317552783, 0.0030917388855837566, 0.003092539439412235, 0.0030933399932407133, 0.0030941405470691916, 0.00309494110089767, 0.0030957416547261482, 0.0030965422085546266, 0.003097342762383105, 0.003098143316211583, 0.0030989438700400615, 0.00309974442386854, 0.003100544977697018, 0.0031013455315254965, 0.003102146085353975, 0.003102946639182453, 0.0031037471930109315, 0.00310454774683941, 0.003105348300667888, 0.0031061488544963664, 0.0031069494083248448, 0.003107749962153323, 0.0031085505159818014, 0.0031093510698102797, 0.003110151623638758, 0.0031109521774672364, 0.0031117527312957147, 0.003112553285124193, 0.0031133538389526713, 0.0031141543927811497, 0.003114954946609628, 0.0031157555004381063, 0.0031165560542665846, 0.003117356608095063, 0.0031181571619235413, 0.0031189577157520196, 0.003119758269580498, 0.0031205588234089763, 0.0031213593772374546, 0.003122159931065933, 0.0031229604848944112, 0.0031237610387228896, 0.003124561592551368, 0.003125362146379846, 0.0031261627002083245, 0.003126963254036803, 0.003127763807865281, 0.0031285643616937595, 0.003129364915522238, 0.003130165469350716, 0.0031309660231791945, 0.0031317665770076728, 0.003132567130836151, 0.0031333676846646294, 0.0031341682384931078, 0.003134968792321586, 0.0031357693461500644, 0.0031365698999785427, 0.003137370453807021, 0.0031381710076354994, 0.0031389715614639777, 0.003139772115292456, 0.0031405726691209343, 0.0031413732229494127, 0.003142173776777891, 0.0031429743306063693, 0.0031437748844348476, 0.003144575438263326, 0.0031453759920918043, 0.0031461765459202826, 0.003146977099748761, 0.0031477776535772392, 0.0031485782074057176, 0.003149378761234196, 0.003150179315062674, 0.0031509798688911525, 0.003151780422719631, 0.003152580976548109, 0.0031533815303765875, 0.003154182084205066, 0.003154982638033544, 0.0031557831918620225, 0.003156583745690501, 0.003157384299518979, 0.0031581848533474574, 0.0031589854071759358, 0.003159785961004414, 0.0031605865148328924, 0.0031613870686613707, 0.003162187622489849, 0.0031629881763183274, 0.0031637887301468057, 0.003164589283975284, 0.0031653898378037624, 0.0031661903916322407, 0.003166990945460719, 0.0031677914992891973, 0.0031685920531176756, 0.003169392606946154, 0.0031701931607746323, 0.0031709937146031106, 0.003171794268431589, 0.0031725948222600673, 0.0031733953760885456, 0.003174195929917024, 0.0031749964837455022, 0.0031757970375739806, 0.003176597591402459, 0.003177398145230937, 0.0031781986990594155, 0.003178999252887894, 0.003179799806716372, 0.0031806003605448505, 0.003181400914373329, 0.003182201468201807, 0.0031830020220302855, 0.003183802575858764, 0.003184603129687242, 0.0031854036835157204, 0.0031862042373441988, 0.003187004791172677, 0.0031878053450011554, 0.0031886058988296337, 0.003189406452658112, 0.0031902070064865904, 0.0031910075603150687, 0.003191808114143547, 0.0031926086679720253, 0.0031934092218005037, 0.003194209775628982, 0.0031950103294574603, 0.0031958108832859386, 0.003196611437114417, 0.0031974119909428953, 0.0031982125447713736, 0.003199013098599852, 0.0031998136524283302, 0.0032006142062568086, 0.003201414760085287, 0.0032022153139137652, 0.0032030158677422435, 0.003203816421570722, 0.0032046169753992, 0.0032054175292276785, 0.003206218083056157, 0.003207018636884635, 0.0032078191907131135, 0.003208619744541592, 0.00320942029837007, 0.0032102208521985484, 0.0032110214060270268, 0.003211821959855505, 0.0032126225136839834, 0.0032134230675124617, 0.00321422362134094, 0.0032150241751694184, 0.0032158247289978967, 0.003216625282826375, 0.0032174258366548534, 0.0032182263904833317, 0.00321902694431181, 0.0032198274981402883, 0.0032206280519687666, 0.003221428605797245, 0.0032222291596257233, 0.0032230297134542016, 0.00322383026728268, 0.0032246308211111583, 0.0032254313749396366, 0.003226231928768115, 0.0032270324825965932, 0.0032278330364250716, 0.00322863359025355, 0.003229434144082028, 0.0032302346979105065, 0.003231035251738985, 0.003231835805567463, 0.0032326363593959415, 0.00323343691322442, 0.003234237467052898, 0.0032350380208813765, 0.003235838574709855, 0.003236639128538333, 0.0032374396823668114, 0.0032382402361952898, 0.003239040790023768, 0.0032398413438522464, 0.0032406418976807247, 0.003241442451509203, 0.0032422430053376814, 0.0032430435591661597, 0.003243844112994638, 0.0032446446668231163, 0.0032454452206515947, 0.003246245774480073, 0.0032470463283085513, 0.0032478468821370296, 0.003248647435965508, 0.0032494479897939863, 0.0032502485436224646, 0.003251049097450943, 0.0032518496512794213, 0.0032526502051078996, 0.003253450758936378, 0.0032542513127648562, 0.0032550518665933345, 0.003255852420421813, 0.003256652974250291, 0.0032574535280787695, 0.003258254081907248, 0.003259054635735726, 0.0032598551895642045, 0.003260655743392683, 0.003261456297221161, 0.0032622568510496395, 0.0032630574048781178, 0.003263857958706596, 0.0032646585125350744, 0.0032654590663635527, 0.003266259620192031, 0.0032670601740205094, 0.0032678607278489877, 0.003268661281677466, 0.0032694618355059444, 0.0032702623893344227, 0.003271062943162901, 0.0032718634969913793, 0.0032726640508198577, 0.003273464604648336, 0.0032742651584768143, 0.0032750657123052926, 0.003275866266133771, 0.0032766668199622493, 0.0032774673737907276, 0.003278267927619206, 0.0032790684814476842, 0.0032798690352761626, 0.003280669589104641, 0.003281470142933119, 0.0032822706967615975, 0.003283071250590076, 0.003283871804418554, 0.0032846723582470325, 0.003285472912075511, 0.003286273465903989, 0.0032870740197324675, 0.003287874573560946, 0.003288675127389424, 0.0032894756812179024, 0.0032902762350463808, 0.003291076788874859, 0.0032918773427033374, 0.0032926778965318157, 0.003293478450360294, 0.0032942790041887724, 0.0032950795580172507, 0.003295880111845729, 0.0032966806656742073, 0.0032974812195026857, 0.003298281773331164, 0.0032990823271596423, 0.0032998828809881206, 0.003300683434816599, 0.0033014839886450773, 0.0033022845424735556, 0.003303085096302034, 0.0033038856501305123, 0.0033046862039589906, 0.003305486757787469, 0.0033062873116159472, 0.0033070878654444255, 0.003307888419272904, 0.003308688973101382, 0.0033094895269298605, 0.003310290080758339, 0.003311090634586817, 0.0033118911884152955, 0.003312691742243774, 0.003313492296072252, 0.0033142928499007305, 0.0033150934037292088, 0.003315893957557687, 0.0033166945113861654, 0.0033174950652146437, 0.003318295619043122, 0.0033190961728716004, 0.0033198967267000787, 0.003320697280528557, 0.0033214978343570354, 0.0033222983881855137, 0.003323098942013992, 0.0033238994958424703, 0.0033247000496709487, 0.003325500603499427, 0.0033263011573279053, 0.0033271017111563836, 0.003327902264984862, 0.0033287028188133403, 0.0033295033726418186, 0.003330303926470297, 0.0033311044802987752, 0.0033319050341272536, 0.003332705587955732, 0.00333350614178421, 0.0033343066956126885, 0.003335107249441167, 0.003335907803269645, 0.0033367083570981235, 0.003337508910926602, 0.00333830946475508, 0.0033391100185835585, 0.003339910572412037, 0.003340711126240515, 0.0033415116800689934, 0.0033423122338974718, 0.00334311278772595, 0.0033439133415544284, 0.0033447138953829067, 0.003345514449211385, 0.0033463150030398634, 0.0033471155568683417, 0.00334791611069682, 0.0033487166645252983, 0.0033495172183537767, 0.003350317772182255, 0.0033511183260107333, 0.0033519188798392116, 0.00335271943366769, 0.0033535199874961683, 0.0033543205413246466, 0.003355121095153125, 0.0033559216489816033, 0.0033567222028100816, 0.00335752275663856, 0.0033583233104670382, 0.0033591238642955165, 0.003359924418123995, 0.003360724971952473, 0.0033615255257809515, 0.00336232607960943, 0.003363126633437908, 0.0033639271872663865, 0.003364727741094865, 0.003365528294923343, 0.0033663288487518215, 0.0033671294025802998, 0.003367929956408778, 0.0033687305102372564, 0.0033695310640657348, 0.003370331617894213, 0.0033711321717226914, 0.0033719327255511697, 0.003372733279379648, 0.0033735338332081264, 0.0033743343870366047, 0.003375134940865083, 0.0033759354946935613, 0.0033767360485220397, 0.003377536602350518, 0.0033783371561789963, 0.0033791377100074746, 0.003379938263835953, 0.0033807388176644313, 0.0033815393714929096, 0.003382339925321388, 0.0033831404791498662, 0.0033839410329783446, 0.003384741586806823, 0.003385542140635301, 0.0033863426944637795, 0.003387143248292258, 0.003387943802120736, 0.0033887443559492145, 0.003389544909777693, 0.003390345463606171, 0.0033911460174346495, 0.003391946571263128, 0.003392747125091606, 0.0033935476789200844, 0.0033943482327485628, 0.003395148786577041, 0.0033959493404055194, 0.0033967498942339977, 0.003397550448062476, 0.0033983510018909544, 0.0033991515557194327, 0.003399952109547911, 0.0034007526633763894, 0.0034015532172048677, 0.003402353771033346, 0.0034031543248618243, 0.0034039548786903026, 0.003404755432518781, 0.0034055559863472593, 0.0034063565401757376, 0.003407157094004216, 0.0034079576478326943, 0.0034087582016611726, 0.003409558755489651, 0.0034103593093181292, 0.0034111598631466076, 0.003411960416975086, 0.003412760970803564, 0.0034135615246320425, 0.003414362078460521, 0.003415162632288999, 0.0034159631861174775, 0.003416763739945956, 0.003417564293774434, 0.0034183648476029125, 0.003419165401431391, 0.003419965955259869, 0.0034207665090883474, 0.0034215670629168258, 0.003422367616745304, 0.0034231681705737824, 0.0034239687244022607, 0.003424769278230739, 0.0034255698320592174, 0.0034263703858876957, 0.003427170939716174, 0.0034279714935446523, 0.0034287720473731307, 0.003429572601201609, 0.0034303731550300873, 0.0034311737088585656, 0.003431974262687044, 0.0034327748165155223, 0.0034335753703440006, 0.003434375924172479, 0.0034351764780009572, 0.0034359770318294356, 0.003436777585657914, 0.003437578139486392, 0.0034383786933148705, 0.003439179247143349, 0.003439979800971827, 0.0034407803548003055, 0.003441580908628784, 0.003442381462457262, 0.0034431820162857405, 0.003443982570114219, 0.003444783123942697, 0.0034455836777711754, 0.0034463842315996538, 0.003447184785428132, 0.0034479853392566104, 0.0034487858930850887, 0.003449586446913567, 0.0034503870007420454, 0.0034511875545705237, 0.003451988108399002, 0.0034527886622274804, 0.0034535892160559587, 0.003454389769884437, 0.0034551903237129153, 0.0034559908775413936, 0.003456791431369872, 0.0034575919851983503, 0.0034583925390268286, 0.003459193092855307, 0.0034599936466837853, 0.0034607942005122636, 0.003461594754340742, 0.0034623953081692202, 0.0034631958619976986, 0.003463996415826177, 0.003464796969654655, 0.0034655975234831335, 0.003466398077311612, 0.00346719863114009, 0.0034679991849685685, 0.003468799738797047, 0.003469600292625525, 0.0034704008464540035, 0.003471201400282482, 0.00347200195411096, 0.0034728025079394384, 0.0034736030617679168, 0.003474403615596395, 0.0034752041694248734, 0.0034760047232533517, 0.00347680527708183, 0.0034776058309103084, 0.0034784063847387867, 0.003479206938567265, 0.0034800074923957433, 0.0034808080462242217, 0.0034816086000527, 0.0034824091538811783, 0.0034832097077096566, 0.003484010261538135, 0.0034848108153666133, 0.0034856113691950916, 0.00348641192302357, 0.0034872124768520482, 0.0034880130306805266, 0.003488813584509005, 0.0034896141383374832, 0.0034904146921659615, 0.00349121524599444, 0.003492015799822918, 0.0034928163536513965, 0.003493616907479875, 0.003494417461308353, 0.0034952180151368315, 0.00349601856896531, 0.003496819122793788, 0.0034976196766222665, 0.0034984202304507448, 0.003499220784279223, 0.0035000213381077014, 0.0035008218919361797, 0.003501622445764658, 0.0035024229995931364, 0.0035032235534216147, 0.003504024107250093, 0.0035048246610785714, 0.0035056252149070497, 0.003506425768735528, 0.0035072263225640063, 0.0035080268763924847, 0.003508827430220963, 0.0035096279840494413, 0.0035104285378779196, 0.003511229091706398, 0.0035120296455348763, 0.0035128301993633546, 0.003513630753191833, 0.0035144313070203112, 0.0035152318608487896, 0.003516032414677268, 0.003516832968505746, 0.0035176335223342245, 0.003518434076162703, 0.003519234629991181, 0.0035200351838196595, 0.003520835737648138, 0.003521636291476616, 0.0035224368453050945, 0.003523237399133573, 0.003524037952962051, 0.0035248385067905294, 0.0035256390606190078, 0.003526439614447486, 0.0035272401682759644, 0.0035280407221044427, 0.003528841275932921, 0.0035296418297613994, 0.0035304423835898777, 0.003531242937418356, 0.0035320434912468343, 0.0035328440450753127, 0.003533644598903791, 0.0035344451527322693, 0.0035352457065607476, 0.003536046260389226, 0.0035368468142177043, 0.0035376473680461826, 0.003538447921874661, 0.0035392484757031393, 0.0035400490295316176, 0.003540849583360096, 0.0035416501371885742, 0.0035424506910170525, 0.003543251244845531, 0.003544051798674009, 0.0035448523525024875, 0.003545652906330966, 0.003546453460159444, 0.0035472540139879225, 0.003548054567816401, 0.003548855121644879, 0.0035496556754733575, 0.0035504562293018358, 0.003551256783130314, 0.0035520573369587924, 0.0035528578907872707, 0.003553658444615749, 0.0035544589984442274, 0.0035552595522727057, 0.003556060106101184, 0.0035568606599296624, 0.0035576612137581407, 0.003558461767586619, 0.0035592623214150973, 0.0035600628752435757, 0.003560863429072054, 0.0035616639829005323, 0.0035624645367290106, 0.003563265090557489, 0.0035640656443859673, 0.0035648661982144456, 0.003565666752042924, 0.0035664673058714022, 0.0035672678596998806, 0.003568068413528359, 0.003568868967356837, 0.0035696695211853155, 0.003570470075013794, 0.003571270628842272, 0.0035720711826707505, 0.003572871736499229, 0.003573672290327707, 0.0035744728441561855, 0.003575273397984664, 0.003576073951813142, 0.0035768745056416204, 0.0035776750594700988, 0.003578475613298577, 0.0035792761671270554, 0.0035800767209555337, 0.003580877274784012, 0.0035816778286124904, 0.0035824783824409687, 0.003583278936269447, 0.0035840794900979253, 0.0035848800439264037, 0.003585680597754882, 0.0035864811515833603, 0.0035872817054118386, 0.003588082259240317, 0.0035888828130687953, 0.0035896833668972736, 0.003590483920725752, 0.0035912844745542303, 0.0035920850283827086, 0.003592885582211187, 0.0035936861360396652, 0.0035944866898681435, 0.003595287243696622, 0.0035960877975251, 0.0035968883513535785, 0.003597688905182057, 0.003598489459010535, 0.0035992900128390135, 0.003600090566667492, 0.00360089112049597, 0.0036016916743244485, 0.0036024922281529268, 0.003603292781981405, 0.0036040933358098834, 0.0036048938896383617, 0.00360569444346684, 0.0036064949972953184, 0.0036072955511237967, 0.003608096104952275, 0.0036088966587807534, 0.0036096972126092317, 0.00361049776643771, 0.0036112983202661883, 0.0036120988740946667, 0.003612899427923145, 0.0036136999817516233, 0.0036145005355801016, 0.00361530108940858, 0.0036161016432370583, 0.0036169021970655366, 0.003617702750894015, 0.0036185033047224932, 0.0036193038585509716, 0.00362010441237945, 0.003620904966207928, 0.0036217055200364065, 0.003622506073864885, 0.003623306627693363, 0.0036241071815218415, 0.00362490773535032, 0.003625708289178798, 0.0036265088430072765, 0.003627309396835755, 0.003628109950664233, 0.0036289105044927114, 0.0036297110583211898, 0.003630511612149668, 0.0036313121659781464, 0.0036321127198066247, 0.003632913273635103, 0.0036337138274635814, 0.0036345143812920597, 0.003635314935120538, 0.0036361154889490164, 0.0036369160427774947, 0.003637716596605973, 0.0036385171504344513, 0.0036393177042629296, 0.003640118258091408, 0.0036409188119198863, 0.0036417193657483646, 0.003642519919576843, 0.0036433204734053213, 0.0036441210272337996, 0.003644921581062278, 0.0036457221348907562, 0.0036465226887192346, 0.003647323242547713, 0.003648123796376191, 0.0036489243502046695, 0.003649724904033148, 0.003650525457861626, 0.0036513260116901045, 0.003652126565518583, 0.003652927119347061, 0.0036537276731755395, 0.003654528227004018, 0.003655328780832496, 0.0036561293346609744, 0.0036569298884894528, 0.003657730442317931, 0.0036585309961464094, 0.0036593315499748877, 0.003660132103803366, 0.0036609326576318444, 0.0036617332114603227, 0.003662533765288801, 0.0036633343191172793, 0.0036641348729457577, 0.003664935426774236, 0.0036657359806027143, 0.0036665365344311926, 0.003667337088259671, 0.0036681376420881493, 0.0036689381959166276, 0.003669738749745106, 0.0036705393035735842, 0.0036713398574020626, 0.003672140411230541, 0.003672940965059019, 0.0036737415188874975, 0.003674542072715976, 0.003675342626544454, 0.0036761431803729325, 0.003676943734201411, 0.003677744288029889, 0.0036785448418583675, 0.003679345395686846, 0.003680145949515324, 0.0036809465033438024, 0.0036817470571722808, 0.003682547611000759, 0.0036833481648292374, 0.0036841487186577157, 0.003684949272486194, 0.0036857498263146724, 0.0036865503801431507, 0.003687350933971629, 0.0036881514878001074, 0.0036889520416285857, 0.003689752595457064, 0.0036905531492855423, 0.0036913537031140206, 0.003692154256942499, 0.0036929548107709773, 0.0036937553645994556, 0.003694555918427934, 0.0036953564722564123, 0.0036961570260848906, 0.003696957579913369, 0.0036977581337418472, 0.0036985586875703256, 0.003699359241398804, 0.003700159795227282, 0.0037009603490557605, 0.003701760902884239, 0.003702561456712717, 0.0037033620105411955, 0.003704162564369674, 0.003704963118198152, 0.0037057636720266305, 0.003706564225855109, 0.003707364779683587, 0.0037081653335120654, 0.0037089658873405438, 0.003709766441169022, 0.0037105669949975004, 0.0037113675488259787, 0.003712168102654457, 0.0037129686564829354, 0.0037137692103114137, 0.003714569764139892, 0.0037153703179683703, 0.0037161708717968487, 0.003716971425625327, 0.0037177719794538053, 0.0037185725332822836, 0.003719373087110762, 0.0037201736409392403, 0.0037209741947677186, 0.003721774748596197, 0.0037225753024246752, 0.0037233758562531536, 0.003724176410081632, 0.0037249769639101102, 0.0037257775177385885, 0.003726578071567067, 0.003727378625395545, 0.0037281791792240235, 0.003728979733052502, 0.00372978028688098, 0.0037305808407094585, 0.003731381394537937, 0.003732181948366415, 0.0037329825021948934, 0.0037337830560233718, 0.00373458360985185, 0.0037353841636803284, 0.0037361847175088067, 0.003736985271337285, 0.0037377858251657634, 0.0037385863789942417, 0.00373938693282272, 0.0037401874866511984, 0.0037409880404796767, 0.003741788594308155, 0.0037425891481366333, 0.0037433897019651117, 0.00374419025579359, 0.0037449908096220683, 0.0037457913634505466, 0.003746591917279025, 0.0037473924711075033, 0.0037481930249359816, 0.00374899357876446, 0.0037497941325929382, 0.0037505946864214166, 0.003751395240249895, 0.003752195794078373, 0.0037529963479068515, 0.00375379690173533, 0.003754597455563808, 0.0037553980093922865, 0.003756198563220765, 0.003756999117049243, 0.0037577996708777215, 0.0037586002247062, 0.003759400778534678, 0.0037602013323631564, 0.0037610018861916348, 0.003761802440020113, 0.0037626029938485914, 0.0037634035476770697, 0.003764204101505548, 0.0037650046553340264, 0.0037658052091625047, 0.003766605762990983, 0.0037674063168194613, 0.0037682068706479397, 0.003769007424476418, 0.0037698079783048963, 0.0037706085321333746, 0.003771409085961853, 0.0037722096397903313, 0.0037730101936188096, 0.003773810747447288, 0.0037746113012757663, 0.0037754118551042446, 0.003776212408932723, 0.0037770129627612012, 0.0037778135165896795, 0.003778614070418158, 0.003779414624246636, 0.0037802151780751145, 0.003781015731903593, 0.003781816285732071, 0.0037826168395605495, 0.003783417393389028, 0.003784217947217506, 0.0037850185010459845, 0.0037858190548744628, 0.003786619608702941, 0.0037874201625314194, 0.0037882207163598977, 0.003789021270188376, 0.0037898218240168544, 0.0037906223778453327, 0.003791422931673811, 0.0037922234855022894, 0.0037930240393307677, 0.003793824593159246, 0.0037946251469877243, 0.0037954257008162027, 0.003796226254644681, 0.0037970268084731593, 0.0037978273623016376, 0.003798627916130116, 0.0037994284699585943, 0.0038002290237870726, 0.003801029577615551, 0.0038018301314440292, 0.0038026306852725076, 0.003803431239100986, 0.003804231792929464, 0.0038050323467579425, 0.003805832900586421, 0.003806633454414899, 0.0038074340082433775, 0.003808234562071856, 0.003809035115900334, 0.0038098356697288125, 0.003810636223557291, 0.003811436777385769, 0.0038122373312142474, 0.0038130378850427258, 0.003813838438871204, 0.0038146389926996824, 0.0038154395465281607, 0.003816240100356639, 0.0038170406541851174, 0.0038178412080135957, 0.003818641761842074, 0.0038194423156705523, 0.0038202428694990307, 0.003821043423327509, 0.0038218439771559873, 0.0038226445309844656, 0.003823445084812944, 0.0038242456386414223, 0.0038250461924699006, 0.003825846746298379, 0.0038266473001268573, 0.0038274478539553356, 0.003828248407783814, 0.0038290489616122922, 0.0038298495154407705, 0.003830650069269249, 0.003831450623097727, 0.0038322511769262055, 0.003833051730754684, 0.003833852284583162, 0.0038346528384116405, 0.003835453392240119, 0.003836253946068597, 0.0038370544998970755, 0.0038378550537255538, 0.003838655607554032, 0.0038394561613825104, 0.0038402567152109887, 0.003841057269039467, 0.0038418578228679454, 0.0038426583766964237, 0.003843458930524902, 0.0038442594843533804, 0.0038450600381818587, 0.003845860592010337, 0.0038466611458388153, 0.0038474616996672937, 0.003848262253495772, 0.0038490628073242503, 0.0038498633611527286, 0.003850663914981207, 0.0038514644688096853, 0.0038522650226381636, 0.003853065576466642, 0.0038538661302951202, 0.0038546666841235986, 0.003855467237952077, 0.003856267791780555, 0.0038570683456090335, 0.003857868899437512, 0.00385866945326599, 0.0038594700070944685, 0.003860270560922947, 0.003861071114751425, 0.0038618716685799035, 0.003862672222408382, 0.00386347277623686, 0.0038642733300653384, 0.0038650738838938168, 0.003865874437722295, 0.0038666749915507734, 0.0038674755453792517, 0.00386827609920773, 0.0038690766530362084, 0.0038698772068646867, 0.003870677760693165, 0.0038714783145216434, 0.0038722788683501217, 0.0038730794221786, 0.0038738799760070783, 0.0038746805298355566, 0.003875481083664035, 0.0038762816374925133, 0.0038770821913209916, 0.00387788274514947, 0.0038786832989779483, 0.0038794838528064266, 0.003880284406634905, 0.0038810849604633832, 0.0038818855142918616, 0.00388268606812034, 0.003883486621948818, 0.0038842871757772965, 0.003885087729605775, 0.003885888283434253, 0.0038866888372627315, 0.00388748939109121, 0.003888289944919688, 0.0038890904987481665, 0.0038898910525766448, 0.003890691606405123, 0.0038914921602336014, 0.0038922927140620798, 0.003893093267890558, 0.0038938938217190364, 0.0038946943755475147, 0.003895494929375993, 0.0038962954832044714, 0.0038970960370329497, 0.003897896590861428, 0.0038986971446899063, 0.0038994976985183847, 0.003900298252346863, 0.0039010988061753413, 0.0039018993600038196, 0.003902699913832298, 0.0039035004676607763, 0.0039043010214892546, 0.003905101575317733, 0.0039059021291462112, 0.00390670268297469, 0.003907503236803168, 0.003908303790631646, 0.0039091043444601245, 0.003909904898288603, 0.003910705452117081, 0.0039115060059455595, 0.003912306559774038, 0.003913107113602516, 0.0039139076674309945, 0.003914708221259473, 0.003915508775087951, 0.0039163093289164294, 0.003917109882744908, 0.003917910436573386, 0.003918710990401864, 0.003919511544230343, 0.003920312098058821, 0.003921112651887299, 0.003921913205715778, 0.003922713759544256, 0.003923514313372734, 0.003924314867201213, 0.003925115421029691, 0.003925915974858169, 0.003926716528686648, 0.003927517082515126, 0.003928317636343604, 0.003929118190172083, 0.003929918744000561, 0.003930719297829039, 0.003931519851657518, 0.003932320405485996, 0.003933120959314474, 0.0039339215131429526, 0.003934722066971431, 0.003935522620799909, 0.0039363231746283875, 0.003937123728456866, 0.003937924282285344, 0.0039387248361138225, 0.003939525389942301, 0.003940325943770779, 0.0039411264975992575, 0.003941927051427736, 0.003942727605256214, 0.003943528159084692, 0.003944328712913171, 0.003945129266741649, 0.003945929820570127, 0.003946730374398606, 0.003947530928227084, 0.003948331482055562, 0.003949132035884041, 0.003949932589712519, 0.003950733143540997, 0.003951533697369476, 0.003952334251197954, 0.003953134805026432, 0.003953935358854911, 0.003954735912683389, 0.003955536466511867, 0.003956337020340346, 0.003957137574168824, 0.003957938127997302, 0.003958738681825781, 0.003959539235654259, 0.003960339789482737, 0.0039611403433112155, 0.003961940897139694, 0.003962741450968172, 0.0039635420047966505, 0.003964342558625129, 0.003965143112453607, 0.0039659436662820855, 0.003966744220110564, 0.003967544773939042, 0.0039683453277675204, 0.003969145881595999, 0.003969946435424477, 0.003970746989252955, 0.003971547543081434, 0.003972348096909912, 0.00397314865073839, 0.003973949204566869, 0.003974749758395347, 0.003975550312223825, 0.003976350866052304, 0.003977151419880782, 0.00397795197370926, 0.003978752527537739, 0.003979553081366217, 0.003980353635194695, 0.003981154189023174, 0.003981954742851652, 0.00398275529668013, 0.003983555850508609, 0.003984356404337087, 0.003985156958165565, 0.0039859575119940436, 0.003986758065822522, 0.003987558619651, 0.0039883591734794785, 0.003989159727307957, 0.003989960281136435, 0.0039907608349649135, 0.003991561388793392, 0.00399236194262187, 0.0039931624964503485, 0.003993963050278827, 0.003994763604107305, 0.0039955641579357834, 0.003996364711764262, 0.00399716526559274, 0.003997965819421218, 0.003998766373249697, 0.003999566927078175, 0.004000367480906653, 0.004001168034735132, 0.00400196858856361, 0.004002769142392088]

net_absorption_emission_protons_inv_s_inv_cm3_1emin4 = [0.0, -1.4598140225168276e+24, -3.3056035103064366e+24, -2.7756623027699933e+24, -2.737314189936762e+24, -1.399990349438045e+24, -3.283887247748693e+24, -4.829556922404012e+24, -4.57945918290882e+24, -4.236530656171168e+24, -3.041979805954168e+24, -4.842562989080349e+24, -6.127828995180693e+24, -6.074391236576928e+24, -5.686652367077419e+24, -4.6446384251577073e+24, -6.342855150810338e+24, -7.46119280819554e+24, -7.527142585417885e+24, -7.204532799571186e+24, -6.306707902921686e+24, -7.97560221018873e+24, -8.97196829145529e+24, -9.177798170904165e+24, -8.999717850576143e+24, -8.383200132761985e+24, -9.854279053275063e+24, -1.0711525940701489e+25, -1.102654958481141e+25, -1.1024104852556706e+25, -1.073414154663493e+25, -1.1964069042738265e+25, -1.2654606485994136e+25, -1.3085642322269569e+25, -1.3224942177128884e+25, -1.3190276141112294e+25, -1.417076771607216e+25, -1.4740646631978398e+25, -1.5164800742231465e+25, -1.5370629944625847e+25, -1.5457226777472435e+25, -1.6254137295477019e+25, -1.6723457926572044e+25, -1.7105544619108615e+25, -1.7365500414874483e+25, -1.749455636858123e+25, -1.821228003194991e+25, -1.866699197901755e+25, -1.9045438700193634e+25, -1.937618217275065e+25, -1.959957189176922e+25, -2.03423677810837e+25, -2.090776169338515e+25, -2.1392500147472306e+25, -2.1874717398834674e+25, -2.2289357556924954e+25, -2.3193719705413317e+25, -2.395991781116133e+25, -2.468636971444178e+25, -2.5452068613959363e+25, -2.6168311564079303e+25, -2.729491312525879e+25, -2.835269553441429e+25, -2.9376782452552357e+25, -3.047598063051684e+25, -3.158563037802819e+25, -3.2971279521748878e+25, -3.4324245610979034e+25, -3.57292261520836e+25, -3.7211441209694107e+25, -3.874695321514184e+25, -4.045552458331735e+25, -4.2217349034100265e+25, -4.408391315311218e+25, -4.597911793173871e+25, -4.795874838437925e+25, -5.0064973605445295e+25, -5.223119447422491e+25, -5.457987452334945e+25, -5.692265163057276e+25, -5.936394821989345e+25, -6.192080933440176e+25, -6.450515843448546e+25, -6.734531651539102e+25, -7.011252174219099e+25, -7.3029741109877235e+25, -7.608735213690098e+25, -7.913025872114986e+25, -8.240632932423037e+25, -8.56077266092684e+25, -8.888458495839482e+25, -9.237652938762694e+25, -9.5843811655973e+25, -9.95613380183419e+25, -1.0325487708810447e+26, -1.0704488761620347e+26, -1.1099249862925491e+26, -1.1504330557110703e+26, -1.1931139489963196e+26, -1.2368537173040432e+26, -1.2819221643006261e+26, -1.3290747009058775e+26, -1.3773227028859676e+26, -1.4278939636913472e+26, -1.4792335137305619e+26, -1.5327172330922838e+26, -1.5878891612061018e+26, -1.644458173508741e+26, -1.7029703375405124e+26, -1.762812360419435e+26, -1.824008198080833e+26, -1.887684907498469e+26, -1.9530377282366508e+26, -2.0207431162323905e+26, -2.0903097142336216e+26, -2.1617837787629957e+26, -2.236207470022355e+26, -2.3128820472745043e+26, -2.392459630531187e+26, -2.4749196058172612e+26, -2.5602971613643656e+26, -2.6484731279734423e+26, -2.7390560518734086e+26, -2.8324091106593317e+26, -2.9293431677032935e+26, -3.0285672047688714e+26, -3.1311803956517744e+26, -3.23694380869616e+26, -3.34641174538242e+26, -3.4595727946664815e+26, -3.5764419052636695e+26, -3.6973017035579846e+26, -3.822077861408406e+26, -3.951206415838484e+26, -4.085575520831836e+26, -4.224325017192286e+26, -4.367881481489278e+26, -4.516543053899831e+26, -4.67019857062191e+26, -4.829168295353772e+26, -4.993724362811449e+26, -5.163940916341471e+26, -5.339998300128048e+26, -5.522298693778846e+26, -5.709774989353818e+26, -5.903595005337398e+26, -6.1039647795312844e+26, -6.311140285079268e+26, -6.526283207589773e+26, -6.748219751676122e+26, -6.977685035478848e+26, -7.21544983517746e+26, -7.461314611841266e+26, -7.716052801296552e+26, -7.979246318809674e+26, -8.250620236328331e+26, -8.531481100397513e+26, -8.820529019375036e+26, -9.119897615571826e+26, -9.430672706414583e+26, -9.751207464566664e+26, -1.0082287752943761e+27, -1.0424689997387966e+27, -1.0778949915094673e+27, -1.1145008718618424e+27, -1.1524140731039251e+27, -1.1916183778494439e+27, -1.232152411471072e+27, -1.2741058915422956e+27, -1.317567559029409e+27, -1.3624977066152764e+27, -1.4089211341251427e+27, -1.4569092678328592e+27, -1.5065143974264546e+27, -1.557788084273077e+27, -1.6107989223522145e+27, -1.6656321244815753e+27, -1.7223338926979554e+27, -1.78103400854639e+27, -1.8416342307055315e+27, -1.9043240098662644e+27, -1.9691897727866623e+27, -2.0362464877483218e+27, -2.1055255875554492e+27, -2.1771943484964532e+27, -2.2513354325409564e+27, -2.3280138740857885e+27, -2.407279416370994e+27, -2.489188555069097e+27, -2.57381919705897e+27, -2.661363329629128e+27, -2.7519093269500545e+27, -2.845476460658963e+27, -2.942214834719885e+27, -3.042319040215546e+27, -3.145840262659604e+27, -3.252877041462201e+27, -3.363538208813858e+27, -3.477929394607708e+27, -3.596247408999404e+27, -3.7186121772809797e+27, -3.8450749034191866e+27, -3.9758095811821437e+27, -4.111041347504413e+27, -4.25087834536033e+27, -4.395412359154934e+27, -4.544846280101487e+27, -4.69933632396695e+27, -4.8591346066478816e+27, -5.024373570132651e+27, -5.195206961069223e+27, -5.371821524491487e+27, -5.554475009295835e+27, -5.743407753378964e+27, -5.938693410290741e+27, -6.1406278140278e+27, -6.349361225580378e+27, -6.565192146677608e+27, -6.788376125412175e+27, -7.019060229900763e+27, -7.257659039604917e+27, -7.504320866699669e+27, -7.759374191511053e+27, -8.023085277766289e+27, -8.295708596474703e+27, -8.577615018509212e+27, -8.869142353541751e+27, -9.170536607329152e+27, -9.482291345379876e+27, -9.804568543454129e+27, -1.0137792833182448e+28, -1.048231175699004e+28, -1.083852077073513e+28, -1.1206810955993701e+28, -1.1587687084364705e+28, -1.1981534151981336e+28, -1.238871297758022e+28, -1.280970893935713e+28, -1.3244906814910456e+28, -1.3694927592185296e+28, -1.4160315586467618e+28, -1.4641452878704565e+28, -1.5138967258756878e+28, -1.5653395974615826e+28, -1.6185292556466288e+28, -1.6735187231457252e+28, -1.7303856413558298e+28, -1.789182355319825e+28, -1.8499781486065125e+28, -1.9128333110673382e+28, -1.9778216757476046e+28, -2.045017802889381e+28, -2.114505160351939e+28, -2.186349184743771e+28, -2.2606324154993754e+28, -2.3374472197875917e+28, -2.4168764848209752e+28, -2.498992951778711e+28, -2.58389944690373e+28, -2.671690086082753e+28, -2.762466768348429e+28, -2.8563285350529485e+28, -2.953374328927963e+28, -3.053721363548204e+28, -3.1574694351092584e+28, -3.2647454194430312e+28, -3.375670778499434e+28, -3.490362462354977e+28, -3.6089491821037326e+28, -3.7315689096634543e+28, -3.858351882020312e+28, -3.98944963815959e+28, -4.124995239828346e+28, -4.265146533676457e+28, -4.410055212071542e+28, -4.559897189080971e+28, -4.7148202354628966e+28, -4.875008540348401e+28, -5.0406449305924855e+28, -5.211905606516394e+28, -5.3889821584038325e+28, -5.572079309949252e+28, -5.761398463615658e+28, -5.9571465987430315e+28, -6.159544379991017e+28, -6.36882177209389e+28, -6.5852136874165445e+28, -6.808953145055189e+28, -7.040297343588159e+28, -7.2794978099257075e+28, -7.52683055731215e+28, -7.7825660830405455e+28, -8.04698849404358e+28, -8.320391478019946e+28, -8.603086629187775e+28, -8.895389003677317e+28, -9.197623773713529e+28, -9.510123302572487e+28, -9.833247030409673e+28, -1.0167349419477017e+29, -1.0512798177140498e+29, -1.0869988931222707e+29, -1.1239315560923695e+29, -1.1621189557156557e+29, -1.201604033535435e+29, -1.2424306427841442e+29, -1.2846445079939507e+29, -1.3282925736775809e+29, -1.3734239739074141e+29, -1.4200885822517521e+29, -1.4683390490621932e+29, -1.518228843116101e+29, -1.569813864493973e+29, -1.6231515940350983e+29, -1.678301515697854e+29, -1.7353255065608436e+29, -1.7942871074385094e+29, -1.8552521487625772e+29, -1.918288589436267e+29, -1.9834670528626105e+29, -2.050859999648845e+29, -2.1205430755665207e+29, -2.1925937749524285e+29, -2.2670925329167486e+29, -2.344122791777531e+29, -2.4237706211108608e+29, -2.5061245111481145e+29, -2.5912767138079395e+29, -2.679322411914783e+29, -2.770359738132653e+29, -2.8644905129240548e+29, -2.961819707131406e+29, -3.0624559667220345e+29, -3.1665117390758578e+29, -3.274103369726369e+29, -3.385350757378707e+29, -3.500378271452244e+29, -3.619314254125893e+29, -3.7422915487225494e+29, -3.869447443247254e+29, -4.0009242082066785e+29, -4.1368684552516514e+29, -4.2774318104700476e+29, -4.4227714944124305e+29, -4.573049693477726e+29, -4.728434260369604e+29, -4.8890986593714004e+29, -5.055222353649323e+29, -5.226990833595003e+29, -5.404595793736825e+29, -5.5882358939878034e+29, -5.778115875847194e+29, -5.974447752747169e+29, -6.177451087639485e+29, -6.387352244212932e+29, -6.60438581724267e+29, -6.828794030465308e+29, -7.060827541063696e+29, -7.300745483696693e+29, -7.548815673568316e+29, -7.805315355184332e+29, -8.070530661311495e+29, -8.344757967279038e+29, -8.62830330767317e+29, -8.921483504290944e+29, -9.224625880475702e+29, -9.538068864473535e+29, -9.86216262658048e+29, -1.0197269059789787e+30, -1.054376221868042e+30, -1.09020292234476e+30, -1.127247002869359e+30, -1.1655498389747538e+30, -1.2051541903440066e+30, -1.2461043000748394e+30, -1.2884458922840412e+30, -1.3322262389404896e+30, -1.3774942335882803e+30, -1.4243004320217556e+30, -1.4726971009674607e+30, -1.52273827847594e+30, -1.574479863724743e+30, -1.6279796206691202e+30, -1.683297296130792e+30, -1.7404946643613447e+30, -1.7996356078625028e+30, -1.8607861445255358e+30, -1.924014588085565e+30, -1.9893915420253684e+30, -2.0569899935183343e+30, -2.1268854494851654e+30, -2.1991559568565195e+30, -2.2738822135179865e+30, -2.35114768401368e+30, -2.4310386315736962e+30, -2.5136442796956095e+30, -2.599056885965933e+30, -2.6873718278507915e+30, -2.7786877168671266e+30, -2.873106534814318e+30, -2.9707337107665463e+30, -3.0716782850134254e+30, -3.176052964742395e+30, -3.2839743204552713e+30, -3.395562856847139e+30, -3.510943200365818e+30, -3.6302441864319994e+30, -3.753599049541469e+30, -3.8811455436337725e+30, -4.0130260954212913e+30, -4.1493879612722675e+30, -4.2903834535087857e+30, -4.4361700027546126e+30, -4.5869104109633577e+30, -4.742773020870593e+30, -4.903931900284648e+30, -5.0705669906292866e+30, -5.242864393873255e+30, -5.421016500432781e+30, -5.605222292364081e+30, -5.795687439890893e+30, -5.992624655629341e+30, -6.196253862548877e+30, -6.406802461388004e+30, -6.624505565790509e+30, -6.84960628728027e+30, -7.082356012358222e+30, -7.323014659581033e+30, -7.571850966982807e+30, -7.829142841836238e+30, -8.095177572857239e+30, -8.370252259211248e+30, -8.654674102290437e+30, -8.948760694113716e+30, -9.2528404664145e+30, -9.567252991852039e+30, -9.892349381401626e+30, -1.0228492662280114e+31, -1.0576058234946626e+31, -1.0935434219422968e+31, -1.1307021955272474e+31, -1.1691236385095002e+31, -1.208850658442286e+31, -1.2499276192605287e+31, -1.292400392412911e+31, -1.3363164083213674e+31, -1.3817247104286132e+31, -1.428676004420455e+31, -1.477222724509496e+31, -1.527419082345419e+31, -1.5793211342762257e+31, -1.6329868406802878e+31, -1.6884761313514226e+31, -1.7458509721865508e+31, -1.8051754349466601e+31, -1.8665157680428243e+31, -1.9299404724993386e+31, -1.9955203763106699e+31, -2.0633287131395917e+31, -2.133441206717648e+31, -2.205936154614746e+31, -2.2808945105077756e+31, -2.3583999849411057e+31, -2.4385391293050587e+31, -2.5214014354438973e+31, -2.6070794376045562e+31, -2.6956688148272653e+31, -2.7872684980744584e+31, -2.881980775516883e+31, -2.9799114150631404e+31, -3.0811697777513237e+31, -3.1858689404773677e+31, -3.294125824626219e+31, -3.4060613203842037e+31, -3.5218004276470436e+31, -3.6414723926041357e+31, -3.765210858759434e+31, -3.893154004093004e+31, -4.0254447037706e+31, -4.162230688689833e+31, -4.303664709784303e+31, -4.449904707485928e+31, -4.601113990450767e+31, -4.757461412932254e+31, -4.919121571401946e+31, -5.086274991394862e+31, -5.259108334347794e+31, -5.43781460381696e+31, -5.622593361922472e+31, -5.813650950201356e+31, -6.0112007214236845e+31, -6.215463282230888e+31, -6.426666728056316e+31, -6.645046911728595e+31, -6.870847696400204e+31, -7.104321230271351e+31, -7.345728234034129e+31, -7.595338279239239e+31, -7.853430106624049e+31, -8.120291918526439e+31, -8.396221718872895e+31, -8.681527631386661e+31, -8.97652824925557e+31, -9.28155299444716e+31, -9.596942480801188e+31, -9.923048891276844e+31, -1.0260236379354876e+32, -1.0608881473173686e+32, -1.0969373489560285e+32, -1.1342114973896548e+32, -1.1727522153740813e+32, -1.2126025393194148e+32, -1.2538069682927415e+32, -1.2964115126823038e+32, -1.3404637472137972e+32, -1.3860128619573742e+32, -1.4331097189201979e+32, -1.4818069078498854e+32, -1.5321588054898169e+32, -1.5842216360561574e+32, -1.6380535338502532e+32, -1.6937146086116294e+32, -1.751267012036556e+32, -1.810775007558281e+32, -1.8723050418991082e+32, -1.9359258193615633e+32, -2.0017083782527757e+32, -2.0697261707260454e+32, -2.140055143934335e+32, -2.2127738254890563e+32, -2.2879634109637906e+32, -2.365707854066512e+32, -2.446093961143134e+32, -2.529211486999114e+32, -2.6151532358684605e+32, -2.7040151650951573e+32, -2.7958964913844257e+32, -2.8908998019609335e+32, -2.9891311696123984e+32, -3.0907002697933987e+32, -3.1957205042428066e+32, -3.304309126555514e+32, -3.4165873738877846e+32, -3.532680601153129e+32, -3.65271842192896e+32, -3.776834852070465e+32, -3.905168460427537e+32, -4.037862521782064e+32, -4.175065178316801e+32, -4.3169296040562236e+32, -4.4636141752030075e+32, -4.6152826479909985e+32, -4.772104340187925e+32, -4.934254320587221e+32, -5.101913604299153e+32, -5.275269353715793e+32, -5.4545150884206694e+32, -5.63985090000921e+32, -5.831483675449243e+32, -6.029627328368324e+32, -6.234503036580159e+32, -6.446339488992199e+32, -6.665373140654472e+32, -6.891848476142077e+32, -7.126018282192427e+32, -7.368143929225354e+32, -7.618495662680555e+32, -7.87735290424737e+32, -8.145004562844105e+32, -8.421749357065616e+32, -8.707896147675486e+32, -9.003764281357508e+32, -9.309683946904834e+32, -9.625996542501267e+32, -9.95305505632476e+32, -1.0291224459110974e+33, -1.0640882111335209e+33, -1.100241818224627e+33, -1.137623608482202e+33, -1.176275292494749e+33, -1.216239996537175e+33, -1.2575623105163618e+33, -1.300288337671382e+33, -1.3444657458141727e+33, -1.3901438203274425e+33, -1.4373735190457792e+33, -1.4862075288698736e+33, -1.536700324401417e+33, -1.5889082284267328e+33, -1.6428894745965898e+33, -1.698704272093992e+33, -1.7564148726092405e+33, -1.8160856394053754e+33, -1.877783118893832e+33, -1.9415761145111838e+33, -2.0075357631070772e+33, -2.0757356139380144e+33, -2.1462517103034535e+33, -2.2191626739478107e+33, -2.294549792250454e+33, -2.372497108398642e+33, -2.4530915146613127e+33, -2.536422848656818e+33, -2.6225839929200966e+33, -2.711670977929646e+33, -2.803783088407724e+33, -2.8990229733013977e+33, -2.9974967594973706e+33, -3.0993141692042347e+33, -3.204588641500378e+33, -3.3134374576964105e+33, -3.4259818711462705e+33, -3.5423472412035135e+33, -3.6626631718552455e+33, -3.7870636548501503e+33, -3.9156872176851045e+33, -4.048677076554592e+33, -4.186181294342204e+33, -4.328352943964301e+33, -4.475350277219773e+33, -4.627336899053576e+33, -4.784481947951218e+33, -4.94696028200343e+33, -5.1149526715670063e+33, -5.28864599788341e+33, -5.468233458710001e+33, -5.653914780528504e+33, -5.845896437977552e+33, -6.04439188038788e+33, -6.249621766005783e+33, -6.46181420383353e+33, -6.681205003668776e+33, -6.908037934113925e+33, -7.142564989466526e+33, -7.385046665105305e+33, -7.635752242222312e+33, -7.894960081677129e+33, -8.162957927823133e+33, -8.440043221990272e+33, -8.726523426601964e+33, -9.022716359561415e+33, -9.32895053996559e+33, -9.645565544817953e+33, -9.972912377589014e+33, -1.031135384871764e+34, -1.0661264968500892e+34, -1.1023033352802823e+34, -1.139705964197332e+34, -1.1783757933173093e+34, -1.218355622700938e+34, -1.2596896888237647e+34, -1.3024237121618716e+34, -1.3466049462903053e+34, -1.3922822285745318e+34, -1.4395060324693356e+34, -1.4883285215143594e+34, -1.5388036050313013e+34, -1.5909869956046964e+34, -1.644936268396397e+34, -1.7007109223297438e+34, -1.7583724432111993e+34, -1.817984368846929e+34, -1.879612356200637e+34, -1.9433242506641153e+34, -2.0091901574943195e+34, -2.077282515463057e+34, -2.14767617281214e+34, -2.2204484655551713e+34, -2.295679298185253e+34, -2.373451226880893e+34, -2.4538495452400174e+34, -2.536962372648993e+34, -2.622880745319169e+34, -2.711698710085665e+34, -2.8035134210279836e+34, -2.8984252389817715e+34, -2.9965378340247474e+34, -3.0979582909929095e+34, -3.202797218115953e+34, -3.3111688588368386e+34, -3.4231912068928003e+34, -3.5389861247292733e+34, -3.658679465320237e+34, -3.7824011974803713e+34, -3.910285534724684e+34, -4.0424710677633907e+34, -4.179100900699355e+34, -4.320322791000581e+34, -4.466289293323198e+34, -4.617157907239377e+34, -4.773091228956797e+34, -4.934257107091101e+34, -5.100828802537522e+34, -5.272985152525449e+34, -5.450910738899851e+34, -5.634796060685993e+34, -5.824837710989389e+34, -6.021238558268825e+34, -6.224207932032167e+34, -6.433961812990141e+34, -6.650723027674028e+34, -6.8747214475683325e+34, -7.1061941927489795e+34, -7.345385840042533e+34, -7.5925486356942485e+34, -7.847942712537009e+34, -8.111836311618058e+34, -8.38450600826661e+34, -8.666236942526333e+34, -8.957323053899143e+34, -9.258067320318668e+34, -9.568782001226968e+34, -9.889788884665274e+34, -1.0221419538218543e+35, -1.0564015563654731e+35, -1.0917928855070107e+35, -1.128352186032656e+35, -1.166116784555369e+35, -1.2051251162414115e+35, -1.2454167517860348e+35, -1.2870324246025521e+35, -1.3300140581887712e+35, -1.3744047936269438e+35, -1.4202490171747974e+35, -1.4675923878938717e+35, -1.516481865263113e+35, -1.5669657367140399e+35, -1.6190936450259977e+35, -1.6729166155040733e+35, -1.7284870828665466e+35, -1.785858917753372e+35, -1.8450874527668903e+35, -1.9062295079411388e+35, -1.969343415537257e+35, -2.0344890440428837e+35, -2.1017278212561993e+35, -2.171122756315663e+35, -2.2427384605333325e+35, -2.316641166874058e+35, -2.3928987479157637e+35, -2.4715807321122767e+35, -2.5527583181663876e+35, -2.63650438731063e+35, -2.722893513278045e+35, -2.81200196973007e+35, -2.9039077348962853e+35, -2.9986904931620853e+35, -3.0964316333260044e+35, -3.1972142432314322e+35, -3.301123100458657e+35, -3.40824465874597e+35, -3.518667029789195e+35, -3.632479960049823e+35, -3.749774802182401e+35, -3.8706444806691785e+35, -3.995183451232605e+35, -4.123487653570269e+35, -4.2556544569408e+35, -4.391782598101287e+35, -4.531972111079033e+35, -4.6763242482369746e+35, -4.824941392069352e+35, -4.977926957143738e+35, -5.135385281583158e+35, -5.2974215074631405e+35, -5.464141449476383e+35, -5.635651451202357e+35, -5.812058228298099e+35, -5.9934686979163605e+35, -6.179989793638337e+35, -6.371728265201758e+35, -6.5687904622956685e+35, -6.77128210168836e+35, -6.979308016956626e+35, -7.192971890082665e+35, -7.412375964204457e+35, -7.637620736806436e+35, -7.868804632669243e+35, -8.106023655921752e+35, -8.34937102056961e+35, -8.598936758927937e+35, -8.854807307430967e+35, -9.117065069362303e+35, -9.38578795412115e+35, -9.661048892729396e+35, -9.942915329384272e+35, -1.0231448688980404e+36, -1.052670382064955e+36, -1.082872841752053e+36, -1.1137562413058186e+36, -1.1453237354530948e+36, -1.1775775754348516e+36, -1.2105190420241072e+36, -1.2441483765484345e+36, -1.2784647100644032e+36, -1.313465990859106e+36, -1.3491489104846868e+36, -1.3855088285641628e+36, -1.422539696641681e+36, -1.460233981386802e+36, -1.4985825875012131e+36, -1.5375747807160867e+36, -1.5771981113108077e+36, -1.6174383386261702e+36, -1.6582793570896853e+36, -1.699703124315043e+36, -1.7416895918827125e+36, -1.784216639452963e+36, -1.8272600129061998e+36, -1.8707932672472823e+36, -1.9147877150502628e+36, -1.959212381256252e+36, -2.004033965169667e+36, -2.0492168105255373e+36, -2.0947228845224757e+36, -2.1405117667299347e+36, -2.1865406487858452e+36, -2.232764345797084e+36, -2.2791353203437562e+36, -2.3256037199629877e+36, -2.372117428952579e+36, -2.418622135284652e+36, -2.465061413356138e+36, -2.511376823224464e+36, -2.5575080268831713e+36, -2.603392922023095e+36, -2.648967793599951e+36, -2.694167483389301e+36, -2.738925577554708e+36, -2.7831746120866285e+36, -2.826846295787772e+36, -2.869871750288623e+36, -2.912181766374924e+36, -2.953707075700991e+36, -2.994378636749881e+36, -3.0341279336879144e+36, -3.07288728654918e+36, -3.1105901709785155e+36, -3.1471715455638944e+36, -3.1825681846022807e+36, -3.21671901397184e+36, -3.2495654476307335e+36, -3.281051722130014e+36, -3.3111252264191693e+36, -3.339736824138721e+36, -3.36684116553582e+36, -3.3923969861072066e+36, -3.4163673890681995e+36, -3.438720108766781e+36, -3.4594277522056475e+36, -3.478468015901557e+36, -3.495823875397243e+36, -3.511483744844638e+36, -3.525441604196436e+36, -3.5376970916746506e+36, -3.54825555932775e+36, -3.557128089642649e+36, -3.5643314713449444e+36, -3.569888132703384e+36, -3.573826030856676e+36, -3.576178495910434e+36, -3.5769840288159496e+36, -3.576286052353976e+36, -3.574132614915248e+36, -3.5705760472104886e+36, -3.565672572567649e+36, -3.559481872096735e+36, -3.552066606731403e+36, -3.543491898997606e+36, -3.5338247783137067e+36, -3.52313359468457e+36, -3.511487406799749e+36, -3.498955351753541e+36, -3.485606004836528e+36, -3.471506739052095e+36, -3.4567230951285005e+36, -3.441318173755524e+36, -3.4253520625000273e+36, -3.40888131026698e+36, -3.3919584621961486e+36, -3.374631667453422e+36, -3.3569443714358615e+36, -3.3389351024327017e+36, -3.320637360766248e+36, -3.302079615907435e+36, -3.2832854140827186e+36, -3.264273595563698e+36, -3.245058617285487e+36, -3.225650972834803e+36, -3.2060576983562246e+36, -3.1862829497277267e+36, -3.1663286336285754e+36, -3.146195073021539e+36, -3.1258816862215505e+36, -3.105387658210523e+36, -3.0847125832188285e+36, -3.063857058815496e+36, -3.042823213766744e+36, -3.021615154629263e+36, -3.0002393192948676e+36, -2.978704729328174e+36, -2.957023136755985e+36, -2.9352090647925716e+36, -2.913279745648448e+36, -2.891254961919681e+36, -2.869156800967958e+36, -2.8470093340878623e+36, -2.82483823406092e+36, -2.802670345892002e+36, -2.780533226119315e+36, -2.7584546661172095e+36, -2.736462214323222e+36, -2.7145827113847275e+36, -2.692841850914241e+36, -2.6712637769451596e+36, -2.64987072737532e+36, -2.6286827307515685e+36, -2.6077173617574112e+36, -2.586989558783954e+36, -2.566511505047961e+36, -2.546292572918205e+36, -2.5263393294608425e+36, -2.506655599746602e+36, -2.4872425831989666e+36, -2.468099017216572e+36, -2.4492213814843237e+36, -2.4306041357950213e+36, -2.4122399838351858e+36, -2.3941201552356823e+36, -2.3762346982380232e+36, -2.358572775565009e+36, -2.341122956491237e+36, -2.323873498664094e+36, -2.3068126139060268e+36, -2.289928713009923e+36, -2.273210625395156e+36, -2.256647790397052e+36, -2.2402304178901543e+36, -2.2239496168708386e+36, -2.2077974915232944e+36, -2.1917672051420127e+36, -2.1758530130645028e+36, -2.160050266462295e+36, -2.14435538943404e+36, -2.1287658323308186e+36, -2.1132800046157348e+36, -2.0978971908141975e+36, -2.0826174532513975e+36, -2.0674415253017932e+36, -2.052370698802421e+36, -2.0374067091156834e+36, -2.0225516210824048e+36, -2.0078077187938562e+36, -1.993177401748689e+36, -1.9786630895603112e+36, -1.964267136957805e+36, -1.9499917603921836e+36, -1.9358389771323358e+36, -1.921810557322861e+36, -1.9079079890889195e+36, -1.8941324564190733e+36, -1.8804848292424596e+36, -1.8669656648458192e+36, -1.8535752195522342e+36, -1.840313469407558e+36, -1.8271801384928982e+36, -1.8141747334009544e+36, -1.8012965823770816e+36, -1.7885448776300675e+36, -1.7759187193594505e+36, -1.7634171601184572e+36, -1.7510392482337023e+36, -1.738784069124365e+36, -1.7266507835051874e+36, -1.7146386616092037e+36, -1.7027471127261636e+36, -1.690975709516004e+36, -1.6793242067182152e+36, -1.6677925540354696e+36, -1.656380903119474e+36, -1.6450896087263373e+36, -1.63391922423514e+36, -1.6228704918366384e+36, -1.6119443277959104e+36, -1.6011418032744196e+36, -1.5904641212622066e+36, -1.5799125902200206e+36, -1.569488595064392e+36, -1.5591935661471463e+36, -1.5490289468850675e+36, -1.5389961606866542e+36, -1.5290965778027965e+36, -1.5193314826972448e+36, -1.5097020424936437e+36, -1.500209277008917e+36, -1.4908540308303826e+36, -1.4816369478366917e+36, -1.472558448502868e+36, -1.463618710267855e+36, -1.4548176511808184e+36, -1.4461549169807046e+36, -1.4376298717037768e+36, -1.4292415918559873e+36, -1.4209888641330917e+36, -1.412870186620871e+36, -1.4048837733615645e+36, -1.39702756213143e+36, -1.3892992252379626e+36, -1.38169618311395e+36, -1.3742156204596106e+36, -1.3668545046631267e+36, -1.359609606214388e+36, -1.352477520815345e+36, -1.3454546928849072e+36, -1.338537440154103e+36, -1.3317219790494226e+36, -1.3250044505689159e+36, -1.3183809463641801e+36, -1.3118475347546543e+36, -1.3054002864148481e+36, -1.2990352994928725e+36, -1.2927487239372643e+36, -1.286536784829725e+36, -1.280395804542766e+36, -1.2743222235633987e+36, -1.2683126198467862e+36, -1.2623637265861185e+36, -1.2564724483076015e+36, -1.2506358752216537e+36, -1.2448512957819864e+36, -1.2391162074257944e+36, -1.2334283254863813e+36, -1.2277855902888487e+36, -1.2221861724554776e+36, -1.2166284764633466e+36, -1.2111111425105006e+36, -1.2056330467593954e+36, -1.2001933000372715e+36, -1.194791245082345e+36, -1.1894264524326914e+36, -1.1840987150606496e+36, -1.17880804186079e+36, -1.1735546501026571e+36, -1.1683389569617242e+36, -1.1631615702430048e+36, -1.1580232784112027e+36, -1.152925040040481e+36, -1.1478679727940788e+36, -1.1428533420413052e+36, -1.1378825492156242e+36, -1.1329571200125265e+36, -1.1280786925214238e+36, -1.1232490053799128e+36, -1.1184698860328534e+36, -1.1137432391727582e+36, -1.1090710354312958e+36, -1.1044553003856081e+36, -1.0998981039361596e+36, -1.0954015501067066e+36, -1.0909677673102579e+36, -1.086598899118681e+36, -1.0822970955675562e+36, -1.0780645050215601e+36, -1.0739032666205464e+36, -1.0698155033205611e+36, -1.0658033155392129e+36, -1.0618687754098493e+36, -1.0580139216446131e+36, -1.0542407550023483e+36, -1.0505512343533422e+36, -1.0469472733296596e+36, -1.0434307375466836e+36, -1.0400034423783373e+36, -1.0366671512668839e+36, -1.0334235745449005e+36, -1.030274368746336e+36, -1.0272211363811725e+36, -1.0242654261475958e+36, -1.021408733554255e+36, -1.0186525019243762e+36, -1.0159981237537019e+36, -1.0134469423927015e+36, -1.0110002540248672e+36, -1.008659309911793e+36, -1.0064253188767931e+36, -1.0042994499987309e+36, -1.0022828354886819e+36, -1.0003765737222221e+36, -9.985817324014373e+35, -9.968993518209803e+35, -9.953304482139616e+35, -9.938760171538537e+35, -9.925370369901094e+35, -9.913144722957059e+35, -9.902092773061172e+35, -9.892223993301896e+35, -9.883547821144002e+35, -9.876073691428217e+35, -9.869811068562398e+35, -9.864769477747795e+35, -9.860958535090247e+35, -9.858387976458854e+35, -9.857067684955889e+35, -9.85700771687596e+35, -9.858218326032322e+35, -9.860709986335956e+35, -9.864493412521456e+35, -9.86957957890998e+35, -9.875979736110116e+35, -9.883705425554989e+35, -9.892768491777973e+35, -9.903181092328332e+35, -9.914955705229157e+35, -9.928105133878558e+35, -9.942642509292819e+35, -9.958581289587357e+35, -9.97593525659015e+35, -9.99471850947678e+35, -1.0014945455310162e+36, -1.0036630796366734e+36, -1.005978951412179e+36, -1.0084436849763606e+36, -1.0110588281096411e+36, -1.013825949568987e+36, -1.0167466360122738e+36, -1.019822488516424e+36, -1.0230551186729673e+36, -1.0264461442442271e+36, -1.0299971843625234e+36, -1.0337098542547334e+36, -1.0375857594740988e+36, -1.0416264896207729e+36, -1.045833611532851e+36, -1.050208661929701e+36, -1.054753139489518e+36, -1.0594684963441201e+36, -1.0643561289744751e+36, -1.0694173684918632e+36, -1.0746534702912504e+36, -1.0800656030651405e+36, -1.0856548371691237e+36, -1.0914221323327851e+36, -1.0973683247136405e+36, -1.1034941132953037e+36, -1.1098000456366043e+36, -1.1162865029831203e+36, -1.1229536847592315e+36, -1.1298015924657953e+36, -1.1368300130161733e+36, -1.144038501552497e+36, -1.1514263637931498e+36, -1.1589926379736298e+36, -1.1667360764541736e+36, -1.1746551270801445e+36, -1.1827479143946557e+36, -1.1910122208173281e+36, -1.1994454679185125e+36, -1.208044697933926e+36, -1.2168065556823346e+36, -1.2257272710652941e+36, -1.2348026423467652e+36, -1.2440280204278047e+36, -1.2533982943502358e+36, -1.26290787828032e+36, -1.2725507002416804e+36, -1.2823201928820203e+36, -1.2922092865740249e+36, -1.3022104051633079e+36, -1.3123154646874227e+36, -1.3225158753978213e+36, -1.3328025474208613e+36, -1.3431659003945194e+36, -1.3535958774126337e+36, -1.3640819635987468e+36, -1.3746132096151974e+36, -1.3851782603900937e+36, -1.3957653893142003e+36, -1.4063625381205907e+36, -1.416957362612808e+36, -1.4275372843498445e+36, -1.4380895483301594e+36, -1.4486012866405148e+36, -1.459059587949353e+36, -1.469451572628425e+36, -1.4797644731816517e+36, -1.4899857195459404e+36, -1.5001030287079167e+36, -1.510104497952426e+36, -1.5199787009264253e+36, -1.5297147855665808e+36, -1.539302572803437e+36, -1.5487326548214632e+36, -1.5579964915260604e+36, -1.5670865037489384e+36, -1.5759961616158623e+36, -1.5847200664083137e+36, -1.59325402417889e+36, -1.6015951093315285e+36, -1.6097417163562587e+36, -1.6176935979186902e+36, -1.6254518875488084e+36, -1.6330191052565632e+36, -1.6403991445248327e+36, -1.6475972392953452e+36, -1.654619909772754e+36, -1.661474886124632e+36, -1.6681710094516758e+36, -1.6747181097404486e+36, -1.6811268608884837e+36, -1.6874086133029595e+36, -1.6935752050173483e+36, -1.6996387527355752e+36, -1.705611424696444e+36, -1.7115051977413892e+36, -1.7173316014582777e+36, -1.72310145275215e+36, -1.7288245846500447e+36, -1.7345095735697414e+36, -1.7401634696596192e+36, -1.7457915351379902e+36, -1.7513969958116684e+36, -1.756980811125585e+36, -1.7625414681752425e+36, -1.768074805092684e+36, -1.7735738690841985e+36, -1.7790288141466363e+36, -1.7844268431125958e+36, -1.7897521981682112e+36, -1.7949862033493157e+36, -1.8001073617531602e+36, -1.8050915093076505e+36, -1.8099120259273082e+36, -1.8145401037659316e+36, -1.8189450710684385e+36, -1.8230947688496288e+36, -1.8269559763126678e+36, -1.830494879595723e+36, -1.8336775771373562e+36, -1.8364706137191224e+36, -1.8388415341169125e+36, -1.8407594463157408e+36, -1.842195583453996e+36, -1.843123853104369e+36, -1.8435213622010336e+36, -1.8433689059147642e+36, -1.8426514090790277e+36, -1.8413583093862374e+36, -1.8394838725057933e+36, -1.837027430506026e+36, -1.8339935364673478e+36, -1.8303920299141985e+36, -1.8262380096234362e+36, -1.8215517124298065e+36, -1.816358298785565e+36, -1.8106875479768233e+36, -1.8045734679896768e+36, -1.7980538269937165e+36, -1.7911696152125423e+36, -1.783964447530274e+36, -1.776483918501239e+36, -1.7687749224541223e+36, -1.7608849520928864e+36, -1.7528613893848408e+36, -1.7447508025900048e+36, -1.736598263034759e+36, -1.7284466946831602e+36, -1.7203362687328137e+36, -1.7123038543872557e+36, -1.704382535664899e+36, -1.6966012026285868e+36, -1.6889842237967507e+36, -1.6815512047626374e+36, -1.6743168362389302e+36, -1.6672908328990376e+36, -1.660477962538825e+36, -1.6538781632700413e+36, -1.647486744714586e+36, -1.641294667530128e+36, -1.6352888940963355e+36, -1.629452801856541e+36, -1.6237666496711265e+36, -1.6182080866210974e+36, -1.612752692023785e+36, -1.6073745350042778e+36, -1.602046741816179e+36, -1.5967420592276875e+36, -1.5914334026817577e+36, -1.5860943785907574e+36, -1.580699771019749e+36, -1.575225984123122e+36, -1.5696514329939138e+36, -1.563956877026881e+36, -1.5581256914416584e+36, -1.5521440742158135e+36, -1.546001187291338e+36, -1.5396892324951526e+36, -1.533203464110055e+36, -1.526542141405253e+36, -1.5197064256510295e+36, -1.5127002271708763e+36, -1.5055300088066332e+36, -1.498204552774213e+36, -1.4907346982673667e+36, -1.4831330573260248e+36, -1.4754137164384452e+36, -1.4675919311072676e+36, -1.459683820204158e+36, -1.4517060663905744e+36, -1.4436756282230384e+36, -1.4356094688194278e+36, -1.4275243051684568e+36, -1.4194363813436154e+36, -1.411361268064231e+36, -1.4033136902486703e+36, -1.3953073834512382e+36, -1.3873549793772534e+36, -1.3794679200434766e+36, -1.371656399602042e+36, -1.3639293323781416e+36, -1.3562943452893972e+36, -1.3487577925143102e+36, -1.341324790057854e+36, -1.3339992677174077e+36, -1.3267840358769113e+36, -1.3196808645430717e+36, -1.3126905720779861e+36, -1.3058131211687441e+36, -1.2990477196994181e+36, -1.2923929243452116e+36, -1.2858467448868406e+36, -1.2794067474368012e+36, -1.2730701549737503e+36, -1.2668339437892195e+36, -1.2606949346598e+36, -1.2546498777618604e+36, -1.2486955305430087e+36, -1.2428287279506636e+36, -1.2370464445933564e+36, -1.2313458485712284e+36, -1.2257243468592645e+36, -1.2201796222591453e+36, -1.2147096620529625e+36, -1.20931277859508e+36, -1.2039876221676037e+36, -1.1987331865007007e+36, -1.1935488074224505e+36, -1.1884341551550716e+36, -1.1833892208157134e+36, -1.178414297712022e+36, -1.173509958046075e+36, -1.168677025655624e+36, -1.1639165454303866e+36, -1.1592297500429905e+36, -1.1546180246308648e+36, -1.1500828700563216e+36, -1.1456258653584756e+36, -1.1412486299924188e+36, -1.1369527864280864e+36, -1.1327399236547544e+36, -1.1286115621059408e+36, -1.1245691204847254e+36, -1.1206138849307492e+36, -1.1167469809273483e+36, -1.1129693483015e+36, -1.1092817196193316e+36, -1.1056846022272206e+36, -1.102178264132763e+36, -1.0987627238620576e+36, -1.0954377443691238e+36, -1.0922028310127644e+36, -1.0890572335536858e+36, -1.0859999520634234e+36, -1.0830297465765215e+36, -1.0801451502585961e+36, -1.0773444858088928e+36, -1.0746258847647754e+36, -1.0719873093305127e+36, -1.0694265763142918e+36, -1.066941382725563e+36, -1.0645293325619246e+36, -1.0621879643004202e+36, -1.0599147786029952e+36, -1.0577072657512029e+36, -1.0555629323391945e+36, -1.0534793267784939e+36, -1.051454063201064e+36, -1.0494848433886328e+36, -1.04756947640505e+36, -1.0457058956630407e+36, -1.0438921732166836e+36, -1.0421265311331657e+36, -1.0404073498619338e+36, -1.0387331735830018e+36, -1.0371027125784243e+36, -1.0355148427298901e+36, -1.0339686022988338e+36, -1.0324631861934907e+36, -1.030997937968197e+36, -1.0295723398324799e+36, -1.0281860009727105e+36, -1.0268386445043174e+36, -1.0255300933806288e+36, -1.0242602555835271e+36, -1.0230291089129144e+36, -1.0218366856773075e+36, -1.0206830575667864e+36, -1.0195683209641982e+36, -1.0184925829208266e+36, -1.0174559479913484e+36, -1.0164585060886143e+36, -1.0155003214854585e+36, -1.0145814230569237e+36, -1.013701795823969e+36, -1.0128613738294822e+36, -1.0120600343498994e+36, -1.011297593420343e+36, -1.0105738026305385e+36, -1.0098883471297366e+36, -1.0092408447647029e+36, -1.0086308462633105e+36, -1.0080578363679399e+36, -1.0075212358178561e+36, -1.0070204040767848e+36, -1.0065546427019926e+36, -1.0061231992526275e+36, -1.0057252716384568e+36, -1.0053600128147599e+36, -1.0050265357348808e+36, -1.0047239184780434e+36, -1.0044512094768395e+36, -1.00420743277589e+36, -1.003991593259751e+36, -1.0038026817952953e+36, -1.0036396802401679e+36, -1.0035015662751811e+36, -1.0033873180241502e+36, -1.0032959184304437e+36, -1.003226359364088e+36, -1.0031776454380565e+36, -1.0031487975165075e+36, -1.0031388559015588e+36, -1.003146883188606e+36, -1.0031719667834212e+36, -1.0032132210774658e+36, -1.0032697892802159e+36, -1.0033408449103459e+36, -1.0034255929496945e+36, -1.0035232706665583e+36, -1.0036331481168895e+36, -1.003754528334168e+36, -1.003886747220561e+36, -1.0040291731542249e+36, -1.0041812063288034e+36, -1.0043422778431702e+36, -1.0045118485607423e+36, -1.0046894077587336e+36, -1.0048744715889712e+36, -1.0050665813724238e+36, -1.0052653017500061e+36, -1.0054702187125771e+36, -1.005680937532691e+36, -1.0058970806201433e+36, -1.006118285322891e+36, -1.0063442016929919e+36, -1.0065744902369105e+36, -1.006808819666302e+36, -1.007046864664541e+36, -1.0072883036812501e+36, -1.007532816764522e+36, -1.007780083438174e+36, -1.0080297806280194e+36, -1.008281580638503e+36, -1.0085351491780395e+36, -1.0087901434285462e+36, -1.0090462101516905e+36, -1.0093029838223784e+36, -1.0095600847768741e+36, -1.0098171173619915e+36, -1.0100736680695667e+36, -1.0103293036394947e+36, -1.0105835691143606e+36, -1.0108359858282416e+36, -1.0110860493132466e+36, -1.0113332271081639e+36, -1.0115769564557772e+36, -1.0118166418775682e+36, -1.0120516526179492e+36, -1.0122813199532913e+36, -1.012504934366065e+36, -1.0127217425883122e+36, -1.0129309445248333e+36, -1.0131316900714973e+36, -1.0133230758503731e+36, -1.013504141889783e+36, -1.0136738682836324e+36, -1.0138311718709345e+36, -1.0139749029828955e+36, -1.0141038423113494e+36, -1.0142166979579519e+36, -1.0143121027293757e+36, -1.014388611748586e+36, -1.014444700456378e+36, -1.0144787630810206e+36, -1.014489111656027e+36, -1.0144739756673527e+36, -1.0144315024115753e+36, -1.0143597581450723e+36, -1.0142567301017158e+36, -1.0141203294520312e+36, -1.0139483952712613e+36, -1.0137386995760571e+36, -1.0134889534807085e+36, -1.0131968145128258e+36, -1.012859895116516e+36, -1.0124757723570761e+36, -1.0120419988266965e+36, -1.0115561147340979e+36, -1.0110156611440963e+36, -1.010418194315437e+36, -1.0097613010665136e+36, -1.0090426150805773e+36, -1.0082598340432474e+36, -1.0074107374876489e+36, -1.0064932052050724e+36, -1.0055052360634405e+36, -1.0044449670609351e+36, -1.0033106924300928e+36, -1.0021008825966029e+36, -1.0008142027891474e+36, -9.994495310910015e+35, -9.98005975721453e+35, -9.964828913353072e+35, -9.948798941320615e+35, -9.93196875572668e+35, -9.914340145113896e+35, -9.895917875628089e+35, -9.876709775388885e+35, -9.856726798096637e+35, -9.83598306461053e+35, -9.81449588145969e+35, -9.79228573549331e+35, -9.769376264132634e+35, -9.745794200958195e+35, -9.721569296641095e+35, -9.696734215502958e+35, -9.671324408272206e+35, -9.645377961871695e+35, -9.618935427338053e+35, -9.59203962722419e+35, -9.564735444070671e+35, -9.537069591747732e+35, -9.509090371663611e+35, -9.48084741600474e+35, -9.45239142031649e+35, -9.423773867849275e+35, -9.395046748179181e+35, -9.366262272670772e+35, -9.337472589372529e+35, -9.30872949993152e+35, -9.280084181076606e+35, -9.251586913155135e+35, -9.223286818112016e+35, -9.195231609180222e+35, -9.167467354403794e+35, -9.140038255945402e+35, -9.112986446938301e+35, -9.086351807436888e+35, -9.060171800792988e+35, -9.03448133155334e+35, -9.009312625728343e+35, -8.984695134033917e+35, -8.960655458458918e+35, -8.937217302261114e+35, -8.914401443249707e+35, -8.892225729981663e+35, -8.870705100266131e+35, -8.849851621163906e+35, -8.82967454947315e+35, -8.810180411505362e+35, -8.791373100801902e+35, -8.773253992294351e+35, -8.75582207129212e+35, -8.73907407557789e+35, -8.72300464881476e+35, -8.707606503405942e+35, -8.69287059091027e+35, -8.678786278100184e+35, -8.665341526745864e+35, -8.652523075230498e+35, -8.640316620139065e+35, -8.628706996014632e+35, -8.617678351551704e+35, -8.607214320575307e+35, -8.597298186262736e+35, -8.587913037173971e+35, -8.579041913787889e+35, -8.57066794438301e+35, -8.562774469250397e+35, -8.555345152394449e+35, -8.548364080047635e+35, -8.541815845509461e+35, -8.535685620010897e+35, -8.52995920949981e+35, -8.524623097449736e+35, -8.519664473991888e+35, -8.515071251882383e+35, -8.510832070016647e+35, -8.506936285399254e+35, -8.503373954674983e+35, -8.500135806496799e+35, -8.497213206177986e+35, -8.494598114214825e+35, -8.492283040388064e+35, -8.490260995244377e+35, -8.488525440818824e+35, -8.487070242483888e+35, -8.485889623796714e+35, -8.484978126156434e+35, -8.484330574986271e+35, -8.483942054003447e+35, -8.483807888952361e+35, -8.483923641937071e+35, -8.484285117214516e+35, -8.484888378996388e+35, -8.48572978146273e+35, -8.486806010822713e+35, -8.48811413887864e+35, -8.489651687160314e+35, -8.491416700315152e+35, -8.493407827079197e+35, -8.495624406816571e+35, -8.49806655931632e+35, -8.500735275296017e+35, -8.503632504866772e+35, -8.50676124109718e+35, -8.510125595763716e+35, -8.513730864399979e+35, -8.517583577859648e+35, -8.521691537785451e+35, -8.526063833621756e+35, -8.530710839120721e+35, -8.535644186660152e+35, -8.540876718096132e+35, -8.546422411331928e+35, -8.552296282234973e+35, -8.558514262025546e+35, -8.565093050716667e+35, -8.572049947652972e+35, -8.579402660613713e+35, -8.587169095343262e+35, -8.595367127716853e+35, -8.604014361048994e+35, -8.613127871294483e+35, -8.622723943081125e+35, -8.632817799645626e+35, -8.643423329824303e+35, -8.654552815275678e+35, -8.6662166610926e+35, -8.678423132902146e+35, -8.691178103448359e+35, -8.70448481152655e+35, -8.718343635976346e+35, -8.732751887269312e+35, -8.74770361902782e+35, -8.763189461609656e+35, -8.77919647967728e+35, -8.795708055451001e+35, -8.812703799124595e+35, -8.830159487690406e+35, -8.848047033202398e+35, -8.866334481272568e+35, -8.884986040370554e+35, -8.903962142268782e+35, -8.923219533746467e+35, -8.942711399435429e+35, -8.962387515458663e+35, -8.982194433285092e+35, -9.002075692985581e+35, -9.02197206484618e+35, -9.04182181806089e+35, -9.061561014996313e+35, -9.081123829292104e+35, -9.100442885840696e+35, -9.119449620474169e+35, -9.138074656978224e+35, -9.156248198864007e+35, -9.173900433144871e+35, -9.190961943204847e+35, -9.207364127700741e+35, -9.223039622322056e+35, -9.237922721131318e+35, -9.25194979414049e+35, -9.26505969773474e+35, -9.277194174543712e+35, -9.288298239373803e+35, -9.298320547871252e+35, -9.307213744659701e+35, -9.31493478781376e+35, -9.321445246671085e+35, -9.32671157016367e+35, -9.330705323048865e+35, -9.333403387656684e+35, -9.334788129026459e+35, -9.334847521588874e+35, -9.333575235852093e+35, -9.330970683879484e+35, -9.327039022682566e+35, -9.321791115013163e+35, -9.315243447405267e+35, -9.307418005693556e+35, -9.298342108626583e+35, -9.288048200571605e+35, -9.27657360470845e+35, -9.263960238491178e+35, -9.25025429354243e+35, -9.23550588252331e+35, -9.21976865588115e+35, -9.203099391731994e+35, -9.185557562458982e+35, -9.167204881918845e+35, -9.148104837420269e+35, -9.128322210888743e+35, -9.107922593831785e+35, -9.086971900884506e+35, -9.065535886827756e+35, -9.043679672028171e+35, -9.021467281251893e+35, -8.998961200741696e+35, -8.976221958315172e+35, -8.953307731048733e+35, -8.930273984841455e+35, -8.907173149816521e+35, -8.88405433511376e+35, -8.860963086152476e+35, -8.837941186918985e+35, -8.815026509242232e+35, -8.792252910397473e+35, -8.769650179710592e+35, -8.747244034148635e+35, -8.725056162182492e+35, -8.703104314509178e+35, -8.68140243954071e+35, -8.659960860909551e+35, -8.638786493633202e+35, -8.617883095022326e+35, -8.59725154592582e+35, -8.576890157494764e+35, -8.55679499831612e+35, -8.536960236526326e+35, -8.517378491374255e+35, -8.49804118864978e+35, -8.478938914442136e+35, -8.460061761832413e+35, -8.441399665349101e+35, -8.422942718325984e+35, -8.404681468682208e+35, -8.38660718908794e+35, -8.368712117982881e+35, -8.350989668451143e+35, -8.333434602535097e+35, -8.316043169162154e+35, -8.298813204465264e+35, -8.28174419388368e+35, -8.264837296021351e+35, -8.248095328820977e+35, -8.231522719158662e+35, -8.215125417477504e+35, -8.198910779558209e+35, -8.18288741794716e+35, -8.167065025946873e+35, -8.151454177396703e+35, -8.136066105740112e+35, -8.120912466087804e+35, -8.106005084133866e+35, -8.091355695879203e+35, -8.076975682143609e+35, -8.062875801827751e+35, -8.04906592780558e+35, -8.035554789191412e+35, -8.022349723550035e+35, -8.00945644238678e+35, -7.996878812994888e+35, -7.984618659431593e+35, -7.972675585074988e+35, -7.961046818859837e+35, -7.949727086933717e+35, -7.93870851110183e+35, -7.927980535061006e+35, -7.917529879058505e+35, -7.907340523259529e+35, -7.897393719771365e+35, -7.887668032960219e+35, -7.878139407409989e+35, -7.86878126261335e+35, -7.859564613261968e+35, -7.850458213803137e+35, -7.841428725774246e+35, -7.832440906291243e+35, -7.82345781596809e+35, -7.814441044472322e+35, -7.805350951870893e+35, -7.796146923894753e+35, -7.786787639241828e+35, -7.777231347038383e+35, -7.76743615259863e+35, -7.757360309637016e+35, -7.746962517119297e+35, -7.736202218955213e+35, -7.725039904767358e+35, -7.713437409983235e+35, -7.701358213518489e+35, -7.688767731326993e+35, -7.675633604097762e+35, -7.661925977381364e+35, -7.647617772424273e+35, -7.632684945982673e+35, -7.617106737383857e+35, -7.600865901095466e+35, -7.583948923063642e+35, -7.566346219084398e+35, -7.548052313482229e+35, -7.529065996395605e+35, -7.50939045799745e+35, -7.489033398028079e+35, -7.468007109080576e+35, -7.446328532155469e+35, -7.424019283100667e+35, -7.401105648668621e+35, -7.377618551059938e+35, -7.353593479980084e+35, -7.329070391416792e+35, -7.304093572547871e+35, -7.27871147241125e+35, -7.252976498218992e+35, -7.226944777461237e+35, -7.200675886238022e+35, -7.174232544562922e+35, -7.147680279710861e+35, -7.121087059023184e+35, -7.094522893938662e+35, -7.068059417385303e+35, -7.041769437035127e+35, -7.015726467297108e+35, -6.990004243289085e+35, -6.964676220380252e+35, -6.93981506324013e+35, -6.915492128628967e+35, -6.89177694645574e+35, -6.868736703856288e+35, -6.846435737237028e+35, -6.824935037361301e+35, -6.804291772620251e+35, -6.784558835634125e+35, -6.76578441824505e+35, -6.748011619814097e+35, -6.7312780934928e+35, -6.715615734823998e+35, -6.701050416624105e+35, -6.68760177362515e+35, -6.67528303980513e+35, -6.664100940723938e+35, -6.65405564251591e+35, -6.645140758478709e+35, -6.6373434134610986e+35, -6.6306443654920256e+35, -6.6250181833374245e+35, -6.620433477925139e+35, -6.6168531848631e+35, -6.614234894600858e+35, -6.612531226173842e+35, -6.611690239920285e+35, -6.6116558841023504e+35, -6.612368469986245e+35, -6.6137651696656115e+35, -6.615780530736949e+35, -6.618347001868714e+35, -6.6213954633432186e+35, -6.62485575678391e+35, -6.628657208511919e+35, -6.632729141296544e+35, -6.6370013696532886e+35, -6.641404674307348e+35, -6.64587125194755e+35, -6.650335136945207e+35, -6.654732592282087e+35, -6.659002467509866e+35, -6.663086522134426e+35, -6.666929713369789e+35, -6.67048044772486e+35, -6.67369079636127e+35, -6.676516674586998e+35, -6.678917986215748e+35, -6.680858733829751e+35, -6.682307096226516e+35, -6.68323547451441e+35, -6.68362050844587e+35, -6.683443064651524e+35, -6.682688198466613e+35, -6.681345091032113e+35, -6.679406963317874e+35, -6.676870968662859e+35, -6.673738065362387e+35, -6.670012870775577e+35, -6.665703498367438e+35, -6.660821379066952e+35, -6.655381068297674e+35, -6.649400040044544e+35, -6.6428984693442405e+35, -6.635899004636931e+35, -6.628426531482501e+35, -6.6205079292329745e+35, -6.612171822339091e+35, -6.603448328074127e+35, -6.5943688025408854e+35, -6.584965586915504e+35, -6.575271755936283e+35, -6.5653208706841244e+35, -6.555146737702718e+35, -6.544783176470241e+35, -6.534263797162826e+35, -6.5236217905321915e+35, -6.512889731566785e+35, -6.502099398408218e+35, -6.491281607768862e+35, -6.480466067833776e+35, -6.469681249357239e+35, -6.4589542753640706e+35, -6.448310829571009e+35, -6.437775083346109e+35, -6.427369640742656e+35, -6.4171155008824664e+35, -6.407032036728978e+35, -6.397136989095282e+35, -6.387446474571359e+35, -6.3779750059403785e+35, -6.368735523584019e+35, -6.359739436349127e+35, -6.350996670364959e+35, -6.342515724352248e+35, -6.334303730056158e+35, -6.3263665165468156e+35, -6.3187086772693716e+35, -6.311333638871419e+35, -6.304243730990271e+35, -6.2974402563347944e+35, -6.290923560538848e+35, -6.284693101394317e+35, -6.27874751717982e+35, -6.2730846938920096e+35, -6.267701831248906e+35, -6.262595507376265e+35, -6.257761742102115e+35, -6.253196058783266e+35, -6.2488935445577426e+35, -6.24484890888673e+35, -6.241056540196159e+35, -6.237510560377596e+35, -6.234204876858559e+35, -6.231133231898474e+35, -6.228289248735025e+35, -6.225666474173896e+35, -6.223258417208437e+35, -6.221058583258529e+35, -6.2190605036449716e+35, -6.217257759958554e+35, -6.2156440030396136e+35, -6.21421296636159e+35, -6.212958473701993e+35, -6.211874441077084e+35, -6.2109548730280576e+35, -6.210193853452487e+35, -6.209585531282885e+35, -6.209124101421608e+35, -6.2088037814385805e+35, -6.2086187846276885e+35, -6.2085632900916806e+35, -6.2086314105923805e+35, -6.208817158941603e+35, -6.2091144137444346e+35, -6.209516885312066e+35, -6.2100180825578976e+35, -6.2106112816645845e+35, -6.2112894972720075e+35, -6.212045456878509e+35, -6.2128715790806574e+35, -6.213759956194387e+35, -6.214702341713342e+35, -6.215690142959298e+35, -6.2167144191823456e+35, -6.217765885256324e+35, -6.218834921017934e+35, -6.2199115861899385e+35, -6.2209856407315095e+35, -6.222046570365505e+35, -6.22308361694999e+35, -6.22408581328276e+35, -6.225042021867702e+35, -6.225940977115061e+35, -6.226771330409127e+35, -6.227521697452406e+35, -6.228180707275941e+35, -6.228737052305312e+35, -6.2291795388835544e+35, -6.229497137667644e+35, -6.229679033348881e+35, -6.229714673184298e+35, -6.229593813869165e+35, -6.229306566331292e+35, -6.228843438079946e+35, -6.2281953727941924e+35, -6.2273537868912966e+35, -6.226310602863322e+35, -6.2250582792219186e+35, -6.223589836933037e+35, -6.2218988822624684e+35, -6.219979625986458e+35, -6.21782689895128e+35, -6.2154361639859506e+35, -6.21280352419165e+35, -6.209925727644561e+35, -6.20680016856112e+35, -6.203424884981701e+35, -6.199798553038798e+35, -6.19592047788495e+35, -6.1917905813640054e+35, -6.187409386527961e+35, -6.182777999113684e+35, -6.177898086118429e+35, -6.17277185163732e+35, -6.167402010156826e+35, -6.161791757529344e+35, -6.155944739892261e+35, -6.149865020829529e+35, -6.1435570471134225e+35, -6.137025613397043e+35, -6.130275826263336e+35, -6.123313068060469e+35, -6.116142960975351e+35, -6.108771331809273e+35, -6.101204177921298e+35, -6.0934476348006366e+35, -6.08550794570568e+35, -6.077391433785587e+35, -6.069104477054438e+35, -6.060653486546167e+35, -6.0520448879159594e+35, -6.043285106693824e+35, -6.034380557327297e+35, -6.0253376360787e+35, -6.016162717771365e+35, -6.006862156307096e+35, -5.9974422888146356e+35, -5.98790944322294e+35, -5.978269949000066e+35, -5.968530150755416e+35, -5.958696424362042e+35, -5.948775195231734e+35, -5.938772958357997e+35, -5.92869629973574e+35, -5.9185519187691e+35, -5.908346651289077e+35, -5.898087492824184e+35, -5.887781621788697e+35, -5.877436422286056e+35, -5.867059506254656e+35, -5.856658734721071e+35, -5.846242237956075e+35, -5.835818434366279e+35, -5.825396047980012e+35, -5.814984124418797e+35, -5.804592045262596e+35, -5.794229540737954e+35, -5.783906700670882e+35, -5.7736339836511804e+35, -5.7634222243599816e+35, -5.753282639006695e+35, -5.743226828819348e+35, -5.733266781518777e+35, -5.7234148706988585e+35, -5.7136838530212326e+35, -5.70408686312058e+35, -5.6946374061056305e+35, -5.685349347532819e+35, -5.676236900720343e+35, -5.66731461127464e+35, -5.658597338697911e+35, -5.650100234957916e+35, -5.6418387199130856e+35, -5.633828453507772e+35, -5.6260853046749914e+35, -5.61862531691959e+35, -5.611464670586019e+35, -5.604619641859201e+35, -5.598106558592613e+35, -5.591941753097856e+35, -5.586141512089618e+35, -5.5807220240217415e+35, -5.575699324103865e+35, -5.571089237333929e+35, -5.566907319932269e+35, -5.563168799597487e+35, -5.559888515051882e+35, -5.557080855366466e+35, -5.554759699589742e+35, -5.552938357218677e+35, -5.551629510064556e+35, -5.550845156067629e+35, -5.550596555615228e+35, -5.550894180899101e+35, -5.551747668833925e+35, -5.553165778026259e+35, -5.555156350248584e+35, -5.557726276832296e+35, -5.56088147033937e+35, -5.564626841823313e+35, -5.568966283923805e+35, -5.573902659979612e+35, -5.5794377992732734e+35, -5.585572498455045e+35, -5.5923065291177004e+35, -5.599638651429535e+35, -5.607566633655604e+35, -5.616087277337164e+35, -5.625196447829491e+35, -5.634889109840354e+35, -5.645159367556927e+35, -5.656000508900243e+35, -5.66740505340261e+35, -5.6793648031714594e+35, -5.691870896375225e+35, -5.7049138626688475e+35, -5.718483679967247e+35, -5.732569831973902e+35, -5.747161365878926e+35, -5.762246949657278e+35, -5.7778149284188584e+35, -5.7938533792970945e+35, -5.8103501643918474e+35, -5.827292981336216e+35, -5.844669411090772e+35, -5.86246696263027e+35, -5.880673114238892e+35, -5.899275351180063e+35, -5.918261199574454e+35, -5.937618256364875e+35, -5.957334215309119e+35, -5.977396888994133e+35, -5.99779422691327e+35, -6.01851432969896e+35, -6.039545459645799e+35, -6.060876047699189e+35, -6.082494697118378e+35, -6.104390184056275e+35, -6.126551455321917e+35, -6.148967623611133e+35, -6.171627960512163e+35, -6.194521887598538e+35, -6.217638965932239e+35, -6.24096888430328e+35, -6.2645014465302525e+35, -6.288226558140313e+35, -6.312134212745762e+35, -6.33621447841608e+35, -6.360457484339371e+35, -6.3848534080462814e+35, -6.409392463459046e+35, -6.43406489000373e+35, -6.458860943011306e+35, -6.4837708856084244e+35, -6.508784982280837e+35, -6.533893494270277e+35, -6.559086676945343e+35, -6.584354779264101e+35, -6.609688045428232e+35, -6.635076718803152e+35, -6.660511048163043e+35, -6.685981296297525e+35, -6.711477750996044e+35, -6.736990738411792e+35, -6.762510638783209e+35, -6.788027904480132e+35, -6.813533080319777e+35, -6.839016826086792e+35, -6.86446994117055e+35, -6.889883391224144e+35, -6.915248336734164e+35, -6.940556163371464e+35, -6.965798513990925e+35, -6.990967322122713e+35, -7.016054846799191e+35, -7.041053708536282e+35, -7.065956926288832e+35, -7.090757955181434e+35, -7.115450724805349e+35, -7.140029677869956e+35, -7.164489808977267e+35, -7.188826703288296e+35, -7.2130365748374e+35, -7.237116304246343e+35, -7.261063475584469e+35, -7.28487641211452e+35, -7.308554210667013e+35, -7.33209677437939e+35, -7.355504843541192e+35, -7.378780024289493e+35, -7.401924814903638e+35, -7.424942629456739e+35, -7.447837818592659e+35, -7.470615687210215e+35, -7.493282508849271e+35, -7.5158455365954e+35, -7.538313010335449e+35, -7.560694160221912e+35, -7.582999206226449e+35, -7.605239353687046e+35, -7.627426784785178e+35, -7.649574645911264e+35, -7.671697030910952e+35, -7.693808960229934e+35, -7.715926356008227e+35, -7.738066013200855e+35, -7.760245566830387e+35, -7.78248345550602e+35, -7.804798881369047e+35, -7.827211766645907e+35, -7.84974270701777e+35, -7.872412922028882e+35, -7.895244202780658e+35, -7.918258857165755e+35, -7.941479652915634e+35, -7.964929758738898e+35, -7.988632683837417e+35, -8.012612216089867e+35, -8.036892359191865e+35, -8.061497269043837e+35, -8.08645118966871e+35, -8.111778388936668e+35, -8.137503094369578e+35, -8.163649429278868e+35, -8.190241349485763e+35, -8.217302580856258e+35, -8.244856557868925e+35, -8.272926363418095e+35, -8.301534670042136e+35, -8.33070368274636e+35, -8.36045508357969e+35, -8.3908099781047e+35, -8.421788843889714e+35, -8.453411481135879e+35, -8.485696965539443e+35, -8.518663603480822e+35, -8.552328889615637e+35, -8.586709466940599e+35, -8.621821089393274e+35, -8.657678587039265e+35, -8.694295833896719e+35, -8.731685718437092e+35, -8.769860116801646e+35, -8.808829868766194e+35, -8.848604756481892e+35, -8.889193486018183e+35, -8.930603671727682e+35, -8.97284182344806e+35, -9.015913336552529e+35, -9.059822484853444e+35, -9.10457241635431e+35, -9.150165151842508e+35, -9.196601586297978e+35, -9.24388149309245e+35, -9.2920035309304e+35, -9.340965253479593e+35, -9.390763121618533e+35, -9.44139251821604e+35, -9.492847765339034e+35, -9.545122143771247e+35, -9.598207914703792e+35, -9.65209634344702e+35, -9.706777724987511e+35, -9.76224141120747e+35, -9.818475839557777e+35, -9.875468562969089e+35, -9.933206280767231e+35, -9.991674870349627e+35, -1.0050859419368965e+36, -1.0110744258164751e+36, -1.0171312992176661e+36, -1.0232548534071681e+36, -1.0294433135319504e+36, -1.035694841695324e+36, -1.0420075399255717e+36, -1.0483794530127979e+36, -1.0548085711899607e+36, -1.0612928326363788e+36, -1.0678301257830185e+36, -1.074418291401229e+36, -1.081055124458606e+36, -1.0877383757280937e+36, -1.0944657531388454e+36, -1.1012349228600176e+36, -1.1080435101116866e+36, -1.114889099699574e+36, -1.1217692362734548e+36, -1.128681424312009e+36, -1.1356231278397689e+36, -1.1425917698849687e+36, -1.149584731689834e+36, -1.1565993516877992e+36, -1.1636329242650947e+36, -1.170682698326647e+36, -1.1777458756889823e+36, -1.1848196093254118e+36, -1.1919010014909654e+36, -1.1989871017569324e+36, -1.2060749049868135e+36, -1.2131613492874798e+36, -1.2202433139710266e+36, -1.2273176175641351e+36, -1.2343810159033079e+36, -1.2414302003550725e+36, -1.2484617962011804e+36, -1.2554723612293343e+36, -1.2624583845700007e+36, -1.2694162858197723e+36, -1.2763424144914152e+36, -1.2832330498296727e+36, -1.2900844010307206e+36, -1.2968926079019241e+36, -1.3036537419957484e+36, -1.3103638082503064e+36, -1.317018747165326e+36, -1.3236144375397891e+36, -1.330146699793599e+36, -1.336611299891831e+36, -1.343003953886193e+36, -1.3493203330834876e+36, -1.3555560698467106e+36, -1.361706764029297e+36, -1.367767990038607e+36, -1.3737353045199653e+36, -1.379604254647765e+36, -1.3853703870063133e+36, -1.3910292570386268e+36, -1.3965764390385087e+36, -1.4020075366575646e+36, -1.4073181938971206e+36, -1.4125041065529996e+36, -1.417561034080398e+36, -1.4224848118458244e+36, -1.4272713637338733e+36, -1.4319167150777018e+36, -1.436417005884365e+36, -1.4407685043284203e+36, -1.4449676204904205e+36, -1.449010920319794e+36, -1.4528951398049588e+36, -1.4566171993364976e+36, -1.46017421825136e+36, -1.463563529548207e+36, -1.4667826947640803e+36, -1.469829519002531e+36, -1.4727020661002917e+36, -1.4753986739162002e+36, -1.477917969719344e+36, -1.480258885645319e+36, -1.4824206741787592e+36, -1.484402923607121e+36, -1.4862055733755206e+36, -1.4878289292545882e+36, -1.4892736782136273e+36, -1.490540902869726e+36, -1.4916320953604443e+36, -1.4925491704627872e+36, -1.49329447775697e+36, -1.4938708126070672e+36, -1.4942814257062774e+36, -1.4945300309099042e+36, -1.4946208110567134e+36, -1.4945584214585634e+36, -1.4943479907210275e+36, -1.4939951185434166e+36, -1.4935058701373256e+36, -1.4928867668986587e+36, -1.492144772969531e+36, -1.4912872773349053e+36, -1.4903220711149726e+36, -1.4892573197380345e+36, -1.4881015297119305e+36, -1.486863509754147e+36, -1.485552326093089e+36, -1.4841772518153827e+36, -1.482747710206261e+36, -1.4812732121133025e+36, -1.4797632874555288e+36, -1.4782274111016455e+36, -1.4766749234507295e+36, -1.47511494616427e+36, -1.4735562936205073e+36, -1.4720073807855753e+36, -1.4704761283215347e+36, -1.4689698658740593e+36, -1.4674952346015302e+36, -1.466058090117122e+36, -1.4646634071160887e+36, -1.4633151870448623e+36, -1.4620163702377243e+36, -1.4607687539938987e+36, -1.4595729180928863e+36, -1.4584281592450656e+36, -1.457332435945858e+36, -1.4562823251445648e+36, -1.4552729920512438e+36, -1.4542981742873058e+36, -1.4533501814377128e+36, -1.4524199108864913e+36, -1.4514968806137707e+36, -1.450569279404679e+36, -1.4496240346724307e+36, -1.4486468978312258e+36, -1.4476225468771787e+36, -1.4465347055489903e+36, -1.4453662781526884e+36, -1.4440994988511185e+36, -1.4427160939444727e+36, -1.4411974554108838e+36, -1.4395248237397247e+36, -1.4376794778828312e+36, -1.4356429299733842e+36, -1.433397122325158e+36, -1.4309246241296814e+36, -1.4282088252182326e+36, -1.4252341242518512e+36, -1.4219861087474227e+36, -1.4184517244397696e+36, -1.4146194316180735e+36, -1.410479346257638e+36, -1.4060233639898426e+36, -1.4012452652110985e+36, -1.3961407999188433e+36, -1.3907077511732835e+36, -1.3849459764106343e+36, -1.3788574261705355e+36, -1.372446140138623e+36, -1.36571822073925e+36, -1.3586817848355833e+36, -1.351346894398788e+36, -1.3437254672893334e+36, -1.3358311695473945e+36, -1.3276792908116155e+36, -1.319286604673922e+36, -1.3106712159308546e+36, -1.3018523968082088e+36, -1.2928504143155322e+36, -1.2836863509315698e+36, -1.2743819208320122e+36, -1.2649592838496171e+36, -1.2554408593055588e+36, -1.245849141773962e+36, -1.236206520739833e+36, -1.2265351059895836e+36, -1.2168565604341309e+36, -1.2071919419111901e+36, -1.1975615553485778e+36, -1.1879848164969855e+36, -1.1784801282612291e+36, -1.16906477047565e+36, -1.1597548037856272e+36, -1.1505649881133072e+36, -1.1415087160052138e+36, -1.1325979609838568e+36, -1.1238432408554784e+36, -1.1152535957648992e+36, -1.1068365806357295e+36, -1.0985982714929749e+36, -1.0905432850344446e+36, -1.0826748107007153e+36, -1.074994654388537e+36, -1.0675032928638687e+36, -1.0601999378546009e+36, -1.0530826087433212e+36, -1.0461482127346989e+36, -1.0393926313421122e+36, -1.03281081202203e+36, -1.0263968637835376e+36, -1.0201441556125433e+36, -1.0140454165755485e+36, -1.0080928365054176e+36, -1.002278166219821e+36, -9.965928162824465e+35, -9.91027953383955e+35, -9.855745934956843e+35, -9.802236910305982e+35, -9.749662233330156e+35, -9.697932699093584e+35, -9.646960859054004e+35, -9.596661694296212e+35, -9.546953224164631e+35, -9.497757048160073e+35, -9.448998819863273e+35, -9.400608652513651e+35, -9.352521456686669e+35, -9.30467721126689e+35, -9.25702116961618e+35, -9.209504003451483e+35, -9.162081887496316e+35, -9.114716528436313e+35, -9.067375142092737e+35, -9.02003038302659e+35, -8.972660231007703e+35, -8.925247838919733e+35, -8.877781346733006e+35, -8.830253666168748e+35, -8.782662240601396e+35, -8.735008784608967e+35, -8.687299007394248e+35, -8.639542324068562e+35, -8.59175155851707e+35, -8.543942641270736e+35, -8.496134305493401e+35, -8.448347783856984e+35, -8.400606508750064e+35, -8.352935817927253e+35, -8.305362667381008e+35, -8.257915352907234e+35, -8.210623241539908e+35, -8.16351651375527e+35, -8.116625917099232e+35, -8.069982531664863e+35, -8.023617547655953e+35, -7.977562055100056e+35, -7.93184684563802e+35, -7.88650222620436e+35, -7.841557844325013e+35, -7.797042524699444e+35, -7.752984116691744e+35, -7.709409352338337e+35, -7.666343714479307e+35, -7.623811314630701e+35, -7.581834780247488e+35, -7.540435151058452e+35, -7.499631784203038e+35, -7.459442267951118e+35, -7.419882343838178e+35, -7.380965837105563e+35, -7.342704595390503e+35, -7.305108435661762e+35, -7.268185099448882e+35, -7.231940216451129e+35, -7.196377276657374e+35, -7.161497611132209e+35, -7.127300381650411e+35, -7.0937825793747e+35, -7.060939032775002e+35, -7.028762424987867e+35, -6.997243320795786e+35, -6.966370203386706e+35, -6.93612952102096e+35, -6.906505743688984e+35, -6.877481429795365e+35, -6.84903730284606e+35, -6.821152338048761e+35, -6.79380385866431e+35, -6.766967641871127e+35, -6.740618033817889e+35, -6.714728073459863e+35, -6.689269624682327e+35, -6.66421351612797e+35, -6.6395296880601834e+35, -6.615187345507113e+35, -6.5911551168553794e+35, -6.567401216981262e+35, -6.543893613950144e+35, -6.5206001982472735e+35, -6.4974889534600994e+35, -6.4745281272920895e+35, -6.45168640176389e+35, -6.4289330614465285e+35, -6.406238158563518e+35, -6.383572673826864e+35, -6.360908671880652e+35, -6.338219450284052e+35, -6.3154796810027245e+35, -6.292665543452109e+35, -6.2697548482049946e+35, -6.246727150561814e+35, -6.22356385327365e+35, -6.200248297805437e+35, -6.176765843631675e+35, -6.15310393516021e+35, -6.1292521559888394e+35, -6.1052022703067306e+35, -6.0809482513535926e+35, -6.056486296958338e+35, -6.0318148322671266e+35, -6.0069344998682245e+35, -5.981848137600728e+35, -5.956560744412654e+35, -5.931079434701822e+35, -5.9054133816300125e+35, -5.879573749958038e+35, -5.853573618983518e+35, -5.82742789620791e+35, -5.80115322237732e+35, -5.774767868569797e+35, -5.748291626009068e+35, -5.7217456892997716e+35, -5.6951525337794706e+35, -5.6685357876843264e+35, -5.6419200998215484e+35, -5.615331003443894e+35, -5.588794777004674e+35, -5.562338302477011e+35, -5.535988921905986e+35, -5.5097742928609276e+35, -5.483722243446899e+35, -5.457860627530877e+35, -5.432217180830597e+35, -5.406819378515424e+35, -5.381694294955816e+35, -5.356868466265814e+35, -5.3323677562652e+35, -5.308217226494405e+35, -5.284441010898974e+35, -5.261062195797434e+35, -5.238102705730808e+35, -5.215583195777696e+35, -5.1935229509013985e+35, -5.171939792871869e+35, -5.150849995275486e+35, -5.130268207098775e+35, -5.1102073853287765e+35, -5.090678736976811e+35, -5.071691670878864e+35, -5.0532537595815645e+35, -5.0353707115587084e+35, -5.018046353949527e+35, -5.001282625940505e+35, -4.985079582850986e+35, -4.9694354109077976e+35, -4.9543464526281495e+35, -4.939807242654277e+35, -4.925810553813523e+35, -4.912347453104347e+35, -4.899407367242242e+35, -4.886978157324885e+35, -4.875046202120282e+35, -4.863596489410933e+35, -4.852612714777394e+35, -4.8420773871494015e+35, -4.831971940404702e+35, -4.822276850258263e+35, -4.812971755646686e+35, -4.8040355837852804e+35, -4.795446678054839e+35, -4.787182927856822e+35, -4.779221899571411e+35, -4.7715409677487845e+35, -4.764117445669344e+35, -4.7569287144241106e+35, -4.749952349679983e+35, -4.74316624532336e+35, -4.7365487332065096e+35, -4.730078698253931e+35, -4.723735688234321e+35, -4.717500017545737e+35, -4.7113528644179566e+35, -4.7052763609873e+35, -4.699253675765921e+35, -4.6932690880872925e+35, -4.687308054174872e+35, -4.681357264554502e+35, -4.675404692598206e+35, -4.6694396340586884e+35, -4.663452737527967e+35, -4.657436025824452e+35, -4.651382908382953e+35, -4.645288184794825e+35, -4.639148039707807e+35, -4.6329600293654964e+35, -4.626723060121376e+35, -4.620437359325518e+35, -4.6141044390270834e+35, -4.607727052989054e+35, -4.6013091475483205e+35, -4.59485580689076e+35, -4.588373193339994e+35, -4.581868483279239e+35, -4.5753497993416514e+35, -4.5688261395106464e+35, -4.562307303778409e+35, -4.555803818996408e+35, -4.549326862550752e+35, -4.5428881854689865e+35, -4.536500035545959e+35, -4.530175081046353e+35, -4.523926335509768e+35, -4.517767084144792e+35, -4.511710812260016e+35, -4.505771136137934e+35, -4.499961736707838e+35, -4.494296296334117e+35, -4.4887884389843526e+35, -4.483451673997572e+35, -4.4782993436260146e+35, -4.473344574478247e+35, -4.468600232952021e+35, -4.464078884698816e+35, -4.459792758129422e+35, -4.455753711931976e+35, -4.451973206541884e+35, -4.44846227947692e+35, -4.445231524422413e+35, -4.442291073932577e+35, -4.439650585596429e+35, -4.437319231497805e+35, -4.435305690793376e+35, -4.433618145220216e+35, -4.43226427733868e+35, -4.431251271317624e+35, -4.4305858160607586e+35, -4.430274110481931e+35, -4.4303218707331495e+35, -4.43073433919712e+35, -4.4315162950606205e+35, -4.4326720662868e+35, -4.434205542818432e+35, -4.436120190842036e+35, -4.438419067955517e+35, -4.441104839085453e+35, -4.444179793006816e+35, -4.4476458593246635e+35, -4.451504625781277e+35, -4.455757355758411e+35, -4.460405005847079e+35, -4.465448243363288e+35, -4.4708874636912595e+35, -4.476722807336104e+35, -4.482954176576681e+35, -4.4895812516059e+35, -4.4966035060534e+35, -4.504020221786376e+35, -4.51183050288805e+35, -4.520033288716411e+35, -4.528627365952084e+35, -4.537611379545643e+35, -4.546983842483089e+35, -4.556743144292816e+35, -4.5668875582246155e+35, -4.577415247040769e+35, -4.588324267365577e+35, -4.599612572551958e+35, -4.611278014032139e+35, -4.623318341134441e+35, -4.6357311993559216e+35, -4.6485141270974854e+35, -4.661664550880314e+35, -4.675179779076414e+35, -4.6890569942002585e+35, -4.703293243824327e+35, -4.7178854301951715e+35, -4.7328302986407715e+35, -4.74812442487444e+35, -4.763764201316424e+35, -4.779745822561451e+35, -4.7960652701414004e+35, -4.812718296733876e+35, -4.829700409986983e+35, -4.847006856130724e+35, -4.864632603559439e+35, -4.8825723265726e+35, -4.900820389464569e+35, -4.919370831159678e+35, -4.938217350586539e+35, -4.957353292984959e+35, -4.9767716373397696e+35, -4.9964649851230614e+35, -5.016425550529361e+35, -5.036645152369874e+35, -5.057115207790422e+35, -5.077826727958667e+35, -5.098770315855732e+35, -5.119936166293101e+35, -5.141314068252634e+35, -5.162893409637694e+35, -5.184663184496314e+35, -5.206612002761336e+35, -5.2287281025291395e+35, -5.2509993648761564e+35, -5.2734133311902585e+35, -5.295957222971712e+35, -5.318617964034136e+35, -5.3413822050147696e+35, -5.364236350080132e+35, -5.3871665856935674e+35, -5.410158911288142e+35, -5.4331991716743125e+35, -5.456273090989518e+35, -5.479366307986897e+35, -5.502464412442854e+35, -5.525552982456997e+35, -5.548617622406968e+35, -5.571644001318006e+35, -5.594617891402053e+35, -5.617525206526425e+35, -5.6403520403700856e+35, -5.6630847040382606e+35, -5.685709762910053e+35, -5.708214072510332e+35, -5.7305848132068945e+35, -5.752809523556995e+35, -5.774876132138497e+35, -5.7967729877251825e+35, -5.8184888876871e+35, -5.84001310451534e+35, -5.8613354103945666e+35, -5.882446099767084e+35, -5.903336009854084e+35, -5.923996539115571e+35, -5.944419663655475e+35, -5.96459795158543e+35, -5.98452457538145e+35, -6.0041933222757026e+35, -6.023598602736231e+35, -6.042735457090778e+35, -6.06159956035573e+35, -6.080187225333442e+35, -6.0984954040351004e+35, -6.116521687484935e+35, -6.134264303956421e+35, -6.151722115677825e+35, -6.168894614042856e+35, -6.1857819133443664e+35, -6.202384743044195e+35, -6.218704438580775e+35, -6.234742930701754e+35, -6.250502733307917e+35, -6.265986929780119e+35, -6.2811991577602325e+35, -6.2961435923494556e+35, -6.3108249276920645e+35, -6.325248356907143e+35, -6.339419550341551e+35, -6.353344632119963e+35, -6.367030154981132e+35, -6.3804830733987255e+35, -6.3937107150029265e+35, -6.40672075033399e+35, -6.4195211609785615e+35, -6.4321202061622495e+35, -6.444526387887806e+35, -6.456748414736942e+35, -6.468795164473006e+35, -6.480675645601279e+35, -6.492398958069838e+35, -6.50397425330996e+35, -6.5154106938335645e+35, -6.5267174126224674e+35, -6.5379034725549335e+35, -6.5489778261296475e+35, -6.559949275748969e+35, -6.5708264348319445e+35, -6.581617690026846e+35, -6.592331164789049e+35, -6.602974684587207e+35, -6.613555743986103e+35, -6.624081475848141e+35, -6.634558622875153e+35, -6.644993511696735e+35, -6.655392029690615e+35, -6.665759604696438e+35, -6.676101187762945e+35, -6.686421239042299e+35, -6.696723716917106e+35, -6.707012070422056e+35, -6.717289234996792e+35, -6.727557631573176e+35, -6.737819168983798e+35, -6.748075249646688e+35, -6.758326778462342e+35, -6.768574174832509e+35, -6.778817387694728e+35, -6.789055913442924e+35, -6.799288816592125e+35, -6.809514753023857e+35, -6.819731995642861e+35, -6.829938462257882e+35, -6.840131745491154e+35, -6.850309144517178e+35, -6.860467698414892e+35, -6.870604220923325e+35, -6.880715336377676e+35, -6.890797516604108e+35, -6.900847118547044e+35, -6.910860422404535e+35, -6.920833670041957e+35, -6.930763103459497e+35, -6.940645003085869e+35, -6.950475725673044e+35, -6.960251741571182e+35, -6.969969671160738e+35, -6.979626320227596e+35, -6.98921871406667e+35, -6.998744130107599e+35, -7.008200128861286e+35, -7.017584582991193e+35, -7.026895704326272e+35, -7.036132068637683e+35, -7.045292638015086e+35, -7.054376780690515e+35, -7.063384288171886e+35, -7.072315389563463e+35, -7.081170762970208e+35, -7.08995154389844e+35, -7.098659330587376e+35, -7.107296186229214e+35, -7.115864638053931e+35, -7.124367673281776e+35, -7.132808731970264e+35, -7.141191696805042e+35, -7.149520879910768e+35, -7.157801006780367e+35, -7.166037197448616e+35, -7.174234945052928e+35, -7.182400091952451e+35, -7.190538803591867e+35, -7.198657540317827e+35, -7.206763027370722e+35, -7.214862223288009e+35, -7.222962286969256e+35, -7.231070543659695e+35, -7.239194450114537e+35, -7.247341559213379e+35, -7.25551948428964e+35, -7.263735863438878e+35, -7.271998324067018e+35, -7.280314447927233e+35, -7.288691736887583e+35, -7.297137579654517e+35, -7.30565921966616e+35, -7.314263724350145e+35, -7.322957955921556e+35, -7.331748543879892e+35, -7.340641859338749e+35, -7.349643991304621e+35, -7.35876072499633e+35, -7.36799752227538e+35, -7.377359504237661e+35, -7.386851435991431e+35, -7.396477713630599e+35, -7.406242353388829e+35, -7.416148982944865e+35, -7.42620083482822e+35, -7.436400741863171e+35, -7.446751134571522e+35, -7.457254040443934e+35, -7.467911084978168e+35, -7.478723494374456e+35, -7.489692099770717e+35, -7.500817342895432e+35, -7.512099283011463e+35, -7.523537605024313e+35, -7.535131628626815e+35, -7.546880318352721e+35, -7.558782294416606e+35, -7.570835844220786e+35, -7.583038934412821e+35, -7.59538922338877e+35, -7.607884074137791e+35, -7.620520567337678e+35, -7.633295514615139e+35, -7.646205471896093e+35, -7.659246752779412e+35, -7.672415441877653e+35, -7.685707408075516e+35, -7.69911831766967e+35, -7.712643647360813e+35, -7.726278697074961e+35, -7.740018602605783e+35, -7.753858348068822e+35, -7.76779277817339e+35, -7.7818166103178e+35, -7.795924446523748e+35, -7.8101107852258e+35, -7.824370032939415e+35, -7.838696515829268e+35, -7.853084491204944e+35, -7.867528158968371e+35, -7.882021673037965e+35, -7.896559152775835e+35, -7.91113469443448e+35, -7.925742382644693e+35, -7.940376301956919e+35, -7.955030548444697e+35, -7.969699241374111e+35, -7.984376534937147e+35, -7.999056630039914e+35, -8.013733786131574e+35, -8.028402333049981e+35, -8.043056682860225e+35, -8.05769134164682e+35, -8.07230092122279e+35, -8.086880150706729e+35, -8.101423887917912e+35, -8.115927130534635e+35, -8.13038502695283e+35, -8.144792886786361e+35, -8.159146190942299e+35, -8.173440601205836e+35, -8.187671969269576e+35, -8.201836345141812e+35, -8.215929984870587e+35, -8.229949357522984e+35, -8.243891151362017e+35, -8.257752279170279e+35, -8.271529882670092e+35, -8.285221335999933e+35, -8.298824248213089e+35, -8.312336464766311e+35, -8.325756067981314e+35, -8.339081376462355e+35, -8.3523109434655e+35, -8.365443554221677e+35, -8.378478222221224e+35, -8.391414184479647e+35, -8.404250895807421e+35, -8.416988022118562e+35, -8.429625432813801e+35, -8.442163192288436e+35, -8.454601550613563e+35, -8.466940933451855e+35, -8.479181931269282e+35, -8.491325287915115e+35, -8.503371888639995e+35, -8.515322747635253e+35, -8.527178995171366e+35, -8.538941864424542e+35, -8.550612678079547e+35, -8.56219283480159e+35, -8.573683795672347e+35, -8.585087070687269e+35, -8.596404205413308e+35, -8.607636767908371e+35, -8.618786336004994e+35, -8.629854485060392e+35, -8.640842776276474e+35, -8.651752745694313e+35, -8.662585893962442e+35, -8.673343676983359e+35, -8.68402749753509e+35, -8.69463869796429e+35, -8.705178554042003e+35, -8.715648270071738e+35, -8.726048975329242e+35, -8.736381721909774e+35, -8.746647484048544e+35, -8.756847158974068e+35, -8.766981569338856e+35, -8.777051467266586e+35, -8.787057540034232e+35, -8.797000417404177e+35, -8.806880680595905e+35, -8.816698872877544e+35, -8.826455511740774e+35, -8.836151102600348e+35, -8.845786153947954e+35, -8.855361193867504e+35, -8.864876787800518e+35, -8.874333557435968e+35, -8.88373220057566e+35, -8.893073511812444e+35, -8.902358403841536e+35, -8.911587929207798e+35, -8.920763302280058e+35, -8.929885921231027e+35, -8.938957389789047e+35, -8.94797953852214e+35, -8.956954445407378e+35, -8.965884455436037e+35, -8.974772199003842e+35, -8.983620608840219e+35, -8.992432935231906e+35, -9.00121275930883e+35, -9.009964004169755e+35, -9.018690943639807e+35, -9.027398208470313e+35, -9.036090789812151e+35, -9.04477403981444e+35, -9.05345366923243e+35, -9.062135741948212e+35, -9.070826666344597e+35, -9.079533183502591e+35, -9.088262352224449e+35, -9.097021530920345e+35, -9.105818356430223e+35, -9.114660719889688e+35, -9.123556739780469e+35, -9.13251473234327e+35, -9.14154317956063e+35, -9.150650694952703e+35, -9.15984598745519e+35, -9.169137823677967e+35, -9.17853498886724e+35, -9.188046246914486e+35, -9.19768029977475e+35, -9.207445746672382e+35, -9.217351043478374e+35, -9.227404462656682e+35, -9.23761405417084e+35, -9.247987607748343e+35, -9.258532616884262e+35, -9.269256244962942e+35, -9.280165293851682e+35, -9.291266175307152e+35, -9.3025648855063e+35, -9.314066982983177e+35, -9.325777570224443e+35, -9.337701279135104e+35, -9.349842260549128e+35, -9.362204177912342e+35, -9.374790205224973e+35, -9.387603029278716e+35, -9.400644856175471e+35, -9.413917422064261e+35, -9.42742200798338e+35, -9.441159458641173e+35, -9.455130204920892e+35, -9.469334289844702e+35, -9.483771397685657e+35, -9.498440885871771e+35, -9.513341819284375e+35, -9.528473006514784e+35, -9.543833037610944e+35, -9.559420322815744e+35, -9.575233131774663e+35, -9.59126963267435e+35, -9.60752793075937e+35, -9.624006105670199e+35, -9.640702247047936e+35, -9.657614487855282e+35, -9.674741034883456e+35, -9.692080195931771e+35, -9.709630403179012e+35, -9.727390232299082e+35, -9.745358416915868e+35, -9.76353385804122e+35, -9.781915628191914e+35, -9.800502969940717e+35, -9.81929528872155e+35, -9.838292139771635e+35, -9.857493209167832e+35, -9.876898288982963e+35, -9.896507246659682e+35, -9.916319988777147e+35, -9.936336419452727e+35, -9.956556393694677e+35, -9.976979666086005e+35, -9.997605835248182e+35, -1.0018434284586769e+36, -1.0039464119878509e+36, -1.0060694104306187e+36, -1.0082122591585242e+36, -1.010374745786302e+36, -1.012556603308979e+36, -1.0147575032586103e+36, -1.0169770489528312e+36, -1.0192147689082097e+36, -1.0214701104896865e+36, -1.0237424338661992e+36, -1.0260310063393735e+36, -1.0283349971092825e+36, -1.0306534725366379e+36, -1.0329853919565718e+36, -1.0353296040931517e+36, -1.0376848441184483e+36, -1.0400497313933433e+36, -1.0424227679208345e+36, -1.0448023375356048e+36, -1.0471867058465117e+36, -1.0495740209421684e+36, -1.0519623148617621e+36, -1.0543495058273802e+36, -1.0567334012267097e+36, -1.0591117013289423e+36, -1.061482003710469e+36, -1.0638418083613626e+36, -1.0661885234385389e+36, -1.0685194716266114e+36, -1.0708318970636618e+36, -1.0731229727851799e+36, -1.0753898086370276e+36, -1.077629459605262e+36, -1.0798389345095452e+36, -1.0820152050053618e+36, -1.0841552148396231e+36, -1.086255889304503e+36, -1.088314144834781e+36, -1.0903268986945606e+36, -1.0922910787013253e+36, -1.0942036329366863e+36, -1.096061539395366e+36, -1.097861815526881e+36, -1.0996015276266945e+36, -1.1012778000367693e+36, -1.1028878241187676e+36, -1.1044288669660153e+36, -1.1058982798239818e+36, -1.1072935061921843e+36, -1.1086120895835845e+36, -1.1098516809211259e+36, -1.111010045553617e+36, -1.1120850698767697e+36, -1.1130747675473215e+36, -1.1139772852813764e+36, -1.1147909082300745e+36, -1.1155140649280901e+36, -1.1161453318124276e+36, -1.1166834373104108e+36, -1.1171272654977028e+36, -1.117475859328026e+36, -1.1177284234374581e+36, -1.1178843265269099e+36, -1.1179431033270812e+36, -1.117904456150484e+36, -1.1177682560353814e+36, -1.1175345434865819e+36, -1.1172035288181307e+36, -1.1167755921023758e+36, -1.1162512827300857e+36, -1.1156313185857347e+36, -1.1149165848417234e+36, -1.1141081323750774e+36, -1.1132071758096353e+36, -1.1122150911863597e+36, -1.1111334132643788e+36, -1.1099638324548646e+36, -1.1087081913897041e+36, -1.1073684811270669e+36, -1.105946836995786e+36, -1.1044455340808933e+36, -1.1028669823527367e+36, -1.1012137214427525e+36, -1.0994884150694013e+36, -1.0976938451188779e+36, -1.0958329053857108e+36, -1.0939085949797956e+36, -1.0919240114074324e+36, -1.0898823433356735e+36, -1.0877868630502943e+36, -1.0856409186200844e+36, -1.0834479257811839e+36, -1.081211359557873e+36, -1.0789347456376254e+36, -1.0766216515206744e+36, -1.0742756774663334e+36, -1.0719004472604235e+36, -1.0694995988304482e+36, -1.067076774737131e+36, -1.0646356125730533e+36, -1.0621797353011619e+36, -1.0597127415676079e+36, -1.057238196025213e+36, -1.054759619705348e+36, -1.052280480477147e+36, -1.0498041836344097e+36, -1.0473340626508588e+36, -1.0448733701451997e+36, -1.0424252690978413e+36, -1.0399928243601057e+36, -1.0375789944975481e+36, -1.0351866240069671e+36, -1.032818435946258e+36, -1.0304770250142873e+36, -1.0281648511162804e+36, -1.025884233447991e+36, -1.0236373451289126e+36, -1.0214262084127834e+36, -1.0192526904994729e+36, -1.0171184999698948e+36, -1.01502518386127e+36, -1.0129741253968347e+36, -1.0109665423797982e+36, -1.0090034862575987e+36, -1.0070858418585115e+36, -1.0052143277982665e+36, -1.003389497550681e+36, -1.0016117411722493e+36, -9.99881287666622e+35, -9.981982079713962e+35, -9.965624185460009e+35, -9.94973685536252e+35, -9.934316294876414e+35, -9.919357305773572e+35, -9.904853343314618e+35, -9.890796577921897e+35, -9.877177960977681e+35, -9.863987294357555e+35, -9.851213303293321e+35, -9.83884371214864e+35, -9.826865322684101e+35, -9.81526409438069e+35, -9.804025226391011e+35, -9.79313324068771e+35, -9.782572065979583e+35, -9.77232512197516e+35, -9.762375403579108e+35, -9.752705564618765e+35, -9.743298000711025e+35, -9.734134930893296e+35, -9.725198477658387e+35, -9.716470745053812e+35, -9.707933894522593e+35, -9.69957021818394e+35, -9.691362209276085e+35, -9.683292629500246e+35, -9.675344573039119e+35, -9.667501527031064e+35, -9.65974742832298e+35, -9.652066716334282e+35, -9.64444438189903e+35, -9.636866011971348e+35, -9.629317830106969e+35, -9.621786732656333e+35, -9.614260320627288e+35, -9.606726927198268e+35, -9.59917564088673e+35, -9.591596324395008e+35, -9.583979629181112e+35, -9.57631700581569e+35, -9.56860071020961e+35, -9.560823805812078e+35, -9.552980161894905e+35, -9.545064448054917e+35, -9.537072125080823e+35, -9.528999432343053e+35, -9.520843371878032e+35, -9.512601689346863e+35, -9.504272852063184e+35, -9.495856024287085e+35, -9.487351039994137e+35, -9.478758373334083e+35, -9.470079106997132e+35, -9.461314898712201e+35, -9.452467946103911e+35, -9.44354095013588e+35, -9.434537077372566e+35, -9.425459921286503e+35, -9.416313462840942e+35, -9.407102030575227e+35, -9.397830260414597e+35, -9.388503055423707e+35, -9.379125545720225e+35, -9.36970304875235e+35, -9.360241030143602e+35, -9.350745065296073e+35, -9.341220801936535e+35, -9.331673923775966e+35, -9.322110115449239e+35, -9.312535028881526e+35, -9.30295425122342e+35, -9.293373274478681e+35, -9.283797466936666e+35, -9.27423204650947e+35, -9.264682056055397e+35, -9.255152340758974e+35, -9.245647527623242e+35, -9.236172007111489e+35, -9.226729916965384e+35, -9.217325128210613e+35, -9.207961233343585e+35, -9.198641536684769e+35, -9.18936904686786e+35, -9.180146471420467e+35, -9.170976213383458e+35, -9.161860369903002e+35, -9.152800732720826e+35, -9.143798790478212e+35, -9.134855732742657e+35, -9.125972455658587e+35, -9.117149569118181e+35, -9.108387405345655e+35, -9.09968602877954e+35, -9.091045247144346e+35, -9.082464623591079e+35, -9.073943489795118e+35, -9.065480959893673e+35, -9.057075945151219e+35, -9.048727169239759e+35, -9.04043318402561e+35, -9.032192385757285e+35, -9.024003031552537e+35, -9.015863256084852e+35, -9.007771088377913e+35, -8.999724468616933e+35, -8.991721264891348e+35, -8.983759289790028e+35, -8.975836316769856e+35, -8.967950096227878e+35, -8.960098371206463e+35, -8.952278892668526e+35, -8.944489434280892e+35, -8.936727806648405e+35, -8.928991870943425e+35, -8.921279551879156e+35, -8.913588849978996e+35, -8.905917853091774e+35, -8.898264747112358e+35, -8.890627825862033e+35, -8.883005500091803e+35, -8.875396305567939e+35, -8.867798910207891e+35, -8.860212120231177e+35, -8.852634885296033e+35, -8.845066302596852e+35, -8.837505619895417e+35, -8.829952237469107e+35, -8.822405708958573e+35, -8.814865741103457e+35, -8.807332192360516e+35, -8.799805070402026e+35, -8.792284528499854e+35, -8.784770860804326e+35, -8.777264496536516e+35, -8.769765993117425e+35, -8.762276028264124e+35, -8.754795391092895e+35, -8.747324972272244e+35, -8.739865753282967e+35, -8.73241879484119e+35, -8.724985224555745e+35, -8.717566223892938e+35, -8.710163014530027e+35, -8.702776844185218e+35, -8.695408972017816e+35, -8.688060653694961e+35, -8.680733126228439e+35, -8.673427592685473e+35, -8.666145206882326e+35, -8.658887058168007e+35, -8.651654156408492e+35, -8.644447417277979e+35, -8.637267647964368e+35, -8.630115533390937e+35, -8.622991623053236e+35, -8.615896318563495e+35, -8.608829861991089e+35, -8.601792325076475e+35, -8.59478359939077e+35, -8.587803387503827e+35, -8.58085119521256e+35, -8.573926324873234e+35, -8.567027869869293e+35, -8.560154710236816e+35, -8.553305509457254e+35, -8.54647871242089e+35, -8.539672544546004e+35, -8.532885012039575e+35, -8.526113903263821e+35, -8.519356791175948e+35, -8.512611036789504e+35, -8.505873793607158e+35, -8.499142012963694e+35, -8.492412450213745e+35, -8.485681671694362e+35, -8.478946062391664e+35, -8.472201834234038e+35, -8.46544503493908e+35, -8.458671557336251e+35, -8.451877149092137e+35, -8.44505742276455e+35, -8.43820786611548e+35, -8.431323852617123e+35, -8.424400652088011e+35, -8.417433441400726e+35, -8.410417315209963e+35, -8.40334729665105e+35, -8.396218347968578e+35, -8.389025381037857e+35, -8.381763267746998e+35, -8.374426850214289e+35, -8.367010950818751e+35, -8.359510382026434e+35, -8.351919956001766e+35, -8.344234493989376e+35, -8.336448835465543e+35, -8.32855784704904e+35, -8.32055643117298e+35, -8.312439534511703e+35, -8.304202156162668e+35, -8.295839355580808e+35, -8.287346260260733e+35, -8.278718073163237e+35, -8.269950079878485e+35, -8.2610376555151e+35, -8.251976271305431e+35, -8.242761500908902e+35, -8.233389026396057e+35, -8.223854643891789e+35, -8.214154268851258e+35, -8.204283940941195e+35, -8.194239828494537e+35, -8.184018232506035e+35, -8.17361559013151e+35, -8.163028477654831e+35, -8.152253612884062e+35, -8.141287856938364e+35, -8.130128215388345e+35, -8.118771838714905e+35, -8.107216022051054e+35, -8.095458204178746e+35, -8.083495965753782e+35, -8.071327026739054e+35, -8.058949243032673e+35, -8.046360602283185e+35, -8.033559218895773e+35, -8.020543328238542e+35, -8.007311280071629e+35, -7.993861531230273e+35, -7.98019263760492e+35, -7.966303245474786e+35, -7.952192082261884e+35, -7.937857946786791e+35, -7.92329969911808e+35, -7.908516250123442e+35, -7.893506550840541e+35, -7.87826958179894e+35, -7.86280434243416e+35, -7.847109840752229e+35, -7.831185083402945e+35, -7.815029066339393e+35, -7.798640766241594e+35, -7.782019132891585e+35, -7.765163082690842e+35, -7.74807149351378e+35, -7.730743201090125e+35, -7.713176997109425e+35, -7.69537162923351e+35, -7.677325803199735e+35, -7.659038187184418e+35, -7.640507418586172e+35, -7.621732113373036e+35, -7.602710878118576e+35, -7.583442324831229e+35, -7.563925088658578e+35, -7.5441578485185e+35, -7.524139350684283e+35, -7.503868435312409e+35, -7.483344065873742e+35, -7.462565361405696e+35, -7.44153163146983e+35, -7.42024241365664e+35, -7.398697513433815e+35, -7.376897046102001e+35, -7.354841480567711e+35, -7.332531684608851e+35, -7.309968971261919e+35, -7.287155145922475e+35, -7.264092553706977e+35, -7.240784126592196e+35, -7.217233429811447e+35, -7.193444706956217e+35, -7.169422923209842e+35, -7.145173806111049e+35, -7.120703883236023e+35, -7.096020516172676e+35, -7.071131930157806e+35, -7.046047238752378e+35, -7.020776462935825e+35, -6.995330544023827e+35, -6.969721349835559e+35, -6.943961673573762e+35, -6.918065224923422e+35, -6.892046612929175e+35, -6.865921320270922e+35, -6.8397056686318e+35, -6.813416774929945e+35, -6.787072498278246e+35, -6.76069137763176e+35, -6.734292560189037e+35, -6.707895720728514e+35, -6.681520972177663e+35, -6.655188767839001e+35, -6.628919795823197e+35, -6.6027348663704464e+35, -6.576654792865111e+35, -6.550700267482138e+35, -6.524891732517401e+35, -6.4992492485685846e+35, -6.473792360840116e+35, -6.4485399649283924e+35, -6.4235101735201325e+35, -6.398720185491537e+35, -6.3741861589273934e+35, -6.34992308959368e+35, -6.325944696377962e+35, -6.302263315176952e+35, -6.278889802646264e+35, -6.2558334511258045e+35, -6.233101915950902e+35, -6.210701156204862e+35, -6.188635389815988e+35, -6.1669070637108645e+35, -6.145516839546239e+35, -6.124463595325573e+35, -6.10374444299441e+35, -6.083354761883351e+35, -6.063288247655444e+35, -6.043536976196437e+35, -6.0240914816888886e+35, -6.00494084792237e+35, -5.9860728117237676e+35, -5.967473877247046e+35, -5.949129439731815e+35, -5.9310239172548184e+35, -5.91314088891344e+35, -5.895463237849622e+35, -5.877973297495969e+35, -5.8606529994444495e+35, -5.843484021361009e+35, -5.826447933435482e+35, -5.809526341924317e+35, -5.792701028441256e+35, -5.775954083756716e+35, -5.759268034985908e+35, -5.742625965173812e+35, -5.726011624419653e+35, -5.7094095318201716e+35, -5.692805067647472e+35, -5.676184555317214e+35, -5.659535332832454e+35, -5.642845813521285e+35, -5.626105536003345e+35, -5.609305203440641e+35, -5.5924367122313e+35, -5.575493170403146e+35, -5.5584689060533794e+35, -5.541359466259582e+35, -5.524161606954335e+35, -5.506873274318001e+35, -5.489493578293722e+35, -5.472022758866281e+35, -5.454462145781635e+35, -5.436814112405952e+35, -5.4190820244333925e+35, -5.401270184163668e+35, -5.383383771063754e+35, -5.365428779322183e+35, -5.347411953087392e+35, -5.329340720059589e+35, -5.311223124078632e+35, -5.293067757316109e+35, -5.274883692643396e+35, -5.2566804167056245e+35, -5.238467764184421e+35, -5.220255853687587e+35, -5.202055025651918e+35, -5.183875782593608e+35, -5.165728731990855e+35, -5.1476245320271665e+35, -5.129573840378337e+35, -5.111587266168063e+35, -5.093675325176578e+35, -5.0758483983351174e+35, -5.058116693497055e+35, -5.040490210440352e+35, -5.0229787090153514e+35, -5.00559168032313e+35, -4.988338320782264e+35, -4.971227508917659e+35, -4.954267784690126e+35, -4.9374673311672356e+35, -4.9208339583338445e+35, -4.9043750888323166e+35, -4.888097745422006e+35, -4.872008539958343e+35, -4.856113663693153e+35, -4.840418878714369e+35, -4.8249295103579735e+35, -4.8096504404412706e+35, -4.794586101188255e+35, -4.7797404697409535e+35, -4.765117063170027e+35, -4.7507189339291876e+35, -4.736548665714805e+35, -4.7226083697246516e+35, -4.708899681327547e+35, -4.695423757186617e+35, -4.682181272892096e+35, -4.669172421189753e+35, -4.6563969109040134e+35, -4.643853966672195e+35, -4.631542329623814e+35, -4.619460259144395e+35, -4.6076055358789035e+35, -4.595975466127012e+35, -4.584566887791397e+35, -4.5733761780361864e+35, -4.562399262805571e+35, -4.5516316283492e+35, -4.541068334884962e+35, -4.530704032519412e+35, -4.520532979523166e+35, -4.5105490630424854e+35, -4.500745822301749e+35, -4.4911164743251024e+35, -4.4816539421801154e+35, -4.47235088571137e+35, -4.4631997347047764e+35, -4.4541927243863e+35, -4.44532193312814e+35, -4.436579322200198e+35, -4.4279567773701764e+35, -4.419446152124591e+35, -4.411039312250572e+35, -4.4027281814870955e+35, -4.39450478793076e+35, -4.386361310849889e+35, -4.3782901275475644e+35, -4.370283859888086e+35, -4.362335420093794e+35, -4.354438055405338e+35, -4.346585391198392e+35, -4.338771472142775e+35, -4.330990801003485e+35, -4.323238374687004e+35, -4.315509717151154e+35, -4.307800908823645e+35, -4.3001086121893965e+35, -4.292430093243801e+35, -4.2847632385384696e+35, -4.277106567583913e+35, -4.269459240415557e+35, -4.261821060169669e+35, -4.254192470564213e+35, -4.246574548225272e+35, -4.238968989848365e+35, -4.231378094231612e+35, -4.223804739268107e+35, -4.216252354029821e+35, -4.208724886123932e+35, -4.201226764545067e+35, -4.1937628582892335e+35, -4.186338431034617e+35, -4.178959092227366e+35, -4.171630744946013e+35, -4.1643595309413944e+35, -4.157151773276342e+35, -4.150013917003329e+35, -4.142952468340312e+35, -4.135973932803936e+35, -4.1290847527764885e+35, -4.1222912449752e+35, -4.115599538292011e+35, -4.1090155124667265e+35, -4.10254473804372e+35, -4.096192418044427e+35, -4.089963331778848e+35, -4.083861781190201e+35, -4.077891540107126e+35, -4.072055806755975e+35, -4.066357159852539e+35, -4.060797518569975e+35, -4.0553781066461556e+35, -4.050099420867312e+35, -4.044961204129542e+35, -4.039962423252107e+35, -4.035101251684666e+35, -4.0303750572172326e+35, -4.025780394774637e+35, -4.021313004342507e+35, -4.0169678140457726e+35, -4.01273894836802e+35, -4.008619741474208e+35, -4.0046027555680946e+35, -4.000679804190037e+35, -3.996841980333973e+35, -3.99307968923492e+35, -3.989382685652957e+35, -3.9857401154560276e+35, -3.9821405612765826e+35, -3.978572091997004e+35, -3.975022315790471e+35, -3.9714784364286745e+35, -3.967927312538903e+35, -3.964355519476859e+35, -3.9607494134614364e+35, -3.9570951975987504e+35, -3.953378989405894e+35, -3.9495868894313084e+35, -3.945705050554334e+35, -3.941719747536307e+35, -3.9376174463867675e+35, -3.933384873105224e+35, -3.929009081352197e+35, -3.9244775186090736e+35, -3.919778090388053e+35, -3.914899222061785e+35, -3.909829917894978e+35, -3.90455981687733e+35, -3.8990792449758986e+35, -3.8933792634492106e+35, -3.887451712894712e+35, -3.8812892527322e+35, -3.874885395860851e+35, -3.868234538266183e+35, -3.8613319833964205e+35, -3.8541739611665864e+35, -3.846757641500057e+35, -3.839081142360338e+35, -3.831143532273924e+35, -3.822944827394206e+35, -3.814485983202439e+35, -3.805768880989591e+35, -3.796796309305759e+35, -3.7875719406084985e+35, -3.778100303381473e+35, -3.768386750029783e+35, -3.758437420894653e+35, -3.748259204758871e+35, -3.737859696238595e+35, -3.727247150482434e+35, -3.716430435610727e+35, -3.705418983342335e+35, -3.69422273826665e+35, -3.6828521062148034e+35, -3.671317902186019e+35, -3.65963129828027e+35, -3.647803772071995e+35, -3.6358470558447464e+35, -3.6237730870909376e+35, -3.611593960650729e+35, -3.5993218828372785e+35, -3.586969127864238e+35, -3.574547996853924e+35, -3.562070779666175e+35, -3.549549719744387e+35, -3.536996982130734e+35, -3.5244246247578595e+35, -3.5118445730688e+35, -3.499268597975827e+35, -3.486708297111545e+35, -3.474175079275993e+35, -3.4616801519381665e+35, -3.449234511595541e+35, -3.436848936755845e+35, -3.424533983257504e+35, -3.412299981609495e+35, -3.400157035997859e+35, -3.388115024573617e+35, -3.3761836006211e+35, -3.3643721941852616e+35, -3.352690013731493e+35, -3.341146047410131e+35, -3.329749063508296e+35, -3.3185076096837517e+35, -3.307430010605088e+35, -3.2965243636498496e+35, -3.2857985323536417e+35, -3.2752601373511403e+35, -3.2649165445986922e+35, -3.254774850727447e+35, -3.244841865438683e+35, -3.2351240909138988e+35, -3.225627698282377e+35, -3.216358501254078e+35, -3.2073219270914434e+35, -3.198522985161088e+35, -3.1899662333657104e+35, -3.1816557428172978e+35, -3.173595061162534e+35, -3.165787175022513e+35, -3.1582344720480106e+35, -3.150938703124309e+35, -3.143900945287378e+35, -3.1371215659313185e+35, -3.130600188894154e+35, -3.124335663011861e+35, -3.1183260337220332e+35, -3.1125685182817123e+35, -3.107059485138696e+35, -3.1017944379632697e+35, -3.0967680048032586e+35, -3.091973932779985e+35, -3.0874050886843794e+35, -3.0830534657695003e+35, -3.0789101969719096e+35, -3.074965574715635e+35, -3.071209077383417e+35, -3.067629402451845e+35, -3.0642145062155846e+35, -3.0609516499348754e+35, -3.057827452166034e+35, -3.054827946950791e+35, -3.0519386474660302e+35, -3.049144614661645e+35, -3.046430530345958e+35, -3.0437807741203217e+35, -3.041179503505826e+35, -3.038610736563316e+35, -3.0360584362705857e+35, -3.033506595891662e+35, -3.030939324558165e+35, -3.028340932277348e+35, -3.025696013580454e+35, -3.0229895290463417e+35, -3.0202068839541843e+35, -3.017334003355959e+35, -3.014357402904147e+35, -3.0112642548184383e+35, -3.0080424484367844e+35, -3.00468064486324e+35, -3.001168325292925e+35, -2.997495832671489e+35, -2.9936544064254587e+35, -2.98963621007783e+35, -2.9854343516456678e+35, -2.9810428967932823e+35, -2.976456874796087e+35, -2.971672277441088e+35, -2.9666860510639523e+35, -2.961496081986417e+35, -2.9561011756795412e+35, -2.9505010300315686e+35, -2.944696203149269e+35, -2.938688076159213e+35, -2.9324788115117487e+35, -2.9260713073152746e+35, -2.9194691482487456e+35, -2.912676553611319e+35, -2.905698323074539e+35, -2.8985397806997595e+35, -2.8912067177772604e+35, -2.883705335028233e+35, -2.876042184696175e+35, -2.8682241130239976e+35, -2.86025820359425e+35, -2.852151721969396e+35, -2.8439120620419138e+35, -2.835546694461775e+35, -2.8270631174731688e+35, -2.818468810448013e+35, -2.8097711903652763e+35, -2.8009775714433387e+35, -2.792095128084182e+35, -2.7831308612587318e+35, -2.7740915684085014e+35, -2.7649838169105484e+35, -2.755813921110477e+35, -2.7465879228929523e+35, -2.7373115757294494e+35, -2.727990332109531e+35, -2.7186293342410746e+35, -2.7092334078709335e+35, -2.699807059068433e+35, -2.6903544737879715e+35, -2.680879520017643e+35, -2.6713857523065235e+35, -2.661876418459048e+35, -2.6523544681759274e+35, -2.6428225634230125e+35, -2.633283090305775e+35, -2.6237381722350804e+35, -2.614189684169249e+35, -2.604639267731369e+35, -2.5950883470015003e+35, -2.5855381448024033e+35, -2.5759896993017664e+35, -2.5664438807690894e+35, -2.556901408341023e+35, -2.5473628666558314e+35, -2.5378287222385938e+35, -2.5282993395289526e+35, -2.518774996455747e+35, -2.5092558994807056e+35, -2.4997421980414447e+35, -2.490233998340334e+35, -2.4807313764366422e+35, -2.4712343906082342e+35, -2.461743092963909e+35, -2.45225754029219e+35, -2.442777804145688e+35, -2.4333039801634267e+35, -2.423836196642994e+35, -2.4143746223820592e+35, -2.404919473806542e+35, -2.3954710214158344e+35, -2.386029595572488e+35, -2.37659559166739e+35, -2.3671694746937872e+35, -2.3577517832626285e+35, -2.348343133094767e+35, -2.3389442200191433e+35, -2.3295558225102444e+35, -2.3201788037927005e+35, -2.3108141135400152e+35, -2.3014627891905865e+35, -2.2921259569015148e+35, -2.2828048321572956e+35, -2.2735007200454346e+35, -2.2642150152106653e+35, -2.2549492014915415e+35, -2.245704851242367e+35, -2.236483624339933e+35, -2.2272872668701348e+35, -2.2181176094865968e+35, -2.2089765654341475e+35, -2.199866128222905e+35, -2.190788368943456e+35, -2.1817454332064446e+35, -2.172739537696602e+35, -2.1637729663244717e+35, -2.154848065967849e+35, -2.145967241789609e+35, -2.1371329521272858e+35, -2.128347702946996e+35, -2.1196140418642013e+35, -2.1109345517314587e+35, -2.102311843801971e+35, -2.0937485504817175e+35, -2.08524731768622e+35, -2.0768107968250932e+35, -2.0684416364395164e+35, -2.060142473527316e+35, -2.051915924589396e+35, -2.043764576438525e+35, -2.0356909768155375e+35, -2.0276976248583632e+35, -2.0197869614749752e+35, -2.0119613596708063e+35, -2.004223114882351e+35, -1.9965744353726156e+35, -1.9890174327363277e+35, -1.9815541125716914e+35, -1.974186365364236e+35, -1.9669159576340867e+35, -1.959744523389817e+35, -1.9526735559337377e+35, -1.9457044000570575e+35, -1.9388382446621166e+35, -1.9320761158464933e+35, -1.9254188704781047e+35, -1.9188671902924e+35, -1.9124215765351325e+35, -1.906082345176339e+35, -1.8998496227179683e+35, -1.8937233426161708e+35, -1.8877032423396574e+35, -1.8817888610831777e+35, -1.875979538156815e+35, -1.8702744120712526e+35, -1.8646724203372572e+35, -1.8591723000015355e+35, -1.853772588935353e+35, -1.8484716278987425e+35, -1.8432675633934412e+35, -1.8381583513250113e+35, -1.8331417614849976e+35, -1.8282153828667738e+35, -1.823376629820007e+35, -1.8186227490486557e+35, -1.8139508274493404e+35, -1.809357800780566e+35, -1.80484046315192e+35, -1.800395477306902e+35, -1.796019385672304e+35, -1.7917086221374974e+35, -1.787459524517403e+35, -1.7832683476469034e+35, -1.7791312770474393e+35, -1.7750444430969172e+35, -1.771003935629808e+35, -1.7670058188899364e+35, -1.763046146748087e+35, -1.7591209780997666e+35, -1.7552263923499458e+35, -1.7513585048946717e+35, -1.7475134825039573e+35, -1.743687558515969e+35, -1.739877047751746e+35, -1.7360783610625575e+35, -1.732288019425239e+35, -1.7285026675086376e+35, -1.7247190866362146e+35, -1.7209342070753107e+35, -1.717145119595055e+35, -1.713349086234115e+35, -1.709543550231676e+35, -1.705726145080435e+35, -1.7018947026636026e+35, -1.698047260449718e+35, -1.6941820677189493e+35, -1.6902975908041227e+35, -1.6863925173324974e+35, -1.6824657594592513e+35, -1.678516456086138e+35, -1.6745439740640843e+35, -1.6705479083791874e+35, -1.6665280813238685e+35, -1.6624845406591072e+35, -1.658417556770889e+35, -1.654327618831064e+35, -1.6502154299672691e+35, -1.6460819014554314e+35, -1.641928145942265e+35, -1.637755469713178e+35, -1.6335653640186725e+35, -1.6293594954777102e+35, -1.6251396955778964e+35, -1.6209079492955873e+35, -1.6166663828663333e+35, -1.6124172507376369e+35, -1.6081629217427522e+35, -1.6039058645428925e+35, -1.5996486323898756e+35, -1.5953938472706279e+35, -1.5911441835011012e+35, -1.5869023508505979e+35, -1.5826710772826092e+35, -1.5784530914067896e+35, -1.5742511047533505e+35, -1.570067793979568e+35, -1.5659057831377185e+35, -1.5617676261312124e+35, -1.5576557895028757e+35, -1.5535726356967988e+35, -1.5495204069452921e+35, -1.5455012099315747e+35, -1.5415170013807162e+35, -1.5375695747306093e+35, -1.5336605480294755e+35, -1.5297913532002114e+35, -1.525963226802208e+35, -1.5221772024128314e+35, -1.5184341047290275e+35, -1.5147345454799842e+35, -1.5110789212177041e+35, -1.5074674130303733e+35, -1.5038999882027869e+35, -1.500376403817581e+35, -1.496896212270097e+35, -1.4934587686377014e+35, -1.4900632398198399e+35, -1.4867086153377733e+35, -1.4833937196565031e+35, -1.4801172258683672e+35, -1.4768776705561984e+35, -1.4736734696335582e+35, -1.4705029349469168e+35, -1.4673642914093604e+35, -1.4642556944323218e+35, -1.4611752474152695e+35, -1.4581210190550267e+35, -1.455091060245303e+35, -1.452083420342431e+35, -1.4490961625942352e+35, -1.446127378539517e+35, -1.4431752012158126e+35, -1.4402378170310439e+35, -1.4373134761858182e+35, -1.4344005015581096e+35, -1.4314972959975471e+35, -1.4286023479979896e+35, -1.425714235754793e+35, -1.4228316296344805e+35, -1.419953293115573e+35, -1.4170780822822857e+35, -1.4142049439721914e+35, -1.4113329127018202e+35, -1.4084611065034194e+35, -1.4055887218243874e+35, -1.4027150276423813e+35, -1.399839358957973e+35, -1.3969611098259161e+35, -1.394079726084089e+35, -1.3911946979327143e+35, -1.3883055525127495e+35, -1.3854118466149976e+35, -1.3825131596476023e+35, -1.379609086968107e+35, -1.3766992336790438e+35, -1.3737832089645596e+35, -1.3708606210344315e+35, -1.367931072723452e+35, -1.3649941577810443e+35, -1.3620494578715488e+35, -1.3590965402905651e+35, -1.3561349563922108e+35, -1.3531642407120652e+35, -1.3501839107561236e+35, -1.3471934674253073e+35, -1.3441923960323195e+35, -1.3411801678638097e+35, -1.33815624223922e+35, -1.3351200690119716e+35, -1.332071091458167e+35, -1.3290087494983863e+35, -1.3259324831974403e+35, -1.3228417364882016e+35, -1.3197359610680126e+35, -1.3166146204196167e+35, -1.3134771939089237e+35, -1.3103231809159197e+35, -1.3071521049612784e+35, -1.3039635177882943e+35, -1.3007570033708448e+35, -1.2975321818152238e+35, -1.2942887131279194e+35, -1.2910263008303472e+35, -1.287744695392744e+35, -1.2844436974760444e+35, -1.2811231609599334e+35, -1.2777829957476137e+35, -1.2744231703326996e+35, -1.2710437141195805e+35, -1.267644719488572e+35, -1.2642263435981813e+35, -1.2607888099205422e+35, -1.2573324095029582e+35, -1.2538575019543966e+35, -1.2503645161538349e+35, -1.2468539506796523e+35, -1.2433263739586634e+35, -1.2397824241384934e+35, -1.236222808682644e+35, -1.2326483036943211e+35, -1.2290597529706606e+35, -1.2254580667967047e+35, -1.2218442204834596e+35, -1.2182192526607497e+35, -1.2145842633337746e+35, -1.2109404117149194e+35, -1.2072889138451384e+35, -1.2036310400171792e+35, -1.199968112017723e+35, -1.1963015002049021e+35, -1.192632620439091e+35, -1.1889629308860035e+35, -1.1852939287121001e+35, -1.1816271466921892e+35, -1.1779641497508937e+35, -1.1743065314581508e+35, -1.1706559104989743e+35, -1.1670139271409364e+35, -1.1633822397152357e+35, -1.1597625211323832e+35, -1.15615645545038e+35, -1.1525657345099857e+35, -1.1489920546525167e+35, -1.1454371135324887e+35, -1.141902607034565e+35, -1.1383902263023219e+35, -1.134901654885606e+35, -1.1314385660073612e+35, -1.1280026199508436e+35, -1.124595461564435e+35, -1.1212187178789324e+35, -1.1178739958302672e+35, -1.1145628800774802e+35, -1.111286930904679e+35, -1.1080476821936683e+35, -1.1048466394531614e+35, -1.1016852778885973e+35, -1.0985650404966068e+35, -1.0954873361688062e+35, -1.0924535377881807e+35, -1.089464980303462e+35, -1.0865229587684697e+35, -1.083628726332184e+35, -1.080783492171737e+35, -1.0779884193584184e+35, -1.0752446226515052e+35, -1.0725531662185223e+35, -1.0699150612804372e+35, -1.0673312636858607e+35, -1.064802671421273e+35, -1.0623301220640635e+35, -1.0599143901947048e+35, -1.0575561847785215e+35, -1.0552561465375871e+35, -1.0530148453311156e+35, -1.0508327775650269e+35, -1.0487103636547757e+35, -1.0466479455647558e+35, -1.0446457844480775e+35, -1.0427040584114419e+35, -1.0408228604304605e+35, -1.0390021964368472e+35, -1.0372419836020076e+35, -1.0355420488360624e+35, -1.033902127522553e+35, -1.0323218625036063e+35, -1.0308008033339067e+35, -1.0293384058094165e+35, -1.02793403178608e+35, -1.0265869492888342e+35, -1.0252963329183368e+35, -1.0240612645534522e+35, -1.0228807343470344e+35, -1.0217536420103577e+35, -1.0206787983789041e+35, -1.0196549272455e+35, -1.0186806674537656e+35, -1.017754575232741e+35, -1.0168751267592358e+35, -1.0160407209282919e+35, -1.0152496823159987e+35, -1.0145002643119348e+35, -1.0137906524035131e+35, -1.01311896759119e+35, -1.0124832699153896e+35, -1.011881562074027e+35, -1.0113117931125929e+35, -1.0107718621680457e+35, -1.0102596222500985e+35, -1.0097728840420895e+35, -1.0093094197094442e+35, -1.0088669667003556e+35, -1.0084432315287016e+35, -1.0080358935286746e+35, -1.007642608574833e+35, -1.0072610127585527e+35, -1.006888726020623e+35, -1.0065233557342616e+35, -1.0061625002391838e+35, -1.0058037523286082e+35, -1.0054447026904397e+35, -1.0050829433074526e+35, -1.004716070822442e+35, -1.0043416898748421e+35, -1.0039574164157222e+35, -1.0035608810127864e+35, -1.0031497321516089e+35, -1.0027216395459023e+35, -1.0022742974661065e+35, -1.001805428097819e+35, -1.0013127849414415e+35, -1.0007941562617122e+35, -1.0002473685997407e+35, -9.996702903580492e+34, -9.990608354632453e+34, -9.984169671212414e+34, -9.977367016663724e+34, -9.970181125133445e+34, -9.96259334217408e+34, -9.954585666422774e+34, -9.946140792413038e+34, -9.937242154475926e+34, -9.92787397174025e+34, -9.918021294161225e+34, -9.907670049528138e+34, -9.89680709134092e+34, -9.885420247476011e+34, -9.873498369470537e+34, -9.861031382289022e+34, -9.848010334377203e+34, -9.834427447802377e+34, -9.820276168233227e+34, -9.80555121454046e+34, -9.79024862768531e+34, -9.774365818655185e+34, -9.757901615093086e+34, -9.74085630629511e+34, -9.723231686225987e+34, -9.705031094190461e+34, -9.686259452774992e+34, -9.66692330268737e+34, -9.647030834090341e+34, -9.626591914045631e+34, -9.605618109680887e+34, -9.584122706693486e+34, -9.562120722801496e+34, -9.539628915803367e+34, -9.516665785884624e+34, -9.49325157185053e+34, -9.469408240989053e+34, -9.44515947229562e+34, -9.42053063281562e+34, -9.395548746922488e+34, -9.370242458352666e+34, -9.34464198488642e+34, -9.318779065619335e+34, -9.29268690078247e+34, -9.266400084153207e+34, -9.239954528123712e+34, -9.213387381587975e+34, -9.186736940791296e+34, -9.160042553421757e+34, -9.133344516220737e+34, -9.10668396644572e+34, -9.080102767599047e+34, -9.053643389853139e+34, -9.027348785658234e+34, -9.00126226107176e+34, -8.975427343367493e+34, -8.949887645526216e+34, -8.924686728238983e+34, -8.899867960095069e+34, -8.875474376603615e+34, -8.851548538764292e+34, -8.828132391876904e+34, -8.805267125299902e+34, -8.782993033859403e+34, -8.761349381602833e+34, -8.74037426859385e+34, -8.720104501402363e+34, -8.700575467944997e+34, -8.68182101728087e+34, -8.663873344974558e+34, -8.646762884546651e+34, -8.630518205540882e+34, -8.615165918678175e+34, -8.600730588507594e+34, -8.587234653941669e+34, -8.57469835699727e+34, -8.563139680009508e+34, -8.552574291553828e+34, -8.543015501238147e+34, -8.534474223478356e+34, -8.526958950334126e+34, -8.520475733414959e+34, -8.515028174818366e+34, -8.510617427024968e+34, -8.50724220162185e+34, -8.50489878668359e+34, -8.503581072602833e+34, -8.503280586123919e+34, -8.503986532295282e+34, -8.505685844035833e+34, -8.508363238965876e+34, -8.512001283152102e+34, -8.51658046136244e+34, -8.522079253448685e+34, -8.528474216427166e+34, -8.535740071824629e+34, -8.543849797855521e+34, -8.552774725991287e+34, -8.56248464145936e+34, -8.572947887218868e+34, -8.584131470979628e+34, -8.596001174797198e+34, -8.608521666800709e+34, -8.62165661461763e+34, -8.635368800056859e+34, -8.64962023462255e+34, -8.664372275435682e+34, -8.679585741166476e+34, -8.695221027554623e+34, -8.711238222153815e+34, -8.727597217895115e+34, -8.744257825143278e+34, -8.761179881836293e+34, -8.778323361440898e+34, -8.79564847832889e+34, -8.813115790322576e+34, -8.830686298088311e+34, -8.84832154110819e+34, -8.865983689988435e+34, -8.88363563484904e+34, -8.901241069591438e+34, -8.918764571850085e+34, -8.936171678452641e+34, -8.953428956230013e+34, -8.970504068082583e+34, -8.987365834166477e+34, -9.003984288140887e+34, -9.020330728427278e+34, -9.03637776446236e+34, -9.052099357909657e+34, -9.067470858904286e+34, -9.082469037343473e+34, -9.09707210931125e+34, -9.11125975871354e+34, -9.125013154273773e+34, -9.138314961981288e+34, -9.151149353182468e+34, -9.163502008456333e+34, -9.175360117472425e+34, -9.18671237502884e+34, -9.197548973456055e+34, -9.207861591622623e+34, -9.217643380744123e+34, -9.226888947217083e+34, -9.235594332703847e+34, -9.243756991673692e+34, -9.25137576663265e+34, -9.258450861225926e+34, -9.264983811426857e+34, -9.270977454989376e+34, -9.276435899340559e+34, -9.281364488072205e+34, -9.285769766177026e+34, -9.289659444143448e+34, -9.293042361043426e+34, -9.295928446672041e+34, -9.298328682822606e+34, -9.300255063759037e+34, -9.30172055590119e+34, -9.302739056740402e+34, -9.303325352982556e+34, -9.303495077902576e+34, -9.303264667866932e+34, -9.30265131797114e+34, -9.301672936746038e+34, -9.300348099846568e+34, -9.298696002654826e+34, -9.296736411719571e+34, -9.294489614950397e+34, -9.291976370474456e+34, -9.289217854116514e+34, -9.286235605416571e+34, -9.283051472134344e+34, -9.279687553231912e+34, -9.276166140296757e+34, -9.272509657435482e+34, -9.26874059966359e+34, -9.264881469864715e+34, -9.260954714396102e+34, -9.25698265749653e+34, -9.252987434626947e+34, -9.248990924952606e+34, -9.24501468318225e+34, -9.241079871034721e+34, -9.237207188606138e+34, -9.233416805965734e+34, -9.229728295309624e+34, -9.226160564063019e+34, -9.222731789271626e+34, -9.219459353726884e+34, -9.216359784181436e+34, -9.213448692111072e+34, -9.210740717383776e+34, -9.208249475263405e+34, -9.205987507121938e+34, -9.20396623521497e+34, -9.202195921881028e+34, -9.200685633443957e+34, -9.199443209112953e+34, -9.198475235118409e+34, -9.19778702424369e+34, -9.197382600934297e+34, -9.197264692071265e+34, -9.197434723440402e+34, -9.197892821926601e+34, -9.198637823365494e+34, -9.199667285965091e+34, -9.20097750914702e+34, -9.202563557627208e+34, -9.204419290511877e+34, -9.206537395135286e+34, -9.208909425359378e+34, -9.211525843995108e+34, -9.21437606900111e+34, -9.217448523105108e+34, -9.220730686450118e+34, -9.224209151880472e+34, -9.227869682480321e+34, -9.231697270956196e+34, -9.235676200471602e+34, -9.239790106565928e+34, -9.24402203976328e+34, -9.248354528520154e+34, -9.252769642172236e+34, -9.257249053560958e+34, -9.261774101018307e+34, -9.266325849464498e+34, -9.270885150332263e+34, -9.275432700111319e+34, -9.27994909730458e+34, -9.28441489761661e+34, -9.288810667237084e+34, -9.293117034073347e+34, -9.29731473687183e+34, -9.301384672116444e+34, -9.305307938702777e+34, -9.309065880322348e+34, -9.312640125592582e+34, -9.316012625932897e+34, -9.31916569124363e+34, -9.322082023443065e+34, -9.324744747933332e+34, -9.3271374431083e+34, -9.329244167981664e+34, -9.33104948807344e+34, -9.332538499670267e+34, -9.333696852582259e+34, -9.334510771540852e+34, -9.33496707637159e+34, -9.335053201056792e+34, -9.334757211839454e+34, -9.334067824478386e+34, -9.332974420756578e+34, -9.331467064368498e+34, -9.329536516248188e+34, -9.327174249437847e+34, -9.324372463520859e+34, -9.321124098700535e+34, -9.317422849503618e+34, -9.3132631781444e+34, -9.308640327513732e+34, -9.303550333770696e+34, -9.297990038479892e+34, -9.291957100216905e+34, -9.28545000559346e+34, -9.278468079552416e+34, -9.2710114948807e+34, -9.26308128077949e+34, -9.25467933040677e+34, -9.245808407237375e+34, -9.236472150134774e+34, -9.226675077005882e+34, -9.216422586908319e+34, -9.205720960530894e+34, -9.194577358906103e+34, -9.182999820302235e+34, -9.170997255207818e+34, -9.15857943933985e+34, -9.145757004673243e+34, -9.132541428428257e+34, -9.11894502004834e+34, -9.104980906183456e+34, -9.090663013693371e+34, -9.076006050773666e+34, -9.061025486250437e+34, -9.045737527140573e+34, -9.030159094609928e+34, -9.014307798430815e+34, -8.998201910096173e+34, -8.981860334718829e+34, -8.965302581882505e+34, -8.948548735603806e+34, -8.931619423558203e+34, -8.914535785740277e+34, -8.897319442712843e+34, -8.879992463594017e+34, -8.862577333942162e+34, -8.8450969236535e+34, -8.827574455022264e+34, -8.810033471050157e+34, -8.792497804123709e+34, -8.77499154512528e+34, -8.75753901304562e+34, -8.74016472514584e+34, -8.722893367699893e+34, -8.70574976731945e+34, -8.688758862855616e+34, -8.671945677859354e+34, -8.655335293550991e+34, -8.638952822237213e+34, -8.62282338111426e+34, -8.606972066362257e+34, -8.591423927426197e+34, -8.576203941394634e+34, -8.56133698733312e+34, -8.546847820471106e+34, -8.532761046099429e+34, -8.519101093046968e+34, -8.505892186613367e+34, -8.493158320823421e+34, -8.480923229866704e+34, -8.469210358618657e+34, -8.458042832112248e+34, -8.447443423865687e+34, -8.437434522978648e+34, -8.428038099898324e+34, -8.419275670820307e+34, -8.4111682606507e+34, -8.403736364516317e+34, -8.396999907808636e+34, -8.390978204775711e+34, -8.385689915693215e+34, -8.381153002676932e+34, -8.377384684203304e+34, -8.374401388459772e+34, -8.372218705646455e+34, -8.370851339364633e+34, -8.370313057310925e+34, -8.370616641428051e+34, -8.37177383777803e+34, -8.373795306358893e+34, -8.376690571153692e+34, -8.380467970682471e+34, -8.385134609387853e+34, -8.390696310153997e+34, -8.39715756833858e+34, -8.404521507626075e+34, -8.412789838116741e+34, -8.421962816992316e+34, -8.432039212161296e+34, -8.443016269252925e+34, -8.454889682370228e+34, -8.467653568949145e+34, -8.481300449137124e+34, -8.495821230033879e+34, -8.511205195171101e+34, -8.527439999540597e+34, -8.544511670528468e+34, -8.562404615026791e+34, -8.581101633002271e+34, -8.60058393776872e+34, -8.620831183169552e+34, -8.641821497849792e+34, -8.663531526750372e+34, -8.685936479933563e+34, -8.709010188772316e+34, -8.732725169537894e+34, -8.75705269433058e+34, -8.781962869266108e+34, -8.807424719794109e+34, -8.833406282942232e+34, -8.859874706253595e+34, -8.886796353124353e+34, -8.914136914180514e+34, -8.941861524317393e+34, -8.969934884915647e+34, -8.99832139075774e+34, -9.026985261063429e+34, -9.055890674068767e+34, -9.085001904457887e+34, -9.114283462991444e+34, -9.143700237552104e+34, -9.173217634867786e+34, -9.202801722079923e+34, -9.232419367318935e+34, -9.262038378422267e+34, -9.291627638921249e+34, -9.321157240371484e+34, -9.350598610163752e+34, -9.379924633865071e+34, -9.409109771238139e+34, -9.438130165031503e+34, -9.46696374170761e+34, -9.49559030327926e+34, -9.52399160948434e+34, -9.552151449589238e+34, -9.580055703147531e+34, -9.607692389132927e+34, -9.635051702953725e+34, -9.662126040915794e+34, -9.688910011821542e+34, -9.715400435489329e+34, -9.741596328084919e+34, -9.7674988742688e+34, -9.793111386279133e+34, -9.818439250194282e+34, -9.843489859723006e+34, -9.868272538002696e+34, -9.892798447976919e+34, -9.91708049205774e+34, -9.941133201836987e+34, -9.964972618742136e+34, -9.988616166590805e+34, -1.0012082517057674e+35, -1.0035391449147847e+35, -1.0058563703791177e+35, -1.008162083471328e+35, -1.0104585056729393e+35, -1.0127479092667254e+35, -1.0150326020003042e+35, -1.0173149118394442e+35, -1.0195971719144952e+35, -1.0218817057662116e+35, -1.0241708129833681e+35, -1.0264667553256599e+35, -1.028771743411715e+35, -1.0310879240441564e+35, -1.0334173682388577e+35, -1.035762060011425e+35, -1.0381238859684969e+35, -1.0405046257414671e+35, -1.0429059432902995e+35, -1.0453293790986613e+35, -1.047776343272782e+35, -1.0502481095481761e+35, -1.052745810203727e+35, -1.0552704318741119e+35, -1.0578228122489603e+35, -1.060403637639089e+35, -1.0630134413894766e+35, -1.065652603112723e+35, -1.0683213487161721e+35, -1.0710197511920977e+35, -1.0737477321403567e+35, -1.0765050639897702e+35, -1.0792913728868387e+35, -1.0821061422176615e+35, -1.0849487167305162e+35, -1.0878183072235902e+35, -1.090713995769619e+35, -1.0936347414412024e+35, -1.0965793865069638e+35, -1.099546663067472e+35, -1.1025352000989633e+35, -1.1055435308765352e+35, -1.1085701007434992e+35, -1.111613275200735e+35, -1.1146713482816724e+35, -1.1177425511869964e+35, -1.120825061143737e+35, -1.1239170104633252e+35, -1.12701649576132e+35, -1.1301215873115056e+35, -1.133230338497826e+35, -1.136340795333811e+35, -1.1394510060137235e+35, -1.1425590304593213e+35, -1.145662949830714e+35, -1.1487608759621842e+35, -1.151850960686985e+35, -1.154931405016961e+35, -1.1580004681394094e+35, -1.1610564761934944e+35, -1.164097830794885e+35, -1.1671230172686118e+35, -1.1701306125596096e+35, -1.1731192927873721e+35, -1.1760878404128186e+35, -1.1790351509877433e+35, -1.1819602394592121e+35, -1.1848622460034033e+35, -1.187740441364645e+35, -1.1905942316797517e+35, -1.1934231627674319e+35, -1.1962269238688676e+35, -1.1990053508252913e+35, -1.2017584286822417e+35, -1.2044862937128851e+35, -1.2071892348570252e+35, -1.209867694571429e+35, -1.2125222690948397e+35, -1.2151537081284881e+35, -1.2177629139390592e+35, -1.2203509398924043e+35, -1.222918988426733e+35, -1.2254684084790972e+35, -1.2280006923772497e+35, -1.230517472214312e+35, -1.2330205157222276e+35, -1.2355117216616445e+35, -1.2379931147503487e+35, -1.2404668401469893e+35, -1.2429351575137836e+35, -1.2454004346806433e+35, -1.2478651409305863e+35, -1.2503318399317986e+35, -1.2528031823390558e+35, -1.2552818980878574e+35, -1.257770788405073e+35, -1.2602727175616568e+35, -1.2627906043907569e+35, -1.2653274135964687e+35, -1.2678861468788429e+35, -1.270469833898479e+35, -1.2730815231115522e+35, -1.2757242724937341e+35, -1.2784011401876741e+35, -1.2811151750958937e+35, -1.2838694074490794e+35, -1.2866668393767543e+35, -1.2895104355108244e+35, -1.2924031136495814e+35, -1.2953477355127626e+35, -1.2983470976187324e+35, -1.301403922311739e+35, -1.304520848973559e+35, -1.3077004254474967e+35, -1.3109450997073827e+35, -1.3142572117993646e+35, -1.3176389860903042e+35, -1.3210925238471814e+35, -1.3246197961789575e+35, -1.328222637363665e+35, -1.3319027385893698e+35, -1.3356616421273863e+35, -1.3395007359602606e+35, -1.3434212488818676e+35, -1.3474242460823622e+35, -1.3515106252320022e+35, -1.3556811130695112e+35, -1.35993626250151e+35, -1.3642764502134288e+35, -1.3687018747887895e+35, -1.3732125553333295e+35, -1.3778083305931283e+35, -1.382488858554796e+35, -1.3872536165139575e+35, -1.3921019015932043e+35, -1.3970328316912527e+35, -1.4020453468410543e+35, -1.407138210955174e+35, -1.4123100139348467e+35, -1.417559174120227e+35, -1.4228839410582693e+35, -1.4282823985665679e+35, -1.4337524680734582e+35, -1.4392919122144605e+35, -1.4448983386688768e+35, -1.4505692042256427e+35, -1.456301819063124e+35, -1.462093351239591e+35, -1.467940831388049e+35, -1.4738411576153583e+35, -1.4797911006074731e+35, -1.4857873089481378e+35, -1.4918263146570006e+35, -1.497904538960023e+35, -1.5040182983060521e+35, -1.5101638106416164e+35, -1.5163372019634463e+35, -1.522534513162384e+35, -1.5287517071764155e+35, -1.5349846764680733e+35, -1.5412292508403956e+35, -1.5474812056015127e+35, -1.553736270088832e+35, -1.5599901365569468e+35, -1.566238469428937e+35, -1.5724769149094133e+35, -1.5787011109488395e+35, -1.5849066975438306e+35, -1.5910893273535365e+35, -1.5972446766043002e+35, -1.6033684562490621e+35, -1.609456423340799e+35, -1.615504392575127e+35, -1.621508247946921e+35, -1.627463954463787e+35, -1.6333675698531595e+35, -1.6392152561913736e+35, -1.6450032913844668e+35, -1.650728080420161e+35, -1.6563861663165102e+35, -1.661974240679465e+35, -1.6674891537932708e+35, -1.672927924157386e+35, -1.678287747391753e+35, -1.683566004431563e+35, -1.688760268936917e+35, -1.6938683138490346e+35, -1.6988881170279252e+35, -1.7038178659165077e+35, -1.7086559611821312e+35, -1.7134010192956752e+35, -1.7180518740212878e+35, -1.7226075767937155e+35, -1.7270673959800058e+35, -1.731430815027905e+35, -1.7356975295151915e+35, -1.7398674431296393e+35, -1.7439406626195553e+35, -1.7479174917626816e+35, -1.7517984244180762e+35, -1.7555841367308655e+35, -1.759275478570214e+35, -1.762873464290645e+35, -1.7663792629093614e+35, -1.769794187802002e+35, -1.7731196860185373e+35, -1.7763573273277754e+35, -1.7795087930925652e+35, -1.782575865084714e+35, -1.785560414338192e+35, -1.788464390137897e+35, -1.791289809236101e+35, -1.794038745377431e+35, -1.796713319209467e+35, -1.799315688640768e+35, -1.8018480397028487e+35, -1.804312577955743e+35, -1.806711520472472e+35, -1.8090470884182568e+35, -1.8113215002348848e+35, -1.8135369654263395e+35, -1.815695678932175e+35, -1.8177998160660698e+35, -1.8198515279860797e+35, -1.821852937657877e+35, -1.8238061362647604e+35, -1.8257131800120472e+35, -1.8275760872714792e+35, -1.829396836007936e+35, -1.83117736142915e+35, -1.832919553801323e+35, -1.8346252563729538e+35, -1.8362962633537753e+35, -1.8379343178984747e+35, -1.8395411100502287e+35, -1.8411182746054987e+35, -1.842667388865552e+35, -1.844189970250238e+35, -1.845687473754834e+35, -1.84716128923764e+35, -1.8486127385355538e+35, -1.850043072410717e+35, -1.8514534673381613e+35, -1.852845022153175e+35, -1.8542187545812957e+35, -1.8555755976796562e+35, -1.8569163962250226e+35, -1.858241903086773e+35, -1.85955277562706e+35, -1.860849572172555e+35, -1.862132748605066e+35, -1.8634026551180698e+35]
tiempo_s_1emin4 = [0.0, 8.005538284800008e-07, 1.601107656959991e-06, 2.401661485439981e-06, 3.202215313919971e-06, 4.002769142399913e-06, 4.8033229708797e-06, 5.603876799359487e-06, 6.4044306278392736e-06, 7.20498445631906e-06, 8.005538284798847e-06, 8.806092113278634e-06, 9.60664594175842e-06, 1.0407199770238208e-05, 1.1207753598717994e-05, 1.2008307427197781e-05, 1.2808861255677568e-05, 1.3609415084157355e-05, 1.4409968912637142e-05, 1.5210522741116928e-05, 1.601107656959748e-05, 1.681163039807808e-05, 1.761218422655868e-05, 1.841273805503928e-05, 1.921329188351988e-05, 2.001384571200048e-05, 2.081439954048108e-05, 2.161495336896168e-05, 2.241550719744228e-05, 2.321606102592288e-05, 2.401661485440348e-05, 2.481716868288408e-05, 2.561772251136468e-05, 2.641827633984528e-05, 2.721883016832588e-05, 2.801938399680648e-05, 2.881993782528708e-05, 2.962049165376768e-05, 3.042104548224828e-05, 3.1221599310730306e-05, 3.202215313921253e-05, 3.282270696769476e-05, 3.3623260796176985e-05, 3.442381462465921e-05, 3.522436845314144e-05, 3.6024922281623664e-05, 3.682547611010589e-05, 3.7626029938588116e-05, 3.842658376707034e-05, 3.922713759555257e-05, 4.0027691424034795e-05, 4.082824525251702e-05, 4.162879908099925e-05, 4.2429352909481474e-05, 4.32299067379637e-05, 4.4030460566445926e-05, 4.483101439492815e-05, 4.563156822341038e-05, 4.6432122051892605e-05, 4.723267588037483e-05, 4.803322970885706e-05, 4.8833783537339284e-05, 4.963433736582151e-05, 5.0434891194303736e-05, 5.123544502278596e-05, 5.203599885126819e-05, 5.2836552679750415e-05, 5.363710650823264e-05, 5.443766033671487e-05, 5.5238214165197094e-05, 5.603876799367932e-05, 5.6839321822161546e-05, 5.763987565064377e-05, 5.8440429479126e-05, 5.9240983307608225e-05, 6.004153713609045e-05, 6.084209096457268e-05, 6.16426447930549e-05, 6.244319862153713e-05, 6.324375245001936e-05, 6.404430627850158e-05, 6.484486010698381e-05, 6.564541393546604e-05, 6.644596776394826e-05, 6.724652159243049e-05, 6.804707542091271e-05, 6.884762924939494e-05, 6.964818307787717e-05, 7.044873690635939e-05, 7.124929073484162e-05, 7.204984456332385e-05, 7.285039839180607e-05, 7.36509522202883e-05, 7.445150604877052e-05, 7.525205987725275e-05, 7.605261370573498e-05, 7.68531675342172e-05, 7.765372136269943e-05, 7.845427519118166e-05, 7.925482901966388e-05, 8.005538284814611e-05, 8.085593667662833e-05, 8.165649050511056e-05, 8.245704433359279e-05, 8.325759816207501e-05, 8.405815199055724e-05, 8.485870581903947e-05, 8.565925964752169e-05, 8.645981347600392e-05, 8.726036730448614e-05, 8.806092113296837e-05, 8.88614749614506e-05, 8.966202878993282e-05, 9.046258261841505e-05, 9.126313644689728e-05, 9.20636902753795e-05, 9.286424410386173e-05, 9.366479793234395e-05, 9.446535176082618e-05, 9.52659055893084e-05, 9.606645941779063e-05, 9.686701324627286e-05, 9.766756707475509e-05, 9.846812090323731e-05, 9.926867473171954e-05, 0.00010006922856020176, 0.00010086978238868399, 0.00010167033621716622, 0.00010247089004564844, 0.00010327144387413067, 0.0001040719977026129, 0.00010487255153109512, 0.00010567310535957735, 0.00010647365918805957, 0.0001072742130165418, 0.00010807476684502403, 0.00010887532067350625, 0.00010967587450198848, 0.0001104764283304707, 0.00011127698215895293, 0.00011207753598743516, 0.00011287808981591738, 0.00011367864364439961, 0.00011447919747288184, 0.00011527975130136406, 0.00011608030512984629, 0.00011688085895832852, 0.00011768141278681074, 0.00011848196661529297, 0.0001192825204437752, 0.00012008307427225742, 0.00012088362810073965, 0.00012168418192922187, 0.0001224847357577007, 0.00012328528958617643, 0.00012408584341465215, 0.00012488639724312787, 0.0001256869510716036, 0.00012648750490007932, 0.00012728805872855504, 0.00012808861255703076, 0.00012888916638550648, 0.0001296897202139822, 0.00013049027404245792, 0.00013129082787093364, 0.00013209138169940936, 0.00013289193552788508, 0.0001336924893563608, 0.00013449304318483653, 0.00013529359701331225, 0.00013609415084178797, 0.0001368947046702637, 0.0001376952584987394, 0.00013849581232721513, 0.00013929636615569085, 0.00014009691998416657, 0.0001408974738126423, 0.00014169802764111802, 0.00014249858146959374, 0.00014329913529806946, 0.00014409968912654518, 0.0001449002429550209, 0.00014570079678349662, 0.00014650135061197234, 0.00014730190444044806, 0.00014810245826892378, 0.0001489030120973995, 0.00014970356592587523, 0.00015050411975435095, 0.00015130467358282667, 0.0001521052274113024, 0.0001529057812397781, 0.00015370633506825383, 0.00015450688889672955, 0.00015530744272520527, 0.000156107996553681, 0.00015690855038215672, 0.00015770910421063244, 0.00015850965803910816, 0.00015931021186758388, 0.0001601107656960596, 0.00016091131952453532, 0.00016171187335301104, 0.00016251242718148676, 0.00016331298100996248, 0.0001641135348384382, 0.00016491408866691393, 0.00016571464249538965, 0.00016651519632386537, 0.0001673157501523411, 0.0001681163039808168, 0.00016891685780929253, 0.00016971741163776825, 0.00017051796546624397, 0.0001713185192947197, 0.00017211907312319542, 0.00017291962695167114, 0.00017372018078014686, 0.00017452073460862258, 0.0001753212884370983, 0.00017612184226557402, 0.00017692239609404974, 0.00017772294992252546, 0.00017852350375100118, 0.0001793240575794769, 0.00018012461140795263, 0.00018092516523642835, 0.00018172571906490407, 0.0001825262728933798, 0.0001833268267218555, 0.00018412738055033123, 0.00018492793437880695, 0.00018572848820728267, 0.0001865290420357584, 0.00018732959586423412, 0.00018813014969270984, 0.00018893070352118556, 0.00018973125734966128, 0.000190531811178137, 0.00019133236500661272, 0.00019213291883508844, 0.00019293347266356416, 0.00019373402649203988, 0.0001945345803205156, 0.00019533513414899133, 0.00019613568797746705, 0.00019693624180594277, 0.0001977367956344185, 0.0001985373494628942, 0.00019933790329136993, 0.00020013845711984565, 0.00020093901094832137, 0.0002017395647767971, 0.00020254011860527282, 0.00020334067243374854, 0.00020414122626222426, 0.00020494178009069998, 0.0002057423339191757, 0.00020654288774765142, 0.00020734344157612714, 0.00020814399540460286, 0.00020894454923307858, 0.0002097451030615543, 0.00021054565689003003, 0.00021134621071850575, 0.00021214676454698147, 0.0002129473183754572, 0.0002137478722039329, 0.00021454842603240863, 0.00021534897986088435, 0.00021614953368936007, 0.0002169500875178358, 0.00021775064134631152, 0.00021855119517478724, 0.00021935174900326296, 0.00022015230283173868, 0.0002209528566602144, 0.00022175341048869012, 0.00022255396431716584, 0.00022335451814564156, 0.00022415507197411728, 0.000224955625802593, 0.00022575617963106873, 0.00022655673345954445, 0.00022735728728802017, 0.0002281578411164959, 0.0002289583949449716, 0.00022975894877344733, 0.00023055950260192305, 0.00023136005643039877, 0.0002321606102588745, 0.00023296116408735021, 0.00023376171791582594, 0.00023456227174430166, 0.00023536282557277738, 0.0002361633794012531, 0.00023696393322972882, 0.00023776448705820454, 0.00023856504088668026, 0.00023936559471515598, 0.0002401661485436317, 0.00024096670237210743, 0.00024176725620058315, 0.00024256781002905887, 0.0002433683638575346, 0.0002441689176860099, 0.0002449694715144726, 0.0002457700253429353, 0.000246570579171398, 0.0002473711329998607, 0.00024817168682832343, 0.00024897224065678614, 0.00024977279448524885, 0.00025057334831371156, 0.00025137390214217427, 0.000252174455970637, 0.0002529750097990997, 0.0002537755636275624, 0.0002545761174560251, 0.0002553766712844878, 0.00025617722511295053, 0.00025697777894141325, 0.00025777833276987596, 0.00025857888659833867, 0.0002593794404268014, 0.0002601799942552641, 0.0002609805480837268, 0.0002617811019121895, 0.0002625816557406522, 0.00026338220956911493, 0.00026418276339757764, 0.00026498331722604035, 0.00026578387105450306, 0.0002665844248829658, 0.0002673849787114285, 0.0002681855325398912, 0.0002689860863683539, 0.0002697866401968166, 0.0002705871940252793, 0.00027138774785374204, 0.00027218830168220475, 0.00027298885551066746, 0.00027378940933913017, 0.0002745899631675929, 0.0002753905169960556, 0.0002761910708245183, 0.000276991624652981, 0.0002777921784814437, 0.00027859273230990643, 0.00027939328613836914, 0.00028019383996683185, 0.00028099439379529456, 0.0002817949476237573, 0.00028259550145222, 0.0002833960552806827, 0.0002841966091091454, 0.0002849971629376081, 0.00028579771676607083, 0.00028659827059453354, 0.00028739882442299625, 0.00028819937825145896, 0.00028899993207992167, 0.0002898004859083844, 0.0002906010397368471, 0.0002914015935653098, 0.0002922021473937725, 0.0002930027012222352, 0.00029380325505069793, 0.00029460380887916064, 0.00029540436270762336, 0.00029620491653608607, 0.0002970054703645488, 0.0002978060241930115, 0.0002986065780214742, 0.0002994071318499369, 0.0003002076856783996, 0.00030100823950686233, 0.00030180879333532504, 0.00030260934716378775, 0.00030340990099225046, 0.00030421045482071317, 0.0003050110086491759, 0.0003058115624776386, 0.0003066121163061013, 0.000307412670134564, 0.0003082132239630267, 0.00030901377779148944, 0.00030981433161995215, 0.00031061488544841486, 0.00031141543927687757, 0.0003122159931053403, 0.000313016546933803, 0.0003138171007622657, 0.0003146176545907284, 0.0003154182084191911, 0.00031621876224765383, 0.00031701931607611654, 0.00031781986990457925, 0.00031862042373304196, 0.0003194209775615047, 0.0003202215313899674, 0.0003210220852184301, 0.0003218226390468928, 0.0003226231928753555, 0.0003234237467038182, 0.00032422430053228094, 0.00032502485436074365, 0.00032582540818920636, 0.00032662596201766907, 0.0003274265158461318, 0.0003282270696745945, 0.0003290276235030572, 0.0003298281773315199, 0.0003306287311599826, 0.00033142928498844533, 0.00033222983881690804, 0.00033303039264537075, 0.00033383094647383346, 0.0003346315003022962, 0.0003354320541307589, 0.0003362326079592216, 0.0003370331617876843, 0.000337833715616147, 0.00033863426944460973, 0.00033943482327307244, 0.00034023537710153515, 0.00034103593092999786, 0.00034183648475846057, 0.0003426370385869233, 0.000343437592415386, 0.0003442381462438487, 0.0003450387000723114, 0.0003458392539007741, 0.00034663980772923683, 0.00034744036155769955, 0.00034824091538616226, 0.00034904146921462497, 0.0003498420230430877, 0.0003506425768715504, 0.0003514431307000131, 0.0003522436845284758, 0.0003530442383569385, 0.00035384479218540123, 0.00035464534601386394, 0.00035544589984232665, 0.00035624645367078936, 0.00035704700749925207, 0.0003578475613277148, 0.0003586481151561775, 0.0003594486689846402, 0.0003602492228131029, 0.0003610497766415656, 0.00036185033047002834, 0.00036265088429849105, 0.00036345143812695376, 0.00036425199195541647, 0.0003650525457838792, 0.0003658530996123419, 0.0003666536534408046, 0.0003674542072692673, 0.00036825476109773, 0.00036905531492619273, 0.00036985586875465544, 0.00037065642258311815, 0.00037145697641158086, 0.0003722575302400436, 0.0003730580840685063, 0.000373858637896969, 0.0003746591917254317, 0.0003754597455538944, 0.00037626029938235713, 0.00037706085321081984, 0.00037786140703928255, 0.00037866196086774526, 0.00037946251469620797, 0.0003802630685246707, 0.0003810636223531334, 0.0003818641761815961, 0.0003826647300100588, 0.0003834652838385215, 0.00038426583766698423, 0.00038506639149544694, 0.00038586694532390965, 0.00038666749915237237, 0.0003874680529808351, 0.0003882686068092978, 0.0003890691606377605, 0.0003898697144662232, 0.0003906702682946859, 0.00039147082212314863, 0.00039227137595161134, 0.00039307192978007405, 0.00039387248360853676, 0.00039467303743699947, 0.0003954735912654622, 0.0003962741450939249, 0.0003970746989223876, 0.0003978752527508503, 0.000398675806579313, 0.00039947636040777574, 0.00040027691423623845, 0.00040107746806470116, 0.00040187802189316387, 0.0004026785757216266, 0.0004034791295500893, 0.000404279683378552, 0.0004050802372070147, 0.0004058807910354774, 0.00040668134486394013, 0.00040748189869240284, 0.00040828245252086555, 0.00040908300634932826, 0.000409883560177791, 0.0004106841140062537, 0.0004114846678347164, 0.0004122852216631791, 0.0004130857754916418, 0.0004138863293201045, 0.00041468688314856724, 0.00041548743697702995, 0.00041628799080549266, 0.00041708854463395537, 0.0004178890984624181, 0.0004186896522908808, 0.0004194902061193435, 0.0004202907599478062, 0.0004210913137762689, 0.00042189186760473163, 0.00042269242143319434, 0.00042349297526165705, 0.00042429352909011976, 0.0004250940829185825, 0.0004258946367470452, 0.0004266951905755079, 0.0004274957444039706, 0.0004282962982324333, 0.00042909685206089603, 0.00042989740588935874, 0.00043069795971782145, 0.00043149851354628416, 0.00043229906737474687, 0.0004330996212032096, 0.0004339001750316723, 0.000434700728860135, 0.0004355012826885977, 0.0004363018365170604, 0.00043710239034552313, 0.00043790294417398584, 0.00043870349800244856, 0.00043950405183091127, 0.000440304605659374, 0.0004411051594878367, 0.0004419057133162994, 0.0004427062671447621, 0.0004435068209732248, 0.00044430737480168753, 0.00044510792863015024, 0.00044590848245861295, 0.00044670903628707566, 0.00044750959011553837, 0.0004483101439440011, 0.0004491106977724638, 0.0004499112516009265, 0.0004507118054293892, 0.0004515123592578519, 0.00045231291308631464, 0.00045311346691477735, 0.00045391402074324006, 0.00045471457457170277, 0.0004555151284001655, 0.0004563156822286282, 0.0004571162360570909, 0.0004579167898855536, 0.0004587173437140163, 0.00045951789754247903, 0.00046031845137094174, 0.00046111900519940445, 0.00046191955902786716, 0.0004627201128563299, 0.0004635206666847926, 0.0004643212205132553, 0.000465121774341718, 0.0004659223281701807, 0.0004667228819986434, 0.00046752343582710614, 0.00046832398965556885, 0.00046912454348403156, 0.00046992509731249427, 0.000470725651140957, 0.0004715262049694197, 0.0004723267587978824, 0.0004731273126263451, 0.0004739278664548078, 0.00047472842028327053, 0.00047552897411173324, 0.00047632952794019595, 0.00047713008176865866, 0.0004779306355971214, 0.0004787311894255841, 0.0004795317432540468, 0.0004803322970825095, 0.0004811328509109722, 0.00048193340473943493, 0.00048273395856789764, 0.00048353451239636035, 0.00048433506622482306, 0.00048513562005328577, 0.0004859361738817485, 0.0004867367277102112, 0.0004875372815386739, 0.0004883378353671385, 0.0004891383891956272, 0.0004899389430241159, 0.0004907394968526047, 0.0004915400506810934, 0.0004923406045095821, 0.0004931411583380708, 0.0004939417121665596, 0.0004947422659950483, 0.000495542819823537, 0.0004963433736520258, 0.0004971439274805145, 0.0004979444813090032, 0.000498745035137492, 0.0004995455889659807, 0.0005003461427944694, 0.0005011466966229582, 0.0005019472504514469, 0.0005027478042799356, 0.0005035483581084244, 0.0005043489119369131, 0.0005051494657654018, 0.0005059500195938905, 0.0005067505734223793, 0.000507551127250868, 0.0005083516810793567, 0.0005091522349078455, 0.0005099527887363342, 0.0005107533425648229, 0.0005115538963933117, 0.0005123544502218004, 0.0005131550040502891, 0.0005139555578787779, 0.0005147561117072666, 0.0005155566655357553, 0.0005163572193642441, 0.0005171577731927328, 0.0005179583270212215, 0.0005187588808497103, 0.000519559434678199, 0.0005203599885066877, 0.0005211605423351764, 0.0005219610961636652, 0.0005227616499921539, 0.0005235622038206426, 0.0005243627576491314, 0.0005251633114776201, 0.0005259638653061088, 0.0005267644191345976, 0.0005275649729630863, 0.000528365526791575, 0.0005291660806200638, 0.0005299666344485525, 0.0005307671882770412, 0.00053156774210553, 0.0005323682959340187, 0.0005331688497625074, 0.0005339694035909962, 0.0005347699574194849, 0.0005355705112479736, 0.0005363710650764623, 0.0005371716189049511, 0.0005379721727334398, 0.0005387727265619285, 0.0005395732803904173, 0.000540373834218906, 0.0005411743880473947, 0.0005419749418758835, 0.0005427754957043722, 0.0005435760495328609, 0.0005443766033613497, 0.0005451771571898384, 0.0005459777110183271, 0.0005467782648468159, 0.0005475788186753046, 0.0005483793725037933, 0.000549179926332282, 0.0005499804801607708, 0.0005507810339892595, 0.0005515815878177482, 0.000552382141646237, 0.0005531826954747257, 0.0005539832493032144, 0.0005547838031317032, 0.0005555843569601919, 0.0005563849107886806, 0.0005571854646171694, 0.0005579860184456581, 0.0005587865722741468, 0.0005595871261026356, 0.0005603876799311243, 0.000561188233759613, 0.0005619887875881018, 0.0005627893414165905, 0.0005635898952450792, 0.0005643904490735679, 0.0005651910029020567, 0.0005659915567305454, 0.0005667921105590341, 0.0005675926643875229, 0.0005683932182160116, 0.0005691937720445003, 0.0005699943258729891, 0.0005707948797014778, 0.0005715954335299665, 0.0005723959873584553, 0.000573196541186944, 0.0005739970950154327, 0.0005747976488439215, 0.0005755982026724102, 0.0005763987565008989, 0.0005771993103293876, 0.0005779998641578764, 0.0005788004179863651, 0.0005796009718148538, 0.0005804015256433426, 0.0005812020794718313, 0.00058200263330032, 0.0005828031871288088, 0.0005836037409572975, 0.0005844042947857862, 0.000585204848614275, 0.0005860054024427637, 0.0005868059562712524, 0.0005876065100997412, 0.0005884070639282299, 0.0005892076177567186, 0.0005900081715852074, 0.0005908087254136961, 0.0005916092792421848, 0.0005924098330706735, 0.0005932103868991623, 0.000594010940727651, 0.0005948114945561397, 0.0005956120483846285, 0.0005964126022131172, 0.0005972131560416059, 0.0005980137098700947, 0.0005988142636985834, 0.0005996148175270721, 0.0006004153713555609, 0.0006012159251840496, 0.0006020164790125383, 0.0006028170328410271, 0.0006036175866695158, 0.0006044181404980045, 0.0006052186943264933, 0.000606019248154982, 0.0006068198019834707, 0.0006076203558119594, 0.0006084209096404482, 0.0006092214634689369, 0.0006100220172974256, 0.0006108225711259144, 0.0006116231249544031, 0.0006124236787828918, 0.0006132242326113806, 0.0006140247864398693, 0.000614825340268358, 0.0006156258940968468, 0.0006164264479253355, 0.0006172270017538242, 0.000618027555582313, 0.0006188281094108017, 0.0006196286632392904, 0.0006204292170677791, 0.0006212297708962679, 0.0006220303247247566, 0.0006228308785532453, 0.0006236314323817341, 0.0006244319862102228, 0.0006252325400387115, 0.0006260330938672003, 0.000626833647695689, 0.0006276342015241777, 0.0006284347553526665, 0.0006292353091811552, 0.0006300358630096439, 0.0006308364168381327, 0.0006316369706666214, 0.0006324375244951101, 0.0006332380783235989, 0.0006340386321520876, 0.0006348391859805763, 0.000635639739809065, 0.0006364402936375538, 0.0006372408474660425, 0.0006380414012945312, 0.00063884195512302, 0.0006396425089515087, 0.0006404430627799974, 0.0006412436166084862, 0.0006420441704369749, 0.0006428447242654636, 0.0006436452780939524, 0.0006444458319224411, 0.0006452463857509298, 0.0006460469395794186, 0.0006468474934079073, 0.000647648047236396, 0.0006484486010648847, 0.0006492491548933735, 0.0006500497087218622, 0.0006508502625503509, 0.0006516508163788397, 0.0006524513702073284, 0.0006532519240358171, 0.0006540524778643059, 0.0006548530316927946, 0.0006556535855212833, 0.0006564541393497721, 0.0006572546931782608, 0.0006580552470067495, 0.0006588558008352383, 0.000659656354663727, 0.0006604569084922157, 0.0006612574623207045, 0.0006620580161491932, 0.0006628585699776819, 0.0006636591238061706, 0.0006644596776346594, 0.0006652602314631481, 0.0006660607852916368, 0.0006668613391201256, 0.0006676618929486143, 0.000668462446777103, 0.0006692630006055918, 0.0006700635544340805, 0.0006708641082625692, 0.000671664662091058, 0.0006724652159195467, 0.0006732657697480354, 0.0006740663235765242, 0.0006748668774050129, 0.0006756674312335016, 0.0006764679850619904, 0.0006772685388904791, 0.0006780690927189678, 0.0006788696465474565, 0.0006796702003759453, 0.000680470754204434, 0.0006812713080329227, 0.0006820718618614115, 0.0006828724156899002, 0.0006836729695183889, 0.0006844735233468777, 0.0006852740771753664, 0.0006860746310038551, 0.0006868751848323439, 0.0006876757386608326, 0.0006884762924893213, 0.0006892768463178101, 0.0006900774001462988, 0.0006908779539747875, 0.0006916785078032762, 0.000692479061631765, 0.0006932796154602537, 0.0006940801692887424, 0.0006948807231172312, 0.0006956812769457199, 0.0006964818307742086, 0.0006972823846026974, 0.0006980829384311861, 0.0006988834922596748, 0.0006996840460881636, 0.0007004845999166523, 0.000701285153745141, 0.0007020857075736298, 0.0007028862614021185, 0.0007036868152306072, 0.000704487369059096, 0.0007052879228875847, 0.0007060884767160734, 0.0007068890305445621, 0.0007076895843730509, 0.0007084901382015396, 0.0007092906920300283, 0.0007100912458585171, 0.0007108917996870058, 0.0007116923535154945, 0.0007124929073439833, 0.000713293461172472, 0.0007140940150009607, 0.0007148945688294495, 0.0007156951226579382, 0.0007164956764864269, 0.0007172962303149157, 0.0007180967841434044, 0.0007188973379718931, 0.0007196978918003819, 0.0007204984456288706, 0.0007212989994573593, 0.000722099553285848, 0.0007229001071143368, 0.0007237006609428255, 0.0007245012147713142, 0.000725301768599803, 0.0007261023224282917, 0.0007269028762567804, 0.0007277034300852692, 0.0007285039839137579, 0.0007293045377422466, 0.0007301050915707354, 0.0007309056453992241, 0.0007317061992277128, 0.0007325067530562016, 0.0007333073068846903, 0.000734107860713179, 0.0007349084145416677, 0.0007357089683701565, 0.0007365095221986452, 0.0007373100760271339, 0.0007381106298556227, 0.0007389111836841114, 0.0007397117375126001, 0.0007405122913410889, 0.0007413128451695776, 0.0007421133989980663, 0.0007429139528265551, 0.0007437145066550438, 0.0007445150604835325, 0.0007453156143120213, 0.00074611616814051, 0.0007469167219689987, 0.0007477172757974875, 0.0007485178296259762, 0.0007493183834544649, 0.0007501189372829536, 0.0007509194911114424, 0.0007517200449399311, 0.0007525205987684198, 0.0007533211525969086, 0.0007541217064253973, 0.000754922260253886, 0.0007557228140823748, 0.0007565233679108635, 0.0007573239217393522, 0.000758124475567841, 0.0007589250293963297, 0.0007597255832248184, 0.0007605261370533072, 0.0007613266908817959, 0.0007621272447102846, 0.0007629277985387733, 0.0007637283523672621, 0.0007645289061957508, 0.0007653294600242395, 0.0007661300138527283, 0.000766930567681217, 0.0007677311215097057, 0.0007685316753381945, 0.0007693322291666832, 0.0007701327829951719, 0.0007709333368236607, 0.0007717338906521494, 0.0007725344444806381, 0.0007733349983091269, 0.0007741355521376156, 0.0007749361059661043, 0.000775736659794593, 0.0007765372136230818, 0.0007773377674515705, 0.0007781383212800592, 0.000778938875108548, 0.0007797394289370367, 0.0007805399827655254, 0.0007813405365940142, 0.0007821410904225029, 0.0007829416442509916, 0.0007837421980794804, 0.0007845427519079691, 0.0007853433057364578, 0.0007861438595649466, 0.0007869444133934353, 0.000787744967221924, 0.0007885455210504128, 0.0007893460748789015, 0.0007901466287073902, 0.000790947182535879, 0.0007917477363643677, 0.0007925482901928564, 0.0007933488440213451, 0.0007941493978498339, 0.0007949499516783226, 0.0007957505055068113, 0.0007965510593353001, 0.0007973516131637888, 0.0007981521669922775, 0.0007989527208207663, 0.000799753274649255, 0.0008005538284777437, 0.0008013543823062325, 0.0008021549361347212, 0.0008029554899632099, 0.0008037560437916987, 0.0008045565976201874, 0.0008053571514486761, 0.0008061577052771648, 0.0008069582591056536, 0.0008077588129341423, 0.000808559366762631, 0.0008093599205911198, 0.0008101604744196085, 0.0008109610282480972, 0.000811761582076586, 0.0008125621359050747, 0.0008133626897335634, 0.0008141632435620522, 0.0008149637973905409, 0.0008157643512190296, 0.0008165649050475184, 0.0008173654588760071, 0.0008181660127044958, 0.0008189665665329846, 0.0008197671203614733, 0.000820567674189962, 0.0008213682280184507, 0.0008221687818469395, 0.0008229693356754282, 0.0008237698895039169, 0.0008245704433324057, 0.0008253709971608944, 0.0008261715509893831, 0.0008269721048178719, 0.0008277726586463606, 0.0008285732124748493, 0.0008293737663033381, 0.0008301743201318268, 0.0008309748739603155, 0.0008317754277888043, 0.000832575981617293, 0.0008333765354457817, 0.0008341770892742705, 0.0008349776431027592, 0.0008357781969312479, 0.0008365787507597366, 0.0008373793045882254, 0.0008381798584167141, 0.0008389804122452028, 0.0008397809660736916, 0.0008405815199021803, 0.000841382073730669, 0.0008421826275591578, 0.0008429831813876465, 0.0008437837352161352, 0.000844584289044624, 0.0008453848428731127, 0.0008461853967016014, 0.0008469859505300902, 0.0008477865043585789, 0.0008485870581870676, 0.0008493876120155563, 0.0008501881658440451, 0.0008509887196725338, 0.0008517892735010225, 0.0008525898273295113, 0.000853390381158, 0.0008541909349864887, 0.0008549914888149775, 0.0008557920426434662, 0.0008565925964719549, 0.0008573931503004437, 0.0008581937041289324, 0.0008589942579574211, 0.0008597948117859099, 0.0008605953656143986, 0.0008613959194428873, 0.000862196473271376, 0.0008629970270998648, 0.0008637975809283535, 0.0008645981347568422, 0.000865398688585331, 0.0008661992424138197, 0.0008669997962423084, 0.0008678003500707972, 0.0008686009038992859, 0.0008694014577277746, 0.0008702020115562634, 0.0008710025653847521, 0.0008718031192132408, 0.0008726036730417296, 0.0008734042268702183, 0.000874204780698707, 0.0008750053345271958, 0.0008758058883556845, 0.0008766064421841732, 0.000877406996012662, 0.0008782075498411507, 0.0008790081036696394, 0.0008798086574981281, 0.0008806092113266169, 0.0008814097651551056, 0.0008822103189835943, 0.0008830108728120831, 0.0008838114266405718, 0.0008846119804690605, 0.0008854125342975493, 0.000886213088126038, 0.0008870136419545267, 0.0008878141957830155, 0.0008886147496115042, 0.0008894153034399929, 0.0008902158572684817, 0.0008910164110969704, 0.0008918169649254591, 0.0008926175187539478, 0.0008934180725824366, 0.0008942186264109253, 0.000895019180239414, 0.0008958197340679028, 0.0008966202878963915, 0.0008974208417248802, 0.000898221395553369, 0.0008990219493818577, 0.0008998225032103464, 0.0009006230570388352, 0.0009014236108673239, 0.0009022241646958126, 0.0009030247185243014, 0.0009038252723527901, 0.0009046258261812788, 0.0009054263800097676, 0.0009062269338382563, 0.000907027487666745, 0.0009078280414952337, 0.0009086285953237225, 0.0009094291491522112, 0.0009102297029806999, 0.0009110302568091887, 0.0009118308106376774, 0.0009126313644661661, 0.0009134319182946549, 0.0009142324721231436, 0.0009150330259516323, 0.0009158335797801211, 0.0009166341336086098, 0.0009174346874370985, 0.0009182352412655873, 0.000919035795094076, 0.0009198363489225647, 0.0009206369027510534, 0.0009214374565795422, 0.0009222380104080309, 0.0009230385642365196, 0.0009238391180650084, 0.0009246396718934971, 0.0009254402257219858, 0.0009262407795504746, 0.0009270413333789633, 0.000927841887207452, 0.0009286424410359408, 0.0009294429948644295, 0.0009302435486929182, 0.000931044102521407, 0.0009318446563498957, 0.0009326452101783844, 0.0009334457640068732, 0.0009342463178353619, 0.0009350468716638506, 0.0009358474254923393, 0.0009366479793208281, 0.0009374485331493168, 0.0009382490869778055, 0.0009390496408062943, 0.000939850194634783, 0.0009406507484632717, 0.0009414513022917605, 0.0009422518561202492, 0.0009430524099487379, 0.0009438529637772267, 0.0009446535176057154, 0.0009454540714342041, 0.0009462546252626929, 0.0009470551790911816, 0.0009478557329196703, 0.000948656286748159, 0.0009494568405766478, 0.0009502573944051365, 0.0009510579482336252, 0.000951858502062114, 0.0009526590558906027, 0.0009534596097190914, 0.0009542601635475802, 0.0009550607173760689, 0.0009558612712045576, 0.0009566618250330464, 0.0009574623788615351, 0.0009582629326900238, 0.0009590634865185126, 0.0009598640403470013, 0.00096066459417549, 0.0009614651480039788, 0.0009622657018324675, 0.0009630662556609562, 0.000963866809489445, 0.0009646673633179337, 0.0009654679171464224, 0.0009662684709749111, 0.0009670690248033999, 0.0009678695786318886, 0.0009686701324603773, 0.0009694706862888661, 0.0009702712401173548, 0.0009710717939458435, 0.0009718723477743323, 0.000972672901602821, 0.0009734734554313097, 0.0009742740092597985, 0.0009750745630882872, 0.0009758751169167759, 0.000976675670745272, 0.0009774762245738128, 0.0009782767784023536, 0.0009790773322308943, 0.0009798778860594351, 0.0009806784398879759, 0.0009814789937165167, 0.0009822795475450574, 0.0009830801013735982, 0.000983880655202139, 0.0009846812090306798, 0.0009854817628592205, 0.0009862823166877613, 0.000987082870516302, 0.0009878834243448428, 0.0009886839781733836, 0.0009894845320019244, 0.0009902850858304652, 0.000991085639659006, 0.0009918861934875467, 0.0009926867473160875, 0.0009934873011446283, 0.000994287854973169, 0.0009950884088017098, 0.0009958889626302506, 0.0009966895164587914, 0.0009974900702873321, 0.000998290624115873, 0.0009990911779444137, 0.0009998917317729544, 0.0010006922856014952, 0.001001492839430036, 0.0010022933932585768, 0.0010030939470871175, 0.0010038945009156583, 0.001004695054744199, 0.0010054956085727399, 0.0010062961624012806, 0.0010070967162298214, 0.0010078972700583622, 0.001008697823886903, 0.0010094983777154437, 0.0010102989315439845, 0.0010110994853725253, 0.001011900039201066, 0.0010127005930296068, 0.0010135011468581476, 0.0010143017006866884, 0.0010151022545152291, 0.00101590280834377, 0.0010167033621723107, 0.0010175039160008515, 0.0010183044698293922, 0.001019105023657933, 0.0010199055774864738, 0.0010207061313150145, 0.0010215066851435553, 0.001022307238972096, 0.0010231077928006369, 0.0010239083466291776, 0.0010247089004577184, 0.0010255094542862592, 0.0010263100081148, 0.0010271105619433407, 0.0010279111157718815, 0.0010287116696004223, 0.001029512223428963, 0.0010303127772575038, 0.0010311133310860446, 0.0010319138849145854, 0.0010327144387431261, 0.001033514992571667, 0.0010343155464002077, 0.0010351161002287485, 0.0010359166540572892, 0.00103671720788583, 0.0010375177617143708, 0.0010383183155429116, 0.0010391188693714523, 0.001039919423199993, 0.0010407199770285339, 0.0010415205308570747, 0.0010423210846856154, 0.0010431216385141562, 0.001043922192342697, 0.0010447227461712377, 0.0010455232999997785, 0.0010463238538283193, 0.00104712440765686, 0.0010479249614854008, 0.0010487255153139416, 0.0010495260691424824, 0.0010503266229710232, 0.001051127176799564, 0.0010519277306281047, 0.0010527282844566455, 0.0010535288382851862, 0.001054329392113727, 0.0010551299459422678, 0.0010559304997708086, 0.0010567310535993493, 0.0010575316074278901, 0.0010583321612564309, 0.0010591327150849717, 0.0010599332689135124, 0.0010607338227420532, 0.001061534376570594, 0.0010623349303991348, 0.0010631354842276755, 0.0010639360380562163, 0.001064736591884757, 0.0010655371457132978, 0.0010663376995418386, 0.0010671382533703794, 0.0010679388071989202, 0.001068739361027461, 0.0010695399148560017, 0.0010703404686845425, 0.0010711410225130833, 0.001071941576341624, 0.0010727421301701648, 0.0010735426839987056, 0.0010743432378272464, 0.0010751437916557871, 0.001075944345484328, 0.0010767448993128687, 0.0010775454531414094, 0.0010783460069699502, 0.001079146560798491, 0.0010799471146270318, 0.0010807476684555725, 0.0010815482222841133, 0.001082348776112654, 0.0010831493299411949, 0.0010839498837697356, 0.0010847504375982764, 0.0010855509914268172, 0.001086351545255358, 0.0010871520990838987, 0.0010879526529124395, 0.0010887532067409803, 0.001089553760569521, 0.0010903543143980618, 0.0010911548682266026, 0.0010919554220551434, 0.0010927559758836841, 0.001093556529712225, 0.0010943570835407657, 0.0010951576373693065, 0.0010959581911978472, 0.001096758745026388, 0.0010975592988549288, 0.0010983598526834695, 0.0010991604065120103, 0.001099960960340551, 0.0011007615141690919, 0.0011015620679976326, 0.0011023626218261734, 0.0011031631756547142, 0.001103963729483255, 0.0011047642833117957, 0.0011055648371403365, 0.0011063653909688773, 0.001107165944797418, 0.0011079664986259588, 0.0011087670524544996, 0.0011095676062830404, 0.0011103681601115811, 0.001111168713940122, 0.0011119692677686627, 0.0011127698215972035, 0.0011135703754257442, 0.001114370929254285, 0.0011151714830828258, 0.0011159720369113666, 0.0011167725907399073, 0.001117573144568448, 0.0011183736983969889, 0.0011191742522255296, 0.0011199748060540704, 0.0011207753598826112, 0.001121575913711152, 0.0011223764675396927, 0.0011231770213682335, 0.0011239775751967743, 0.001124778129025315, 0.0011255786828538558, 0.0011263792366823966, 0.0011271797905109374, 0.0011279803443394782, 0.001128780898168019, 0.0011295814519965597, 0.0011303820058251005, 0.0011311825596536412, 0.001131983113482182, 0.0011327836673107228, 0.0011335842211392636, 0.0011343847749678043, 0.0011351853287963451, 0.0011359858826248859, 0.0011367864364534267, 0.0011375869902819674, 0.0011383875441105082, 0.001139188097939049, 0.0011399886517675898, 0.0011407892055961305, 0.0011415897594246713, 0.001142390313253212, 0.0011431908670817528, 0.0011439914209102936, 0.0011447919747388344, 0.0011455925285673752, 0.001146393082395916, 0.0011471936362244567, 0.0011479941900529975, 0.0011487947438815383, 0.001149595297710079, 0.0011503958515386198, 0.0011511964053671606, 0.0011519969591957013, 0.0011527975130242421, 0.001153598066852783, 0.0011543986206813237, 0.0011551991745098644, 0.0011559997283384052, 0.001156800282166946, 0.0011576008359954868, 0.0011584013898240275, 0.0011592019436525683, 0.001160002497481109, 0.0011608030513096499, 0.0011616036051381906, 0.0011624041589667314, 0.0011632047127952722, 0.001164005266623813, 0.0011648058204523537, 0.0011656063742808945, 0.0011664069281094353, 0.001167207481937976, 0.0011680080357665168, 0.0011688085895950576, 0.0011696091434235984, 0.0011704096972521391, 0.00117121025108068, 0.0011720108049092207, 0.0011728113587377615, 0.0011736119125663022, 0.001174412466394843, 0.0011752130202233838, 0.0011760135740519245, 0.0011768141278804653, 0.001177614681709006, 0.0011784152355375469, 0.0011792157893660876, 0.0011800163431946284, 0.0011808168970231692, 0.00118161745085171, 0.0011824180046802507, 0.0011832185585087915, 0.0011840191123373323, 0.001184819666165873, 0.0011856202199944138, 0.0011864207738229546, 0.0011872213276514954, 0.0011880218814800361, 0.001188822435308577, 0.0011896229891371177, 0.0011904235429656585, 0.0011912240967941992, 0.00119202465062274, 0.0011928252044512808, 0.0011936257582798216, 0.0011944263121083623, 0.001195226865936903, 0.0011960274197654439, 0.0011968279735939846, 0.0011976285274225254, 0.0011984290812510662, 0.001199229635079607, 0.0012000301889081477, 0.0012008307427366885, 0.0012016312965652293, 0.00120243185039377, 0.0012032324042223108, 0.0012040329580508516, 0.0012048335118793924, 0.0012056340657079332, 0.001206434619536474, 0.0012072351733650147, 0.0012080357271935555, 0.0012088362810220962, 0.001209636834850637, 0.0012104373886791778, 0.0012112379425077186, 0.0012120384963362593, 0.0012128390501648001, 0.0012136396039933409, 0.0012144401578218817, 0.0012152407116504224, 0.0012160412654789632, 0.001216841819307504, 0.0012176423731360447, 0.0012184429269645855, 0.0012192434807931263, 0.001220044034621667, 0.0012208445884502078, 0.0012216451422787486, 0.0012224456961072894, 0.0012232462499358302, 0.001224046803764371, 0.0012248473575929117, 0.0012256479114214525, 0.0012264484652499933, 0.001227249019078534, 0.0012280495729070748, 0.0012288501267356156, 0.0012296506805641563, 0.0012304512343926971, 0.001231251788221238, 0.0012320523420497787, 0.0012328528958783194, 0.0012336534497068602, 0.001234454003535401, 0.0012352545573639418, 0.0012360551111924825, 0.0012368556650210233, 0.001237656218849564, 0.0012384567726781049, 0.0012392573265066456, 0.0012400578803351864, 0.0012408584341637272, 0.001241658987992268, 0.0012424595418208087, 0.0012432600956493495, 0.0012440606494778903, 0.001244861203306431, 0.0012456617571349718, 0.0012464623109635126, 0.0012472628647920534, 0.0012480634186205941, 0.001248863972449135, 0.0012496645262776757, 0.0012504650801062164, 0.0012512656339347572, 0.001252066187763298, 0.0012528667415918388, 0.0012536672954203795, 0.0012544678492489203, 0.001255268403077461, 0.0012560689569060019, 0.0012568695107345426, 0.0012576700645630834, 0.0012584706183916242, 0.001259271172220165, 0.0012600717260487057, 0.0012608722798772465, 0.0012616728337057873, 0.001262473387534328, 0.0012632739413628688, 0.0012640744951914096, 0.0012648750490199504, 0.0012656756028484911, 0.001266476156677032, 0.0012672767105055727, 0.0012680772643341135, 0.0012688778181626542, 0.001269678371991195, 0.0012704789258197358, 0.0012712794796482766, 0.0012720800334768173, 0.001272880587305358, 0.0012736811411338989, 0.0012744816949624396, 0.0012752822487909804, 0.0012760828026195212, 0.001276883356448062, 0.0012776839102766027, 0.0012784844641051435, 0.0012792850179336843, 0.001280085571762225, 0.0012808861255907658, 0.0012816866794193066, 0.0012824872332478474, 0.0012832877870763881, 0.001284088340904929, 0.0012848888947334697, 0.0012856894485620105, 0.0012864900023905512, 0.001287290556219092, 0.0012880911100476328, 0.0012888916638761736, 0.0012896922177047143, 0.001290492771533255, 0.0012912933253617959, 0.0012920938791903367, 0.0012928944330188774, 0.0012936949868474182, 0.001294495540675959, 0.0012952960945044997, 0.0012960966483330405, 0.0012968972021615813, 0.001297697755990122, 0.0012984983098186628, 0.0012992988636472036, 0.0013000994174757444, 0.0013008999713042852, 0.001301700525132826, 0.0013025010789613667, 0.0013033016327899075, 0.0013041021866184483, 0.001304902740446989, 0.0013057032942755298, 0.0013065038481040706, 0.0013073044019326113, 0.0013081049557611521, 0.0013089055095896929, 0.0013097060634182337, 0.0013105066172467744, 0.0013113071710753152, 0.001312107724903856, 0.0013129082787323968, 0.0013137088325609375, 0.0013145093863894783, 0.001315309940218019, 0.0013161104940465598, 0.0013169110478751006, 0.0013177116017036414, 0.0013185121555321822, 0.001319312709360723, 0.0013201132631892637, 0.0013209138170178045, 0.0013217143708463453, 0.001322514924674886, 0.0013233154785034268, 0.0013241160323319676, 0.0013249165861605084, 0.0013257171399890491, 0.00132651769381759, 0.0013273182476461307, 0.0013281188014746714, 0.0013289193553032122, 0.001329719909131753, 0.0013305204629602938, 0.0013313210167888345, 0.0013321215706173753, 0.001332922124445916, 0.0013337226782744569, 0.0013345232321029976, 0.0013353237859315384, 0.0013361243397600792, 0.00133692489358862, 0.0013377254474171607, 0.0013385260012457015, 0.0013393265550742423, 0.001340127108902783, 0.0013409276627313238, 0.0013417282165598646, 0.0013425287703884054, 0.0013433293242169461, 0.001344129878045487, 0.0013449304318740277, 0.0013457309857025685, 0.0013465315395311092, 0.00134733209335965, 0.0013481326471881908, 0.0013489332010167315, 0.0013497337548452723, 0.001350534308673813, 0.0013513348625023539, 0.0013521354163308946, 0.0013529359701594354, 0.0013537365239879762, 0.001354537077816517, 0.0013553376316450577, 0.0013561381854735985, 0.0013569387393021393, 0.00135773929313068, 0.0013585398469592208, 0.0013593404007877616, 0.0013601409546163024, 0.0013609415084448431, 0.001361742062273384, 0.0013625426161019247, 0.0013633431699304655, 0.0013641437237590062, 0.001364944277587547, 0.0013657448314160878, 0.0013665453852446286, 0.0013673459390731693, 0.00136814649290171, 0.0013689470467302509, 0.0013697476005587917, 0.0013705481543873324, 0.0013713487082158732, 0.001372149262044414, 0.0013729498158729547, 0.0013737503697014955, 0.0013745509235300363, 0.001375351477358577, 0.0013761520311871178, 0.0013769525850156586, 0.0013777531388441994, 0.0013785536926727402, 0.001379354246501281, 0.0013801548003298217, 0.0013809553541583625, 0.0013817559079869032, 0.001382556461815444, 0.0013833570156439848, 0.0013841575694725256, 0.0013849581233010663, 0.0013857586771296071, 0.0013865592309581479, 0.0013873597847866887, 0.0013881603386152294, 0.0013889608924437702, 0.001389761446272311, 0.0013905620001008518, 0.0013913625539293925, 0.0013921631077579333, 0.001392963661586474, 0.0013937642154150148, 0.0013945647692435556, 0.0013953653230720964, 0.0013961658769006372, 0.001396966430729178, 0.0013977669845577187, 0.0013985675383862595, 0.0013993680922148003, 0.001400168646043341, 0.0014009691998718818, 0.0014017697537004226, 0.0014025703075289634, 0.0014033708613575041, 0.001404171415186045, 0.0014049719690145857, 0.0014057725228431264, 0.0014065730766716672, 0.001407373630500208, 0.0014081741843287488, 0.0014089747381572895, 0.0014097752919858303, 0.001410575845814371, 0.0014113763996429119, 0.0014121769534714526, 0.0014129775072999934, 0.0014137780611285342, 0.001414578614957075, 0.0014153791687856157, 0.0014161797226141565, 0.0014169802764426973, 0.001417780830271238, 0.0014185813840997788, 0.0014193819379283196, 0.0014201824917568604, 0.0014209830455854011, 0.001421783599413942, 0.0014225841532424827, 0.0014233847070710235, 0.0014241852608995642, 0.001424985814728105, 0.0014257863685566458, 0.0014265869223851865, 0.0014273874762137273, 0.001428188030042268, 0.0014289885838708089, 0.0014297891376993496, 0.0014305896915278904, 0.0014313902453564312, 0.001432190799184972, 0.0014329913530135127, 0.0014337919068420535, 0.0014345924606705943, 0.001435393014499135, 0.0014361935683276758, 0.0014369941221562166, 0.0014377946759847574, 0.0014385952298132981, 0.001439395783641839, 0.0014401963374703797, 0.0014409968912989205, 0.0014417974451274612, 0.001442597998956002, 0.0014433985527845428, 0.0014441991066130836, 0.0014449996604416243, 0.001445800214270165, 0.0014466007680987059, 0.0014474013219272466, 0.0014482018757557874, 0.0014490024295843282, 0.001449802983412869, 0.0014506035372414097, 0.0014514040910699505, 0.0014522046448984913, 0.001453005198727032, 0.0014538057525555728, 0.0014546063063841136, 0.0014554068602126544, 0.0014562074140411952, 0.001457007967869736, 0.0014578085216982767, 0.0014586090755268175, 0.0014594096293553582, 0.001460210183183899, 0.0014610107370124398, 0.0014618112908409806, 0.0014626118446695213, 0.0014634123984980621, 0.0014642129523266029, 0.0014650135061551437, 0.0014658140599836844, 0.0014666146138122252, 0.001467415167640766, 0.0014682157214693068, 0.0014690162752978475, 0.0014698168291263883, 0.001470617382954929, 0.0014714179367834698, 0.0014722184906120106, 0.0014730190444405514, 0.0014738195982690922, 0.001474620152097633, 0.0014754207059261737, 0.0014762212597547145, 0.0014770218135832553, 0.001477822367411796, 0.0014786229212403368, 0.0014794234750688776, 0.0014802240288974183, 0.0014810245827259591, 0.0014818251365545, 0.0014826256903830407, 0.0014834262442115814, 0.0014842267980401222, 0.001485027351868663, 0.0014858279056972038, 0.0014866284595257445, 0.0014874290133542853, 0.001488229567182826, 0.0014890301210113669, 0.0014898306748399076, 0.0014906312286684484, 0.0014914317824969892, 0.00149223233632553, 0.0014930328901540707, 0.0014938334439826115, 0.0014946339978111523, 0.001495434551639693, 0.0014962351054682338, 0.0014970356592967746, 0.0014978362131253154, 0.0014986367669538561, 0.001499437320782397, 0.0015002378746109377, 0.0015010384284394785, 0.0015018389822680192, 0.00150263953609656, 0.0015034400899251008, 0.0015042406437536415, 0.0015050411975821823, 0.001505841751410723, 0.0015066423052392639, 0.0015074428590678046, 0.0015082434128963454, 0.0015090439667248862, 0.001509844520553427, 0.0015106450743819677, 0.0015114456282105085, 0.0015122461820390493, 0.00151304673586759, 0.0015138472896961308, 0.0015146478435246716, 0.0015154483973532124, 0.0015162489511817531, 0.001517049505010294, 0.0015178500588388347, 0.0015186506126673755, 0.0015194511664959162, 0.001520251720324457, 0.0015210522741529978, 0.0015218528279815386, 0.0015226533818100793, 0.00152345393563862, 0.0015242544894671609, 0.0015250550432957016, 0.0015258555971242424, 0.0015266561509527832, 0.001527456704781324, 0.0015282572586098647, 0.0015290578124384055, 0.0015298583662669463, 0.001530658920095487, 0.0015314594739240278, 0.0015322600277525686, 0.0015330605815811094, 0.0015338611354096502, 0.001534661689238191, 0.0015354622430667317, 0.0015362627968952725, 0.0015370633507238132, 0.001537863904552354, 0.0015386644583808948, 0.0015394650122094356, 0.0015402655660379763, 0.0015410661198665171, 0.0015418666736950579, 0.0015426672275235987, 0.0015434677813521394, 0.0015442683351806802, 0.001545068889009221, 0.0015458694428377617, 0.0015466699966663025, 0.0015474705504948433, 0.001548271104323384, 0.0015490716581519248, 0.0015498722119804656, 0.0015506727658090064, 0.0015514733196375472, 0.001552273873466088, 0.0015530744272946287, 0.0015538749811231695, 0.0015546755349517103, 0.001555476088780251, 0.0015562766426087918, 0.0015570771964373326, 0.0015578777502658733, 0.0015586783040944141, 0.001559478857922955, 0.0015602794117514957, 0.0015610799655800364, 0.0015618805194085772, 0.001562681073237118, 0.0015634816270656588, 0.0015642821808941995, 0.0015650827347227403, 0.001565883288551281, 0.0015666838423798219, 0.0015674843962083626, 0.0015682849500369034, 0.0015690855038654442, 0.001569886057693985, 0.0015706866115225257, 0.0015714871653510665, 0.0015722877191796073, 0.001573088273008148, 0.0015738888268366888, 0.0015746893806652296, 0.0015754899344937704, 0.0015762904883223111, 0.001577091042150852, 0.0015778915959793927, 0.0015786921498079334, 0.0015794927036364742, 0.001580293257465015, 0.0015810938112935558, 0.0015818943651220965, 0.0015826949189506373, 0.001583495472779178, 0.0015842960266077189, 0.0015850965804362596, 0.0015858971342648004, 0.0015866976880933412, 0.001587498241921882, 0.0015882987957504227, 0.0015890993495789635, 0.0015898999034075043, 0.001590700457236045, 0.0015915010110645858, 0.0015923015648931266, 0.0015931021187216674, 0.0015939026725502081, 0.001594703226378749, 0.0015955037802072897, 0.0015963043340358305, 0.0015971048878643712, 0.001597905441692912, 0.0015987059955214528, 0.0015995065493499936, 0.0016003071031785343, 0.001601107657007075, 0.0016019082108356159, 0.0016027087646641566, 0.0016035093184926974, 0.0016043098723212382, 0.001605110426149779, 0.0016059109799783197, 0.0016067115338068605, 0.0016075120876354013, 0.001608312641463942, 0.0016091131952924828, 0.0016099137491210236, 0.0016107143029495644, 0.0016115148567781051, 0.001612315410606646, 0.0016131159644351867, 0.0016139165182637275, 0.0016147170720922682, 0.001615517625920809, 0.0016163181797493498, 0.0016171187335778906, 0.0016179192874064313, 0.001618719841234972, 0.0016195203950635129, 0.0016203209488920537, 0.0016211215027205944, 0.0016219220565491352, 0.001622722610377676, 0.0016235231642062167, 0.0016243237180347575, 0.0016251242718632983, 0.001625924825691839, 0.0016267253795203798, 0.0016275259333489206, 0.0016283264871774614, 0.0016291270410060022, 0.001629927594834543, 0.0016307281486630837, 0.0016315287024916245, 0.0016323292563201653, 0.001633129810148706, 0.0016339303639772468, 0.0016347309178057876, 0.0016355314716343283, 0.0016363320254628691, 0.0016371325792914099, 0.0016379331331199507, 0.0016387336869484914, 0.0016395342407770322, 0.001640334794605573, 0.0016411353484341138, 0.0016419359022626545, 0.0016427364560911953, 0.001643537009919736, 0.0016443375637482768, 0.0016451381175768176, 0.0016459386714053584, 0.0016467392252338992, 0.00164753977906244, 0.0016483403328909807, 0.0016491408867195215, 0.0016499414405480623, 0.001650741994376603, 0.0016515425482051438, 0.0016523431020336846, 0.0016531436558622254, 0.0016539442096907661, 0.001654744763519307, 0.0016555453173478477, 0.0016563458711763884, 0.0016571464250049292, 0.00165794697883347, 0.0016587475326620108, 0.0016595480864905515, 0.0016603486403190923, 0.001661149194147633, 0.0016619497479761739, 0.0016627503018047146, 0.0016635508556332554, 0.0016643514094617962, 0.001665151963290337, 0.0016659525171188777, 0.0016667530709474185, 0.0016675536247759593, 0.0016683541786045, 0.0016691547324330408, 0.0016699552862615816, 0.0016707558400901224, 0.0016715563939186631, 0.001672356947747204, 0.0016731575015757447, 0.0016739580554042855, 0.0016747586092328262, 0.001675559163061367, 0.0016763597168899078, 0.0016771602707184485, 0.0016779608245469893, 0.00167876137837553, 0.0016795619322040709, 0.0016803624860326116, 0.0016811630398611524, 0.0016819635936896932, 0.001682764147518234, 0.0016835647013467747, 0.0016843652551753155, 0.0016851658090038563, 0.001685966362832397, 0.0016867669166609378, 0.0016875674704894786, 0.0016883680243180194, 0.0016891685781465601, 0.001689969131975101, 0.0016907696858036417, 0.0016915702396321825, 0.0016923707934607232, 0.001693171347289264, 0.0016939719011178048, 0.0016947724549463456, 0.0016955730087748863, 0.001696373562603427, 0.0016971741164319679, 0.0016979746702605087, 0.0016987752240890494, 0.0016995757779175902, 0.001700376331746131, 0.0017011768855746717, 0.0017019774394032125, 0.0017027779932317533, 0.001703578547060294, 0.0017043791008888348, 0.0017051796547173756, 0.0017059802085459164, 0.0017067807623744572, 0.001707581316202998, 0.0017083818700315387, 0.0017091824238600795, 0.0017099829776886202, 0.001710783531517161, 0.0017115840853457018, 0.0017123846391742426, 0.0017131851930027833, 0.0017139857468313241, 0.0017147863006598649, 0.0017155868544884057, 0.0017163874083169464, 0.0017171879621454872, 0.001717988515974028, 0.0017187890698025688, 0.0017195896236311095, 0.0017203901774596503, 0.001721190731288191, 0.0017219912851167318, 0.0017227918389452726, 0.0017235923927738134, 0.0017243929466023542, 0.001725193500430895, 0.0017259940542594357, 0.0017267946080879765, 0.0017275951619165173, 0.001728395715745058, 0.0017291962695735988, 0.0017299968234021396, 0.0017307973772306804, 0.0017315979310592211, 0.001732398484887762, 0.0017331990387163027, 0.0017339995925448434, 0.0017348001463733842, 0.001735600700201925, 0.0017364012540304658, 0.0017372018078590065, 0.0017380023616875473, 0.001738802915516088, 0.0017396034693446289, 0.0017404040231731696, 0.0017412045770017104, 0.0017420051308302512, 0.001742805684658792, 0.0017436062384873327, 0.0017444067923158735, 0.0017452073461444143, 0.001746007899972955, 0.0017468084538014958, 0.0017476090076300366, 0.0017484095614585774, 0.0017492101152871181, 0.001750010669115659, 0.0017508112229441997, 0.0017516117767727405, 0.0017524123306012812, 0.001753212884429822, 0.0017540134382583628, 0.0017548139920869035, 0.0017556145459154443, 0.001756415099743985, 0.0017572156535725259, 0.0017580162074010666, 0.0017588167612296074, 0.0017596173150581482, 0.001760417868886689, 0.0017612184227152297, 0.0017620189765437705, 0.0017628195303723113, 0.001763620084200852, 0.0017644206380293928, 0.0017652211918579336, 0.0017660217456864744, 0.0017668222995150151, 0.001767622853343556, 0.0017684234071720967, 0.0017692239610006375, 0.0017700245148291782, 0.001770825068657719, 0.0017716256224862598, 0.0017724261763148006, 0.0017732267301433413, 0.001774027283971882, 0.0017748278378004229, 0.0017756283916289636, 0.0017764289454575044, 0.0017772294992860452, 0.001778030053114586, 0.0017788306069431267, 0.0017796311607716675, 0.0017804317146002083, 0.001781232268428749, 0.0017820328222572898, 0.0017828333760858306, 0.0017836339299143714, 0.0017844344837429122, 0.001785235037571453, 0.0017860355913999937, 0.0017868361452285345, 0.0017876366990570752, 0.001788437252885616, 0.0017892378067141568, 0.0017900383605426976, 0.0017908389143712383, 0.0017916394681997791, 0.0017924400220283199, 0.0017932405758568607, 0.0017940411296854014, 0.0017948416835139422, 0.001795642237342483, 0.0017964427911710238, 0.0017972433449995645, 0.0017980438988281053, 0.001798844452656646, 0.0017996450064851868, 0.0018004455603137276, 0.0018012461141422684, 0.0018020466679708092, 0.00180284722179935, 0.0018036477756278907, 0.0018044483294564315, 0.0018052488832849723, 0.001806049437113513, 0.0018068499909420538, 0.0018076505447705946, 0.0018084510985991353, 0.0018092516524276761, 0.001810052206256217, 0.0018108527600847577, 0.0018116533139132984, 0.0018124538677418392, 0.00181325442157038, 0.0018140549753989208, 0.0018148555292274615, 0.0018156560830560023, 0.001816456636884543, 0.0018172571907130839, 0.0018180577445416246, 0.0018188582983701654, 0.0018196588521987062, 0.001820459406027247, 0.0018212599598557877, 0.0018220605136843285, 0.0018228610675128693, 0.00182366162134141, 0.0018244621751699508, 0.0018252627289984916, 0.0018260632828270324, 0.0018268638366555731, 0.001827664390484114, 0.0018284649443126547, 0.0018292654981411955, 0.0018300660519697362, 0.001830866605798277, 0.0018316671596268178, 0.0018324677134553585, 0.0018332682672838993, 0.00183406882111244, 0.0018348693749409809, 0.0018356699287695216, 0.0018364704825980624, 0.0018372710364266032, 0.001838071590255144, 0.0018388721440836847, 0.0018396726979122255, 0.0018404732517407663, 0.001841273805569307, 0.0018420743593978478, 0.0018428749132263886, 0.0018436754670549294, 0.0018444760208834701, 0.001845276574712011, 0.0018460771285405517, 0.0018468776823690925, 0.0018476782361976332, 0.001848478790026174, 0.0018492793438547148, 0.0018500798976832556, 0.0018508804515117963, 0.001851681005340337, 0.0018524815591688779, 0.0018532821129974186, 0.0018540826668259594, 0.0018548832206545002, 0.001855683774483041, 0.0018564843283115817, 0.0018572848821401225, 0.0018580854359686633, 0.001858885989797204, 0.0018596865436257448, 0.0018604870974542856, 0.0018612876512828264, 0.0018620882051113672, 0.001862888758939908, 0.0018636893127684487, 0.0018644898665969895, 0.0018652904204255302, 0.001866090974254071, 0.0018668915280826118, 0.0018676920819111526, 0.0018684926357396933, 0.0018692931895682341, 0.0018700937433967749, 0.0018708942972253157, 0.0018716948510538564, 0.0018724954048823972, 0.001873295958710938, 0.0018740965125394787, 0.0018748970663680195, 0.0018756976201965603, 0.001876498174025101, 0.0018772987278536418, 0.0018780992816821826, 0.0018788998355107234, 0.0018797003893392642, 0.001880500943167805, 0.0018813014969963457, 0.0018821020508248865, 0.0018829026046534273, 0.001883703158481968, 0.0018845037123105088, 0.0018853042661390496, 0.0018861048199675903, 0.0018869053737961311, 0.001887705927624672, 0.0018885064814532127, 0.0018893070352817534, 0.0018901075891102942, 0.001890908142938835, 0.0018917086967673758, 0.0018925092505959165, 0.0018933098044244573, 0.001894110358252998, 0.0018949109120815389, 0.0018957114659100796, 0.0018965120197386204, 0.0018973125735671612, 0.001898113127395702, 0.0018989136812242427, 0.0018997142350527835, 0.0019005147888813243, 0.001901315342709865, 0.0019021158965384058, 0.0019029164503669466, 0.0019037170041954874, 0.0019045175580240281, 0.001905318111852569, 0.0019061186656811097, 0.0019069192195096504, 0.0019077197733381912, 0.001908520327166732, 0.0019093208809952728, 0.0019101214348238135, 0.0019109219886523543, 0.001911722542480895, 0.0019125230963094359, 0.0019133236501379766, 0.0019141242039665174, 0.0019149247577950582, 0.001915725311623599, 0.0019165258654521397, 0.0019173264192806805, 0.0019181269731092213, 0.001918927526937762, 0.0019197280807663028, 0.0019205286345948436, 0.0019213291884233844, 0.0019221297422519251, 0.001922930296080466, 0.0019237308499090067, 0.0019245314037375475, 0.0019253319575660882, 0.001926132511394629, 0.0019269330652231698, 0.0019277336190517106, 0.0019285341728802513, 0.001929334726708792, 0.0019301352805373329, 0.0019309358343658736, 0.0019317363881944144, 0.0019325369420229552, 0.001933337495851496, 0.0019341380496800367, 0.0019349386035085775, 0.0019357391573371183, 0.001936539711165659, 0.0019373402649941998, 0.0019381408188227406, 0.0019389413726512814, 0.0019397419264798221, 0.001940542480308363, 0.0019413430341369037, 0.0019421435879654445, 0.0019429441417939852, 0.001943744695622526, 0.0019445452494510668, 0.0019453458032796076, 0.0019461463571081483, 0.001946946910936689, 0.0019477474647652299, 0.0019485480185937707, 0.0019493485724223114, 0.0019501491262508522, 0.001950949680079393, 0.0019517502339079337, 0.0019525507877364745, 0.001953351341564986, 0.0019541518953934225, 0.001954952449221859, 0.001955753003050296, 0.0019565535568787326, 0.0019573541107071693, 0.001958154664535606, 0.0019589552183640426, 0.0019597557721924793, 0.001960556326020916, 0.0019613568798493527, 0.0019621574336777894, 0.001962957987506226, 0.0019637585413346628, 0.0019645590951630995, 0.001965359648991536, 0.001966160202819973, 0.0019669607566484095, 0.0019677613104768462, 0.001968561864305283, 0.0019693624181337196, 0.0019701629719621563, 0.001970963525790593, 0.0019717640796190297, 0.0019725646334474664, 0.001973365187275903, 0.0019741657411043397, 0.0019749662949327764, 0.001975766848761213, 0.00197656740258965, 0.0019773679564180865, 0.001978168510246523, 0.00197896906407496, 0.0019797696179033966, 0.0019805701717318333, 0.00198137072556027, 0.0019821712793887066, 0.0019829718332171433, 0.00198377238704558, 0.0019845729408740167, 0.0019853734947024534, 0.00198617404853089, 0.0019869746023593268, 0.0019877751561877635, 0.0019885757100162, 0.001989376263844637, 0.0019901768176730735, 0.0019909773715015102, 0.001991777925329947, 0.0019925784791583836, 0.0019933790329868203, 0.001994179586815257, 0.0019949801406436937, 0.0019957806944721304, 0.001996581248300567, 0.0019973818021290037, 0.0019981823559574404, 0.001998982909785877, 0.001999783463614314, 0.0020005840174427505, 0.002001384571271187, 0.002002185125099624, 0.0020029856789280606, 0.0020037862327564973, 0.002004586786584934, 0.0020053873404133706, 0.0020061878942418073, 0.002006988448070244, 0.0020077890018986807, 0.0020085895557271174, 0.002009390109555554, 0.0020101906633839908, 0.0020109912172124275, 0.002011791771040864, 0.002012592324869301, 0.0020133928786977375, 0.0020141934325261742, 0.002014993986354611, 0.0020157945401830476, 0.0020165950940114843, 0.002017395647839921, 0.0020181962016683577, 0.0020189967554967944, 0.002019797309325231, 0.0020205978631536677, 0.0020213984169821044, 0.002022198970810541, 0.002022999524638978, 0.0020238000784674145, 0.002024600632295851, 0.002025401186124288, 0.0020262017399527246, 0.0020270022937811613, 0.002027802847609598, 0.0020286034014380346, 0.0020294039552664713, 0.002030204509094908, 0.0020310050629233447, 0.0020318056167517814, 0.002032606170580218, 0.0020334067244086548, 0.0020342072782370915, 0.002035007832065528, 0.002035808385893965, 0.0020366089397224015, 0.0020374094935508382, 0.002038210047379275, 0.0020390106012077116, 0.0020398111550361483, 0.002040611708864585, 0.0020414122626930217, 0.0020422128165214584, 0.002043013370349895, 0.0020438139241783317, 0.0020446144780067684, 0.002045415031835205, 0.002046215585663642, 0.0020470161394920785, 0.002047816693320515, 0.002048617247148952, 0.0020494178009773886, 0.0020502183548058253, 0.002051018908634262, 0.0020518194624626986, 0.0020526200162911353, 0.002053420570119572, 0.0020542211239480087, 0.0020550216777764454, 0.002055822231604882, 0.0020566227854333188, 0.0020574233392617555, 0.002058223893090192, 0.002059024446918629, 0.0020598250007470655, 0.0020606255545755022, 0.002061426108403939, 0.0020622266622323756, 0.0020630272160608123, 0.002063827769889249, 0.0020646283237176857, 0.0020654288775461224, 0.002066229431374559, 0.0020670299852029957, 0.0020678305390314324, 0.002068631092859869, 0.002069431646688306, 0.0020702322005167425, 0.002071032754345179, 0.002071833308173616, 0.0020726338620020526, 0.0020734344158304893, 0.002074234969658926, 0.0020750355234873626, 0.0020758360773157993, 0.002076636631144236, 0.0020774371849726727, 0.0020782377388011094, 0.002079038292629546, 0.002079838846457983, 0.0020806394002864195, 0.002081439954114856, 0.002082240507943293, 0.0020830410617717295, 0.0020838416156001662, 0.002084642169428603, 0.0020854427232570396, 0.0020862432770854763, 0.002087043830913913, 0.0020878443847423497, 0.0020886449385707864, 0.002089445492399223, 0.0020902460462276598, 0.0020910466000560964, 0.002091847153884533, 0.00209264770771297, 0.0020934482615414065, 0.002094248815369843, 0.00209504936919828, 0.0020958499230267166, 0.0020966504768551533, 0.00209745103068359, 0.0020982515845120266, 0.0020990521383404633, 0.0020998526921689, 0.0021006532459973367, 0.0021014537998257734, 0.00210225435365421, 0.002103054907482647, 0.0021038554613110835, 0.00210465601513952, 0.002105456568967957, 0.0021062571227963935, 0.0021070576766248302, 0.002107858230453267, 0.0021086587842817036, 0.0021094593381101403, 0.002110259891938577, 0.0021110604457670137, 0.0021118609995954504, 0.002112661553423887, 0.0021134621072523238, 0.0021142626610807604, 0.002115063214909197, 0.002115863768737634, 0.0021166643225660705, 0.002117464876394507, 0.002118265430222944, 0.0021190659840513806, 0.0021198665378798173, 0.002120667091708254, 0.0021214676455366907, 0.0021222681993651273, 0.002123068753193564, 0.0021238693070220007, 0.0021246698608504374, 0.002125470414678874, 0.002126270968507311, 0.0021270715223357475, 0.002127872076164184, 0.002128672629992621, 0.0021294731838210576, 0.0021302737376494942, 0.002131074291477931, 0.0021318748453063676, 0.0021326753991348043, 0.002133475952963241, 0.0021342765067916777, 0.0021350770606201144, 0.002135877614448551, 0.0021366781682769878, 0.0021374787221054244, 0.002138279275933861, 0.002139079829762298, 0.0021398803835907345, 0.002140680937419171, 0.002141481491247608, 0.0021422820450760446, 0.0021430825989044813, 0.002143883152732918, 0.0021446837065613547, 0.0021454842603897913, 0.002146284814218228, 0.0021470853680466647, 0.0021478859218751014, 0.002148686475703538, 0.002149487029531975, 0.0021502875833604115, 0.002151088137188848, 0.002151888691017285, 0.0021526892448457216, 0.0021534897986741582, 0.002154290352502595, 0.0021550909063310316, 0.0021558914601594683, 0.002156692013987905, 0.0021574925678163417, 0.0021582931216447784, 0.002159093675473215, 0.0021598942293016518, 0.0021606947831300885, 0.002161495336958525, 0.002162295890786962, 0.0021630964446153985, 0.002163896998443835, 0.002164697552272272, 0.0021654981061007086, 0.0021662986599291453, 0.002167099213757582, 0.0021678997675860187, 0.0021687003214144553, 0.002169500875242892, 0.0021703014290713287, 0.0021711019828997654, 0.002171902536728202, 0.002172703090556639, 0.0021735036443850755, 0.002174304198213512, 0.002175104752041949, 0.0021759053058703856, 0.0021767058596988222, 0.002177506413527259, 0.0021783069673556956, 0.0021791075211841323, 0.002179908075012569, 0.0021807086288410057, 0.0021815091826694424, 0.002182309736497879, 0.0021831102903263158, 0.0021839108441547525, 0.002184711397983189, 0.002185511951811626, 0.0021863125056400625, 0.002187113059468499, 0.002187913613296936, 0.0021887141671253726, 0.0021895147209538093, 0.002190315274782246, 0.0021911158286106827, 0.0021919163824391194, 0.002192716936267556, 0.0021935174900959927, 0.0021943180439244294, 0.002195118597752866, 0.002195919151581303, 0.0021967197054097395, 0.002197520259238176, 0.002198320813066613, 0.0021991213668950496, 0.0021999219207234862, 0.002200722474551923, 0.0022015230283803596, 0.0022023235822087963, 0.002203124136037233, 0.0022039246898656697, 0.0022047252436941064, 0.002205525797522543, 0.0022063263513509798, 0.0022071269051794165, 0.002207927459007853, 0.00220872801283629, 0.0022095285666647265, 0.002210329120493163, 0.0022111296743216, 0.0022119302281500366, 0.0022127307819784733, 0.00221353133580691, 0.0022143318896353467, 0.0022151324434637834, 0.00221593299729222, 0.0022167335511206567, 0.0022175341049490934, 0.00221833465877753, 0.002219135212605967, 0.0022199357664344035, 0.00222073632026284, 0.002221536874091277, 0.0022223374279197136, 0.0022231379817481503, 0.002223938535576587, 0.0022247390894050236, 0.0022255396432334603, 0.002226340197061897, 0.0022271407508903337, 0.0022279413047187704, 0.002228741858547207, 0.0022295424123756438, 0.0022303429662040805, 0.002231143520032517, 0.002231944073860954, 0.0022327446276893905, 0.0022335451815178272, 0.002234345735346264, 0.0022351462891747006, 0.0022359468430031373, 0.002236747396831574, 0.0022375479506600107, 0.0022383485044884474, 0.002239149058316884, 0.0022399496121453207, 0.0022407501659737574, 0.002241550719802194, 0.002242351273630631, 0.0022431518274590675, 0.002243952381287504, 0.002244752935115941, 0.0022455534889443776, 0.0022463540427728143, 0.002247154596601251, 0.0022479551504296876, 0.0022487557042581243, 0.002249556258086561, 0.0022503568119149977, 0.0022511573657434344, 0.002251957919571871, 0.0022527584734003078, 0.0022535590272287445, 0.002254359581057181, 0.002255160134885618, 0.0022559606887140545, 0.0022567612425424912, 0.002257561796370928, 0.0022583623501993646, 0.0022591629040278013, 0.002259963457856238, 0.0022607640116846747, 0.0022615645655131114, 0.002262365119341548, 0.0022631656731699847, 0.0022639662269984214, 0.002264766780826858, 0.002265567334655295, 0.0022663678884837315, 0.002267168442312168, 0.002267968996140605, 0.0022687695499690416, 0.0022695701037974783, 0.002270370657625915, 0.0022711712114543516, 0.0022719717652827883, 0.002272772319111225, 0.0022735728729396617, 0.0022743734267680984, 0.002275173980596535, 0.0022759745344249718, 0.0022767750882534085, 0.002277575642081845, 0.002278376195910282, 0.0022791767497387185, 0.0022799773035671552, 0.002280777857395592, 0.0022815784112240286, 0.0022823789650524653, 0.002283179518880902, 0.0022839800727093387, 0.0022847806265377754, 0.002285581180366212, 0.0022863817341946487, 0.0022871822880230854, 0.002287982841851522, 0.002288783395679959, 0.0022895839495083955, 0.002290384503336832, 0.002291185057165269, 0.0022919856109937056, 0.0022927861648221423, 0.002293586718650579, 0.0022943872724790156, 0.0022951878263074523, 0.002295988380135889, 0.0022967889339643257, 0.0022975894877927624, 0.002298390041621199, 0.0022991905954496358, 0.0022999911492780725, 0.002300791703106509, 0.002301592256934946, 0.0023023928107633825, 0.0023031933645918192, 0.002303993918420256, 0.0023047944722486926, 0.0023055950260771293, 0.002306395579905566, 0.0023071961337340027, 0.0023079966875624394, 0.002308797241390876, 0.0023095977952193127, 0.0023103983490477494, 0.002311198902876186, 0.002311999456704623, 0.0023128000105330595, 0.002313600564361496, 0.002314401118189933, 0.0023152016720183696, 0.0023160022258468063, 0.002316802779675243, 0.0023176033335036796, 0.0023184038873321163, 0.002319204441160553, 0.0023200049949889897, 0.0023208055488174264, 0.002321606102645863, 0.0023224066564743, 0.0023232072103027365, 0.002324007764131173, 0.00232480831795961, 0.0023256088717880465, 0.0023264094256164832, 0.00232720997944492, 0.0023280105332733566, 0.0023288110871017933, 0.00232961164093023, 0.0023304121947586667, 0.0023312127485871034, 0.00233201330241554, 0.0023328138562439768, 0.0023336144100724134, 0.00233441496390085, 0.002335215517729287, 0.0023360160715577235, 0.00233681662538616, 0.002337617179214597, 0.0023384177330430336, 0.0023392182868714703, 0.002340018840699907, 0.0023408193945283436, 0.0023416199483567803, 0.002342420502185217, 0.0023432210560136537, 0.0023440216098420904, 0.002344822163670527, 0.002345622717498964, 0.0023464232713274005, 0.002347223825155837, 0.002348024378984274, 0.0023488249328127105, 0.0023496254866411472, 0.002350426040469584, 0.0023512265942980206, 0.0023520271481264573, 0.002352827701954894, 0.0023536282557833307, 0.0023544288096117674, 0.002355229363440204, 0.0023560299172686408, 0.0023568304710970774, 0.002357631024925514, 0.002358431578753951, 0.0023592321325823875, 0.002360032686410824, 0.002360833240239261, 0.0023616337940676976, 0.0023624343478961343, 0.002363234901724571, 0.0023640354555530077, 0.0023648360093814443, 0.002365636563209881, 0.0023664371170383177, 0.0023672376708667544, 0.002368038224695191, 0.002368838778523628, 0.0023696393323520645, 0.002370439886180501, 0.002371240440008938, 0.0023720409938373745, 0.0023728415476658112, 0.002373642101494248, 0.0023744426553226846, 0.0023752432091511213, 0.002376043762979558, 0.0023768443168079947, 0.0023776448706364314, 0.002378445424464868, 0.0023792459782933048, 0.0023800465321217414, 0.002380847085950178, 0.002381647639778615, 0.0023824481936070515, 0.002383248747435488, 0.002384049301263925, 0.0023848498550923616, 0.0023856504089207983, 0.002386450962749235, 0.0023872515165776717, 0.0023880520704061083, 0.002388852624234545, 0.0023896531780629817, 0.0023904537318914184, 0.002391254285719855, 0.002392054839548292, 0.0023928553933767285, 0.002393655947205165, 0.002394456501033602, 0.0023952570548620386, 0.0023960576086904752, 0.002396858162518912, 0.0023976587163473486, 0.0023984592701757853, 0.002399259824004222, 0.0024000603778326587, 0.0024008609316610954, 0.002401661485489532, 0.0024024620393179688, 0.0024032625931464054, 0.002404063146974842, 0.002404863700803279, 0.0024056642546317155, 0.002406464808460152, 0.002407265362288589, 0.0024080659161170256, 0.0024088664699454623, 0.002409667023773899, 0.0024104675776023357, 0.0024112681314307723, 0.002412068685259209, 0.0024128692390876457, 0.0024136697929160824, 0.002414470346744519, 0.002415270900572956, 0.0024160714544013925, 0.002416872008229829, 0.002417672562058266, 0.0024184731158867026, 0.0024192736697151392, 0.002420074223543576, 0.0024208747773720126, 0.0024216753312004493, 0.002422475885028886, 0.0024232764388573227, 0.0024240769926857594, 0.002424877546514196, 0.0024256781003426328, 0.0024264786541710695, 0.002427279207999506, 0.002428079761827943, 0.0024288803156563795, 0.002429680869484816, 0.002430481423313253, 0.0024312819771416896, 0.0024320825309701263, 0.002432883084798563, 0.0024336836386269997, 0.0024344841924554364, 0.002435284746283873, 0.0024360853001123097, 0.0024368858539407464, 0.002437686407769183, 0.00243848696159762, 0.0024392875154260565, 0.002440088069254493, 0.00244088862308293, 0.0024416891769113666, 0.0024424897307398032, 0.00244329028456824, 0.0024440908383966766, 0.0024448913922251133, 0.00244569194605355, 0.0024464924998819867, 0.0024472930537104234, 0.00244809360753886, 0.0024488941613672968, 0.0024496947151957335, 0.00245049526902417, 0.002451295822852607, 0.0024520963766810435, 0.00245289693050948, 0.002453697484337917, 0.0024544980381663536, 0.0024552985919947903, 0.002456099145823227, 0.0024568996996516637, 0.0024577002534801004, 0.002458500807308537, 0.0024593013611369737, 0.0024601019149654104, 0.002460902468793847, 0.002461703022622284, 0.0024625035764507205, 0.002463304130279157, 0.002464104684107594, 0.0024649052379360306, 0.0024657057917644673, 0.002466506345592904, 0.0024673068994213406, 0.0024681074532497773, 0.002468908007078214, 0.0024697085609066507, 0.0024705091147350874, 0.002471309668563524, 0.0024721102223919608, 0.0024729107762203975, 0.002473711330048834, 0.002474511883877271, 0.0024753124377057075, 0.0024761129915341442, 0.002476913545362581, 0.0024777140991910176, 0.0024785146530194543, 0.002479315206847891, 0.0024801157606763277, 0.0024809163145047644, 0.002481716868333201, 0.0024825174221616377, 0.0024833179759900744, 0.002484118529818511, 0.002484919083646948, 0.0024857196374753845, 0.002486520191303821, 0.002487320745132258, 0.0024881212989606946, 0.0024889218527891313, 0.002489722406617568, 0.0024905229604460046, 0.0024913235142744413, 0.002492124068102878, 0.0024929246219313147, 0.0024937251757597514, 0.002494525729588188, 0.0024953262834166248, 0.0024961268372450615, 0.002496927391073498, 0.002497727944901935, 0.0024985284987303715, 0.0024993290525588082, 0.002500129606387245, 0.0025009301602156816, 0.0025017307140441183, 0.002502531267872555, 0.0025033318217009917, 0.0025041323755294284, 0.002504932929357865, 0.0025057334831863017, 0.0025065340370147384, 0.002507334590843175, 0.002508135144671612, 0.0025089356985000485, 0.002509736252328485, 0.002510536806156922, 0.0025113373599853586, 0.0025121379138137953, 0.002512938467642232, 0.0025137390214706686, 0.0025145395752991053, 0.002515340129127542, 0.0025161406829559787, 0.0025169412367844154, 0.002517741790612852, 0.0025185423444412888, 0.0025193428982697255, 0.002520143452098162, 0.002520944005926599, 0.0025217445597550355, 0.0025225451135834722, 0.002523345667411909, 0.0025241462212403456, 0.0025249467750687823, 0.002525747328897219, 0.0025265478827256557, 0.0025273484365540924, 0.002528148990382529, 0.0025289495442109657, 0.0025297500980394024, 0.002530550651867839, 0.002531351205696276, 0.0025321517595247125, 0.002532952313353149, 0.002533752867181586, 0.0025345534210100226, 0.0025353539748384593, 0.002536154528666896, 0.0025369550824953326, 0.0025377556363237693, 0.002538556190152206, 0.0025393567439806427, 0.0025401572978090794, 0.002540957851637516, 0.0025417584054659528, 0.0025425589592943895, 0.002543359513122826, 0.002544160066951263, 0.0025449606207796995, 0.0025457611746081362, 0.002546561728436573, 0.0025473622822650096, 0.0025481628360934463, 0.002548963389921883, 0.0025497639437503197, 0.0025505644975787564, 0.002551365051407193, 0.0025521656052356297, 0.0025529661590640664, 0.002553766712892503, 0.00255456726672094, 0.0025553678205493765, 0.002556168374377813, 0.00255696892820625, 0.0025577694820346866, 0.0025585700358631233, 0.00255937058969156, 0.0025601711435199966, 0.0025609716973484333, 0.00256177225117687, 0.0025625728050053067, 0.0025633733588337434, 0.00256417391266218, 0.0025649744664906168, 0.0025657750203190535, 0.00256657557414749, 0.002567376127975927, 0.0025681766818043635, 0.0025689772356328002, 0.002569777789461237, 0.0025705783432896736, 0.0025713788971181103, 0.002572179450946547, 0.0025729800047749837, 0.0025737805586034204, 0.002574581112431857, 0.0025753816662602937, 0.0025761822200887304, 0.002576982773917167, 0.002577783327745604, 0.0025785838815740405, 0.002579384435402477, 0.002580184989230914, 0.0025809855430593506, 0.0025817860968877873, 0.002582586650716224, 0.0025833872045446606, 0.0025841877583730973, 0.002584988312201534, 0.0025857888660299707, 0.0025865894198584074, 0.002587389973686844, 0.002588190527515281, 0.0025889910813437175, 0.002589791635172154, 0.002590592189000591, 0.0025913927428290275, 0.0025921932966574642, 0.002592993850485901, 0.0025937944043143376, 0.0025945949581427743, 0.002595395511971211, 0.0025961960657996477, 0.0025969966196280844, 0.002597797173456521, 0.0025985977272849578, 0.0025993982811133944, 0.002600198834941831, 0.002600999388770268, 0.0026017999425987045, 0.002602600496427141, 0.002603401050255578, 0.0026042016040840146, 0.0026050021579124513, 0.002605802711740888, 0.0026066032655693246, 0.0026074038193977613, 0.002608204373226198, 0.0026090049270546347, 0.0026098054808830714, 0.002610606034711508, 0.002611406588539945, 0.0026122071423683815, 0.002613007696196818, 0.002613808250025255, 0.0026146088038536915, 0.0026154093576821282, 0.002616209911510565, 0.0026170104653390016, 0.0026178110191674383, 0.002618611572995875, 0.0026194121268243117, 0.0026202126806527484, 0.002621013234481185, 0.0026218137883096218, 0.0026226143421380584, 0.002623414895966495, 0.002624215449794932, 0.0026250160036233685, 0.002625816557451805, 0.002626617111280242, 0.0026274176651086786, 0.0026282182189371153, 0.002629018772765552, 0.0026298193265939887, 0.0026306198804224253, 0.002631420434250862, 0.0026322209880792987, 0.0026330215419077354, 0.002633822095736172, 0.002634622649564609, 0.0026354232033930455, 0.002636223757221482, 0.002637024311049919, 0.0026378248648783555, 0.0026386254187067922, 0.002639425972535229, 0.0026402265263636656, 0.0026410270801921023, 0.002641827634020539, 0.0026426281878489757, 0.0026434287416774124, 0.002644229295505849, 0.0026450298493342858, 0.0026458304031627224, 0.002646630956991159, 0.002647431510819596, 0.0026482320646480325, 0.002649032618476469, 0.002649833172304906, 0.0026506337261333426, 0.0026514342799617793, 0.002652234833790216, 0.0026530353876186527, 0.0026538359414470893, 0.002654636495275526, 0.0026554370491039627, 0.0026562376029323994, 0.002657038156760836, 0.002657838710589273, 0.0026586392644177095, 0.002659439818246146, 0.002660240372074583, 0.0026610409259030196, 0.0026618414797314562, 0.002662642033559893, 0.0026634425873883296, 0.0026642431412167663, 0.002665043695045203, 0.0026658442488736397, 0.0026666448027020764, 0.002667445356530513, 0.0026682459103589498, 0.0026690464641873865, 0.002669847018015823, 0.00267064757184426, 0.0026714481256726965, 0.002672248679501133, 0.00267304923332957, 0.0026738497871580066, 0.0026746503409864433, 0.00267545089481488, 0.0026762514486433167, 0.0026770520024717533, 0.00267785255630019, 0.0026786531101286267, 0.0026794536639570634, 0.0026802542177855, 0.002681054771613937, 0.0026818553254423735, 0.00268265587927081, 0.002683456433099247, 0.0026842569869276836, 0.0026850575407561202, 0.002685858094584557, 0.0026866586484129936, 0.0026874592022414303, 0.002688259756069867, 0.0026890603098983037, 0.0026898608637267404, 0.002690661417555177, 0.0026914619713836138, 0.0026922625252120505, 0.002693063079040487, 0.002693863632868924, 0.0026946641866973605, 0.002695464740525797, 0.002696265294354234, 0.0026970658481826706, 0.0026978664020111073, 0.002698666955839544, 0.0026994675096679807, 0.0027002680634964174, 0.002701068617324854, 0.0027018691711532907, 0.0027026697249817274, 0.002703470278810164, 0.002704270832638601, 0.0027050713864670375, 0.002705871940295474, 0.002706672494123911, 0.0027074730479523476, 0.0027082736017807842, 0.002709074155609221, 0.0027098747094376576, 0.0027106752632660943, 0.002711475817094531, 0.0027122763709229677, 0.0027130769247514044, 0.002713877478579841, 0.0027146780324082778, 0.0027154785862367145, 0.002716279140065151, 0.002717079693893588, 0.0027178802477220245, 0.002718680801550461, 0.002719481355378898, 0.0027202819092073346, 0.0027210824630357713, 0.002721883016864208, 0.0027226835706926447, 0.0027234841245210814, 0.002724284678349518, 0.0027250852321779547, 0.0027258857860063914, 0.002726686339834828, 0.002727486893663265, 0.0027282874474917015, 0.002729088001320138, 0.002729888555148575, 0.0027306891089770116, 0.0027314896628054483, 0.002732290216633885, 0.0027330907704623216, 0.0027338913242907583, 0.002734691878119195, 0.0027354924319476317, 0.0027362929857760684, 0.002737093539604505, 0.0027378940934329418, 0.0027386946472613785, 0.002739495201089815, 0.002740295754918252, 0.0027410963087466885, 0.0027418968625751252, 0.002742697416403562, 0.0027434979702319986, 0.0027442985240604353, 0.002745099077888872, 0.0027458996317173087, 0.0027467001855457454, 0.002747500739374182, 0.0027483012932026187, 0.0027491018470310554, 0.002749902400859492, 0.002750702954687929, 0.0027515035085163655, 0.002752304062344802, 0.002753104616173239, 0.0027539051700016756, 0.0027547057238301123, 0.002755506277658549, 0.0027563068314869856, 0.0027571073853154223, 0.002757907939143859, 0.0027587084929722957, 0.0027595090468007324, 0.002760309600629169, 0.0027611101544576058, 0.0027619107082860425, 0.002762711262114479, 0.002763511815942916, 0.0027643123697713525, 0.0027651129235997892, 0.002765913477428226, 0.0027667140312566626, 0.0027675145850850993, 0.002768315138913536, 0.0027691156927419727, 0.0027699162465704094, 0.002770716800398846, 0.0027715173542272827, 0.0027723179080557194, 0.002773118461884156, 0.002773919015712593, 0.0027747195695410295, 0.002775520123369466, 0.002776320677197903, 0.0027771212310263396, 0.0027779217848547763, 0.002778722338683213, 0.0027795228925116496, 0.0027803234463400863, 0.002781124000168523, 0.0027819245539969597, 0.0027827251078253964, 0.002783525661653833, 0.0027843262154822698, 0.0027851267693107065, 0.002785927323139143, 0.00278672787696758, 0.0027875284307960165, 0.0027883289846244532, 0.00278912953845289, 0.0027899300922813266, 0.0027907306461097633, 0.0027915311999382, 0.0027923317537666367, 0.0027931323075950734, 0.00279393286142351, 0.0027947334152519467, 0.0027955339690803834, 0.00279633452290882, 0.002797135076737257, 0.0027979356305656935, 0.00279873618439413, 0.002799536738222567, 0.0028003372920510036, 0.0028011378458794403, 0.002801938399707877, 0.0028027389535363136, 0.0028035395073647503, 0.002804340061193187, 0.0028051406150216237, 0.0028059411688500604, 0.002806741722678497, 0.0028075422765069338, 0.0028083428303353705, 0.002809143384163807, 0.002809943937992244, 0.0028107444918206805, 0.0028115450456491172, 0.002812345599477554, 0.0028131461533059906, 0.0028139467071344273, 0.002814747260962864, 0.0028155478147913007, 0.0028163483686197374, 0.002817148922448174, 0.0028179494762766107, 0.0028187500301050474, 0.002819550583933484, 0.002820351137761921, 0.0028211516915903575, 0.002821952245418794, 0.002822752799247231, 0.0028235533530756676, 0.0028243539069041043, 0.002825154460732541, 0.0028259550145609776, 0.0028267555683894143, 0.002827556122217851, 0.0028283566760462877, 0.0028291572298747244, 0.002829957783703161, 0.0028307583375315978, 0.0028315588913600345, 0.002832359445188471, 0.002833159999016908, 0.0028339605528453445, 0.0028347611066737812, 0.002835561660502218, 0.0028363622143306546, 0.0028371627681590913, 0.002837963321987528, 0.0028387638758159647, 0.0028395644296444014, 0.002840364983472838, 0.0028411655373012747, 0.0028419660911297114, 0.002842766644958148, 0.002843567198786585, 0.0028443677526150215, 0.002845168306443458, 0.002845968860271895, 0.0028467694141003316, 0.0028475699679287683, 0.002848370521757205, 0.0028491710755856416, 0.0028499716294140783, 0.002850772183242515, 0.0028515727370709517, 0.0028523732908993884, 0.002853173844727825, 0.002853974398556262, 0.0028547749523846985, 0.002855575506213135, 0.002856376060041572, 0.0028571766138700085, 0.0028579771676984452, 0.002858777721526882, 0.0028595782753553186, 0.0028603788291837553, 0.002861179383012192, 0.0028619799368406287, 0.0028627804906690654, 0.002863581044497502, 0.0028643815983259388, 0.0028651821521543754, 0.002865982705982812, 0.002866783259811249, 0.0028675838136396855, 0.002868384367468122, 0.002869184921296559, 0.0028699854751249956, 0.0028707860289534323, 0.002871586582781869, 0.0028723871366103057, 0.0028731876904387423, 0.002873988244267179, 0.0028747887980956157, 0.0028755893519240524, 0.002876389905752489, 0.002877190459580926, 0.0028779910134093625, 0.002878791567237799, 0.002879592121066236, 0.0028803926748946725, 0.0028811932287231092, 0.002881993782551546, 0.0028827943363799826, 0.0028835948902084193, 0.002884395444036856, 0.0028851959978652927, 0.0028859965516937294, 0.002886797105522166, 0.0028875976593506028, 0.0028883982131790394, 0.002889198767007476, 0.002889999320835913, 0.0028907998746643495, 0.002891600428492786, 0.002892400982321223, 0.0028932015361496596, 0.0028940020899780963, 0.002894802643806533, 0.0028956031976349697, 0.0028964037514634063, 0.002897204305291843, 0.0028980048591202797, 0.0028988054129487164, 0.002899605966777153, 0.00290040652060559, 0.0029012070744340265, 0.002902007628262463, 0.0029028081820909, 0.0029036087359193366, 0.0029044092897477732, 0.00290520984357621, 0.0029060103974046466, 0.0029068109512330833, 0.00290761150506152, 0.0029084120588899567, 0.0029092126127183934, 0.00291001316654683, 0.0029108137203752668, 0.0029116142742037034, 0.00291241482803214, 0.002913215381860577, 0.0029140159356890135, 0.00291481648951745, 0.002915617043345887, 0.0029164175971743236, 0.0029172181510027603, 0.002918018704831197, 0.0029188192586596337, 0.0029196198124880703, 0.002920420366316507, 0.0029212209201449437, 0.0029220214739733804, 0.002922822027801817, 0.002923622581630254, 0.0029244231354586905, 0.002925223689287127, 0.002926024243115564, 0.0029268247969440006, 0.0029276253507724372, 0.002928425904600874, 0.0029292264584293106, 0.0029300270122577473, 0.002930827566086184, 0.0029316281199146207, 0.0029324286737430574, 0.002933229227571494, 0.0029340297813999308, 0.0029348303352283675, 0.002935630889056804, 0.002936431442885241, 0.0029372319967136775, 0.002938032550542114, 0.002938833104370551, 0.0029396336581989876, 0.0029404342120274243, 0.002941234765855861, 0.0029420353196842977, 0.0029428358735127343, 0.002943636427341171, 0.0029444369811696077, 0.0029452375349980444, 0.002946038088826481, 0.002946838642654918, 0.0029476391964833545, 0.002948439750311791, 0.002949240304140228, 0.0029500408579686646, 0.0029508414117971012, 0.002951641965625538, 0.0029524425194539746, 0.0029532430732824113, 0.002954043627110848, 0.0029548441809392847, 0.0029556447347677214, 0.002956445288596158, 0.0029572458424245948, 0.0029580463962530315, 0.002958846950081468, 0.002959647503909905, 0.0029604480577383415, 0.002961248611566778, 0.002962049165395215, 0.0029628497192236516, 0.0029636502730520883, 0.002964450826880525, 0.0029652513807089617, 0.0029660519345373984, 0.002966852488365835, 0.0029676530421942717, 0.0029684535960227084, 0.002969254149851145, 0.002970054703679582, 0.0029708552575080185, 0.002971655811336455, 0.002972456365164892, 0.0029732569189933286, 0.0029740574728217653, 0.002974858026650202, 0.0029756585804786386, 0.0029764591343070753, 0.002977259688135512, 0.0029780602419639487, 0.0029788607957923854, 0.002979661349620822, 0.0029804619034492588, 0.0029812624572776955, 0.002982063011106132, 0.002982863564934569, 0.0029836641187630055, 0.0029844646725914422, 0.002985265226419879, 0.0029860657802483156, 0.0029868663340767523, 0.002987666887905189, 0.0029884674417336257, 0.0029892679955620624, 0.002990068549390499, 0.0029908691032189357, 0.0029916696570473724, 0.002992470210875809, 0.002993270764704246, 0.0029940713185326825, 0.002994871872361119, 0.002995672426189556, 0.0029964729800179926, 0.0029972735338464293, 0.002998074087674866, 0.0029988746415033026, 0.0029996751953317393, 0.003000475749160176, 0.0030012763029886127, 0.0030020768568170494, 0.003002877410645486, 0.0030036779644739228, 0.0030044785183023595, 0.003005279072130796, 0.003006079625959233, 0.0030068801797876695, 0.0030076807336161062, 0.003008481287444543, 0.0030092818412729796, 0.0030100823951014163, 0.003010882948929853, 0.0030116835027582897, 0.0030124840565867264, 0.003013284610415163, 0.0030140851642435997, 0.0030148857180720364, 0.003015686271900473, 0.00301648682572891, 0.0030172873795573465, 0.003018087933385783, 0.00301888848721422, 0.0030196890410426566, 0.0030204895948710933, 0.00302129014869953, 0.0030220907025279666, 0.0030228912563564033, 0.00302369181018484, 0.0030244923640132767, 0.0030252929178417134, 0.00302609347167015, 0.0030268940254985868, 0.0030276945793270235, 0.00302849513315546, 0.003029295686983897, 0.0030300962408123335, 0.0030308967946407702, 0.003031697348469207, 0.0030324979022976436, 0.0030332984561260803, 0.003034099009954517, 0.0030348995637829537, 0.0030357001176113904, 0.003036500671439827, 0.0030373012252682637, 0.0030381017790967004, 0.003038902332925137, 0.003039702886753574, 0.0030405034405820105, 0.003041303994410447, 0.003042104548238884, 0.0030429051020673206, 0.0030437056558957573, 0.003044506209724194, 0.0030453067635526306, 0.0030461073173810673, 0.003046907871209504, 0.0030477084250379407, 0.0030485089788663774, 0.003049309532694814, 0.0030501100865232508, 0.0030509106403516875, 0.003051711194180124, 0.003052511748008561, 0.0030533123018369975, 0.0030541128556654342, 0.003054913409493871, 0.0030557139633223076, 0.0030565145171507443, 0.003057315070979181, 0.0030581156248076177, 0.0030589161786360544, 0.003059716732464491, 0.0030605172862929277, 0.0030613178401213644, 0.003062118393949801, 0.003062918947778238, 0.0030637195016066745, 0.003064520055435111, 0.003065320609263548, 0.0030661211630919846, 0.0030669217169204213, 0.003067722270748858, 0.0030685228245772946, 0.0030693233784057313, 0.003070123932234168, 0.0030709244860626047, 0.0030717250398910414, 0.003072525593719478, 0.0030733261475479148, 0.0030741267013763515, 0.003074927255204788, 0.003075727809033225, 0.0030765283628616615, 0.0030773289166900982, 0.003078129470518535, 0.0030789300243469716, 0.0030797305781754083, 0.003080531132003845, 0.0030813316858322817, 0.0030821322396607184, 0.003082932793489155, 0.0030837333473175917, 0.0030845339011460284, 0.003085334454974465, 0.003086135008802902, 0.0030869355626313385, 0.003087736116459775, 0.003088536670288212, 0.0030893372241166486, 0.0030901377779450853, 0.003090938331773522, 0.0030917388856019586, 0.0030925394394303953, 0.003093339993258832, 0.0030941405470872687, 0.0030949411009157054, 0.003095741654744142, 0.003096542208572579, 0.0030973427624010155, 0.003098143316229452, 0.003098943870057889, 0.0030997444238863255, 0.0031005449777147622, 0.003101345531543199, 0.0031021460853716356, 0.0031029466392000723, 0.003103747193028509, 0.0031045477468569457, 0.0031053483006853824, 0.003106148854513819, 0.0031069494083422558, 0.0031077499621706924, 0.003108550515999129, 0.003109351069827566, 0.0031101516236560025, 0.003110952177484439, 0.003111752731312876, 0.0031125532851413126, 0.0031133538389697493, 0.003114154392798186, 0.0031149549466266226, 0.0031157555004550593, 0.003116556054283496, 0.0031173566081119327, 0.0031181571619403694, 0.003118957715768806, 0.003119758269597243, 0.0031205588234256795, 0.003121359377254116, 0.003122159931082553, 0.0031229604849109895, 0.0031237610387394262, 0.003124561592567863, 0.0031253621463962996, 0.0031261627002247363, 0.003126963254053173, 0.0031277638078816097, 0.0031285643617100464, 0.003129364915538483, 0.0031301654693669198, 0.0031309660231953564, 0.003131766577023793, 0.00313256713085223, 0.0031333676846806665, 0.003134168238509103, 0.00313496879233754, 0.0031357693461659766, 0.0031365698999944133, 0.00313737045382285, 0.0031381710076512867, 0.0031389715614797233, 0.00313977211530816, 0.0031405726691365967, 0.0031413732229650334, 0.00314217377679347, 0.003142974330621907, 0.0031437748844503435, 0.00314457543827878, 0.003145375992107217, 0.0031461765459356535, 0.0031469770997640902, 0.003147777653592527, 0.0031485782074209636, 0.0031493787612494003, 0.003150179315077837, 0.0031509798689062737, 0.0031517804227347104, 0.003152580976563147, 0.0031533815303915838, 0.0031541820842200204, 0.003154982638048457, 0.003155783191876894, 0.0031565837457053305, 0.003157384299533767, 0.003158184853362204, 0.0031589854071906406, 0.0031597859610190773, 0.003160586514847514, 0.0031613870686759507, 0.0031621876225043873, 0.003162988176332824, 0.0031637887301612607, 0.0031645892839896974, 0.003165389837818134, 0.003166190391646571, 0.0031669909454750075, 0.003167791499303444, 0.003168592053131881, 0.0031693926069603176, 0.0031701931607887542, 0.003170993714617191, 0.0031717942684456276, 0.0031725948222740643, 0.003173395376102501, 0.0031741959299309377, 0.0031749964837593744, 0.003175797037587811, 0.0031765975914162478, 0.0031773981452446844, 0.003178198699073121, 0.003178999252901558, 0.0031797998067299945, 0.003180600360558431, 0.003181400914386868, 0.0031822014682153046, 0.0031830020220437413, 0.003183802575872178, 0.0031846031297006147, 0.0031854036835290513, 0.003186204237357488, 0.0031870047911859247, 0.0031878053450143614, 0.003188605898842798, 0.003189406452671235, 0.0031902070064996715, 0.003191007560328108, 0.003191808114156545, 0.0031926086679849816, 0.0031934092218134182, 0.003194209775641855, 0.0031950103294702916, 0.0031958108832987283, 0.003196611437127165, 0.0031974119909556017, 0.0031982125447840384, 0.003199013098612475, 0.0031998136524409118, 0.0032006142062693485, 0.003201414760097785, 0.003202215313926222, 0.0032030158677546585, 0.003203816421583095, 0.003204616975411532, 0.0032054175292399686, 0.0032062180830684053, 0.003207018636896842, 0.0032078191907252787, 0.0032086197445537154, 0.003209420298382152, 0.0032102208522105887, 0.0032110214060390254, 0.003211821959867462, 0.003212622513695899, 0.0032134230675243355, 0.003214223621352772, 0.003215024175181209, 0.0032158247290096456, 0.0032166252828380822, 0.003217425836666519, 0.0032182263904949556, 0.0032190269443233923, 0.003219827498151829, 0.0032206280519802657, 0.0032214286058087024, 0.003222229159637139, 0.0032230297134655758, 0.0032238302672940125, 0.003224630821122449, 0.003225431374950886, 0.0032262319287793225, 0.003227032482607759, 0.003227833036436196, 0.0032286335902646326, 0.0032294341440930693, 0.003230234697921506, 0.0032310352517499427, 0.0032318358055783794, 0.003232636359406816, 0.0032334369132352527, 0.0032342374670636894, 0.003235038020892126, 0.003235838574720563, 0.0032366391285489995, 0.003237439682377436, 0.003238240236205873, 0.0032390407900343096, 0.0032398413438627463, 0.003240641897691183, 0.0032414424515196196, 0.0032422430053480563, 0.003243043559176493, 0.0032438441130049297, 0.0032446446668333664, 0.003245445220661803, 0.0032462457744902398, 0.0032470463283186765, 0.003247846882147113, 0.00324864743597555, 0.0032494479898039865, 0.0032502485436324232, 0.00325104909746086, 0.0032518496512892966, 0.0032526502051177333, 0.00325345075894617, 0.0032542513127746067, 0.0032550518666030434, 0.00325585242043148, 0.0032566529742599167, 0.0032574535280883534, 0.00325825408191679, 0.003259054635745227, 0.0032598551895736635, 0.0032606557434021, 0.003261456297230537, 0.0032622568510589736, 0.0032630574048874103, 0.003263857958715847, 0.0032646585125442836, 0.0032654590663727203, 0.003266259620201157, 0.0032670601740295937, 0.0032678607278580304, 0.003268661281686467, 0.0032694618355149038, 0.0032702623893433405, 0.003271062943171777, 0.003271863497000214, 0.0032726640508286505, 0.0032734646046570872, 0.003274265158485524, 0.0032750657123139606, 0.0032758662661423973, 0.003276666819970834, 0.0032774673737992707, 0.0032782679276277074, 0.003279068481456144, 0.0032798690352845807, 0.0032806695891130174, 0.003281470142941454, 0.003282270696769891, 0.0032830712505983275, 0.003283871804426764, 0.003284672358255201, 0.0032854729120836376, 0.0032862734659120743, 0.003287074019740511, 0.0032878745735689476, 0.0032886751273973843, 0.003289475681225821, 0.0032902762350542577, 0.0032910767888826944, 0.003291877342711131, 0.0032926778965395678, 0.0032934784503680045, 0.003294279004196441, 0.003295079558024878, 0.0032958801118533145, 0.0032966806656817512, 0.003297481219510188, 0.0032982817733386246, 0.0032990823271670613, 0.003299882880995498, 0.0033006834348239347, 0.0033014839886523714, 0.003302284542480808, 0.0033030850963092447, 0.0033038856501376814, 0.003304686203966118, 0.003305486757794555, 0.0033062873116229915, 0.003307087865451428, 0.003307888419279865, 0.0033086889731083016, 0.0033094895269367383, 0.003310290080765175, 0.0033110906345936116, 0.0033118911884220483, 0.003312691742250485, 0.0033134922960789217, 0.0033142928499073584, 0.003315093403735795, 0.0033158939575642318, 0.0033166945113926685, 0.003317495065221105, 0.003318295619049542, 0.0033190961728779785, 0.0033198967267064152, 0.003320697280534852, 0.0033214978343632886, 0.0033222983881917253, 0.003323098942020162, 0.0033238994958485987, 0.0033247000496770354, 0.003325500603505472, 0.0033263011573339087, 0.0033271017111623454, 0.003327902264990782, 0.003328702818819219, 0.0033295033726476555, 0.003330303926476092, 0.003331104480304529, 0.0033319050341329656, 0.0033327055879614023, 0.003333506141789839, 0.0033343066956182756, 0.0033351072494467123, 0.003335907803275149, 0.0033367083571035857, 0.0033375089109320224, 0.003338309464760459, 0.0033391100185888958, 0.0033399105724173325, 0.003340711126245769, 0.003341511680074206, 0.0033423122339026425, 0.0033431127877310792, 0.003343913341559516, 0.0033447138953879526, 0.0033455144492163893, 0.003346315003044826, 0.0033471155568732627, 0.0033479161107016994, 0.003348716664530136, 0.0033495172183585727, 0.0033503177721870094, 0.003351118326015446, 0.003351918879843883, 0.0033527194336723195, 0.003353519987500756, 0.003354320541329193, 0.0033551210951576296, 0.0033559216489860663, 0.003356722202814503, 0.0033575227566429396, 0.0033583233104713763, 0.003359123864299813, 0.0033599244181282497, 0.0033607249719566864, 0.003361525525785123, 0.00336232607961356, 0.0033631266334419965, 0.003363927187270433, 0.00336472774109887, 0.0033655282949273065, 0.0033663288487557432, 0.00336712940258418, 0.0033679299564126166, 0.0033687305102410533, 0.00336953106406949, 0.0033703316178979267, 0.0033711321717263634, 0.0033719327255548, 0.0033727332793832368, 0.0033735338332116734, 0.00337433438704011, 0.003375134940868547, 0.0033759354946969835, 0.00337673604852542, 0.003377536602353857, 0.0033783371561822936, 0.0033791377100107303, 0.003379938263839167, 0.0033807388176676036, 0.0033815393714960403, 0.003382339925324477, 0.0033831404791529137, 0.0033839410329813504, 0.003384741586809787, 0.003385542140638224, 0.0033863426944666605, 0.003387143248295097, 0.003387943802123534, 0.0033887443559519705, 0.0033895449097804072, 0.003390345463608844, 0.0033911460174372806, 0.0033919465712657173, 0.003392747125094154, 0.0033935476789225907, 0.0033943482327510274, 0.003395148786579464, 0.0033959493404079008, 0.0033967498942363374, 0.003397550448064774, 0.003398351001893211, 0.0033991515557216475, 0.003399952109550084, 0.003400752663378521, 0.0034015532172069576, 0.0034023537710353943, 0.003403154324863831, 0.0034039548786922677, 0.0034047554325207043, 0.003405555986349141, 0.0034063565401775777, 0.0034071570940060144, 0.003407957647834451, 0.003408758201662888, 0.0034095587554913245, 0.003410359309319761, 0.003411159863148198, 0.0034119604169766346, 0.0034127609708050712, 0.003413561524633508, 0.0034143620784619446, 0.0034151626322903813, 0.003415963186118818, 0.0034167637399472547, 0.0034175642937756914, 0.003418364847604128, 0.0034191654014325648, 0.0034199659552610014, 0.003420766509089438, 0.003421567062917875, 0.0034223676167463115, 0.003423168170574748, 0.003423968724403185, 0.0034247692782316216, 0.0034255698320600583, 0.003426370385888495, 0.0034271709397169317, 0.0034279714935453683, 0.003428772047373805, 0.0034295726012022417, 0.0034303731550306784, 0.003431173708859115, 0.003431974262687552, 0.0034327748165159885, 0.003433575370344425, 0.003434375924172862, 0.0034351764780012986, 0.0034359770318297352, 0.003436777585658172, 0.0034375781394866086, 0.0034383786933150453, 0.003439179247143482, 0.0034399798009719187, 0.0034407803548003554, 0.003441580908628792, 0.0034423814624572288, 0.0034431820162856655, 0.003443982570114102, 0.003444783123942539, 0.0034455836777709755, 0.003446384231599412, 0.003447184785427849, 0.0034479853392562856, 0.0034487858930847223, 0.003449586446913159, 0.0034503870007415957, 0.0034511875545700323, 0.003451988108398469, 0.0034527886622269057, 0.0034535892160553424, 0.003454389769883779, 0.003455190323712216, 0.0034559908775406525, 0.003456791431369089, 0.003457591985197526, 0.0034583925390259626, 0.0034591930928543992, 0.003459993646682836, 0.0034607942005112726, 0.0034615947543397093, 0.003462395308168146, 0.0034631958619965827, 0.0034639964158250194, 0.003464796969653456, 0.0034655975234818928, 0.0034663980773103295, 0.003467198631138766, 0.003467999184967203, 0.0034687997387956395, 0.003469600292624076, 0.003470400846452513, 0.0034712014002809496, 0.0034720019541093863, 0.003472802507937823, 0.0034736030617662597, 0.0034744036155946964, 0.003475204169423133, 0.0034760047232515697, 0.0034768052770800064, 0.003477605830908443, 0.00347840638473688, 0.0034792069385653165, 0.003480007492393753, 0.00348080804622219, 0.0034816086000506266, 0.0034824091538790632, 0.0034832097077075, 0.0034840102615359366, 0.0034848108153643733, 0.00348561136919281, 0.0034864119230212467, 0.0034872124768496834, 0.00348801303067812, 0.0034888135845065568, 0.0034896141383349935, 0.00349041469216343, 0.003491215245991867, 0.0034920157998203035, 0.00349281635364874, 0.003493616907477177, 0.0034944174613056136, 0.0034952180151340503, 0.003496018568962487, 0.0034968191227909237, 0.0034976196766193604, 0.003498420230447797, 0.0034992207842762337, 0.0035000213381046704, 0.003500821891933107, 0.003501622445761544, 0.0035024229995899805, 0.003503223553418417, 0.003504024107246854, 0.0035048246610752906, 0.0035056252149037273, 0.003506425768732164, 0.0035072263225606006, 0.0035080268763890373, 0.003508827430217474, 0.0035096279840459107, 0.0035104285378743474, 0.003511229091702784, 0.0035120296455312208, 0.0035128301993596575, 0.003513630753188094, 0.003514431307016531, 0.0035152318608449675, 0.0035160324146734042, 0.003516832968501841, 0.0035176335223302776, 0.0035184340761587143, 0.003519234629987151, 0.0035200351838155877, 0.0035208357376440244, 0.003521636291472461, 0.0035224368453008977, 0.0035232373991293344, 0.003524037952957771, 0.003524838506786208, 0.0035256390606146445, 0.003526439614443081, 0.003527240168271518, 0.0035280407220999546, 0.0035288412759283913, 0.003529641829756828, 0.0035304423835852646, 0.0035312429374137013, 0.003532043491242138, 0.0035328440450705747, 0.0035336445988990114, 0.003534445152727448, 0.0035352457065558848, 0.0035360462603843215, 0.003536846814212758, 0.003537647368041195, 0.0035384479218696315, 0.0035392484756980682, 0.003540049029526505, 0.0035408495833549416, 0.0035416501371833783, 0.003542450691011815, 0.0035432512448402517, 0.0035440517986686884, 0.003544852352497125, 0.0035456529063255617, 0.0035464534601539984, 0.003547254013982435, 0.003548054567810872, 0.0035488551216393085, 0.003549655675467745, 0.003550456229296182, 0.0035512567831246186, 0.0035520573369530553, 0.003552857890781492, 0.0035536584446099286, 0.0035544589984383653, 0.003555259552266802, 0.0035560601060952387, 0.0035568606599236754, 0.003557661213752112, 0.0035584617675805488, 0.0035592623214089855, 0.003560062875237422, 0.003560863429065859, 0.0035616639828942955, 0.0035624645367227322, 0.003563265090551169, 0.0035640656443796056, 0.0035648661982080423, 0.003565666752036479, 0.0035664673058649157, 0.0035672678596933524, 0.003568068413521789, 0.0035688689673502257, 0.0035696695211786624, 0.003570470075007099, 0.003571270628835536, 0.0035720711826639725, 0.003572871736492409, 0.003573672290320846, 0.0035744728441492826, 0.0035752733979777193, 0.003576073951806156, 0.0035768745056345926, 0.0035776750594630293, 0.003578475613291466, 0.0035792761671199027, 0.0035800767209483394, 0.003580877274776776, 0.0035816778286052128, 0.0035824783824336495, 0.003583278936262086, 0.003584079490090523, 0.0035848800439189595, 0.0035856805977473962, 0.003586481151575833, 0.0035872817054042696, 0.0035880822592327063, 0.003588882813061143, 0.0035896833668895797, 0.0035904839207180164, 0.003591284474546453, 0.0035920850283748897, 0.0035928855822033264, 0.003593686136031763, 0.0035944866898602, 0.0035952872436886365, 0.003596087797517073, 0.00359688835134551, 0.0035976889051739466, 0.0035984894590023833, 0.00359929001283082, 0.0036000905666592566, 0.0036008911204876933, 0.00360169167431613, 0.0036024922281445667, 0.0036032927819730034, 0.00360409333580144, 0.003604893889629877, 0.0036056944434583135, 0.00360649499728675, 0.003607295551115187, 0.0036080961049436235, 0.0036088966587720602, 0.003609697212600497, 0.0036104977664289336, 0.0036112983202573703, 0.003612098874085807, 0.0036128994279142437, 0.0036136999817426804, 0.003614500535571117, 0.0036153010893995538, 0.0036161016432279904, 0.003616902197056427, 0.003617702750884864, 0.0036185033047133005, 0.003619303858541737, 0.003620104412370174, 0.0036209049661986106, 0.0036217055200270473, 0.003622506073855484, 0.0036233066276839206, 0.0036241071815123573, 0.003624907735340794, 0.0036257082891692307, 0.0036265088429976674, 0.003627309396826104, 0.003628109950654541, 0.0036289105044829775, 0.003629711058311414, 0.003630511612139851, 0.0036313121659682875, 0.0036321127197967242, 0.003632913273625161, 0.0036337138274535976, 0.0036345143812820343, 0.003635314935110471, 0.0036361154889389077, 0.0036369160427673444, 0.003637716596595781, 0.0036385171504242178, 0.0036393177042526544, 0.003640118258081091, 0.003640918811909528, 0.0036417193657379645, 0.003642519919566401, 0.003643320473394838, 0.0036441210272232746, 0.0036449215810517113, 0.003645722134880148, 0.0036465226887085847, 0.0036473232425370213, 0.003648123796365458, 0.0036489243501938947, 0.0036497249040223314, 0.003650525457850768, 0.003651326011679205, 0.0036521265655076415, 0.003652927119336078, 0.003653727673164515, 0.0036545282269929515, 0.0036553287808213882, 0.003656129334649825, 0.0036569298884782616, 0.0036577304423066983, 0.003658530996135135, 0.0036593315499635717, 0.0036601321037920084, 0.003660932657620445, 0.0036617332114488818, 0.0036625337652773184, 0.003663334319105755, 0.003664134872934192, 0.0036649354267626285, 0.003665735980591065, 0.003666536534419502, 0.0036673370882479386, 0.0036681376420763753, 0.003668938195904812, 0.0036697387497332487, 0.0036705393035616853, 0.003671339857390122, 0.0036721404112185587, 0.0036729409650469954, 0.003673741518875432, 0.003674542072703869, 0.0036753426265323055, 0.003676143180360742, 0.003676943734189179, 0.0036777442880176156, 0.0036785448418460522, 0.003679345395674489, 0.0036801459495029256, 0.0036809465033313623, 0.003681747057159799, 0.0036825476109882357, 0.0036833481648166724, 0.003684148718645109, 0.0036849492724735458, 0.0036857498263019824, 0.003686550380130419, 0.003687350933958856, 0.0036881514877872925, 0.003688952041615729, 0.003689752595444166, 0.0036905531492726026, 0.0036913537031010393, 0.003692154256929476, 0.0036929548107579127, 0.0036937553645863493, 0.003694555918414786, 0.0036953564722432227, 0.0036961570260716594, 0.003696957579900096, 0.003697758133728533, 0.0036985586875569695, 0.003699359241385406, 0.003700159795213843, 0.0037009603490422796, 0.0037017609028707162, 0.003702561456699153, 0.0037033620105275896, 0.0037041625643560263, 0.003704963118184463, 0.0037057636720128997, 0.0037065642258413364, 0.003707364779669773, 0.0037081653334982098, 0.0037089658873266465, 0.003709766441155083, 0.00371056699498352, 0.0037113675488119565, 0.003712168102640393, 0.00371296865646883, 0.0037137692102972666, 0.0037145697641257033, 0.00371537031795414, 0.0037161708717825767, 0.0037169714256110134, 0.00371777197943945, 0.0037185725332678867, 0.0037193730870963234, 0.00372017364092476, 0.003720974194753197, 0.0037217747485816335, 0.00372257530241007, 0.003723375856238507, 0.0037241764100669436, 0.0037249769638953802, 0.003725777517723817, 0.0037265780715522536, 0.0037273786253806903, 0.003728179179209127, 0.0037289797330375637, 0.0037297802868660004, 0.003730580840694437, 0.0037313813945228738, 0.0037321819483513105, 0.003732982502179747, 0.003733783056008184, 0.0037345836098366205, 0.003735384163665057, 0.003736184717493494, 0.0037369852713219306, 0.0037377858251503673, 0.003738586378978804, 0.0037393869328072407, 0.0037401874866356774, 0.003740988040464114, 0.0037417885942925507, 0.0037425891481209874, 0.003743389701949424, 0.003744190255777861, 0.0037449908096062975, 0.003745791363434734, 0.003746591917263171, 0.0037473924710916076, 0.0037481930249200443, 0.003748993578748481, 0.0037497941325769176, 0.0037505946864053543, 0.003751395240233791, 0.0037521957940622277, 0.0037529963478906644, 0.003753796901719101, 0.0037545974555475378, 0.0037553980093759745, 0.003756198563204411, 0.003756999117032848, 0.0037577996708612845, 0.0037586002246897212, 0.003759400778518158, 0.0037602013323465946, 0.0037610018861750313, 0.003761802440003468, 0.0037626029938319047, 0.0037634035476603414, 0.003764204101488778, 0.0037650046553172147, 0.0037658052091456514, 0.003766605762974088, 0.003767406316802525, 0.0037682068706309615, 0.003769007424459398, 0.003769807978287835, 0.0037706085321162716, 0.0037714090859447083, 0.003772209639773145, 0.0037730101936015816, 0.0037738107474300183, 0.003774611301258455, 0.0037754118550868917, 0.0037762124089153284, 0.003777012962743765, 0.0037778135165722018, 0.0037786140704006385, 0.003779414624229075, 0.003780215178057512, 0.0037810157318859485, 0.0037818162857143852, 0.003782616839542822, 0.0037834173933712586, 0.0037842179471996953, 0.003785018501028132, 0.0037858190548565687, 0.0037866196086850054, 0.003787420162513442, 0.0037882207163418787, 0.0037890212701703154, 0.003789821823998752, 0.003790622377827189, 0.0037914229316556255, 0.003792223485484062, 0.003793024039312499, 0.0037938245931409356, 0.0037946251469693723, 0.003795425700797809, 0.0037962262546262456, 0.0037970268084546823, 0.003797827362283119, 0.0037986279161115557, 0.0037994284699399924, 0.003800229023768429, 0.0038010295775968658, 0.0038018301314253025, 0.003802630685253739, 0.003803431239082176, 0.0038042317929106125, 0.0038050323467390492, 0.003805832900567486, 0.0038066334543959226, 0.0038074340082243593, 0.003808234562052796, 0.0038090351158812327, 0.0038098356697096694, 0.003810636223538106, 0.0038114367773665427, 0.0038122373311949794, 0.003813037885023416, 0.003813838438851853, 0.0038146389926802895, 0.003815439546508726, 0.003816240100337163, 0.0038170406541655996, 0.0038178412079940363, 0.003818641761822473, 0.0038194423156509096, 0.0038202428694793463, 0.003821043423307783, 0.0038218439771362197, 0.0038226445309646564, 0.003823445084793093, 0.0038242456386215298, 0.0038250461924499665, 0.003825846746278403, 0.00382664730010684, 0.0038274478539352765, 0.0038282484077637132, 0.00382904896159215, 0.0038298495154205866, 0.0038306500692490233, 0.00383145062307746, 0.0038322511769058967, 0.0038330517307343334, 0.00383385228456277, 0.0038346528383912067, 0.0038354533922196434, 0.00383625394604808, 0.003837054499876517, 0.0038378550537049535, 0.00383865560753339, 0.003839456161361827, 0.0038402567151902636, 0.0038410572690187003, 0.003841857822847137, 0.0038426583766755736, 0.0038434589305040103, 0.003844259484332447, 0.0038450600381608837, 0.0038458605919893204, 0.003846661145817757, 0.0038474616996461938, 0.0038482622534746305, 0.003849062807303067, 0.003849863361131504, 0.0038506639149599405, 0.0038514644687883772, 0.003852265022616814, 0.0038530655764452506, 0.0038538661302736873, 0.003854666684102124, 0.0038554672379305607, 0.0038562677917589974, 0.003857068345587434, 0.0038578688994158707, 0.0038586694532443074, 0.003859470007072744, 0.003860270560901181, 0.0038610711147296175, 0.003861871668558054, 0.003862672222386491, 0.0038634727762149276, 0.0038642733300433643, 0.003865073883871801, 0.0038658744377002376, 0.0038666749915286743, 0.003867475545357111, 0.0038682760991855477, 0.0038690766530139844, 0.003869877206842421, 0.003870677760670858, 0.0038714783144992945, 0.003872278868327731, 0.003873079422156168, 0.0038738799759846045, 0.0038746805298130412, 0.003875481083641478, 0.0038762816374699146, 0.0038770821912983513, 0.003877882745126788, 0.0038786832989552247, 0.0038794838527836614, 0.003880284406612098, 0.0038810849604405348, 0.0038818855142689714, 0.003882686068097408, 0.003883486621925845, 0.0038842871757542815, 0.003885087729582718, 0.003885888283411155, 0.0038866888372395916, 0.0038874893910680283, 0.003888289944896465, 0.0038890904987249016, 0.0038898910525533383, 0.003890691606381775, 0.0038914921602102117, 0.0038922927140386484, 0.003893093267867085, 0.003893893821695522, 0.0038946943755239585, 0.003895494929352395, 0.003896295483180832, 0.0038970960370092685, 0.0038978965908377052, 0.003898697144666142, 0.0038994976984945786, 0.0039002982523230153, 0.003901098806151452, 0.0039018993599798887, 0.0039026999138083254, 0.003903500467636762, 0.0039043010214651988, 0.0039051015752936354, 0.003905902129122072, 0.003906702682950391, 0.00390750323677862, 0.003908303790606848, 0.003909104344435077, 0.003909904898263305, 0.003910705452091534, 0.0039115060059197624, 0.003912306559747991, 0.0039131071135762195, 0.003913907667404448, 0.0039147082212326765, 0.003915508775060905, 0.003916309328889134, 0.003917109882717362, 0.003917910436545591, 0.003918710990373819, 0.003919511544202048, 0.003920312098030276, 0.003921112651858505, 0.003921913205686733, 0.003922713759514962, 0.00392351431334319, 0.003924314867171419, 0.003925115420999647, 0.003925915974827876, 0.003926716528656104, 0.003927517082484333, 0.003928317636312561, 0.00392911819014079, 0.0039299187439690185, 0.003930719297797247, 0.0039315198516254755, 0.003932320405453704, 0.0039331209592819326, 0.003933921513110161, 0.00393472206693839, 0.003935522620766618, 0.003936323174594847, 0.003937123728423075, 0.003937924282251304, 0.003938724836079532, 0.003939525389907761, 0.003940325943735989, 0.003941126497564218, 0.003941927051392446, 0.003942727605220675, 0.003943528159048903, 0.003944328712877132, 0.00394512926670536, 0.003945929820533589, 0.0039467303743618174, 0.003947530928190046, 0.0039483314820182745, 0.003949132035846503, 0.0039499325896747315, 0.00395073314350296, 0.003951533697331189, 0.003952334251159417, 0.003953134804987646, 0.003953935358815874, 0.003954735912644103, 0.003955536466472331, 0.00395633702030056, 0.003957137574128788, 0.003957938127957017, 0.003958738681785245, 0.003959539235613474, 0.003960339789441702, 0.003961140343269931, 0.003961940897098159, 0.003962741450926388, 0.003963542004754616, 0.003964342558582845, 0.0039651431124110735, 0.003965943666239302, 0.0039667442200675305, 0.003967544773895759, 0.0039683453277239876, 0.003969145881552216, 0.003969946435380445, 0.003970746989208673, 0.003971547543036902, 0.00397234809686513, 0.003973148650693359, 0.003973949204521587, 0.003974749758349816, 0.003975550312178044, 0.003976350866006273, 0.003977151419834501, 0.00397795197366273, 0.003978752527490958, 0.003979553081319187, 0.003980353635147415, 0.003981154188975644, 0.0039819547428038725, 0.003982755296632101, 0.0039835558504603295, 0.003984356404288558, 0.0039851569581167865, 0.003985957511945015, 0.003986758065773244, 0.003987558619601472, 0.003988359173429701, 0.003989159727257929, 0.003989960281086158, 0.003990760834914386, 0.003991561388742615, 0.003992361942570843, 0.003993162496399072, 0.0039939630502273, 0.003994763604055529, 0.003995564157883757, 0.003996364711711986, 0.003997165265540214, 0.003997965819368443, 0.0039987663731966714, 0.0039995669270249, 0.0040003674808531285, 0.004001168034681357, 0.0040019685885095855, 0.004002769142337814]

net_absorption_emission_protons_inv_s_inv_cm3_1emin3 = [0.0, -2.2577680957322018e+23, -5.2777657253027716e+23, -8.817749488812008e+23, -1.294594837241556e+24, -1.7024228615484543e+24, -2.295265235391164e+24, -2.744473531113943e+24, -3.1860073376498935e+24, -3.647545208590426e+24, -4.2376451668574925e+24, -4.858730267936102e+24, -5.430819504585012e+24, -6.021637884076655e+24, -6.563626130554372e+24, -6.953308234199892e+24, -7.342839208476792e+24, -7.670657821483107e+24, -8.078410516754563e+24, -8.477506078034767e+24, -8.843544565552414e+24, -9.251050120887106e+24, -9.642627857838846e+24, -1.0032130631841881e+25, -1.0370715831138135e+25, -1.068459563985553e+25, -1.103964473243861e+25, -1.1351151875857505e+25, -1.167757832472996e+25, -1.201632151872312e+25, -1.2364550668605688e+25, -1.2734639349302564e+25, -1.3123427023661568e+25, -1.350887037509336e+25, -1.3865890196719456e+25, -1.4229337097001822e+25, -1.458941055554752e+25, -1.495906899173998e+25, -1.5319671392477042e+25, -1.5666331532395665e+25, -1.602924973113826e+25, -1.637241841880882e+25, -1.66897572323516e+25, -1.700264673777112e+25, -1.727293186210167e+25, -1.7508446963546144e+25, -1.7726791342837392e+25, -1.7937781510940587e+25, -1.8121898428542499e+25, -1.8281146136982164e+25, -1.8401066108118929e+25, -1.8496235812511956e+25, -1.8573714144301686e+25, -1.8634068945222804e+25, -1.8674942001981887e+25, -1.8708141770351278e+25, -1.873621393582011e+25, -1.8756516975271147e+25, -1.877086188980678e+25, -1.8780121434477995e+25, -1.8788685533106483e+25, -1.8794909776993307e+25, -1.879924280461458e+25, -1.880224202566632e+25, -1.88036658056146e+25, -1.8803788067542186e+25, -1.8803839197651993e+25, -1.8803887906114026e+25, -1.880393448566087e+25, -1.880400247890495e+25, -1.8804050671782979e+25, -1.8804094756498918e+25, -1.8804140527729173e+25, -1.8804169209671705e+25, -1.8804213809971657e+25, -1.8804238167504686e+25, -1.8804263409839448e+25, -1.8804288652174213e+25, -1.8804309643282613e+25, -1.8804344422540152e+25, -1.8804478511577484e+25, -1.8804692171608206e+25, -1.8806224672836452e+25, -1.880849861185406e+25, -1.8811887225957256e+25, -1.8818447492682906e+25, -1.8832685862212875e+25, -1.8854693209019215e+25, -1.8886456225094078e+25, -1.8936113294723034e+25, -1.9010147662252925e+25, -1.9112173380842808e+25, -1.9240636282149851e+25, -1.9394036832523378e+25, -1.957194222492001e+25, -1.9789096057417816e+25, -2.009748587198043e+25, -2.0538999266764363e+25, -2.1141037811979205e+25, -2.190404590556078e+25, -2.281573802448944e+25, -2.388153573166882e+25, -2.5046527702012072e+25, -2.634945732305596e+25, -2.792231869257842e+25, -2.9706244863737015e+25, -3.167598911879639e+25, -3.3774050443671124e+25, -3.6014260873330383e+25, -3.8454942583765093e+25, -4.117634195235316e+25, -4.4211763641209815e+25, -4.766611153751045e+25, -5.153465670982984e+25, -5.574962929397665e+25, -6.03537039204045e+25, -6.520278403174273e+25, -7.049702791122324e+25, -7.631260285428461e+25, -8.279329569076276e+25, -8.98160992443511e+25, -9.716272993888956e+25, -1.0475470555196128e+26, -1.1289324979638303e+26, -1.216836573939331e+26, -1.3115696099887216e+26, -1.4137791875898399e+26, -1.5222611440371296e+26, -1.634061377847883e+26, -1.749991421515918e+26, -1.8743660844021724e+26, -2.003175459572652e+26, -2.138572408449997e+26, -2.280933704934637e+26, -2.4308403355740592e+26, -2.589110466022332e+26, -2.755902727255918e+26, -2.92991123995182e+26, -3.1090742995804064e+26, -3.297925991603868e+26, -3.493973367210524e+26, -3.698747858523436e+26, -3.9081108451726546e+26, -4.1245844734363105e+26, -4.3463587003245406e+26, -4.573925985141102e+26, -4.810709609193257e+26, -5.054779802597869e+26, -5.305804893635386e+26, -5.567129786318441e+26, -5.8357156164184766e+26, -6.1130435174579385e+26, -6.399323623045025e+26, -6.695835520631705e+26, -6.999302141564101e+26, -7.310175154533259e+26, -7.632882733276257e+26, -7.964000450073021e+26, -8.300348640692607e+26, -8.646149475065852e+26, -9.003349636954381e+26, -9.371022897587815e+26, -9.746641922115406e+26, -1.0129859837000244e+27, -1.0527529345280356e+27, -1.0939543462582958e+27, -1.136310902053968e+27, -1.1799147063768655e+27, -1.2252035207782253e+27, -1.2720304024932865e+27, -1.32031209621826e+27, -1.3699907975365856e+27, -1.4212618575947083e+27, -1.474318521285759e+27, -1.5293128153253775e+27, -1.586395081406244e+27, -1.6445988371591954e+27, -1.7042533792507876e+27, -1.7659137355686288e+27, -1.8292559658639244e+27, -1.8946719358947294e+27, -1.9616015058902128e+27, -2.030805830207792e+27, -2.1027005790533815e+27, -2.1770320581308015e+27, -2.253817023328047e+27, -2.3329790730589475e+27, -2.415197216860788e+27, -2.5004786189731813e+27, -2.5888173830694583e+27, -2.680315314961818e+27, -2.7745913578700977e+27, -2.871588703962467e+27, -2.9718478801589245e+27, -3.075082945471697e+27, -3.181730430560424e+27, -3.291508388534905e+27, -3.4046602250802547e+27, -3.5215610000885987e+27, -3.64229611699024e+27, -3.767213007492544e+27, -3.8971101185074904e+27, -4.0310264320857504e+27, -4.1696797253053966e+27, -4.3132714978678404e+27, -4.4619350207722007e+27, -4.615988395571094e+27, -4.774998633351074e+27, -4.939459096441175e+27, -5.109825918935175e+27, -5.285156792817922e+27, -5.466479737272082e+27, -5.653767015757679e+27, -5.847553947281604e+27, -6.048224080572598e+27, -6.255576311881218e+27, -6.46981573634023e+27, -6.691534510383043e+27, -6.921721839520644e+27, -7.160077211808575e+27, -7.406305215671609e+27, -7.660532753076849e+27, -7.923745685859132e+27, -8.196254317450434e+27, -8.477829776225569e+27, -8.768631944930429e+27, -9.069385205662543e+27, -9.380905522752167e+27, -9.703222199876731e+27, -1.0035375490213794e+28, -1.0378535719534582e+28, -1.073322609973271e+28, -1.110024175105575e+28, -1.1479887998147022e+28, -1.1872846652464744e+28, -1.2278704983406744e+28, -1.2699087297204839e+28, -1.3134762533048912e+28, -1.358471509993557e+28, -1.404946185593277e+28, -1.4530489808298384e+28, -1.5027999492854187e+28, -1.5542585598132276e+28, -1.6074615283979784e+28, -1.6625574078701303e+28, -1.7194153846997535e+28, -1.7781407889022852e+28, -1.8388776737547566e+28, -1.9017093474861914e+28, -1.966748271478857e+28, -2.0340016512684824e+28, -2.10356854870338e+28, -2.17554287165026e+28, -2.250026024394274e+28, -2.327008229783596e+28, -2.4065472144787334e+28, -2.4888041246874407e+28, -2.573871111285254e+28, -2.6618620536525735e+28, -2.7528410143998992e+28, -2.8468158864908655e+28, -2.9440414938452533e+28, -3.0446857806522873e+28, -3.1486459341184562e+28, -3.2561377944523716e+28, -3.367311567955525e+28, -3.4822955265834146e+28, -3.601222802807478e+28, -3.72409821665791e+28, -3.851147304517944e+28, -3.982664782123625e+28, -4.118588632984821e+28, -4.2591617531687526e+28, -4.404461608297246e+28, -4.554666729609524e+28, -4.710111774208656e+28, -4.8707885670861095e+28, -5.0369899264810325e+28, -5.208814707835933e+28, -5.386522747194699e+28, -5.570342577455046e+28, -5.760371181728031e+28, -5.956824980629274e+28, -6.1599592735155755e+28, -6.370067503365889e+28, -6.587388883517352e+28, -6.812064677968903e+28, -7.044364329057432e+28, -7.284587161856308e+28, -7.532966621658994e+28, -7.789769419852452e+28, -8.055336089557858e+28, -8.329943462019554e+28, -8.613940254242948e+28, -8.907583747045985e+28, -9.211243464173423e+28, -9.52519302225455e+28, -9.849938865227543e+28, -1.0185731057170831e+29, -1.0532959789066794e+29, -1.0892030848134884e+29, -1.1263315804727852e+29, -1.1647230090404055e+29, -1.2044234479588745e+29, -1.2454739739683377e+29, -1.2879242217685868e+29, -1.3318147625044781e+29, -1.3772018400364948e+29, -1.4241294160444287e+29, -1.4726593726049329e+29, -1.5228459389867448e+29, -1.5747440911066813e+29, -1.6284104342210672e+29, -1.6839026606633073e+29, -1.741287675636458e+29, -1.8006255059425144e+29, -1.861980936611851e+29, -1.9254287556252626e+29, -1.9910369097685764e+29, -2.058881036376155e+29, -2.129041580382879e+29, -2.2015879111627855e+29, -2.2766055971880927e+29, -2.3541830456271346e+29, -2.4343973683857532e+29, -2.5173410306074358e+29, -2.603110415920693e+29, -2.6918089295943488e+29, -2.7835286289131313e+29, -2.8783656237627963e+29, -2.976435486510439e+29, -3.07784545499938e+29, -3.182716878073989e+29, -3.291160788945359e+29, -3.403291519679848e+29, -3.519244119666572e+29, -3.639151902426055e+29, -3.763143034934279e+29, -3.891352358439327e+29, -4.0239301162635334e+29, -4.1610248996542146e+29, -4.302793598102148e+29, -4.449391316455053e+29, -4.600976723930711e+29, -4.7577271901447035e+29, -4.9198190953971665e+29, -5.087435472155072e+29, -5.2607634232479455e+29, -5.439995135836648e+29, -5.625332533914699e+29, -5.816979313145269e+29, -6.01515521863239e+29, -6.220086809838959e+29, -6.431996331438548e+29, -6.65112763243501e+29, -6.877726155262833e+29, -7.1120398596478e+29, -7.354339368445867e+29, -7.604890810532687e+29, -7.863977670874653e+29, -8.131890066072565e+29, -8.408929111819319e+29, -8.69540832655827e+29, -8.991643904127877e+29, -9.297971115700592e+29, -9.61473634080846e+29, -9.942290952698544e+29, -1.0281007608947451e+30, -1.0631259413561609e+30, -1.0993448535810992e+30, -1.136797196403063e+30, -1.1755257784551764e+30, -1.2155737127659757e+30, -1.2569855106751873e+30, -1.299808205584399e+30, -1.3440899181603443e+30, -1.3898801235953472e+30, -1.4372303512097067e+30, -1.4861932494833916e+30, -1.536824747371639e+30, -1.5891807377653993e+30, -1.6433204747355662e+30, -1.6993046069365692e+30, -1.7571960752736463e+30, -1.8170596891475376e+30, -1.8789629875250657e+30, -1.9429745929203462e+30, -2.0091672433165591e+30, -2.077614718248583e+30, -2.1483942207356023e+30, -2.2215850246398058e+30, -2.2972690377136392e+30, -2.375531414272599e+30, -2.4564601659713556e+30, -2.5401455497649853e+30, -2.6266821358987406e+30, -2.7161667849781634e+30, -2.80870000311383e+30, -2.904385531335006e+30, -3.003330686909365e+30, -3.105646703199743e+30, -3.211448358773634e+30, -3.320854491039367e+30, -3.433987812750916e+30, -3.550975371498365e+30, -3.671948515913516e+30, -3.797042677828416e+30, -3.926398514086656e+30, -4.0601610718677356e+30, -4.198480634324251e+30, -4.341512490769132e+30, -4.48941704035358e+30, -4.642360195341172e+30, -4.8005137766969726e+30, -4.9640552198062597e+30, -5.13316827837389e+30, -5.30804255496825e+30, -5.488874304762152e+30, -5.675866564292256e+30, -5.869229113411849e+30, -6.06917904709596e+30, -6.275940902530073e+30, -6.489746411924109e+30, -6.71083578054491e+30, -6.939457283949146e+30, -7.175867373690202e+30, -7.420331193657798e+30, -7.67312335278402e+30, -7.934527482814003e+30, -8.204836922244426e+30, -8.484355295124392e+30, -8.77339604861117e+30, -9.072283785101907e+30, -9.38135382501121e+30, -9.700953070404376e+30, -1.0031440307661993e+31, -1.0373186515193437e+31, -1.072657507375819e+31, -1.1092002734535034e+31, -1.1469879547415277e+31, -1.1860629769630989e+31, -1.2264691829965094e+31, -1.2682519282809895e+31, -1.3114581026930715e+31, -1.356136211849299e+31, -1.4023363888806379e+31, -1.4501104918144469e+31, -1.4995121328968883e+31, -1.5505967721660214e+31, -1.6034217326638768e+31, -1.658046305557967e+31, -1.7145318068302885e+31, -1.7729416236669313e+31, -1.8333413111677597e+31, -1.895798669879203e+31, -1.9603837886278224e+31, -2.0271691653224066e+31, -2.0962297448067185e+31, -2.1676430454065455e+31, -2.2414892140659734e+31, -2.3178511331941817e+31, -2.39681451403243e+31, -2.478467966708777e+31, -2.5629031460921084e+31, -2.650214810953397e+31, -2.7405009670672466e+31, -2.8338629313581476e+31, -2.930405498932923e+31, -3.0302370211282373e+31, -3.133469541785955e+31, -3.24021892276228e+31, -3.350604974522468e+31, -3.464751590127703e+31, -3.582786876636664e+31, -3.704843315556656e+31, -3.831057884954647e+31, -3.9615722519185543e+31, -4.096532892668365e+31, -4.236091272562663e+31, -4.380404032276181e+31, -4.529633130449421e+31, -4.683946058969163e+31, -4.843516000587468e+31, -5.008522051518505e+31, -5.179149396112837e+31, -5.355589537796386e+31, -5.538040499462501e+31, -5.726707049228727e+31, -5.921800929078678e+31, -6.123541098973622e+31, -6.332153975489702e+31, -6.54787368379837e+31, -6.7709423351341425e+31, -7.0016102778602505e+31, -7.2401363911154065e+31, -7.486788378503994e+31, -7.74184305530104e+31, -8.005586670755485e+31, -8.278315222936783e+31, -8.560334795797604e+31, -8.851961898835834e+31, -9.15352382158241e+31, -9.465359002029481e+31, -9.78781740846877e+31, -1.0121260932239384e+32, -1.046606378959722e+32, -1.082261294510012e+32, -1.119130854324231e+32, -1.1572564360211463e+32, -1.1966808266891872e+32, -1.2374482706772557e+32, -1.2796045200540347e+32, -1.3231968843545108e+32, -1.3682742851752604e+32, -1.4148873107063012e+32, -1.46308827189241e+32, -1.5129312620245733e+32, -1.5644722163385276e+32, -1.6177689752547106e+32, -1.672881349937783e+32, -1.729871188406098e+32, -1.7888024453178878e+32, -1.8497412538742394e+32, -1.912755999812698e+32, -1.9779173977412927e+32, -2.045298570938826e+32, -2.1149751336501538e+32, -2.187025275189155e+32, -2.261529848034787e+32, -2.3385724584344917e+32, -2.4182395602106765e+32, -2.500620551639506e+32, -2.5858078756791897e+32, -2.673897123677785e+32, -2.764987142841694e+32, -2.859180146133741e+32, -2.9565818285714103e+32, -3.0573014843488052e+32, -3.1614521292535752e+32, -3.269150628500639e+32, -3.3805178270879145e+32, -3.495678685194233e+32, -3.614762418697264e+32, -3.7379026430869595e+32, -3.8652375245447656e+32, -3.996909934166996e+32, -4.133067607474431e+32, -4.273863311568383e+32, -4.419455015211292e+32, -4.570006065451597e+32, -4.725685371951281e+32, -4.886667595498354e+32, -5.053133343720895e+32, -5.225269373329171e+32, -5.403268800388525e+32, -5.5873313154726506e+32, -5.777663406977979e+32, -5.9744785944460165e+32, -6.1779976655942876e+32, -6.388448925299354e+32, -6.606068449713402e+32, -6.831100351380526e+32, -7.063797053961651e+32, -7.304419572386329e+32, -7.553237805756817e+32, -7.810530840551661e+32, -8.076587262536981e+32, -8.35170547937044e+32, -8.636194054675894e+32, -8.930372053363391e+32, -9.234569398464004e+32, -9.549127241014329e+32, -9.874398340797469e+32, -1.0210747460950129e+33, -1.0558551776141485e+33, -1.0918201294382806e+33, -1.129009929269812e+33, -1.167466276838676e+33, -1.2072322904528087e+33, -1.2483525551899794e+33, -1.290873172786569e+33, -1.3348418130773535e+33, -1.3803077671305838e+33, -1.4273220025043393e+33, -1.475937220020602e+33, -1.5262079125659885e+33, -1.5781904259829573e+33, -1.6319430218664888e+33, -1.6875259426103213e+33, -1.745001478562264e+33, -1.8044340374045362e+33, -1.8658902161203302e+33, -1.9294388749430104e+33, -1.995151214280566e+33, -2.063100853947342e+33, -2.1333639150635974e+33, -2.2060191047592838e+33, -2.2811478038267642e+33, -2.3588341571635497e+33, -2.4391651673594457e+33, -2.5222307912385195e+33, -2.608124039980227e+33, -2.6969410821931656e+33, -2.788781350758444e+33, -2.883747653098878e+33, -2.9819462850633204e+33, -3.0834871488858205e+33, -3.18848387473037e+33, -3.2970539464802656e+33, -3.4093188317761095e+33, -3.525404116207651e+33, -3.645439642090406e+33, -3.76955965173487e+33, -3.897902935552933e+33, -4.0306129850626606e+33, -4.1678381508743335e+33, -4.3097318058914274e+33, -4.456452513956966e+33, -4.608164203979083e+33, -4.765036349875685e+33, -4.927244156133443e+33, -5.094968749704632e+33, -5.268397378175187e+33, -5.447723614089214e+33, -5.633147566081366e+33, -5.824876096914455e+33, -6.023123048454935e+33, -6.228109473766466e+33, -6.440063876799449e+33, -6.659222459789968e+33, -6.885829378262282e+33, -7.120137004326903e+33, -7.362406198321946e+33, -7.612906588684668e+33, -7.871916860954318e+33, -8.139725055505468e+33, -8.416628874568127e+33, -8.702935998624081e+33, -8.998964412593426e+33, -9.305042741800485e+33, -9.621510598077307e+33, -9.948718936253916e+33, -1.0287030421131229e+34, -1.063681980518493e+34, -1.099847431748903e+34, -1.1372394063495484e+34, -1.1758992436356436e+34, -1.2158696539701695e+34, -1.2571947622245866e+34, -1.299920152398764e+34, -1.3440929134622868e+34, -1.3897616863850355e+34, -1.4369767123891038e+34, -1.485789882423604e+34, -1.536254787837308e+34, -1.5884267722999991e+34, -1.6423629849134454e+34, -1.6981224345103982e+34, -1.7557660451527548e+34, -1.8153567127771187e+34, -1.8769593629699977e+34, -1.9406410098057906e+34, -2.0064708157492955e+34, -2.0745201525274121e+34, -2.144862662899773e+34, -2.2175743232593767e+34, -2.2927335069890076e+34, -2.370421048417241e+34, -2.4507203072668696e+34, -2.533717233474625e+34, -2.6195004321696147e+34, -2.708161228649031e+34, -2.7997937331450723e+34, -2.8944949051077333e+34, -2.9923646167544223e+34, -3.09350571555547e+34, -3.198024085346095e+34, -3.306028705622848e+34, -3.417631708632819e+34, -3.532948433768452e+34, -3.6520974787049915e+34, -3.775200746680815e+34, -3.9023834893064584e+34, -4.0337743440741103e+34, -4.169505365845412e+34, -4.309712051326888e+34, -4.454533355609235e+34, -4.604111699608163e+34, -4.758592967236299e+34, -4.918126490943742e+34, -5.08286502417137e+34, -5.252964699122915e+34, -5.428584968064699e+34, -5.609888526249736e+34, -5.797041214345158e+34, -5.990211898101709e+34, -6.189572322700493e+34, -6.395296939132468e+34, -6.607562699630842e+34, -6.826548818972213e+34, -7.052436498186367e+34, -7.2854086070109015e+34, -7.525649321049049e+34, -7.773343709396722e+34, -8.028677268145275e+34, -8.291835394928013e+34, -8.563002799311607e+34, -8.842362843643488e+34, -9.13009680858077e+34, -9.426383077379909e+34, -9.731396232680497e+34, -1.0045306059455719e+35, -1.0368276447563169e+35, -1.0700464187310076e+35, -1.1042017651490755e+35, -1.1393075357435865e+35, -1.1753764402897569e+35, -1.2124198770038826e+35, -1.2504477492368728e+35, -1.2894682680349364e+35, -1.3294877402468803e+35, -1.3705103420057081e+35, -1.4125378775814533e+35, -1.4555695238299218e+35, -1.4996015607226843e+35, -1.5446270887562041e+35, -1.5906357344239285e+35, -1.6376133453556788e+35, -1.6855416772459135e+35, -1.73439807525741e+35, -1.7841551532353248e+35, -1.834780474790169e+35, -1.8862362410970982e+35, -1.938478991114506e+35, -1.9914593208399867e+35, -2.0451216291649775e+35, -2.099403898861146e+35, -2.1542375221730994e+35, -2.209547181383219e+35, -2.265250795495081e+35, -2.3212595447796596e+35, -2.3774779852766027e+35, -2.4338042653454227e+35, -2.4901304559061313e+35, -2.5463430049902493e+35, -2.602323325512225e+35, -2.657948522638173e+35, -2.7130922636663783e+35, -2.767625788827542e+35, -2.821419055802495e+35, -2.8743420040315512e+35, -2.9262659170990723e+35, -2.9770648528080125e+35, -3.0266171012874156e+35, -3.074806622062632e+35, -3.1215244021098863e+35, -3.166669669328472e+35, -3.2101508906197866e+35, -3.251886482057851e+35, -3.2918051617318822e+35, -3.3298458850476646e+35, -3.365957318694955e+35, -3.400096833950556e+35, -3.432229032676982e+35, -3.462323859741617e+35, -3.490354401916702e+35, -3.516294522785365e+35, -3.5401165315441734e+35, -3.5617891254544024e+35, -3.58127587470865e+35, -3.5985345279664785e+35, -3.613517400451735e+35, -3.6261730593682656e+35, -3.636449440967127e+35, -3.644298420672453e+35, -3.649681717263322e+35, -3.652577853524794e+35, -3.652989732661959e+35, -3.650952239083926e+35, -3.646539152810957e+35, -3.639868597670157e+35, -3.631106241279357e+35, -3.620465541668847e+35, -3.608204496670818e+35, -3.594618594991175e+35, -3.580029979925734e+35, -3.564773196390392e+35, -3.549178269123378e+35, -3.533552217834595e+35, -3.518160413202385e+35, -3.5032093755461205e+35, -3.48883267989535e+35, -3.47508153063422e+35, -3.461921293776154e+35, -3.449234831220691e+35, -3.4368328958085474e+35, -3.424471166137233e+35, -3.411872791717009e+35, -3.398754660171902e+35, -3.384855070708199e+35, -3.36996017713151e+35, -3.353926506959181e+35, -3.3366971010750076e+35, -3.3183093476943926e+35, -3.2988933655500774e+35, -3.278660750761476e+35, -3.2578845394202854e+35, -3.2368722371733026e+35, -3.2159346090914615e+35, -3.1953535009224613e+35, -3.175352195022123e+35, -3.1560716462625515e+35, -3.137555394626201e+35, -3.1197450583407287e+35, -3.1024871643668738e+35, -3.0855507970777504e+35, -3.0686542881497555e+35, -3.0514980825792385e+35, -3.0338001345455237e+35, -3.0153298170288997e+35, -2.9959364274783965e+35, -2.975568938481488e+35, -2.9542846188241004e+35, -2.93224542569572e+35, -2.9097024930981254e+35, -2.886970444204667e+35, -2.8643944665262137e+35, -2.8423139600480302e+35, -2.8210269918343127e+35, -2.8007597111364462e+35, -2.7816443021845613e+35, -2.7637080429244437e+35, -2.746874714760521e+35, -2.730978127746409e+35, -2.715786065279397e+35, -2.701031689852515e+35, -2.6864485429444594e+35, -2.6718048327838333e+35, -2.6569327922255726e+35, -2.6417494987428482e+35, -2.626266607125917e+35, -2.6105878226294164e+35, -2.5948944662141408e+35, -2.5794209631508498e+35, -2.56442333610946e+35, -2.5501446482014654e+35, -2.536781713812556e+35, -2.5244572311437096e+35, -2.5132008128442095e+35, -2.5029412859958024e+35, -2.4935112386005857e+35, -2.48466328055584e+35, -2.4760960508405275e+35, -2.467486817802115e+35, -2.4585267324241634e+35, -2.448954500516318e+35, -2.438584470553904e+35, -2.4273258536785637e+35, -2.4151909035156145e+35, -2.402291240603873e+35, -2.3888229363923107e+35, -2.375042297670987e+35, -2.361235355129981e+35, -2.3476847382200565e+35, -2.334637842527621e+35, -2.3222799525645123e+35, -2.3107153157033475e+35, -2.2999581635953868e+35, -2.2899344727586034e+35, -2.2804939918965096e+35, -2.2714308873498304e+35, -2.2625104025478366e+35, -2.2534982964161166e+35, -2.24418958403624e+35, -2.234433269192312e+35, -2.224150303905145e+35, -2.213342861770274e+35, -2.202094062069189e+35, -2.190558401413776e+35, -2.1789442060039084e+35, -2.1674902902377564e+35, -2.1564396036324065e+35, -2.1460129121835583e+35, -2.1363854778005896e+35, -2.1276692945975894e+35, -2.1199027704371673e+35, -2.113048886288794e+35, -2.1070019181801414e+35, -2.1016018641929562e+35, -2.096654875523415e+35, -2.091957328820724e+35, -2.0873207634879932e+35, -2.0825947878852523e+35, -2.0776852521599994e+35, -2.0725654827764878e+35, -2.0672791326833118e+35, -2.0619341479187343e+35, -2.056688386243623e+35, -2.0517284288659834e+35, -2.0472439816916515e+35, -2.0434008592669307e+35, -2.040315801615582e+35, -2.0380362503455925e+35, -2.0365277108279418e+35, -2.0356705034528163e+35, -2.0352666500933055e+35, -2.0350564712191297e+35, -2.0347433176013168e+35, -2.034023860386963e+35, -2.0326206326161055e+35, -2.0303131466928004e+35, -2.0269639636465938e+35, -2.0225365757493357e+35, -2.0171028495039005e+35, -2.0108389752592198e+35, -2.0040102500371675e+35, -1.996946414784893e+35, -1.990010496011251e+35, -1.983564996570779e+35, -1.9779397112457914e+35, -1.973405338564308e+35, -1.9701564201368683e+35, -1.9683060318524364e+35, -1.9678932049914535e+35, -1.968902434521355e+35, -1.9712930142626194e+35, -1.9750344936528476e+35, -1.9801434279517665e+35, -1.986715922897898e+35, -1.994950373938586e+35, -2.0051553775626564e+35, -2.0177391305725533e+35, -2.0331787474152194e+35, -2.051970704855929e+35, -2.074566780280597e+35, -2.101302914644733e+35, -2.132330810828119e+35, -2.1675631833208034e+35, -2.206642976852427e+35, -2.2489444363691464e+35, -2.2936098644389296e+35, -2.339620800741081e+35, -2.385896981387769e+35, -2.4314116387156144e+35, -2.475308267218202e+35, -2.517002496254855e+35, -2.556253489906182e+35, -2.593192351481385e+35, -2.628300072703231e+35, -2.6623341258967515e+35, -2.6962101512282586e+35, -2.730852471402867e+35, -2.7670333429149642e+35, -2.805224778465346e+35, -2.8454873384332612e+35, -2.887416706531403e+35, -2.9301610409184045e+35, -2.9725108952537357e+35, -3.013050780616923e+35, -3.050349689430534e+35, -3.0831596909830643e+35, -3.11058900035251e+35, -3.1322195654268388e+35, -3.1481487078906695e+35, -3.158947939237078e+35, -3.1655471231328646e+35, -3.169065723784208e+35, -3.170622308596921e+35, -3.1711569241420074e+35, -3.1712977707850617e+35, -3.1712943963983397e+35, -3.1710262442902336e+35, -3.1700805341571246e+35, -3.1678801936228755e+35, -3.163833651099522e+35, -3.1574755397008022e+35, -3.1485711205572323e+35, -3.137166399040364e+35, -3.1235782143747697e+35, -3.1083312209610023e+35, -3.0920589621738674e+35, -3.0753921692515243e+35, -3.0588580617896804e+35, -3.0428100305843175e+35, -3.0273989606000405e+35, -3.012587634432731e+35, -2.9982004057208316e+35, -2.983993613183225e+35, -2.969729262142396e+35, -2.95523562302053e+35, -2.9404429457498142e+35, -2.9253891411240045e+35, -2.9101974067570245e+35, -2.8950338335862318e+35, -2.8800568581329114e+35, -2.865371414328448e+35, -2.8509987755982753e+35, -2.8368689155721555e+35, -2.8228367516397307e+35, -2.808718109285879e+35, -2.79433687075685e+35, -2.779572461335293e+35, -2.7643969861948937e+35, -2.7488937903611795e+35, -2.733253309505045e+35, -2.7177468486148086e+35, -2.7026833519134024e+35, -2.6883574558384715e+35, -2.6749986028176545e+35, -2.6627305680348953e+35, -2.6515486129760336e+35, -2.6413181275931437e+35, -2.63179474626323e+35, -2.622662258885274e+35, -2.6135818272780406e+35, -2.604244481349606e+35, -2.5944187544433237e+35, -2.5839865009058284e+35, -2.5729621017314692e+35, -2.5614929827221425e+35, -2.5498422011787542e+35, -2.5383563960696023e+35, -2.527424307325693e+35, -2.5174321117201923e+35, -2.508721877004284e+35, -2.501558531330805e+35, -2.496109065834088e+35, -2.492435550029239e+35, -2.490501330001949e+35, -2.4901878768927402e+35, -2.491318449036091e+35, -2.493684175574905e+35, -2.4970683575814292e+35, -2.501265579893841e+35, -2.5060934201051666e+35, -2.511395898961458e+35, -2.5170391370836242e+35, -2.5229008192411203e+35, -2.5288559264388797e+35, -2.5347617220244144e+35, -2.5404451327282656e+35, -2.5456954200518657e+35, -2.550264380827571e+35, -2.5538752778770705e+35, -2.556240376539345e+35, -2.5570855185468804e+35, -2.556178828673897e+35, -2.553359667963483e+35, -2.54856352955784e+35, -2.5418388376318777e+35, -2.533352549760978e+35, -2.5233829419639882e+35, -2.512299735903629e+35, -2.50053352001361e+35, -2.4885379375197136e+35, -2.4767491377379404e+35, -2.4655473749479397e+35, -2.4552253547757777e+35, -2.445967029172569e+35, -2.4378391622522057e+35, -2.4307963184871716e+35, -2.4246981801348126e+35, -2.4193365078622213e+35, -2.414467828256541e+35, -2.4098472372437985e+35, -2.405258660186811e+35, -2.400537533934524e+35, -2.3955831008352012e+35, -2.390359158237341e+35, -2.3848839377464043e+35, -2.379211502612598e+35, -2.373408363703495e+35, -2.367529701097506e+35, -2.361599521611302e+35, -2.3555982943327704e+35, -2.349460225954848e+35, -2.3430806062017625e+35, -2.3363318688364152e+35, -2.3290854785059544e+35, -2.321235726216632e+35, -2.3127211696212643e+35, -2.303539849655392e+35, -2.293755489670387e+35, -2.2834934558091553e+35, -2.2729270524089692e+35, -2.2622564184835705e+35, -2.2516835665933147e+35, -2.241387726699248e+35, -2.2315050197638295e+35, -2.2221156409803165e+35, -2.2132403753022623e+35, -2.204846683784719e+35, -2.1968630905168746e+35, -2.1891994153093333e+35, -2.181769685304222e+35, -2.1745143570695507e+35, -2.1674187380268917e+35, -2.1605251101126363e+35, -2.153936913901065e+35, -2.1478143415916088e+35, -2.1423617186685236e+35, -2.1378080373281738e+35, -2.134382845711197e+35, -2.132290297102107e+35, -2.1316844340242297e+35, -2.132648668670843e+35, -2.135181924027932e+35, -2.1391930854460482e+35, -2.1445044023384812e+35, -2.150863425944042e+35, -2.157962119314616e+35, -2.1654610436663584e+35, -2.1730160751601072e+35, -2.180304953500025e+35, -2.1870510876657773e+35, -2.1930424040218825e+35, -2.198143569036797e+35, -2.2023005997938356e+35, -2.2055376311799184e+35, -2.207946371666481e+35, -2.2096694766556967e+35, -2.2108796287785483e+35, -2.21175648252563e+35, -2.2124637759206974e+35, -2.2131288343427876e+35, -2.2138264170898365e+35, -2.2145684279367365e+35, -2.215300472945017e+35, -2.2159056435669457e+35, -2.2162152650870248e+35, -2.2160257116833122e+35, -2.2151197876005018e+35, -2.213290659229595e+35, -2.2103659577976292e+35, -2.206229522764696e+35, -2.2008383745052136e+35, -2.1942329117161306e+35, -2.186538998596945e+35, -2.177961465217218e+35, -2.1687694806861287e+35, -2.159275147190667e+35, -2.149807389204435e+35, -2.1406836940442807e+35, -2.1321824582118896e+35, -2.12451861067009e+35, -2.117824852348609e+35, -2.1121403207050968e+35, -2.1074078146318454e+35, -2.1034799533871858e+35, -2.1001338475303767e+35, -2.0970930856269194e+35, -2.0940551471192125e+35, -2.0907218013165095e+35, -2.0868297041470348e+35, -2.082178305729432e+35, -2.0766523590777692e+35, -2.07023677072614e+35, -2.0630222233293607e+35, -2.0552008646186133e+35, -2.047052310341954e+35, -2.038921152762078e+35, -2.0311880021377397e+35, -2.0242367287996594e+35, -2.018420951864147e+35, -2.0140329003403455e+35, -2.011277549264909e+35, -2.01025443530735e+35, -2.0109488373854302e+35, -2.013233141118413e+35, -2.0168782737777694e+35, -2.021574182553901e+35, -2.0269575120839658e+35, -2.0326439862012982e+35, -2.038262568874176e+35, -2.0434883085881265e+35, -2.0480708771909135e+35, -2.051856194435699e+35, -2.0547991552766554e+35, -2.0569662967995387e+35, -2.0585281821934445e+35, -2.0597422499245025e+35, -2.0609277779674557e+35, -2.062435348578916e+35, -2.0646136875211146e+35, -2.0677769402844922e+35, -2.0721753217815894e+35, -2.077971660629548e+35, -2.0852257156175076e+35, -2.0938873560260328e+35, -2.1037988649460517e+35, -2.114705836476139e+35, -2.126275467379377e+35, -2.138120540255062e+35, -2.149827080103881e+35, -2.1609835362355826e+35, -2.1712093755954964e+35, -2.1801811411105766e+35, -2.187654298020346e+35, -2.193479535669538e+35, -2.19761259137139e+35, -2.2001170992505204e+35, -2.2011604203717264e+35, -2.2010028540256827e+35, -2.1999810303676605e+35, -2.198486607889747e+35, -2.1969416212560477e+35, -2.195771939921967e+35, -2.195380321144262e+35, -2.1961205038408155e+35, -2.1982737275713662e+35, -2.2020289983064276e+35, -2.207468362816822e+35, -2.2145583745625233e+35, -2.2231487942149315e+35, -2.232979317813253e+35, -2.2436947227646567e+35, -2.2548682461120193e+35, -2.2660322740677093e+35, -2.2767145802989476e+35, -2.2864774971701974e+35, -2.2949566669974758e+35, -2.301895544575064e+35, -2.307171745530109e+35, -2.310811756725187e+35, -2.312991475667155e+35, -2.314021466925381e+35, -2.3143175632547924e+35, -2.314359270607073e+35, -2.31464009364207e+35, -2.3156151235030333e+35, -2.3176518176793634e+35, -2.3209897374003597e+35, -2.325714085124048e+35, -2.3317463085197716e+35, -2.3388530089975028e+35, -2.346672179186852e+35, -2.354753687128108e+35, -2.3626091991396918e+35, -2.3697656054341506e+35, -2.3758156169391092e+35, -2.3804595806243892e+35, -2.383533667708126e+35, -2.385021299917157e+35, -2.3850468009872266e+35, -2.38385254279725e+35, -2.381763003436062e+35, -2.3791408596801525e+35, -2.376341224265668e+35, -2.3736702274145075e+35, -2.3713532961447617e+35, -2.3695168378415153e+35, -2.368184873424116e+35, -2.3672898728357617e+35, -2.3666950165660375e+35, -2.3662236641597305e+35, -2.3656911417782196e+35, -2.3649340933008244e+35, -2.363833457131252e+35, -2.362328421849229e+35, -2.360420230138862e+35, -2.3581662087030258e+35, -2.355665712543639e+35, -2.3530406465863276e+35, -2.350413775610848e+35, -2.3478881111939266e+35, -2.345530281520344e+35, -2.3433600182937135e+35, -2.3413468684154425e+35, -2.339414134569204e+35, -2.337449055376286e+35, -2.3353175058673074e+35, -2.3328811177461495e+35, -2.330014688078498e+35, -2.3266219933667987e+35, -2.3226485402313935e+35, -2.3180902490361343e+35, -2.3129975001479844e+35, -2.307474340303582e+35, -2.3016729582895497e+35, -2.295783825449501e+35, -2.29002218240481e+35, -2.284611839613117e+35, -2.2797675190986966e+35, -2.275677154426068e+35, -2.2724856431310796e+35, -2.2702814841856866e+35, -2.2690875295920617e+35, -2.268856752106287e+35, -2.269473512105254e+35, -2.2707603334651698e+35, -2.2724897099257e+35, -2.2743999993814245e+35, -2.276214065106763e+35, -2.277659032822086e+35, -2.2784853914174738e+35, -2.2784837041661433e+35, -2.2774974287793326e+35, -2.2754307544428976e+35, -2.2722509080214808e+35, -2.267984990053907e+35, -2.262711989131027e+35, -2.256551106820458e+35, -2.2496478381257414e+35, -2.2421593590636803e+35, -2.2342406747793815e+35, -2.2260327148465267e+35, -2.2176531881278145e+35, -2.209190599519583e+35, -2.200701452553805e+35, -2.1922103659173803e+35, -2.1837126452235547e+35, -2.1751787749327334e+35, -2.166560308320745e+35, -2.1577967006308356e+35, -2.1488227121954578e+35, -2.1395760695018338e+35, -2.1300050909051415e+35, -2.1200759553845684e+35, -2.109779230531096e+35, -2.099135207066014e+35, -2.0881975456156235e+35, -2.077054759346841e+35, -2.0658291548859118e+35, -2.0546730384092464e+35, -2.0437622491985415e+35, -2.033287377015884e+35, -2.023443308229905e+35, -2.014417981176383e+35, -2.006381371826378e+35, -1.9994757484398735e+35, -1.9938081194279656e+35, -1.9894455639317308e+35, -1.9864138099147007e+35, -1.9846990532086602e+35, -1.9842526420031814e+35, -1.9849979317227233e+35, -1.9868383837635503e+35, -1.989665864489917e+35, -1.993368110878934e+35, -1.9978344658828137e+35, -2.0029592377981936e+35, -2.00864238071937e+35, -2.0147875935831456e+35, -2.0212983482520615e+35, -2.0280727273801375e+35, -2.0349982199043986e+35, -2.041947729170406e+35, -2.0487779557574157e+35, -2.0553310137030477e+35, -2.061439653826055e+35, -2.066935869786927e+35, -2.0716620502749674e+35, -2.075483324755268e+35, -2.0782994295193915e+35, -2.0800543614253468e+35, -2.08074230774153e+35, -2.080408810599715e+35, -2.0791467695395447e+35, -2.0770876039116985e+35, -2.0743885786056362e+35, -2.07121784151988e+35, -2.0677390529603882e+35, -2.0640975614433444e+35, -2.0604098886094298e+35, -2.0567578544364415e+35, -2.053188059978613e+35, -2.0497167296723337e+35, -2.04633919397371e+35, -2.0430426624294578e+35, -2.0398204819901753e+35, -2.0366858560799865e+35, -2.03368304442381e+35, -2.030894363621347e+35, -2.0284418220774847e+35, -2.026482882143958e+35, -2.025200563407878e+35, -2.0247887956414576e+35, -2.0254345155939412e+35, -2.0272984111465665e+35, -2.030496404014524e+35, -2.0350839099842288e+35, -2.0410446348611024e+35, -2.0482851941063657e+35, -2.0566362461593825e+35, -2.0658601781117595e+35, -2.0756647531616e+35, -2.08572158717123e+35, -2.0956879126668547e+35, -2.1052298359490075e+35, -2.1140451983536175e+35, -2.1218842021448223e+35, -2.128566133239331e+35, -2.1339907853850627e+35, -2.1381435476666507e+35, -2.1410935504143192e+35, -2.1429847683336006e+35, -2.14402054410301e+35, -2.1444425970388335e+35, -2.144506174632779e+35, -2.1444535220825578e+35, -2.1444882034991794e+35, -2.144752926234956e+35, -2.1453133364439466e+35, -2.1461497508216827e+35, -2.1471580011463062e+35, -2.1481595825644826e+35, -2.1489202409559118e+35, -2.1491751530498903e+35, -2.1486580790172482e+35, -2.1471314014734154e+35, -2.1444138595862374e+35, -2.1404030426398124e+35, -2.1350902780932656e+35, -2.128566355249772e+35, -2.1210174691162783e+35, -2.112711747767773e+35, -2.1039776456401214e+35, -2.095176263695724e+35, -2.0866702317541186e+35, -2.0787921132884198e+35, -2.0718153426153956e+35, -2.0659304728901997e+35, -2.0612290170355846e+35, -2.0576964424967683e+35, -2.055214997308011e+35, -2.0535760817043356e+35, -2.0525009313105417e+35, -2.0516675428660587e+35, -2.0507411417767155e+35, -2.0494051345297227e+35, -2.047389452962113e+35, -2.0444934930389482e+35, -2.040601453505889e+35, -2.035688729443056e+35, -2.0298190210485926e+35, -2.0231328643066567e+35, -2.0158292510356583e+35, -2.0081427575136186e+35, -2.000319040109273e+35, -1.992591617792245e+35, -1.985162532587633e+35, -1.9781888076789897e+35, -1.971775712857573e+35, -1.9659768422474493e+35, -1.9608000675645915e+35, -1.956217694010129e+35, -1.9521787168004494e+35, -1.948620999902262e+35, -1.945481461766783e+35, -1.942702892381983e+35, -1.9402367430661012e+35, -1.938042007896618e+35, -1.9360810339192035e+35, -1.9343136483912273e+35, -1.9326912913660802e+35, -1.9311528421351445e+35, -1.929623523747513e+35, -1.928017705433358e+35, -1.9262456896983228e+35, -1.924223795832196e+35, -1.9218863751801774e+35, -1.9191979423864516e+35, -1.9161634659149524e+35, -1.9128350547178683e+35, -1.9093137646584837e+35, -1.9057459330935245e+35, -1.902314207580646e+35, -1.8992241382307622e+35, -1.896687749296309e+35, -1.89490582987766e+35, -1.8940507655189172e+35, -1.894251590298329e+35, -1.8955826193171034e+35, -1.898056586209961e+35, -1.9016227248387473e+35, -1.9061697581999828e+35, -1.9115333382070676e+35, -1.91750714984451e+35, -1.9238566693419705e+35, -1.9303344519792625e+35, -1.936695813448941e+35, -1.942713844378277e+35, -1.9481928410607856e+35, -1.9529794252124596e+35, -1.956970840368477e+35, -1.9601201326950688e+35, -1.9624381325481233e+35, -1.963992336512275e+35, -1.9649029383528165e+35, -1.9653363661388106e+35, -1.9654967511135215e+35, -1.9656157855148186e+35, -1.965941429017352e+35, -1.966725907103923e+35, -1.968213420712688e+35, -1.9706279654125565e+35, -1.9741616476522555e+35, -1.9789638885384216e+35, -1.9851319201408765e+35, -1.9927029986460697e+35, -2.0016487721880508e+35, -2.0118722365429955e+35, -2.0232076772850388e+35, -2.035423923347078e+35, -2.0482311193154857e+35, -2.0612910622901866e+35, -2.0742309487162788e+35, -2.0866601464494654e+35, -2.098189360017701e+35, -2.1084513079215952e+35, -2.117121797750024e+35, -2.123939888657181e+35, -2.1287256952873098e+35, -2.1313943395813922e+35, -2.131964625355015e+35, -2.1305612211024232e+35, -2.12740950674735e+35, -2.122822771517952e+35, -2.1171821204018904e+35, -2.1109102039112197e+35, -2.1044406482865872e+35, -2.098185725474901e+35, -2.092505249860695e+35, -2.087679819563006e+35, -2.0838912672308035e+35, -2.0812125376795346e+35, -2.079608222969679e+35, -2.078945779035575e+35, -2.0790161873500357e+35, -2.079561693296699e+35, -2.0803074165717195e+35, -2.0809932082592756e+35, -2.081402178179557e+35, -2.0813828162304067e+35, -2.080862499420126e+35, -2.079851281368731e+35, -2.078436047852402e+35, -2.0767662341177387e+35, -2.0750332008908587e+35, -2.0734459567877505e+35, -2.072206143260177e+35, -2.071485064134296e+35, -2.0714050932604267e+35, -2.0720271168967395e+35, -2.073344870127205e+35, -2.0752862204652283e+35, -2.0777207344767734e+35, -2.0804723056219407e+35, -2.0833352612553016e+35, -2.086092208753002e+35, -2.0885319037542224e+35, -2.0904655895447515e+35, -2.0917405219289643e+35, -2.0922497179046443e+35, -2.0919373175173325e+35, -2.0907993058236645e+35, -2.0888796943248987e+35, -2.08626260134975e+35, -2.083060990100163e+35, -2.0794031066285263e+35, -2.0754178846861157e+35, -2.0712207200884578e+35, -2.0669010328609956e+35, -2.062512906977222e+35, -2.058069817502201e+35, -2.0535440390203373e+35, -2.048870818437796e+35, -2.0439568520697753e+35, -2.0386921056188162e+35, -2.032963628771392e+35, -2.0266698005654234e+35, -2.0197334273345625e+35, -2.0121122995416425e+35, -2.0038061635190777e+35, -1.9948595218752405e+35, -1.9853601728067226e+35, -1.97543386513614e+35, -1.965235825367183e+35, -1.9549401668994e+35, -1.944728302794727e+35, -1.9347774559180892e+35, -1.925250214956472e+35, -1.9162858553835296e+35, -1.9079938714187465e+35, -1.900449890303697e+35, -1.8936939013486363e+35, -1.8877305573491986e+35, -1.8825312100923247e+35, -1.878037324580084e+35, -1.874164963604793e+35, -1.870810119412428e+35, -1.8678547605033514e+35, -1.865173527982653e+35, -1.862641033316897e+35, -1.860139666211899e+35, -1.857567720806561e+35, -1.8548475080184924e+35, -1.851932970137538e+35, -1.848816184423727e+35, -1.8455320681231423e+35, -1.8421606034606044e+35, -1.8388260017066177e+35, -1.8356924204853093e+35, -1.8329561243049313e+35, -1.8308343089438425e+35, -1.8295511607714344e+35, -1.829322052319859e+35, -1.830337044893746e+35, -1.832745041774206e+35, -1.83663998485386e+35, -1.8420503997965545e+35, -1.848933372119872e+35, -1.857173696944276e+35, -1.866588520569159e+35, -1.8769373244318365e+35, -1.887936637974533e+35, -1.8992784519671528e+35, -1.910650977145323e+35, -1.921760183799703e+35, -1.9323504829335325e+35, -1.94222297244225e+35, -1.9512498636710526e+35, -1.9593840049988138e+35, -1.9666628017635512e+35, -1.97320626225927e+35, -1.9792093416485614e+35, -1.9849291742315503e+35, -1.9906681479809267e+35, -1.996754058200816e+35, -2.00351876185058e+35, -2.0112768313921977e+35, -2.020305676153579e+35, -2.030828467035874e+35, -2.0430009800773485e+35, -2.0569031838536376e+35, -2.0725360560580723e+35, -2.089823748664392e+35, -2.108620851932805e+35, -2.1287241575339273e+35, -2.1498880109846006e+35, -2.1718420919515478e+35, -2.194310283769514e+35, -2.217029203781401e+35, -2.239764973494824e+35, -2.2623269174358587e+35, -2.2845770919610064e+35, -2.3064348530578967e+35, -2.327876059818491e+35, -2.348926952905362e+35, -2.3696532109879607e+35, -2.3901451319303684e+35, -2.4105002654040765e+35, -2.4308050977399456e+35, -2.451117524479511e+35, -2.4714518208615476e+35, -2.4917676319988584e+35, -2.5119641673495916e+35, -2.531880329089842e+35, -2.551300973957766e+35, -2.5699689523457467e+35, -2.5876020366093667e+35, -2.6039133880282576e+35, -2.618633855881958e+35, -2.6315341805428747e+35, -2.6424451035182763e+35, -2.6512734803982564e+35, -2.658012748770327e+35, -2.662746514893073e+35, -2.665644572647635e+35, -2.666951325542449e+35, -2.666967301639318e+35, -2.6660251701445912e+35, -2.6644623113711e+35, -2.6625924764118937e+35, -2.6606793218186646e+35, -2.6589145601002618e+35, -2.6574031050321074e+35, -2.6561569315017724e+35, -2.655098478462054e+35, -2.654073404384888e+35, -2.652871483197349e+35, -2.6512535318883134e+35, -2.6489815959600163e+35, -2.6458492577994677e+35, -2.6417089063237665e+35, -2.636493104572909e+35, -2.6302277742862256e+35, -2.623035720341759e+35, -2.6151299675264805e+35, -2.606797393626085e+35, -2.598374126691221e+35, -2.5902150378267937e+35, -2.5826603134821034e+35, -2.5760024543459013e+35, -2.5704570673513088e+35, -2.5661404751364385e+35, -2.563056489767868e+35, -2.5610937550605914e+35, -2.5600339613462927e+35, -2.5595701072378938e+35, -2.5593329581610254e+35, -2.5589230499149082e+35, -2.557945095008745e+35, -2.5560415146851406e+35, -2.5529220370642174e+35, -2.5483868229552243e+35, -2.5423413217062015e+35, -2.534801916937396e+35, -2.525892291992736e+35, -2.515831238443722e+35, -2.5049132848193795e+35, -2.4934840030739157e+35, -2.481912148815589e+35, -2.4705609169096036e+35, -2.4597605629268086e+35, -2.449784468980918e+35, -2.440830432281806e+35, -2.433008537137731e+35, -2.426336451049788e+35, -2.4207423880602745e+35, -2.4160753470522454e+35, -2.4121216126014046e+35, -2.408625962667619e+35, -2.405315620929398e+35, -2.4019247695209396e+35, -2.3982174263107117e+35, -2.3940066894200798e+35, -2.38916873453248e+35, -2.3836504720343555e+35, -2.377470374301051e+35, -2.3707126095012653e+35, -2.3635152129086582e+35, -2.356053544161731e+35, -2.348520682579902e+35, -2.3411066739464148e+35, -2.3339786396853238e+35, -2.327263679691882e+35, -2.321036240859032e+35, -2.315311197489455e+35, -2.3100433289920025e+35, -2.305133235614919e+35, -2.300439071028992e+35, -2.2957928653538657e+35, -2.29101973415273e+35, -2.2859579737760297e+35, -2.280477963788622e+35, -2.2744979382837207e+35, -2.2679950281934362e+35, -2.2610104730692594e+35, -2.253648495582183e+35, -2.246068961176478e+35, -2.238474545979704e+35, -2.2310936521569888e+35, -2.2241606964566152e+35, -2.2178956234992208e+35, -2.2124845442591315e+35, -2.2080632715924486e+35, -2.2047052331121674e+35, -2.202414815820133e+35, -2.201126677230382e+35, -2.2007109939450107e+35, -2.2009840653097895e+35, -2.201723201376293e+35, -2.2026844496289647e+35, -2.203621491020061e+35, -2.204303983773513e+35, -2.2045337548241033e+35, -2.2041575155328737e+35, -2.2030751747382336e+35, -2.2012432898801917e+35, -2.1986736813266496e+35, -2.1954276828735913e+35, -2.1916068673516957e+35, -2.187341338230249e+35, -2.182776800055984e+35, -2.178061612985311e+35, -2.1733349147294284e+35, -2.168716683084752e+35, -2.1643003463962714e+35, -2.1601482618775968e+35, -2.156290103990813e+35, -2.1527239624146215e+35, -2.1494197588783e+35, -2.1463244633329054e+35, -2.1433685236926235e+35, -2.1404729145505945e+35, -2.137556249217919e+35, -2.1345414740546584e+35, -2.131361761567507e+35, -2.1279653270444075e+35, -2.1243190020664044e+35, -2.120410498741725e+35, -2.1162493848543852e+35, -2.1118668585811762e+35, -2.1073144603434673e+35, -2.1026618891064564e+35, -2.0979941031105957e+35, -2.09340788411051e+35, -2.0890080342044928e+35, -2.0849033599677325e+35, -2.0812025841499588e+35, -2.0780103135955474e+35, -2.0754231844806746e+35, -2.0735263015428434e+35, -2.0723900842058496e+35, -2.072067625968844e+35, -2.0725926608732496e+35, -2.073978210249924e+35, -2.0762159540805483e+35, -2.0792763361093563e+35, -2.083109373950504e+35, -2.0876461094156415e+35, -2.0928006045306156e+35, -2.098472368540635e+35, -2.1045490921777815e+35, -2.1109095671154406e+35, -2.1174266788487905e+35, -2.123970377099927e+35, -2.130410545989355e+35, -2.136619713861666e+35, -2.1424755579838697e+35, -2.1478631715461955e+35, -2.152677069626935e+35, -2.1568229176211908e+35, -2.1602189709058378e+35, -2.162797218920223e+35, -2.1645042308550996e+35, -2.1653017040355123e+35, -2.1651667199675194e+35, -2.164091716944326e+35, -2.16208419209145e+35, -2.159166149758105e+35, -2.15537331720272e+35, -2.1507541524093093e+35, -2.1453686724165684e+35, -2.139287133372506e+35, -2.1325885953725892e+35, -2.1253594056046014e+35, -2.1176916322270324e+35, -2.109681478687779e+35, -2.1014277039733227e+35, -2.0930300689959085e+35, -2.0845878235724414e+35, -2.0761982430157074e+35, -2.067955219142128e+35, -2.05994790829968e+35, -2.0522594394909802e+35, -2.044965689110137e+35, -2.038134135093788e+35, -2.0318228117548857e+35, -2.0260793961414922e+35, -2.0209404658904067e+35, -2.0164309755993898e+35, -2.012564001982527e+35, -2.0093408061807898e+35, -2.0067512536950344e+35, -2.0047746183553787e+35, -2.0033807771772508e+35, -2.0025317792555047e+35, -2.002183745931086e+35, -2.0022890335986862e+35, -2.0027985669751294e+35, -2.0036642314888638e+35, -2.0048412004142396e+35, -2.0062900668008793e+35, -2.0079786531285225e+35, -2.009883383677059e+35, -2.011990126137686e+35, -2.0142944397915096e+35, -2.0168012064896617e+35, -2.0195236653308146e+35, -2.022481918635826e+35, -2.025701020712538e+35, -2.0292087965675436e+35, -2.0330335602101996e+35, -2.0372019079740222e+35, -2.0417367501806124e+35, -2.0466557160082542e+35, -2.0519700255997545e+35, -2.057683875947426e+35, -2.0637943392561423e+35, -2.0702917300203927e+35, -2.0771603642732963e+35, -2.084379613663097e+35, -2.091925148539093e+35, -2.0997702666771317e+35, -2.1078872151793205e+35, -2.116248429428792e+35, -2.1248276317762152e+35, -2.1336007513774862e+35, -2.142546643436117e+35, -2.1516475999930463e+35, -2.1608896550189926e+35, -2.1702626942112533e+35, -2.179760385264207e+35, -2.18937994832805e+35, -2.199121789762331e+35, -2.2089890256569673e+35, -2.218986925222107e+35, -2.229122307845441e+35, -2.2394029309110515e+35, -2.249836907641071e+35, -2.260432194546022e+35, -2.271196185921287e+35, -2.282135447833337e+35, -2.293255616205382e+35, -2.304561473215013e+35, -2.3160572038848566e+35, -2.327746821297585e+35, -2.3396347353133535e+35, -2.3517264270476673e+35, -2.3640291808175136e+35, -2.376552817816418e+35, -2.3893103723922582e+35, -2.4023186531691036e+35, -2.4155986377472545e+35, -2.429175661214732e+35, -2.443079374582836e+35, -2.457343468341319e+35, -2.472005177000613e+35, -2.4871046008412436e+35, -2.5026838991158227e+35, -2.5187864228020503e+35, -2.5354558631439873e+35, -2.5527354936480924e+35, -2.5706675775221534e+35, -2.5892930001064115e+35, -2.6086511676214925e+35, -2.628780191249696e+35, -2.6497173512320048e+35, -2.6714998117246965e+35, -2.6941655357955663e+35, -2.7177543329462194e+35, -2.7423089597455483e+35, -2.7678761872229943e+35, -2.794507744955295e+35, -2.822261048613802e+35, -2.8511996120633054e+35, -2.8813930345756172e+35, -2.912916438109757e+35, -2.945849212157233e+35, -2.9802729119518284e+35, -3.0162681616715385e+35, -3.0539104518973132e+35, -3.0932648034228736e+35, -3.134379405312531e+35, -3.177278520966127e+35, -3.2219551742713746e+35, -3.268364345519446e+35, -3.3164175783009187e+35, -3.365979975056084e+35, -3.416870499687058e+35, -3.468866291113415e+35, -3.521711332247136e+35, -3.575129355896749e+35, -3.628840366052131e+35, -3.682579679235091e+35, -3.7361180033694975e+35, -3.789280804022034e+35, -3.841965068292442e+35, -3.894151560789712e+35, -3.945910775698746e+35, -3.997401047480043e+35, -4.04885773801398e+35, -4.100573125022329e+35, -4.152867602924937e+35, -4.206054031125737e+35, -4.260398383447585e+35, -4.3160810203581916e+35, -4.373163613250733e+35, -4.431566703637536e+35, -4.4910619010943534e+35, -4.551280833940486e+35, -4.611740422891545e+35, -4.671881306054193e+35, -4.731113858443142e+35, -4.7888647438839664e+35, -4.844616685293188e+35, -4.897935289770962e+35, -4.948479215595237e+35, -4.9959933735841384e+35, -5.0402886529458005e+35, -5.081215117733995e+35, -5.118637915682138e+35, -5.152425523126682e+35, -5.1824579413714874e+35, -5.208658083851798e+35, -5.231043514982587e+35, -5.2497891968449686e+35, -5.265286605613667e+35, -5.2781820691818596e+35, -5.289378495712962e+35, -5.299989981402493e+35, -5.311247293301304e+35, -5.324362275320619e+35, -5.3403687010388635e+35, -5.359963869915329e+35, -5.3833776789405545e+35, -5.410293263971362e+35, -5.439835961365685e+35, -5.470636671176897e+35, -5.500963733100152e+35, -5.5289063237563485e+35, -5.552584026198943e+35, -5.5703529549386095e+35, -5.58097937561494e+35, -5.583757209088004e+35, -5.578555455304562e+35, -5.5657938337720386e+35, -5.546357425724205e+35, -5.521471016914517e+35, -5.492558685909726e+35, -5.461112617079884e+35, -5.428587583210402e+35, -5.3963263760953436e+35, -5.365510306400877e+35, -5.337121535529206e+35, -5.3119030608595286e+35, -5.2903081388845555e+35, -5.27244190765666e+35, -5.258010197919834e+35, -5.246299406299227e+35, -5.2362127017735356e+35, -5.2263793737461755e+35, -5.215336345807517e+35, -5.201757860866202e+35, -5.184688297289115e+35, -5.1637223711552125e+35, -5.139082851028986e+35, -5.111569079904516e+35, -5.0823837833957235e+35, -5.0528793941146724e+35, -5.02428654537884e+35, -4.9974893557122134e+35, -4.9728951061457425e+35, -4.950417068608826e+35, -4.929558774229622e+35, -4.9095650020592e+35, -4.889594478665404e+35, -4.86887243460807e+35, -4.846794763970493e+35, -4.822974321554874e+35, -4.797237944235591e+35, -4.7695949905468676e+35, -4.740201523986187e+35, -4.709338557858146e+35, -4.677410760310198e+35, -4.644958419215531e+35, -4.612665448415726e+35, -4.5813436302011855e+35, -4.551879358432811e+35, -4.5251419952859164e+35, -4.501868130022385e+35, -4.482547949420786e+35, -4.467343889619709e+35, -4.4560657784376786e+35, -4.4482124774330796e+35, -4.443072286747291e+35, -4.439858861342479e+35, -4.437850731612897e+35, -4.4365027519678474e+35, -4.43550616259618e+35, -4.434787660320078e+35, -4.434453164955053e+35, -4.4346949942891174e+35, -4.435688771260925e+35, -4.437506818985898e+35, -4.440068207204607e+35, -4.4431341402300916e+35, -4.44634457211714e+35, -4.4492816238381124e+35, -4.451540370713184e+35, -4.452788740770803e+35, -4.4528043819709615e+35, -4.451484720051818e+35, -4.448834031106045e+35, -4.444936007559137e+35, -4.439921269794199e+35, -4.433937240907959e+35, -4.427124273721314e+35, -4.419598520272689e+35, -4.411439912532161e+35, -4.4026831843869925e+35, -4.393310818996325e+35, -4.3832484229954705e+35, -4.372364468093183e+35, -4.3604769327215466e+35, -4.34736880520936e+35, -4.3328127695374196e+35, -4.316603139440697e+35, -4.298590914220616e+35, -4.2787164017610644e+35, -4.257033693571348e+35, -4.233722505788154e+35, -4.209085199619859e+35, -4.183529507141064e+35, -4.157539858974324e+35, -4.1316416630430296e+35, -4.1063632051721646e+35, -4.0821992034094134e+35, -4.059578873304754e+35, -4.038840133498338e+35, -4.020210667899269e+35, -4.0037961235172556e+35, -3.989575708618018e+35, -3.9774056521766864e+35, -3.967031109012616e+35, -3.958106887823526e+35, -3.950226698492185e+35, -3.9429594918068566e+35, -3.935890114553201e+35, -3.9286602733704994e+35, -3.921005067307641e+35, -3.91278039206543e+35, -3.9039774270483074e+35, -3.894722051657171e+35, -3.885259074172954e+35, -3.8759231781184055e+35, -3.867100109304679e+35, -3.859182589411683e+35, -3.8525256904215175e+35, -3.847406065079217e+35, -3.843988739644739e+35, -3.8423043753899716e+35, -3.842239134968518e+35, -3.8435385457556905e+35, -3.845825916164162e+35, -3.848634783863019e+35, -3.851453476470558e+35, -3.853778204557545e+35, -3.855169400029406e+35, -3.855304601623229e+35, -3.8540204746415225e+35, -3.851336914318765e+35, -3.847457889489854e+35, -3.842746788362453e+35, -3.837678259233689e+35, -3.832773221818695e+35, -3.828527810476158e+35, -3.825349281087487e+35, -3.823511371971106e+35, -3.8231379357124254e+35, -3.8242175057887516e+35, -3.826644380454789e+35, -3.830275758503447e+35, -3.834991111597424e+35, -3.8407401010305724e+35, -3.847568631476047e+35, -3.8556179315891254e+35, -3.865097389150943e+35, -3.876236953784714e+35, -3.889228428808554e+35, -3.904166586473848e+35, -3.921000817254734e+35, -3.939506237010557e+35, -3.959280193906783e+35, -3.979766351957833e+35, -4.000304429064812e+35, -4.02019972410136e+35, -4.038803298157867e+35, -4.055591585349519e+35, -4.070233718655738e+35, -4.0826362002842116e+35, -4.0929576673065375e+35, -4.101590984239487e+35, -4.1091149712462816e+35, -4.116222774746285e+35, -4.123637263287714e+35, -4.1320252533835516e+35, -4.1419217173992714e+35, -4.153672812778966e+35, -4.167403366890165e+35, -4.1830111674921676e+35, -4.200187606809152e+35, -4.218462064888526e+35, -4.237265695807995e+35, -4.2560086493623355e+35, -4.2741629870223065e+35, -4.291341721139496e+35, -4.307362992257527e+35, -4.322288149169439e+35, -4.336424181502884e+35, -4.350285047354209e+35, -4.364512823024854e+35, -4.3797674202017565e+35, -4.396601320907394e+35, -4.4153414257803536e+35, -4.4360018489921386e+35, -4.458248173682456e+35, -4.481425358919397e+35, -4.504649601205058e+35, -4.526951539897119e+35, -4.547447208576924e+35, -4.565506598151812e+35, -4.5808890294750825e+35, -4.593819735894131e+35, -4.604991838271489e+35, -4.615490143342391e+35, -4.626645624213681e+35, -4.639840180519076e+35, -4.656289169564275e+35, -4.676833692671159e+35, -4.70177535477009e+35, -4.730782680447481e+35, -4.7628899724721354e+35, -4.796596056801899e+35, -4.830053316259974e+35, -4.861319680561884e+35, -4.888632124670772e+35, -4.9106539804524056e+35, -4.926652473620823e+35, -4.936576879048429e+35, -4.941028016015308e+35, -4.941131030987061e+35, -4.938340055579127e+35, -4.9342117632569324e+35, -4.93018444455962e+35, -4.927392166746813e+35, -4.926533558030186e+35, -4.92780512114507e+35, -4.930901456045514e+35, -4.935079195633718e+35, -4.939276566594029e+35, -4.942275100608435e+35, -4.942883971555822e+35, -4.940121852106588e+35, -4.9333681329242996e+35, -4.9224570021986354e+35, -4.907695510285753e+35, -4.88979986287233e+35, -4.8697602887379696e+35, -4.848659925234801e+35, -4.827483055658376e+35, -4.806949823948438e+35, -4.787407759930218e+35, -4.768797186366676e+35, -4.750691747687219e+35, -4.7324011560678215e+35, -4.713113981672277e+35, -4.692055164771798e+35, -4.6686351205983086e+35, -4.642572683608777e+35, -4.6139802021782116e+35, -4.583404043822651e+35, -4.551817085316547e+35, -4.520562145732393e+35, -4.491248070190669e+35, -4.465604327230054e+35, -4.445305683965357e+35, -4.4317848552852196e+35, -4.42605634361392e+35, -4.428577219282958e+35, -4.439169007433422e+35, -4.457018626736709e+35, -4.480765888979387e+35, -4.508671733654943e+35, -4.5388471959054426e+35, -4.5695106241316055e+35, -4.599232503070652e+35, -4.627125649634721e+35, -4.652944915560601e+35, -4.677074899981454e+35, -4.700404836397179e+35, -4.724113065760847e+35, -4.7494039134655976e+35, -4.777251370706918e+35, -4.808202257119668e+35, -4.842275756866165e+35, -4.8789705136191214e+35, -4.917362975169634e+35, -4.9562605677733986e+35, -4.9943669481623845e+35, -5.030424801507082e+35, -5.063319239335131e+35, -5.0921431622044346e+35, -5.1162370621252814e+35, -5.1352160798768415e+35, -5.148988730188241e+35, -5.157760629413479e+35, -5.162009699449054e+35, -5.162421240539652e+35, -5.159782297052766e+35, -5.1548507765360214e+35, -5.148228743443589e+35, -5.1402740430411074e+35, -5.131076199057782e+35, -5.120503512166378e+35, -5.1083060081762895e+35, -5.094242605230194e+35, -5.078196684359673e+35, -5.060252190578805e+35, -5.04071729985631e+35, -5.020097806321433e+35, -4.999032955891999e+35, -4.978211166575194e+35, -4.958283214834636e+35, -4.9397880476217284e+35, -4.9231027219985394e+35, -4.90842338327244e+35, -4.8957785462718046e+35, -4.885069531453024e+35, -4.87612686442234e+35, -4.868767511648739e+35, -4.862837566884574e+35, -4.8582289916590024e+35, -4.854866387836246e+35, -4.852668461185104e+35, -4.851496281086163e+35, -4.851104576811794e+35, -4.851112180796945e+35, -4.851003624241243e+35, -4.850166975852229e+35, -4.847964771614714e+35, -4.8438267347254765e+35, -4.837346304869242e+35, -4.828359329291757e+35, -4.81698426506538e+35, -4.803609946781927e+35, -4.788828730427148e+35, -4.7733267037492376e+35, -4.757754218473487e+35, -4.7426052582960855e+35, -4.7281315154089324e+35, -4.714307946583628e+35, -4.700854455511373e+35, -4.6873067888958826e+35, -4.673121074524586e+35, -4.657791467479236e+35, -4.640958958065751e+35, -4.6224911918385384e+35, -4.602517978432828e+35, -4.581414841725796e+35, -4.559736863610789e+35, -4.538115734159871e+35, -4.5171419942360045e+35, -4.497259182827325e+35, -4.47869477454125e+35, -4.461443901625329e+35, -4.445307820096128e+35, -4.429974012201176e+35, -4.415113644507864e+35, -4.4004685726522505e+35, -4.3859051569251935e+35, -4.371423629300509e+35, -4.3571253574302616e+35, -4.343151622471899e+35, -4.329613508614472e+35, -4.3165324862104714e+35, -4.303806477585761e+35, -4.291208804828347e+35, -4.2784195696716036e+35, -4.26508221271588e+35, -4.250873065380581e+35, -4.235569082230045e+35, -4.219098969682488e+35, -4.201565853023655e+35, -4.1832352735754106e+35, -4.16448976584131e+35, -4.145758884809247e+35, -4.127439413375153e+35, -4.1098230443713056e+35, -4.093047471406576e+35, -4.077081950671161e+35, -4.061751232550336e+35, -4.046793903478403e+35, -4.031944173252929e+35, -4.0170212144039446e+35, -4.002008107295333e+35, -3.987103651794941e+35, -3.97273474789e+35, -3.959524214656079e+35, -3.9482177065985324e+35, -3.939582099652998e+35, -3.934294320400669e+35, -3.932842252935422e+35, -3.9354571101703335e+35, -3.94208984718897e+35, -3.952434483020057e+35, -3.965991055058461e+35, -3.982152857552411e+35, -4.000298417266944e+35, -4.0198690429977886e+35, -4.0404173427775545e+35, -4.0616195696742885e+35, -4.08325329412541e+35, -4.105149873858738e+35, -4.12713689587566e+35, -4.148988047034395e+35, -4.170396150526152e+35, -4.190979431157881e+35, -4.210322206544319e+35, -4.228040802740547e+35, -4.243856217976946e+35, -4.257650199022506e+35, -4.26948373530157e+35, -4.27956713782423e+35, -4.288186177454211e+35, -4.295603769311162e+35, -4.301965551299271e+35, -4.307237020398474e+35, -4.311190298708692e+35, -4.313444134466904e+35, -4.3135466113455766e+35, -4.3110803236681634e+35, -4.3057662107548416e+35, -4.297544338379841e+35, -4.2866158906660144e+35, -4.273438480295996e+35, -4.2586750709309304e+35, -4.24310454772158e+35, -4.227509034726493e+35, -4.212559112001044e+35, -4.1987218028295294e+35, -4.186214866212297e+35, -4.17502166745949e+35, -4.16496365310298e+35, -4.1558074924125625e+35, -4.147370558943045e+35, -4.13958924859076e+35, -4.1325298785025124e+35, -4.126343903573763e+35, -4.1211876796583074e+35, -4.117135424876085e+35, -4.1141118647403304e+35, -4.111861981927194e+35, -4.109963991542034e+35, -4.1078816242424075e+35, -4.105044793102135e+35, -4.10094410713483e+35, -4.0952239207143195e+35, -4.087759661502319e+35, -4.078707087381529e+35, -4.0685134857612016e+35, -4.0578841252578314e+35, -4.047702566853586e+35, -4.0389114228585665e+35, -4.032369955563654e+35, -4.028713676953737e+35, -4.02824492092414e+35, -4.030879253372196e+35, -4.036160514663624e+35, -4.0433406286105114e+35, -4.051504531843656e+35, -4.059710841927522e+35, -4.067117923426721e+35, -4.073072268299617e+35, -4.077148333083347e+35, -4.079141810218459e+35, -4.079028024854031e+35, -4.07690182094129e+35, -4.0729150556518465e+35, -4.067224222189738e+35, -4.0599557801539234e+35, -4.051192092722813e+35, -4.0409772202199245e+35, -4.029339173298041e+35, -4.016323120028066e+35, -4.002028048620455e+35, -3.986637560885374e+35, -3.9704345194627974e+35, -3.953790372021339e+35, -3.937124185989079e+35, -3.9208338917348195e+35, -3.905211576786791e+35, -3.8903630308326004e+35, -3.876155610283152e+35, -3.862215222122489e+35, -3.8479824794616584e+35, -3.832822454514742e+35, -3.8161667780147835e+35, -3.797656305045684e+35, -3.777250620605356e+35, -3.755277670460955e+35, -3.732410265197496e+35, -3.709572114750939e+35, -3.6877905914851804e+35, -3.668023998681062e+35, -3.6509964298821776e+35, -3.6370728072429615e+35, -3.6262002288921794e+35, -3.617929711703291e+35, -3.6115162468207755e+35, -3.6060777506620775e+35, -3.600779180102696e+35, -3.595001064725646e+35, -3.5884547290123515e+35, -3.581219376192153e+35, -3.573695627270572e+35, -3.566490527590326e+35, -3.560264680260989e+35, -3.5555789687343274e+35, -3.552775086662793e+35, -3.5519125760101904e+35, -3.5527692264618806e+35, -3.554896253550958e+35, -3.557708700282344e+35, -3.5605872009275804e+35, -3.562969526094339e+35, -3.564417221571845e+35, -3.564651268298511e+35, -3.563558323923722e+35, -3.561174050648822e+35, -3.557651861115154e+35, -3.553224755191292e+35, -3.5481659456657256e+35, -3.542751832611772e+35, -3.537229336186875e+35, -3.53178886406504e+35, -3.526544077505036e+35, -3.52151972431209e+35, -3.516648728889271e+35, -3.511779226599642e+35, -3.5066912816975455e+35, -3.501121825765776e+35, -3.494795219663142e+35, -3.487456100037864e+35, -3.478901001656848e+35, -3.46900559605524e+35, -3.457745000199599e+35, -3.4452051862325426e+35, -3.431583921911001e+35, -3.417180018188869e+35, -3.402370291870143e+35, -3.387574899397154e+35, -3.3732136420218876e+35, -3.3596581624263825e+35, -3.347186965136802e+35, -3.335951072664712e+35, -3.32595723718013e+35, -3.317072818120645e+35, -3.3090522066016024e+35, -3.3015800726788607e+35, -3.2943230006130797e+35, -3.2869793246421633e+35, -3.2793176814688937e+35, -3.271197735423794e+35, -3.2625708705215486e+35, -3.2534632192734677e+35, -3.243947086224254e+35, -3.2341088340793343e+35, -3.2240213288908506e+35, -3.2137272658041976e+35, -3.203236650030387e+35, -3.1925381022795367e+35, -3.1816202406339515e+35, -3.1704968525575023e+35, -3.1592284885790144e+35, -3.1479338528704023e+35, -3.1367869240888424e+35, -3.125999533705624e+35, -3.1157930084344113e+35, -3.1063650924079717e+35, -3.0978588256189147e+35, -3.090338565379998e+35, -3.083776144328814e+35, -3.0780487174455494e+35, -3.0729496452043578e+35, -3.0682137269548114e+35, -3.063556364378643e+35, -3.0587218865782174e+35, -3.0535307872285005e+35, -3.0479124649356963e+35, -3.041912250270893e+35, -3.035669112707646e+35, -3.0293701778523124e+35, -3.0231955654936013e+35, -3.01726920054471e+35, -3.0116281111195837e+35, -3.0062163591834146e+35, -3.0009027361971568e+35, -2.9955157164138404e+35, -2.989886002276063e+35, -2.9838866078446287e+35, -2.977462446417249e+35, -2.9706449799729314e+35, -2.9635515680041454e+35, -2.9563726831089237e+35, -2.9493523968800294e+35, -2.9427681498376318e+35, -2.936914860096733e+35, -2.9320962562179264e+35, -2.9286235197367787e+35, -2.9268186135333828e+35, -2.927017791606216e+35, -2.9295702917944745e+35, -2.93482827260884e+35, -2.943126347101939e+35, -2.954751889267579e+35, -2.9699098420340346e+35, -2.9886874387274293e+35, -3.0110248052549167e+35, -3.0366968773580706e+35, -3.065310627866143e+35, -3.0963194638064126e+35, -3.1290540664440126e+35, -3.1627662654156813e+35, -3.196680268791173e+35, -3.2300442781171967e+35, -3.262175607837384e+35, -3.292493940711021e+35, -3.320539892322048e+35, -3.3459789449571355e+35, -3.368593335948368e+35, -3.388266150286971e+35, -3.4049624689620345e+35, -3.418711992261952e+35, -3.4295962501744435e+35, -3.437741583543937e+35, -3.443316917557037e+35, -3.446533491624655e+35, -3.447642734125887e+35, -3.4469287619630646e+35, -3.444693523279916e+35, -3.441234925870305e+35, -3.436820682732198e+35, -3.431662365371255e+35, -3.4258948428348e+35, -3.419565784468986e+35, -3.4126383714214726e+35, -3.40500811707922e+35, -3.39653214425483e+35, -3.3870668471379945e+35, -3.376508020824709e+35, -3.364826664541419e+35, -3.352094024819073e+35, -3.338491092359953e+35, -3.324300464976793e+35, -3.309881715083917e+35, -3.295634442414225e+35, -3.281955342426873e+35, -3.269196393172426e+35, -3.2576305540809014e+35, -3.247429475694699e+35, -3.2386552088114158e+35, -3.23126540547935e+35, -3.225129522296809e+35, -3.220052335824212e+35, -3.215800698307264e+35, -3.2121297720324194e+35, -3.208805765988877e+35, -3.20562321384994e+35, -3.202415850395903e+35, -3.199060998480108e+35, -3.1954780021379205e+35, -3.1916216740227592e+35, -3.1874720833916774e+35, -3.1830224071001895e+35, -3.1782670120487252e+35, -3.173192280379369e+35, -3.1677726383769347e+35, -3.161973518711157e+35, -3.1557614936266376e+35, -3.1491198485741934e+35, -3.1420660456070986e+35, -3.134666565407545e+35, -3.1270449523624944e+35, -3.1193804351904318e+35, -3.111896700515864e+35, -3.1048425301078452e+35, -3.098467507174236e+35, -3.0929966162726688e+35, -3.088607369481164e+35, -3.085412325165702e+35, -3.0834488053430883e+35, -3.0826765004806057e+35, -3.0829826312608935e+35, -3.0841934948226173e+35, -3.086090603410854e+35, -3.0884292770887556e+35, -3.0909575384146827e+35, -3.093433501614416e+35, -3.0956400911441662e+35, -3.097396698681101e+35, -3.098568062585187e+35, -3.099071016191621e+35, -3.0988796886422406e+35, -3.0980292885285666e+35, -3.096617928788057e+35, -3.0948053203614928e+35, -3.0928068481771115e+35, -3.0908817699982638e+35, -3.089315153849454e+35, -3.0883946188016856e+35, -3.0883846544205558e+35, -3.0895027155148587e+35, -3.09190175965389e+35, -3.095662922268186e+35, -3.100799614158634e+35, -3.1072711513146108e+35, -3.1150012460225666e+35, -3.123895445038502e+35, -3.1338524651531946e+35, -3.1447670573250942e+35, -3.156525507189686e+35, -3.1689978442055188e+35, -3.182032189155742e+35, -3.1954559001251357e+35, -3.2090854931578893e+35, -3.22274362479824e+35, -3.2362781011911017e+35, -3.24957630769106e+35, -3.262569445083828e+35, -3.2752242707887725e+35, -3.2875243456670828e+35, -3.2994462380845718e+35, -3.3109373235612574e+35, -3.321900485416752e+35, -3.332188113714889e+35, -3.341604822559233e+35, -3.349916480144838e+35, -3.356862885237592e+35, -3.362172351411803e+35, -3.365577803461758e+35, -3.366835012907962e+35, -3.365743830474394e+35, -3.362172514057263e+35, -3.356083538922134e+35, -3.34755698165449e+35, -3.3368055517448446e+35, -3.324174861862516e+35, -3.3101246200567846e+35, -3.2951910239332785e+35, -3.2799361369353236e+35, -3.264893966549444e+35, -3.250523502485415e+35, -3.2371760844176813e+35, -3.2250798866422103e+35, -3.2143402979599265e+35, -3.2049529419672392e+35, -3.1968259640068295e+35, -3.1898089014686187e+35, -3.183725752848968e+35, -3.178409251422671e+35, -3.1737322056509805e+35, -3.1696310080890202e+35, -3.1661169039589644e+35, -3.163272656485175e+35, -3.161235376188019e+35, -3.1601693980900165e+35, -3.1602349977998254e+35, -3.1615588392477265e+35, -3.1642107419241776e+35, -3.1681896790074664e+35, -3.173420603823939e+35, -3.1797625287079724e+35, -3.1870264504796976e+35, -3.1949990002891854e+35, -3.203465159961236e+35, -3.2122228458364464e+35, -3.221084736576911e+35, -3.229867791818629e+35, -3.2383763210630443e+35, -3.24638775845322e+35, -3.253650007016464e+35, -3.2598954522417248e+35, -3.2648709930013358e+35, -3.26837781103991e+35, -3.270310999124375e+35, -3.270688646862271e+35, -3.2696624955536588e+35, -3.267506819495189e+35, -3.2645872628393178e+35, -3.261315497574272e+35, -3.258097821666401e+35, -3.2552859779730346e+35, -3.2531370031513783e+35, -3.251786628970779e+35, -3.251238439017944e+35, -3.2513690843503718e+35, -3.251948426112329e+35, -3.2526722753370352e+35, -3.2532041932053574e+35, -3.2532215474310938e+35, -3.252459914652997e+35, -3.250749388304342e+35, -3.248036812407938e+35, -3.244389632872363e+35, -3.2399798363633308e+35, -3.235049900325745e+35, -3.2298661413680076e+35, -3.2246675845230777e+35, -3.2196198414347367e+35, -3.2147830820432053e+35, -3.2101009510968854e+35, -3.205413531950947e+35, -3.20049285805469e+35, -3.1950949377045757e+35, -3.1890188074272908e+35, -3.182161632562858e+35, -3.1745597366483505e+35, -3.1664083372621927e+35, -3.158056633688421e+35, -3.1499784054076818e+35, -3.1427206862859492e+35, -3.1368348197224235e+35, -3.1327965952818436e+35, -3.1309260424588247e+35, -3.131321714740124e+35, -3.1338259340719203e+35, -3.138033527602352e+35, -3.1433467062772168e+35, -3.1490660816091186e+35, -3.1544975448344635e+35, -3.1590510540679285e+35, -3.1623113974651988e+35, -3.1640705574619366e+35, -3.164322440678282e+35, -3.1632296370501862e+35, -3.161076147902205e+35, -3.158219151902868e+35, -3.1550479894981216e+35, -3.1519519438170575e+35, -3.1492929114515977e+35, -3.1473771634770177e+35, -3.1464231410498027e+35, -3.146528425204182e+35, -3.1476453987703103e+35, -3.1495777592255482e+35, -3.1520065184592237e+35, -3.154545083620725e+35, -3.1568121134561038e+35, -3.1585030523645263e+35, -3.1594403874441293e+35, -3.1595894859976496e+35, -3.1590385170834238e+35, -3.1579524233930456e+35, -3.1565175997194e+35, -3.1548939946014015e+35, -3.1531862343468865e+35, -3.1514384111114172e+35, -3.149651215419251e+35, -3.1478157747729883e+35, -3.1459546496574632e+35, -3.1441561083449276e+35, -3.142585072898723e+35, -3.141457711195609e+35, -3.1409799271222337e+35, -3.141269671272501e+35, -3.14229788511058e+35, -3.143880584893056e+35, -3.145732261868244e+35, -3.1475602560781846e+35, -3.1491599586481135e+35, -3.150472914118136e+35, -3.1515900186245285e+35, -3.152706294150588e+35, -3.1540503130119557e+35, -3.1558161269680257e+35, -3.158120185251525e+35, -3.1609938337415375e+35, -3.1644077616783926e+35, -3.1683130198308557e+35, -3.17267845974192e+35, -3.1775085347229804e+35, -3.1828361023295745e+35, -3.1886966845406453e+35, -3.1950979442203237e+35, -3.2019982606099373e+35, -3.2093025847594293e+35, -3.216876155032471e+35, -3.224570942563655e+35, -3.2322573827009407e+35, -3.239854010840549e+35, -3.247348193710695e+35, -3.2548016299501553e+35, -3.2623359209010954e+35, -3.2700977769517977e+35, -3.2782099304454064e+35, -3.2867198128941175e+35, -3.29556019446521e+35, -3.304532850903555e+35, -3.313319420559693e+35, -3.321516272121042e+35, -3.3286854001290775e+35, -3.3344121139005526e+35, -3.338361518155409e+35, -3.3403275125694415e+35, -3.340268847800907e+35, -3.338326692169908e+35, -3.3348184481823536e+35, -3.3302046773570784e+35, -3.325030473630629e+35, -3.319848557552056e+35, -3.3151368080334973e+35, -3.3112259205093254e+35, -3.3082522113745812e+35, -3.3061462264184586e+35, -3.3046606216433175e+35, -3.3034322391743246e+35, -3.3020652950642267e+35, -3.300217265562794e+35, -3.297668261406379e+35, -3.2943591200970666e+35, -3.2903919691558664e+35, -3.2859967771585936e+35, -3.28147510484312e+35, -3.2771357091801045e+35, -3.2732356986387217e+35, -3.2699371770921272e+35, -3.2672848173832568e+35, -3.2652058824785886e+35, -3.2635310412624228e+35, -3.262031571524884e+35, -3.2604661821717325e+35, -3.258629173519841e+35, -3.256391415324244e+35, -3.2537266820552834e+35, -3.250718065447621e+35, -3.2475424052442984e+35, -3.24443478730784e+35, -3.2416394838948964e+35, -3.2393570249738978e+35, -3.2376981854953546e+35, -3.2366540830157194e+35, -3.2360878207287157e+35, -3.2357483596131056e+35, -3.2353028506560993e+35, -3.234380507875762e+35, -3.2326197865229323e+35, -3.2297111446860242e+35, -3.2254295271670513e+35, -3.2196531852979812e+35, -3.2123678816653036e+35, -3.2036575637962076e+35, -3.1936841366657814e+35, -3.1826600417408502e+35, -3.170817966561727e+35, -3.1583821271972227e+35, -3.145545128310382e+35, -3.132453373949898e+35, -3.119202439512684e+35, -3.105841967179325e+35, -3.092387915472443e+35, -3.078838770683308e+35, -3.0651918166773433e+35, -3.0514557457953802e+35, -3.037656690929108e+35, -3.0238361658138886e+35, -3.0100414599494083e+35, -2.996311543859174e+35, -2.982663744881415e+35, -2.9690871377825454e+35, -2.955546789376484e+35, -2.9419989286033587e+35, -2.9284125252088386e+35, -2.9147900181796047e+35, -2.9011804298513182e+35, -2.8876812821466393e+35, -2.874429654381687e+35, -2.8615855569172646e+35, -2.849311896580851e+35, -2.8377551160398488e+35, -2.8270298129727448e+35, -2.8172096611250506e+35, -2.8083257711677952e+35, -2.800372198720589e+35, -2.7933167995566605e+35, -2.7871144585547338e+35, -2.781719332058507e+35, -2.777093346233591e+35, -2.7732095386025846e+35, -2.770050273777115e+35, -2.7676012284210902e+35, -2.7658420811745374e+35, -2.7647344907910654e+35, -2.7642080485913496e+35, -2.7641460483048083e+35, -2.7643748383425878e+35, -2.764661952237743e+35, -2.7647276282017403e+35, -2.7642710703435773e+35, -2.7630078596271414e+35, -2.7607106187142174e+35, -2.757243621184055e+35, -2.7525841363967695e+35, -2.746827549200455e+35, -2.740177457794056e+35, -2.7329244332481284e+35, -2.7254176044421457e+35, -2.7180324747614543e+35, -2.7111376928456493e+35, -2.705063928551019e+35, -2.7000792861108992e+35, -2.6963760204369296e+35, -2.694070842369238e+35, -2.6932160818024213e+35, -2.693814370280699e+35, -2.695828694422127e+35, -2.6991835098805355e+35, -2.7037586694678324e+35, -2.7093825597693287e+35, -2.7158317851017166e+35, -2.7228420649239896e+35, -2.7301304193769746e+35, -2.7374243004334387e+35, -2.7444907046347396e+35, -2.751158204785182e+35, -2.7573269997761756e+35, -2.762965483353151e+35, -2.7680951993925442e+35, -2.7727683910612345e+35, -2.777043254072168e+35, -2.7809616136753425e+35, -2.7845325151642584e+35, -2.787723643018594e+35, -2.7904609321571575e+35, -2.792635413183706e+35, -2.7941153441951317e+35, -2.7947610965068418e+35, -2.7944401542344674e+35, -2.7930399981593218e+35, -2.790477498769693e+35, -2.7867044935035524e+35, -2.7817100792576898e+35, -2.775520441194564e+35, -2.768196641421955e+35, -2.759830005509136e+35, -2.7505342400119724e+35, -2.7404338870460835e+35, -2.7296503820682e+35, -2.7182891765993062e+35, -2.706432734008134e+35, -2.694143340713912e+35, -2.6814762851732846e+35, -2.668499343920993e+35, -2.655311064181403e+35, -2.642050029324263e+35, -2.6288903253525208e+35, -2.616023132496788e+35, -2.6036286269649395e+35, -2.5918450598454438e+35, -2.5807429030678986e+35, -2.5703113002150304e+35, -2.560461336344323e+35, -2.551045884082282e+35, -2.5418904195300447e+35, -2.5328256796320555e+35, -2.5237131971558394e+35, -2.5144584443183274e+35, -2.5050114686479793e+35, -2.495358937594761e+35, -2.485512901910601e+35, -2.4755005108662604e+35, -2.4653566960158515e+35, -2.4551199016371327e+35, -2.444829994169501e+35, -2.434527442872202e+35, -2.4242532343339577e+35, -2.414049307891202e+35, -2.403959381030098e+35, -2.394029920418744e+35, -2.3843108664195276e+35, -2.3748556847544845e+35, -2.3657204607343047e+35, -2.3569620320423502e+35, -2.3486354677914918e+35, -2.3407914171280234e+35, -2.3334738760817466e+35, -2.32671873353506e+35, -2.3205531136207043e+35, -2.314995150960093e+35, -2.3100535556642293e+35, -2.3057262603754818e+35, -2.3019976443896564e+35, -2.2988342734048422e+35, -2.2961796738395984e+35, -2.2939492194996185e+35, -2.2920265769850538e+35, -2.290263212258877e+35, -2.2884821756926233e+35, -2.286486832852779e+35, -2.2840745302743224e+35, -2.281054487896214e+35, -2.2772684965650256e+35, -2.2726122079028404e+35, -2.267053989482115e+35, -2.2606478139414584e+35, -2.2535369585330672e+35, -2.245946692117907e+35, -2.2381663448564225e+35, -2.2305234247576956e+35, -2.223353936367359e+35, -2.216973329811209e+35, -2.2116516723257715e+35, -2.207595185427329e+35, -2.204934806686994e+35, -2.2037213294704987e+35, -2.2039261049912725e+35, -2.205446198802941e+35, -2.208113095621531e+35, -2.211704331076787e+35, -2.2159576286711393e+35, -2.2205871495833184e+35, -2.2253013173595047e+35, -2.2298213946585075e+35, -2.23389959872569e+35, -2.237335088796599e+35, -2.2399857714283965e+35, -2.2417738297123574e+35, -2.2426835405488905e+35, -2.242751449046596e+35, -2.242050974712581e+35, -2.2406751947413282e+35, -2.238722003965813e+35, -2.236284772842134e+35, -2.233449528936773e+35, -2.2302975837553735e+35, -2.2269112568791123e+35, -2.2233801281715583e+35, -2.2198057160397986e+35, -2.2163031240985186e+35, -2.2129987692737705e+35, -2.2100239072531196e+35, -2.20750452690405e+35, -2.205549283057416e+35, -2.2042380907762393e+35, -2.2036142569109566e+35, -2.203682257229946e+35, -2.2044116827182154e+35, -2.2057461244004006e+35, -2.2076145550891596e+35, -2.2099424869129898e+35, -2.2126607612129803e+35, -2.215710874931591e+35, -2.2190468292508115e+35, -2.222634302532984e+35, -2.2264483705242563e+35, -2.230471008484659e+35, -2.2346892748268706e+35, -2.2390945370514868e+35, -2.243682578509838e+35, -2.2484541414922253e+35, -2.253415513194615e+35, -2.2585790310016013e+35, -2.2639636101169847e+35, -2.269595356217274e+35, -2.275507999273252e+35, -2.2817424724079766e+35, -2.2883447452849977e+35, -2.295361196971728e+35, -2.3028313798611862e+35, -2.3107788069018403e+35, -2.3192011229750585e+35, -2.3280614591579013e+35, -2.337282803042052e+35, -2.3467468990393924e+35, -2.356298675462647e+35, -2.3657566099930074e+35, -2.3749287805498858e+35, -2.383633470362462e+35, -2.391722029737977e+35, -2.399100444369236e+35, -2.405745236224602e+35, -2.4117095148019507e+35, -2.417116482563458e+35, -2.4221402156098593e+35, -2.4269763707656168e+35, -2.4318077696236183e+35, -2.4367709534199672e+35, -2.4419295636098703e+35, -2.447258950924017e+35, -2.4526442014911957e+35, -2.4578913715037376e+35, -2.4627496844272455e+35, -2.4669411175665676e+35, -2.4701932559310016e+35, -2.4722713502044123e+35, -2.473005952714346e+35, -2.4723132191799544e+35, -2.4702060048053776e+35, -2.466795272905542e+35, -2.4622828528686884e+35, -2.456947756059549e+35, -2.4511286047930668e+35, -2.4452041105805486e+35, -2.4395723045944225e+35, -2.4346280577701736e+35, -2.4307379961669163e+35, -2.428212574666639e+35, -2.4272767464944333e+35, -2.428042861758897e+35, -2.4304912716965202e+35, -2.4344645117709975e+35, -2.439679025437462e+35, -2.4457542829987648e+35, -2.4522543919820445e+35, -2.458734220202712e+35, -2.4647820689226487e+35, -2.4700533047147994e+35, -2.4742920122032615e+35, -2.477339357340031e+35, -2.479128415378125e+35, -2.4796669054308948e+35, -2.4790118623181205e+35, -2.4772426928795572e+35, -2.474439804488742e+35, -2.4706742937849992e+35, -2.46601043915874e+35, -2.4605182160281353e+35, -2.454289496718654e+35, -2.447450760118723e+35, -2.4401677108362935e+35, -2.432641689569154e+35, -2.425100962780302e+35, -2.417789656008939e+35, -2.4109542574059892e+35, -2.4048258553202282e+35, -2.399598265071512e+35, -2.395406744502394e+35, -2.3923147121517146e+35, -2.390313396091318e+35, -2.3893329371439756e+35, -2.38925802152696e+35, -2.3899404042360817e+35, -2.391204450544564e+35, -2.3928471494121184e+35, -2.3946380918686263e+35, -2.396325880454431e+35, -2.3976545921124756e+35, -2.398388182410541e+35, -2.3983352417885147e+35, -2.397365150427659e+35, -2.39541082146498e+35, -2.3924600878147317e+35, -2.388542461115787e+35, -2.383717677437728e+35, -2.3780686681726824e+35, -2.371698110629927e+35, -2.364726689733586e+35, -2.3572919057716423e+35, -2.3495469111806e+35, -2.341658706616907e+35, -2.333804556386659e+35, -2.3261654217271042e+35, -2.3189158626272098e+35, -2.312211109453466e+35, -2.3061734907865797e+35, -2.300881565605802e+35, -2.2963654941240372e+35, -2.292610971108774e+35, -2.2895716200884202e+35, -2.2871869226129946e+35, -2.2854005768083507e+35, -2.2841734362175113e+35, -2.2834862419723004e+35, -2.2833301918160476e+35, -2.2836874174646166e+35, -2.2845073893310016e+35, -2.2856874563891287e+35, -2.287064987350439e+35, -2.2884250554844377e+35, -2.2895227152970095e+35, -2.290114491950406e+35, -2.289991169496674e+35, -2.2890039973245167e+35, -2.287078905454792e+35, -2.2842172318507586e+35, -2.280485158509781e+35, -2.275996064245514e+35, -2.2708901241662113e+35, -2.265314822447434e+35, -2.2594096602091262e+35, -2.253297942098265e+35, -2.2470869463002792e+35, -2.240874847533022e+35, -2.2347601178283347e+35, -2.228848771917073e+35, -2.223257011475298e+35, -2.2181097907834908e+35, -2.2135374400188722e+35, -2.2096719388298293e+35, -2.206642569384612e+35, -2.2045690654770398e+35, -2.203550210103662e+35, -2.2036473713239038e+35, -2.2048650937210373e+35, -2.207133535881299e+35, -2.2102991780145852e+35, -2.214129890053862e+35, -2.218337600596634e+35, -2.2226167604004838e+35, -2.226691152921904e+35, -2.2303578169335575e+35, -2.2335168094276547e+35, -2.236179338904138e+35, -2.2384527807998227e+35, -2.24050705874832e+35, -2.2425312593128765e+35, -2.2446914203188576e+35, -2.2470999504172602e+35, -2.2498040877924105e+35, -2.252795605388902e+35, -2.256037784688085e+35, -2.2595003542890956e+35, -2.2631905263301322e+35, -2.2671695855178105e+35, -2.2715494426573883e+35, -2.2764704659870454e+35, -2.282068157879537e+35, -2.288439446063696e+35, -2.2956184828065438e+35, -2.3035677696686345e+35, -2.3121852919768526e+35, -2.3213242061473288e+35, -2.3308194077661837e+35, -2.340514823955203e+35, -2.350285925981356e+35, -2.3600534013286983e+35, -2.369786089832448e+35, -2.37949396016401e+35, -2.3892144060444024e+35, -2.398996522334501e+35, -2.4088876127030972e+35, -2.418924122092956e+35, -2.4291265068742474e+35, -2.439495712319438e+35, -2.4500090184805715e+35, -2.4606150603524038e+35, -2.4712305701973305e+35, -2.481742892325371e+35, -2.4920211772823062e+35, -2.5019356040237772e+35, -2.5113799604583898e+35, -2.52029087437997e+35, -2.5286581182529685e+35, -2.536523844197961e+35, -2.5439721976646904e+35, -2.5511127265168536e+35, -2.558061037400665e+35, -2.5649192754695425e+35, -2.571758476869772e+35, -2.5786050537559294e+35, -2.585433903859752e+35, -2.5921699067449605e+35, -2.5986977015394538e+35, -2.6048774888373424e+35, -2.610563254857976e+35, -2.6156197430504198e+35, -2.6199353994073483e+35, -2.6234298034146104e+35, -2.6260553357279023e+35, -2.6277938532003354e+35, -2.6286498939504604e+35, -2.6286423723019666e+35, -2.627796783268435e+35, -2.626139596392444e+35, -2.6236958454363824e+35, -2.6204900696778693e+35, -2.6165499361239352e+35, -2.6119112539586776e+35, -2.6066227974238885e+35, -2.6007494056000868e+35, -2.594372178678872e+35, -2.5875851583927208e+35, -2.5804885841585222e+35, -2.573179576763257e+35, -2.565741807294747e+35, -2.558236191949545e+35, -2.55069471820006e+35, -2.543119025718367e+35, -2.5354843642506297e+35, -2.5277482337894134e+35, -2.519861707704126e+35, -2.511780528500317e+35, -2.5034729136430507e+35, -2.494921853198029e+35, -2.486121468568558e+35, -2.477069260321333e+35, -2.467757971876649e+35, -2.4581714617154267e+35, -2.4482879460450302e+35, -2.4380914853485538e+35, -2.4275894861237047e+35, -2.4168313216905093e+35, -2.405921834244911e+35, -2.3950240502479687e+35, -2.384348061805602e+35, -2.374127102580476e+35, -2.3645859586356946e+35, -2.3559092994680926e+35, -2.348217252662922e+35, -2.3415527998617296e+35, -2.3358816639320637e+35, -2.3311020216038638e+35, -2.327059942970632e+35, -2.3235671882732375e+35, -2.320419905206302e+35, -2.3174182529257364e+35, -2.3143868614597206e+35, -2.3111943991013282e+35, -2.3077686654803954e+35, -2.3041031815279105e+35, -2.300252965141661e+35, -2.2963203302606273e+35, -2.29243445473981e+35, -2.2887297697213663e+35, -2.2853276774076594e+35, -2.2823243534131387e+35, -2.2797853292320474e+35, -2.277745886802039e+35, -2.2762154013705615e+35, -2.275183644089551e+35, -2.2746274214202654e+35, -2.2745164321614425e+35, -2.2748176489685228e+35, -2.2754978591462277e+35, -2.2765243070442e+35, -2.2778637043372825e+35, -2.279480151087782e+35, -2.281332634447363e+35, -2.2833727045490516e+35, -2.285542743573214e+35, -2.2877750736578423e+35, -2.289992072971193e+35, -2.292107468496854e+35, -2.294028963970461e+35, -2.295662258768972e+35, -2.2969162828657404e+35, -2.2977091395943575e+35, -2.297973899280266e+35, -2.2976631688637218e+35, -2.296751443623133e+35, -2.2952347174699508e+35, -2.2931275754965457e+35, -2.2904586815545016e+35, -2.287265839465871e+35, -2.283591550017605e+35, -2.2794794753659507e+35, -2.2749718505064163e+35, -2.2701078194730034e+35, -2.264922784276955e+35, -2.2594489009646267e+35, -2.2537167357441923e+35, -2.2477578700976865e+35, -2.2416080379797086e+35, -2.2353102634061103e+35, -2.228917448397643e+35, -2.2224939060827583e+35, -2.2161153993720977e+35, -2.2098673116846573e+35, -2.2038406866295737e+35, -2.1981261445080023e+35, -2.1928062117448173e+35, -2.1879473006996056e+35, -2.183593120814945e+35, -2.1797612878993718e+35, -2.1764441867367935e+35, -2.173614016215566e+35, -2.1712309010419822e+35, -2.169252307947158e+35, -2.1676417462664898e+35, -2.166374753054518e+35, -2.1654405414295442e+35, -2.1648386401577705e+35, -2.1645713392604617e+35, -2.1646342585040094e+35, -2.1650081096117105e+35, -2.1656542761829516e+35, -2.166515378901186e+35, -2.167520204897261e+35, -2.1685909877422384e+35, -2.1696504718864716e+35, -2.170626554332284e+35, -2.1714533458098357e+35, -2.172068841316164e+35, -2.172410592225267e+35, -2.172411452211846e+35, -2.1719974210022638e+35, -2.1710888650437376e+35, -2.1696052207555e+35, -2.1674721053653966e+35, -2.1646289959799883e+35, -2.1610355508037075e+35, -2.1566752315181676e+35, -2.1515558766848894e+35, -2.1457078653728446e+35, -2.139181127964945e+35, -2.1320423259358658e+35, -2.1243730926964373e+35, -2.1162695332807953e+35, -2.1078424911046715e+35, -2.0992176072033046e+35, -2.090534028657519e+35, -2.0819408137037734e+35, -2.0735906300417005e+35, -2.065631150670903e+35, -2.058195356889665e+35, -2.051392392664684e+35, -2.0453004237105535e+35, -2.0399622261270125e+35, -2.035383393216352e+35, -2.0315326002032214e+35, -2.028343532268554e+35, -2.0257186881003974e+35, -2.0235358641923997e+35, -2.021658212788201e+35, -2.0199480628225776e+35, -2.0182833000064026e+35, -2.0165735612779855e+35, -2.014772611650752e+35, -2.0128836730790324e+35, -2.010956192411364e+35, -2.0090748858051043e+35, -2.0073438647799016e+35, -2.0058694827494467e+35, -2.0047451299901872e+35, -2.00404001438912e+35, -2.0037926576807837e+35, -2.004008894761642e+35, -2.004663727619471e+35, -2.0057063162054797e+35, -2.007067431128421e+35, -2.0086686299677297e+35, -2.010432170042981e+35, -2.012290332119205e+35, -2.0141926407105483e+35, -2.0161096846553783e+35, -2.0180329624264966e+35, -2.019971212297874e+35, -2.021944622933955e+35, -2.023978748000115e+35, -2.0260997152860712e+35, -2.028331593590677e+35, -2.0306959082182268e+35, -2.0332126056113194e+35, -2.0359014400842728e+35, -2.038782809601875e+35, -2.0418774023779582e+35, -2.0452044670689523e+35, -2.0487789181041293e+35, -2.0526077263204087e+35, -2.056686119331198e+35, -2.060994112150473e+35, -2.065493908936508e+35, -2.0701287905016775e+35, -2.0748241466678977e+35, -2.0794911801047897e+35, -2.084033394500463e+35, -2.088355322195527e+35, -2.0923722452747424e+35, -2.096019201966903e+35, -2.0992575666698734e+35, -2.1020779791700106e+35, -2.1044991953099946e+35, -2.1065632371227058e+35, -2.10832777431374e+35, -2.109856875830982e+35, -2.1112112233666357e+35, -2.112438774869873e+35, -2.1135668688859696e+35, -2.1145968840710533e+35, -2.115502645524482e+35, -2.1162335343176358e+35, -2.1167225171475298e+35, -2.1168981258711297e+35, -2.1166981376067707e+35, -2.116081862724139e+35, -2.1150379974526696e+35, -2.1135860943135072e+35, -2.1117715675190346e+35, -2.1096560976868305e+35, -2.107306495735698e+35, -2.1047849872683202e+35, -2.102142606473768e+35, -2.099415696928729e+35, -2.0966243359727053e+35, -2.093771385665072e+35, -2.0908417428183683e+35, -2.087802616096293e+35, -2.0846065508585274e+35, -2.0811989101066983e+35, -2.0775304666798643e+35, -2.0735739635138545e+35, -2.0693415915550122e+35, -2.064899113631249e+35, -2.060372487442292e+35, -2.055944482670693e+35, -2.051841418383303e+35, -2.048312718605209e+35, -2.0456074844152665e+35, -2.0439523077279156e+35, -2.04353345605276e+35, -2.044485073105503e+35, -2.0468837594633552e+35, -2.050749015895735e+35, -2.0560484587916776e+35, -2.062706327489753e+35, -2.070613584600666e+35, -2.0796379383959114e+35, -2.0896324498913014e+35, -2.1004419865867645e+35, -2.1119074979427176e+35, -2.1238686994701135e+35, -2.1361660707227913e+35, -2.14864301766644e+35, -2.16114869779203e+35, -2.1735415502256894e+35, -2.185693218408264e+35, -2.19749241474198e+35, -2.2088483426878185e+35, -2.2196934534100304e+35, -2.229985425391629e+35, -2.239708199433744e+35, -2.248871666099576e+35, -2.2575093430638086e+35, -2.265673384997091e+35, -2.2734267766246715e+35, -2.280833507557793e+35, -2.2879484692135087e+35, -2.2948091684606815e+35, -2.3014308398581555e+35, -2.3078054216370246e+35, -2.3139037555409607e+35, -2.3196797980149628e+35, -2.325075705209476e+35, -2.330027120728111e+35, -2.3344684831039106e+35, -2.3383384292389253e+35, -2.3415853455745406e+35, -2.3441728990560706e+35, -2.346085108286045e+35, -2.347330323498468e+35, -2.3479434569354914e+35, -2.347985965238601e+35, -2.3475433936219616e+35, -2.3467206695705818e+35, -2.345635698968265e+35, -2.3444121087848368e+35, -2.3431721574334292e+35, -2.3420308575172244e+35, -2.3410921794854967e+35, -2.3404477994730214e+35, -2.340178255606278e+35, -2.3403557152284987e+35, -2.341047033870631e+35, -2.342315600012072e+35, -2.344220697974306e+35, -2.3468137213187995e+35, -2.3501313478186657e+35, -2.3541865391743867e+35, -2.3589588196490824e+35, -2.3643856743062502e+35, -2.3703570728867563e+35, -2.376714996504042e+35, -2.3832592885130747e+35, -2.3897600929868594e+35, -2.3959757257861774e+35, -2.4016734475216308e+35, -2.406649782300703e+35, -2.4107470861421628e+35, -2.413863984145641e+35, -2.4159587008672513e+35, -2.417045718925896e+35, -2.417187244346651e+35, -2.4164814851772253e+35, -2.4150498063198466e+35, -2.4130245421011338e+35, -2.4105387644469806e+35, -2.4077187236357496e+35, -2.4046790890794523e+35, -2.4015206153017863e+35, -2.3983295410919627e+35, -2.3951779538913402e+35, -2.3921244825360646e+35, -2.3892148989264995e+35, -2.386482387979716e+35, -2.383947359380526e+35, -2.381616817612876e+35, -2.3794835813894914e+35, -2.377526010709378e+35, -2.3757091396449664e+35, -2.373987967592598e+35, -2.3723130571707893e+35, -2.370637748100659e+35, -2.3689256397771084e+35, -2.367156861734104e+35, -2.3653320808006697e+35, -2.363473918173627e+35, -2.361626094906903e+35, -2.359850945189692e+35, -2.358225913997539e+35, -2.356839434532311e+35, -2.3557863466507275e+35, -2.3551628996652366e+35, -2.3550614130433353e+35, -2.3555647810101045e+35, -2.356741078944122e+35, -2.3586384702372854e+35, -2.3612804819621186e+35, -2.3646617740305857e+35, -2.368745069238956e+35, -2.373460914963931e+35, -2.378712756370788e+35, -2.3843893368914255e+35, -2.3903840603693577e+35, -2.396617364130421e+35, -2.4030553983408112e+35, -2.4097183855548563e+35, -2.416675273491951e+35, -2.4240260072298215e+35, -2.4318765632387967e+35, -2.4403134381294047e+35, -2.4493837341556247e+35, -2.4590852630735206e+35, -2.4693688855380065e+35, -2.4801527434181337e+35, -2.4913451665349688e+35, -2.5028704277746628e+35, -2.514690368065022e+35, -2.526816201594789e+35, -2.539308277048254e+35, -2.552265537432215e+35, -2.565808930357814e+35, -2.5800632194030505e+35, -2.595140140044097e+35, -2.6111239187916168e+35, -2.628058858426757e+35, -2.6459383633757683e+35, -2.6646953007809188e+35, -2.6841945947136823e+35, -2.7042300087173295e+35, -2.7245277531889985e+35, -2.7447594323169313e+35, -2.7645655989273055e+35, -2.783588822538445e+35, -2.8015122555056477e+35, -2.8180972781054555e+35, -2.833213038481376e+35, -2.8468520679844547e+35, -2.8591291616235487e+35, -2.870264248498553e+35, -2.880552984453005e+35, -2.890330696692459e+35, -2.899935900895323e+35, -2.909678876373741e+35, -2.919818906822887e+35, -2.930551311062649e+35, -2.9420031322765113e+35, -2.9542350672154707e+35, -2.9672471239182092e+35, -2.980986216027001e+35, -2.995354767421771e+35, -3.010219920590659e+35, -3.0254230498679913e+35, -3.0407892533963986e+35, -3.0561366582470566e+35, -3.0712858030231505e+35, -3.0860698093945337e+35, -3.100346095678625e+35, -3.114009743146202e+35, -3.127007405944583e+35, -3.1393493353359716e+35, -3.1511162666327696e+35, -3.1624580402269257e+35, -3.1735820788510413e+35, -3.184732149565274e+35, -3.196160820055627e+35, -3.208101800261444e+35, -3.2207495473383684e+35, -3.2342517712219374e+35, -3.2487156911335988e+35, -3.2642229343657862e+35, -3.280843967906882e+35, -3.2986432004509042e+35, -3.3176701588671933e+35, -3.337938039382379e+35, -3.359395833276897e+35, -3.38190279020716e+35, -3.4052141197909615e+35, -3.428984810852909e+35, -3.452794464700298e+35, -3.476190630976818e+35, -3.4987426944542835e+35, -3.52009490526126e+35, -3.5400072535020566e+35, -3.5583766095842365e+35, -3.5752362379519385e+35, -3.590737065933783e+35, -3.6051171345312594e+35, -3.618665963626157e+35, -3.6316889141259316e+35, -3.644474528921224e+35, -3.657266574820227e+35, -3.670242386774483e+35, -3.683499451161584e+35, -3.69705198251253e+35, -3.710837950682866e+35, -3.7247347379085036e+35, -3.7385793021050535e+35, -3.752187771692803e+35, -3.7653706294009644e+35, -3.777942528749318e+35, -3.789728702304668e+35, -3.8005713347151245e+35, -3.8103387211475755e+35, -3.8189381221943914e+35, -3.826331009045801e+35, -3.832547758549386e+35, -3.8376982498092716e+35, -3.841975189167517e+35, -3.845647981048522e+35, -3.849046195554757e+35, -3.852533060488805e+35, -3.856471069431616e+35, -3.8611837668979705e+35, -3.8669195614155746e+35, -3.873824086154587e+35, -3.881926368596384e+35, -3.891140955260867e+35, -3.901284334886111e+35, -3.912101161122554e+35, -3.9232949528222326e+35, -3.9345589694459736e+35, -3.945604698574587e+35, -3.956186696715801e+35, -3.966122937888219e+35, -3.975309642884854e+35, -3.983729379731046e+35, -3.991451467420054e+35, -3.998624435057818e+35, -4.005461208601658e+35, -4.01221838682826e+35, -4.019171119665757e+35, -4.026584792263835e+35, -4.034684460290258e+35, -4.043623449606882e+35, -4.053454053798421e+35, -4.064105373879008e+35, -4.075374762334053e+35, -4.086938520763833e+35, -4.0983836678832676e+35, -4.109256591354907e+35, -4.1191187604746536e+35, -4.127597403636245e+35, -4.1344215564570214e+35, -4.139439677728243e+35, -4.142620596248264e+35, -4.144041998157541e+35, -4.1438698741989275e+35, -4.14233068969044e+35, -4.139677995909178e+35, -4.136157148818775e+35, -4.131973919033866e+35, -4.127272880757301e+35, -4.122128880234102e+35, -4.116550816640759e+35, -4.110493549404078e+35, -4.1038727124533084e+35, -4.096578918517587e+35, -4.088490956720051e+35, -4.079489942644363e+35, -4.069476460726693e+35, -4.0583906911273494e+35, -4.0462327552861335e+35, -4.0330785714393755e+35, -4.019086217955801e+35, -4.004489263597154e+35, -3.989576404584476e+35, -3.9746604026442935e+35, -3.9600427149808146e+35, -3.945981979017057e+35, -3.932673471716243e+35, -3.9202426964099526e+35, -3.908750929916394e+35, -3.898206589070233e+35, -3.8885757923550716e+35, -3.8797885178917246e+35, -3.871741142605346e+35, -3.864299118033197e+35, -3.857303514147316e+35, -3.850582495284213e+35, -3.843965253303932e+35, -3.837293613266145e+35, -3.8304268746831616e+35, -3.82323858223608e+35, -3.815608517865234e+35, -3.8074169919071055e+35, -3.798549257777253e+35, -3.788914536637059e+35, -3.7784775484470765e+35, -3.767293240887355e+35, -3.755531060130809e+35, -3.74347623784054e+35, -3.731502331796833e+35, -3.720019009818654e+35, -3.709407786753489e+35, -3.6999627919536236e+35, -3.691852174078531e+35, -3.6851088046688585e+35, -3.679648721482204e+35, -3.675306280600045e+35, -3.671870722920699e+35, -3.669111939690449e+35, -3.666791722451656e+35, -3.6646659357233205e+35, -3.662488251376164e+35, -3.660025056047765e+35, -3.6570843255242264e+35, -3.6535515472937124e+35, -3.649418075516284e+35, -3.644786686650039e+35, -3.6398472080812556e+35, -3.6348277286252666e+35, -3.629936407433027e+35, -3.625310344647374e+35, -3.620983139149893e+35, -3.616877125334492e+35, -3.612822321369473e+35, -3.608599328161143e+35, -3.603995922929252e+35, -3.598860840429969e+35, -3.5931394133997146e+35, -3.586884696534239e+35, -3.5802477023377144e+35, -3.573454875843478e+35, -3.56677950183811e+35, -3.5605105026103006e+35, -3.554920581822808e+35, -3.550236274440278e+35, -3.546613522024043e+35, -3.544122459564978e+35, -3.5427439637022174e+35, -3.5423787552632095e+35, -3.542868020082692e+35, -3.544022884686859e+35, -3.5456589149051206e+35, -3.5476314036452925e+35, -3.549867577888473e+35, -3.552392370539913e+35, -3.5553441493345076e+35, -3.558975469131103e+35, -3.5636327417444824e+35, -3.569710351670678e+35, -3.577581423206786e+35, -3.5875181105389205e+35, -3.599623373925359e+35, -3.6137966510819306e+35, -3.6297449203977425e+35, -3.647033325320172e+35, -3.665155324486796e+35, -3.6835985505082495e+35, -3.701889710380751e+35, -3.719614388793662e+35, -3.736418125173584e+35, -3.751998763790289e+35, -3.766096891039631e+35, -3.778485630932089e+35, -3.788958573880932e+35, -3.797317173727637e+35, -3.8033635120066354e+35, -3.806905730241618e+35, -3.807779379643924e+35, -3.805880990199714e+35, -3.801204804469331e+35, -3.793872219872073e+35, -3.784145505755106e+35, -3.77242127751887e+35, -3.7592037133753806e+35, -3.7450614575273026e+35, -3.7305747548321796e+35, -3.7162804894297745e+35, -3.702622805368707e+35, -3.6899161744146616e+35, -3.678326243587365e+35, -3.667871454006175e+35, -3.658445022086211e+35, -3.649852525287137e+35, -3.6418563611527745e+35, -3.6342170161230125e+35, -3.626723679024186e+35, -3.6192119656961954e+35, -3.6115712762351416e+35, -3.603746109076084e+35, -3.595734312249432e+35, -3.587582506486015e+35, -3.5793769422690924e+35, -3.571228123559244e+35, -3.563249521940221e+35, -3.555533422650397e+35, -3.54812888646297e+35, -3.541026945120621e+35, -3.534156358383525e+35, -3.527390318700869e+35, -3.520561605286862e+35, -3.51348192062403e+35, -3.505960973113016e+35, -3.497822125265931e+35, -3.4889134637281876e+35, -3.479115139399638e+35, -3.468345019612185e+35, -3.4565645954341956e+35, -3.443785595956766e+35, -3.430075293356978e+35, -3.415556077843412e+35, -3.4003941409200696e+35, -3.384774573826511e+35, -3.368866016950744e+35, -3.352784728018081e+35, -3.336571193840591e+35, -3.3201890792382138e+35, -3.3035476228168414e+35, -3.2865395280481597e+35, -3.269081648234051e+35, -3.251146556163001e+35, -3.2327776311949858e+35, -3.214086119276115e+35, -3.195233952592156e+35, -3.176409742801993e+35, -3.1578062799322367e+35, -3.1396056442696397e+35, -3.121973525740406e+35, -3.105059655567033e+35, -3.08899857012415e+35, -3.07390522072295e+35, -3.059862814355675e+35, -3.0469045994506197e+35, -3.0349956153777557e+35, -3.024022806399851e+35, -3.0138005862993996e+35, -3.0040937059766097e+35, -2.9946523437557022e+35, -2.9852492677668724e+35, -2.9757082153036197e+35, -2.965916303509847e+35, -2.9558193463812864e+35, -2.9454047779275846e+35, -2.9346802554665785e+35, -2.9236559091075022e+35, -2.91233522286806e+35, -2.900715500822016e+35, -2.888795601324914e+35, -2.8765869608308756e+35, -2.8641237539080355e+35, -2.851468837605409e+35, -2.8387134637901958e+35, -2.8259703086395626e+35, -2.8133608878032043e+35, -2.800999613371713e+35, -2.7889773913879633e+35, -2.7773476928420204e+35, -2.7661175265142008e+35, -2.755244826825338e+35, -2.7446425893859735e+35, -2.7341888279626737e+35, -2.7237403504034873e+35, -2.7131477571104625e+35, -2.702269165774877e+35, -2.690980944844179e+35, -2.6791849117943854e+35, -2.666812576635454e+35, -2.6538276763099546e+35, -2.6402282399054064e+35, -2.6260487933993876e+35, -2.611362322041952e+35, -2.5962806588082108e+35, -2.580951483191045e+35, -2.56555040915284e+35, -2.5502677672768253e+35, -2.5352913405329592e+35, -2.5207878902054643e+35, -2.5068871465208406e+35, -2.493671653046574e+35, -2.481174548864413e+35, -2.4693855679667526e+35, -2.4582638585531788e+35, -2.4477550874103064e+35, -2.437809766208132e+35, -2.4283996575213004e+35, -2.419529324614704e+35, -2.4112403857887465e+35, -2.4036070057852112e+35, -2.396722786000822e+35, -2.390681387564569e+35, -2.3855553188599352e+35, -2.3813783598592786e+35, -2.3781362917539327e+35, -2.3757680074289706e+35, -2.3741757605233825e+35, -2.3732407542957834e+35, -2.3728394598469455e+35, -2.372856950943204e+35, -2.3731953272170323e+35, -2.3737770142978894e+35, -2.374543811648434e+35, -2.3754529409822277e+35, -2.3764712759923276e+35, -2.377568713139748e+35, -2.3787114815257616e+35, -2.3798561608024745e+35, -2.380945250701264e+35, -2.3819052298106546e+35, -2.3826480478225677e+35, -2.383076802789454e+35, -2.383095864421087e+35, -2.3826248692889867e+35, -2.3816149025797102e+35, -2.3800640371258607e+35, -2.378028633329968e+35, -2.3756268566024137e+35, -2.373031986923773e+35, -2.370455122954768e+35, -2.36811925624664e+35, -2.3662286720452044e+35, -2.3649386675606263e+35, -2.364330545231689e+35, -2.364396033102791e+35, -2.3650341415933993e+35, -2.3660621857505826e+35, -2.367241042591979e+35, -2.3683122095592425e+35, -2.3690408723521883e+35, -2.369256086634365e+35, -2.368878348385466e+35, -2.3679277693436978e+35, -2.36651212618967e+35, -2.364800333628315e+35, -2.362990255617844e+35, -2.361279039642651e+35, -2.3598406272140168e+35, -2.3588111726260212e+35, -2.3582806515867455e+35, -2.3582884986527757e+35, -2.358822089810112e+35, -2.3598182011558045e+35, -2.3611682733173096e+35, -2.362727987313504e+35, -2.3643305579502073e+35, -2.3658019537527784e+35, -2.366975638005098e+35, -2.36770468339045e+35, -2.3678700209448315e+35, -2.36738462219453e+35, -2.366194136119491e+35, -2.3642747817312018e+35, -2.361629280526661e+35, -2.3582815261657504e+35, -2.3542707080185513e+35, -2.349645834400752e+35, -2.3444619915122938e+35, -2.3387798971387056e+35, -2.3326698552670906e+35, -2.3262198221613214e+35, -2.319545309474555e+35, -2.312797161435841e+35, -2.3061627827556798e+35, -2.2998576552760764e+35, -2.294106756130435e+35, -2.28911883429054e+35, -2.2850590642382348e+35, -2.282026355223152e+35, -2.2800404715168885e+35, -2.279041781177643e+35, -2.2789036371170688e+35, -2.279454681915711e+35, -2.2805064190927852e+35, -2.281880854050938e+35, -2.283433930203291e+35, -2.28507213204395e+35, -2.2867610801592522e+35, -2.2885258446597827e+35, -2.2904433200764007e+35, -2.2926277151279663e+35, -2.2952110748216244e+35, -2.2983215178990768e+35, -2.3020622363615163e+35, -2.306494111959015e+35, -2.311624086711316e+35, -2.317400341521427e+35, -2.323714136441966e+35, -2.3304071540351994e+35, -2.337282664601681e+35, -2.3441189724413452e+35, -2.35068431315801e+35, -2.3567532367095367e+35, -2.3621249108011426e+35, -2.3666432102455034e+35, -2.3702169075199982e+35, -2.372836414531385e+35, -2.3745824753480565e+35, -2.375622931034167e+35, -2.3761962153553817e+35, -2.3765834882310933e+35, -2.3770736867305875e+35, -2.3779263830803237e+35, -2.3793366361969062e+35, -2.3814052903468697e+35, -2.3841183612059976e+35, -2.3873400371670495e+35, -2.390824072082592e+35, -2.3942463593515295e+35, -2.3972566589697155e+35, -2.3995410755649947e+35, -2.400881789440612e+35, -2.401199531803064e+35, -2.400568357131249e+35, -2.399199904469049e+35, -2.3974024742454997e+35, -2.39552591821387e+35, -2.3939050795766785e+35, -2.392812692782764e+35, -2.392428776345776e+35, -2.3928292025599654e+35, -2.3939922484517525e+35, -2.3958189554373994e+35, -2.39816144148604e+35, -2.4008532786969096e+35, -2.4037374739198187e+35, -2.406689461303777e+35, -2.4096336244877984e+35, -2.4125517712813014e+35, -2.4154813170295493e+35, -2.4185008388836715e+35, -2.4217021177267666e+35, -2.4251512676001552e+35, -2.4288466675127253e+35, -2.4326862169033888e+35, -2.4364575083825656e+35, -2.4398589875428355e+35, -2.4425489134556927e+35, -2.4442074692619185e+35, -2.4445924197172945e+35, -2.4435729244437253e+35, -2.441136191345443e+35, -2.4373715021958035e+35, -2.432441500575279e+35, -2.4265509591175705e+35, -2.4199203939038395e+35, -2.4127679904815768e+35, -2.4052997264301672e+35, -2.3977050682723675e+35, -2.390154607130354e+35, -2.3827965582886112e+35, -2.3757507949877555e+35, -2.369101241826237e+35, -2.362889165088209e+35, -2.35711053318336e+35, -2.3517199139975366e+35, -2.3466414706712958e+35, -2.3417850903908296e+35, -2.3370634901448673e+35, -2.332405334147081e+35, -2.32776055754289e+35, -2.323096884318012e+35, -2.3183897576315177e+35, -2.313610176933991e+35, -2.308715440854592e+35, -2.3036465539985627e+35, -2.2983336439580024e+35, -2.292707898063979e+35, -2.286716102275056e+35, -2.280332763920788e+35, -2.273565657264963e+35, -2.2664531921471337e+35, -2.2590550361141844e+35, -2.2514394714649077e+35, -2.243671346945504e+35, -2.235803499711184e+35, -2.2278729700074692e+35, -2.2199018844998423e+35, -2.2119018899683966e+35, -2.203880565871102e+35, -2.1958482601971872e+35, -2.1878240951334387e+35, -2.1798402427835604e+35, -2.171943787830682e+35, -2.1641955333551325e+35, -2.1566651285842994e+35, -2.1494222159142714e+35, -2.1425241993426383e+35, -2.136002730883137e+35, -2.1298525918094485e+35, -2.124027340602994e+35, -2.1184449425543556e+35, -2.1130034099115218e+35, -2.107602421262364e+35, -2.1021640628107666e+35, -2.096645964797355e+35, -2.0910432247698493e+35, -2.085379896209201e+35, -2.0796942750476375e+35, -2.074023458900705e+35, -2.068391746246421e+35, -2.0628052670244712e+35, -2.0572528250284504e+35, -2.051711130403973e+35, -2.046151909928265e+35, -2.040548788226496e+35, -2.034882781824942e+35, -2.0291460337503753e+35, -2.0233437223509625e+35, -2.0174941035198952e+35, -2.011626711877158e+35, -2.005778930579506e+35, -1.9999913457716576e+35, -1.994302508226351e+35, -1.98874397799888e+35, -1.983336742640625e+35, -1.978090023215098e+35, -1.9730029360067487e+35, -1.968068606304987e+35, -1.963279569458551e+35, -1.958633060111052e+35, -1.954135171502111e+35, -1.949803568530338e+35, -1.9456689974632966e+35, -1.9417759161854614e+35, -1.9381821501132064e+35, -1.9349568513109187e+35, -1.932175651132693e+35, -1.9299121363436902e+35, -1.9282257723634605e+35, -1.9271479091834658e+35, -1.9266689671577648e+35, -1.92673059403998e+35, -1.9272259626221266e+35, -1.9280093965870407e+35, -1.928913788549177e+35, -1.9297718994592274e+35, -1.930436658573753e+35, -1.930796462403883e+35, -1.930783719858711e+35, -1.930377287594857e+35, -1.9296006917670546e+35, -1.928517630422799e+35, -1.927224918834026e+35, -1.9258422561746875e+35, -1.924498947865119e+35, -1.9233195764820635e+35, -1.9224120523576325e+35, -1.921861204241027e+35, -1.9217290193885205e+35, -1.9220598468048937e+35, -1.9228867169182978e+35, -1.9242344571789757e+35, -1.9261168657638968e+35, -1.9285283718386557e+35, -1.931434049066645e+35, -1.93476378388398e+35, -1.9384154973356455e+35, -1.942268725060751e+35, -1.946205399864921e+35, -1.9501317106127764e+35, -1.9539947137340703e+35, -1.95778953786424e+35, -1.961556108881515e+35, -1.9653669472593907e+35, -1.9693090750467326e+35, -1.9734635000990416e+35, -1.9778856468793458e+35, -1.9825899858639366e+35, -1.98754204084033e+35, -1.9926603882815945e+35, -1.9978294745392727e+35, -2.002921014218441e+35, -2.007818596949429e+35, -2.0124387909753264e+35, -2.016743544322236e+35, -2.0207423038971472e+35, -2.0244860161865197e+35, -2.028057281745322e+35, -2.031560793395164e+35, -2.0351162166936545e+35, -2.0388529374659056e+35, -2.0429040712570694e+35, -2.0473970706341024e+35, -2.05244026922987e+35, -2.0581074540898917e+35, -2.064424397160637e+35, -2.071361385391683e+35, -2.0788343822060175e+35, -2.08671527405589e+35, -2.0948496059766562e+35, -2.103079022160601e+35, -2.1112652704706496e+35, -2.119312311715032e+35, -2.1271825459388667e+35, -2.1349035740742534e+35, -2.1425642690022144e+35, -2.150302204488148e+35, -2.1582864858180464e+35, -2.1666998345719367e+35, -2.1757223485570755e+35, -2.1855179949867285e+35, -2.1962241044930645e+35, -2.2079437634847925e+35, -2.220740838794968e+35, -2.2346373591065612e+35, -2.2496131114193972e+35, -2.2656074964301148e+35, -2.28252377695123e+35, -2.3002357476532646e+35, -2.3185965631257465e+35, -2.3374490899793066e+35, -2.3566368333994346e+35, -2.3760143351138025e+35, -2.395455987616377e+35, -2.4148624234343983e+35, -2.4341639381015804e+35, -2.4533207226496598e+35, -2.472319992368678e+35, -2.4911704056826004e+35, -2.5098944593517002e+35, -2.5285197912697796e+35, -2.5470704868066068e+35, -2.565559548571457e+35, -2.5839836335936004e+35, -2.6023209563284624e+35, -2.620532872472242e+35, -2.638569105608305e+35, -2.656375928764602e+35, -2.67390599773421e+35, -2.691128103818304e+35, -2.7080349910652817e+35, -2.724647622915656e+35, -2.7410148653333175e+35, -2.7572083804096496e+35, -2.7733134236532284e+35, -2.7894169909904638e+35, -2.8055951782000735e+35, -2.8219016287156143e+35, -2.838358646620303e+35, -2.854952121370697e+35, -2.8716309920133796e+35, -2.8883115894923984e+35, -2.904886746149009e+35, -2.9212389483529158e+35, -2.937256021385942e+35, -2.9528470273202862e+35, -2.967955537956726e+35, -2.9825675786672184e+35, -2.9967125289890502e+35, -3.0104569438545587e+35, -3.023893054317471e+35, -3.037124904973963e+35, -3.050255247680168e+35, -3.0633755247206872e+35, -3.076560038647695e+35, -3.0898642758890614e+35, -3.103326642427535e+35, -3.1169725989101202e+35, -3.1308201900417002e+35, -3.1448860685384056e+35, -3.1591912053462293e+35, -3.173765526444473e+35, -3.1886507487810426e+35, -3.2039007515496016e+35, -3.2195789637954602e+35, -3.235752518311087e+35, -3.2524833274538507e+35, -3.2698167025589658e+35, -3.2877684527298588e+35, -3.306311304803801e+35, -3.3253609984349975e+35, -3.344762141461764e+35, -3.364274877846312e+35, -3.383566005172269e+35, -3.402210956848148e+35, -3.41971311928319e+35, -3.43554235035457e+35, -3.449186937106728e+35, -3.460207240496707e+35, -3.468278745662948e+35, -3.4732169212193995e+35, -3.47498264479242e+35, -3.47367152096084e+35, -3.4694921067292256e+35, -3.4627377890251375e+35, -3.453756189008918e+35, -3.442919218484412e+35, -3.430596283001391e+35, -3.4171323744161655e+35, -3.4028318820475446e+35, -3.387948107684089e+35, -3.3726779677521224e+35, -3.357161265073059e+35, -3.341484019588537e+35, -3.3256853939302636e+35, -3.3097676012330102e+35, -3.2937079016844764e+35, -3.2774715420907763e+35, -3.2610243880306536e+35, -3.244344054660998e+35, -3.227428502058014e+35, -3.2103012627408152e+35, -3.1930126908545258e+35, -3.1756368924384333e+35, -3.1582643845846668e+35, -3.1409911235648914e+35, -3.123905371563422e+35, -3.1070748130385433e+35, -3.090537017794691e+35, -3.074296232941246e+35, -3.058328143689792e+35, -3.0425917967804364e+35, -3.0270452211384274e+35, -3.0116597242199853e+35, -2.9964283104914886e+35, -2.981366004696596e+35, -2.9665029089185596e+35, -2.9518732020349958e+35, -2.937504210769178e+35, -2.923409157749015e+35, -2.9095857149923485e+35, -2.896020610175602e+35, -2.88269869503511e+35, -2.869613498369523e+35, -2.8567757542901254e+35, -2.8442169705250995e+35, -2.8319866451830133e+35, -2.8201436686794183e+35, -2.8087439949673515e+35, -2.7978273334157493e+35, -2.7874054294926765e+35, -2.7774538816800956e+35, -2.7679087775524453e+35, -2.758668817844027e+35, -2.7496028893971048e+35, -2.7405621483170422e+35, -2.7313947433346106e+35, -2.7219607168090753e+35, -2.7121446987309825e+35, -2.7018648020026537e+35, -2.6910773206270478e+35, -2.6797779033183777e+35, -2.668000373146327e+35, -2.6558141539387258e+35, -2.6433205869178053e+35, -2.6306477393614113e+35, -2.6179430357425606e+35, -2.6053633267816307e+35, -2.5930626897216422e+35, -2.5811789948585734e+35, -2.569820791819216e+35, -2.5590562449730035e+35, -2.5489057165071167e+35, -2.5393392500506084e+35, -2.530279703818985e+35, -2.5216116181409664e+35, -2.5131950698995853e+35, -2.5048828410970583e+35, -2.496538425076196e+35, -2.4880520186280045e+35, -2.479351939561363e+35, -2.4704098610865106e+35, -2.4612395736366793e+35, -2.451890242283525e+35, -2.442435987752909e+35, -2.4329639635224963e+35, -2.4235629857449408e+35, -2.414314310364967e+35, -2.4052854609758735e+35, -2.3965272353908786e+35, -2.3880733416201677e+35, -2.37994170781062e+35, -2.3721364575549284e+35, -2.3646497918743207e+35, -2.3574634225319485e+35, -2.3505495872417486e+35, -2.343871925056731e+35, -2.337386557331058e+35, -2.3310436280109817e+35, -2.324789362502249e+35, -2.318568477073792e+35, -2.312326587625577e+35, -2.306012198789965e+35, -2.2995779386705076e+35, -2.2929809177787273e+35, -2.2861823541832576e+35, -2.2791468262460117e+35, -2.27184162547082e+35, -2.264236670461407e+35, -2.256305330167527e+35, -2.2480263222689896e+35, -2.2393866294935645e+35, -2.2303851429716362e+35, -2.221036536890473e+35, -2.21137474539258e+35, -2.201455376654132e+35, -2.1913564549871054e+35, -2.1811770050820475e+35, -2.1710331718693046e+35, -2.161051827962793e+35, -2.1513619928196313e+35, -2.1420848569777983e+35, -2.1333236480848425e+35, -2.1251547832430223e+35, -2.1176215467331768e+35, -2.1107309197886167e+35, -2.1044534131241838e+35, -2.098725169901345e+35, -2.0934514644601937e+35, -2.0885110142141983e+35, -2.0837610171917416e+35, -2.0790432147206557e+35, -2.0741913229333853e+35, -2.0690398417231996e+35, -2.063433759382837e+35, -2.0572384120949107e+35, -2.05034895634369e+35, -2.0426993037488215e+35, -2.0342703011600965e+35, -2.0250960350224726e+35, -2.015265898205907e+35, -2.004919673241711e+35, -1.9942342745496924e+35, -1.9834036490378257e+35, -1.9726160982223143e+35, -1.9620342215992983e+35, -1.9517812691048016e+35, -1.9419350154165398e+35, -1.9325280217862366e+35, -1.9235522771940402e+35, -1.91496647364458e+35, -1.906704761097336e+35, -1.898686169072158e+35, -1.89082391766772e+35, -1.8830338042437417e+35, -1.8752409393798737e+35, -1.867384348978687e+35, -1.8594192883368388e+35, -1.851317454467475e+35, -1.843065598180396e+35, -1.834663307045076e+35, -1.8261208991100854e+35, -1.8174583264636488e+35, -1.8087056408569344e+35, -1.799904934640299e+35, -1.791112867735164e+35, -1.782402087283037e+35, -1.7738592899528202e+35, -1.7655779170356634e+35, -1.757645187259356e+35, -1.7501262153387414e+35, -1.7430506193620007e+35, -1.73640701183387e+35, -1.730147527186598e+35, -1.7242000977405668e+35, -1.7184834541475055e+35, -1.7129200903667255e+35, -1.7074447178775897e+35, -1.7020081408781777e+35, -1.6965778008629995e+35, -1.6911364125383798e+35, -1.6856797188868744e+35, -1.680213950985138e+35, -1.674753271341676e+35, -1.6693172662155256e+35, -1.6639283863035642e+35, -1.6586091698995382e+35, -1.6533792082172193e+35, -1.6482521104718878e+35, -1.6432329904472517e+35, -1.638316948408511e+35, -1.6334886013589525e+35, -1.6287222840065463e+35, -1.6239826814853607e+35, -1.6192265313977229e+35, -1.614406909953843e+35, -1.6094812076950078e+35, -1.604421641278394e+35, -1.5992243137310165e+35, -1.5939119692488427e+35, -1.5885281528375554e+35, -1.5831250325010834e+35, -1.577750382508691e+35, -1.5724390560050443e+35, -1.5672113802571376e+35, -1.5620775698075117e+35, -1.557045257575836e+35, -1.5521269850981026e+35, -1.5473453814244716e+35, -1.5427349823552664e+35, -1.5383406394341202e+35, -1.534213062794636e+35, -1.5304023330984261e+35, -1.526950388294184e+35, -1.5238836348900867e+35, -1.5212068989515842e+35, -1.5188998034642642e+35, -1.5169162989698237e+35, -1.515187586056969e+35, -1.5136282026002979e+35, -1.5121446572719254e+35, -1.5106456007998003e+35, -1.5090520927338243e+35, -1.5073061974701085e+35, -1.505376244765761e+35, -1.5032578201948943e+35, -1.500970730352666e+35, -1.498553290581305e+35, -1.4960557969106346e+35, -1.4935348209621416e+35, -1.4910492573426088e+35, -1.4886582551122703e+35, -1.486420561564044e+35, -1.4843944829542312e+35, -1.4826375779828926e+35, -1.4812052734447264e+35, -1.4801477879849812e+35, -1.4795050700898992e+35, -1.4792999155565294e+35, -1.4795300176814948e+35, -1.4801603460279358e+35, -1.4811177828492837e+35, -1.4822901213091152e+35, -1.4835310769399978e+35, -1.4846717464370257e+35, -1.4855371752873568e+35, -1.4859650188921462e+35, -1.4858225322349509e+35, -1.485018741234646e+35, -1.483510293240268e+35, -1.4813012674687946e+35, -1.4784383807962097e+35, -1.4750033521467667e+35, -1.4711039692078694e+35, -1.4668650050762058e+35, -1.4624197545457473e+35, -1.4579026208594668e+35, -1.4534428739281082e+35, -1.4491594748783916e+35, -1.4451568082907967e+35, -1.4415213050173502e+35, -1.438319164361827e+35, -1.435595487681386e+35, -1.4333749470032483e+35, -1.4316636491006048e+35, -1.4304513596747865e+35, -1.4297130568213401e+35, -1.4294090872447416e+35, -1.4294839296116195e+35, -1.4298644276681125e+35, -1.430458985438565e+35, -1.4311593520263627e+35, -1.4318461365615919e+35, -1.4323981105829683e+35, -1.4327039350426993e+35, -1.432673755932145e+35, -1.4322478391366526e+35, -1.4314003995500713e+35, -1.4301385890861956e+35, -1.428498206193244e+35, -1.4265381796501459e+35, -1.4243351796486631e+35, -1.4219785342002712e+35, -1.4195648543965774e+35, -1.4171917983994528e+35, -1.4149510163641135e+35, -1.4129210115118561e+35, -1.4111610346338134e+35, -1.4097070957213653e+35, -1.4085708357693076e+35, -1.4077415221671859e+35, -1.4071909433464263e+35, -1.406880557981912e+35, -1.4067699384852457e+35, -1.4068253540183139e+35, -1.4070272694059595e+35, -1.4073755938144958e+35, -1.4078916996097519e+35, -1.4086165546953551e+35, -1.409604780112823e+35, -1.410915061475816e+35, -1.4125980846663587e+35, -1.4146839492713417e+35, -1.4171716516969382e+35, -1.4200234224679946e+35, -1.4231661071612686e+35, -1.4265002246997972e+35, -1.4299150965518987e+35, -1.4333063543515345e+35, -1.4365912702367176e+35, -1.4397182925250017e+35, -1.4426694717662423e+35, -1.4454569268241173e+35, -1.4481160051105342e+35, -1.4506979140990021e+35, -1.4532635977558749e+35, -1.4558791068914585e+35, -1.4586113888660942e+35, -1.4615229376288131e+35, -1.464664363117853e+35, -1.46806535864958e+35, -1.4717260333800136e+35, -1.4756113431322803e+35, -1.4796509261128849e+35, -1.483745123664126e+35, -1.48777601383463e+35, -1.4916208426793647e+35, -1.495164945675027e+35, -1.4983120652189423e+35, -1.5009912756355049e+35, -1.5031608253462333e+35, -1.5048097219239304e+35, -1.5059578405796769e+35, -1.5066549758866454e+35, -1.5069788534049246e+35, -1.5070318628899957e+35, -1.5069362461463178e+35, -1.506827648538551e+35, -1.5068472355062677e+35, -1.5071328758745221e+35, -1.5078101192912145e+35, -1.5089838116521333e+35, -1.5107312045759064e+35, -1.513097333866505e+35, -1.5160932619914545e+35, -1.5196974907247282e+35, -1.5238604729392409e+35, -1.5285117554424936e+35, -1.533568951635447e+35, -1.5389475173682004e+35, -1.5445701704965021e+35, -1.5503747261632997e+35, -1.556319140378409e+35, -1.5623827541936408e+35, -1.5685632014865078e+35, -1.5748691872893275e+35, -1.5813102262999299e+35, -1.587885212633205e+35, -1.5945721152075871e+35, -1.6013209848653835e+35, -1.6080518018887217e+35, -1.6146576466879695e+35, -1.6210125415815456e+35, -1.6269824253153752e+35, -1.6324373266223621e+35, -1.6372629405978825e+35, -1.6413703105616369e+35, -1.6447028924042442e+35, -1.6472406867117696e+35, -1.649001294860323e+35, -1.650037801839401e+35, -1.6504335039537693e+35, -1.6502938057175058e+35, -1.6497360665353567e+35, -1.6488786126421495e+35, -1.6478303616350907e+35, -1.6466824537931641e+35, -1.645502983346418e+35, -1.6443354675712152e+35, -1.6432011599158045e+35, -1.6421047482021044e+35, -1.6410424234598604e+35, -1.6400108438660057e+35, -1.6390152769231058e+35, -1.6380752940037988e+35, -1.637226842144968e+35, -1.6365202327350202e+35, -1.636014379530461e+35, -1.6357683031334094e+35, -1.6358313884809858e+35, -1.6362341081053762e+35, -1.6369808841029429e+35, -1.638046382971606e+35, -1.6393757601158886e+35, -1.6408883123401622e+35, -1.6424830465097508e+35, -1.6440443440360627e+35, -1.6454464763948557e+35, -1.6465569780613899e+35, -1.6472401350385061e+35, -1.6473623692852193e+35, -1.6468007373754663e+35, -1.6454543279681301e+35, -1.6432566949847306e+35, -1.6401863811268513e+35, -1.6362726078731272e+35, -1.6315943794897857e+35, -1.6262730979614865e+35, -1.620460548554338e+35, -1.6143250994687744e+35, -1.608038849209528e+35, -1.6017674214546333e+35, -1.5956626976196252e+35, -1.5898576630575633e+35, -1.5844621970932949e+35, -1.5795590951398665e+35, -1.5752004758028354e+35, -1.5714054362417251e+35, -1.5681599900054963e+35, -1.5654199248668246e+35, -1.563116527138165e+35, -1.5611644975380476e+35, -1.5594710591871005e+35, -1.5579452099109677e+35, -1.5565061083007102e+35, -1.5550895629862713e+35, -1.5536515965270063e+35, -1.552168337613277e+35, -1.5506322395572768e+35, -1.5490456787274902e+35, -1.5474138674111687e+35, -1.5457391988640332e+35, -1.544018446586768e+35, -1.5422429757791706e+35, -1.540400911872313e+35, -1.5384796193972166e+35, -1.5364671051364145e+35, -1.5343518761558741e+35, -1.5321218488754974e+35, -1.5297635545471505e+35, -1.5272627816887312e+35, -1.5246070117666613e+35, -1.5217889923176146e+35, -1.5188101154085919e+35, -1.515682265337256e+35, -1.5124273958224129e+35, -1.509074912004896e+35, -1.5056575551677445e+35, -1.5022067157950235e+35, -1.4987479873434561e+35, -1.495297511906258e+35, -1.4918594380888405e+35, -1.4884246865558862e+35, -1.4849711709859673e+35, -1.4814655684816786e+35, -1.4778665990405334e+35, -1.4741295397923488e+35, -1.4702114199458732e+35, -1.4660761206591956e+35, -1.461698544128226e+35, -1.4570671707407316e+35, -1.4521846683065644e+35, -1.447066662587131e+35, -1.441739199027401e+35, -1.4362357000322575e+35, -1.4305942620554154e+35, -1.4248559110229341e+35, -1.41906398784859e+35, -1.4132642963892442e+35, -1.4075052087222782e+35, -1.401836791087109e+35, -1.3963083101402384e+35, -1.3909641540760702e+35, -1.3858390109175146e+35, -1.3809537279724978e+35, -1.3763133320757756e+35, -1.3719081444793552e+35, -1.3677179777960927e+35, -1.3637184314135112e+35, -1.3598876699597483e+35, -1.3562119653998194e+35, -1.3526886638554423e+35, -1.3493258887894928e+35, -1.3461389605249237e+35, -1.343144042516093e+35, -1.3403499204167975e+35, -1.337749192124406e+35, -1.3353105690527414e+35, -1.3329743406164634e+35, -1.3306529920000709e+35, -1.3282381263544326e+35, -1.3256132047902403e+35, -1.322669715806675e+35, -1.319323092367306e+35, -1.3155246648763865e+35, -1.31126718015473e+35, -1.3065833353257188e+35, -1.301538584204815e+35, -1.296220600866086e+35, -1.2907280395319671e+35, -1.285160745769821e+35, -1.2796126681216281e+35, -1.2741677509462135e+35, -1.2688983444731316e+35, -1.2638652834978467e+35, -1.2591187360404704e+35, -1.2546990776590214e+35, -1.2506372689714717e+35, -1.2469544359630146e+35, -1.2436605859474256e+35, -1.2407526691865772e+35, -1.2382125006201057e+35, -1.2360053020010622e+35, -1.2340797033801585e+35, -1.2323698924430245e+35, -1.230800239280765e+35, -1.2292922318724652e+35, -1.227773030767296e+35, -1.2261844729240197e+35, -1.224490984854047e+35, -1.2226846575670427e+35, -1.2207857788723407e+35, -1.2188375858245867e+35, -1.2168951215927815e+35, -1.2150099206248756e+35, -1.2132143282626105e+35, -1.2115104323392766e+35, -1.209867606263119e+35, -1.2082294122365581e+35, -1.2065267600992722e+35, -1.2046922583993703e+35, -1.202671853949515e+35, -1.200432921011689e+35, -1.197970356984469e+35, -1.1953122750254164e+35, -1.1925249642733887e+35, -1.1897147173428823e+35, -1.1870234669658495e+35, -1.1846162618290355e+35]
tiempo_s_1emin3 = [0.0, 8.005538284799257e-07, 1.6011076569602252e-06, 2.4016614854399944e-06, 3.2022153139193746e-06, 4.0027691423982775e-06, 4.803322970875625e-06, 5.603876799352972e-06, 6.40443062783032e-06, 7.204984456307667e-06, 8.005538284785015e-06, 8.806092113262362e-06, 9.60664594173971e-06, 1.0407199770217057e-05, 1.1207753598694404e-05, 1.2008307427171752e-05, 1.28088612556491e-05, 1.3609415084126447e-05, 1.4409968912603794e-05, 1.5210522741081141e-05, 1.601107656956613e-05, 1.681163039805161e-05, 1.7612184226537088e-05, 1.8412738055022567e-05, 1.9213291883508046e-05, 2.0013845711993525e-05, 2.0814399540479004e-05, 2.1614953368964483e-05, 2.241550719744996e-05, 2.321606102593544e-05, 2.401661485442092e-05, 2.4817168682906398e-05, 2.5617722511391877e-05, 2.6418276339877356e-05, 2.7218830168362835e-05, 2.8019383996848313e-05, 2.8819937825333792e-05, 2.962049165381927e-05, 3.042104548230475e-05, 3.1221599310804534e-05, 3.2022153139306275e-05, 3.282270696780802e-05, 3.362326079630976e-05, 3.44238146248115e-05, 3.522436845331324e-05, 3.6024922281814985e-05, 3.682547611031673e-05, 3.762602993881847e-05, 3.842658376732021e-05, 3.922713759582195e-05, 4.0027691424323694e-05, 4.0828245252825436e-05, 4.162879908132718e-05, 4.242935290982892e-05, 4.322990673833066e-05, 4.4030460566832404e-05, 4.4831014395334146e-05, 4.563156822383589e-05, 4.643212205233763e-05, 4.723267588083937e-05, 4.803322970934111e-05, 4.8833783537842855e-05, 4.96343373663446e-05, 5.043489119484634e-05, 5.123544502334808e-05, 5.203599885184982e-05, 5.2836552680351565e-05, 5.3637106508853307e-05, 5.443766033735505e-05, 5.523821416585679e-05, 5.603876799435853e-05, 5.6839321822860274e-05, 5.7639875651362016e-05, 5.844042947986376e-05, 5.92409833083655e-05, 6.004153713686724e-05, 6.0842090965368984e-05, 6.164264479387073e-05, 6.244319862237247e-05, 6.324375245087421e-05, 6.404430627937595e-05, 6.484486010787769e-05, 6.564541393637943e-05, 6.644596776488118e-05, 6.724652159338292e-05, 6.804707542188466e-05, 6.88476292503864e-05, 6.964818307888814e-05, 7.044873690738989e-05, 7.124929073589163e-05, 7.204984456439337e-05, 7.285039839289511e-05, 7.365095222139685e-05, 7.44515060498986e-05, 7.525205987840034e-05, 7.605261370690208e-05, 7.685316753540382e-05, 7.765372136390556e-05, 7.84542751924073e-05, 7.925482902090905e-05, 8.005538284941079e-05, 8.085593667791253e-05, 8.165649050641427e-05, 8.245704433491601e-05, 8.325759816341776e-05, 8.40581519919195e-05, 8.485870582042124e-05, 8.565925964892298e-05, 8.645981347742472e-05, 8.726036730592647e-05, 8.806092113442821e-05, 8.886147496292995e-05, 8.966202879143169e-05, 9.046258261993343e-05, 9.126313644843518e-05, 9.206369027693692e-05, 9.286424410543866e-05, 9.36647979339404e-05, 9.446535176244214e-05, 9.526590559094388e-05, 9.606645941944563e-05, 9.686701324794737e-05, 9.766756707644911e-05, 9.846812090495085e-05, 9.92686747334526e-05, 0.00010006922856195434, 0.00010086978239045608, 0.00010167033621895782, 0.00010247089004745956, 0.0001032714438759613, 0.00010407199770446305, 0.00010487255153296479, 0.00010567310536146653, 0.00010647365918996827, 0.00010727421301847001, 0.00010807476684697176, 0.0001088753206754735, 0.00010967587450397524, 0.00011047642833247698, 0.00011127698216097872, 0.00011207753598948046, 0.0001128780898179822, 0.00011367864364648395, 0.00011447919747498569, 0.00011527975130348743, 0.00011608030513198917, 0.00011688085896049092, 0.00011768141278899266, 0.0001184819666174944, 0.00011928252044599614, 0.00012008307427449788, 0.00012088362810299963, 0.00012168418193150137, 0.00012248473575996943, 0.00012328528958840612, 0.0001240858434168428, 0.0001248863972452795, 0.0001256869510737162, 0.00012648750490215288, 0.00012728805873058957, 0.00012808861255902626, 0.00012888916638746295, 0.00012968972021589964, 0.00013049027404433633, 0.00013129082787277302, 0.0001320913817012097, 0.0001328919355296464, 0.0001336924893580831, 0.00013449304318651978, 0.00013529359701495647, 0.00013609415084339316, 0.00013689470467182985, 0.00013769525850026654, 0.00013849581232870323, 0.00013929636615713992, 0.0001400969199855766, 0.0001408974738140133, 0.00014169802764244999, 0.00014249858147088668, 0.00014329913529932337, 0.00014409968912776005, 0.00014490024295619674, 0.00014570079678463343, 0.00014650135061307012, 0.00014730190444150681, 0.0001481024582699435, 0.0001489030120983802, 0.00014970356592681688, 0.00015050411975525357, 0.00015130467358369026, 0.00015210522741212695, 0.00015290578124056364, 0.00015370633506900033, 0.00015450688889743702, 0.0001553074427258737, 0.0001561079965543104, 0.0001569085503827471, 0.00015770910421118378, 0.00015850965803962047, 0.00015931021186805716, 0.00016011076569649385, 0.00016091131952493054, 0.00016171187335336723, 0.00016251242718180392, 0.0001633129810102406, 0.0001641135348386773, 0.000164914088667114, 0.00016571464249555068, 0.00016651519632398737, 0.00016731575015242406, 0.00016811630398086075, 0.00016891685780929744, 0.00016971741163773413, 0.00017051796546617082, 0.0001713185192946075, 0.0001721190731230442, 0.00017291962695148089, 0.00017372018077991758, 0.00017452073460835427, 0.00017532128843679096, 0.00017612184226522765, 0.00017692239609366434, 0.00017772294992210102, 0.00017852350375053771, 0.0001793240575789744, 0.0001801246114074111, 0.00018092516523584778, 0.00018172571906428447, 0.00018252627289272116, 0.00018332682672115785, 0.00018412738054959454, 0.00018492793437803123, 0.00018572848820646792, 0.0001865290420349046, 0.0001873295958633413, 0.000188130149691778, 0.00018893070352021468, 0.00018973125734865137, 0.00019053181117708806, 0.00019133236500552475, 0.00019213291883396144, 0.00019293347266239813, 0.00019373402649083482, 0.0001945345803192715, 0.0001953351341477082, 0.0001961356879761449, 0.00019693624180458158, 0.00019773679563301827, 0.00019853734946145496, 0.00019933790328989165, 0.00020013845711832834, 0.00020093901094676503, 0.00020173956477520172, 0.0002025401186036384, 0.0002033406724320751, 0.0002041412262605118, 0.00020494178008894848, 0.00020574233391738517, 0.00020654288774582186, 0.00020734344157425855, 0.00020814399540269524, 0.00020894454923113193, 0.00020974510305956862, 0.0002105456568880053, 0.000211346210716442, 0.00021214676454487868, 0.00021294731837331537, 0.00021374787220175206, 0.00021454842603018875, 0.00021534897985862544, 0.00021614953368706213, 0.00021695008751549882, 0.0002177506413439355, 0.0002185511951723722, 0.0002193517490008089, 0.00022015230282924558, 0.00022095285665768227, 0.00022175341048611896, 0.00022255396431455565, 0.00022335451814299234, 0.00022415507197142903, 0.00022495562579986572, 0.0002257561796283024, 0.0002265567334567391, 0.0002273572872851758, 0.00022815784111361248, 0.00022895839494204917, 0.00022975894877048586, 0.00023055950259892255, 0.00023136005642735924, 0.00023216061025579593, 0.00023296116408423262, 0.0002337617179126693, 0.000234562271741106, 0.0002353628255695427, 0.00023616337939797938, 0.00023696393322641607, 0.00023776448705485276, 0.00023856504088328945, 0.00023936559471172614, 0.00024016614854016283, 0.00024096670236859952, 0.0002417672561970362, 0.0002425678100254729, 0.00024336836385390959, 0.00024416891768234625, 0.00024496947151078294, 0.00024577002533921963, 0.0002465705791676563, 0.000247371132996093, 0.0002481716868245297, 0.0002489722406529664, 0.0002497727944814031, 0.00025057334830983977, 0.00025137390213827646, 0.00025217445596671315, 0.00025297500979514984, 0.0002537755636235865, 0.0002545761174520232, 0.0002553766712804599, 0.0002561772251088966, 0.0002569777789373333, 0.00025777833276577, 0.00025857888659420666, 0.00025937944042264335, 0.00026017999425108004, 0.00026098054807951673, 0.0002617811019079534, 0.0002625816557363901, 0.0002633822095648268, 0.0002641827633932635, 0.0002649833172217002, 0.00026578387105013687, 0.00026658442487857356, 0.00026738497870701025, 0.00026818553253544694, 0.00026898608636388363, 0.0002697866401923203, 0.000270587194020757, 0.0002713877478491937, 0.0002721883016776304, 0.0002729888555060671, 0.00027378940933450377, 0.00027458996316294046, 0.00027539051699137715, 0.00027619107081981384, 0.00027699162464825053, 0.0002777921784766872, 0.0002785927323051239, 0.0002793932861335606, 0.0002801938399619973, 0.000280994393790434, 0.00028179494761887067, 0.00028259550144730736, 0.00028339605527574405, 0.00028419660910418074, 0.0002849971629326174, 0.0002857977167610541, 0.0002865982705894908, 0.0002873988244179275, 0.0002881993782463642, 0.0002889999320748009, 0.00028980048590323756, 0.00029060103973167425, 0.00029140159356011094, 0.00029220214738854763, 0.0002930027012169843, 0.000293803255045421, 0.0002946038088738577, 0.0002954043627022944, 0.0002962049165307311, 0.00029700547035916777, 0.00029780602418760446, 0.00029860657801604115, 0.00029940713184447784, 0.00030020768567291453, 0.0003010082395013512, 0.0003018087933297879, 0.0003026093471582246, 0.0003034099009866613, 0.000304210454815098, 0.00030501100864353467, 0.00030581156247197136, 0.00030661211630040805, 0.00030741267012884474, 0.00030821322395728143, 0.0003090137777857181, 0.0003098143316141548, 0.0003106148854425915, 0.0003114154392710282, 0.0003122159930994649, 0.00031301654692790157, 0.00031381710075633826, 0.00031461765458477495, 0.00031541820841321164, 0.0003162187622416483, 0.000317019316070085, 0.0003178198698985217, 0.0003186204237269584, 0.0003194209775553951, 0.0003202215313838318, 0.00032102208521226847, 0.00032182263904070516, 0.00032262319286914184, 0.00032342374669757853, 0.0003242243005260152, 0.0003250248543544519, 0.0003258254081828886, 0.0003266259620113253, 0.000327426515839762, 0.0003282270696681987, 0.00032902762349663536, 0.00032982817732507205, 0.00033062873115350874, 0.00033142928498194543, 0.0003322298388103821, 0.0003330303926388188, 0.0003338309464672555, 0.0003346315002956922, 0.0003354320541241289, 0.00033623260795256557, 0.00033703316178100226, 0.00033783371560943895, 0.00033863426943787564, 0.00033943482326631233, 0.000340235377094749, 0.0003410359309231857, 0.0003418364847516224, 0.0003426370385800591, 0.0003434375924084958, 0.00034423814623693247, 0.00034503870006536916, 0.00034583925389380585, 0.00034663980772224254, 0.00034744036155067923, 0.0003482409153791159, 0.0003490414692075526, 0.0003498420230359893, 0.000350642576864426, 0.0003514431306928627, 0.00035224368452129937, 0.00035304423834973606, 0.00035384479217817275, 0.00035464534600660944, 0.0003554458998350461, 0.0003562464536634828, 0.0003570470074919195, 0.0003578475613203562, 0.0003586481151487929, 0.0003594486689772296, 0.00036024922280566626, 0.00036104977663410295, 0.00036185033046253964, 0.00036265088429097633, 0.000363451438119413, 0.0003642519919478497, 0.0003650525457762864, 0.0003658530996047231, 0.0003666536534331598, 0.00036745420726159647, 0.00036825476109003316, 0.00036905531491846985, 0.00036985586874690654, 0.00037065642257534323, 0.0003714569764037799, 0.0003722575302322166, 0.0003730580840606533, 0.00037385863788909, 0.0003746591917175267, 0.00037545974554596337, 0.00037626029937440006, 0.00037706085320283675, 0.00037786140703127344, 0.00037866196085971013, 0.0003794625146881468, 0.0003802630685165835, 0.0003810636223450202, 0.0003818641761734569, 0.0003826647300018936, 0.00038346528383033027, 0.00038426583765876696, 0.00038506639148720365, 0.00038586694531564034, 0.000386667499144077, 0.0003874680529725137, 0.0003882686068009504, 0.0003890691606293871, 0.0003898697144578238, 0.0003906702682862605, 0.00039147082211469716, 0.00039227137594313385, 0.00039307192977157054, 0.00039387248360000723, 0.0003946730374284439, 0.0003954735912568806, 0.0003962741450853173, 0.000397074698913754, 0.0003978752527421907, 0.00039867580657062737, 0.00039947636039906406, 0.00040027691422750075, 0.00040107746805593744, 0.00040187802188437413, 0.0004026785757128108, 0.0004034791295412475, 0.0004042796833696842, 0.0004050802371981209, 0.0004058807910265576, 0.00040668134485499427, 0.00040748189868343096, 0.00040828245251186765, 0.00040908300634030434, 0.00040988356016874103, 0.0004106841139971777, 0.0004114846678256144, 0.0004122852216540511, 0.0004130857754824878, 0.0004138863293109245, 0.00041468688313936117, 0.00041548743696779786, 0.00041628799079623455, 0.00041708854462467124, 0.0004178890984531079, 0.0004186896522815446, 0.0004194902061099813, 0.000420290759938418, 0.0004210913137668547, 0.0004218918675952914, 0.00042269242142372807, 0.00042349297525216476, 0.00042429352908060144, 0.00042509408290903813, 0.0004258946367374748, 0.0004266951905659115, 0.0004274957443943482, 0.0004282962982227849, 0.0004290968520512216, 0.0004298974058796583, 0.00043069795970809496, 0.00043149851353653165, 0.00043229906736496834, 0.00043309962119340503, 0.0004339001750218417, 0.0004347007288502784, 0.0004355012826787151, 0.0004363018365071518, 0.0004371023903355885, 0.00043790294416402517, 0.00043870349799246186, 0.00043950405182089855, 0.00044030460564933524, 0.00044110515947777193, 0.0004419057133062086, 0.0004427062671346453, 0.000443506820963082, 0.0004443073747915187, 0.0004451079286199554, 0.00044590848244839207, 0.00044670903627682876, 0.00044750959010526545, 0.00044831014393370214, 0.00044911069776213883, 0.0004499112515905755, 0.0004507118054190122, 0.0004515123592474489, 0.0004523129130758856, 0.0004531134669043223, 0.00045391402073275897, 0.00045471457456119566, 0.00045551512838963235, 0.00045631568221806904, 0.0004571162360465057, 0.0004579167898749424, 0.0004587173437033791, 0.0004595178975318158, 0.0004603184513602525, 0.0004611190051886892, 0.00046191955901712586, 0.00046272011284556255, 0.00046352066667399924, 0.00046432122050243593, 0.0004651217743308726, 0.0004659223281593093, 0.000466722881987746, 0.0004675234358161827, 0.0004683239896446194, 0.00046912454347305607, 0.00046992509730149276, 0.00047072565112992945, 0.00047152620495836614, 0.00047232675878680283, 0.0004731273126152395, 0.0004739278664436762, 0.0004747284202721129, 0.0004755289741005496, 0.0004763295279289863, 0.00047713008175742297, 0.00047793063558585966, 0.00047873118941429635, 0.00047953174324273304, 0.00048033229707116973, 0.0004811328508996064, 0.0004819334047280431, 0.0004827339585564798, 0.0004835345123849165, 0.0004843350662133532, 0.00048513562004178987, 0.00048593617387022656, 0.00048673672769866325, 0.00048753728152709994, 0.0004883378353555366, 0.0004891383891839733, 0.00048993894301241, 0.0004907394968408467, 0.0004915400506692834, 0.0004923406044977201, 0.0004931411583261568, 0.0004939417121545935, 0.0004947422659830301, 0.0004955428198114668, 0.0004963433736399035, 0.0004971439274683402, 0.0004979444812967769, 0.0004987450351252136, 0.0004995455889536503, 0.000500346142782087, 0.0005011466966105237, 0.0005019472504389604, 0.000502747804267397, 0.0005035483580958337, 0.0005043489119242704, 0.0005051494657527071, 0.0005059500195811438, 0.0005067505734095805, 0.0005075511272380172, 0.0005083516810664539, 0.0005091522348948906, 0.0005099527887233272, 0.0005107533425517639, 0.0005115538963802006, 0.0005123544502086373, 0.000513155004037074, 0.0005139555578655107, 0.0005147561116939474, 0.0005155566655223841, 0.0005163572193508208, 0.0005171577731792575, 0.0005179583270076941, 0.0005187588808361308, 0.0005195594346645675, 0.0005203599884930042, 0.0005211605423214409, 0.0005219610961498776, 0.0005227616499783143, 0.000523562203806751, 0.0005243627576351877, 0.0005251633114636244, 0.000525963865292061, 0.0005267644191204977, 0.0005275649729489344, 0.0005283655267773711, 0.0005291660806058078, 0.0005299666344342445, 0.0005307671882626812, 0.0005315677420911179, 0.0005323682959195546, 0.0005331688497479913, 0.0005339694035764279, 0.0005347699574048646, 0.0005355705112333013, 0.000536371065061738, 0.0005371716188901747, 0.0005379721727186114, 0.0005387727265470481, 0.0005395732803754848, 0.0005403738342039215, 0.0005411743880323582, 0.0005419749418607948, 0.0005427754956892315, 0.0005435760495176682, 0.0005443766033461049, 0.0005451771571745416, 0.0005459777110029783, 0.000546778264831415, 0.0005475788186598517, 0.0005483793724882884, 0.000549179926316725, 0.0005499804801451617, 0.0005507810339735984, 0.0005515815878020351, 0.0005523821416304718, 0.0005531826954589085, 0.0005539832492873452, 0.0005547838031157819, 0.0005555843569442186, 0.0005563849107726553, 0.0005571854646010919, 0.0005579860184295286, 0.0005587865722579653, 0.000559587126086402, 0.0005603876799148387, 0.0005611882337432754, 0.0005619887875717121, 0.0005627893414001488, 0.0005635898952285855, 0.0005643904490570222, 0.0005651910028854588, 0.0005659915567138955, 0.0005667921105423322, 0.0005675926643707689, 0.0005683932181992056, 0.0005691937720276423, 0.000569994325856079, 0.0005707948796845157, 0.0005715954335129524, 0.000572395987341389, 0.0005731965411698257, 0.0005739970949982624, 0.0005747976488266991, 0.0005755982026551358, 0.0005763987564835725, 0.0005771993103120092, 0.0005779998641404459, 0.0005788004179688826, 0.0005796009717973193, 0.000580401525625756, 0.0005812020794541926, 0.0005820026332826293, 0.000582803187111066, 0.0005836037409395027, 0.0005844042947679394, 0.0005852048485963761, 0.0005860054024248128, 0.0005868059562532495, 0.0005876065100816862, 0.0005884070639101228, 0.0005892076177385595, 0.0005900081715669962, 0.0005908087253954329, 0.0005916092792238696, 0.0005924098330523063, 0.000593210386880743, 0.0005940109407091797, 0.0005948114945376164, 0.000595612048366053, 0.0005964126021944897, 0.0005972131560229264, 0.0005980137098513631, 0.0005988142636797998, 0.0005996148175082365, 0.0006004153713366732, 0.0006012159251651099, 0.0006020164789935466, 0.0006028170328219833, 0.00060361758665042, 0.0006044181404788566, 0.0006052186943072933, 0.00060601924813573, 0.0006068198019641667, 0.0006076203557926034, 0.0006084209096210401, 0.0006092214634494768, 0.0006100220172779135, 0.0006108225711063502, 0.0006116231249347868, 0.0006124236787632235, 0.0006132242325916602, 0.0006140247864200969, 0.0006148253402485336, 0.0006156258940769703, 0.000616426447905407, 0.0006172270017338437, 0.0006180275555622804, 0.0006188281093907171, 0.0006196286632191537, 0.0006204292170475904, 0.0006212297708760271, 0.0006220303247044638, 0.0006228308785329005, 0.0006236314323613372, 0.0006244319861897739, 0.0006252325400182106, 0.0006260330938466473, 0.000626833647675084, 0.0006276342015035206, 0.0006284347553319573, 0.000629235309160394, 0.0006300358629888307, 0.0006308364168172674, 0.0006316369706457041, 0.0006324375244741408, 0.0006332380783025775, 0.0006340386321310142, 0.0006348391859594509, 0.0006356397397878875, 0.0006364402936163242, 0.0006372408474447609, 0.0006380414012731976, 0.0006388419551016343, 0.000639642508930071, 0.0006404430627585077, 0.0006412436165869444, 0.0006420441704153811, 0.0006428447242438178, 0.0006436452780722544, 0.0006444458319006911, 0.0006452463857291278, 0.0006460469395575645, 0.0006468474933860012, 0.0006476480472144379, 0.0006484486010428746, 0.0006492491548713113, 0.000650049708699748, 0.0006508502625281846, 0.0006516508163566213, 0.000652451370185058, 0.0006532519240134947, 0.0006540524778419314, 0.0006548530316703681, 0.0006556535854988048, 0.0006564541393272415, 0.0006572546931556782, 0.0006580552469841149, 0.0006588558008125515, 0.0006596563546409882, 0.0006604569084694249, 0.0006612574622978616, 0.0006620580161262983, 0.000662858569954735, 0.0006636591237831717, 0.0006644596776116084, 0.0006652602314400451, 0.0006660607852684818, 0.0006668613390969184, 0.0006676618929253551, 0.0006684624467537918, 0.0006692630005822285, 0.0006700635544106652, 0.0006708641082391019, 0.0006716646620675386, 0.0006724652158959753, 0.000673265769724412, 0.0006740663235528487, 0.0006748668773812853, 0.000675667431209722, 0.0006764679850381587, 0.0006772685388665954, 0.0006780690926950321, 0.0006788696465234688, 0.0006796702003519055, 0.0006804707541803422, 0.0006812713080087789, 0.0006820718618372155, 0.0006828724156656522, 0.0006836729694940889, 0.0006844735233225256, 0.0006852740771509623, 0.000686074630979399, 0.0006868751848078357, 0.0006876757386362724, 0.0006884762924647091, 0.0006892768462931458, 0.0006900774001215824, 0.0006908779539500191, 0.0006916785077784558, 0.0006924790616068925, 0.0006932796154353292, 0.0006940801692637659, 0.0006948807230922026, 0.0006956812769206393, 0.000696481830749076, 0.0006972823845775127, 0.0006980829384059493, 0.000698883492234386, 0.0006996840460628227, 0.0007004845998912594, 0.0007012851537196961, 0.0007020857075481328, 0.0007028862613765695, 0.0007036868152050062, 0.0007044873690334429, 0.0007052879228618796, 0.0007060884766903162, 0.0007068890305187529, 0.0007076895843471896, 0.0007084901381756263, 0.000709290692004063, 0.0007100912458324997, 0.0007108917996609364, 0.0007116923534893731, 0.0007124929073178098, 0.0007132934611462464, 0.0007140940149746831, 0.0007148945688031198, 0.0007156951226315565, 0.0007164956764599932, 0.0007172962302884299, 0.0007180967841168666, 0.0007188973379453033, 0.00071969789177374, 0.0007204984456021767, 0.0007212989994306133, 0.00072209955325905, 0.0007229001070874867, 0.0007237006609159234, 0.0007245012147443601, 0.0007253017685727968, 0.0007261023224012335, 0.0007269028762296702, 0.0007277034300581069, 0.0007285039838865436, 0.0007293045377149802, 0.0007301050915434169, 0.0007309056453718536, 0.0007317061992002903, 0.000732506753028727, 0.0007333073068571637, 0.0007341078606856004, 0.0007349084145140371, 0.0007357089683424738, 0.0007365095221709105, 0.0007373100759993471, 0.0007381106298277838, 0.0007389111836562205, 0.0007397117374846572, 0.0007405122913130939, 0.0007413128451415306, 0.0007421133989699673, 0.000742913952798404, 0.0007437145066268407, 0.0007445150604552773, 0.000745315614283714, 0.0007461161681121507, 0.0007469167219405874, 0.0007477172757690241, 0.0007485178295974608, 0.0007493183834258975, 0.0007501189372543342, 0.0007509194910827709, 0.0007517200449112076, 0.0007525205987396442, 0.0007533211525680809, 0.0007541217063965176, 0.0007549222602249543, 0.000755722814053391, 0.0007565233678818277, 0.0007573239217102644, 0.0007581244755387011, 0.0007589250293671378, 0.0007597255831955745, 0.0007605261370240111, 0.0007613266908524478, 0.0007621272446808845, 0.0007629277985093212, 0.0007637283523377579, 0.0007645289061661946, 0.0007653294599946313, 0.000766130013823068, 0.0007669305676515047, 0.0007677311214799414, 0.000768531675308378, 0.0007693322291368147, 0.0007701327829652514, 0.0007709333367936881, 0.0007717338906221248, 0.0007725344444505615, 0.0007733349982789982, 0.0007741355521074349, 0.0007749361059358716, 0.0007757366597643083, 0.0007765372135927449, 0.0007773377674211816, 0.0007781383212496183, 0.000778938875078055, 0.0007797394289064917, 0.0007805399827349284, 0.0007813405365633651, 0.0007821410903918018, 0.0007829416442202385, 0.0007837421980486751, 0.0007845427518771118, 0.0007853433057055485, 0.0007861438595339852, 0.0007869444133624219, 0.0007877449671908586, 0.0007885455210192953, 0.000789346074847732, 0.0007901466286761687, 0.0007909471825046054, 0.000791747736333042, 0.0007925482901614787, 0.0007933488439899154, 0.0007941493978183521, 0.0007949499516467888, 0.0007957505054752255, 0.0007965510593036622, 0.0007973516131320989, 0.0007981521669605356, 0.0007989527207889723, 0.0007997532746174089, 0.0008005538284458456, 0.0008013543822742823, 0.000802154936102719, 0.0008029554899311557, 0.0008037560437595924, 0.0008045565975880291, 0.0008053571514164658, 0.0008061577052449025, 0.0008069582590733392, 0.0008077588129017758, 0.0008085593667302125, 0.0008093599205586492, 0.0008101604743870859, 0.0008109610282155226, 0.0008117615820439593, 0.000812562135872396, 0.0008133626897008327, 0.0008141632435292694, 0.000814963797357706, 0.0008157643511861427, 0.0008165649050145794, 0.0008173654588430161, 0.0008181660126714528, 0.0008189665664998895, 0.0008197671203283262, 0.0008205676741567629, 0.0008213682279851996, 0.0008221687818136363, 0.000822969335642073, 0.0008237698894705096, 0.0008245704432989463, 0.000825370997127383, 0.0008261715509558197, 0.0008269721047842564, 0.0008277726586126931, 0.0008285732124411298, 0.0008293737662695665, 0.0008301743200980032, 0.0008309748739264398, 0.0008317754277548765, 0.0008325759815833132, 0.0008333765354117499, 0.0008341770892401866, 0.0008349776430686233, 0.00083577819689706, 0.0008365787507254967, 0.0008373793045539334, 0.00083817985838237, 0.0008389804122108067, 0.0008397809660392434, 0.0008405815198676801, 0.0008413820736961168, 0.0008421826275245535, 0.0008429831813529902, 0.0008437837351814269, 0.0008445842890098636, 0.0008453848428383003, 0.000846185396666737, 0.0008469859504951736, 0.0008477865043236103, 0.000848587058152047, 0.0008493876119804837, 0.0008501881658089204, 0.0008509887196373571, 0.0008517892734657938, 0.0008525898272942305, 0.0008533903811226672, 0.0008541909349511038, 0.0008549914887795405, 0.0008557920426079772, 0.0008565925964364139, 0.0008573931502648506, 0.0008581937040932873, 0.000858994257921724, 0.0008597948117501607, 0.0008605953655785974, 0.0008613959194070341, 0.0008621964732354707, 0.0008629970270639074, 0.0008637975808923441, 0.0008645981347207808, 0.0008653986885492175, 0.0008661992423776542, 0.0008669997962060909, 0.0008678003500345276, 0.0008686009038629643, 0.000869401457691401, 0.0008702020115198376, 0.0008710025653482743, 0.000871803119176711, 0.0008726036730051477, 0.0008734042268335844, 0.0008742047806620211, 0.0008750053344904578, 0.0008758058883188945, 0.0008766064421473312, 0.0008774069959757679, 0.0008782075498042045, 0.0008790081036326412, 0.0008798086574610779, 0.0008806092112895146, 0.0008814097651179513, 0.000882210318946388, 0.0008830108727748247, 0.0008838114266032614, 0.0008846119804316981, 0.0008854125342601347, 0.0008862130880885714, 0.0008870136419170081, 0.0008878141957454448, 0.0008886147495738815, 0.0008894153034023182, 0.0008902158572307549, 0.0008910164110591916, 0.0008918169648876283, 0.000892617518716065, 0.0008934180725445016, 0.0008942186263729383, 0.000895019180201375, 0.0008958197340298117, 0.0008966202878582484, 0.0008974208416866851, 0.0008982213955151218, 0.0008990219493435585, 0.0008998225031719952, 0.0009006230570004319, 0.0009014236108288685, 0.0009022241646573052, 0.0009030247184857419, 0.0009038252723141786, 0.0009046258261426153, 0.000905426379971052, 0.0009062269337994887, 0.0009070274876279254, 0.0009078280414563621, 0.0009086285952847988, 0.0009094291491132354, 0.0009102297029416721, 0.0009110302567701088, 0.0009118308105985455, 0.0009126313644269822, 0.0009134319182554189, 0.0009142324720838556, 0.0009150330259122923, 0.000915833579740729, 0.0009166341335691656, 0.0009174346873976023, 0.000918235241226039, 0.0009190357950544757, 0.0009198363488829124, 0.0009206369027113491, 0.0009214374565397858, 0.0009222380103682225, 0.0009230385641966592, 0.0009238391180250959, 0.0009246396718535325, 0.0009254402256819692, 0.0009262407795104059, 0.0009270413333388426, 0.0009278418871672793, 0.000928642440995716, 0.0009294429948241527, 0.0009302435486525894, 0.0009310441024810261, 0.0009318446563094628, 0.0009326452101378994, 0.0009334457639663361, 0.0009342463177947728, 0.0009350468716232095, 0.0009358474254516462, 0.0009366479792800829, 0.0009374485331085196, 0.0009382490869369563, 0.000939049640765393, 0.0009398501945938297, 0.0009406507484222663, 0.000941451302250703, 0.0009422518560791397, 0.0009430524099075764, 0.0009438529637360131, 0.0009446535175644498, 0.0009454540713928865, 0.0009462546252213232, 0.0009470551790497599, 0.0009478557328781965, 0.0009486562867066332, 0.0009494568405350699, 0.0009502573943635066, 0.0009510579481919433, 0.00095185850202038, 0.0009526590558488167, 0.0009534596096772534, 0.0009542601635056901, 0.0009550607173341268, 0.0009558612711625634, 0.0009566618249910001, 0.0009574623788194368, 0.0009582629326478735, 0.0009590634864763102, 0.0009598640403047469, 0.0009606645941331836, 0.0009614651479616203, 0.000962265701790057, 0.0009630662556184937, 0.0009638668094469303, 0.000964667363275367, 0.0009654679171038037, 0.0009662684709322404, 0.0009670690247606771, 0.0009678695785891138, 0.0009686701324175505, 0.0009694706862459872, 0.0009702712400744239, 0.0009710717939028606, 0.0009718723477312972, 0.0009726729015597339, 0.0009734734553881706, 0.0009742740092166073, 0.000975074563045044, 0.0009758751168734807, 0.0009766756707019174, 0.000977476224530354, 0.0009782767783587908, 0.0009790773321872275, 0.0009798778860156641, 0.0009806784398441008, 0.0009814789936725375, 0.0009822795475009742, 0.000983080101329411, 0.0009838806551578476, 0.0009846812089862843, 0.000985481762814721, 0.0009862823166431577, 0.0009870828704715943, 0.000987883424300031, 0.0009886839781284677, 0.0009894845319569044, 0.000990285085785341, 0.0009910856396137778, 0.0009918861934422145, 0.0009926867472706512, 0.0009934873010990879, 0.0009942878549275246, 0.0009950884087559612, 0.000995888962584398, 0.0009966895164128346, 0.0009974900702412713, 0.000998290624069708, 0.0009990911778981447, 0.0009998917317265814, 0.001000692285555018, 0.0010014928393834548, 0.0010022933932118915, 0.0010030939470403281, 0.0010038945008687648, 0.0010046950546972015, 0.0010054956085256382, 0.001006296162354075, 0.0010070967161825116, 0.0010078972700109483, 0.001008697823839385, 0.0010094983776678217, 0.0010102989314962584, 0.001011099485324695, 0.0010119000391531317, 0.0010127005929815684, 0.0010135011468100051, 0.0010143017006384418, 0.0010151022544668785, 0.0010159028082953152, 0.0010167033621237519, 0.0010175039159521886, 0.0010183044697806252, 0.001019105023609062, 0.0010199055774374986, 0.0010207061312659353, 0.001021506685094372, 0.0010223072389228087, 0.0010231077927512454, 0.001023908346579682, 0.0010247089004081188, 0.0010255094542365555, 0.0010263100080649921, 0.0010271105618934288, 0.0010279111157218655, 0.0010287116695503022, 0.001029512223378739, 0.0010303127772071756, 0.0010311133310356123, 0.001031913884864049, 0.0010327144386924857, 0.0010335149925209224, 0.001034315546349359, 0.0010351161001777957, 0.0010359166540062324, 0.0010367172078346691, 0.0010375177616631058, 0.0010383183154915425, 0.0010391188693199792, 0.0010399194231484159, 0.0010407199769768526, 0.0010415205308052893, 0.001042321084633726, 0.0010431216384621626, 0.0010439221922905993, 0.001044722746119036, 0.0010455232999474727, 0.0010463238537759094, 0.001047124407604346, 0.0010479249614327828, 0.0010487255152612195, 0.0010495260690896561, 0.0010503266229180928, 0.0010511271767465295, 0.0010519277305749662, 0.001052728284403403, 0.0010535288382318396, 0.0010543293920602763, 0.001055129945888713, 0.0010559304997171497, 0.0010567310535455864, 0.001057531607374023, 0.0010583321612024597, 0.0010591327150308964, 0.0010599332688593331, 0.0010607338226877698, 0.0010615343765162065, 0.0010623349303446432, 0.0010631354841730799, 0.0010639360380015166, 0.0010647365918299533, 0.00106553714565839, 0.0010663376994868266, 0.0010671382533152633, 0.0010679388071437, 0.0010687393609721367, 0.0010695399148005734, 0.00107034046862901, 0.0010711410224574468, 0.0010719415762858835, 0.0010727421301143202, 0.0010735426839427568, 0.0010743432377711935, 0.0010751437915996302, 0.001075944345428067, 0.0010767448992565036, 0.0010775454530849403, 0.001078346006913377, 0.0010791465607418137, 0.0010799471145702504, 0.001080747668398687, 0.0010815482222271237, 0.0010823487760555604, 0.0010831493298839971, 0.0010839498837124338, 0.0010847504375408705, 0.0010855509913693072, 0.0010863515451977439, 0.0010871520990261806, 0.0010879526528546173, 0.001088753206683054, 0.0010895537605114906, 0.0010903543143399273, 0.001091154868168364, 0.0010919554219968007, 0.0010927559758252374, 0.001093556529653674, 0.0010943570834821108, 0.0010951576373105475, 0.0010959581911389842, 0.0010967587449674208, 0.0010975592987958575, 0.0010983598526242942, 0.001099160406452731, 0.0010999609602811676, 0.0011007615141096043, 0.001101562067938041, 0.0011023626217664777, 0.0011031631755949144, 0.001103963729423351, 0.0011047642832517877, 0.0011055648370802244, 0.0011063653909086611, 0.0011071659447370978, 0.0011079664985655345, 0.0011087670523939712, 0.0011095676062224079, 0.0011103681600508446, 0.0011111687138792813, 0.001111969267707718, 0.0011127698215361546, 0.0011135703753645913, 0.001114370929193028, 0.0011151714830214647, 0.0011159720368499014, 0.001116772590678338, 0.0011175731445067748, 0.0011183736983352115, 0.0011191742521636482, 0.0011199748059920848, 0.0011207753598205215, 0.0011215759136489582, 0.001122376467477395, 0.0011231770213058316, 0.0011239775751342683, 0.001124778128962705, 0.0011255786827911417, 0.0011263792366195784, 0.001127179790448015, 0.0011279803442764517, 0.0011287808981048884, 0.0011295814519333251, 0.0011303820057617618, 0.0011311825595901985, 0.0011319831134186352, 0.0011327836672470719, 0.0011335842210755086, 0.0011343847749039453, 0.001135185328732382, 0.0011359858825608186, 0.0011367864363892553, 0.001137586990217692, 0.0011383875440461287, 0.0011391880978745654, 0.001139988651703002, 0.0011407892055314388, 0.0011415897593598755, 0.0011423903131883122, 0.0011431908670167489, 0.0011439914208451855, 0.0011447919746736222, 0.001145592528502059, 0.0011463930823304956, 0.0011471936361589323, 0.001147994189987369, 0.0011487947438158057, 0.0011495952976442424, 0.001150395851472679, 0.0011511964053011157, 0.0011519969591295524, 0.0011527975129579891, 0.0011535980667864258, 0.0011543986206148625, 0.0011551991744432992, 0.0011559997282717359, 0.0011568002821001726, 0.0011576008359286093, 0.001158401389757046, 0.0011592019435854826, 0.0011600024974139193, 0.001160803051242356, 0.0011616036050707927, 0.0011624041588992294, 0.001163204712727666, 0.0011640052665561028, 0.0011648058203845395, 0.0011656063742129762, 0.0011664069280414129, 0.0011672074818698495, 0.0011680080356982862, 0.001168808589526723, 0.0011696091433551596, 0.0011704096971835963, 0.001171210251012033, 0.0011720108048404697, 0.0011728113586689064, 0.001173611912497343, 0.0011744124663257798, 0.0011752130201542164, 0.0011760135739826531, 0.0011768141278110898, 0.0011776146816395265, 0.0011784152354679632, 0.0011792157892963999, 0.0011800163431248366, 0.0011808168969532733, 0.00118161745078171, 0.0011824180046101467, 0.0011832185584385833, 0.00118401911226702, 0.0011848196660954567, 0.0011856202199238934, 0.00118642077375233, 0.0011872213275807668, 0.0011880218814092035, 0.0011888224352376402, 0.0011896229890660769, 0.0011904235428945135, 0.0011912240967229502, 0.001192024650551387, 0.0011928252043798236, 0.0011936257582082603, 0.001194426312036697, 0.0011952268658651337, 0.0011960274196935704, 0.001196827973522007, 0.0011976285273504438, 0.0011984290811788804, 0.0011992296350073171, 0.0012000301888357538, 0.0012008307426641905, 0.0012016312964926272, 0.001202431850321064, 0.0012032324041495006, 0.0012040329579779373, 0.001204833511806374, 0.0012056340656348107, 0.0012064346194632473, 0.001207235173291684, 0.0012080357271201207, 0.0012088362809485574, 0.001209636834776994, 0.0012104373886054308, 0.0012112379424338675, 0.0012120384962623042, 0.0012128390500907409, 0.0012136396039191776, 0.0012144401577476142, 0.001215240711576051, 0.0012160412654044876, 0.0012168418192329243, 0.001217642373061361, 0.0012184429268897977, 0.0012192434807182344, 0.001220044034546671, 0.0012208445883751078, 0.0012216451422035444, 0.0012224456960319811, 0.0012232462498604178, 0.0012240468036888545, 0.0012248473575172912, 0.001225647911345728, 0.0012264484651741646, 0.0012272490190026013, 0.001228049572831038, 0.0012288501266594747, 0.0012296506804879113, 0.001230451234316348, 0.0012312517881447847, 0.0012320523419732214, 0.001232852895801658, 0.0012336534496300948, 0.0012344540034585315, 0.0012352545572869682, 0.0012360551111154049, 0.0012368556649438416, 0.0012376562187722782, 0.001238456772600715, 0.0012392573264291516, 0.0012400578802575883, 0.001240858434086025, 0.0012416589879144617, 0.0012424595417428984, 0.001243260095571335, 0.0012440606493997718, 0.0012448612032282085, 0.0012456617570566451, 0.0012464623108850818, 0.0012472628647135185, 0.0012480634185419552, 0.001248863972370392, 0.0012496645261988286, 0.0012504650800272653, 0.001251265633855702, 0.0012520661876841387, 0.0012528667415125753, 0.001253667295341012, 0.0012544678491694487, 0.0012552684029978854, 0.001256068956826322, 0.0012568695106547588, 0.0012576700644831955, 0.0012584706183116322, 0.0012592711721400689, 0.0012600717259685056, 0.0012608722797969422, 0.001261672833625379, 0.0012624733874538156, 0.0012632739412822523, 0.001264074495110689, 0.0012648750489391257, 0.0012656756027675624, 0.001266476156595999, 0.0012672767104244358, 0.0012680772642528725, 0.0012688778180813091, 0.0012696783719097458, 0.0012704789257381825, 0.0012712794795666192, 0.001272080033395056, 0.0012728805872234926, 0.0012736811410519293, 0.001274481694880366, 0.0012752822487088027, 0.0012760828025372394, 0.001276883356365676, 0.0012776839101941127, 0.0012784844640225494, 0.0012792850178509861, 0.0012800855716794228, 0.0012808861255078595, 0.0012816866793362962, 0.0012824872331647329, 0.0012832877869931696, 0.0012840883408216063, 0.001284888894650043, 0.0012856894484784796, 0.0012864900023069163, 0.001287290556135353, 0.0012880911099637897, 0.0012888916637922264, 0.001289692217620663, 0.0012904927714490998, 0.0012912933252775365, 0.0012920938791059731, 0.0012928944329344098, 0.0012936949867628465, 0.0012944955405912832, 0.00129529609441972, 0.0012960966482481566, 0.0012968972020765933, 0.00129769775590503, 0.0012984983097334667, 0.0012992988635619034, 0.00130009941739034, 0.0013008999712187767, 0.0013017005250472134, 0.0013025010788756501, 0.0013033016327040868, 0.0013041021865325235, 0.0013049027403609602, 0.0013057032941893969, 0.0013065038480178336, 0.0013073044018462703, 0.001308104955674707, 0.0013089055095031436, 0.0013097060633315803, 0.001310506617160017, 0.0013113071709884537, 0.0013121077248168904, 0.001312908278645327, 0.0013137088324737638, 0.0013145093863022005, 0.0013153099401306372, 0.0013161104939590738, 0.0013169110477875105, 0.0013177116016159472, 0.001318512155444384, 0.0013193127092728206, 0.0013201132631012573, 0.001320913816929694, 0.0013217143707581307, 0.0013225149245865674, 0.001323315478415004, 0.0013241160322434407, 0.0013249165860718774, 0.0013257171399003141, 0.0013265176937287508, 0.0013273182475571875, 0.0013281188013856242, 0.0013289193552140609, 0.0013297199090424976, 0.0013305204628709343, 0.001331321016699371, 0.0013321215705278076, 0.0013329221243562443, 0.001333722678184681, 0.0013345232320131177, 0.0013353237858415544, 0.001336124339669991, 0.0013369248934984278, 0.0013377254473268645, 0.0013385260011553012, 0.0013393265549837378, 0.0013401271088121745, 0.0013409276626406112, 0.001341728216469048, 0.0013425287702974846, 0.0013433293241259213, 0.001344129877954358, 0.0013449304317827947, 0.0013457309856112314, 0.001346531539439668, 0.0013473320932681047, 0.0013481326470965414, 0.0013489332009249781, 0.0013497337547534148, 0.0013505343085818515, 0.0013513348624102882, 0.0013521354162387249, 0.0013529359700671616, 0.0013537365238955983, 0.001354537077724035, 0.0013553376315524716, 0.0013561381853809083, 0.001356938739209345, 0.0013577392930377817, 0.0013585398468662184, 0.001359340400694655, 0.0013601409545230918, 0.0013609415083515285, 0.0013617420621799652, 0.0013625426160084018, 0.0013633431698368385, 0.0013641437236652752, 0.001364944277493712, 0.0013657448313221486, 0.0013665453851505853, 0.001367345938979022, 0.0013681464928074587, 0.0013689470466358954, 0.001369747600464332, 0.0013705481542927687, 0.0013713487081212054, 0.0013721492619496421, 0.0013729498157780788, 0.0013737503696065155, 0.0013745509234349522, 0.0013753514772633889, 0.0013761520310918256, 0.0013769525849202623, 0.001377753138748699, 0.0013785536925771356, 0.0013793542464055723, 0.001380154800234009, 0.0013809553540624457, 0.0013817559078908824, 0.001382556461719319, 0.0013833570155477558, 0.0013841575693761925, 0.0013849581232046292, 0.0013857586770330658, 0.0013865592308615025, 0.0013873597846899392, 0.001388160338518376, 0.0013889608923468126, 0.0013897614461752493, 0.001390562000003686, 0.0013913625538321227, 0.0013921631076605594, 0.001392963661488996, 0.0013937642153174327, 0.0013945647691458694, 0.0013953653229743061, 0.0013961658768027428, 0.0013969664306311795, 0.0013977669844596162, 0.0013985675382880529, 0.0013993680921164896, 0.0014001686459449263, 0.001400969199773363, 0.0014017697536017996, 0.0014025703074302363, 0.001403370861258673, 0.0014041714150871097, 0.0014049719689155464, 0.001405772522743983, 0.0014065730765724198, 0.0014073736304008565, 0.0014081741842292932, 0.0014089747380577299, 0.0014097752918861665, 0.0014105758457146032, 0.00141137639954304, 0.0014121769533714766, 0.0014129775071999133, 0.00141377806102835, 0.0014145786148567867, 0.0014153791686852234, 0.00141617972251366, 0.0014169802763420968, 0.0014177808301705334, 0.0014185813839989701, 0.0014193819378274068, 0.0014201824916558435, 0.0014209830454842802, 0.0014217835993127169, 0.0014225841531411536, 0.0014233847069695903, 0.001424185260798027, 0.0014249858146264636, 0.0014257863684549003, 0.001426586922283337, 0.0014273874761117737, 0.0014281880299402104, 0.001428988583768647, 0.0014297891375970838, 0.0014305896914255205, 0.0014313902452539572, 0.0014321907990823939, 0.0014329913529108305, 0.0014337919067392672, 0.001434592460567704, 0.0014353930143961406, 0.0014361935682245773, 0.001436994122053014, 0.0014377946758814507, 0.0014385952297098874, 0.001439395783538324, 0.0014401963373667608, 0.0014409968911951974, 0.0014417974450236341, 0.0014425979988520708, 0.0014433985526805075, 0.0014441991065089442, 0.001444999660337381, 0.0014458002141658176, 0.0014466007679942543, 0.001447401321822691, 0.0014482018756511277, 0.0014490024294795643, 0.001449802983308001, 0.0014506035371364377, 0.0014514040909648744, 0.001452204644793311, 0.0014530051986217478, 0.0014538057524501845, 0.0014546063062786212, 0.0014554068601070579, 0.0014562074139354945, 0.0014570079677639312, 0.001457808521592368, 0.0014586090754208046, 0.0014594096292492413, 0.001460210183077678, 0.0014610107369061147, 0.0014618112907345514, 0.001462611844562988, 0.0014634123983914248, 0.0014642129522198614, 0.0014650135060482981, 0.0014658140598767348, 0.0014666146137051715, 0.0014674151675336082, 0.001468215721362045, 0.0014690162751904816, 0.0014698168290189183, 0.001470617382847355, 0.0014714179366757917, 0.0014722184905042283, 0.001473019044332665, 0.0014738195981611017, 0.0014746201519895384, 0.001475420705817975, 0.0014762212596464118, 0.0014770218134748485, 0.0014778223673032852, 0.0014786229211317219, 0.0014794234749601586, 0.0014802240287885952, 0.001481024582617032, 0.0014818251364454686, 0.0014826256902739053, 0.001483426244102342, 0.0014842267979307787, 0.0014850273517592154, 0.001485827905587652, 0.0014866284594160888, 0.0014874290132445254, 0.0014882295670729621, 0.0014890301209013988, 0.0014898306747298355, 0.0014906312285582722, 0.001491431782386709, 0.0014922323362151456, 0.0014930328900435823, 0.001493833443872019, 0.0014946339977004557, 0.0014954345515288923, 0.001496235105357329, 0.0014970356591857657, 0.0014978362130142024, 0.001498636766842639, 0.0014994373206710758, 0.0015002378744995125, 0.0015010384283279492, 0.0015018389821563859, 0.0015026395359848226, 0.0015034400898132592, 0.001504240643641696, 0.0015050411974701326, 0.0015058417512985693, 0.001506642305127006, 0.0015074428589554427, 0.0015082434127838794, 0.001509043966612316, 0.0015098445204407528, 0.0015106450742691895, 0.0015114456280976261, 0.0015122461819260628, 0.0015130467357544995, 0.0015138472895829362, 0.001514647843411373, 0.0015154483972398096, 0.0015162489510682463, 0.001517049504896683, 0.0015178500587251197, 0.0015186506125535564, 0.001519451166381993, 0.0015202517202104297, 0.0015210522740388664, 0.0015218528278673031, 0.0015226533816957398, 0.0015234539355241765, 0.0015242544893526132, 0.0015250550431810499, 0.0015258555970094866, 0.0015266561508379232, 0.00152745670466636, 0.0015282572584947966, 0.0015290578123232333, 0.00152985836615167, 0.0015306589199801067, 0.0015314594738085434, 0.00153226002763698, 0.0015330605814654168, 0.0015338611352938535, 0.0015346616891222901, 0.0015354622429507268, 0.0015362627967791635, 0.0015370633506076002, 0.001537863904436037, 0.0015386644582644736, 0.0015394650120929103, 0.001540265565921347, 0.0015410661197497837, 0.0015418666735782204, 0.001542667227406657, 0.0015434677812350937, 0.0015442683350635304, 0.0015450688888919671, 0.0015458694427204038, 0.0015466699965488405, 0.0015474705503772772, 0.0015482711042057139, 0.0015490716580341506, 0.0015498722118625873, 0.001550672765691024, 0.0015514733195194606, 0.0015522738733478973, 0.001553074427176334, 0.0015538749810047707, 0.0015546755348332074, 0.001555476088661644, 0.0015562766424900808, 0.0015570771963185175, 0.0015578777501469541, 0.0015586783039753908, 0.0015594788578038275, 0.0015602794116322642, 0.001561079965460701, 0.0015618805192891376, 0.0015626810731175743, 0.001563481626946011, 0.0015642821807744477, 0.0015650827346028844, 0.001565883288431321, 0.0015666838422597577, 0.0015674843960881944, 0.0015682849499166311, 0.0015690855037450678, 0.0015698860575735045, 0.0015706866114019412, 0.0015714871652303779, 0.0015722877190588146, 0.0015730882728872513, 0.001573888826715688, 0.0015746893805441246, 0.0015754899343725613, 0.001576290488200998, 0.0015770910420294347, 0.0015778915958578714, 0.001578692149686308, 0.0015794927035147448, 0.0015802932573431815, 0.0015810938111716182, 0.0015818943650000548, 0.0015826949188284915, 0.0015834954726569282, 0.001584296026485365, 0.0015850965803138016, 0.0015858971341422383, 0.001586697687970675, 0.0015874982417991117, 0.0015882987956275484, 0.001589099349455985, 0.0015898999032844217, 0.0015907004571128584, 0.0015915010109412951, 0.0015923015647697318, 0.0015931021185981685, 0.0015939026724266052, 0.0015947032262550419, 0.0015955037800834786, 0.0015963043339119153, 0.001597104887740352, 0.0015979054415687886, 0.0015987059953972253, 0.001599506549225662, 0.0016003071030540987, 0.0016011076568825354, 0.001601908210710972, 0.0016027087645394088, 0.0016035093183678455, 0.0016043098721962822, 0.0016051104260247188, 0.0016059109798531555, 0.0016067115336815922, 0.001607512087510029, 0.0016083126413384656, 0.0016091131951669023, 0.001609913748995339, 0.0016107143028237757, 0.0016115148566522124, 0.001612315410480649, 0.0016131159643090857, 0.0016139165181375224, 0.0016147170719659591, 0.0016155176257943958, 0.0016163181796228325, 0.0016171187334512692, 0.0016179192872797059, 0.0016187198411081426, 0.0016195203949365793, 0.001620320948765016, 0.0016211215025934526, 0.0016219220564218893, 0.001622722610250326, 0.0016235231640787627, 0.0016243237179071994, 0.001625124271735636, 0.0016259248255640728, 0.0016267253793925095, 0.0016275259332209462, 0.0016283264870493828, 0.0016291270408778195, 0.0016299275947062562, 0.001630728148534693, 0.0016315287023631296, 0.0016323292561915663, 0.001633129810020003, 0.0016339303638484397, 0.0016347309176768764, 0.001635531471505313, 0.0016363320253337497, 0.0016371325791621864, 0.0016379331329906231, 0.0016387336868190598, 0.0016395342406474965, 0.0016403347944759332, 0.0016411353483043699, 0.0016419359021328066, 0.0016427364559612433, 0.00164353700978968, 0.0016443375636181166, 0.0016451381174465533, 0.00164593867127499, 0.0016467392251034267, 0.0016475397789318634, 0.0016483403327603, 0.0016491408865887368, 0.0016499414404171735, 0.0016507419942456102, 0.0016515425480740469, 0.0016523431019024835, 0.0016531436557309202, 0.001653944209559357, 0.0016547447633877936, 0.0016555453172162303, 0.001656345871044667, 0.0016571464248731037, 0.0016579469787015404, 0.001658747532529977, 0.0016595480863584137, 0.0016603486401868504, 0.0016611491940152871, 0.0016619497478437238, 0.0016627503016721605, 0.0016635508555005972, 0.0016643514093290339, 0.0016651519631574706, 0.0016659525169859073, 0.001666753070814344, 0.0016675536246427806, 0.0016683541784712173, 0.001669154732299654, 0.0016699552861280907, 0.0016707558399565274, 0.001671556393784964, 0.0016723569476134008, 0.0016731575014418375, 0.0016739580552702742, 0.0016747586090987109, 0.0016755591629271475, 0.0016763597167555842, 0.001677160270584021, 0.0016779608244124576, 0.0016787613782408943, 0.001679561932069331, 0.0016803624858977677, 0.0016811630397262044, 0.001681963593554641, 0.0016827641473830778, 0.0016835647012115144, 0.0016843652550399511, 0.0016851658088683878, 0.0016859663626968245, 0.0016867669165252612, 0.0016875674703536979, 0.0016883680241821346, 0.0016891685780105713, 0.001689969131839008, 0.0016907696856674446, 0.0016915702394958813, 0.001692370793324318, 0.0016931713471527547, 0.0016939719009811914, 0.001694772454809628, 0.0016955730086380648, 0.0016963735624665015, 0.0016971741162949382, 0.0016979746701233749, 0.0016987752239518115, 0.0016995757777802482, 0.001700376331608685, 0.0017011768854371216, 0.0017019774392655583, 0.001702777993093995, 0.0017035785469224317, 0.0017043791007508684, 0.001705179654579305, 0.0017059802084077418, 0.0017067807622361784, 0.0017075813160646151, 0.0017083818698930518, 0.0017091824237214885, 0.0017099829775499252, 0.001710783531378362, 0.0017115840852067986, 0.0017123846390352353, 0.001713185192863672, 0.0017139857466921087, 0.0017147863005205453, 0.001715586854348982, 0.0017163874081774187, 0.0017171879620058554, 0.001717988515834292, 0.0017187890696627288, 0.0017195896234911655, 0.0017203901773196022, 0.0017211907311480389, 0.0017219912849764756, 0.0017227918388049122, 0.001723592392633349, 0.0017243929464617856, 0.0017251935002902223, 0.001725994054118659, 0.0017267946079470957, 0.0017275951617755324, 0.001728395715603969, 0.0017291962694324058, 0.0017299968232608424, 0.0017307973770892791, 0.0017315979309177158, 0.0017323984847461525, 0.0017331990385745892, 0.001733999592403026, 0.0017348001462314626, 0.0017356007000598993, 0.001736401253888336, 0.0017372018077167727, 0.0017380023615452093, 0.001738802915373646, 0.0017396034692020827, 0.0017404040230305194, 0.001741204576858956, 0.0017420051306873928, 0.0017428056845158295, 0.0017436062383442662, 0.0017444067921727029, 0.0017452073460011396, 0.0017460078998295762, 0.001746808453658013, 0.0017476090074864496, 0.0017484095613148863, 0.001749210115143323, 0.0017500106689717597, 0.0017508112228001964, 0.001751611776628633, 0.0017524123304570698, 0.0017532128842855065, 0.0017540134381139431, 0.0017548139919423798, 0.0017556145457708165, 0.0017564150995992532, 0.00175721565342769, 0.0017580162072561266, 0.0017588167610845633, 0.001759617314913, 0.0017604178687414367, 0.0017612184225698733, 0.00176201897639831, 0.0017628195302267467, 0.0017636200840551834, 0.00176442063788362, 0.0017652211917120568, 0.0017660217455404935, 0.0017668222993689302, 0.0017676228531973669, 0.0017684234070258036, 0.0017692239608542402, 0.001770024514682677, 0.0017708250685111136, 0.0017716256223395503, 0.001772426176167987, 0.0017732267299964237, 0.0017740272838248604, 0.001774827837653297, 0.0017756283914817338, 0.0017764289453101705, 0.0017772294991386071, 0.0017780300529670438, 0.0017788306067954805, 0.0017796311606239172, 0.001780431714452354, 0.0017812322682807906, 0.0017820328221092273, 0.001782833375937664, 0.0017836339297661007, 0.0017844344835945374, 0.001785235037422974, 0.0017860355912514107, 0.0017868361450798474, 0.0017876366989082841, 0.0017884372527367208, 0.0017892378065651575, 0.0017900383603935942, 0.0017908389142220309, 0.0017916394680504676, 0.0017924400218789042, 0.001793240575707341, 0.0017940411295357776, 0.0017948416833642143, 0.001795642237192651, 0.0017964427910210877, 0.0017972433448495244, 0.001798043898677961, 0.0017988444525063978, 0.0017996450063348345, 0.0018004455601632711, 0.0018012461139917078, 0.0018020466678201445, 0.0018028472216485812, 0.001803647775477018, 0.0018044483293054546, 0.0018052488831338913, 0.001806049436962328, 0.0018068499907907647, 0.0018076505446192014, 0.001808451098447638, 0.0018092516522760747, 0.0018100522061045114, 0.0018108527599329481, 0.0018116533137613848, 0.0018124538675898215, 0.0018132544214182582, 0.0018140549752466949, 0.0018148555290751316, 0.0018156560829035683, 0.001816456636732005, 0.0018172571905604416, 0.0018180577443888783, 0.001818858298217315, 0.0018196588520457517, 0.0018204594058741884, 0.001821259959702625, 0.0018220605135310618, 0.0018228610673594985, 0.0018236616211879352, 0.0018244621750163718, 0.0018252627288448085, 0.0018260632826732452, 0.001826863836501682, 0.0018276643903301186, 0.0018284649441585553, 0.001829265497986992, 0.0018300660518154287, 0.0018308666056438654, 0.001831667159472302, 0.0018324677133007387, 0.0018332682671291754, 0.0018340688209576121, 0.0018348693747860488, 0.0018356699286144855, 0.0018364704824429222, 0.0018372710362713589, 0.0018380715900997956, 0.0018388721439282323, 0.001839672697756669, 0.0018404732515851056, 0.0018412738054135423, 0.001842074359241979, 0.0018428749130704157, 0.0018436754668988524, 0.001844476020727289, 0.0018452765745557258, 0.0018460771283841625, 0.0018468776822125992, 0.0018476782360410358, 0.0018484787898694725, 0.0018492793436979092, 0.001850079897526346, 0.0018508804513547826, 0.0018516810051832193, 0.001852481559011656, 0.0018532821128400927, 0.0018540826666685294, 0.001854883220496966, 0.0018556837743254027, 0.0018564843281538394, 0.0018572848819822761, 0.0018580854358107128, 0.0018588859896391495, 0.0018596865434675862, 0.0018604870972960229, 0.0018612876511244596, 0.0018620882049528963, 0.001862888758781333, 0.0018636893126097696, 0.0018644898664382063, 0.001865290420266643, 0.0018660909740950797, 0.0018668915279235164, 0.001867692081751953, 0.0018684926355803898, 0.0018692931894088265, 0.0018700937432372632, 0.0018708942970656998, 0.0018716948508941365, 0.0018724954047225732, 0.00187329595855101, 0.0018740965123794466, 0.0018748970662078833, 0.00187569762003632, 0.0018764981738647567, 0.0018772987276931934, 0.00187809928152163, 0.0018788998353500667, 0.0018797003891785034, 0.0018805009430069401, 0.0018813014968353768, 0.0018821020506638135, 0.0018829026044922502, 0.0018837031583206869, 0.0018845037121491236, 0.0018853042659775603, 0.001886104819805997, 0.0018869053736344336, 0.0018877059274628703, 0.001888506481291307, 0.0018893070351197437, 0.0018901075889481804, 0.001890908142776617, 0.0018917086966050538, 0.0018925092504334905, 0.0018933098042619272, 0.0018941103580903638, 0.0018949109119188005, 0.0018957114657472372, 0.001896512019575674, 0.0018973125734041106, 0.0018981131272325473, 0.001898913681060984, 0.0018997142348894207, 0.0019005147887178574, 0.001901315342546294, 0.0019021158963747307, 0.0019029164502031674, 0.0019037170040316041, 0.0019045175578600408, 0.0019053181116884775, 0.0019061186655169142, 0.0019069192193453509, 0.0019077197731737876, 0.0019085203270022243, 0.001909320880830661, 0.0019101214346590976, 0.0019109219884875343, 0.001911722542315971, 0.0019125230961444077, 0.0019133236499728444, 0.001914124203801281, 0.0019149247576297178, 0.0019157253114581545, 0.0019165258652865912, 0.0019173264191150279, 0.0019181269729434645, 0.0019189275267719012, 0.001919728080600338, 0.0019205286344287746, 0.0019213291882572113, 0.001922129742085648, 0.0019229302959140847, 0.0019237308497425214, 0.001924531403570958, 0.0019253319573993947, 0.0019261325112278314, 0.0019269330650562681, 0.0019277336188847048, 0.0019285341727131415, 0.0019293347265415782, 0.0019301352803700149, 0.0019309358341984516, 0.0019317363880268883, 0.001932536941855325, 0.0019333374956837616, 0.0019341380495121983, 0.001934938603340635, 0.0019357391571690717, 0.0019365397109975084, 0.001937340264825945, 0.0019381408186543818, 0.0019389413724828185, 0.0019397419263112552, 0.0019405424801396919, 0.0019413430339681285, 0.0019421435877965652, 0.001942944141625002, 0.0019437446954534386, 0.0019445452492818753, 0.001945345803110312, 0.0019461463569387487, 0.0019469469107671854, 0.001947747464595622, 0.0019485480184240588, 0.0019493485722524954, 0.0019501491260809321, 0.0019509496799093688, 0.0019517502337378055, 0.0019525507875662422, 0.0019533513413949734, 0.001954151895224451, 0.0019549524490539284, 0.001955753002883406, 0.0019565535567128835, 0.001957354110542361, 0.0019581546643718385, 0.001958955218201316, 0.0019597557720307936, 0.001960556325860271, 0.0019613568796897486, 0.001962157433519226, 0.0019629579873487036, 0.001963758541178181, 0.0019645590950076587, 0.0019653596488371362, 0.0019661602026666137, 0.0019669607564960913, 0.001967761310325569, 0.0019685618641550463, 0.001969362417984524, 0.0019701629718140014, 0.001970963525643479, 0.0019717640794729564, 0.001972564633302434, 0.0019733651871319115, 0.001974165740961389, 0.0019749662947908665, 0.001975766848620344, 0.0019765674024498216, 0.001977367956279299, 0.0019781685101087766, 0.001978969063938254, 0.0019797696177677316, 0.001980570171597209, 0.0019813707254266867, 0.0019821712792561642, 0.0019829718330856417, 0.0019837723869151193, 0.001984572940744597, 0.0019853734945740743, 0.001986174048403552, 0.0019869746022330294, 0.001987775156062507, 0.0019885757098919844, 0.001989376263721462, 0.0019901768175509395, 0.001990977371380417, 0.0019917779252098945, 0.001992578479039372, 0.0019933790328688496, 0.001994179586698327, 0.0019949801405278046, 0.001995780694357282, 0.0019965812481867596, 0.001997381802016237, 0.0019981823558457147, 0.0019989829096751922, 0.0019997834635046697, 0.0020005840173341473, 0.002001384571163625, 0.0020021851249931023, 0.00200298567882258, 0.0020037862326520574, 0.002004586786481535, 0.0020053873403110124, 0.00200618789414049, 0.0020069884479699675, 0.002007789001799445, 0.0020085895556289225, 0.0020093901094584, 0.0020101906632878776, 0.002010991217117355, 0.0020117917709468326, 0.00201259232477631, 0.0020133928786057876, 0.002014193432435265, 0.0020149939862647427, 0.0020157945400942202, 0.0020165950939236977, 0.0020173956477531753, 0.002018196201582653, 0.0020189967554121303, 0.002019797309241608, 0.0020205978630710854, 0.002021398416900563, 0.0020221989707300404, 0.002022999524559518, 0.0020238000783889955, 0.002024600632218473, 0.0020254011860479505, 0.002026201739877428, 0.0020270022937069056, 0.002027802847536383, 0.0020286034013658606, 0.002029403955195338, 0.0020302045090248157, 0.002031005062854293, 0.0020318056166837707, 0.0020326061705132482, 0.0020334067243427257, 0.0020342072781722033, 0.002035007832001681, 0.0020358083858311583, 0.002036608939660636, 0.0020374094934901134, 0.002038210047319591, 0.0020390106011490684, 0.002039811154978546, 0.0020406117088080235, 0.002041412262637501, 0.0020422128164669785, 0.002043013370296456, 0.0020438139241259336, 0.002044614477955411, 0.0020454150317848886, 0.002046215585614366, 0.0020470161394438437, 0.002047816693273321, 0.0020486172471027987, 0.0020494178009322762, 0.0020502183547617537, 0.0020510189085912313, 0.002051819462420709, 0.0020526200162501863, 0.002053420570079664, 0.0020542211239091414, 0.002055021677738619, 0.0020558222315680964, 0.002056622785397574, 0.0020574233392270515, 0.002058223893056529, 0.0020590244468860065, 0.002059825000715484, 0.0020606255545449616, 0.002061426108374439, 0.0020622266622039166, 0.002063027216033394, 0.0020638277698628717, 0.002064628323692349, 0.0020654288775218267, 0.0020662294313513042, 0.0020670299851807817, 0.0020678305390102593, 0.002068631092839737, 0.0020694316466692143, 0.002070232200498692, 0.0020710327543281694, 0.002071833308157647, 0.0020726338619871244, 0.002073434415816602, 0.0020742349696460795, 0.002075035523475557, 0.0020758360773050345, 0.002076636631134512, 0.0020774371849639896, 0.002078237738793467, 0.0020790382926229446, 0.002079838846452422, 0.0020806394002818997, 0.002081439954111377, 0.0020822405079408547, 0.0020830410617703322, 0.0020838416155998097, 0.0020846421694292873, 0.002085442723258765, 0.0020862432770882423, 0.00208704383091772, 0.0020878443847471974, 0.002088644938576675, 0.0020894454924061524, 0.00209024604623563, 0.0020910466000651075, 0.002091847153894585, 0.0020926477077240625, 0.00209344826155354, 0.0020942488153830176, 0.002095049369212495, 0.0020958499230419726, 0.00209665047687145, 0.0020974510307009277, 0.002098251584530405, 0.0020990521383598827, 0.0020998526921893602, 0.0021006532460188377, 0.0021014537998483153, 0.002102254353677793, 0.0021030549075072703, 0.002103855461336748, 0.0021046560151662254, 0.002105456568995703, 0.0021062571228251804, 0.002107057676654658, 0.0021078582304841355, 0.002108658784313613, 0.0021094593381430905, 0.002110259891972568, 0.0021110604458020456, 0.002111860999631523, 0.0021126615534610006, 0.002113462107290478, 0.0021142626611199557, 0.002115063214949433, 0.0021158637687789107, 0.0021166643226083882, 0.0021174648764378657, 0.0021182654302673433, 0.002119065984096821, 0.0021198665379262983, 0.002120667091755776, 0.0021214676455852534, 0.002122268199414731, 0.0021230687532442084, 0.002123869307073686, 0.0021246698609031635, 0.002125470414732641, 0.0021262709685621185, 0.002127071522391596, 0.0021278720762210736, 0.002128672630050551, 0.0021294731838800286, 0.002130273737709506, 0.0021310742915389837, 0.002131874845368461, 0.0021326753991979387, 0.0021334759530274162, 0.0021342765068568938, 0.0021350770606863713, 0.002135877614515849, 0.0021366781683453263, 0.002137478722174804, 0.0021382792760042814, 0.002139079829833759, 0.0021398803836632364, 0.002140680937492714, 0.0021414814913221915, 0.002142282045151669, 0.0021430825989811465, 0.002143883152810624, 0.0021446837066401016, 0.002145484260469579, 0.0021462848142990566, 0.002147085368128534, 0.0021478859219580117, 0.002148686475787489, 0.0021494870296169667, 0.0021502875834464442, 0.0021510881372759218, 0.0021518886911053993, 0.002152689244934877, 0.0021534897987643543, 0.002154290352593832, 0.0021550909064233094, 0.002155891460252787, 0.0021566920140822644, 0.002157492567911742, 0.0021582931217412195, 0.002159093675570697, 0.0021598942294001745, 0.002160694783229652, 0.0021614953370591296, 0.002162295890888607, 0.0021630964447180846, 0.002163896998547562, 0.0021646975523770397, 0.002165498106206517, 0.0021662986600359947, 0.0021670992138654722, 0.0021678997676949498, 0.0021687003215244273, 0.002169500875353905, 0.0021703014291833823, 0.00217110198301286, 0.0021719025368423374, 0.002172703090671815, 0.0021735036445012924, 0.00217430419833077, 0.0021751047521602475, 0.002175905305989725, 0.0021767058598192025, 0.00217750641364868, 0.0021783069674781576, 0.002179107521307635, 0.0021799080751371126, 0.00218070862896659, 0.0021815091827960677, 0.002182309736625545, 0.0021831102904550227, 0.0021839108442845002, 0.0021847113981139778, 0.0021855119519434553, 0.002186312505772933, 0.0021871130596024103, 0.002187913613431888, 0.0021887141672613654, 0.002189514721090843, 0.0021903152749203204, 0.002191115828749798, 0.0021919163825792755, 0.002192716936408753, 0.0021935174902382305, 0.002194318044067708, 0.0021951185978971856, 0.002195919151726663, 0.0021967197055561406, 0.002197520259385618, 0.0021983208132150957, 0.002199121367044573, 0.0021999219208740507, 0.0022007224747035282, 0.0022015230285330058, 0.0022023235823624833, 0.002203124136191961, 0.0022039246900214383, 0.002204725243850916, 0.0022055257976803934, 0.002206326351509871, 0.0022071269053393484, 0.002207927459168826, 0.0022087280129983035, 0.002209528566827781, 0.0022103291206572585, 0.002211129674486736, 0.0022119302283162136, 0.002212730782145691, 0.0022135313359751686, 0.002214331889804646, 0.0022151324436341237, 0.002215932997463601, 0.0022167335512930787, 0.0022175341051225562, 0.0022183346589520338, 0.0022191352127815113, 0.002219935766610989, 0.0022207363204404663, 0.002221536874269944, 0.0022223374280994214, 0.002223137981928899, 0.0022239385357583764, 0.002224739089587854, 0.0022255396434173315, 0.002226340197246809, 0.0022271407510762865, 0.002227941304905764, 0.0022287418587352416, 0.002229542412564719, 0.0022303429663941966, 0.002231143520223674, 0.0022319440740531517, 0.002232744627882629, 0.0022335451817121067, 0.0022343457355415842, 0.0022351462893710618, 0.0022359468432005393, 0.002236747397030017, 0.0022375479508594943, 0.002238348504688972, 0.0022391490585184494, 0.002239949612347927, 0.0022407501661774044, 0.002241550720006882, 0.0022423512738363595, 0.002243151827665837, 0.0022439523814953145, 0.002244752935324792, 0.0022455534891542696, 0.002246354042983747, 0.0022471545968132246, 0.002247955150642702, 0.0022487557044721797, 0.002249556258301657, 0.0022503568121311347, 0.0022511573659606122, 0.0022519579197900898, 0.0022527584736195673, 0.002253559027449045, 0.0022543595812785223, 0.002255160135108, 0.0022559606889374774, 0.002256761242766955, 0.0022575617965964324, 0.00225836235042591, 0.0022591629042553875, 0.002259963458084865, 0.0022607640119143425, 0.00226156456574382, 0.0022623651195732976, 0.002263165673402775, 0.0022639662272322526, 0.00226476678106173, 0.0022655673348912077, 0.002266367888720685, 0.0022671684425501627, 0.0022679689963796402, 0.0022687695502091178, 0.0022695701040385953, 0.002270370657868073, 0.0022711712116975503, 0.002271971765527028, 0.0022727723193565054, 0.002273572873185983, 0.0022743734270154604, 0.002275173980844938, 0.0022759745346744155, 0.002276775088503893, 0.0022775756423333705, 0.002278376196162848, 0.0022791767499923256, 0.002279977303821803, 0.0022807778576512806, 0.002281578411480758, 0.0022823789653102357, 0.002283179519139713, 0.0022839800729691907, 0.0022847806267986682, 0.0022855811806281458, 0.0022863817344576233, 0.002287182288287101, 0.0022879828421165783, 0.002288783395946056, 0.0022895839497755334, 0.002290384503605011, 0.0022911850574344884, 0.002291985611263966, 0.0022927861650934435, 0.002293586718922921, 0.0022943872727523985, 0.002295187826581876, 0.0022959883804113536, 0.002296788934240831, 0.0022975894880703086, 0.002298390041899786, 0.0022991905957292637, 0.002299991149558741, 0.0023007917033882187, 0.0023015922572176962, 0.0023023928110471738, 0.0023031933648766513, 0.002303993918706129, 0.0023047944725356063, 0.002305595026365084, 0.0023063955801945614, 0.002307196134024039, 0.0023079966878535164, 0.002308797241682994, 0.0023095977955124715, 0.002310398349341949, 0.0023111989031714265, 0.002311999457000904, 0.0023128000108303816, 0.002313600564659859, 0.0023144011184893366, 0.002315201672318814, 0.0023160022261482917, 0.002316802779977769, 0.0023176033338072467, 0.0023184038876367242, 0.0023192044414662018, 0.0023200049952956793, 0.002320805549125157, 0.0023216061029546343, 0.002322406656784112, 0.0023232072106135894, 0.002324007764443067, 0.0023248083182725444, 0.002325608872102022, 0.0023264094259314995, 0.002327209979760977, 0.0023280105335904545, 0.002328811087419932, 0.0023296116412494096, 0.002330412195078887, 0.0023312127489083646, 0.002332013302737842, 0.0023328138565673197, 0.002333614410396797, 0.0023344149642262747, 0.0023352155180557522, 0.0023360160718852298, 0.0023368166257147073, 0.002337617179544185, 0.0023384177333736623, 0.00233921828720314, 0.0023400188410326174, 0.002340819394862095, 0.0023416199486915724, 0.00234242050252105, 0.0023432210563505275, 0.002344021610180005, 0.0023448221640094825, 0.00234562271783896, 0.0023464232716684376, 0.002347223825497915, 0.0023480243793273926, 0.00234882493315687, 0.0023496254869863477, 0.002350426040815825, 0.0023512265946453027, 0.0023520271484747802, 0.0023528277023042578, 0.0023536282561337353, 0.002354428809963213, 0.0023552293637926903, 0.002356029917622168, 0.0023568304714516454, 0.002357631025281123, 0.0023584315791106004, 0.002359232132940078, 0.0023600326867695555, 0.002360833240599033, 0.0023616337944285105, 0.002362434348257988, 0.0023632349020874656, 0.002364035455916943, 0.0023648360097464206, 0.002365636563575898, 0.0023664371174053757, 0.002367237671234853, 0.0023680382250643307, 0.0023688387788938082, 0.0023696393327232858, 0.0023704398865527633, 0.002371240440382241, 0.0023720409942117183, 0.002372841548041196, 0.0023736421018706734, 0.002374442655700151, 0.0023752432095296284, 0.002376043763359106, 0.0023768443171885835, 0.002377644871018061, 0.0023784454248475385, 0.002379245978677016, 0.0023800465325064936, 0.002380847086335971, 0.0023816476401654486, 0.002382448193994926, 0.0023832487478244037, 0.002384049301653881, 0.0023848498554833587, 0.0023856504093128362, 0.0023864509631423138, 0.0023872515169717913, 0.002388052070801269, 0.0023888526246307463, 0.002389653178460224, 0.0023904537322897014, 0.002391254286119179, 0.0023920548399486564, 0.002392855393778134, 0.0023936559476076115, 0.002394456501437089, 0.0023952570552665665, 0.002396057609096044, 0.0023968581629255216, 0.002397658716754999, 0.0023984592705844766, 0.002399259824413954, 0.0024000603782434317, 0.002400860932072909, 0.0024016614859023867, 0.0024024620397318642, 0.0024032625935613418, 0.0024040631473908193, 0.002404863701220297, 0.0024056642550497743, 0.002406464808879252, 0.0024072653627087294, 0.002408065916538207, 0.0024088664703676844, 0.002409667024197162, 0.0024104675780266395, 0.002411268131856117, 0.0024120686856855945, 0.002412869239515072, 0.0024136697933445496, 0.002414470347174027, 0.0024152709010035046, 0.002416071454832982, 0.0024168720086624597, 0.002417672562491937, 0.0024184731163214147, 0.0024192736701508922, 0.0024200742239803698, 0.0024208747778098473, 0.002421675331639325, 0.0024224758854688023, 0.00242327643929828, 0.0024240769931277574, 0.002424877546957235, 0.0024256781007867124, 0.00242647865461619, 0.0024272792084456675, 0.002428079762275145, 0.0024288803161046225, 0.0024296808699341, 0.0024304814237635776, 0.002431281977593055, 0.0024320825314225326, 0.00243288308525201, 0.0024336836390814877, 0.002434484192910965, 0.0024352847467404427, 0.0024360853005699202, 0.0024368858543993978, 0.0024376864082288753, 0.002438486962058353, 0.0024392875158878303, 0.002440088069717308, 0.0024408886235467854, 0.002441689177376263, 0.0024424897312057404, 0.002443290285035218, 0.0024440908388646955, 0.002444891392694173, 0.0024456919465236505, 0.002446492500353128, 0.0024472930541826056, 0.002448093608012083, 0.0024488941618415606, 0.002449694715671038, 0.0024504952695005157, 0.002451295823329993, 0.0024520963771594707, 0.0024528969309889482, 0.0024536974848184258, 0.0024544980386479033, 0.002455298592477381, 0.0024560991463068583, 0.002456899700136336, 0.0024577002539658134, 0.002458500807795291, 0.0024593013616247684, 0.002460101915454246, 0.0024609024692837235, 0.002461703023113201, 0.0024625035769426785, 0.002463304130772156, 0.0024641046846016336, 0.002464905238431111, 0.0024657057922605886, 0.002466506346090066, 0.0024673068999195437, 0.002468107453749021, 0.0024689080075784987, 0.0024697085614079762, 0.0024705091152374538, 0.0024713096690669313, 0.002472110222896409, 0.0024729107767258863, 0.002473711330555364, 0.0024745118843848414, 0.002475312438214319, 0.0024761129920437964, 0.002476913545873274, 0.0024777140997027515, 0.002478514653532229, 0.0024793152073617065, 0.002480115761191184, 0.0024809163150206616, 0.002481716868850139, 0.0024825174226796166, 0.002483317976509094, 0.0024841185303385717, 0.002484919084168049, 0.0024857196379975267, 0.0024865201918270042, 0.0024873207456564818, 0.0024881212994859593, 0.002488921853315437, 0.0024897224071449143, 0.002490522960974392, 0.0024913235148038694, 0.002492124068633347, 0.0024929246224628244, 0.002493725176292302, 0.0024945257301217795, 0.002495326283951257, 0.0024961268377807345, 0.002496927391610212, 0.0024977279454396896, 0.002498528499269167, 0.0024993290530986446, 0.002500129606928122, 0.0025009301607575997, 0.002501730714587077, 0.0025025312684165547, 0.0025033318222460322, 0.0025041323760755098, 0.0025049329299049873, 0.002505733483734465, 0.0025065340375639423, 0.00250733459139342, 0.0025081351452228974, 0.002508935699052375, 0.0025097362528818524, 0.00251053680671133, 0.0025113373605408075, 0.002512137914370285, 0.0025129384681997625, 0.00251373902202924, 0.0025145395758587176, 0.002515340129688195, 0.0025161406835176726, 0.00251694123734715, 0.0025177417911766277, 0.002518542345006105, 0.0025193428988355827, 0.0025201434526650602, 0.0025209440064945378, 0.0025217445603240153, 0.002522545114153493, 0.0025233456679829703, 0.002524146221812448, 0.0025249467756419254, 0.002525747329471403, 0.0025265478833008804, 0.002527348437130358, 0.0025281489909598355, 0.002528949544789313, 0.0025297500986187905, 0.002530550652448268, 0.0025313512062777456, 0.002532151760107223, 0.0025329523139367006, 0.002533752867766178, 0.0025345534215956557, 0.002535353975425133, 0.0025361545292546107, 0.0025369550830840882, 0.0025377556369135658, 0.0025385561907430433, 0.002539356744572521, 0.0025401572984019983, 0.002540957852231476, 0.0025417584060609534, 0.002542558959890431, 0.0025433595137199084, 0.002544160067549386, 0.0025449606213788635, 0.002545761175208341, 0.0025465617290378185, 0.002547362282867296, 0.0025481628366967736, 0.002548963390526251, 0.0025497639443557286, 0.002550564498185206, 0.0025513650520146837, 0.002552165605844161, 0.0025529661596736387, 0.0025537667135031162, 0.0025545672673325938, 0.0025553678211620713, 0.002556168374991549, 0.0025569689288210263, 0.002557769482650504, 0.0025585700364799814, 0.002559370590309459, 0.0025601711441389364, 0.002560971697968414, 0.0025617722517978915, 0.002562572805627369, 0.0025633733594568465, 0.002564173913286324, 0.0025649744671158016, 0.002565775020945279, 0.0025665755747747566, 0.002567376128604234, 0.0025681766824337117, 0.002568977236263189, 0.0025697777900926667, 0.0025705783439221442, 0.0025713788977516218, 0.0025721794515810993, 0.002572980005410577, 0.0025737805592400543, 0.002574581113069532, 0.0025753816668990094, 0.002576182220728487, 0.0025769827745579644, 0.002577783328387442, 0.0025785838822169195, 0.002579384436046397, 0.0025801849898758745, 0.002580985543705352, 0.0025817860975348296, 0.002582586651364307, 0.0025833872051937846, 0.002584187759023262, 0.0025849883128527397, 0.002585788866682217, 0.0025865894205116947, 0.0025873899743411722, 0.0025881905281706498, 0.0025889910820001273, 0.002589791635829605, 0.0025905921896590823, 0.00259139274348856, 0.0025921932973180374, 0.002592993851147515, 0.0025937944049769924, 0.00259459495880647, 0.0025953955126359475, 0.002596196066465425, 0.0025969966202949025, 0.00259779717412438, 0.0025985977279538576, 0.002599398281783335, 0.0026001988356128126, 0.00260099938944229, 0.0026017999432717677, 0.002602600497101245, 0.0026034010509307227, 0.0026042016047602002, 0.0026050021585896778, 0.0026058027124191553, 0.002606603266248633, 0.0026074038200781103, 0.002608204373907588, 0.0026090049277370654, 0.002609805481566543, 0.0026106060353960204, 0.002611406589225498, 0.0026122071430549755, 0.002613007696884453, 0.0026138082507139305, 0.002614608804543408, 0.0026154093583728856, 0.002616209912202363, 0.0026170104660318406, 0.002617811019861318, 0.0026186115736907957, 0.002619412127520273, 0.0026202126813497507, 0.0026210132351792283, 0.0026218137890087058, 0.0026226143428381833, 0.002623414896667661, 0.0026242154504971383, 0.002625016004326616, 0.0026258165581560934, 0.002626617111985571, 0.0026274176658150484, 0.002628218219644526, 0.0026290187734740035, 0.002629819327303481, 0.0026306198811329585, 0.002631420434962436, 0.0026322209887919136, 0.002633021542621391, 0.0026338220964508686, 0.002634622650280346, 0.0026354232041098237, 0.002636223757939301, 0.0026370243117687787, 0.0026378248655982563, 0.0026386254194277338, 0.0026394259732572113, 0.002640226527086689, 0.0026410270809161663, 0.002641827634745644, 0.0026426281885751214, 0.002643428742404599, 0.0026442292962340764, 0.002645029850063554, 0.0026458304038930315, 0.002646630957722509, 0.0026474315115519865, 0.002648232065381464, 0.0026490326192109416, 0.002649833173040419, 0.0026506337268698966, 0.002651434280699374, 0.0026522348345288517, 0.002653035388358329, 0.0026538359421878067, 0.0026546364960172843, 0.0026554370498467618, 0.0026562376036762393, 0.002657038157505717, 0.0026578387113351943, 0.002658639265164672, 0.0026594398189941494, 0.002660240372823627, 0.0026610409266531044, 0.002661841480482582, 0.0026626420343120595, 0.002663442588141537, 0.0026642431419710145, 0.002665043695800492, 0.0026658442496299696, 0.002666644803459447, 0.0026674453572889246, 0.002668245911118402, 0.0026690464649478797, 0.002669847018777357, 0.0026706475726068347, 0.0026714481264363123, 0.0026722486802657898, 0.0026730492340952673, 0.002673849787924745, 0.0026746503417542223, 0.0026754508955837, 0.0026762514494131774, 0.002677052003242655, 0.0026778525570721324, 0.00267865311090161, 0.0026794536647310875, 0.002680254218560565, 0.0026810547723900425, 0.00268185532621952, 0.0026826558800489976, 0.002683456433878475, 0.0026842569877079526, 0.00268505754153743, 0.0026858580953669077, 0.002686658649196385, 0.0026874592030258627, 0.0026882597568553403, 0.0026890603106848178, 0.0026898608645142953, 0.002690661418343773, 0.0026914619721732503, 0.002692262526002728, 0.0026930630798322054, 0.002693863633661683, 0.0026946641874911604, 0.002695464741320638, 0.0026962652951501155, 0.002697065848979593, 0.0026978664028090705, 0.002698666956638548, 0.0026994675104680256, 0.002700268064297503, 0.0027010686181269806, 0.002701869171956458, 0.0027026697257859357, 0.002703470279615413, 0.0027042708334448907, 0.0027050713872743683, 0.0027058719411038458, 0.0027066724949333233, 0.002707473048762801, 0.0027082736025922784, 0.002709074156421756, 0.0027098747102512334, 0.002710675264080711, 0.0027114758179101884, 0.002712276371739666, 0.0027130769255691435, 0.002713877479398621, 0.0027146780332280985, 0.002715478587057576, 0.0027162791408870536, 0.002717079694716531, 0.0027178802485460086, 0.002718680802375486, 0.0027194813562049637, 0.002720281910034441, 0.0027210824638639187, 0.0027218830176933963, 0.0027226835715228738, 0.0027234841253523513, 0.002724284679181829, 0.0027250852330113064, 0.002725885786840784, 0.0027266863406702614, 0.002727486894499739, 0.0027282874483292164, 0.002729088002158694, 0.0027298885559881715, 0.002730689109817649, 0.0027314896636471265, 0.002732290217476604, 0.0027330907713060816, 0.002733891325135559, 0.0027346918789650366, 0.002735492432794514, 0.0027362929866239917, 0.002737093540453469, 0.0027378940942829467, 0.0027386946481124243, 0.0027394952019419018, 0.0027402957557713793, 0.002741096309600857, 0.0027418968634303344, 0.002742697417259812, 0.0027434979710892894, 0.002744298524918767, 0.0027450990787482444, 0.002745899632577722, 0.0027467001864071995, 0.002747500740236677, 0.0027483012940661545, 0.002749101847895632, 0.0027499024017251096, 0.002750702955554587, 0.0027515035093840646, 0.002752304063213542, 0.0027531046170430197, 0.002753905170872497, 0.0027547057247019747, 0.0027555062785314523, 0.0027563068323609298, 0.0027571073861904073, 0.002757907940019885, 0.0027587084938493624, 0.00275950904767884, 0.0027603096015083174, 0.002761110155337795, 0.0027619107091672724, 0.00276271126299675, 0.0027635118168262275, 0.002764312370655705, 0.0027651129244851825, 0.00276591347831466, 0.0027667140321441376, 0.002767514585973615, 0.0027683151398030926, 0.00276911569363257, 0.0027699162474620477, 0.002770716801291525, 0.0027715173551210027, 0.0027723179089504803, 0.0027731184627799578, 0.0027739190166094353, 0.002774719570438913, 0.0027755201242683904, 0.002776320678097868, 0.0027771212319273454, 0.002777921785756823, 0.0027787223395863004, 0.002779522893415778, 0.0027803234472452555, 0.002781124001074733, 0.0027819245549042105, 0.002782725108733688, 0.0027835256625631656, 0.002784326216392643, 0.0027851267702221206, 0.002785927324051598, 0.0027867278778810757, 0.002787528431710553, 0.0027883289855400307, 0.0027891295393695083, 0.0027899300931989858, 0.0027907306470284633, 0.002791531200857941, 0.0027923317546874184, 0.002793132308516896, 0.0027939328623463734, 0.002794733416175851, 0.0027955339700053284, 0.002796334523834806, 0.0027971350776642835, 0.002797935631493761, 0.0027987361853232385, 0.002799536739152716, 0.0028003372929821936, 0.002801137846811671, 0.0028019384006411486, 0.002802738954470626, 0.0028035395083001037, 0.002804340062129581, 0.0028051406159590587, 0.0028059411697885363, 0.002806741723618014, 0.0028075422774474913, 0.002808342831276969, 0.0028091433851064464, 0.002809943938935924, 0.0028107444927654014, 0.002811545046594879, 0.0028123456004243565, 0.002813146154253834, 0.0028139467080833115, 0.002814747261912789, 0.0028155478157422665, 0.002816348369571744, 0.0028171489234012216, 0.002817949477230699, 0.0028187500310601766, 0.002819550584889654, 0.0028203511387191317, 0.002821151692548609, 0.0028219522463780867, 0.0028227528002075643, 0.002823553354037042, 0.0028243539078665193, 0.002825154461695997, 0.0028259550155254744, 0.002826755569354952, 0.0028275561231844294, 0.002828356677013907, 0.0028291572308433845, 0.002829957784672862, 0.0028307583385023395, 0.002831558892331817, 0.0028323594461612945, 0.002833159999990772, 0.0028339605538202496, 0.002834761107649727, 0.0028355616614792046, 0.002836362215308682, 0.0028371627691381597, 0.002837963322967637, 0.0028387638767971147, 0.0028395644306265923, 0.00284036498445607, 0.0028411655382855473, 0.002841966092115025, 0.0028427666459445024, 0.00284356719977398, 0.0028443677536034574, 0.002845168307432935, 0.0028459688612624125, 0.00284676941509189, 0.0028475699689213675, 0.002848370522750845, 0.0028491710765803225, 0.0028499716304098, 0.0028507721842392776, 0.002851572738068755, 0.0028523732918982326, 0.00285317384572771, 0.0028539743995571877, 0.002854774953386665, 0.0028555755072161427, 0.0028563760610456203, 0.002857176614875098, 0.0028579771687045753, 0.002858777722534053, 0.0028595782763635304, 0.002860378830193008, 0.0028611793840224854, 0.002861979937851963, 0.0028627804916814405, 0.002863581045510918, 0.0028643815993403955, 0.002865182153169873, 0.0028659827069993505, 0.002866783260828828, 0.0028675838146583056, 0.002868384368487783, 0.0028691849223172606, 0.002869985476146738, 0.0028707860299762157, 0.002871586583805693, 0.0028723871376351707, 0.0028731876914646483, 0.002873988245294126, 0.0028747887991236033, 0.002875589352953081, 0.0028763899067825584, 0.002877190460612036, 0.0028779910144415134, 0.002878791568270991, 0.0028795921221004685, 0.002880392675929946, 0.0028811932297594235, 0.002881993783588901, 0.0028827943374183785, 0.002883594891247856, 0.0028843954450773336, 0.002885195998906811, 0.0028859965527362886, 0.002886797106565766, 0.0028875976603952437, 0.002888398214224721, 0.0028891987680541987, 0.0028899993218836763, 0.002890799875713154, 0.0028916004295426313, 0.002892400983372109, 0.0028932015372015864, 0.002894002091031064, 0.0028948026448605414, 0.002895603198690019, 0.0028964037525194965, 0.002897204306348974, 0.0028980048601784515, 0.002898805414007929, 0.0028996059678374065, 0.002900406521666884, 0.0029012070754963616, 0.002902007629325839, 0.0029028081831553166, 0.002903608736984794, 0.0029044092908142717, 0.002905209844643749, 0.0029060103984732267, 0.0029068109523027043, 0.002907611506132182, 0.0029084120599616593, 0.002909212613791137, 0.0029100131676206144, 0.002910813721450092, 0.0029116142752795694, 0.002912414829109047, 0.0029132153829385245, 0.002914015936768002, 0.0029148164905974795, 0.002915617044426957, 0.0029164175982564346, 0.002917218152085912, 0.0029180187059153896, 0.002918819259744867, 0.0029196198135743446, 0.002920420367403822, 0.0029212209212332997, 0.0029220214750627772, 0.0029228220288922547, 0.0029236225827217323, 0.00292442313655121, 0.0029252236903806873, 0.002926024244210165, 0.0029268247980396424, 0.00292762535186912, 0.0029284259056985974, 0.002929226459528075, 0.0029300270133575525, 0.00293082756718703, 0.0029316281210165075, 0.002932428674845985, 0.0029332292286754626, 0.00293402978250494, 0.0029348303363344176, 0.002935630890163895, 0.0029364314439933726, 0.00293723199782285, 0.0029380325516523277, 0.0029388331054818052, 0.0029396336593112827, 0.0029404342131407603, 0.002941234766970238, 0.0029420353207997153, 0.002942835874629193, 0.0029436364284586704, 0.002944436982288148, 0.0029452375361176254, 0.002946038089947103, 0.0029468386437765805, 0.002947639197606058, 0.0029484397514355355, 0.002949240305265013, 0.0029500408590944906, 0.002950841412923968, 0.0029516419667534456, 0.002952442520582923, 0.0029532430744124006, 0.002954043628241878, 0.0029548441820713557, 0.0029556447359008332, 0.0029564452897303107, 0.0029572458435597883, 0.002958046397389266, 0.0029588469512187433, 0.002959647505048221, 0.0029604480588776984, 0.002961248612707176, 0.0029620491665366534, 0.002962849720366131, 0.0029636502741956085, 0.002964450828025086, 0.0029652513818545635, 0.002966051935684041, 0.0029668524895135186, 0.002967653043342996, 0.0029684535971724736, 0.002969254151001951, 0.0029700547048314286, 0.002970855258660906, 0.0029716558124903837, 0.0029724563663198612, 0.0029732569201493387, 0.0029740574739788163, 0.002974858027808294, 0.0029756585816377713, 0.002976459135467249, 0.0029772596892967264, 0.002978060243126204, 0.0029788607969556814, 0.002979661350785159, 0.0029804619046146365, 0.002981262458444114, 0.0029820630122735915, 0.002982863566103069, 0.0029836641199325466, 0.002984464673762024, 0.0029852652275915016, 0.002986065781420979, 0.0029868663352504566, 0.002987666889079934, 0.0029884674429094117, 0.0029892679967388892, 0.0029900685505683667, 0.0029908691043978443, 0.002991669658227322, 0.0029924702120567993, 0.002993270765886277, 0.0029940713197157544, 0.002994871873545232, 0.0029956724273747094, 0.002996472981204187, 0.0029972735350336645, 0.002998074088863142, 0.0029988746426926195, 0.002999675196522097, 0.0030004757503515746, 0.003001276304181052, 0.0030020768580105296, 0.003002877411840007, 0.0030036779656694847, 0.003004478519498962, 0.0030052790733284397, 0.0030060796271579172, 0.0030068801809873947, 0.0030076807348168723, 0.00300848128864635, 0.0030092818424758273, 0.003010082396305305, 0.0030108829501347824, 0.00301168350396426, 0.0030124840577937374, 0.003013284611623215, 0.0030140851654526925, 0.00301488571928217, 0.0030156862731116475, 0.003016486826941125, 0.0030172873807706026, 0.00301808793460008, 0.0030188884884295576, 0.003019689042259035, 0.0030204895960885127, 0.00302129014991799, 0.0030220907037474677, 0.0030228912575769452, 0.0030236918114064227, 0.0030244923652359003, 0.003025292919065378, 0.0030260934728948553, 0.003026894026724333, 0.0030276945805538104, 0.003028495134383288, 0.0030292956882127654, 0.003030096242042243, 0.0030308967958717205, 0.003031697349701198, 0.0030324979035306755, 0.003033298457360153, 0.0030340990111896306, 0.003034899565019108, 0.0030357001188485856, 0.003036500672678063, 0.0030373012265075407, 0.003038101780337018, 0.0030389023341664957, 0.0030397028879959732, 0.0030405034418254507, 0.0030413039956549283, 0.003042104549484406, 0.0030429051033138833, 0.003043705657143361, 0.0030445062109728384, 0.003045306764802316, 0.0030461073186317934, 0.003046907872461271, 0.0030477084262907485, 0.003048508980120226, 0.0030493095339497035, 0.003050110087779181, 0.0030509106416086586, 0.003051711195438136, 0.0030525117492676136, 0.003053312303097091, 0.0030541128569265687, 0.003054913410756046, 0.0030557139645855237, 0.0030565145184150012, 0.0030573150722444787, 0.0030581156260739563, 0.003058916179903434, 0.0030597167337329113, 0.003060517287562389, 0.0030613178413918664, 0.003062118395221344, 0.0030629189490508214, 0.003063719502880299, 0.0030645200567097765, 0.003065320610539254, 0.0030661211643687315, 0.003066921718198209, 0.0030677222720276866, 0.003068522825857164, 0.0030693233796866416, 0.003070123933516119, 0.0030709244873455967, 0.003071725041175074, 0.0030725255950045517, 0.0030733261488340292, 0.0030741267026635067, 0.0030749272564929843, 0.003075727810322462, 0.0030765283641519393, 0.003077328917981417, 0.0030781294718108944, 0.003078930025640372, 0.0030797305794698494, 0.003080531133299327, 0.0030813316871288045, 0.003082132240958282, 0.0030829327947877595, 0.003083733348617237, 0.0030845339024467146, 0.003085334456276192, 0.0030861350101056696, 0.003086935563935147, 0.0030877361177646247, 0.003088536671594102, 0.0030893372254235797, 0.0030901377792530572, 0.0030909383330825347, 0.0030917388869120123, 0.00309253944074149, 0.0030933399945709673, 0.003094140548400445, 0.0030949411022299224, 0.0030957416560594, 0.0030965422098888774, 0.003097342763718355, 0.0030981433175478325, 0.00309894387137731, 0.0030997444252067875, 0.003100544979036265, 0.0031013455328657426, 0.00310214608669522, 0.0031029466405246976, 0.003103747194354175, 0.0031045477481836527, 0.00310534830201313, 0.0031061488558426077, 0.0031069494096720852, 0.0031077499635015628, 0.0031085505173310403, 0.003109351071160518, 0.0031101516249899953, 0.003110952178819473, 0.0031117527326489504, 0.003112553286478428, 0.0031133538403079054, 0.003114154394137383, 0.0031149549479668605, 0.003115755501796338, 0.0031165560556258155, 0.003117356609455293, 0.0031181571632847706, 0.003118957717114248, 0.0031197582709437256, 0.003120558824773203, 0.0031213593786026807, 0.003122159932432158, 0.0031229604862616357, 0.0031237610400911132, 0.0031245615939205908, 0.0031253621477500683, 0.003126162701579546, 0.0031269632554090233, 0.003127763809238501, 0.0031285643630679784, 0.003129364916897456, 0.0031301654707269334, 0.003130966024556411, 0.0031317665783858885, 0.003132567132215366, 0.0031333676860448435, 0.003134168239874321, 0.0031349687937037986, 0.003135769347533276, 0.0031365699013627536, 0.003137370455192231, 0.0031381710090217087, 0.003138971562851186, 0.0031397721166806637, 0.0031405726705101412, 0.0031413732243396188, 0.0031421737781690963, 0.003142974331998574, 0.0031437748858280513, 0.003144575439657529, 0.0031453759934870064, 0.003146176547316484, 0.0031469771011459614, 0.003147777654975439, 0.0031485782088049165, 0.003149378762634394, 0.0031501793164638715, 0.003150979870293349, 0.0031517804241228266, 0.003152580977952304, 0.0031533815317817816, 0.003154182085611259, 0.0031549826394407367, 0.003155783193270214, 0.0031565837470996917, 0.0031573843009291692, 0.0031581848547586468, 0.0031589854085881243, 0.003159785962417602, 0.0031605865162470793, 0.003161387070076557, 0.0031621876239060344, 0.003162988177735512, 0.0031637887315649894, 0.003164589285394467, 0.0031653898392239445, 0.003166190393053422, 0.0031669909468828995, 0.003167791500712377, 0.0031685920545418546, 0.003169392608371332, 0.0031701931622008096, 0.003170993716030287, 0.0031717942698597647, 0.003172594823689242, 0.0031733953775187197, 0.0031741959313481972, 0.0031749964851776748, 0.0031757970390071523, 0.00317659759283663, 0.0031773981466661073, 0.003178198700495585, 0.0031789992543250624, 0.00317979980815454, 0.0031806003619840174, 0.003181400915813495, 0.0031822014696429725, 0.00318300202347245, 0.0031838025773019275, 0.003184603131131405, 0.0031854036849608826, 0.00318620423879036, 0.0031870047926198376, 0.003187805346449315, 0.0031886059002787927, 0.00318940645410827, 0.0031902070079377477, 0.0031910075617672252, 0.0031918081155967028, 0.0031926086694261803, 0.003193409223255658, 0.0031942097770851353, 0.003195010330914613, 0.0031958108847440904, 0.003196611438573568, 0.0031974119924030454, 0.003198212546232523, 0.0031990131000620005, 0.003199813653891478, 0.0032006142077209555, 0.003201414761550433, 0.0032022153153799106, 0.003203015869209388, 0.0032038164230388656, 0.003204616976868343, 0.0032054175306978207, 0.003206218084527298, 0.0032070186383567757, 0.0032078191921862532, 0.0032086197460157308, 0.0032094202998452083, 0.003210220853674686, 0.0032110214075041633, 0.003211821961333641, 0.0032126225151631184, 0.003213423068992596, 0.0032142236228220734, 0.003215024176651551, 0.0032158247304810285, 0.003216625284310506, 0.0032174258381399835, 0.003218226391969461, 0.0032190269457989386, 0.003219827499628416, 0.0032206280534578936, 0.003221428607287371, 0.0032222291611168487, 0.003223029714946326, 0.0032238302687758037, 0.0032246308226052812, 0.0032254313764347588, 0.0032262319302642363, 0.003227032484093714, 0.0032278330379231913, 0.003228633591752669, 0.0032294341455821464, 0.003230234699411624, 0.0032310352532411014, 0.003231835807070579, 0.0032326363609000565, 0.003233436914729534, 0.0032342374685590115, 0.003235038022388489, 0.0032358385762179666, 0.003236639130047444, 0.0032374396838769216, 0.003238240237706399, 0.0032390407915358767, 0.003239841345365354, 0.0032406418991948317, 0.0032414424530243092, 0.0032422430068537868, 0.0032430435606832643, 0.003243844114512742, 0.0032446446683422193, 0.003245445222171697, 0.0032462457760011744, 0.003247046329830652, 0.0032478468836601294, 0.003248647437489607, 0.0032494479913190845, 0.003250248545148562, 0.0032510490989780395, 0.003251849652807517, 0.0032526502066369946, 0.003253450760466472, 0.0032542513142959496, 0.003255051868125427, 0.0032558524219549047, 0.003256652975784382, 0.0032574535296138597, 0.0032582540834433372, 0.0032590546372728148, 0.0032598551911022923, 0.00326065574493177, 0.0032614562987612473, 0.003262256852590725, 0.0032630574064202024, 0.00326385796024968, 0.0032646585140791574, 0.003265459067908635, 0.0032662596217381125, 0.00326706017556759, 0.0032678607293970675, 0.003268661283226545, 0.0032694618370560226, 0.0032702623908855, 0.0032710629447149776, 0.003271863498544455, 0.0032726640523739327, 0.00327346460620341, 0.0032742651600328877, 0.0032750657138623652, 0.0032758662676918428, 0.0032766668215213203, 0.003277467375350798, 0.0032782679291802753, 0.003279068483009753, 0.0032798690368392304, 0.003280669590668708, 0.0032814701444981854, 0.003282270698327663, 0.0032830712521571405, 0.003283871805986618, 0.0032846723598160955, 0.003285472913645573, 0.0032862734674750506, 0.003287074021304528, 0.0032878745751340056, 0.003288675128963483, 0.0032894756827929607, 0.003290276236622438, 0.0032910767904519157, 0.0032918773442813932, 0.0032926778981108708, 0.0032934784519403483, 0.003294279005769826, 0.0032950795595993033, 0.003295880113428781, 0.0032966806672582584, 0.003297481221087736, 0.0032982817749172134, 0.003299082328746691, 0.0032998828825761685, 0.003300683436405646, 0.0033014839902351235, 0.003302284544064601, 0.0033030850978940786, 0.003303885651723556, 0.0033046862055530336, 0.003305486759382511, 0.0033062873132119887, 0.003307087867041466, 0.0033078884208709437, 0.0033086889747004212, 0.0033094895285298988, 0.0033102900823593763, 0.003311090636188854, 0.0033118911900183313, 0.003312691743847809, 0.0033134922976772864, 0.003314292851506764, 0.0033150934053362414, 0.003315893959165719, 0.0033166945129951965, 0.003317495066824674, 0.0033182956206541515, 0.003319096174483629, 0.0033198967283131066, 0.003320697282142584, 0.0033214978359720616, 0.003322298389801539, 0.0033230989436310167, 0.003323899497460494, 0.0033247000512899717, 0.0033255006051194492, 0.0033263011589489268, 0.0033271017127784043, 0.003327902266607882, 0.0033287028204373593, 0.003329503374266837, 0.0033303039280963144, 0.003331104481925792, 0.0033319050357552694, 0.003332705589584747, 0.0033335061434142245, 0.003334306697243702, 0.0033351072510731795, 0.003335907804902657, 0.0033367083587321346, 0.003337508912561612, 0.0033383094663910896, 0.003339110020220567, 0.0033399105740500447, 0.003340711127879522, 0.0033415116817089997, 0.0033423122355384772, 0.0033431127893679548, 0.0033439133431974323, 0.00334471389702691, 0.0033455144508563873, 0.003346315004685865, 0.0033471155585153424, 0.00334791611234482, 0.0033487166661742974, 0.003349517220003775, 0.0033503177738332525, 0.00335111832766273, 0.0033519188814922075, 0.003352719435321685, 0.0033535199891511626, 0.00335432054298064, 0.0033551210968101176, 0.003355921650639595, 0.0033567222044690727, 0.00335752275829855, 0.0033583233121280277, 0.0033591238659575052, 0.0033599244197869828, 0.0033607249736164603, 0.003361525527445938, 0.0033623260812754153, 0.003363126635104893, 0.0033639271889343704, 0.003364727742763848, 0.0033655282965933254, 0.003366328850422803, 0.0033671294042522805, 0.003367929958081758, 0.0033687305119112355, 0.003369531065740713, 0.0033703316195701906, 0.003371132173399668, 0.0033719327272291456, 0.003372733281058623, 0.0033735338348881007, 0.003374334388717578, 0.0033751349425470557, 0.0033759354963765332, 0.0033767360502060108, 0.0033775366040354883, 0.003378337157864966, 0.0033791377116944433, 0.003379938265523921, 0.0033807388193533984, 0.003381539373182876, 0.0033823399270123534, 0.003383140480841831, 0.0033839410346713085, 0.003384741588500786, 0.0033855421423302635, 0.003386342696159741, 0.0033871432499892186, 0.003387943803818696, 0.0033887443576481736, 0.003389544911477651, 0.0033903454653071287, 0.003391146019136606, 0.0033919465729660837, 0.0033927471267955612, 0.0033935476806250388, 0.0033943482344545163, 0.003395148788283994, 0.0033959493421134713, 0.003396749895942949, 0.0033975504497724264, 0.003398351003601904, 0.0033991515574313814, 0.003399952111260859, 0.0034007526650903365, 0.003401553218919814, 0.0034023537727492915, 0.003403154326578769, 0.0034039548804082466, 0.003404755434237724, 0.0034055559880672016, 0.003406356541896679, 0.0034071570957261567, 0.003407957649555634, 0.0034087582033851117, 0.0034095587572145892, 0.0034103593110440668, 0.0034111598648735443, 0.003411960418703022, 0.0034127609725324993, 0.003413561526361977, 0.0034143620801914544, 0.003415162634020932, 0.0034159631878504094, 0.003416763741679887, 0.0034175642955093645, 0.003418364849338842, 0.0034191654031683195, 0.003419965956997797, 0.0034207665108272746, 0.003421567064656752, 0.0034223676184862296, 0.003423168172315707, 0.0034239687261451847, 0.003424769279974662, 0.0034255698338041397, 0.0034263703876336172, 0.0034271709414630948, 0.0034279714952925723, 0.00342877204912205, 0.0034295726029515273, 0.003430373156781005, 0.0034311737106104824, 0.00343197426443996, 0.0034327748182694374, 0.003433575372098915, 0.0034343759259283925, 0.00343517647975787, 0.0034359770335873475, 0.003436777587416825, 0.0034375781412463026, 0.00343837869507578, 0.0034391792489052576, 0.003439979802734735, 0.0034407803565642127, 0.00344158091039369, 0.0034423814642231677, 0.0034431820180526452, 0.0034439825718821228, 0.0034447831257116003, 0.003445583679541078, 0.0034463842333705553, 0.003447184787200033, 0.0034479853410295104, 0.003448785894858988, 0.0034495864486884654, 0.003450387002517943, 0.0034511875563474205, 0.003451988110176898, 0.0034527886640063755, 0.003453589217835853, 0.0034543897716653306, 0.003455190325494808, 0.0034559908793242856, 0.003456791433153763, 0.0034575919869832407, 0.003458392540812718, 0.0034591930946421957, 0.0034599936484716732, 0.0034607942023011508, 0.0034615947561306283, 0.003462395309960106, 0.0034631958637895833, 0.003463996417619061, 0.0034647969714485384, 0.003465597525278016, 0.0034663980791074934, 0.003467198632936971, 0.0034679991867664485, 0.003468799740595926, 0.0034696002944254035, 0.003470400848254881, 0.0034712014020843586, 0.003472001955913836, 0.0034728025097433136, 0.003473603063572791, 0.0034744036174022687, 0.003475204171231746, 0.0034760047250612237, 0.0034768052788907012, 0.0034776058327201788, 0.0034784063865496563, 0.003479206940379134, 0.0034800074942086113, 0.003480808048038089, 0.0034816086018675664, 0.003482409155697044, 0.0034832097095265214, 0.003484010263355999, 0.0034848108171854765, 0.003485611371014954, 0.0034864119248444315, 0.003487212478673909, 0.0034880130325033866, 0.003488813586332864, 0.0034896141401623416, 0.003490414693991819, 0.0034912152478212967, 0.003492015801650774, 0.0034928163554802517, 0.0034936169093097292, 0.0034944174631392068, 0.0034952180169686843, 0.003496018570798162, 0.0034968191246276393, 0.003497619678457117, 0.0034984202322865944, 0.003499220786116072, 0.0035000213399455494, 0.003500821893775027, 0.0035016224476045045, 0.003502423001433982, 0.0035032235552634595, 0.003504024109092937, 0.0035048246629224146, 0.003505625216751892, 0.0035064257705813696, 0.003507226324410847, 0.0035080268782403247, 0.003508827432069802, 0.0035096279858992797, 0.0035104285397287572, 0.0035112290935582348, 0.0035120296473877123, 0.00351283020121719, 0.0035136307550466673, 0.003514431308876145, 0.0035152318627056224, 0.0035160324165351, 0.0035168329703645774, 0.003517633524194055, 0.0035184340780235325, 0.00351923463185301, 0.0035200351856824875, 0.003520835739511965, 0.0035216362933414426, 0.00352243684717092, 0.0035232374010003976, 0.003524037954829875, 0.0035248385086593527, 0.00352563906248883, 0.0035264396163183077, 0.0035272401701477852, 0.0035280407239772628, 0.0035288412778067403, 0.003529641831636218, 0.0035304423854656953, 0.003531242939295173, 0.0035320434931246504, 0.003532844046954128, 0.0035336446007836054, 0.003534445154613083, 0.0035352457084425605, 0.003536046262272038, 0.0035368468161015155, 0.003537647369930993, 0.0035384479237604706, 0.003539248477589948, 0.0035400490314194256, 0.003540849585248903, 0.0035416501390783807, 0.003542450692907858, 0.0035432512467373357, 0.0035440518005668132, 0.0035448523543962908, 0.0035456529082257683, 0.003546453462055246, 0.0035472540158847233, 0.003548054569714201, 0.0035488551235436784, 0.003549655677373156, 0.0035504562312026334, 0.003551256785032111, 0.0035520573388615885, 0.003552857892691066, 0.0035536584465205435, 0.003554459000350021, 0.0035552595541794986, 0.003556060108008976, 0.0035568606618384536, 0.003557661215667931, 0.0035584617694974087, 0.003559262323326886, 0.0035600628771563637, 0.0035608634309858412, 0.0035616639848153188, 0.0035624645386447963, 0.003563265092474274, 0.0035640656463037513, 0.003564866200133229, 0.0035656667539627064, 0.003566467307792184, 0.0035672678616216614, 0.003568068415451139, 0.0035688689692806165, 0.003569669523110094, 0.0035704700769395715, 0.003571270630769049, 0.0035720711845985266, 0.003572871738428004, 0.0035736722922574816, 0.003574472846086959, 0.0035752733999164367, 0.003576073953745914, 0.0035768745075753917, 0.0035776750614048692, 0.0035784756152343468, 0.0035792761690638243, 0.003580076722893302, 0.0035808772767227793, 0.003581677830552257, 0.0035824783843817344, 0.003583278938211212, 0.0035840794920406894, 0.003584880045870167, 0.0035856805996996445, 0.003586481153529122, 0.0035872817073585995, 0.003588082261188077, 0.0035888828150175546, 0.003589683368847032, 0.0035904839226765096, 0.003591284476505987, 0.0035920850303354647, 0.003592885584164942, 0.0035936861379944197, 0.0035944866918238973, 0.0035952872456533748, 0.0035960877994828523, 0.00359688835331233, 0.0035976889071418073, 0.003598489460971285, 0.0035992900148007624, 0.00360009056863024, 0.0036008911224597174, 0.003601691676289195, 0.0036024922301186725, 0.00360329278394815, 0.0036040933377776275, 0.003604893891607105, 0.0036056944454365826, 0.00360649499926606, 0.0036072955530955376, 0.003608096106925015, 0.0036088966607544927, 0.00360969721458397, 0.0036104977684134477, 0.0036112983222429253, 0.0036120988760724028, 0.0036128994299018803, 0.003613699983731358, 0.0036145005375608353, 0.003615301091390313, 0.0036161016452197904, 0.003616902199049268, 0.0036177027528787454, 0.003618503306708223, 0.0036193038605377005, 0.003620104414367178, 0.0036209049681966555, 0.003621705522026133, 0.0036225060758556106, 0.003623306629685088, 0.0036241071835145656, 0.003624907737344043, 0.0036257082911735207, 0.003626508845002998, 0.0036273093988324757, 0.0036281099526619533, 0.0036289105064914308, 0.0036297110603209083, 0.003630511614150386, 0.0036313121679798633, 0.003632112721809341, 0.0036329132756388184, 0.003633713829468296, 0.0036345143832977734, 0.003635314937127251, 0.0036361154909567285, 0.003636916044786206, 0.0036377165986156835, 0.003638517152445161, 0.0036393177062746386, 0.003640118260104116, 0.0036409188139335936, 0.003641719367763071, 0.0036425199215925487, 0.003643320475422026, 0.0036441210292515037, 0.0036449215830809813, 0.0036457221369104588, 0.0036465226907399363, 0.003647323244569414, 0.0036481237983988913, 0.003648924352228369, 0.0036497249060578464, 0.003650525459887324, 0.0036513260137168014, 0.003652126567546279, 0.0036529271213757565, 0.003653727675205234, 0.0036545282290347115, 0.003655328782864189, 0.0036561293366936666, 0.003656929890523144, 0.0036577304443526216, 0.003658530998182099, 0.0036593315520115767, 0.003660132105841054, 0.0036609326596705317, 0.0036617332135000093, 0.0036625337673294868, 0.0036633343211589643, 0.003664134874988442, 0.0036649354288179193, 0.003665735982647397, 0.0036665365364768744, 0.003667337090306352, 0.0036681376441358294, 0.003668938197965307, 0.0036697387517947845, 0.003670539305624262, 0.0036713398594537395, 0.003672140413283217, 0.0036729409671126946, 0.003673741520942172, 0.0036745420747716496, 0.003675342628601127, 0.0036761431824306047, 0.003676943736260082, 0.0036777442900895597, 0.0036785448439190373, 0.0036793453977485148, 0.0036801459515779923, 0.00368094650540747, 0.0036817470592369473, 0.003682547613066425, 0.0036833481668959024, 0.00368414872072538, 0.0036849492745548574, 0.003685749828384335, 0.0036865503822138125, 0.00368735093604329, 0.0036881514898727675, 0.003688952043702245, 0.0036897525975317226, 0.0036905531513612, 0.0036913537051906776, 0.003692154259020155, 0.0036929548128496327, 0.00369375536667911, 0.0036945559205085877, 0.0036953564743380653, 0.0036961570281675428, 0.0036969575819970203, 0.003697758135826498, 0.0036985586896559754, 0.003699359243485453, 0.0037001597973149304, 0.003700960351144408, 0.0037017609049738854, 0.003702561458803363, 0.0037033620126328405, 0.003704162566462318, 0.0037049631202917955, 0.003705763674121273, 0.0037065642279507506, 0.003707364781780228, 0.0037081653356097056, 0.003708965889439183, 0.0037097664432686607, 0.003710566997098138, 0.0037113675509276157, 0.0037121681047570933, 0.0037129686585865708, 0.0037137692124160483, 0.003714569766245526, 0.0037153703200750034, 0.003716170873904481, 0.0037169714277339584, 0.003717771981563436, 0.0037185725353929134, 0.003719373089222391, 0.0037201736430518685, 0.003720974196881346, 0.0037217747507108235, 0.003722575304540301, 0.0037233758583697786, 0.003724176412199256, 0.0037249769660287336, 0.003725777519858211, 0.0037265780736876887, 0.003727378627517166, 0.0037281791813466437, 0.0037289797351761213, 0.0037297802890055988, 0.0037305808428350763, 0.003731381396664554, 0.0037321819504940314, 0.003732982504323509, 0.0037337830581529864, 0.003734583611982464, 0.0037353841658119414, 0.003736184719641419, 0.0037369852734708965, 0.003737785827300374, 0.0037385863811298515, 0.003739386934959329, 0.0037401874887888066, 0.003740988042618284, 0.0037417885964477616, 0.003742589150277239, 0.0037433897041067167, 0.003744190257936194, 0.0037449908117656717, 0.0037457913655951493, 0.0037465919194246268, 0.0037473924732541043, 0.003748193027083582, 0.0037489935809130594, 0.003749794134742537, 0.0037505946885720144, 0.003751395242401492, 0.0037521957962309694, 0.003752996350060447, 0.0037537969038899245, 0.003754597457719402, 0.0037553980115488795, 0.003756198565378357, 0.0037569991192078346, 0.003757799673037312, 0.0037586002268667896, 0.003759400780696267, 0.0037602013345257447, 0.003761001888355222, 0.0037618024421846997, 0.0037626029960141773, 0.0037634035498436548, 0.0037642041036731323, 0.00376500465750261, 0.0037658052113320874, 0.003766605765161565, 0.0037674063189910424, 0.00376820687282052, 0.0037690074266499974, 0.003769807980479475, 0.0037706085343089525, 0.00377140908813843, 0.0037722096419679075, 0.003773010195797385, 0.0037738107496268626, 0.00377461130345634, 0.0037754118572858176, 0.003776212411115295, 0.0037770129649447727, 0.00377781351877425, 0.0037786140726037277, 0.0037794146264332053, 0.0037802151802626828, 0.0037810157340921603, 0.003781816287921638, 0.0037826168417511154, 0.003783417395580593, 0.0037842179494100704, 0.003785018503239548, 0.0037858190570690254, 0.003786619610898503, 0.0037874201647279805, 0.003788220718557458, 0.0037890212723869355, 0.003789821826216413, 0.0037906223800458906, 0.003791422933875368, 0.0037922234877048456, 0.003793024041534323, 0.0037938245953638007, 0.003794625149193278, 0.0037954257030227557, 0.0037962262568522333, 0.003797026810681711, 0.0037978273645111883, 0.003798627918340666, 0.0037994284721701434, 0.003800229025999621, 0.0038010295798290984, 0.003801830133658576, 0.0038026306874880535, 0.003803431241317531, 0.0038042317951470085, 0.003805032348976486, 0.0038058329028059635, 0.003806633456635441, 0.0038074340104649186, 0.003808234564294396, 0.0038090351181238736, 0.003809835671953351, 0.0038106362257828287, 0.003811436779612306, 0.0038122373334417837, 0.0038130378872712613, 0.003813838441100739, 0.0038146389949302163, 0.003815439548759694, 0.0038162401025891714, 0.003817040656418649, 0.0038178412102481264, 0.003818641764077604, 0.0038194423179070815, 0.003820242871736559, 0.0038210434255660365, 0.003821843979395514, 0.0038226445332249915, 0.003823445087054469, 0.0038242456408839466, 0.003825046194713424, 0.0038258467485429016, 0.003826647302372379, 0.0038274478562018567, 0.003828248410031334, 0.0038290489638608117, 0.0038298495176902893, 0.003830650071519767, 0.0038314506253492443, 0.003832251179178722, 0.0038330517330081994, 0.003833852286837677, 0.0038346528406671544, 0.003835453394496632, 0.0038362539483261095, 0.003837054502155587, 0.0038378550559850645, 0.003838655609814542, 0.0038394561636440195, 0.003840256717473497, 0.0038410572713029746, 0.003841857825132452, 0.0038426583789619296, 0.003843458932791407, 0.0038442594866208847, 0.003845060040450362, 0.0038458605942798397, 0.0038466611481093173, 0.003847461701938795, 0.0038482622557682723, 0.00384906280959775, 0.0038498633634272274, 0.003850663917256705, 0.0038514644710861824, 0.00385226502491566, 0.0038530655787451375, 0.003853866132574615, 0.0038546666864040925, 0.00385546724023357, 0.0038562677940630475, 0.003857068347892525, 0.0038578689017220026, 0.00385866945555148, 0.0038594700093809576, 0.003860270563210435, 0.0038610711170399127, 0.00386187167086939, 0.0038626722246988677, 0.0038634727785283453, 0.003864273332357823, 0.0038650738861873003, 0.003865874440016778, 0.0038666749938462554, 0.003867475547675733, 0.0038682761015052104, 0.003869076655334688, 0.0038698772091641655, 0.003870677762993643, 0.0038714783168231205, 0.003872278870652598, 0.0038730794244820755, 0.003873879978311553, 0.0038746805321410306, 0.003875481085970508, 0.0038762816397999856, 0.003877082193629463, 0.0038778827474589407, 0.003878683301288418, 0.0038794838551178957, 0.0038802844089473733, 0.003881084962776851, 0.0038818855166063283, 0.003882686070435806, 0.0038834866242652834, 0.003884287178094761, 0.0038850877319242384, 0.003885888285753716, 0.0038866888395831935, 0.003887489393412671, 0.0038882899472421485, 0.003889090501071626, 0.0038898910549011036, 0.003890691608730581, 0.0038914921625600586, 0.003892292716389536, 0.0038930932702190136, 0.003893893824048491, 0.0038946943778779687, 0.0038954949317074462, 0.0038962954855369237, 0.0038970960393664013, 0.003897896593195879, 0.0038986971470253563, 0.003899497700854834, 0.0039002982546843114, 0.003901098808513789, 0.0039018993623432664, 0.003902699916172744, 0.0039035004700022215, 0.003904301023831699, 0.0039051015776611765, 0.003905902131490654, 0.003906702685318954, 0.00390750323914635, 0.003908303792973746, 0.003909104346801142, 0.0039099049006285375, 0.003910705454455933, 0.003911506008283329, 0.003912306562110725, 0.003913107115938121, 0.003913907669765517, 0.003914708223592913, 0.0039155087774203085, 0.003916309331247704, 0.0039171098850751, 0.003917910438902496, 0.003918710992729892, 0.003919511546557288, 0.003920312100384684, 0.0039211126542120795, 0.003921913208039475, 0.003922713761866871, 0.003923514315694267, 0.003924314869521663, 0.003925115423349059, 0.003925915977176455, 0.0039267165310038505, 0.003927517084831246, 0.003928317638658642, 0.003929118192486038, 0.003929918746313434, 0.00393071930014083, 0.003931519853968226, 0.0039323204077956215, 0.003933120961623017, 0.003933921515450413, 0.003934722069277809, 0.003935522623105205, 0.003936323176932601, 0.003937123730759997, 0.0039379242845873925, 0.003938724838414788, 0.003939525392242184, 0.00394032594606958, 0.003941126499896976, 0.003941927053724372, 0.003942727607551768, 0.0039435281613791635, 0.003944328715206559, 0.003945129269033955, 0.003945929822861351, 0.003946730376688747, 0.003947530930516143, 0.003948331484343539, 0.0039491320381709345, 0.00394993259199833, 0.003950733145825726, 0.003951533699653122, 0.003952334253480518, 0.003953134807307914, 0.00395393536113531, 0.0039547359149627054, 0.003955536468790101, 0.003956337022617497, 0.003957137576444893, 0.003957938130272289, 0.003958738684099685, 0.003959539237927081, 0.0039603397917544764, 0.003961140345581872, 0.003961940899409268, 0.003962741453236664, 0.00396354200706406, 0.003964342560891456, 0.003965143114718852, 0.003965943668546247, 0.003966744222373643, 0.003967544776201039, 0.003968345330028435, 0.003969145883855831, 0.003969946437683227, 0.0039707469915106226, 0.003971547545338018, 0.003972348099165414, 0.00397314865299281, 0.003973949206820206, 0.003974749760647602, 0.003975550314474998, 0.0039763508683023936, 0.003977151422129789, 0.003977951975957185, 0.003978752529784581, 0.003979553083611977, 0.003980353637439373, 0.003981154191266769, 0.0039819547450941645, 0.00398275529892156, 0.003983555852748956, 0.003984356406576352, 0.003985156960403748, 0.003985957514231144, 0.00398675806805854, 0.0039875586218859355, 0.003988359175713331, 0.003989159729540727, 0.003989960283368123, 0.003990760837195519, 0.003991561391022915, 0.003992361944850311, 0.0039931624986777065, 0.003993963052505102, 0.003994763606332498, 0.003995564160159894, 0.00399636471398729, 0.003997165267814686, 0.003997965821642082, 0.0039987663754694775, 0.003999566929296873, 0.004000367483124269, 0.004001168036951665, 0.004001968590779061, 0.004002769144606457]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s_1emin3, net_absorption_emission_protons_inv_s_inv_cm3_1emin3, label=r"$10^{-3}$")
ax.plot(tiempo_s_1emin4, net_absorption_emission_protons_inv_s_inv_cm3_1emin4, label=r"$10^{-4}$")
ax.plot(tiempo_s_1emin5, net_absorption_emission_protons_inv_s_inv_cm3_1emin5, label=r"$10^{-5}$")
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
# ax.set_xlim(0.0, 0.005)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{Y}_e \, n_b = \dot{n}_p = \dot{n}_{\bar{\nu}_e} - \dot{n}_{\nu_e} \, [\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
ax.set_title('Ye time derivative.\n But it is not integrated over time in the \nsimulation since we use a frozen fluid.\n')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)